# Nutrelle — sétima execução (seção 17: a faixa acima de 25 g e a densidade; só cabeças sobre features congeladas)

Notebook gerado por `eval/treino/kaggle/monta_pacote.py`; **não edite à mão** (edite `treino_gpu.py` e gere de novo). Lê a saída do kernel da etapa 0 (`nutrelle-deteccao-r5`, anexada em *Input*), `listas/conferencia_r5.json` e `listas/ref_r6_prato.csv` (R4c e H0c-m fora-do-fold da sexta execução, para a paridade). Baixa as fotos (as do ACETADA lidas do zip remoto por intervalo de bytes), roda o detector da etapa 0 no FoodBD e grava as folhas de contato dele, extrai as features do So400m (recorte e, no FoodBD, também a foto inteira), treina as cabeças de 17.3 (sem ajuste fino; cada cabeça de densidade com 3 sementes) e grava previsões cruas (fora-do-fold no treino e nas partes seguradas, **sem pontuar nada; nenhuma do teste2**), as features de todas as bases (`feat7__*.npy`), os arquivos para servir e `servico.json`.

**Versão do repositório: só bases liberadas** (Nutrition5k, Flavoria, LeFood-Set, GT Mensa e FoodBD). As configurações que usam OmniFood8K, FPB, ACETADA ou as receitas ficam de fora e o manifesto diz por quê. A versão completa é montada fora do repositório com `python3 monta_pacote.py push <pasta>`.

Kaggle: *Settings → Accelerator → GPU T4 x2* e *Internet → On*. Instruções completas em `eval/treino/kaggle/LEIA-ME.md`.

In [ ]:
# Parâmetros (os padrões servem para o Kaggle)
import json, os
RODADA = '7'         # este pacote só traz as listas desta execução
ORCAMENTO_H = 3.0    # horas de relógio; o que não couber inteiro não começa (a ordem de prioridade decide o que fica de fora)
SAIDA = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content/nutrelle_saida'   # no Colab, aponte para o Drive para retomar
EXTRA = []           # ex.: ['--so', 'ridge']; ['--dominios', 'flavoria,mensa']; ['--retoma', '<pasta>']; ['--espelho', '<pasta com o zip do OmniFood8K>']
# as variáveis de ambiente NUTRELLE_SAIDA e NUTRELLE_EXTRA (lista em JSON) substituem os valores acima sem editar o notebook
SAIDA = os.environ.get('NUTRELLE_SAIDA') or SAIDA
EXTRA = json.loads(os.environ.get('NUTRELLE_EXTRA') or 'null') or EXTRA
os.makedirs('/tmp/nutrelle_codigo/listas', exist_ok=True)

In [ ]:
# treino_gpu.py (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/treino_gpu.py', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<E};T{#^hOh(bq-BQwKy4>SE$m+hy=@aif<Hp(x6uSRyx75<tfLmlr9v8=rA^vK9U_wKSxGTYDxj-jWATV6)EKC(Hm
YNGRX;bmU+;6s%e2My;AdQ+%8O$p?ZCbjDzN$>!xfgl28Q=I#TvJTAmr^77~D@Zu{uA8j}=Ah_;`+(Wh7Pf7bceFsb$RGZ$;$m)?GKB?9<~?!B1vArY#MT#@
mIAho^Z=^{$g_MwVIU6R+-vlNl8?BI_hHwb%$T%+eNqIhigtVv!`j{?O7Q+YhT=3|SaqgO8$j!W`>k|m9zhNv0uDp5Xr3Q^6;J_^66184ImAO>Ppp50eUOUl
VD&>9$J+8n9FytF5)CM~Sc#tdYxtF`hWZz|<d_JaC{_Y4R^}Ij(EC{*VE_;CXsX}VhcYbuwO8~%Y#jlw@lc?7#bpik-7eRYpnBU+;&;cT?6a+Jgm6*e!rBs;
mKwl`A@b7S_1st7`0v2=*3SJ@q!Ng+Dz436NEB|wU%m8E96_)<P-l^1)8Z?<H?6y_C|Pe<q{q@wqXQX!dJ#C~>+^DJ`dry)O_26bgb{w+1LNuP?4__ckTBNH
8ZM%F+=GRyv@U9Xriv?F0cc*hfgBhv0A&Xfo0dg?gCwhr_Nv2Cvv2xm=Kg~d9_%egGaM2gHrbpRp<p_^7`FByodDM#@>~_1{YUjb^lk%zA9{jw^h9RP*#)bp
rqu7<)>u@JT`MoDkSEXn+zcToA|eb4jJ7QgSU)TBk`U1mEvwZ1d1Mo_5dBC793jB&#7;t7sK70M5}=QTo5t;EWX_fVIw*rY2fC=vCunV^cn(e~faHkK9yb>I
-ID}rWTL!B3<fDBL}uT+Am2>PTAH~+P5RN;$qv?)0KjM&1k&KG3`=wOe3?b46*LwXfw_$!Vv!Z7;tV|2_ZS5IwTQz68?1dAI~=u$OR7y;MGV+<P82%<so7rv
-GxjvyNMxuznP@-EK#>sRosEsNu)Cb1xW%JhNWLu8%%20`kE<9yU7d8@{j{|D6Hq}(Sxywk?hubJq9N4gaTO3$1JX%$W^`NwyMy(s4w1|3h|Jjie6d4YB~Q2
of6stzib(J%>po^qmT#yz3z>W;)88Gf{nX}+&Ym+aeZ+^Y?!J%A~&m$b6|@`AExp#Y%ozq@OA2TY%RD~p1ZZHQ6C>a|BD$+pI7iK5y#?tG9efCY4~UAm^w?C
OGEcuiTXKYQwMR3oZyhdL5qLcA%K2niCI?4&)(T|pJhn6IO}y!0v>kldb($B;WkYdtN}v~tEAWpqbx=NNUYM=>y_!`HMYg#tL7FsfKJGq9TkJH1e5+gtwYkp
jB!kU=|OcOdtkrqqD4H-7p5(EjZpXT8}cI4Pql@8Z@^u1d^XQV7C?Yc!3#W>99B+#I1A2|Xb||1>LcZ+tyg?kNlqg$-4WLW2R>3T$;d_T8HR&?CVMY*nq^RB
9yS6dF*U;T0wwZ%7OvJeg)K_p9ilWw$RyGtU91@qI@yf$P2civ_bRp1ys7pRn>B@~7yShI4Q&?SUCX<DVwkStv)|rJBr?eS4bd=He@kXdP`5)ax7&I_Fw-iB
2e{{U_hvt`P{yBka#iD3@BJIeaP8g`>8gEpxSl*+5bB%-u<>JkfGN!F=ZW12Echb0!#=a69`qoE5I=rHr8KO}`TOXY;*%}e=b47B*N0mlSXAZZFH{e=cd~}X
iAW=wv$>OC1EHtIqXxG*;?FbQ8uLSHZ;!tRXa3b*KAmyB_R_J>J`~jFp}$TN>ziKC;Dl$!ayk{@&-oWm%JoTAuOz3Pd2m+YksbbT%oa6Qt0XCJy1V3&OtYd#
n-R0w1kM{WRJ!S|LL8KftvrA8E$E<sjjMV<N8Ye>GKMM{)7{QoFeL?t87cO<sYGTba<<wZH2IMl%p<p7^ai(hQxey)wuGTd+DS&SU(2vYV&;<h@3yi&$^(jo
;K<E9ZE!>yioi(FNzhsYy|9ScmyQA12T6`Vgi0h1?o}T~@JT$k6_+;-$TFHtOK;i(>@pWz+*2z=Pv-1}+-R~rcK}LyVPSP1`rYkS<!}%dGRF(65<E>n#^0xr
q^$WO91}M%uGWupdWcuD=m!o;D;6l04;1!#w53s;#yv{!X$#7G*gm@eKLjrZfi@sviNY>kKGQ0EX&`b=1WwLGQ)zG))!P9xm_%oQvBJSy>``!3V2GQ}De~ns
>O4~Hfw?uXr4mzU2!d_H$*NQuyVAhh*jJkj0G-n~5MhQ!9M}0jgfs6Vdy1Fv9byO@<t1j3)H6pwaxaCC-+MS}6S)8KURSZC9{Brq3oH)=l!?6c9ajp-G4GX|
0n}J+0yAZGnkM_(Be&M%(6`i^Qw*6UGUT8Ev*ql(Sc50Bm}apCPfb?E%rNgPc%?b9TI7udl~+O~%ltQ%_)RQFW~x|>u6p>FUb|Vo%&@V<XC&Bk5(^*O<p!9o
qg5)q<YbL5g7jb|is7$th$%LF77QguoV#U4x6&pgXD2+F&Oo&VBSI{<%^onm8j}cCc=|Iyx<oqfOi;=4@uvlCw;x$K7~al{Xmy`bHemR5Zq+182Y+qt>x8N3
hI^hgk>=YA!uWIEw|O!wlAV9)FT4<$r5sn%&Otwt%o;FK)qqWrZOAtpB3}LlfuJPdw!m@vdq__VP+9MZ#F?|`OPt~83=+tk%|4(+sU#GkWk5}1>!)<m$8(7m
f$}tPq0E}CCs5=H4d^<7@J?nWIu;>5l<cIL9RSg{{a%}Fk6ouKs-X=!_wcHhwADSa&aq4X>t;b#i5cWt!J9jm@2i^DWFqS1D!^uuY8zQ=auvHTVaPa$S}yh(
W~AXBDLQRItIayu6FIn7ms#emS9@jOc|r;zm5Z>%@mxWSEMW?l!}T@V2!{|ExV@e3D4aG9&8u<&+@ak|(x<Q98xR`7d(jF<7g&&H2BtrAs+$0{{C)DABifSa
e9`SVYzg2Y#6=oSii#YFZ;41DB%MeyUE7!ccJsM*i4A<2W6Vf(G<4z}Ob3YeCxbf9w69tv`LN2y@Xf!+bqbne{i=+baOMq`Kmt8N5hzH1;{6kk;EN%a;g1{|
a#~RSl<bc{%c58GEEAA+XQ9)m5?<Szwmdd~8DLG_LaaUq$$80SkQtsl0e`Dy&z&}n6>Ma50P$(0J)}7%Y3QekKn`OGx-gYaM3?VKhen{h{trnQsv?<*FdwIH
=rVa3X$xf?@O}n5GfjS=RS%(k%WS+C6~5(1yC|$AD6~oG0A%*J&VegTF4}$y0b^x`?W%8cKX@~pZFEPD?nL~z`)oXf;wty&=<xbXj?=>&tCrubyxEcIC|Bfd
GbJ`<1|FPmT5j4MAT1=&X3{V66OPf{o*T=Lce=l=SBlrN7ixzX2J+vdCNah@XbESIoa6>Ds*I(b<x6_upyX~1PL^%D&YK%=wkgWCu%&y~x7|t!%j^px5u>9~
y>O-syynWAaY5O_$||3v8EOolx2|!vkk3?$`!CuuTpX)m=b<UCJ|-T-5srw`9v)p?g;BAHh5T&B3rxc|v{qA;ZuLLhrK$Z6w$9s6vrk0|WX$o&072Al=X>Da
GGq};Um65tEY-3s%>r<bM*}XB1(tt;TRus1xopDv%u7EC<Ag-2OW$DcMusaLv#KWADM3H?=<l&n{lG*!i$CA|jizgxTClj%NFLhk#OzJFa5#c?-er9<$sy<Q
9Yq5ILxl5Yx|hZBpi)^APD_XJ<bh^G>O6l=f4F0|FD_+8ZZE~n^;s3R!EEe@+d~9izHKy)e>#ctiGDaB%W{ofzX$<js)=sKHxWH^R|L=NP=B7>ZQDNF?W<8G
r8(s`J)@pTR%T<(7BKj9$l>48|FCogishAT1IfT`vc0!W3>={Kn%;IHd^3DBmIBCSx$G|U_6Utdti117^}(S8thuiCx!pP|kRsaS*RS)$<zuZZJut|?VBW+%
QIdPB(xQe<7MmDjF&*KJKc2kc=2=XBOiKhl8nKrWIx)6J7z%B|xX)XRuG)suL$07DQ{=TOdMOrCd0Kt0!0`eh`~c9_*hTM=!1nZ$GDI6izhvPUjtTFX*FBMX
;$TEw_-E_yIQ3xOYzFYvEEa2_fqkdXc{Ws<x*GJb)nP?z&mnWBsflUhdF=IDEA<}ZCmZv9p%zK_TYc`~?10~fOEI(Td1HsoO+^;?<cedIlv2Z>qJ6kY<75tY
-D^1|3e@IY*P#-3sC7=BkIZ5f*9Oq0Q|M-V4M$AzG$ms5xm^k7Y->xq-f@|g?GN4Rq)pQe=a5(xRoU+4f`V@ggOG(1@mxPc<wIMLvcao<^B1$WI%9l@k6D=z
9F6$i@5p;Oq<n|)UOt#9;r18Tj?VcO>DylzFkOREj(>qpH1D^h5TM#aR!*eBfd+UXR`vixVnMtV=_O~;WBRXsD{B2Moq~5kINFY0JKQ&`&iQ^M0%7F3gZ!{H
i?a4~n8whYWu30uB`Kse<#eSb8oMvFHepYL>X+nMK8Bc9qBr{QOyN?-pijR8FfPxoW?c^@rp@=?r*xawmtdOD{;1&de~grK)H~t!1ex1Wx9MnhuN7$B_?9~o
;{O8GnWj~^Lk1sy<2SHsAB-beICd5G69E@)DF$o%d&se(Txu>$&x+1-=nV+4!NJ>k$cbl74M&Q)+~}puMt<12z#o>gFR|+`{-!~+)@QC^)|D%<hcO~w*MM0j
n$&*R^YNLM<wr74Ekaze=0Ir9+|g%d93|!!7wf*^v_So9HLT~(#UX8k!@QPU0esizO%K|tcIj=L`$)ocD@aEZ7}pOziD>^o&)H(WHDavsrZnJsqK)<aYdDLb
^T|d6YDPS<T$V#x=RE14t0Tz0>}mI^8QGz{L%it;OT;|s;YE*hELqlT&{X@JKT)GCWJJCTl)}L0bw@HEhz(q=4s|}5+R<H&69xDlwsUrGFQ-qU<_fEHtk|By
;>a2|Sca5Xi^JR4x9E`fF9)Q<-(q5o%kw?0vChr@)@oV@@tX!K0Cu`hepgYIVY{XJW)YD3Cd%E1fj)zUey;G1k+&Nsr0y<s&Naf`q@O6>voN_{?z9R;0>S9-
gx;cB854h`LiShazuqdqp9&-sqv9qCUjGMbj@d8=!H0$lCD&isZoil5y+Nq4*i)OBWIGOkHF_9C4E_7acvBIucj><piC{4w=di2377fG<Pfd>C@p|aFh~x9G
kvFn0!Zywbv9g%hs`kIr8&cGnxGV27o78N<XY)J79n*vl!1wv!I;<l0a1bWx)lYp8B3IfNeKeD<${SusmDTRoir`*>@k|=JWjX$SvV5Pt^yj%u6g^}f8%RMM
Cnxy!55=^^c^6q#ft3x!?pK5(JUK^G?c!=HW#c~H6-+en(j#L=E-mQWz7Kn`1;ed9DCVWVk8`{5adm;|&`eA*0dS+F5lG^Dp!Ua!_);#EbKtZg^igJxyJ)ns
JdUrZLDTUe%~xAQz}75FG~7_`(lsOE>n<oXE^F8wQ4jRqLLQ<<p_GDJ4>PseHtYE$BaS=`7f$5>klzC^Hh%t6<vJUjsWXpdXtDa=ROH>AdC9#BPQWK@fOVlp
ePdBK6T`FG45*d^ZvZ%KughM|fwQ7O&qRL6`CXat^r69~!*}4Sd1X|LcVRt_z>oAlFG_ehp6z^g(T$H6sL{2%#Ahafcv3dAMG5x0%J^19ZZ;w6>KTv&e!1gE
SIbB<X<8K%_(-`2>=4*wnQ<ZuG2phGPb9Jy3}?poZTYwxp_MwGx;EJ2fFpbr)CSym-SZ&A^cAo1)WTPTB(&2j!(hR6d#`ze6*#5(Nj3rDf)OjoV6oau^0Tez
YTH=qQihR~G`T!GV<g<JkDZvooDHe-hh7H_d1~$llJ&HYQW?c861&{TUq~=MM&rV699OWMa08K{0oq9SM2<pv=$j-?ojE4!_3=WTlJ?<GTbJcx6vBPRNP(2(
y4U<{Vgr6NHGE^H^5-yP8-D5iH5qz^t}YkL!}cYWYA2pC0YA<g8Gq<fLv&1@9tN@TMd83NPvQfk9X;88CPHcc$sg(U?8-S3r~wjOjMkGyl1D8=l0N}h<d9oT
oQQ1!&^h{k*w~=b-jb<zkPDQtxpi^zxKmzOcv|_sk&!d>JtuRbIxI@o!c6)lz41tg%PI<kCwj2fH^%-#={m`d+h66+ibp&bi?6l+VHlWd5&6{RPrwW_6C#y2
)mxe$^q4wYje==qNbG0R5mEKnb_B8Q3xazRCGnFBt|_+brGhER=FplOxX3yxAL>4x{I33UMTYajo5pN2%8zEz)81GTxvaBEDz;V>88VEPuE8VxV@Dctb!y{v
LHp;qL5$*Q$gG|;;2Bsx(Q^bR50d;PTn-E|k1B8l!Dp9>^8_t6BHO(y&X6<r@-9U5AQG9nPng00>;Zd6NcB)62)WO=V+y!E<S%J#p%!0!!Uxt?0{)aeJj@aE
G1{!yt%=li-H=kSc1E6W!()nE^EdX$^*wUcaxUzvHU=g%^ydcJwGZ%dbAowO{DMm6WT0_Dfg2UxMGc6iln`;A_-V4w2GsH2vOyFjSf_bJ_#}wjX0;$Y#~>V$
#M#2{bVV)*l1WsVExh7Fb!tN3Is?O~cfen?+Ref4+kD!XtZNTi5Yhi!z-of5%uPURNnO2zi!g`KYsenvD0TGz#RfqIUJmuY9Z$<xsSktuzF)SIvrwxNp@oLJ
G8TgJZQz{h$>NNx%HwWW_M_A{C{R#TqM^+Ge!NIoIw;B|W5>qz(E6?hEmFMBh8q}oQpTubf~9mSnAOv^xuaToER)-$LjHckY?Aapvdl5avq~OuR$2r4@K)mP
p8tQ~>PiC@9lI`o9+y~*&KM0mj-9Wkni1lHtQ0xT<ly#=qT>8&DT$IwnmSzTjpml-PbpuGF|HmW0qOKqE@(1A^eC%t%pe&d)a4x*QhV1*#(jy|1rZoS!|iNE
MEq|70bi?BNTS<5)9A@XJ(FwB{{HlfcDis1I{PG6j|Th^J<sch?0#wy=EF|WTqxun6pYy1yvtw~SYoFkL*wM#!rwnmGrQwxTM{1>=m+0WfVwp`ftT7hg-^rt
lg49X!@#V07Y7<`+f>D-^V}X)Z(Xhn-0eD?gqsjpS4qcR(3^!idcV@ld;$Bdzw!xcmoL5L_9D!kfT=n1HEF{<@$Pz1l`|}@az$@JyytjRvhU}*+aue|X+XLo
tCbQ3xHxHg7LBw!@)+v<sI?s!tBZUa@f#ieS!hwZa!~=zx^kdMzQ6;z;x*RlnZ!P>FWLx~AXGW$WE|5I#p<l9eK)@WLt6?XG7pp&Z81xt+(geeM{shbRxW30
#od#;bQ(*>QC8-z)#MTtGN^;3+uH549Bno;xy>1}7d4dxY?Em|3aE5lFE%&9=;1D>`G35N-#cac&9(1Q#ypRCHJi@K`jvC1>U|2=ba@0!qdWL`L7_KZCdvoa
^Bbk4CyUY5v%)uNEyJPOUsow7i;Bk=UGr)GFwrLYmiR|~*Z{CF%c?=c;K6_)fdaA<HEEtNZQf*DAzJK6Qon~CCVDw4JMDXd(?xeTYuB`s&8Z@OnVCiQ^U5&n
OJ#V3NUgO#YWQ@pbM|jK@{chmOYh<-Pb4Q&jF61;Du`$4MTlBjzXno!vs2*A&8m(u`+&7Mj<0h%6LP6-%hZv7rN(x&ZF**y{W9&QZZx4bhFv5~R>!-D3fUuU
q8h~mY@GM0<b_?(N=>FKoUq|wP&>Gf60rqMCOuFppqru?Q-IsRRydLiK&0|pGpJX6L=!ogQxJf>qG`xMYDSG5v*I+XIoHq7<67`ot$3n}M-^Oj@LZ6p0O%~p
hNF=~ep)FZ+(j1s+PhYas)^LV3*4MidU1Uk6Ze0+OOcJh(|;R=Q6Tyzh*8a&hZaOFPRbI)sH+H*G>S*dKmrE4Ye-8jbo!hN4#;k~wPz}^N4IsNsv-sBciCK^
x-Sc7Jz2x$gQ`klfID_IA?$Zp0}XHqNC9eToIa-DTeVs=gxEFU0($VU>P%3R+c`(>Ygb-<v4s5Jtg>rI?zDFMp8Fm>{&+%8rbeXGR|oHv9mNkK79b_yRy22n
B`JlpBXveQk>~uzHu1ko!&`6iiyF(Yp$`(TnTl0dNpAjfl(nT0!qZkK<_NUB6IpZH;0oG415PyQW#ltTI=Xfj=7jtQ12S(vq>dk*_>$}sjh4bqw97+M!w5_B
Ai%C~W(`k$eq-<nG>(04lN`@BPle$d=K^`r5`#SQf9iwBqzy)ju@i21|8kE3vH}<he_~_XleojjZ7wiDQv#YuOQD&e?$44$#d|`RePuxPqrhLOY~!}%dIRd=
^z{;3cfK9Ym68!EFta}#{GyZ5&QkvB7C{-fQ+^*DNEjh@p4O!R(u*^)G$??!nOxq5-N-?KuE5rsQL0xRiD!l0jDlKI+x2pEt>G#Mlto<S9mca!LYoRV%xp);
fu_G_yY$0=w3G2GXw-q1FL^QDRrcH4t*ZAjRpOaJOewoU{JE8D_S=FePN||s&YX2B-)kY_N6OeOUMT_4)<iP$8cvLdObzh8N<^#OE{2$c=dl=ovb?C5S0RH0
hWO{hEq8@?`7&u`+#N7CkpVDdA<aTqXo%Ci19jxfja+RxH{(1b?XM;aZeI(-@<~XQ$R_dmW>6b+po}v#ohtw+iZllpcI!rb1z=GXb0we~83b{)@5k`n=zon!
aMFAdY`2c@?FyEPe{v6!r7veW93Y{G<638ZL}T!>(2ua6Gvue^wo%1Cvo<#cevfJl^}#Kw`nN~!DI6#g%B=)@k_&O1vTH~V^V1)-FBLw3+roW1#y^7J0&nJI
xO%bLmmO)53A*-)rR``~M&!Ni{_wnCfLv3rIli_9bh1;avMju0UXF>%sz@3n&=~mpgZ{aGkl(Nn3M_X$`dqza<}j<Lic0yX?digw!|VHk=&wn=y91vqbOVqI
9yb$ok8~=gS5==eSH<ZI*?-i@B}Gt8M|H86+?+7YJw=~26|^8<qUkM8`^u(7nAbMAi)#^-rTMBF(W|LMQEMSO`&Fm0?)MhEnEFZ{G&_>E9?b36{Na6J-g*{a
k+CYA=d=(v_0d_#2E0un24+pJ3Nntz{WP8|w`Tyt7`H;HMrNQ+G&o~S-c6I+dg}Z@|JPRO#!!WcS#|W63xp#x?SA_W|J)$BfH(!fz?O+UEYj}ZW8R%e4Sv<i
{rD0@Q$A^!yqD8{yLwwH>Oe0wGfaJ-W}uSE0wgnR;8^2rnzNWXiK-rZRv<iYr^x;v-g1TTr%tgEjv$|FJO5m2LcI#_cc^Bu`%!!-mvQU3aS+fDH9}GP1k4V3
UuFbNnt|#;CXe)=qo5-HqTql$szm8_$Mt{=aMcEdZf&eQ`XvEGbfpGBpR0X{#RcNLv=F5^&}^?}2`eHKtFC<kQ7fsp14Z9A#Aox5gS~?kUEZa$^btB+Y;=r@
;&VA`N7(YxR}TBm*iJFrr^a#IT4T{zRK-Ph5Z0ogln}ct#E)IRfbfm39uG7~v6a)1&?uPjMfZAM9;tJi5!9=)ripHf9{!8QscL03>UH;2Ff&VM$`|{W8*Fd!
x{QBN=U9p@McTlcbPAP2X0F6^Og}!F<QX?S1-CgLzy7_4f18JinPk5kO5BgkUmYWVjVR(2D<M;3M)_%bBr1c)y2~gY3?z(qx$|X$)vW1B<qY%z=dcG~K>=)~
MjXi}yKF{o)xI&sV(Ez;kgfu=UmsXQ8QPD(i-~fy`~5T~5G#I_US-$#`}p9Vxc3hYv`rc#>|SA^#xRQ=v}Id$@T$yGveNR4hq68r!D(_c<qOY<kq*T$y0n)3
)`bHaSdYb)Ny(eSmmvcP3{N}WJ)46kyZG{ZQnImNRFIm~|KfJ5!PZtx1ke+dq8k(27Dn*%Ft_UPP0wjbEzDmO;sv2ubD^bFSyb{;zJgaO<6Zz*dITH!c&S+9
TAad)C1jadV-Ba#W=i!8Y=HGfl2u!@qExvok3gUzzP^VvQqs4B_<Vndyj+mrX}=$aQddL2t2WbJ@;zw&jv5Q-jd1>5VjiB_%cmU%0sCN43@P@3Ad@=s@nsZ=
`bo9~ne<DnAYup?7i==iAFEh5MMT1z1d}tsp+zO8@IvxbjRH{z=)m^mEcjY~-d*gBsL!r>ii5xvlu^Sow3|Q_;alF*GUNsOD9;Se*$xG3vzGaJ>%Ou?pvG^i
&|RK%oT8M1BZrZSNwq1f36+tu_gO-~{?8c`t(jkKwR6m~k;&K<_JMK&!A9Fvy#s2UvaFmFsPZKmQ_4C8-AOhZhA|^`Vc$?o{mm^^aheu(5NZo@3ST!K%&*BB
B}mro$x62tZh579xLBH_*!U>uF*_V%B)&AHAPR3NpW{@}i<dZtAg|gCY{YwD9%R=keuRH7uU+QeI<JzXP;;6A)DVl1klJzD?B~yb&*P-}==<_w&J+hv_w%r0
T_02XVKW0RUaH6y&NTV%^totMIOLC0(wh)vKnKnzkDxEl^J*?~a08YlHPt1FN$s_#xt?YlT`DU`v}rJxRp4ml0JCNodWv6&2>|KLIP0bjAN4xEAqOMbL&88t
zvro~Cr$xV2bsLqEOui<hpzH{eE4J&cYqfPh7M|+3-t0i(aqU}Nv>IGWBJFt>98Nz0<vFfhNKVC<KU15%8O8~a=yH!vC6%a5Q*g*8_di;@OupnhIsr{82*1|
MKpJ~DN^WAWH;m^BHhs#$|%^$vR!MPB40d@47wXu&8mQ%>-dOzH`c27tURt6BF+D?Lw0;+UvKH`6@iy2+Z~x~PZWKRd{^6WInDzEH<p6BRbeD!laV{7jo`-q
U7k$*YOAY?Txb}s=0(Ui8X`0j50b`ow@R}_Zq$c3j`Fk|M*4R~j783!acp;TS<h{0md0r{=g3hfS66~U@Emen-D7huI*SO?1GBRwNHXg4z)~jMb7sRlcuf^x
tYuf-#z<D!EY#{*s+qvA3SsVX0Dj3f5VF2zvcT=wLHL3Da}eqTgT~5O0k-YLFY!X(3W(=FA}a`j&*Z(Py3^y@Ew>Q_I~h|r7fkNij<QuZ?H^GcCC?mr;&4D<
;w|gmdPH$$2x2764klJ)8#zEl+M*kp_#}dLdyO1^V)XYwr$i<x_R&vM9OG1(vwTp(*KinmpdIEs96EM^Xm6k(WyvIE>eD1-N}zBHn~8G&J?2Ob8m*fzUe)AN
RG#}H?R_htO0WU&u3h7@oY^2giVdyAK9*n0<Ua}~fhf5x#qd`4-Uz8%WLGwPSl9mcr>Z9;eO^=%o5=`!`0bp*`U7IB+Abn2CAdBlA3t+t$MG9)`P+G;25BMO
-rQMj*?gDh1^}Y9sxxc#an>(YF_u=i!jplNe+PDLLzaC2Hej4_wlkWDWF)uymZANe$BU=Ph;D_}CXU4q%FIw;D<u2=G@wOJgs~R^@hrzHz|^zIdjjNQrZAcv
EMCcUzzZQSkCd^4UXm3~JX#|1HX7!X*;v<_cvt~rWl4Xg>btV&PUm6xW|RB6xM4*KI9;BE(pDPbqy&GwUp%_8T8RRy*^W+g7rcv|tKd|5k#;x@Ge{=7h8AA?
0)b~v?jsH=heyo-!L)WP&v&J1T*wV=$9&4;Yyk4^lE|ugMQ*T?G>{_*mH7fhl}B`W3aXm^ogPFSDs$<BQj<gy*;PCDzA+WB>yzf+(-6ko^`;gW7SiykZ;ms7
sJ`9EGZ$?%isJ!2%k0=1suAy#h$Nt#GdYjZT?k5ZGs=44jI7VE#x6`q$^sctrgFk=;ZqXr6U@Lla`^MB8E|33tuZwU3K#E<A86EhkD9Ic1?$zxeIGyFMzP#A
(Fg7g+tLLV8mzYK_@BCfMrXfOA7lU!VLdX}au}yhxxlb6a_p_yTEBNU3(LL8gc|9v{gXRNgfUG~wM2?{q5WeX(Ge0^$aNCRsP39N2)wrD&F`z!cyRXC!qD5a
H#nye>K?(IBZH?;0>zbehuh%<In>;YPPRcMKN;s&JL`<ZFyMxckoytD=&Uc0>f&HDF2j{`hk-$^pwLi+r5=zZfFw~W9y^d#waPqvbC>@%aJ$Gj??piOXDfz#
tza#-yqOmJc9Fpt8!Jg1r=z>7#fuFsbr$%mWp=4jDP@-lJw2uiR?!{LWw3?U2rmZV#snXs5-xD4{kKoZ0IwZY-moDvP7E_3gpl=e4Yexn+tpI4q#RCISqTKO
-gfu0*lP<RFn=>rc$fqr@NsQPbFIcE`#qWGG?!qxHz(m+7}S%0H*}A;0T~6hV9G>C7^-_ZVRc=aOg2EJ1bTMtB$O8;<pc(C;(Q;?1H905Umj#!2zPF)WtVsr
c|R>wr>+)g`OGw+kVhXkf;Vp7Y`7(l$j)~`?^LgpIX&ME13Mn&J4(^6=xkG{Wvul;JHEmwjWWKY(zjw;1wXXQFI7qgEso`3$T>JY2sQz1ERPukWBMbw+)IGZ
Q`qJXub6Ou90qwes$L8^ygr;@Ow7XvN`pln2}Nv#0z-abjzZLY$sgf7VcZ6RV+f|yp-9R6h^yoS0s!rq)@1hgHybC>i%T{>@UK-9O&aWq-(tt{sJWfTQFrM6
llrzd?yFu?);BteOu~iNl;(~#=CF~Sk(I}as!OL|hcs~JNnEcK=0MKxs}@M~5R+Ciraen1jW0hV0j#cI+X2JpnH9rEos)B%j*4z>xADxuyIcSdBR1D<I!$Wo
!|RQTW~j!LIoDR_yffX4>qZC#+%++8?cj=)C2TR*&zi&Q-PrYCcRxHmgHAJ$oJIzf3-$1OUSV{+Q)ESHbEs0thTNOeQPbrz$uMFkiEUuvO<6lpPU{;upKIcc
2p@9f1K*YcH}wWm{<f~`-{TNBOeU@NNg_L$hZNJo1J@+ZL#8j4jdi4OArVI85}P>o`FnPmtLN^N(OTt@0?91o&IAo5HfSLH3Yg^He99D<&iK`DG26qX{`b?d
+6PQij%-i8>Oe1l5BQ~gMJK3fJ^@_FjvyM+zw(*hqb)TSppY(H&HcN6U&h+Y6+j#%_wVWIk!#r>b95PujGkT`RE7bTb9$~!fAIXT>4Qu9R`xm!X9Rxu<x{K_
EyjTyN_vZcEgJM*CC(b_piEG_F<!rsw3ff<i96MGnVWi%&YZWU(I-e1Mf_<H(vGO0ESA1DuyQ%{is4Vmqbl!;HU;`sYUJ&r->w_4+pp{OG^tBY!Gaw}g6IrG
lg9dIajnziQN<UQ8H4-LmQlTGI;jf*If(Uw{132yno*D<3hbJ-8l?CIAj~=5TRVQIOUiAxkkGs@LJ^c*KCy{`Hrwo9@d~A>QBG%B2>@F2zS6dgNz~d`FrRYM
tyU$!>yAlyZ9d;SI}eig^zkFrHU$*il11J}N0)(In~uv~ZKri{J*q|b>VD(X>ARR{9J3G6`s!ElCf~3P^DGHAE5T{E$Qx=%ED7nY47Bw!jKxmZPZrf1Al;<l
A|s+Qc;IXkl?|<L9Y$zZ5ifrs*s)VDxbmOOX%~t{%qti2X%DTh#QyBg>JoIKss3hX$0%Q6PN)RlM4u98;0t5h#hz_U{EmRq(Fw!Q!JpY0f<oj8H7}&)Ys5cI
H@@ME(XWr}^4DGhb0RLz$EEbeoHn%**hqnRPTQp_aLQ9yd*NCrQ!E=|0EPq!ACo+7m`r?Wo|N?-i#DRVgb?vqh08rjqs;bKaqp#AUk_hQX>yzd%QMtu3TPV;
%M#9J!kV}p>~!u8xUl1U8re+om6m7ubU2I=x`@$vH`1YWzaa;Qofknu;gy<)%@avKgLhPml}Jv78|f{F?rF8upesVd{en}loO9k99F#@d`%|zHv)k4&vC@$H
n4?3AZAh(oM3h?`%whg>zzU(x5`q~{&Rd}P(HuGA7*;r#3wis<!<JkUHnQ6zw$)@BnK=t#rc`u{TF&{RXv(r5cfMhuanr&J#K%ucJ(tu%OPJ3YiBQh`M}idg
hvk^eeP(})y=zFI!Gny;8`>zurwm+j=9M07t!Si><(wW|J;cz#p?m+L$w@SF{LIO#d=4ZQBaLSbJG9D9{)`_ZuEZGcs@(z^mQE{m`T7a;vX*?s=X!5MWnpnd
y#m2kdJx4f*iOGgpoBKY92Jk?8fFI<NL)c}sYx~07n$pF$$P82^TDK?<U*)<%q2a65_ahh2;eca2Tp%cs5j~6w`PPhwKABYXyLF*CSzsMjyr*h+Sro7r$nfm
q%(*Rda?+ZkuMs1iF~Ul>##Lt-dgkTrlb8iEO&~#<&PgHw;63)yLvCm?+696pbFa6S8un>Ws!q$n&v{aGC%-wm17yr^H|yYn~G*x1?l7nH6t*1i;w$pmi&s&
JgalL0#t{~y1J6Z$OaA)qRd+URkiRetJXyUZPq`;8nRx-&8q}+Z<>Xwn|(Vb%i7i`Phx#;n(6pC0QiApk7~HyvK#gmz8iq2{S_V=YY31K?Nv#deB6xat7z*a
j3oz@azZG<l*5d2q{Us~`%X14j>S;z6;Zm)&qqMC#j{%ArBNbvi97z9SyPGw+Cl&W$F`|>%A9RvAArxsXFs4}6ozhuPF&HRqfaIqDUXN=K>-)TeXJ|!c40f8
$Eak#K0TtNm*bJw)CMk9ORZqJa`gq_uR8gBZ#pYq@q#=cSBa`#XT&APEoaLD+8*_SL$1pLtF&a7SMf8FYIM+_2<}WCYkpe0P8w5Y*3CYUd+&_?{syQEIS^B}
LrKT@*vpt6TWxF16QoB0)CUx-Exw7BdZ8xh842Vm0w2?SGU#&KAV^V6D0=BH^91`~Ca=7$46=_QqKuEQ&t*c*>zy2&tSTF0m%XPmST$8~3Aj#oFHWcwqrH)$
&RfZa$<c)*@u7k(Mt9@9UEq+5=dhh5m<5~)ZT{AobAvt=rh|Y>1rCM5*``uvt<!BIC+>y75-pgx9N}YrBiQ6w(=>ERGkOPb`*kJC{c1e#5*Q{cpF+t+y{8Qk
l2GH!j{@svo|3OG2Bm?#x-B=Z8<vy{S#f$X#G)(AADxbXbAIT_On!_Z^bX(EH0L91kwwg16{?Z7NG;4=^m|ty;vT3r9_5Mk0$D6&kn!5H;QYVD5~0)dJHn%f
Y#S5%ntb2a<s*XO*ve3ATcjkBp{gO>P11~iJ#Nz<!wSHW1l_K-A5bhUy8rGu13(&V&QpThL3ML@_{@$!;HSyzX@ghN*1lHOhQW?r5oFr6<r9I6JVX#}`k71L
%s_qUz3Vb#HnMb5UR5y40$(~`cp?;%@hN@TlWD9UY{-o0NRkqGRbaP%VfjApzea{-YRN|hM)Iu3vw4L!56m2~CrkLr4h<+Ta^^hR2nEVl;D{2PxW44WL)hO$
tG$)x@paZ8vg&XbG&JY=Hf+!%1%kr94tjv2?0;zwacjZAncoIEk<5+dW($%@(~gl(t4@B7HVU#U3<j~v4v&P*T|Y7<rI>?ydM@-1DiALjp2)Z|h!Cl?dZ#qU
o>nv5{F@w#(PeH5SOcW<C)={H?w=K3`dJo3@hdrxb4)GX&&IkKIq*(zOySI*f*cCHeGRMeew(&>)|=n%iE<M)m+k=z<($b-8{su<!o#B_{}@vt1K5O3TZ$F^
v=jMD$*Oz&cHzg|s*^!L;?N6*1v|dXWZxnEbyzK0LaY#3d<y)$HM)-M>&!GQbFd8M&~kV*;up%=TZxluc%ROwwrAkK7Yp4dnHUa+s{g1gu?$XnuPQ6zBdseq
K9_#l-Y8~}75@7!Bm@LUa~?aAZs`?M4E*8Gd{P4VQ=Qsei7o)uoqfSnL9p8|`s2C$Dh${=3l!C`g<Y3+Q#k+E4i?5obf4M@M=1q8Fyx&>VAo1KYq8onQx{YA
Rh_DnM)}LM`rC{O;&OWkY|}S{0HF&co2afUjNOoncAJ~jt|~{CDZ$D0@1qKDW={eiDu-$N(tS?TQ0kkK&1T}Hdu_A)bf;GAR&oFR8l~$?SrZrrTbZ!%#6~?m
5x78>7Ypl3!+wq&iwhdFcmYO>NpDXnaj}HEj$9;B+H=$^s9VSXA!3F+vf_w|UQavJCf+K1E3`RbN6nokTX`&GeZBXZRwuY|6T|r@5eTa$%ncqL*7Mx?!!59n
lJ8W|c?b;IpMqBnWg)E?5Jv0Hp}vMcLXY^Yd9p`?-W63=6?MRL0GizvfT>s9`J1QLP9H?1CDn8ipFy&g-x}o`32$iTs&jj1RBOPc$8b<ajv*Q^JbX6&P^e}@
QGiKAVE`I!>U&%)BKK@?m%wMve-tI6e>|9!VL@SRlw4noBjC<b3LMFlITY7aeDZpr?Y>yYyf}|<S{B6EMUJYsY|)Nzy9pwiv;-e-pbP2eapkt=U!YJN$+zh-
oVN6iGVh-ZIqrFQV&gBLO6UM3gJVg3deVAvU=5EOuxUOzY~cZ_2>`5wT7w-e@*;@L<1-FUTz<nThLH{U<o?9thF>X|N?5UNU8E~NF84(-*|Iysyaj%c%k|+r
5TwvLY&0z0aj!+_i*IjG6so4*yN%_Bt&nM1MQz^~kT{7QP;#4JDu4y(i9O!xZ#QXJKn>ZcdpYKN2-39oh;@~YU7AV2)1{e7Ly2@hxH-MQQ%XmAQd`u0Zor{w
p%BQ+;@Asm^N9Z74&7JDey(^V$6BOFjL`w{99h)bLtDv2Km?ed7{@-S-XJ;}<;&h?Y)!!dyRC2#j}ZT!6v{sI#~piZBE;_DSt!)ou>&H^n=$WYmP!^op7Ji}
TII$O)+dtO@fzE#KXyVxN-Cz7+>sw}HA6y;Q9uZTf8PICzLRmQ_%I~+jVF2+XIpc`N$fy4k9tT6wd261Gcpf;#hxng!_-2)PqS<B&M!{?Q*U_xW57_2IU2Uj
j&RGy0cifnfx=mBU#TwAu&wwgxeHXq-O^;^6pL))a-2SJWS09cJu=mlLQj44^@wPtIP_#pJdVw7r2pH_>to4BX%tyhrvXusHar910X5fzaBc(^I?qJ;3%5!f
giBaJYv^H^H<Ip`tG<41bnl@Vxgx%~CFY+eg;g$pj;wUCx-VNNwa!$6;H7b*p#Lpakl+jQFvffE#e5#hg5}(wo&#BUBAi1-T{hHF4?SMFNR^Bg*jE!P_MVVx
^1uz-q(8-g>IZ3ev4yYfmV;3iE^S4ZiRX<Y+XRcze-5YWvXF8D47aJ|;fZDU{&5*it7&WXA~|{TeP!g=<B5ReUJ2E<SkDTBzhCFfSr9M|9vPbPW!A9G;4VlQ
^oyUPp=Ch$N4Jc|$YzdnA1s{<z57s4SWK@rkd^C{?O}cTbzRaf@PGuxfT&MH!C!Gyt1ws9fu97tLziF8Xc9HL$wG#qwcEQ?QYn0$<b-&oR+1b_X$9Gce=bN?
IbxmooDdd2zl{<P@P1yum{OhX!I?p%JRxa3e#zS@fR9v#=0wbhx~c_96*~(|?s>wh^_G@uJj~mu7!0vkDhqFlNbCIIhZI4uZIvII90WobR+LusJyeI-9&ID0
R>$QE(q1YK%-}n8Xu|ycoAk?dU*Fu=Qo69cd$I@FM&33#a@&I)6bb}N^iMr6JRH+zU~1nj86c&O&CJyLXU;B_A9EKx)$Tv=rC2$@uGbiveht`I7kMIAv;G1U
57?)!Zz9icy^E3i$5aZ7mnE%p+sdW^-XT#HTjZqs7KcqSu|05(x)ie(&c?_xs9YaYw0tr{tN>h?fsNyQIq@ik^rJL95|PO$A`E%y6kwGiO$S`8pQ107V5zrh
lia9Qb$&^C5mj;IR(!vwZDA|o({sq7U@M#<<Bkx$y`?F{h`fN+6bkoh7bj=!2P);TFU4A_bUq&f3%9*0_B?ZQgUCY3uM36H<sr0cg6!`!wx%#4pLW8irx}Mj
!pnxLM!Tj|m<62G(Yi=E^s#)|X@t1-^SzSG-K3f12`*5FmQfdr#Cs!rc7;`!S>9VtqYn94PK)XNJjjwM>z6fJcZ{)i=eq??HvKCD&r%bwd68J8tGQwosG?U&
ZStpQ$2{sXH(D-=*fzP1K|5Il)B6bsA^*A_;?hbvW}Nf{kOz>Ls}<&+G5$@4EMbQ@oNsWXKF15v&Y^L><WIZ98mPaZj7*>l3nR}?TUPTRMjF}`2`3mAnb7yy
{}a$PPU=i4nk!5MWM6Z1-}f#_X5QnG>5YOZm<SIQqJbmK2F>5^SXf228CiayRPM8wc@{l=&!7|A21-3_jS9r*>}S^Gve&p%v5@c?+*K5(^S(z>VPCoI)DGa<
afnQj^$&5iqW$)-u1_H2^G#zq^#mfh3_y5%9aUIkt`RM|;HT|9X9&V4DcqWk(Y2sayNPb0tSnOGvd{@FC?bEPp{{ZH+m@HDY@K<O2=i-!^X=M28m^;O<5oCZ
15?a-{T3y}ducPO#;b0%!sY6y1(D7bk6Uwnl=Z(k7i#gdj#O7h4%k{LzDoE;(rphi1X^0Dm6>5Pe^1lz!$6wDo?PZtLz<1gS(*>LHH#8K5{Kz861OGdtfP`n
Nk1Ca=hsJ3pB2dHmoz4n%x!$NxnSE53BEGS-Rpx6C`o&UcQI7UMd_!+92(+;_W7}y3f<$NyP)894i?udc+`l++vP}_Z(q>0HULT$3Xs@P_&Clj&XPjA7<vD#
mP`TzXdD*gj6SuWXABo%$etiW!VBv&%wADL0z{69noAL^zfEo0DTgxrq1~|9^ksRgEc|RO?m#!=z>&|9clqzI_YxH9+Sz`32o761+eoWgEdi{QM}`<}FgMD<
<=s<1F$)&-^VzrBm3E+rmlhz9&^ksnLrLz`a#YTcr89iwyoi1orcGDvRyafYPaBKNS6vUd0BJ^(UB0wNaEnK@p6MTUt)fEO_0+hh$!V~c%`$Z9F*3RkPw3en
v1+xn5~Qz%@+@F92YFB@QuE@8o9%)r)|XyW(SGVTvv<F$+*DmlI_(?dbCA;txY!`GvcW5uA?@iR-lH~Q#TIHiXyYM33e;VSxb<$>MwbW8Z+_fm0vx05*sAoV
0yK*}VBIf5VxawJu{3ZvNp+ZI0`CaXg)2!WU;399CZ&(SC-1Mdk<()zBF#bgis`&aAAd~sze!?$m9%R(aT0xyrZ)$EdI$sSRz7WvO*P}a$WQOF_P;Q?(drfZ
%nGcONhvwrQT%qaeh&J$_a4)m4)01w&XOJUkN6?9v)soH=T!q19m-OXc^|=i@r?XhH-{n&R*>D~7uY#NlK6|8`RylDUL?<^y5+6%xJ?A{y>m+2E8tT4(b9Fm
g$qRb5>cY-<p5NQl05r{k@Y2mQ7mY+=Z}Nar$gS)h|d+WQ4-GpKGwQR>g!Uv=v*>Rd9v=hx(C@u&aSuVf}l(akUp*`DH!f>DWM-M8bs1wFTfpq&j}PdN#(;_
uP?!{R12Ja?$xdYk9T$U+am|8Y{G#jJQi0Fzr|zjjr!Yx(~^r&i~JBNe=^g$$V$%-12so>M(U}$Ln$h=F_gGXL~8&m#|O51Tp0*bGAaYY+v-52!1`8o#yb83
n?7{AT5gP<KS}*<9Eu{N%EFJOVYM?LO^@}ff<)0{&kZzrKHYPb>*bE{!1;SW*6Xez&UZqPz^9_tm$Yv<{1*x)CqjR$J2cAk3yQGkCE9of>?lBT$9~pHRzi)}
)=aAk@r_16ZQT}!(_cPDAAdEL4FtyGcT+k(HG%jFq3GW2%6l-LR_inoEXnCtq^TocPm!MH=!l{uv?QA$t?YY04(uw-qME%rA(L~Wd(ti=E$Uc1((jR=@7duv
h$4g>-Tn!QthI(qjJZN7u<7$UW^D}9BQ5!*N9)>_%^*fw+dQ()UUFrg#+aJLo;lPD+~_JISd{XV6$czahj~Aeldgk>Q?Cjwtp(|dRCgER*H8s$?ql{qIu4d4
NIeV?B~a<vmHxLLw@?LS#g)bhp>=HrMQ_;b&y73LI!c$Nncr=IHJeY-RLLO&O?*+r_Qb@1-1|!V2{kvm`ux9mOz@@<6|D8U>;Hh~sz@@P6(RI!1O`_8|7!TP
F&@z{m~o^4o=ecK-K-aC9`LZy+g|AIV!QTRuMJJeF<BOT;^gWMsJm?v->)@crIP*~lV%^0Nn&O0oMRHgSu{rG0B<Xm$Q<nLO8{@bPqMI~e`yT6(8eI1$=PG6
{=inw5NC&QnYew<W2a+`wAXr<iNFg91{03~o%6GGLCx#V^biwL9-CKcB!ie+z;3zvb&WquisMeB?~h@+y*l0*#0LD(A(nX$vdJyCJ^3RLuH7Gfetu)QSs-GM
nS9@zttbGEP6OHJ(_t0o2|e0`8Qz~Q>49Nn=4fF7rj<$BoEPo{(JJ8q|0BnO-P`l}c%egmy{?}6fun~Q_NLly8zBtJnH2D1QS8`4{VjgY`fh`|8v&z%$P?OL
KqNQWVYoYBx&S~1=xouM<n}{AJfbj^UDHF54=A+$h<piP0dQe>5bwCWEk8%4=nn+vU;N_C#>c)4M73iRLi=5x?*m392+2tnS)UGa0|PJH$Kn!LD33kB{IZ$R
*k0V0MYq7%P0QyR(<{#dtvm@RlZmcpW6QpHr9pVh7W{egGT9wV<6GP!gV_S4DwFAVv6)ykFOzQf(F5I8VL<GN{SpJeedZEQN8YqJ#x|=4zXh|7;v`rg%R!BH
#zLVU<|^!#KcIJypcwl+X;9H?=fl2q*A8iNSHXwQQ)^?T+s)<OXa%}#Y-W58C+OOC%%`c3!q8WwK|!^!dE`jJE*B16QDF16cCvu&p;T-V*!m;DQSj!-OAi4g
XfdY5FwX?xMvH0=M+T`t1S6D;4T<-Wr(D}uv(`q!`f{zTz<W^@5-)TyuA(AMmxK02q$dx*yVJG|qVf{r5VE?^0d*1k|3**Gs{6%|7Z5sp2+9dtSVbA5t(Pj%
>bzZ)rthVrc-Ipce3@hPuBWDR{$b@Bv=|?l{Mf9eApWjP1B3Sx<1qh}z}rzed@Tto^jB+9FsfJqj-b$lG5Uj{^coaHgU15fLdsRTn3NxR=6}8M;VG2P;oi<)
ISEt1G;c~pVF8l0d)qIL8FXL#4hL5I^#k&$4KsUBAA2)2els$*yfaf{=k!Z~Fhf^gqSQjKU(+kwxi!x8co9&+H=y)UsK1QceL<RD5!$UXv&RfiQ5_=2{t%at
P}%)z&1B6fHGNoM^q;sEOE|>h$-S#`w^%k(h?`7L$Eu)V4__!s$P3rSk9`mpe*{?_#=W7AmT1>Yj9E<^F_}Z|dtDFI-(6E~Z5nZVuTh9Vz?po{QxAA6Nl>~g
v3<MmS=dA_vUm|KG&G^iV5a=nyH&!*vc~Y>q^p6c65(veD4YS+oE&3rnO)=+pKiE*g=CVZj!heIq@<gxA&S)O0mQLX<=cT8siFNm&NwUlxPavZ=mW8VNMaRT
IlUW_SMGHt;5d-;bZu6}LA0Lcg&-Ps4<-m%tqEqU0G-BQ&aZCq5J}X_C=d5!#L*}GCt#YlGRAG{&udqD#gjgHDG8T_^Y1gZ3E{)<;D;BqFII<zAIUc{rto_^
Y7CgF2?(VG^X4i~OO3-L&VFc;%CfKvBUB$2i;ii_k-m^9od+xB33LDSojDgFmyk;8o+WWUWnf3qmk_Z+WU13l0eEWuZvrQ^zm=H}t8`bs!T4^EbOt-M#0Ml_
%PGsy621bBt^9Z^x(46pU8WOcoY{K>PjFK`V%wO2vlk_aJs8W<J0H+@zT(??9!{_hM9F2!h{;hG!U!<?d~$4ynkT9_u*0qvt-L_8&ExfJx|Qj%H*ac{<Ir<K
at4H!O(iNN%uwt{)qE(6wL5N~i*AWXR81TO--5LsaJ_gBZ(*vuiv7w68hllaGO`_J*)yI<IQW4}Ziv)Iz9QsW(oBp_Y<!{azQ+FqVvgpMM%jTtiwfK^0>A02
X1`623DuTgDOQ(ot{2GI!+A0dMMggLgoR^&B73=5x=L|0+hzzV>{`wSdl4Z}sa|=F_$i#dSo7SOgvi)L_3?-5uZ6lN>ba3Rj{jxFe)JC(wP2CU+ogV)w1<jK
zLTuT!|l|+)y`elF8ZU9tY+C8P=k)@bOF0+AuS!>>Xnk}Z%9z!&x%kgdXSOa6;fQS!#nZ@MxE8Qi$Wk%*j<IoKQo>o+UI*#AXB8*au03_T^?Rg8rwyix^U7>
U%m2fK9)0$*-lKtb2!kscr|n+mtdGA3a2*c*K3jU<JSyw5{V%F`p~jV{vLH1`zLAdaW&u)Y!X}U?;c^+!MFNOjdP=~O&zbY>{LH8H#YEX?iQKwBZB%4Fuj5H
Sw&`2gFASuS`@H5|M<dr>S^1V@3DyH@Ss&mq}kU1^ZE)GC&N}GIv`8xuZWO<XTAcsc<l5Uz`T3VtLynoy1MX56$=VRuAoVuK=yuOl2j7Sd>cHosDnbO(x`3e
abRcmr=BC<R!RE=n~O#X3VU+3QjZzJzsV_=K)!X6Xd467@uJ(Iv6HW@g60R>tw`LNl}TYVks`-JGhueMt7HGl{$4#mDXFR`9rI$KZkAClF;KSi+`ezh8udwJ
9y$FZi&eNBd9dA|bKM<Y^I;{D(`l0WYyDpr!W(#@MY>u$ouI5N+sxzw#I7#nssAA5<j9?gct~0@m1SBrch|+zS+~ecAVGfgXr#Na*cPdfQU|S>&JxiSxS$ld
!5F*<jDjtLwZ;4Fyo~vau=9Fc9j@~yIGqwd{#-*JAl_n<RmNBNurG_s0X!%i>x94!mb4XROS#s!8-BX82pw3^>=^Oy-zgL+rQ8*Qn_J6|D_qY|BEEd;%8$1h
l#64?&NoR%e#%h30p--&_YTpNo~ik!D&RAS*LrVQXW^E95=j2+k{I>^*DkZErPCJ=zCPB1Hn2(89Ulkz!G8X+U?8{l+s8*%8g9;6w&X6O$(wAX=3ZVhQ&3D(
Ft>oSI08g^u3S-Rd=UO;OY^D7%<2VM*g+}dQ56(r1QZ+Snf9cS39*&km*Srgq<K_7?)0{P+#Ht0LqS&QDb*0cu5Hv-da9Q0Kg~7l$GxdmFKM{2kKN32gEAd3
c1Wjs-uH<>UeJ91N0xLc^h(od8b^vFGMhriVG%Z5#ymXVKGGlL{19pUcdsjN`<;t;WKU;a=9gdMF)D)jK4+E*PXqI`&}FGw__6Sk5eC5XWM9v8oc5Mw{h)a<
LC$FV2)(t1i764%qa0)ase*=7d|3aI@~pKvrW$i%3r(`$`2B1R7X_Cg#^+DpL|0Zrb*4{hIa_Rk^qSl+15}2v>SL33Ih;3NMN*TN2P0{{%A%~e7}b*~09Bwq
Oe#{7T3Pgy{pG4AHSp6ihax=vdk&K{bYmLw;F`>O3<IT-(6-Hxcd}_EZ(OMXW3I~XB~l&O0>Fe-wdRl%T<#Z^0=8|G>UYV50>0WI%edWCf`)k7ox?dCUsZ&Z
%MXfVp3ncIU;9u2Z2`+;C6%Kd6iih$4H*J%|DD@^jkRu<oOpZts?LcnC3g%}z0|WlJCb^KS602zGL>K%k@Q(tDK}Xz#xqS6w&73Iw3l&`8<y%S(m-0ua_8$~
mX>gGII3&u+*{GW<&#<mR{-*%G!!~&gBErH(w_Bi&<+vHN<4lCu8(1ENGlH9%#`E3!U_@9P8?%c($DK=4*#FUUN~!(g4OpvI3Dz8mO92(a7kkg%78<6^>Z>T
rMHFnDQAFrMC_P{orKGy=u;y#skPUpu}*Cd+Hq;R8{9!yW<KKC&~ky!f_j;~DqoD&UnN+q#oBXWbv6OAd|$su_E|<%Y3#KgG8)QYq>wQd-ToSBnsNh1_$@p*
0{{lDRO=e(MhvWXVF|b7WAH*3O$|P8HMDd*G)@viigl>3buV}Ab-%^X5~cbL4pVIAyv{&~a(H<(m7VoCX)l;FwI!5`)a$>@kYn|FF*wv)_L8uF?ZT@E$v6ub
B-lj4Re+0d2^Sh6n@K(VGObwQ%U(BE?n$6KS4`)%I}^+4kZvRZt_1zW;$z0<Otl-lWgvU%M2a~v$Gv69Fz1Jzzv4882U{LA01j09(+;Pi_mz)bN5zu`SVP*>
!2m{O(}HlAy&mwE58-d8zv7k9vl-<oU@f_&xHGDrUi4v_?aZFue^76}N5I;a-cCy6aYq0HH-PamhK8Z)?zL6sXVuRqCF1Y%6BGJ`_3FY?9%sC*h|yJb0(PC3
u&?{{8c`mGta25s`@T+T*L0x<9{38u2!c%^2`b5kBJMZn^O0jbB{IW<+y7Yr=G_ARunD)fxlTy)PQN2C0jdp)D#{z%Gjldmpc~evM~h>5nBk+lnfli;m!fC6
sGEk()5@+6rx!F?tPlTyBn*nhSVC+iFHrVgyUpeAJuI2?(Lv0D;lKXvk}7)#-6nbbVi%xk?r}>=KB&u({eUKS9qIoQ%#U723f3<AZIEyE8r!=o(Wbflia`PN
Rz>z@a%{bVtMaB3yv|<F5Hbm0s>trbDb$<WrgY6wn}w(^^tC@crVgr6#rh~4Znn;;kUA|of&iu6;5B<G$_H776zzIJiIL!u$;+Y%mcJdDIw)aQiPb`Huan)5
qrRHf_ZexC%{;Xfl>C(Gq+289xs4)38A-UlB7vFbVhK?<K=QF9su`%ZlA3xSL00={9qV*+e$gpxURPru#e9)$iIL9dx)Z;W;TZYW_gXYOhma-6m&>Tt40o4}
eU<mzm4$K1zim3%E9WZ|m(IzEP6{>7?B49!HT&bzUwu{4Qj<?4*_i}qo$rKe^tt>>YVy;l)9o}{&cYjlzRHXVjozxh6KO|LR6fHHR|;+Gz(kKNwK$Eq(eM|3
EGC<dirZVZ?7cpr+}_RO`04%eADdR<x)d$O`VHbOFDDe-f3GGa3CsGjSmcn}Y`<zEe`909y+-)EwZ7eDza-xkr`G#;0+g(aE`b#Giv1VjDA%ImV%5U%c!IIO
b+p+Z;RSz$%*m$r-UOSpe(rzNl8aDjEFa(HQ#J;RVWxl9>!qD@x49kRo#qZvGIaPHNLYZFS1~Xr%0@O930wWhZ-^yq`u|;d><UCqbQf=Tbl;`9JBXT(TD9%&
?Qc4Qu0BT(1~PcIYwc`EU&&d^pJaVbcBk^MAFFEoX-1FY8tM4uNYJb3R{ugCi;^2(_OyQq#q81cY~rdh%+X3#m_5@Fcn=0Dv+>s!-hM{Ar<cLaa3z0ELqHPq
o|JUz{;MeCB$zWdt7<QqHZ{tqC7>_b=E5=GnLmT}jaPI#rn=>_7NCUJIMQyhJ%$pS!V}}N#^EyLOJNO&=+;D!Wr(5lV{z}&u%HPLvta>mi~1?UeaAO@t)|&|
h<oh#ACm+TuI@z|s6{&3HOBT|RP;KlsXLd^Ve7w+HufeH+pK=wzxmg>N*!E^<&cn~zL0J(nFd;Sl!reBl#NYBD4?+?`AxFfOU(-=xgA+ob<lhY*I33e{;p0d
&a&eocNj1YZ>xqJVOlqu&OWF!eK&oue~`y((?K=T6H5kQp~K^Q(wj9gRd0a3sI#Tn3xmXJACNy4p}qivfn#%d4U4`mpfF{_)K5nCW88)6snr#<3?C5H{a>fw
fiBN!ct97jA}**vz*(l(vg4WPIc5rgp@rVFPQ&i|-mx~Se>8+XH!QTws=KuN^?MQ$5y6{Aug}WVL;sNaVU{(Z-sbu?*zb>IQa>G0lgte~9e>I&>^YQNJdw}Q
BI1jvTj)tO49kk6qGC_^y1y|^UrDMhe5UO1AVYWdhf*TubBSwpX9Z!8cP`f?U<acYCDf41MH@zP92)%1Fo30$4(Zjh`(!J<vit8e__GkbEkw7}wAtoQoU%_~
JMBv<(uLWL)6=XeY_M%`n72mqZOT=P?v<)ewZIC|H-)%upavZ`5#5U_ndasw80bvSk{f}P@p&)ew1@C8mCE1>?vHs3-aTeW{EM~s(vPqtI-Q^)X=((Mpz6A3
EPWxeaF#PkSZReg=H&6k3!eB{4e!%d@?lt@>Xi6-CS+=GFKtW3V?v2?GlEnk#52evywI`XD(2Sjd23~S#TRUmgUV9IEzq3`cvo-6GWnT%aByQok@A;q_flJ)
59+y~bPq!kwiiZFh?7B$$Q~u{iVXB+szQC0d{XAgwu8FJDRxUNT=-k9azRtnlONnoxQSZBEH3(G`CMlrW%Mf*u&Q6uEFGaK>%}JnR#$q%(?FgAoy4F*)}m!&
S;&*aBP0uwQv+Ep=6LcSs$u;24~dV|1=(XwqlTLH&lgrA?9W0cWIh$`57{NP)EBW9;M&BO2|h|i^~Wz>wp-uf`ZBgs11|KUv32(LLmCbLx_yDM11$TKQc>wq
IE<{^MKA}9w4-(-bXZX0dnGvF7zYX)vn0x8eGbkPHIE>KoC)<|$*~@(u?Np(<!Rp~t73L%!QtaC{OGwP?!dbUHH_8jQIbiHbUu;Oat^dZ?gUZH)_k68Liy5Z
fcTal^?7y>OT9765R70c?9|U5=e#CWHBnX>JXPuph+2q6hW^lb;HM|$!!`V|%9=K8KD)^@D@}fzi5sq^K`k~p0>pPFLhcz)EcqE(6KU2*Q)RbjG)}Z}MfS^4
Ey4v8ry)6s{VKz&*x_IlhEKu#AaJ>uk7i(QDtc6+e<Y%DcwGkE7d>3XLQVUE<cIEkLI)EzS<pDjksLc(65z2qMPQ%SpOXfqRqHHUcs751<{$8W3C60C7^(RZ
NkGfL8O9r$u^V>mA_*r(Kzg`shV?(adB{hqAFEoz2sN|-Bp@1A)6inJfoD+p*6_mQ7J;S|O#m@p#|b+6lf}}%&R6d`1c)I-oafZo*r+>cj};0uEeB;=o_a=3
O?L}3;C0FC$Q#?*l!N{UBqE81H{peHg3`4XQRZlD?Ndj=!y*Lf=bLvv!gip<=HRNg^cWP7GH>)I!U_+<3zLNpf-T~6ahjRF>=<veaHzq(N0He|Btsydy5iIn
J`SR2pS*t%!!Y_+$F=y*141S;w?ck2O#(3C)Ef?4avZ~HOtXb}{hkME<U!R)4<(oPRmi7affZ^f*uLfT^~*MMt>6uAEPOD4&1#_+z95+AtZo+D^``nmiqhkJ
hFlxC<#}X8=IsEQ45nfBpZPB)I*vU*{=)_Rn#1oA%hd$go^}h5_Tnu;b)I4oh-5Va?Kb!`C1h^UNwgYw33Q}Tw3dCyjfw89m<|32mD!+45ilc<4s>p_UX~U>
2CmhxD{B^mtwX&2gtFipqusI$xm-v&62=?x<&S64a0y%@qQx{kGLu67r7BnFpb4ZI4~~7hjA@pFcCu{~Zh8InmS<Vj2mN)*Fp!o~G!)&3ff_vt`(M%Fz*Zwc
|G@7cU5SiaX%*x)JzZZa-(z+g-d?E=41R8}ETKW%hf7K82j>#}B6Xfh55<>48c*06^(XV6j#KT)Pg_rEqA*@R-5kZ>Y}V1<WbyZUSuXPEj)Hgs*?aL_w{PWf
Ed6mV$u92JL_Ae+J(1>)P#2w@Dn>5Ff(IdAJU$Jhl1-<AEo%H1VhtbB$rlw`x(y?Z+nc;bW!Sjio&kTEpe7RU6eDr8s~y0Pw>?v!dEKMWX<J&)lFs7!rXPE4
qTnqnA@?A%z&kq3$#B9(0msK1TJ&Jk@JZ<CDHVVYTRhI;{#c75YFi7bQahoxT-&F`zi6-{!8BRZkS~LYUD4tK#t>)Y9?D2NyQdg@{f5_l-G9pn$3F(f43u-J
j*l)9ET(~Z>0wX{ZT@OZC?9d7X!WFx%gxEutt15=+@X6lu8qh<(e&{p8aAgSWy8Lqm!utY!%yQ@4Gbb;=sdxKc8c5+Q*DhW4saiDp(8}A2^9=BzO{KCXk*HY
DTlg6cU_(zwl^~JG(a2Q?;<5^%|R`#Z&;sD1UECFn7iZ=kINUcI2zQ-QpL9*u4Ecdsbw%5h6@ELX?6VVRtJhjoq?peb&GpTxS*OofNK%>BUs_}pIk)uI&kkC
b7-(F6anBMofhZV&c|hbzK)5Y2et7Cb~gLX%EJ2keVcp6lZ}gx3i;hty*Ox?T=@n}|IO~RUXq)kU0r?p3^d((A_5ZXpyVkKu}Hq-8(Snrc^yObi()a|u)>&i
Zg2Q+v~KkDD~9Ikp(#h(SjNfnz^Xp}fW};vo{4tNhe@1BKzPsyFClo>_BtqiG#W*==hal+tV&@$tuUKS`>rID3I*JMiC)nz>jyO`^$&1?<}-|B$JjwD!Bu&X
IPU1`MCT29imcb=e*UMF`P3}055wJev8`9jS4t`how;@OB`Nn;bt~3EP&_lh!ZaZ8(sfB(`8<h=Cl*3iCuIjYV|Du^xb^!`#NZrDdmfquCgPMi?6l6zmvUOf
QNK%WY7l%24{GQn0QWvUFdxNsy_q0%i{-9UzOF%j6ha`OsdEar>*1D%ajMts(TVhM{`=3c=U;=Sq~Wylh<;c*#eDR0SH^|V5`*!Ev}h`$y{x>psy-Dh>mf)y
dhS2(ze|<M2C{AD25w0j=<BMBDlfY-t}sgZp~mppz5@N5IpNI_!i~E`(YT#bYuNW-LhsQF`6HgHx4rKO{qaEhbsUb_uQ>bDWb|OHu1_vD2NrQo+7%TSZfrZQ
Gpd~8d|dcN-Jn#cN-`{T)p3`cQH2H<st6O~YukTh)bH2j!A}YUVmrhwHI$t?xjUSBn>pFdiyf7k^CSOyZWzGUR?+=dW2-Y+x`UWj4~UL3NR_7!PAyI%h-z32
z&HdOu+jm{SsSqSpo9BQM(Nr|EZ{CnZEbgy@RJ(lM^AdRWt0>9=~P6p?~mt2YfHse4|TEzPvU0$6G9Od<ymHN7+tAMZ&EE~qQ_pimL=wIzc$6rI2_?m+<zH>
ScX~8m~=Q?wY5;aV}8jgm>z6FCGK1>uqEId4A8-R+bP=@l#6a1k9wv>Ob;aq^6av*Sk60H>Fr<?=4t$}Str7D&(k`mBi5%vGdGl_u<voVGk0P4B_!zcJkbIv
*n3+r0RV%KpB)4xR%YzMPcG}lnq>IMGB@kwz&EQ*<lWLg-di;??FMB2w+&3YA^Cx`2+>H&n)|HhV=GZu;@xl7?pZCO=HBGEt3ee_FAlTwLwb7x4P`Kf9nVcx
u*ThC#*_d#F8z25oF!c#_~r#9oaCy@jectf0{&;|t|@06cyRB^e<v{U^U(kOf6h3=r0dtoqnJremQ%TLt<;hm<i{nJ7~io}0?Rr&o_U4mEnTZCBR&#YRr13y
hWB4N0q~Jpm6Y%otO6nPD<wDZyfx^<Y0%oB_9+rM<gRX-ZNos>2Fu_L>K<)9%6WGz8<H(_zVG#Sj3W7Rw%mUL9V1)HC-@C066Sc4RU}%3csW-OsL~;rg<!kD
v;_T8Ib<PpgJwE|g*>l{I=HggQEyvnElkKz?3B22%{J~1Z_#xcIOsQirW{5p$SRRp5B%VibI<VZw~WF8pf6!K+u&S-J2b!EHnjWu0y3`<<z%%BF7;ouY}zHY
QtVV(naeiGenW)d8xu5@Ama@b#`wrr(^saayLi<D2UPUyu}o%+VxOn)FSe}mrDz}dDcLF*0J>lw+EV4>vN*F5chv(_AA)zD;u5w8407N*qXY8v-ql?|prk=4
^J<3V^DcreFrm!G37+TZ6|+Bu`lQnD&LeJB<_jUGh3WT6N9j0b|E9Mm7Oyt<pZj1!8};u6h|Bly%FC2X+KyF&$8~4EsKra6J!dBbdMd&v6#AGuWuY!Kw{N0K
<<KfsG&p?2HuwRq#;jwe**C&*I4qP{h`BR0_QtvX8!BHU>ue{1eKRccI%4)R2zOq%c%rJ1Naq6w7{^FX92w0@LhZ>(ML0NlslJA|h)ITXN!;n1m)wCbh5>kC
G!dG=2WKe|w#x<d6X|Uk%cI2YKNy>G;G@SUpkB=$gWdRtCmEv2_<gw0&~oK4R2}ykfVc5avhgU^uYjwM5RnU%pVMxfG`D9j-^k}1^=et)VxL=Qg=v}w;p5pT
bf{RA^GrsT>i^r#9=hW#C&TH1Y_Jj+XNz2`WLDG*dp(>^9G)V3Uq?fJB6UNOk3k<HvMYyW5<9~j2Gui1!Z9zr&wzjpU7Itvsmk5oV>Fu4?&!_^@>A^@6-2U|
vG`!>EwR{hp;aQ};kGhpchwsB`Ml+-!V}5!hULp7^Fc5oTk^$Qn2RIRp#0<>UGy8LeDgSYC}pk}{E$g-8nCs6Z$ve-w5+QEFq4IO58UC>$(7aGhLdf_9;(1?
tXs4EewrG^WY)i1(_y7FfYMsp?we;%Fb$9#fPP858$>~i60BE-`?IM6ng9jWv~?!h0le0EnY>hB$F{^PaBYSe{jHF_x=A$CvjA9_AZyd$|FB~^04PkiG55=Q
&uWy7e{j?f>Ge_s=XP|UfP7zN@N*QuWnKnuD&`#)_)Ab=E?SC-^UdViU=*2m1IAA18<9ywbe{(eaPFghe)+eTNtSrp-h^-f#BTNdG((hgUqMi4TJOmhjc4!@
5sO--eac1~);!T&0_AC}qJXoSnY24o`YAC+I{-fP!Gdko7wT(|uEWd93YO55xh_PA+^4mPI<l)G3n&UZdM%=FX$3m!CsR1Y9Of)*Jh@&-_-x5q_<|Bekt4>m
hymzn!|e~qSyk2UV1loXiN^HHeEj6ttC{Od(0AT_V>9sAn<O8N@x7wW8N@iX+~VxWbDfjEqd}2v4TqO>Jv;<9g}y!UzW5#y-JzwysCH4!2Kjbhxu%)Z4jYQf
jxX9rP%xa9F$)m*A)JaZcY(<6y}y~eqR_Vj+hBu7#q)^C(%aTJ;}Qzgwdx&4YGgd^vU=Gzf~BdixFOrSt6GSXXX02VIepG7;>6}-j%yXgSe5PCAw(M7DFhX8
Q=$k+P14A$Q+?3KhK%OTdQ)Hp?F+k}V25rkWM0%ECPT3A<_2R|CH&gmu1zhTU?3BZe%&F63?LH?afADLWCN?qSU#d0KlRDUme4BZ#6-le(??Jki?WOJlux|(
RevrF=(;OuxsH_hL!;XAFyc8l7-YfB<(~GJ4(-)|_16p~w{dLHk=>f*b4)$mQ+9`Jg#^@SW2JKMRt+lehTyT^w}t&kTLf;%uEjTAryxZ_72fs{!OAS;OEkbC
qqRWu3~V%rr_K9T!lB@;B61C#QwCEajbDwVzHy8rb*s&kv6?9tC;&tG)vEiXE19*`1hrj^*KdK@j>diik~f+}(DcrJ>eUwGp|8uMq*mM`s!H+&;cRd%C_~#z
!thYacKxn(bkpK4i!9cVV>dYYeBV3xc^rr1)o7OQ<h!9{>vkcD>s}&V0v8+6Jap(-muZjVA7NCCJOg0xH+wsk<S76Y#$9b%>QCi);@81Z9KgvUT!WHVozuyl
SF|}7Q#qfq7z|=RD<w(L-7COUcQG$@WFj4g4JQaj2^_0!`WflP*WxGzdi8^5a=V)8867sB=@S{Nti&jn@sl2+?vajWb&GZTK9A6fHgZ~`9dbzMD5O1OIriw6
Ey0w8v!~z=mEQCkVu!IZMMHQ=OJ?J@>qovt_uO{Y=&V0=SrpbpK2%h*4w1GHYp92e#XyTCxdjI6PVnCX?*fhMay{*DM%h)ga4DM=G+}_XN?QksK*HA6-7jJs
bXh;(A@5uuB&V+gNTYtO@aV9T9M0rLQ$as;b!F{HpHcj#<4NMRS28V^B4CKu0KD3jQ<YM`c}|~_pJxDsZRS1VaOrjpT%GMkg>MR*;hDA53S+<dC-v3XMY~N$
XGqCzYsZx|sVB7|B~Hc4jpd~X@^1e|rv}-w(DjQ2qqlbVT4c250=o)HzSG=($^ZD|j-`e1U#N~bvub|`tdHIq-n)gvF)AwiTHE}&#>|GzPe5p1)GlBBHvxg3
U^Oe4sM1uuqC5bHaM5KHV^y?u_oB1c^=f<JO776}s%N<6#stQMcN$Iyw2C$^7P_TFZ4bRt@hLb2weKhq7KbK(1`ohQ0VuCvpWwq)&x@>yo}7B}^rueWl^2Jb
Cvw+_)lK)t+}|K#$k~_ab4XFR88U4B2N@Ro-a!Z>6#qo!R><H^fm0?7L;Q;{bWH7-mx9~CDm|1l1F}t}mcmWUY*P;6pEQb%j&<UU8rCAgEc_VeDOt7xCmU!3
j{VXN)<&5DLNiC_d=q}O%nRneE6Se1LFhpqm*39_{logt9F%c8!9N)}o_@2-bH;O2b&QGPJ$t|ZI0SYpQ<5(%NyY>}u>GnEcE7MwZSRyf^{=+FH^>#G{95Y?
x7Be*Ge%F4;%Sl_*E?*nBWu<@ZatSRYoCay)PMkZcIoS2Uez17<*LTSd$ifESkYRFO9K1Q?$nQ^NKB!s$(FEwQ2Mp{rQ*ptG}BRu4P?kf+{Xp$+-*@_&$$uN
XuN_EJ@AsA&!v9(C&a$rR5zVY8}`f^KyFL;je;cQBLBMIV_1od1pNUwSgP9wQ%5QXXAPstGpxVYf@#KD;g?%oAtfesQNVV5<E<(?s7Q*|CUA=f7Yn={Z+tr*
H^uBIvU%E8wKOx1Qx)4Bv2~vS5Wn#ka7~j9VFN&PYQ#T1P#=G{U8FZXiq3l6x4<>o^vCO`5W6W@R_Ar;bESlh7cl@>s2n~D?LAtL>26<HRHR44`#x=i!n1~c
zvYSi9aOS{HuVAb8#;^V_ZKY`*PLj67(noJK9Nux6#AR~RT5qOx{06<?nH*+J7-W1<nIM*FOdt2`@zwgl8w)%2B1BV-~DB*f8i*Bqw1XIa>1O?@M&LM3ZA?i
<8L{5Zr)~gR)mKzhDW#f74|V3_9StOIod1-h(mV_!sgY0ZCPd!#kPmI6G4-O%p@jJ;0$LDoeQHTkvdoCI}EXr{3Mj!z~mEcFsxz&f%mkhQ<ikk6{b{zCpSnU
KRv<*GHf~jIFPabOkjcBgI5^0!IL55U3jk7ktu7oT@X3>;<Y1T6F}18pSbK+G>-81B&}}c>r`GSgVfX~T2LL%(3Mv`o@l8(%b(JV8ZTP`SxM9$(*t7?oX9$n
i7B1S0!TPzaR4H7()cthvLUqYzTGOW)%V;A+a&fXUCVOnF@y8o4N0*XqqW%WLf)n%UsuLW*z{|y?5RA)n@Ix_4R}On<7^w6#@D3($WD;(ycMR^2Xd2w{F)O3
!Rq%Ii?GQJ6h&IyZzh;NW2mPSAfd_;Gy1co{qK#Yq)oV%bsdxzZu&frfaPu_#JtiGme$pmZbPe9jl7->{Ciz;r;@&MroUnR92^P51=5W_^gkGdsd2)!oGiS?
WBf7vR~L*a_ucDv_|syA84rS{FLBJQi@@aU-jE>Z>#}b8;1xa|Iwl24EH(O+kHHbQfO~Yy1~wKjt6i#ck+xddt{)|xPu3y8zjh;jXHi*f>%jp*@D6g!^`?SN
AoPEqiFcbb+#v4V*U||H`b36$?~q2OGlXA___QF=RM?V<+LAV)f#E+dvgxtRlWzPNpF+Kvunjyv?K}<Ji8(5n)ZH1d>1SGaUO17DLqr$o0CO0m9Hf3yTBe&+
PQIfo^$hSHc_$|<zNgLH0-XbgO!y?uR}Dfo`{dJjZS%S9B)WH(0%Xmq`7y6Dc8v<ZZl*tKiaTbR;x@RnBET!gPVYq1ZL{Qf`o0Y2A#Etf0J^6|t4-*?J3>S)
8sF_JlX-2VOuRy|Rj8y@N~6Qbvexo1xH~v4d-0gKY5Nim@RQKrCRAUYu$!7x*ulS97WuliZ)aGa{@wwM7*G3Pp2%K=$Ugzp+OWvkp^@RM!iMcdmV49pFBGqV
&B#8Wwr}>+R@ywpjiynF>_UL*grcp`F6C3kI(hEX`Ox9F$9d+h@7~A2^x5w?dHT>(W9~IuJFF~K(aGFQ%(KS%r2UOjE){C)MrEi8lZNECfr)ktYvu``_wK%k
zw%x01t56{eEf|jj^p3~OAdj4%0d8l-|w(rNMq5cyQ6AM9zTV-zn5-W^9viTx#W=ved7;DrLtvjE(tgNIvPYsq!UO!mDHvU-&xG_&(i({%?bac2tKd7DKoGG
THLr9f<gP+{NS@ReEJ5Nv(5AglG>*$TD{s8%;*OYObSBFA=+$jNrIudarwySjg@9DK-}gS+EoH}ice;)dwV-LMc4Ehovfvh;ufRh%mM>iGM*|q;0`fL%~r!?
L|pT_?S{N}%Pa3}t)R5So=R8>#>la{6D**!F}r?|zFx9s1Kz7VioS3h|A#_^8#}gfbLurp!kEdl68SB(3pqFfHXPTN18l+DG!?7j8=6`G7(!peJ>=8Ca7M(T
0G^=b`4CJ-3?C&bEv~RFvG*YL$69nyNQi;SZ;2UXRdxVKf|FAWpcP)3@|xH*GV-5Ci1Z_ReWU{yST{;O_I!U;&7M~~uJ2bS;0(EqHy;)5l|iQi)-W_h`d&yy
T5IAR=;)}_>0Jyu3hGQZCYpNKQL*-;mu=J=Fx1ppKFynLXhrA+UX_ME?>sLfUcWs5St0L1X?;<801}Y%q9ij1f|e|dSN|hUuo9glHPF=Hfcj+<;1ea_Zu31K
s4(yKM59~BmLp3Oz^KVjcVO^0mwIy7n5zV;S=l@31I6P?isirNj-ezgz^0J7Wb^1ZlZiPENIl)U(MA7ZKmGw1Cc$|3u%=irxz@UE3gXgEdZuTXutTNMF$^3t
=pg{)SmTUtpNQ|_?0+1${4y&QTm4+T7;8ZNXIDw#?OHim0|QuL1n$g2f^;$vG3P$w0)^m-C+&KMGI`G|09$vSzKKb*4vCs;*}M_^`QX}}1z2V#&d`4J$zM|^
*_DZW1cF57RH|2MFfnaa&|-^K*dM}h=dqP!F*`h*tj!{~qRu0Acfe$AC9ZWS?{4Z5w$@NrKFC==syv=LpKS7rd1?xjr7~S8X<91Ynxvcp=1mIRM#v#%mA`kZ
g_0!0SaYF@4AcAK4h4canGI%Lqo$c^?jD%kC~t0*$&GBI5p_`)rM1;GciZk=%d#OLA1piG4mAQaD*UsnH@0sQs43d}MxmAAHtI{?!kG$~o9?BGt%{ee0R4f<
31Qe361nhu>yJnGQ9YjfOj^=!9;2Dyx22d*TCFLYiFJYbwkqw%Vrpyu#1(l8hkKgqE_B4$-j@NVDofoO=FYn@_hgN5JZw}7(cfJD!GfscHY4e~p?Os#c2DvO
^%TOc8E%cAs<a7uka{NO;otDwNhCv#TM;;O-%Y@x9^F6t#ofK#Hq&xrphvH%q5-YAA<%g&@4E$PN2_!f>XAzTswajV(eLG}_(m-|9OOvjH7SDy;0dHTD=3Gc
m?zTS&MB|I#qC46PXEVsjEM!APH6|xWy;V|j?>3#QqqjyguCS9uHjCDbe{VM)UOJLz0H-hZJsA@v<@O`)EiN<`*BJN&59~K0P+$?gsgG3@Q?wgnGY@s#w&pA
5hDIJgc(h|nY$1K77ap;!wBqCvnov91c`+9^0C~@dVgs#!1h)kb35B%$UPIe(3YsWJbx`XdFBJ1ol2nr4KpNRY^o~ZRDNTUSLxnU87zPkSy@CuC@<b8Ut<)n
X6!Px!jT@V@2J*+^O5UtO!C>Y$Ls7k1Ad>0km7Yb=`CPZ*m7aNM5}^1d!;*5*vGH&q|@J5!xo0`IG81{9TZyV(wb)~b@WY1t`RlDX$^2gU1eHS?iE>tYG(85
RriC@5l&Ie4^@V}<{J|<y-;sLDkEA|s5|?%bAlF+T4$obU?g{;=j+qiS=GXL&yq{ex+=1d`kmtGqHka|s>W=;HyGf?s-a$A3j@l58jY3gFmK&qLNX)NvdoyA
#Oz<j)mVC+02J7HdfWj^m>aIICqSHhL~sHxrY@n%%A3}ra79dQqHw{TEUCOBFDr#ga>JbuWiB|H|HfQNLMj^SdNX0J%8eY6?gEZEwbfX^Dm4Ag1CR`hGwa?B
Z#35U(55U>+?@ud3-j<j7<ziyB7H#23*4ym6D-5`Dyw_#{0eUGGa7TEpdLr8!_BWHy`k90x%Vd$vfh4JwITh8aopvR^<k&-`7*OvSSDzubZmpisSFk03(~`9
R8UgvgtL~?netCh-`iob{OFlrjlyFVjF0+c2P~j9VK1p7=xcK!$5c=EUWs+iQZStO@IRv%$YS3$qPWtbG#_3kF;cP+Ic2%<VNMG$QeSzHR%oOWJ8Ie2qO+!#
bElua9Vv)aiV;~0A<riV0bS({R7d?dn=?0_lm4oF;WdnUHj7HYn~yvRz!D%Gf7ef0xLI@bRa^S#Nzx;a@2v&FFsr*oHa{}tytMqwRJZ&d=ilC?SeaXD2OEQ#
t5&ape-F(72HSI^a~rur)kVyM+AgD-I**^J$#VyN&))CP`VN0QZ4hYxfDG4y`0^XPwJ^Cqes}5rzS)rqI3T%eNQ=?|99YdjxE3KgZ+9=T0V1T#G#X)Ojc6IB
@G_+c>WH`bV%pIuPuLv*ygl-ux_VsEbN|_XTMn+~s)=Ths0JKhM8<L0WpAc>u-}#p0tA><NY5R!6_UB=uiN73F-TxjQ1+Zek`Z+@+#$3)*nx9tKxrhQlAaIa
L4kYVobF6&HWck~-*ZaQ{*M2m#RQRGDaiL`2)6$%hSsu4GOio-u6-STfz!76<f5Q;YpLsOCH_Vggx@c;<=*Ldp0!?K4T@iStOXc_GaznGrA}>W0wa8Nr@x5m
WzZ&whKW?xa-`tBd-5X2f21V2+F<%8`t$7b9Nt{<pR1w}5!?pfng6|Hj}sm)neCSW4s1dHZN28-06;qQ=vloH7ckGYnq7m$1ORtVeO>fZLWnSg-+P#nxU>5H
V9OoSnHOn9cL-Zb+o&_PP`dW%lEy{E#W;x>p9+93r(=!5slqg0B1fzv8&kR!_1gk(m^Zb8zLR#I6@uuq+3H@5@HslS*S$Y2CK|asp%55!jmw4vu<EBKly;Q?
pukuK=l}7F1LRX?(e_)ku(!V(x}QnQ@S&&q<eX`!d?FWOLhU_6A)%aIf{)O_wes{FJ+70#Qw5X^O)eCauN=qE=U&?EE7J&r;!)KBI-;b(hhM3OuK^Zbo|LVu
hn}T!PVf@4HC!FqkG<Z*5j?FWi<?J~<KAcO-O4WR-@N;qzMYX_0$=R9Sq4!v!;u&X?UoJ7cpMqB5U5>$6z%)&N45MUMOm!FFOKAgyVg4MJKfKHjWX=VGRJrW
gkzXXglcC>>Y1}mF#Tde_H(z0wDMn1007jX49J-@bnQ_52%;iJ`K0o&o$>K!N@A9jyKDBPsm>#d6cGTVdyP?4=_%lmHdEAm1VjF|e?8<w;#ixZ<<eeo!ZLBL
>v=gQP-~lc82|*hei3YLK3-}}LFu`EUj>bcQ<nFUsn)m#yvll{83@~AR-==8>x1`W^c;;<iiPxr!(1_jc*n&hXFalq_t^QV*v^I=0IdjnZv-<(ouWXo@N{;<
l7Cxw{W+Bi`yt7*2?F_nbfZoG-!FDkoNLs$FCaenQH%}+x`{vI%dJ`gWAJZ6{7bn`tKN#P3L|-Bc(;y)WR)1YAcgxTkeBFsTtz(LQ`_4%ItH%)>12j!L6;5k
rO1^=GMpR4IodRgbr=>pH~zlcgaQ(cSfi#jf9%ZT_|MgTW@Z)rt->Gw&cgO~7BgfhZp$$&81t?FISiGd!&&$%+;C&?LjhbGYzU~@$^OiJ7{lo-9;(jf=x*x3
z7-VPd#~FsQj%<>`BxJG=4v+ulQ-b5LQ|F+TZ>~zC_?3G=~A0rF;Ek)x4La{d>l7AzLPAkGC#W3@P%^-avpT0P}5{U2j&wid^vF{noOG;j(usa3^gDT9asSq
P{J<s-^~^*ZAYWNmN9Vb(q0`?;aan6N_o{jHcP=vF-Zd#aNZMl?~*v54mmBz{Vi8Ce3f?Es=49U(|@a3p3GCs)NLC`MXvV*FzSb(`M(>hJP6;<%R$p7pGmUX
eQCei8tEy1fQD<AO?8Nu!6`bpx!i`UGr^w0(CB`wK%6#+-R6F#eHmm^FJJE2DQ6Ifb}7$`m6lMcVwCz%7KXITUF(qQCo5N0Dtkgwk^tbnvwYn6*YMw}2F_(B
xX0>Zc~xn{kFE7sk%DZOUr85Tubdv2oOHe?x0GU&;Ayx&Mc2CVx*BZT!)ujLGYc@|d3JPjyjKbA!kb^6_sxR!S|A}sB30Ha%mu^tvPALC&;qqQI|Yv}u=d33
gT{O0QUbY>9E;Q;IShE&nkqq61ex`N0eS2n_dXWWHx&Ua1jUQUEg~eUfhPA5GD5jmT>XB8Q5@Zg8^d9?Sh$q$Km!Lz$LsJhV}EMMW3YnMTUcRr#q3&VprN|n
BBzB0-U96=<%h79u&#1hMm(=^)i*s2d9`4cCRCe?W7}(cRHKv+>m?1kfGp~ZFh&z0qJLyb#Pc$H2caE4enK)pzbk1Gjuvz))aZg)F#s!6I~-feNFgxxJBUX4
UUExX%no@Yx4{V)Dx^6eC>ktwIpfV&M9zm>gb&)k%@`4IIwsu}2OTK6)j3W{P$!#=wV710hEQ{O#9j5AAepqpN?dHs5nu+;CIECJDT2yip|sfLB<ct1{2J`o
S5<vg_kI^!CXoz4Z-y6K${R3!khFEi=dhe!toxl6m7XaYolU`q*QfeD{NZ+rO=67Mc$@N>XOIz{ei73{Mz?K$^CpXIo%c-sipbyuNo=d6mkKQ;>1VMTh#bg1
%599xB<|;DpP2&a5c0ATJzyz!F3t{B;J}0Aj!saxo-pgvTkF=^bXK+l9kMkJ6(Is!Gg$VRZl{4kOnWSBdV#QZ{(=TNTh8OzII~LwHhfZvCu=sVagS%D&BpS$
q4&xfC2PVrCG0gMqT3b;fKs8)D%SlHl?-BUO<?c>i^JyzfG66)l|3qo+LhLn#F!-RU8(P|knr5D*{|y)N<B6m#;6=NluoiTa(7tF3ENyAY@2&8#=?6{LUUTv
weQV!LcFvw{0xXBOJzUM#<J3uBOSCq=-Y>J?l7Q7h$|jV4@lsmH(+G%9-V?ZZ3Wq#IRJyK0hBj18z9b?PnIOUYE%7_VUn<ED0bnh_VCfQtrL~tIh|Yj+@BKu
XMA#MSr9K!4}4l>lYv8C;U(K*s}}hkH$KV`m)j<fnlu(;I<}#p_L9_SHX%xm3pid0h#D?0$_N9KZbk~$#R+RVY9(c7UqU7ZFVoO!gn_z@0J%)2d8AgMALh%j
9Atf%V-KSXUYo}e^<t{JYrbu}&?XpDiCH#<#|2P>N9qcS67ckzb6ye)9?}qDLvtIJw&7>KwuW!VqDJVLA%vtP0)SAA!6p9lTtx)w-S})4WxElkw*&_WiZ}U(
{?a_4Vmt|=ReMqdmN^3Zct9`@vu3t#m~%7XC7K;^o&g7CQ0zxEju5X9ZSKJMjlu5He(r&PYQ{vQ*fAizK4z82bL`P?kyl~TthM)H9N^Z&cr-<|HJ%&^-GZFr
%HG{pu@__(Q-8-&k5>@oEb5gix%ui!tey7Y@8((M^`XdEDW47QLA5e#)gZ~Fe_~)8{4!c-MhsQ)w2rJ}H`gY!;*>WqO7}`fJ+cD8bPo*B$8ZpqtKggEl*GtC
Bbe%N3VjWVK!B7rsy4i3>Ve&vsRFN#m8jaY3T(71$%BaYxew&>jMvdR{?ykpedpvCezm53IK`5Sj0mA#C;16`Dx5;Kn1C2H^`v|+q4%W!*SB&_NY5;CeU%mB
UH9B&?r#-Gs(N3nH9XI(3A<ou8^(d&%VNC#LUGsv(8g)+UMt&s;%aSqYA*wCHXQZuxV}qbZ*ew)|3q5r;ik8vQ9?9SooX)V(p)CHeYR>-q<EiY3BKLa7)Vh<
Z`k_k^&g!ci~d!=ymIefBcGL)PeTKygV{mA`Vzc9XyZ3s?bl&{>qfHmtN2Ivtgj`M-SAt=?n*m5aFm_4BBt}y!{tEViCR4jXk9K>(ow|Q4h7{xZ%B|$W$CTq
=>cF}(`gqAB927Ai;Ta0G}5P+0jX};_bRr5O$~0=LSFcbDVTy6YOX11SJ_IPcu}ouxb-I9?8@yk&Od_>){30Q>$_MN$CeC|O*0G<A#3bi-xz$N&e|EnEMMJ<
hn2ckaqdwBnDq2lE*NTFdDF&408ZYp8;CoEn2p8T$JV2M{+hbSQ;yEe?%0Z2WNY(&O-gNf2!}NF(>Bf=cQ#dSW_fA2gmNjtq)(PFMrcaNUcpv$&)+7!%AV6k
+c9amd<6J)cT(mV76ngk;)F%zbpXR{szVSLzgIfXSSkDorv_VN{GPVS$61)p5bm9r=)=?Sq_x}cap9lEEV8-zHsVQyLNy54P4FOG#^#q`Mljgx=BWY;8rV#9
Gb!lXqWWwtItlhXVL=N>)=OA-H4JsXFmPV8EsRV`*3p#&!LTYe70V*q0*B{(P4@0#t|K9$TSX^0^*+2%C&MYfPw3?N%`XD$^h}<Q76pvXsj20Wn*_v1;}7W!
TSeM@$LWxExs<8&rA1N}m_9!QKk_edr{G+zkN*%B1XdT@#~gD2ypb)|kQ{lYYiWBM>7<MnHM>-wL;bPKF;b1<9yXrXg`!Zgws~cOD+Z5?jjWq(ATl44QsY$B
9nQlE-#i0OevJATgZX?04d;lAlKaqh2oN5%^}Jui?lWjP@mhb3<7VfHtcX-*5_>@b%L5A>YNNS3*3`iO=<<k`xgO_R>rQ)qgqYH)E4TV^*p5z%pH{QeKJdA_
yY;Rm>2D6^f(-wSDi9pE!v-zj__2XQUeKQgkCBO&3O2lm1VHn4e@s1|BD6~52(-~Xg^6Yy;UGNDEiur~Ye(!JRtRLMcr!wM4e<QwT-$|{xt03vq*1b6{XCxD
@{1QqiBP$8)7N}=R5=zAy|U%Y>-@Ac`A7?q3heM(WcT4fUhRY9!kA{3XR4W6j5!#4+-|A9qR`FmcQrZ1extZ1(M(bB)X54e-fLKr0@Q~-i%pRS`82~X8Z@I-
GA}gLexcq+6QoeX1;A0zK{l7mTSQ4_q$YIRun|Wa6%>bYQ2Um5x8g%Z9C-7N%Z><?6VAjo6^8dLGceOHbo0y`)Hl;&R;4<zuXdW44-h31A|?51V3d}LY4i1x
KOGOKXM;R8D&dL08%7`Z*y29CIXRSX^wkfdM|>4orvz48ic=@6@Pwh$JoW;8{*;#-HVK9fI$Arz<A+U-8ZCjw*5lRnv=}!iEIDCpUv65x&1J_w7JXjjQUy_I
UHu$L-@i<p_HgLfEV*Y!{hyC?OqIVve&cjaoDTsi2CD*#nWoE~KoB1((5EFO_t(5UpuGa#0S`3oyVs=aIy9bE)&2>9;eL*rBiu+afG>^q7V%JFE2bEni-*bh
02c(xkX+dWv1i)Tgfr$0kmHuA74j+>36PZU#|Gm73T-&R$1;E1a8J!Nvh6l`)u6FNJ%Rwwi0vPAw!`74Rd<o-V!(RZZRGGx|9&z5w>SUeyRG^&cUX%&H4pQ`
YM!0)F(DzD{(CNhjwRI4wn-&{(F0B{fSI{{1;mO6C~*2wG7LgnnnDWx!&O0EeNLGrQc#Z66VZl}&EOzyOX?B$^vPJe`jE}pylI6u@(p32l?m|+CO?k(Xw}l9
l?*^X!tbAMsTO2dK`N!q`%90Hf1zBz3Op^dt102L>O={?*d$$O3<tPv<{x0=@K%mkoXc%ZG_lMvK8*vI>~kUXb)dt2p<`>5PTVuPjiW_~Hp$N<z|JPQEV4s(
1yStC*jq?z_$U-CD^^3?inkms8b{lWIHV^h&*<T?WzY?#^@y;GR>D1KnOme0-{7@ZE++BWGlb#ZTFF<nIXd&##NL)N5)wIodB_M!KF|j*607Hfs}iFl01dv+
h44z7VzBv!Hg@<T7`r>GF(b_ntKwjus_{)oDz@z+*R-U#m*!=Gxz*ZAHceEr30m-Us_$z&z>T+1&b(&1cAJk@_T+#YKAktU==dYS99+?>ji6GTX;afc?u^4P
7HCC#rWb<4_3OV^ly#B*|J`2BJayNqe~OmQ?Wk1jYj6=gHs<(m)N{hx9Os!KvWXc{QlM=x*{Xdie_ZYm%nr<l0F<DC4EH+LAXns`$F`eKWw56ZuV-aAN+10g
C16(llXJvM<&>kWL85?n>G%lTmca64^n6B=8;|o!DP3h6+Xt(1Yh&=VOhB5C_N7Vq^{WGDK%fEH-9XQ*YN2^_`9yZIg<?6?EG#uL)cMne!#I@E_0A_%H%4FP
&@hpR{+-Gw-ZN#qe(^ta##6m-#>FE7%(sj?z9vP>tC8U{rMlJc57Gj!%uDK<=SnZx>Sk^Oa5wO(>FxG+L5uMANG%HFelCC&R>>bTv2JB?_{mJrM6rHpy`i0>
hcfSSPwQ7}?2G0lBYycV?vMWfQ{GHg6S4E8$Ot-2m|U~76i&MfhYf6B8(@l4f7?Z-;Sxm90Tb9VwAdgh!e99VHRysmET&th9>sqEOQfEtu2E9g+aS{a8)fAB
S3k(b^U<pGzF8qadC`C16z{6}NIEpcE%Upvlm0Dddg|DrL;nAEH`;;BD*P#P08;c=&Dd$$E{cQ`5sY$C1K(<R<laT-`qcU6>=j8G+s>+UJQUxQ%^B#0$TCd5
40$36XfIHCQwd36b}Bb)MzrQv13BMaj|8Qdka;v($!QNA08m!_896ok-tC=<Xx=-@MMfe$7dvt{it{awQIP6S;YjDiDFlXAc7tOopoq@=Lm+vml~)mB#?Eh_
BxHI;IEQCL0I^GlZGC6qHXeS`OMBj}pbz?K`gXWr>lKC5DJ<r=+v&F_e8M_jpWu=w;H$)f*;15><}6hn_!c8726`~Fqsz7Ju*KU+>a3A-_%A*VGPlP(FBtlx
u#rJ*uLP*=Jy<jdefR%+K(L;XQ)Y6{_v|wVCi@*e4WL_-j#xrUc2uzmAtegA=rg!ccALYuU9jmEJ214Sp!eb>8*3)t64QwJe9D`%1d)2R+zUf-WOalxu@WiM
=WJ$nH>RXlwpzrY^!i^V-3a{uwxDWHjJo4+D4n=_y7SQw38dbleh<Q2D0A&1>jl9F_zq3laDaTK^JDf-r>&i)tFWK0)9k0gkLfD)cOZzqvwoOLl@Zs@kDrVc
Ke3>TirVc-=haB8VSHP!OiaC6TJm<dcCa^VC?@Bp5BU*X^U6r4hCTpCmFYAtBwC@dn$_g<B;YbmWyv0BD0NP>e#d~2R8RoM`dA|Jm3_pfV3_*Fx#hP33lG>y
Jxo#mrsOS22`n^=w9wG1&{Oi@%`N{G0J}vNx_2#J1WL1<!AtLRU#9}7qf=4+WvnNG#N=s5#`q+l<^nR-&Z?_1kFB9vmy}Ps*J#zXCs3yCgz>RxvMM?k`rOp6
Sfk_!L3=0aoo=k;VN7)QEkItUkv~vm_Q&vIk*{iMNIX~}bo|hM?WYsV=_N0>CH44Py#_kraOL^5&z-e|VL(9?PD`?kIpi~WIKi~sGmUd<cw?5gNQ+34DFe#p
2#L#$n*J`BvH=W45bJawF^x1zEQb@n*zuc$yOEW9XGmt~cV4G%{-G;c3rQZZFn+`4n#kUs50H2Z`m<ZMOpYhXfh<@RhWkZ`|B9W-vmD5irz>`UOl?XeAT($-
z5a*c-ZX1<WoSc9ZRNcOq@ih<1LrPtsLgX;azKYozeFa)Jpr|@HK}yc=D?dXD>2;Ia`xhMG$NZ#u7b#68SgE0h>f{{pEZu5We{2Ca>M#~^2sYl%Sc6#B=xF6
o0`WVq6$xI#iHiJz*09p(gLp%_T%aBsnb#(kek=Ko;haHp1(N$zfycRS~<Z1rbq=|yl+$Fp+FKwPqU_H75Szr!2ppJ2~jN{ZyRRzOt-~eH6dNLz|<;_d7=7Q
__}1mX3EtR0CTA6<Be!C=KwV?CeC!$wU7=6F4WDH5j>+J1gKQ4Q>l{q1HxEc*K~PL*yYAccVxk|q{S(fD;rIW5xtu|6mzhga#BNTQ0W`>U+x^6E0)rT2<L#E
YZF{a?=wR;=~1Qu5DuJMG}mdHA4pAF)p-sP>i<n+S#LH;sp76=_r1dYXqp{RQePC^TkaG1h;TIeeNID<Ao%&&gh8D}lN^`MZn(u<jJ<-7r_t<sWAe&^qxIFA
@sT-a_nYK+tC)J0bWkh7-ZjrDK$}7$8lAfdh9X977?dlgykP8ulCGb3$4QR&T@b2n>jB0r!$tv2zmtTM)ZAq8{a9}d0v(TN9$*7P0|U-T%`Y%5Qk$9i;fcPm
8KeTTwbjOcgMLU>FLd{Bw}KAfthFr>o}FKqI3#8~8e!|woBO9GsNGw*;XW6qipSxYI&I_P9$9s_23ioiw2?oq=agoLQ*`@V6)T=sNXa<CF!*RRyF&2VJ8R!Y
!kGvNK7sPM<IcH7L`LL(cu6YB8^^D%=(kFv+VedW9_5ROI17KshwSGoJ#}0A_&JK^@qR8^R{~!`_~TKC8yofE5)9qwN_p_x#IpYioSnRxOV98|XBd!3q+HCN
Gk2qjkq(9>ixqS97Cge`5TOUdLSo$OCa&X9h*3A%vNG<o$oD9{Lp9shCpH2UfcDBV<(%3l-=88Uv_7+IB!S!RTC*781!e+@4J#;uNE)mv1PbNfm-olOK6(h&
+NpnES7*M~Q+P*O2+2|mje<`_S^%5xl^|||>!X>ZGaHyd`8@uxSio3ek9tW<Y}?ln=LGCfY>7(A)#Mg^0FtvIei>VCyrDO23~%zQK}~z$5Hq4%PnG1#Qr?(h
2zKB7CG=40P+LnGsV!dyLT1zFxZl=WZ*<;!PcfE28{H@bmRlrDDej#to#c(|>Ls*jZ!y`!9g=$z5AtQEPBl72oBCa~&&sNby{G|W*X`!0+$K<AP}M@zIa;wp
I9sMkWO7m3rXb!W=6rbx4t<2BJX+V)gQd(vjUoy#pEQt8dNnM-cZRm)l6goK915ox86?R6i#`y*(@$y4UJ|s8;xn)X*Oof31SQff<Q$bd?84Nytx1dU`OLZ<
F96s@z=!XO7h(cm!1c6m6v+qZt{+fd`&g_H)(h$H)W6iOe>7194x7Z5C@=i?dZSoG6JPxiW!X0(2oAS<D@p-S;&}LCd*jICMh$K$=f|Rwwv2cblhqbiH~1f!
f^adBK0Bs^HYb6Od7n$)OI`LKSDF$Lz!0W`%is~^fXP%HA50GhOvSVlcXx*Qv<t?nr!D|`L88+s*pV(P&bzhbts|wu{d)y4s?Fv8OBJWRT~K<(74y{`Wk;X2
FzIj(985^)$~2z+9G;>QA?NNPKwO>FJZ4?YK&Wb{=H&USr@tSVE50DHN4u;j;m$%CJ7*{LFf!k#JwI|ok0shb1AvF_#F!23SbXn#mA&lF9l_ZN=f>!y+RRsL
_<336zkyJ%Zf2PSE<|pIr9_5BV92$40KDwC6uo9grkzPHQH6OaF3!(I^Y{Vnhk$o+=U4<gbMm}BhfO+ytln&Kh10=k8%DU0{c#=@<!Q(L;kb9dITBd@8!w^f
twV4e3m-9XwgNB>rAd(_Zax+#3zoxTd3kc-Y|em@Gt0lJ2P>^T-1J@t-;Ps%8W|FADK1TQyOeSY{Mw=%)=`FoX&mf9Xn`<B$!YffqDjRUqZ@Y%7eP!;-nOdW
AG3n3H;9R10<G(V7X@W-3-bY9E%VDQDAlaUPhC2AtdeG`?ZgMAtGg5%eoBoJ-^bGH+ha<%Nd1N7rO6NmY{^n<!zDJ~0B_1?^l-=;tn&gZsGo1V;vB#{D+^uL
3&Hm1mQBjZn?RMJ(%9sZE7UZrpt?G66H!w@X=LG9FYmxilCbINv8pJ}hW^q_V$b=r3Gp{?&6w@)sDDS7A$Xq2Z69HVUt~pjB)JhJvj?9#FboVKRBT?1qO6en
6hyPTQ*K<DkN?)Q&e}|c-z%9q4=W2k^J&bD)($A16^JDqU3v5aG>%a6Rn%#80`uw@Ztx@-aL72g#$i;5{hWV}w~DWP(GW6^i$aIF@Kd{Dx35-D4-DT^z}Sfc
z3kVQ(W(jNx`D)P2{?@Y7)yfaT!8f3g_4Ep?Fccr7YvFHLtxSb(PT(i(e5u@@<~U|j!?O=x!pC<Pl`slj|a4aE!y3_PO!VavLC)p<4VUgrOQc-*r0i-p`L4r
>w6d64kcpWK*5VOQh^9aV;JpgiN_davHZm6lWg7$qc^FgyKxB~iNi-<amUsDJgE<Qd_4lVA^TA4x_u*#VYA_~ik5k}7WWIF+xaX@b1q`dJu7^V;yemCH$R$#
x;9N~-{ssWeT;&?g64PM3#h#a)Acale>Yw{Jhi#8(>R7O*>3-ekdBXQ^(95hIJ(?H`Qg|K8cO{)roUzF`EZ3jXn46`eH}AX|3gn9R4e$Z4!&ZB3QkD2kEQ%Q
&oWtDU~a=vNN;q&%|7gUV~-8SMFdWFvEs>@gK7Xm-dBdYfZtf<a`tz)ITW_*1M4P&LOTYM#BWKId%YmA<Fn`+bMcxwwO6LDjPHYTrLE0ob!nlgzQs>}O<2YT
U=1F4dv4oux3nwwkrQtJ>*S6k-E&u`<8M#bIH7l)ymFj%B-&%eSrrPfbG5$9+K#Vq?3Fih;wXbYY#MW`3@6{s@s+6X>~BKi<@AmJERI$8%`rUzIs7>7Y^z*e
fdOs~kzMD^*BM>DQr3n{RKI0O+1ANL$rL!a3rIPAOkg@K{U11*xk%Ohi0sD;p$D7cF~=PT+=7YhOVapl3pE{I#1Se9c6&3xMdh?Mzr7(uZJGr(^7kHmg|Im7
Ww&2(T=`(>`jHzmbeX#G2v#^CBOku1AcUuRDur<;zDeUjUA_2CjUVr*QlCbXD05Jt5{G_7aFr2wexNGpChdeU<oBrH!D7C0h3EpGU$QAAi5rv9_kxThaV3F|
XWoQZhF2GqH2U@;g)W*FyLB-Yma+2jf`}_}XISv`&kyn`bL9um^;FQrT0fT6Bbq3dcn67~C-w3NRc5vgkCL`Nzfx~fk3E+aB<Zy34I1MfIwh*3f`ZD=od|Zg
vcVE`Pt8_lzuGPr>6Q6Nuj6h5<Si@xwFSkc$q3|*DF&PNYrhJrJsGOnmWqvXTymTY^@GqV>-In@ZgWE$%604N@+4hW&95E0Td5aoKdaC=!$I#wEW$#2NOQ=d
JR3Z4Ev0-Dks61nq^r;2OaNf-Qi*ES)IJ_TBy(8-s5NyJ+l0hB_EuX(32*ZqJHgau9TLTJhTy1TV-5%IgAro}0-)Z5(}Dq1>YZT=5EMH%m|$wnH9_D0AIoq0
+hrEvQz6q5wM9nDvKMki+b^W*BSRW{!S%<!h`N=a0xHp+0un^9o>LItn&Q>M#*#8t)A+S1Nb`Qv!R#Db+xJ<5HFJFP&MzG!r5IYX0=N=_bpHlC1ti`<{9{w2
Gp781<F*z$wNZ^3EXU@%xMoGu{)V`Z<GhM>uCfA(S=MCBIs@nZj5QO}oS_Gu_dP%@Phkma9t0^>zB({V2%RgCS3C$UPGGUv`QZS1qP(<{3{$7Gw<c;_)LCo*
C8lyJHxp2vq$NuDXyp&dj3Hc);KI;?;Wqd<Sh@(RK18AF_o8eg0$DIErU3;}wLDznMC1%N9dj{wy30n1TbZHWezrHwp|t3FNL?Fp#S?JVR9Ms+<fojcXQUin
4;O=`xXW2M3=}7x!m5<v1aLQOw<B9UOsv>fJD_YrWrJQcr&T+Y|JeE+GU51*?Uv$dsH)A4)b}s#A<>v9%5D8L>Mrt5dSAf+I>(_A`)8|B)!$u0R+%)ho}q=V
i2Fq;MfxhaRqef*V5D{s=8`@9A0oB<Gg2&1xjXeN>3x6#(Pmrv&vV6Ko3xUiAEU`f3!*<5XJnBfH*f>{l5SUE-tA_x>Rh#sdT%J`rCPb1ckjC{%04jD17&T}
mzhCA+k+{3<)MscKEdeHj~ei>P(m--Tdd+?0fu}m+3?wz?Xo<&jMw?3i|Zlq9{Bh;xi6GrbCYTO`CJs<Y3!D@4ZBL1Bl-IuqQ`ZxUc=0V3b0ONI$M0-@FQyi
#^4R6%&bfw+P8D-P~5u*lvZi{tB!rs$0sykWay<VKnsiJQHhO9d#J#DqNUbQARK^dQdiU~aYBci+PL38T1P)-h4KCG*FpkWb^nsZQYp|SUD5rKL~1M?<Bq76
v6l{3()!s1Y&t(|!u3c1S=k!tFf;ZHGZwd4H3%jT#9&A;O9FjTw~Qxm;)CFA9a6%xtcw-Q4&hN~K0(gihxo~FCIW!8y%D-y8E5rs)mEfd?9(LKz193Z)52J(
5ZIv>F*y(=5X`ArH+_x5uwC7p`x$jR%s+8T!9%NgUaN@uf1QTU>V@~d>24yQ_;&K2H|5tbul7h;FfY|uM_gZEIP)&xXB#*sAE#7cP@TPd@?wB9E!86+m2b+>
6SD(_8JeF)(2m#<Z#}l!2L_{{t0=50v09xcIg7L33aAqc(Pj!z7;prD-T$w`rgnlg-1hP?_zLJMl$mFW*@3~WzE+rI4;$pPKJ6L<bZ23XLat`g{ruvIbJ5t4
I7Bprgc+oz`c`Vv+Gm^C<CXSdGfswgq*{`{fFmi#MlOS<h}h%$wWx1dyt|?Z9z#JkRXBKtOES%+8Nq%&J<tT%<})c~l@-%u*5U2~j(4{U$Bf(-3-gx?rkdfc
2DM@+V>q{vs{BZImO0krAQ$S2>nV~_D$!lxS&}$R)I2Ic){oCkeY(sF*E*Xrm*PxEXVxpUpL4E4l0mhw+D|5+Bo^4As2ONd3XoH6!Of8+vL3JxtAyf9?$22(
yiBdBJV87i5!v<~vLZ}%;>9}>HckX~){Cx>dK5X-&X5!1UqJR~%Qe}s?>P)xh^?+`h{zu;GZ<RUN;sCpoel+wS_);@0zsAjum4xORIKO0tZehc9^8NB;dFhm
yroU>P-jUxy|MYiO&oLBf|$j9@A}SHzE8;k({*%1#inSxpKYMU%8xGz_Q>g;i)=wWp^D>~oXKj=M|zdEk$*2pg#9+ZnCpO=qkW#{SSWXXx!SQkY`OB)U8m1w
;>QDv|D|l3f3Kf~^}|I;?D#Wwoq}GalLY9)#4mQMOjPP16=1}+ydn)!NqlidNBp;9AL8mNhJ-M9fMtG`qtu5lPct+@s4A6G<*B&FuQ}cl-!)&NIW+5H@k=pk
J}L)z&G;Rkgk~*0G2v2>>x29>V>HW&FqgvyrOH9n?Rkxo`NW~+;eUAs3p;z{P}^Uv0xnj8OGCpz?N_;xO?r$E8U<JWgeW2C8x`p0*=fPLcQC?0*`A>XkgNil
y4+z^kaNbGCKPU*G&O{_kLbtmfQ$U-c?mU{s3YiDMQ4{0yM9*N^Z=#|iyYiAn*8^p7t{hJQWZX030q4vKy1;w%#S$J80HDP?P&^4BVJ}nI9d7tCF-}=;j+g}
R(Yo?{G@Ta5J)*hnQ7GiP9jlEfp1d;MkPIao4RE<{5iKh2N0-JYIHxr$6sNhYt*wu@s}%HoZ2@HeZ;ksdjgT7pA7T$8AFdRi|ECtwvs);Mm}sLQPd4nsS<^i
0M_OvZ{)xth4~|eKEzbnFk)y-a$BmZ5W%uz#)7&D@f9AjzuPNVr|;~JdCtzmJE(fc!6I~u0;K?VGB&!%ZS+bZnb!)RvyzwABg;uF#MVJ@h-?>&e$CHzZ=J{C
mmVR`+zho;#jWpLNE>ST;+=c)8T5bCa5VWn$(kf~>sbY{0aAyU=Nm{Cu9Rq=aZNtFwYm_18qL?ib3GIvD=ia!`b!{}ik|UjmWW<ko|-aH<+X?&KJ0RXofn!p
U>VGk_~Q|XS-;q>mK2-V3F_p0Ki`8q;^iM}G(CG1JEC<jBd?9@1D`zQ?db$<(hLfRg(=aysRP@EN8qyYs%nV?9DMWAH#@j#k9tt0Muy1w?~h1f4D#AObTcYw
{{$-dmqI>!ACRIND*H>jBP<%W+S*^p06KWTo-S$SH7@Tf>Jc6f<<x0D2R^Rsq(FO;fe(Q8PbJN)rKw&TA*K|;;=sL}oy<S$QEj9^iU!syMXnkQ8Kn4Y#38}V
`187vYHEf;-0i=-n=*N&OEo}*@kcs5`!LZST&ggiHl53RhC!VjNA!izg+>Pd@>y2H@*~?65(X1hO>Ou#k<m}lE)F5>h3P!*!9d+ZRMz!@QZ;t4I!CvFGEroo
jZhB(8)DB5wp*ZuX|U^5)+fe7x(mx5UDAxuZ8)7WTA2jG#kuSECbM+}lCcxKOlUtb#06*}^dxT>IE&Wt1W;6JHDk~KAY6b~FXnI7`rZorjI4yBk6pJBOpl7%
Y%A7hp8Weno{ptC*n2RATJf<>?8I7H*+|W2iAEHaaOslgqHiQt0wRHPKsbTaA6OZ1123c&xM5Me8<mudT?TZMWF`u|sz=e-j?vlicI+-B#pJfw+ZDBUQ!}HP
4vZ7?GjUWY#bY>8FaI*kNI5Eh%<OTD+HeW&`{@d^g$*21&G{tE^nmvIPkE(QH4#2<%>C-UbpiDZ&K!>1iP8WNa+}L4-SyDy&L(l0uTKaJAkozY(xL(6UKnhm
9$Ms5FPPHQVne;p%L-)4q$-Z2brZ4kQ(SEJC7ZFdfMmY=y@c<za)P_Hh2})i8C4@okiVkPWfl>YuUP?Dq*}>_vG;hcNgaqE2jDlPmh!&-?u2kl57l5rc`&`T
w5c22b?V~2`lID;r5yZ=kDSx4p6fM6Y<8%G*?sWhCXDPtwtQ=+YuS~fb%Ddv?ts65LMh{U+!g-Y4ht1TyjVPg8lj&O-Yk*P&O!_>@3Nqrww^|Ca(@|lHbUy{
uztdak07iPaQ1_vFIXdUc62LQ)K_7G#S1(^80lQXJq?M37bj~$xU?S|XhRd|BJig^Y&zh<I=|hy!W9KF(qI+^)FHe=&4`*$KsIX}9VP89m{&McKEYEE*U{9H
3W2EIn7m2o#L*~*eGD7#@akn^sYPXLJ_LSLJ5w%EngIA18-g2$PwV5fKV_Xfe1U6wQB6?Z%PO*8zDN@EQLSy%jc?`j`SWH>QT=kbz<rY5QV#O(*7Lx;oYPsK
-a-<L;8in#vN6sCMIcyI<Y06{y^J>BPrTlXTbhNLhlYTEt}v;9iKtlQ6V}79TaWYHMA0>6C(DsS(c)k%vJ(qb+>fga8#+-jx)eU6(K?4;O9)t138smbll@^0
9u_5(@#H$Sav`+iQ@LFX{xOaUuURbSr>SE)&S+*&%-kkk5hV*yGGK`P|Ht*02|vaW(9gzmt1O^vG1#t>d<}QE>i<kuD>{Gvc;PmsG5iJ>eYle!l~y65(1qiz
)$gL9%-jB$Z)1|V@?Ld^iE&jtiBk$&l_Eb>SIU8aQn!oJFDqi{t{GEElNkn4NgV^hIwTALGC!G>S{Xb!X=N_`P@3<fqC2FGE4QZdPBv=!UX=08x-CQKb9C<x
svQRM*mz-aM|d6~bDNompLHpQ!0M~8=*rf?;m~sD=$4;X{0z-4sJ<mDY9fb7dtLi{)3)i+Zo>!3X0kv0A!zo~6_U}(G;oCS^9&QA<(1H^q_HB>Es34eJvrOD
;6Xv*T=<4XKP3nIpKg(eB#xsW3GwfQiYZiPQ^idJmSw2__Qgf7#q>LxL53yk8c*$Nq>A*hHz(oLezU?${1}>)<<puyThDB{gDEPNsJV{QlC^rM_EAtCn}k@r
0TiKVksP!q;}TMG!q#xh#Yv(!MdJ*z3``i#T9l->ty42ZgKJ$NT!Orhe9?-oUif|MB=hX<x!DT+Umy$!vs^nC79Jp{g~C}EhmA>1-apqO_D6O7^+4)9{xs5O
WEI7)b+xi#MjXmiX@fwl@Q(6q^k}VmZHMjcyrwtaVkW@NJ`pGbZ{DXkfMwAuHIN2x_PF#GsKtKinL^Me-&l6}0ZS7&v?gW^=~aZOK@0LC;ml|S=z1`~%q*&G
c=6cM^+d2Pty=Dm(65U@gYMnhST99YugJU6OVp_0K?D<(CGFU#g<}4R{umW_cc$VN6fLXjrgFcB>llS3hH^X{&UD6>pki;6RK7J4X)V+Z==i;S%WVLlD69S2
1cYwiHZb{O%G#hme=Ve!QX&p8|A_&0zuilHM5QLWY`nSJN)k@)u^d-^X?=ovqJ(glk-_L+5TJ$EBdU%n@0t%ORD`6+iEPRFz4Vdq=N)bAEj>37M<u;?`H`?s
lK_-#N-*Qu1K-HcM^XrZQo+zQK(0>z0%r9?2|Ibg=2gw?2&RF&#xu@IahgY2(*9CXX;Q{vs3g^vc<vE=^@cH2ZcvZj_ZYDL&}hxt-VShJOQ4{&uQ9<<gZ8w8
DVwzjz%~anRjp5e^DK{i75^F4r1IGk=Go(^tMbHLoP?RLa{TrU8zj7!OA0y&D%860fZ&J#T#3{lwB6lInp_+O?US<Peae%+%!V1Bh0tezYahq(YM$F9@=w_X
IF@?GXa6O{tkMx8Uzn4!(4#&R{6*H%T!Dw`?Xhnm_OyjCRN?HTQSK$rv>Jf))ag9L!A^0T<HPLV<@;i^0!cpi_OFrBBVj6*RpgI;daUW)0swyEB1FNS%k)%b
S91r9dt#@V?5!1rJSA8iKCNPY)SO`*oS-piO1N7CKRC_1djBATx2P!AkWcyG2zDFIT7<Vii~Yp6QlyYjd_C+~kIWMijlvd}D5}m4sgs)_1C8*<D=Icjd&MtX
w*qy3D3suiV(ow)G1OW^Hv6<~HIJq1kgyV;U>b=oudJ$Al;9}tV%Ctu?X)=0|3%%&&XH8ivO3pX8i6s1-5)eRGZO+y{3|jY@!9j4>2xiLKiH7lfgGCxKuT!@
9D{#q6-F0JR$rbEx9-%-*9K2~1Xd#dDW(1^Np+BDKw%l}oJ?&_pw%dBtBSLdvzH3x_MmxKLBq_-O+L=ul^brCw_U!FlQOU$Y^)>eMe&H=2qNYWKA7R|#<0w}
JqQUxYrf;8iv)Ij{F_7hgOe=%DHADcT1EU2t15kp+%7?JoH`V6e)ov7Lol9JJi?SWK{5+1aSa(8l3J|CSERtZ9iCg>H+8B;pA%CS1*oMiq<ejKE^mS?aFS1~
Vd_%~c8%b!g8`tm;NPddWc=IXQTgF3(KA3k{G8}B!xFcr+5pc^Z>7(N*H?3hffOJIns&Oo%BO$z)UQV+r@FqsN?h3bd1gKry6gT#QEnwBFQfx@#&^)|nH~dQ
v6tsWC)XbI+0UL-UAz{p0GJLeiNrWSt=%pAd<#KNgTS0MTQ3d5%>grT)zv4nM`{*mQ!lCjNRO4IT^B;ll05_;+NUJ-j`G+Z>(x&HO-a6wpkRjMLglrXRx{p2
Ix0)372#z|h2%fH#HdBqs=||PiRyWIvU4$O&aPGo`zPq2HgS9<zjMjB`)!@mwzN!?O+V<NSFOG_{`D56LlBk~H59>i){iH?bNBvBSEVBjBvy`p7-B`7Quuk{
0lu?`H{vQKfdc!`L1C5{PK(+<^cbfdF8mHB)k-?3Yb;+j)oY~mntY2T36e44%1jaaJFGS{p3ryhXvVlC^US8m{{IQ2OVi5O>tTxq4qosXe?GXSKoEKM_oo4Q
PQm!=7iYS)<W$41NftsUcJ)LuiSd6_MK6p+94kJu(z>9K!>5A;4P7gyb)};|u8|>%X=^(9PO+rEKsvP@D(K%V-2)QBCD3xpvbJS?S4enqdkZ1<5Uwi!tkg``
wx^mCvCQKZpy4+*Cf$hk)Yq`}>B?!_(XH=HIW=imB=UG6i*5`Cn-&!r4bK=zZ|QgARL)&LT+nFLIcFJR<M`{ZmLHc&4J1y4l|mAfm1|Kp$VqV|iTz;j?g>_W
{riN#zoZbQvrg;qg$d7e^32OptaniSaaJ}51%Kqo6HT=9oCkx1)+V`>xzJ6R@x@LpMNJ9kH|H_itxD6aj6}LG*S(*pRSNv$VEt<sI)1$YLQo8irPdtGPNlg+
<s^2!L8}GtN_#4Zg`R0Aq@)~UGS~{`f&ZYJIyHH~WJ#J*?Ozd!uSSedsUkiQYv9bjUs2SJYU;j)PO-2<i_FHydtlRo?NjANc~rCBERp!za0!kT0%k8F%wDZl
%FnCTd^HxLJY>)G_%2_?HEL=?E(g0|si$_S8t4P-SAK=7h22$_!KXyEVS=ontPY&UcGV#*bx<qLJ0BwqipBzcg^&q7yl_|NAC1mXUG0-%ZgYEB?Y*nG%MqL0
Cuw6DRIhHF9oO+j=GgB@SxY0W`|Ne_eg7D(W}d6#9>b_9&b_P&e)eUjAg+dxNUVZB+m6V)>}_ke`XA+&@4}>k4~{{Yb1ZVr+?bj=-l)kAvTEX+Il2v%)-jm!
RnFBHWBkBv{{@j1Dpw&Pn)9Er<T|Fw3D<vVs7&f+^+!M%{`51NxKSxl0;ojMw5!=s_O5_~;O?3{f<BoZnEWeUg~|g5PkG2z%d7Qh2CvSvqhcEk>Upy0;CV>I
N;{vrP)dI;8-rGk6Dl<k=hvW4pP<SM%|XXBoWOJHO-){H!BX>dh5s6Nv*%#PSh^{WtFO8cVWs%iltUWhE-0>Prhw+D^~H_STw@}2jZ2yfJ@?w@(^!hsLhpLa
t~KDTNcmm-38sa>lPn3nC8V?SJ74)$O5@4aLv~%HFMD`;%`gwaeQFQt_Au{907l2Ej%bF-o3q8?K}&}41AhX=eSVz$bgr@yrKjvo`0|FFP`{=iTzRvwSp&JB
BSz_L(YvE_aNN3ISg3y)_XZ*2&*XY6D<>)|ILGqX$pOof*%WEOlU~tdgdPoy{pw+5^CZfYHv{y%O@5OYdTV#oeHzoOPQ<wl{IIF(<NjaXNanp-OFHz8A}b8Y
jfqQl`oeIy;XkcY)=n&o;R5XLN_i||u$r?+R+3M`a#v{0a|73=-lyq>WRpNp&PO6V`)NUpzdlBt4ZGemSu<N&RD(-Apt-R6gIbuM><|QLld`gK8Zedt$!l!X
uf`USOZvii2G(*WG3)_UB1XG(L*nOth*dkY&XkDt|C5N1N&gFoC!SK&jZOK?O;?zsI^RJ3nU4$?ac8Fo`N(+Jw`dI{)QqQZ0}JYLG=AmEEztl1Vb&v+LwO+h
NIdt-6qHH=tDRC4_j?{dADge$-1+f?_vfy%m>=Pg(a}u}Ee5e%-XN8PE1=M3mD1rUqb{fJIPk#1IX!}vV>VNVjw1>3`$?nzW>|o|P5L(_jvX6RU6RQ8;Xq%s
lbMKK17G0`P)W0c0PCz7<!Bb}fPE6BLGvuT-mP((;_Mp^^&=sl`ZaBn5&MYZAYeOY=ceak=!=f~E#W#};Hb)1-Gofu@XfWrw>t)Q$JqoIa2qcsn>6jLEd~qX
=Au-w-djxJ&DMPrxi+t+vlR!<NQ`of@W>_e*gL*hbEoszNz5BAL<?JF8<nh~kQ{%$43AiM;>@2QI!(l9)AY_SjCaL1@GfM?Xo}E<#KMCMkGr~@vei<g?DIlb
0ppu6Bon;3Ajx|C%LkylJhz@8zsk0v1m)nSA-yf$sXRW_0p*1_)-&b${2g%#(jG;+SLCtbER@n++yo1X!%yF??p6n#!l3=HDI>R624f<*CYFizZ$+!<Y-v0x
k!s0S&w#rD(Z_6LKjz;eK(}yrMM5oTKAly2UP4lM`<4NsDf|k@ev{-9FD?09h%6?aei}EiH|>~f?rJA{qF{$;Hwpj%^e5nG1{1D%WK0AqL{Ihs!LFus0!fb?
VjO|R5j??`cxpW!>s46HyRP0>zVl)Emcb`8gNRZ30e>=wN0D9#FW?Bp1)vy@yW`bxJR8_S-EU<RcPc|s;$VZ<c-~^hwlF-r2eU|79C=*Y@5OrTia`O?nU6La
!Q0n_;GXLgw<pebCd|<?UtEDy1vCght0h0&M_wCn+V+)}tC5>8fUhwoyDH^32Y35)hNp=l#P3noHok=ztY$X%P+(m+-6IeY{YD8jYP++fw49nep%#W_vAJ^q
W1NDxJ(x;2S;2S;t71*X$B*-z=Z4cEYCG-gKobRbbiJO3Czz3TD^+t1;qon&ky75KM;KY00Xnw&ga{)vV0vONmo0?zC(w>5aZA=|GUTmyp&W-2#CpL~P1wA|
I`U7ftx4{GU14?*Rp(s6%%qHC2GX2vA{Rh*3Rt3Uv9el|%8T8>V&)v`yH;gFN(2y<q;5-+PSJ^0_6JQoCl?;QJ49LGdWFXn9(2aBq!y1<cdU)HkC37hy){n*
&g5Zvic@Zzy3(IZ$N!VrwJh|>za_2kJw2`dz{^sdm-Pgsx6q;9-1}T$)$-}kM6ghjR=W!RGbMdz3K%^;F%3r%v}55>fsEK?CVX#DUTZV;sqp?|CS(CkmhC;}
xQ8m80x8KTJEq)VB8_uB02_wUgo$*Xae6=sMH0FOsT7j0@TU@Nxo8_{)l#6F47CO@4wHxViW_HNzxbApL73=R1eaPXa^!#j#ob&Lf`c9k`<1_J?>vxVC@45}
+|H0qVd!vX?bGa=KwMe7{edMIt-RH&m1f#DZ6b40KC7fX);h|`<vAtnCVle?sI_R?Ya2>WJdRRpEQ=ToC@wm-*7UNUpA%M|PI{oqBxIqjxLX0Q!X9Gf$*mlE
#?2a8^!d9*G_CjN>+$AG=s~G{#powOro>)_cn&%t?!RepeiWI)%+f$ic+)7+awmL*)}+nUwcIs5>*uyub=zn0_+tt=@M>n@%rFkg9*^Kmjh$#{+lNK<qLzHV
?UQ7(&;7B^h@4^6kj)HnZt)OJLjv0>m~J&>(9IwJ>=|g~T^D>(YT?juf3s|G`z*`taFciqw@-ql%Ci&+P<6805`o))r3S>otgD43TS?mS`&TTA;e_||KYLrB
a520Wpw?Mx4ng+<`tFgPWN~<7Q#y}SklNBBUv{Xe8<&mzWk7ceN5L2!e8b>mr~}xQ<UKaOkNUgC_6)*BU^a9wQt=7>QGt2>==OMr2w5*ZFC8|C%Ty;}M?$ty
2E^qZQnXf)8Iyh=<aq6~FieyrMI3w?T!Ui2pfX$fy%!UX(wR|wFfApOo*MVlSF{%A13xR2y;V=)f1!Sby3a(TVN%nLf?h{zxJ(?0`L&H|f0O>qVFR3S|8x=`
3(Fr(V^bA#Np$PtwHN$SaJc)~Yls%%xGdl(5IF+LhD&qo()mjeNNQfye<ow3ycq}dw5>52`Rgre=eQ!{KWcDI<X5sqJDfhWI7{;sqL6did-)N-GBD1rnw@X_
O(Yadf7{Uw`prAQOfw`-6`x`BjIsQ!M?ZCW2d#Egy4=FdX``p*hpONxL2i8&0o%JP0>WD7?SIg|#s9N$-B_8^s6&455|B>kl%sdhy9E4tTVFf<lI_2}l?*AJ
AHVrE-DMNI?SuzLv3&6@FN`$;4wYX1Hd4<+LA#7264a-w`>}rKcC0Fk$%>m#s8rD(`KcF%dw!4zk^v3})Kfh{Ac$Pg<J@B(UkG79tpB$D5&A^6y_3uzdy8!>
W$aaUo{#lEc4HC8`f06NB`!9=6s`Z1uinG-Z$La*gWIxy`^wUL=VjCsV^AVK4xiu)U_WJ47T);{-9vZh{3DE&xd_BqcEY1_pPu%Mj7}_{7Ow!rJeb;%FJA|C
q_W1K7q8QJb{G-Elh}1_s{vhVPMDnzX}N~BUseW{x>Ay{8!7mMxRO`+evEndy1bg7b^A6jo!u4<-HNuXTAl(@b+#ON)=?A%1JKuGhsrBW^H$sXr8nMr8)%%%
1xP1~mU=+T&%z?Hq5UYip9FHW+76R$5D&>AC8dB~k*)Kp0k((x2CpiA6m5NKt0NvnWiRmWu4G{SwN7_!1$Dc(6|?I_VDS7o#m`FwXN8u}3`#@AEj3&v!;jm$
m*f}8Wp$|&axQH~0$nqc>LpUREkVk%o%&)e2A1b*J*$EKy^P&&CZxwjhus<i`p+SFyjkTYFjj5hAB6C`F-4^6!h4lFiJx(gU!L-h4nM3=t??6DoZ-?Bf@qvS
6~>Oxa-KY;9?B#LPhk%I`iuQ(qAdTsd@YlJs25H>YF#VwIjVkE+kSX9Z%TW&n*Yw3999NwHKVk*)qGdVB&XoG)BM)4krpS}HG$BOvc-k?SHQ29NJJ+pSb-qE
_R})02vwjr{SUQM3KWlpG)<dLD8U}q!roc!2vr2VS3(|EG^LXV7aL00y22lAXaS*+`YuHpQJUx{?>#Q0bL<C$?QF*?T{1;fKixkZE|raDbbdbxGK>XQ-lO^0
_MWe`4;xQH0Y=4PEO^`DkV;{<$#7|V!LwPtQc@An)IEQO7!29-j0od6BY!17sRoyE=q%Qa97x1yBEa~1<0n10vD(-N5*?kP*0xRX`_4zVJbClo7L4Juq)$wO
1u|xaqgK-@r6YZ``jZS^@2OJ_IhB>eJ(s~S@&f+)st}GtQn#o#`bqHp^`{!785Wg19F?hpKkl7dDh>hkf^-&}u~AfNarv86{pl>JSMh=Jjckd3d7!KmuA=qx
>>J<mG>0u#WR{3yr?SQX)Hyky9PF!#e=|q|joa)l_?mOs&#SF~%nfal%_-#VXdt}`mQ;?<8i!VV8|Ntme>Okg!^&U}Do_k(v^B=MGfG3X5<ru7D_56xar4br
SO^IKKvTpZ4a8H@Je>E3q=DgCp&TjbfYGKRK)k`yWELQPWsUVx)i0c^rhP~NJ@R+AzX+~9#7lX4<Xg`<TsX?*4eDK(+k>n=1=7!(bE?}(=DF*lD#=w)m!FDH
noi}3SqVHj`m59+il*@;i`1n`6+#=<HA8ojH+jone|RJUx4G-J0<xD88jMB?4i<i71(?CyTxdbK92KbtD(NMTN5ewS$WnQ%y6eaRY4ajING5HE%gpOjG{_xP
dnypum}cMuD-bk9dN!1U$)Ryf^lnZt)))|{G6QK@qr%R~b8NLZZ@P**R1k0X``ZT$lSE>EhttW2iU6#U1A8CS!^AZ4EMDqs(DINT(&QB-uZmsPo`d{&-9XWh
u|$~DwX`CCL%?8tBWucLImrgyNC+74?Cb{{PMQ)jQnn}E(Xk20E<&$cDe&b<b3A)Dn6N8x`TBx++JaH-xfK7sN)-0Hj<ofle=d1O%#2vBU34C{OB?u0JKUCJ
UvolQ>jH$7`CF8UH}KnSO{qAuGpxVS)rrpWf%icnX9-&P^sqkzo<fo(Y%mSAJg`KfAY4u$>sQT_F2Oek!$nC%w^{R0KT(-`S-OoK`L9sYvHnF#=b3lsA5-#v
uiav{t;D@nv~WSx1~qdF(C1CH1lf!UIS+`-gg)Lqcuo{LAF4`nu?jZS+WBFL|7jPa<Rx=}`M)1Pz;Ej&GYsX|R%_Qu`<9XN=yBQavPN%m*jUqoLa;hzGU?d>
l#A@N&iSOrg;68*Ug8TjpA{$d=I2!@`kO(&>w`h;Um1MrGVSFlS&U@V%`ZfU@2pWLBv}>ho;dd6o-x9y%eIhsYO^8uEg>uL0+)r6#qd}-mag@<A9m5@;91Be
F^|XfE91V^8>%ImsIDo8UDAyyLjo`*m{icC8uIpcPAdD{K~TgFbn^>~<SS!<TEM1I8oa9j?ec9MUFV9wG)j~=itaTy1^xDD%UI(Q&G}OF7=PgxBP@`{Co+|f
dE1cc9aNqwyqdY3G?snL>ot6Y`-<jUxgjk*dD?H97$n>x?&nO2sSS1qDDsnlbB!1*0F4R$Xd#PGld=cAA4qn7K&o?)M@GHC;myb*(H<Jg2dO@5eSQu7@i1~H
$grf5NUlfN>5yh%Nsx{M-KZeTa2=x*Sg3b=@t~l>Q|CoPc?w%9$FbMIf&qBkp1xMn7_(kxi+I)<h$>?B;=zBD9Q*S;2wYp8XQ`N&HYrPdO$zgDS994(x64r%
4`p@;Tg_o0m(I`5&IG+C%}VML6jvd@xR~m8Oy6foBgb<b8nt1Pq?K|!HR6yI1I!Pbb#~qCvB=3g<$MhF`u8~62CKZN5kq)9Sk-9}Wy*nkue&a8`sVS)iFBSO
W%wWf0#>reI5L|@ES-jaT1JcfEe_U?5;Bz2FK->$f@Pd-dx9Yv!%Z$fz?*N#cqV!O(9KsRa*BoZUBiJ<=~Wy)6IN7mZiMD8#Fg8UqCqr1-)5988~>sK@`Dm<
PR@jD4JH0#o8z3UoPoX&h;f=Trk{kx_YZ$4y+k(*&5)laS%UIXx%P=Sgv+uEvCG*>wpK4p<G8aA$KTK1rTp1e2$lD$I-X`K2qK@f_q3@H)WB=UEw08qvO^Ku
ZNAG7#A+T}YCb0u;O&S{i>$a3^5h@tiiq7l0Ukf^Yt+o;;USF~oFkHAzc&zj!1>PV2d*b9)MU7A<~hG8YNEwOMUt-e3N}w_iDxks^s}8Sv+2QN;v<pPRfi<8
r>De%(cm)MKSBJ+2XMGP$mi+Qm)cbF)20l1O)){<F7s=*=+0k0ONyTO5(Kn>q!x**U*I%>l&g4n3HXO8E(?Fdcocow^Qp(>a7X*xus~iXA@PR-dSLnvixuEw
Z@fW`uFI?L9+)}44`bAUT?cpzr?|(vXeK9`Nlz11pWipT;P;HDdIWw?G&yEum?>cs9atHafbmvQ97wHn$`|*mZU+#q9Y?h0Ict%9t>5z(Ci@MM1GgEzovl-A
b(*eh`fb)pf>P0-G3MUHcQ^7J)K98HuLa>NRLa>-Hv@jRH%bDBQyLXnzzfc9H?oX(A3}woLRUgJOAFN*t=kU>T#U9;EI&-UAqow!ZZ52UO`$vZVxq<@6ouJ5
&PvMD;;YZ$g?lz-=>Q=@doZ^g20}8<4Cq?x^qJ3r{fbpdMi38=#*|4tN*S&uu_8mwQ3Bcwm-6Nq=D%syN=GgiG^`(ZeUFW`r)I<OPK<;aD8K-=kbPwrOEa!(
(5xbQ!H{~<t1rJq41w7eCM{xqJhA{cyN}Q|Vd%Pga}X3Fv2Yg+-E-L(_JM>=xsn)a|6d4atvf;)GngBPmS53&p6S}XO%nNn(2Hy%V^Ba|B01k^V5L3_GWcPx
R5XVNuRsUt7s!)XhwNQH&7!5D3C8ky6a#5Szp@8i1z}>gg-XmYIUCg!9BzsTbbEZ^&~h@sUchP^P;oA!@bC(yeX|~I-Ko^j8S&Akq)Ye%JF10!CYY!_`}Z)4
CVud5rm8$b)%0VLc?$KVC6Q|h@U@(Cz~@r;F0tdFV^DFrZSH@$+St(@R*;^|wbCn>X>3f)U*07DD7wmZ*Er-|YgWGYSo}V^?SFpJ<o>VkCXSX-Jt?A*=Kvg2
vv*OQ&%_~n>rXu>sFe}#U?<h-1MIolyC(=oieyAIuPHlKtm&94)?|xPIasN3nx#SIHsB-yz=nN!q42xfc7LCJ-2rR(9p+-b!$3w*t4QSPPp>vg&u|@M+k3o4
?1wY^r&iz1#Ot$Pp^43MIPWSCz<~TB&OFE<^Sw6$1J&RUc@+2llD&vIzr`|!U7=dGrB=fa&wgAXuaRHsLXmxJ;9%Wcm^p$)lUkT1SDFcUp6@Y+{9k!ZrvtrT
>F>QCL;wpKUM&AhAWHAEWWUP;Zsfrtz5<Iv<DA#xP2CqZuUJ0_IT^sZ*PtBP4wk_^MCjJ)4&QLyCNwa1SoOyLT|B=4r>JdiE9=yNF1x(Lu;!6*Y^&G3pSa9A
ZL(K?BHDuKvGPl81iG+FmFw8oO=7Hny&1D&>#25xCau)P{4_i-U29T|)0|{JE-EYDD;<@B>rUYUsm4mNjliBq27CF$4a|Q%uIZ)LSX#~z>6W2r+G6G+L7)7J
;X`@$BcK${n<FZk5V{G@3W+Qt&ViLrd6K1VxTG|#jU_`?EfBpZZ9W)KF$lTZlFJh%j3pptv$eAH6-J++Ep#Ea)Tqt&7j(kPylsP;^T^<su<h0Bz|RZd307o6
8nk6nPv)jx07KoXCyl=GUR70vFI}n4R)h-=fR(yLx9Nx>_urY8FqMq+P&JsEb5|Fo^;=*CbiWOKh^w;-N(Hr^m5Xj;J0>*gmt50{a}w(}d*PF>?*EyA3@y5A
S3GnsJjOynMUXvvTwvWM+iCYUib6|?7P;TG9|sfmfpAh<IMrejcWfSlH4h2FltDaG`gk+A*5x}io#MBkDVbwRV8ig5qhV29_3Wjvh2yRhUUtWt?!d6W5aGjZ
Sa(}UHRp~9*RNyRd^4ZBXwMheUXtRCsPq9$J<BmFwxH!_p9#DXiRG4d!uoNOak>$zX_*$P@lePW2haQ4Fa0Vnat5}JK0m?@gP&X<`j}wOG5i&Eu_IH1bq1#h
|Moc3lQRBmt|)NEl9d$irvN!U1q73nI4TZKhmv7n*u|wXS$%;!1HG)*2%wW07Zx?-I^$M>gwZo?$%&mI%twsBDG=@vQqTL%6C6LKNh-9IlzUqa+c{$aZU83;
?Bp=)h2L=v5`P{7_Kq6o)JS3YB|&v52UCKYGX8%03Zj2YJn8Of`;XuqidglBK)$8#AXE#YHm_w<2uSZJp}0YR&QiIT?iTaQtVoj5Fp>B{>=<VtttJ0%^Eokb
p+31=Z#+Z1XdYw{*e;YyqKdX`f2*m$VJ4B8R!vUsEB8zr9{`q(lF%720#v((tW`MSf9NMAOd3jwdk2EE`@lMR<FS%oONlxb^Rc4WQLL~uB@w^AG96I|X-|{a
(EmLb_<!y#I~@Xp_I{u}m$RJbBW?A@LcSswN=XS0l0S9B%xaX*zGrntLw6z2N=ROa3j@k-S-SS1P(WR79oFFRmJFtn$^R9mUF1H0@X}Ba@^j8+wgks%dW?E<
WKuWmDm_|`y+0>!PCNJ`%3ysJzJ~@XvR|d`L{7$(RtoJcCxqQQRi-ZZlj1Kn15&=6s;?MHSn{wP!g<mLy$1OTLN?vd{iDQI74Fk;nU@#k@cuH?8dPNoMywQ2
&C<Hgam)gp^bs|_4%|azttd%X(oVDww;cnZ6S|Jc-GxCc5&=x3_bj>P;RZP%N8B?k(^R2+Khc%BsiO9}{(txtgfM)xnJZAgSRj>r6glB(jQ%X@@0V~`RufWh
X(-&(zr!M~hl{5$h^#bXw;|hj7{UqZA(Fk(qJ)mN+(`4qduK<J2CmaWQnCqMBfGD+-Pws~LKy^tEC<fmASm0(DhAf{OE2EWB_Kh!?>h7hj71x&qS0o3VVq;K
X)T>046i>1L2q=Qlx^*lrP`5zh{@1ULygQY^Zb02(-e3+#y~mb&XWUz+i5+&m`%^#<*!D1o|+_`W}X~m1o>|u_(eKQmz_Tt)v1E}D{-%QP1PIdYKG6TO<qe9
vGVpyYEl_KwvX)@qmh1S)M+h(+c0X~WSwuFg3YLyim<WLrT`f*Sh!C=B!vHtxO^(ds=h83Qbe{MT)}I%>_WVI)&1QfL(6Exu)-WQ)YUMi-nV?`NioSJNzk4d
hSBI4-ITLzD+flBh+ElR?&(a*_e`=}?$0Wh99LeO*L?X&;u4tPRAlGx_#*>j03<dXaG}FEve;4Ul4W5IONn>RRKxhG#RMSJ4JW<e;etS5?@DIP01`;pr9CH5
y~K^Re!yzH{h*~sD^UNzH6qag=pcvr{(xnAMTV2w$+W?!@FyCcr`C0QwT7{|B2ao7lGb$_PTBC;qS6P2)JRw{Svd67s;3wONc9{6fntANrS|1y*XnW_hDyzH
*-Ws>%W%{?WL}_^*rssddIhTvzQ%cE_-M|xZcbKNZ)BK&(mC@JQt&o|bPeYf%DJ}nXaA@m<NPJ-7nXk;PQ^3ss8O^aXEe{K1@IR&D7U~@f(bp*4~!;-YZyIF
XUbP~2*vwKw>ETJ5IQT`O(%0_h5yhu??DmQF#N<Qx3COY=~{z;zaKd^I4(A1rgHaj(_!p7#&baHmucC`D-PbDJXLZ^ZwZ!8Bnn4S74XTxvP!a2J%ZtZk@SF;
g&_v#Q<%v#Pgn_0uZ=y2J*tpLx$O$iKz=d_4(3l``V{oCaru1qH^uoiAP(sf|0y?qHMYy&$%r~`uW%g@RU^>L2v$<n$HB<Xvb#fFZKft7JX<zMgWJH}5FLiO
VzMP63tOn}M68QlE!nb#!Ap8H_sn->pZAaQc`V*&sWl@W&{AGf^Z%|avF;8?#7s{qn)r*s921V^#C96kn<|8L>nRfqKgwBzhG!1ZsZ6ppDB@6sddS;LY@L(`
Lu;kmTRSa}(gxL|@uR%XaH>O~Cdy!NO<prgfYqgi9;`P|T{VCPY$I57Y4T5p3QM{Vi;|7&mfdVCi9QRlwkb)y&NXQ@!=Bv6d}%KH5m)p7>PSSO?a;$K6G@Z-
-Q8hle)p$-<3rCMfAoU9cdjc}<{FQ7xeb_j-7ysTY(i2|wFUWydRqSM7Y?o?Q0bw}#&p5vBJJ|(fuk#98|4YT`x~hI#a9l#y=x4RAv}DH`sZwilYTpN;f}4O
HY!sm?K0<9#a}0MJH5xcGXMJl(21ks&0u<<UU-wHJBIame0V9X?IBGaMppjZ`Xylq_Eco$x(`}I8cF5vX9~{lsw212w$N_q9A80HFH55yVnp5NLo5eG1_s&D
dkz+b>?-X%vfbR!k0rXw>&?4`4@cXmp1KJz<{Ipa1WZj-Frf5U?Ob64{DHp~qlK#ZSZ0V#6Cv>MDJa#HOEWxml=3aIsEh?p`gi3)t%{l(_^TX!6(L@gbRb;D
Vsm`rI}RQKK+9lc*0wg$Z-K)?6W#PKM1NVL#MXC=x8$!SD&x2xt>2>%)FSOFrrH3DsMBwOH@q|}^XrhRI`%sY4^#ailDeEH*!O7a{0r78by&(mus(TRSBc$-
Q$%=D>_3L88UyRE`DV{8ivxiYpaS_U)=yeD%~srt<2Z(<Z4291SDIZ**TTju7_sG?{_Ajp3=yGU67htnrjnk!i5!>Sg~C+4_Wrh_)MPJ%EtNG#S8c5W&wTC~
=a^NW*X@D&IkeA0hO51paar1uMGEv%n?W-HQo+e4tXXU%_4q;Dc%MTR-yQQ#w%)y<uwZnP1y}umL{E;bHW2|c4&mW$u;2{rwc3w{ENZMB&tJ+kI`LicM)V0?
x<26m-LgjE@f$ztQbgbOVFdvvR|#MzP9n+Rnak2-#wiN>ykd<4-bZm9`ZF;d!%Dy{`#gzm(VTn$|IOk^G-au2IyESJ0EAliAzq5^cTBgN8*jYywcUIA9wrmM
ov-~G$^x9Dc$cHe0;WmY5?6dy4V^rK7#*Alkp+-x`|E(mC)DIOF=434Q$O<i-!gH08SD}RE{sMNFE%l=No($(Ova863;HG5i(#qKFhGTbe`8LdB~99<z+Yr8
oe9GSoHzv>z|1LcB|Q-T(z%2_VTkh&!0;W?TIJxZA;4e9>+-}`CR{CJq>ix!Phd++(<p~#XIwwX;yLg?t$;%jbgg%ix9y%Q^k~qmO*B@J>GcGBf~Xxj6|gMg
W$tf#%p5D)VFgw+`3oZ4Vx^~!>nx<u%uk_#ko+}GfYa>RY*4|c;v5DZiLFY-KtS}*WS>J*SC_}gSjRv4w4SIOXJ0@Iq)}YW6qc4G6rbRzgcl3f3Goz<(Mq|#
x}n4<L9@GSmgKVh(LX{SX58Z{MWQr(f>f7d?KA&%w;u`zd|eRXD&Zi%1i?a|pq@KRkHQQAi&a-%)xg578j<N@+E_nMWP_3&MiH=kkCW-F3q;`Mumi96J!tcP
;o3Y11dDzBR))Or!z0!xt~_Sws^c7=#r<mW!P-pS^xScXMdrXF>qi-z%Ojs<7F6k>-Q6}1tf@lUi@V2jR25Ci)UhX~I5JrG$a61QYeO7X3uf(<6R5BG=F#RR
GFGA#bGxzOG9M;sd;U@alQd3R&nYAO3$;D>(v4jjic!=^n6l7Xn1N$$l_V5y>ZdmzQPkhx=h9rmiA5-6;mXEi03F@M^n9<dB+bMiVf!Zv7}LH6Z{?frt;gd+
WG8F(pm`^iw@`N7M>`oPnBwd|)SqKIt_2BcR&Ry~*8cuK_En8E8L+f}C!Cp#b4!ON8O=A6bkVoLhTx6kZtl|-VW>o})*F1$-$>*xt9c1PfZ|M(>KBnZ_tX!O
8pVBU8-}<zdFuCM!#JeJt81SQCD9BpcV*gxBc+6M4eB2M+L1+D{BHC95;Xi&g<~W2sl*UKa6at_gj=t8FBQB=E2XgfX<WB>`4l?A)rjp*gbn18m0!zuy*sFb
2zz5^otUVi>744bZYe%Bf6YWK0e-eC5S7}+XwQ|NVOvtTdgd|g!NBQs`*Mb>eL?mV$#?rCJFXPT(vt)-qbDrFRmd^LlDmDfr&iH?O;O``zFvBwyt_~Va(P}K
Mq3SErBt5tPMk9arnr-}op>tLDxRTiGtVz)dkDGLfVVGr`vbQw-^X29<ZQ6!MDjsk1!#1y{CT0Xub&D`FaJE(-9JB{8i4n+)t|+?l-i<gL^-*&_JVfH%y`~o
tQY#$B{VcG6##VjQ?8Ic&{?=uX?y2B=%XBBQZTQE0W)ztMqO!g&F23$)-UjaN0?SJ<SwGoV$4#desRuy)giP1g3i)7X&!-(;#<k7d+izikfGu3K>55AOBg=0
xM}C<1h_zN0n2X2uBp;so!wgfg#YYDA2{7URSzeE?q##5G$#8F9Jnn~u9Gj=IgUl__n1$6TJpPnuC<<M)d>v{CS1vc;r0&2U>500Nxe?a1PXP+`wOx&7Ad%x
hY#5eEU{qDwWQMSL9XKUSGTpOEYvxn{e0qA9njMe=uQthqX-f$NxLI7`K=_A@xzOk;qsEwg8YqG*IS8vKn}65T!GE5*KCCPb`%?VlJ+1BNp|RhfS{Y!G5hBK
l&AI!AKr2fCX~FqyFB9w;5HJqEv%G!=r;KbctMerGUQ(y6g)5c^0$8WbD0q>Y6W{!cJ(Xfex8QPf-at=1>13Jskh0)eFo8hUdLjFAl$=t&bH?l>c-*;N+uU)
??Z7(U;9o^9>Vcy`y=^ZhO4W;8f8TD-qvCmG6WKlsh$QQI$0Ym)#kHTvK(t&t_~vs(|*i_o9<5Cv%HfB;zBfxpIoTVI?_5NF?3jtsw8`awP?|M<%W85ZAH1l
_D|aT^fK}VO6x)^j~XDAj`j5`Lcmxh=HF7jb2WrLUm=|kE{!)DJghAn9h5u&fy~B`*3xZhdNOyv!ap*-6b=X5iZ<F~2rM6;xPk`Vs*vGK9%wb3=Z5-8AZwz-
<M@)xJr?i&YVk)1uG(_W&&8Bm0BGBeL9?egw`&vyfVks4wpLKW)*JG&+A7s^R2>M)!bfjbdPE-H3?y!udPMFwfQ<dTP#2s!Wl=b~#a1!ld81gVMv}3VAgsRT
A*Ymf!QDn5nyMS^?Bl#}-}o_Nn~dLri-OLnY!F|4-YhV$U?l8KPIDz95H%Z@yUP%68cbB4v#RD_f-<X?nG5_M#hmzMV>IiUD=q4|r4WABGXBK%y4R(8v_Vy3
0wuH8iW@t=d_y@9RW_SS+*IVMM#v(erT$06$m<MGRNu<;9?_QHIDds(6}3hXrSjZ7xLfQ;2<j?32lq9Fo(kd=1xrnS6T?uZss@;5a9o}IG<KCIiaSB;)SC@)
hFYzsm<0KSq+4yCnNWU74N}6;+DdmOM|35Xx#AAGj{HcGM~%f1hKN;*SzKuh-~&b0!5k6NM9fOq5+snv&8T@uXp$1l@{C?_O15}JM$DhI*C){fFF7z{NAz7n
@36?B`#XE>rdE3Am{(q*u=djVRuLqP_rPJzPB=3{N{$1wWgV-!h`eKLPs|g(H3l{>@AoOd5)%Y?KZ>wlM+ys&lAqG?=?nG_&X#wUouDUP022G}8eQ2VQp{qF
zT{Jb1R&{TtX1`wF24+v=l5&!Je&mLp{OP%T2UZ9I-X{_ja>2R-!o=&ua@%pe#rvVKdgANt12!gq_A?wAnK?D0D#zMWLq(-=_8k5Kw@gKoc&hrGRVe>r}^ne
2gZAn4Oe_C+{79-D5H@}YiEcwfv7queK}_pvZa9_x*YlsYRxaAl0x~s#A3n$5JGf4j+2<r5#Ql{!ihbr-&ruUMA}UAkt`@iU(?tF*dPByVjFcOeW)-((;p$>
hcRRhW!WbH0~oC7Eq?!EplfoM{t<ILrK|9H!C0YNn^@1{Bz`KKO;UFC7|@#kVI^%~*Q(#)%guwn%;q1z$sMyDa*;q2$ZB{(92R-3{$8KFfd8Km0Y9W6(k&6s
`rhq>W;!G)w=mu&8S(@C61_jbS}Z7G|E!Y*p&(kmCHw;<94pSxMtWM3(wSA4R3j^7PU>Ge=JZz1Kv}U@aZ@C&|E+`+5q||B86nfEje4DY@)W<B(Pi3M#+~ch
LuWSWd!<i*S2qrkmNh1%&k&zTo9+hvAO2^h;$qk(6xW?YiGd!3QEx86MtNIVM^KdRg32j|2(mw`zY!6dyk`#8_j(!oIrI~vUL$%z$~(lS$j<&nV;=0ado-&Y
_j+%M#N3ohL`JtBqdz0s-Ol<gB|TRw>!|!-wU4TGSw904iQe`jsY|5sQLwmBm<e!L7u@rsJb#1WOfcFd@eG>lU(@yu*>^DsUARZq4^pZ)4@=U7Bva-d*2?Oc
9Xok_S;F3n7?3d~G~xseApla%pYJ!|=LlS42<TYoMWQ|!edgWkApS+!A(@qTS(Wl9d|L=Wmz)d=|MsgJJkE8u{dZfs5eL9@NYRlHT^;aop;~d>F9n2df9mtQ
fBW+G#7KScTpO5$%TiP|>!;Kkp0m8?`El;l^XT-_Y^+#{;gS@ri50l_mtcqiea54IR?j`bz*1aQ*pwSP9x@4Z2WGd5(JDX(rt~y2rJfHAkTVSrt|<u1Qfu)b
+#&z*Q4&)Oqy70v46KyABWW_v!OAUY2+z|uwM%GcJ9V3Xg%jn*&kQyffvA=0U9RI+vV#D14yE=Iy-d8b9Yqg}_qePqM7WLjtPbAgWR4yLlT6(p)uUp}|Gwhc
{4K>@FTR%e)$Uic;=ECD{4&NfdyB~GHwSZMq?hNNm(-!)6_D;r09pnGn9#~vzU0gE&N4m_H+QJ`;DMouA<XPvj@8W!M2`2TwT(Er7M}z=cF>d394-E&qdjBc
>!nN-3=F&V)Nz+ESOviwGU(17mg?mO<HTQ<C9y%h@r)l(Ob0j~4;oafmn~gz7ccL#gx03Z-kAmV0L=X-gk*DU`d?_Py;18n$YVC7WI!s`%j<xb(})(WE7m@$
6LzS660H*~>Xh1Yl}EALW;rW9l&hTfz4z@*AaQq%tu>~*wL%=sxPXe&H|W;2*kq-9qbj!TyCu)JjoBIGHsz-}!ZLs?SN#clENSsA`DtO8{Cm7?zve}ZfOpz!
&CrD_pqmv;%fG)@Af8WrP&a5co`FjGSp3{YO^?ef!Mk%4_hZo3{9}(R$?J~hw7Sd>kCef%*ooo<pZek5ngar6S6f>n*<T~PvUZj6&g2`bQXftd)c~m!7u^-_
yNHXOTZD?fT{BwiOH=pD+tw9(isqJ)bSeiA(k4us_qI|aclLV^y*YZ~=LLGzB}P_C+nU#eCdyzE4qr@3!H58c0&8b41?Z=LRJdKbINhfGg<sL&U>Pw1J1ixn
B_~*(zL|FCEi(r>10c)?$otAZA7wJG!frusC<ya0a)r-2&P<stEpYt~9cH$`DpOMf%qq7U6K=Q9Lc<Sg?2o}VU(d{rZ8#EiKXTQCFVSSTw2Mn!CWwK1eK3EP
0|(xU;*w=g|1uU@&>gOQ10hs=s-r}hD7^*(@7u8OMFrE((;E6cR0OjetLK5o*S@a!1~YZhGt)90qV#WZa0Chcy6GLma~2EmF`tV01C3InRY4AfGg`!(t!`9@
Z22>f)_~tQaYj6m==cq_n1|iaL~uif#0m%uT}SXQ(2d>k!jMpFn_dlV5Z-bMeB)e<@p%hw<{Rv?x_<-O@9%3Kt0%%$2K{^;=iXXcPGxh*@|Yqfr@jx;{^hGG
h@gj-*E0-3uTHoi1ZMmeJmj3(wfatV)N5eUiddRj!Pu&)I$O#qDtyc6jdGN-mtz5QG<h8PxM<9tQ{L<QYpEr6=u9>~uA*?WDz6oNbffD;fn0-L08p6EQ4}qw
sXi7kh>miMaEWFc#DbZ1-)UjKqJyF-MGpL27Ut3!w|%>5k2))JT!hUMaG8wSFqvZ7N@RQRZ54t6CpoO8(Cphu0=`=F-@*nKt!`-pHO=mgtW@ZTWqCb@NvtxI
VKn&+-n{_lY{By?5+XFRU}W!{824q*UO|sch$`QsfLX)mL2Q`5sa80Mqz;~t##KMleK#VB*Xp;bPOdP|o;PQ$QpbDSEx8>}BI!(8d{0nrfKAcZXQ4+UlPK+3
`<kSB8hD(wmH<`!<nae(#2N)!*yU0RCB559*T*EdX<}4K$%I|XlYeVW>OMd8dO`}ri9B|r;LTwCTnOIIc;@k8*RW%u?#j$W8!1CA;A89(W~Gj}Ay60w#9<iN
3bt?Mez!-(Y>t+?%H#fhiq+SE-g2jo;#)<IECcuM(s=~tRs@Kwx|avclnL`ft?K>oRH3f|^LeZmN5EUWb$M}e5htK-k7CF-j5KUvmp@%--?4OG?2>|oA^%d7
-}-s@hu3{-`)1R<8J=r7GiEo<5^uN1ss%@j+8My9vFEXp_H^{<KHzai%=osBrY})EsoCn0^S*1lH`N=ZCE>XhhEXS(3EI=O!&nyUYez<hS}nbif508#m~MO>
+0HZ_IKb(5dj@dLz`s8xqXPv$o3PC!5T=$^mW4J!{mbI*l3~LL78rGQS|<-@td~7UVIAC-?A57|Ge#SQ;YzR06t9wk3w;PB5$LctF#YzRzLlx&zM$)j>5+XE
NQKxJnoh!SGqg@Xz%Vl_khVrGD)e*YUI6e)59hw4nls5(Dexy^mZFQd>n?cg{KaYBdedN@;~6d8z8_Ec)99*6oqgIrd`vPEoJjGaXZZRw6_Hok-Lw#0fig}f
;D7ezPEpnL6|fG0|F#kd+RC$A>sB<Kr%F^g<5D&Nr_k%jK}a!pSfqoVY^+VXu26de?%zzGFtNNPS0-R9tJ<Sjs7FPwFTQc|Er7^$|BW)Ti;6zk*ruT$sFdM-
)W*F$$L$s{6gy-?)#b+iN{}%emLHmF=R0?AGc&owGF4>dCK~&|4v-dt^5*=GB+2qa#o3B##a(pawF^e&>2{w&8EdG94-CeSzz{}N0rAj(8fWK1W<8JeBzUox
hmSjnf~SiGt+L#neID2nscW{lo8#P?AZxzs6xk41c!jzHgJ2HS(XY&6Zn#9QB~fm6SsGs`-q;>QffqsoXWJ$5E+pY4^#AdYr*zsgW9&2CY7X^4Fcy{t(+$gk
H_Suxxn*AL^J1_AcIlMF8SxTZsPv;{Wyh*REE1~KamD_lT)OH5I~v(^c?Wb<%6Oi;EWcWU{|O~o>+Vhgw|_<>$+gA-3JdRniTjGZb7}+A$hl$pzcHH?$ePo>
0tv%H>%H6_YwN+jF3P!8FlHv-utu&a6<hgI!|!*}5m}ALAx0H)qRe|a+SM%EMBh^znZg@@l!ju6F%z|Y%mCUg;*2#&50eai1QfErF)Tnd@Yol0X&T}R*HEC6
AOz4P8066w^5b#akmp1}m$Bpe?cRQeRWa4kes`yfhUBT4sbdIsS~1P{7`5_pX5n>5p_pLHH}?pqP2s*~ohk`_8HCGA3?giw@P=ju$oiigA?qlk^E)N>{lUQl
*|06(o_<OATUY3w8%5(CBy(I)b)Dh=&sj(JD%WCO&=~SyWf>kDWkd^BEFmWV`lI+HmD3C+O`d0@rKwPTTh#T8+}X9f?TvAB;Ek)?&@Hg85~sRTS~c%QIUCc8
Lo;jMh$P)@!OJ=yhG67tyLL;?=j@t0f-MF{afvbFTfQ-TDvLN+mRT5Pm-F2UurK(cN))XRxz*Fko(cGaz__?L8{`?`s%^1C!yL5jfjQA`fBuGogjNqnLiiR&
gkl~?ErFe&vELB6h4n`kn|z+yq%7>fv0V22r8wnhN}43i9|lRiB%Ff=kvZE+@uJ_KGZc%JdMuq4*Chep*YJ6<`*xu^6R!-9>cS)1reQC-q9?<AC~}<#Uvy@F
^e2(<el6|22n<)>BIZs{6?h1Ns<0OZHR87Z*o75{AKV^QztQ$(CwA<C)$x@I;d>C3ja4M%-1mqtF|z9;+3_Wexp+~2pe*w@wyq)|AXgbmxLKNaGW^8~yN3Qv
Z$~r>Z2`L&7I_QWyeH*dV3jOzz`p7HlQXLN@D8r2$`UY;ignc3=NxrE1tE~L9vN+v0xJdSnE=+o<VdWu<-ayc7l_P_NU9lC>{$M<uttL{j=V6~mVEiO><SV%
@%T2n|7FhD$YvGhb>y9>4{4fkRmvDj;u68G%v`=32O(2t85Tr?WtKH;nmcW;(w0R<#y0s4Ni!0}$sw+VmbLVf4}RAMvf#<g^v=AGS9<aQ%$N&v(Ic#NcM$=}
?u59AFyB`u^B^t{Db(@hU`T@gDenN$Q)aq3d1Y`zDUv0Gar5aA@iZB0>5H0Ky{F-Zi&7P&b4o3uG(MmD03*oN*T*>K!eJb`t{0Ya?l`7<M%xa9HJv5M&FtM`
Gn0pFE$i%~*Q7ikabNlF1icT`MnBHiViL+93f^o#AJH0dXxO3uV`4om;<C(>>mJblqOg50LU1Qt&o>kqnlQS;NNcv~XxjyAMT357xG6rWx!K~XXm%Z=E%FnD
Aa$i>CQOcf39&N5s{VE;1}?`<(Zex-7@UOu9@1+4oIfWsl4_=B_uN4G-5&R7o3p<>Y$*t_p8xmT(1?v_+G%&KtsI6-?e<)P>MQS>M{(!^<Gt)+Qx!O*-}Oek
xcs(qUxOzQrDVFb{He)zqno>(PgM6#kqit5?(OKry6eW1>goA1;~*u+A_5nq_mdF6dm0G}`8^08XYJz$uy>Llax_kp9x+g*nWY8+u6w6#Yn6<?PYP7Q_=56D
MymX|q}KL}%fp=Yvn%9czI(G)y>!C|?^#}<7j)ayMl1gC>#59;Xr_b8=}7qr%5DZ5=4jzWKc$BTS3`NIBJ}YQ?axUH{kEG7dP4I#mGg!r?6(-<R)pHf3>PXW
`fX%7{z!*R&y3CZOp(r^%?h?33I!wgmAU)M2}73|Qsk@j`x7x_<7g+%gna^k_eJ(2r%ukUM<6i}E5+v1=N{SCm}x-m971CNoClb?&PSdtMOJURQ8O1G#;07r
Y#Q?<qB__34nlnbj>}_E&3?+!-MC9;5ZDG8^sE&Lf~+5w<F)7<Hj3|c%@$7z+P-L@GqunYaa1Za9NuW*EFO5-H*m7keW^2hraab&i6&E*F=`O0#usoAH{<<#
u$=&|ao(Q=`Ui$%^e+S6$yXrtFjE}9>QHJy@GE%H)Z6ZIW&<W3GnX1{MpGwiis7Ng#F#2qMNzN8DR!yv=oVx=hXmya&v1<)LvLHmim-BT|9$7~aL%dRp@eGu
Jn$QmDc~AYoTcsUKqiZZ>q-zwnZpX9V*D$Vy_mHeF8K-xdLT?z<OM=Cr2D_htEaMgQLcaw!GB;<5-ZgaVV*OX0!Jc%kFK|e;F0m4p8$Vd^jaj0YvMx5vj4e8
oO0Sal*IE(x9cP7<^abKfS#Su;-hS@u{;=60WGQf9Q1His)X>ueHC|8q)H!#;G(;U#|Xsq`S+(eYMe{>Qg~b6dFq%-mrr=Af|TNl_njtYm2E8eD*F|mY?LN+
HKuBoDVqen94#eHq4!0E(f}+=1BFF(v#h44JDGVzPaiwqvQ3MvS181><N(4)e@$M;slkOsc;PGAm6%ljnIW|Xte@!DrJ*<hG0s20xwwNjSmZ>6R>OxH-$DC!
=IZjb-J<|FpvzIQPHe92k;a^u9^la03l&A&GZmwZ`eJ<aR&KBQ5+8W)x0bxe_nA8jd^Afrq1aRd@4=xw7$Qfd@mhp3*QyG1=9p+Sg_!E=on|!rGCqa@fR+I}
P)H>)1?~Fo9W{B8`sJzfyrbCRP|Si!xsw{|npLNMZ_7Hf<WTCw_tqTG!0GlAvRGa#vvc_t<)uUE{+qC)omRdQ@>u+w3J=blDcIg1DTT#-xZ0XBQ7R;It9?=n
f5;2<VzPT^>NI;4slcmL+c{s~j;kb(oqwXpxYO;hIGf9^aL0@Ljtx_6zsTx!3NVor5>nN44BzByy?n8@B!=>R@AlbCuX$?D_aN&X)cPEUxkj(fNUnIU9@nMt
GCuqKol*F{g_!CB_lk1OzlV(8J(V=#tNX~r|1FiL{*Iax?f4`0wxo<;Bo!xt-9kIMu%3U}VYuwpM0JV-%DEmQlZPQP1GVLya5XaoplC>)Fd5r(V-lIb!B{J+
eJuj~2uQu!-iB`F=P@^ZAa2|6AYcuScyQGe^Lb)efEuzU&?k34ySUt;Y+KM-<Xh#0Cf`r-FeY-&*a?;h2Y1{?B>V0jlKA0V3!kVfxG`=i2T)!?GXCf8o}uCU
?A!i3(Zzu>pEozIO^_xZ5@$^$slpk9S73JZB~!s~##3h+WJ)ZoLjRLWi0SEi(-laYhbBvFD4(jD<(g-ATA>rzz{5xQ|8{@o0P}C7U7VqSl_6I;6g4W6u{AvP
T$S5!6hiZyqv;2{n$z~^?t_GC?M5K8?v@W^S61zTGYJ2klZf;$bibW&Ih+jvK|~;yzdN@sAF|9{5a(mZi%#b2*~1F*dp!H=$VGg01JLyCzygp)+(u=+t3g^{
#1YiHMlg3zG-(~9veX?82+{OTqAz^72kDu+aH}$(0FpI#MD1DkP7$lR8Gct6Fmz<}UqD5^#H4_piMf4odpSm7@oxr|JK=SN9ddjFBS~FH<R!*Q$vlta9>;Zj
E17J{8zJv9kvX?7?2kOT7tveT$7o1_-0KnEF~8N2(wK>ljav@lYkhq?iPcrx!M{&b<p^RZbhZ-_*_f)=ZRiNx6hwj9fRIez3&C}uSsEc4cI>{}UpH&5%Gfm}
2wvaonB6>{)`Gz#PKdu&BBK7juF$v>_A;Oe6M1EEK04QjH4se*!xH|);A-%~wU&*`jN+97r=d)!X~l9+!r%yW+QZYPMymP~aJ&zGq~m6tU{uqzf>*(Jf1l*m
mUnMhy1;V+zX>trdKrUdw;^Pva?RtfJVE#qlfg)iShbXnoB$PRciqDacRD2hK-=~Qjp9zOC${WSP5Cfa3LI2EDj!-qnJu{5?9Z&y7vz738uz~}jwq(@-m|d&
4Y`Ei)yo$FI>$e0>gq6^sR5)EE)fZdwPZ+KV;ySuHq=PHNsWBQzfx^FPnI&~19yE7K)BKBeASHh_XNwqu%Wli_5OV@jnWwxgPNosXl4WWR|=OY>ZkC5L5Q46
bbcokZVIZ*`a><-!gRLK+XstnQPXBiQJ8!dz}6z$wkz)^5M&?grII(R2yxVtz+Ww-;RdaKjADz`p3<8Mfq8^nICAPqbTl0%CWcQ=kcTJZvx)D==zpoGAa=l}
%OH=ww$KZOJk4$t7CWFL#7y&Sx1nOp#bMy&lTrXf=_jxaXmw&O%Dc^h-`2G?y{9l)p;yN|2K;6^_N@-6Cvy#!6j=*PRgMlC$<b1;;)yS+I~$gqlRH*)PrxeT
I=BrYceGHf`);3jl#N$F?5BrlcbVK%WEcnJC!jAg%Q7VonDMFs+V<1VV09Gq6-4!y1<MQ208^1lVy!%}_|eh_a`K5WJpiRp(8^f7^j~^R(^y`@q38<d+u(wC
`W`4j!WAl?n^IQ6sx1Z)Nrf{tC%xV++#!|bL-&8y*UBcC7{_(*nyB$OgmvIio!R`*4L$;{bx28Ba5Z>7t0I|9L5NgBtQ6-g|C&4zItbR$60?~R`IX~C_uMLK
%1-%Rz#^3slYMRB5y6>dIj3_P5C;M%;tdvO2(aSWj*fSZ!cUld6ThGG5o`;%j@ti#YUrUO2~B<nMZ`q@&f`if3*P|uz_oUo>&aUP)(-~TM%oYRJRT*5stNt4
+RXoxN6fb3FGq$aTCk}9y~1!dBXbW@G)Ej0NtLWoYNP552sH!F9_5-B@$5}wf*d)$2eY-NqNSW_&tN^^pap0_K;2n#i?lCwaW8$2!E2JU@oy6NI@h=&)QH!v
4Ub<<dQMyMPjZR^xAY#7NBly6p`=j53H|-I-14V{rPc$efxi!s@frUgGcW7u#BY(nilL5vF{~!bNezB!=&;1L`s{uK-6PAD@or5}p)?tDXf@jL<(!C&VQ{1O
cu@B1`0`Lp*cYv!bS*x&Whfn-!K-#Mbq=~z9)4`z6UWo4RS!kWd0m?$;!|##^&&RxOd|z3vl{rhM!iyjAktnXd}6;52@G-uUi1XGo%%@0)aOH*GdVoB#dNTR
-8Vc0d!*f7>Nk?d?<LXSDV5Z%7T^+uJC|ZuL81i$0lp4?jK3S<dZy6{?p=CzJ6JfI3AfR_HTI6epXEc-G6(<Ex!08aKdHxKsrv9T(l2@kvhO9iE;{~{J=*K`
-K!P}Y`Li+RSUNxwt{ba!8zpyk#B7JU+n(iQu6lu(hddl#Z~4iGeE<hcSXKPLXD+d*OSxt{<OBh%M_LedfM~U-jEXe@6frJjcCude1i{qT&?IQ(bj?#wqtNA
e5XLwDCs}W`t&`u8icwlF{HG0AW$k<8j*K+xYz{1#!&W?-`%`S?O2<4V^!RH@d;L&mT)j(YRPXcFj)w!t^NS^FE+GuUzhQ66CD~L>M1aB<HSCQq)B8bN%GGo
2S1J>0VLTaV-krgUgeP$>Sahdq6fsvc^tOA>-?A`ZSZn)kzcy$9>n%JgKW_Iho9o89pv+XvFWPM%w_5_j?#ZxS&G;bW>v-EdWqn4=b|z{S)rms_>4SuQaa5c
e|k>L$%MtbTlktyU>39hvBsV?DslG4*QSbzIk)0_DL5wrZ5Xvn^tC5P4)e)!Bt6Z86Gl4l0m__No|2>LC1MUr5BhHGd{M$&`W-{=*g)kO5gn*d&pVlIsmc22
<Qmj+T*ksiE6Yr9cmuNpr1uW(&ZDZtoGVAxgn!)A$jz@P*PQAJ>{HPAW;J^MFe^lw)$4_syC@f?vtgjNKFaK@5zoY0laBvzTQ?up8oklD&{2p0M<5LsTSS#Z
MJo#Dj$Ti-qv*!K&na818k|a~?S$%i&55W)3V`U*s*>GJd+yS;sY?kp{_9aYC-Pxr1Fnn#LcJU^-sqpnJ@ckBy=?Xnf)*)c$iYr5ewxLZA(cQ%&rtu|!g34_
U(9{^wMNv)K>)M5oc)8bU-4%9SsgM=WsCZBz2o}%n7m|pgo1K$keER+vFtg<s9h2k1c?=#VDt_Sb)Sh}2K3lBGztuf8#mgBdX*)=4y0bPyCB})FEV5JfD?a5
Bn6`Fl;9Msqd-n*%>d2%*5)XvJ_1kRzI=Q&piMTe47*87E_}1Kwe9$wS<#t;w1~nuT>fV{JCE+5m6-3j>h$Fd4Aw0Wy_clk@~M`G^YO_Xat~$+i(&}UTIklg
^u)jb^?krqN_XL+O+>t>4I~R$RR>0|en!92mCz;Xbt*k&qE29(2-g`u`6EC%B)Eb8(rS)_2;Mnf=da~i<*1Pr#VI|5M<>;vUI2%v-d)RIkz-*c6WjhyTitq6
M?JFg+lH|Y$$n)HRC@fhe_#H#avjPNTza%EcRR70R%&)e8#X?8rtR^R4&@FsGfnAXWtbF(3A7GY3%MaZWM1pWVI5R!pQwO4Zm&m_R28r-D~KRvYBtxe!G&6w
jAiOw5cut5?zxIXEK9bfol<18V2IFSZTz|KIT~^E*-Y#4XUh1#)dhscs>#b$7QNPq2@tyN6*&~nnf)P^v5h&@Dd-I^v524vO@iF?QboWF157GtrGS%<v$DDv
!Nn4iAWorJXiU4;!k3CrOP)~$A((pEAVB_%KnM?37&VRJjvrjOoyIx0-Cap|*^MGhuwp7|`S^fnYibQz_}xWpZy|KVBeSlFVg0qdY2;7Yfg`@l07TBuo`ag<
-99Q^(JQc$`7|qBGC$uS#o}Eiv$lY+R6_i}LFp1Kch3>$%8G)QP&h#%kXrhm(LO0E<K4QxpjqR$2J7@(*B$Cwj&8}8&?xVgZ*uJ5XM;C=O3By<aSd57&MR4l
{sxb;eh(l8u9lyIc)^G#h(X20OzLrVRbI+52n$&(FV3D!0g|Uv^iKybG1XxmXs7?$<@I!F__Z$u$gS60fbFI}P;{_omfCV(@VW4FzY7;QyGCa|=+Z4~ze<B-
Vjix~hVXo&?Q}JPv5g0;T7qZ}G4a}Bfsq7UJEGtU;GW2nUxc_3nV|mzCrwZ#!<1~xQ&$~*<)7d>dF7wy96g9X4f0!S57qH|jpOrgqHozaYL`zlH3M%5aqbH2
D3wO>jfEv%JaCYe0w9%{!6tPyY;twxH^ETQRNSzH8tUi?A7*@>Yn}JSa-V0ZSra0;U-e&A=D9ipi@G_2sM~FXBJY-*$oY)}mU)4Axm6Zk9b!G8;mlTtA}Gvx
2W&U?Ej%JVeLvRmuICpWDf1y5&?BDb;yRY0@T?+eO4(;^itK)c2KTVqgCP&yxk*F!Y8RME3xwj%LFixOK}mjKGyuxNi{g81^KKo&{`%QN>sDV;_*mwS(t>6W
I@H(Jh`Ha<7gxNcQ*!I)pUbUOrF+;CrxDpAdI-rNB(}RuL%#2wIt<~&{rDS&DV%icVki^JsDJ96b8-zokAlf%02KICqI)^LR~G(y@mVVNEr_GW=`4iJiEwg)
@0&ZhCLMyMD6b~b>g$ED`}lDQsm~#9hv&`9x_IxGsptSHi#8)mUYFaSt0|^Q&kvB)amnj}3$Sz(U<P$K=Fja{HgZwi$z4B~6JTb^&w>TbQe_eAW6d*z6|z_=
eFf_|C2Jpx{rTzYj2^2z${4yzrrcSO@JvOjkhCQ*)x76dqvWb_^xIgi|6Pi~Ek!-twt9(KQFc@(jou>p#eVxVieBJgDNzzb_bmYKfg*aVIRBaR0D^<j1I-eL
uu{~7Z=8qhE-}WHe!N{7K$z{9rbBM1splD_fHw9z=7I&Dto$C8DL<(nC&=Xct*xf5zQku2(!ttlFDKIOKCQ%*bONNLBeW>%o(68+seU?E>fI^+-X1zW(*Mao
@)?1<&A~@OE?gDvF#M+exu<HXWS=a!w%`Hq+s0EI2YvjzBUYiXU>H)!w^iodNpVBrgveHB`R$zH;N0B-(Hp}P@wj+z^g*%zXUnjDY!UR&tg}^tde|^)(Fz&K
Z1<Unq%U;_wk^|5U=q>97uq5G!6_6yj%;UE1i@R$%&KBpZHg0#1Yza+e`!DNuZyajs53j2LU(!KeeOwq?5Vg*hPkEGsHurp2Z-cbb_ip{(x&%We7XS1BTns#
v56wnM*DUw&e_f~&aSrYv@_iWp0qklaOMDTTQk{`9zgOduN=KQU~oppWP3urrYB4VgN1dBi&02W7fA*+T@Z@Y2&mc;OY+QXL7OIT75b1Lr)0YYCw8yH8I5Ce
If)`epj!uW$u&+skj68lI{0G$#c&LeE4s@3Qq&yp(38IJ8oB+Lwcv0Is`zs_iQ|OiU;6kBJKJ09H?XLkyMv=-wB8!bVhNE$SUstdm(NCm6fCwckQkUoNaFb;
hN4>zwJ2WXMH7Zc@&4(0L;&VAt6@Q<3bwG6mwPg4Wt=`<8VTSIl#w7AnnYr*qL?sQP|v+~+m<y9msO6UY=4PksR8be>h&~Cyj3qq-`~)d$U2QXq8MgnlDBLA
9U})T?{LH+LM-{1_0hZJH?EMcIuy%JTpDqy`(kXLHlV4vG2g$NaZv7|%`*?&D7sA;9stO@iQU$Yca`@tcA(U!zXD}^_Xe|mvjwRb400K#-Pn`JT_Jx8ALg=4
@&aI$mREvoJpl(K#BHRpqVXwIU(C)E(?#K+@A7Vz?XcwZ&a;`giBFUq<mNGyNo{Z1qn2;>MrK=9!~KN}#aOme%g>e5*Kc4cZE;KU`>@-^5ziGi>&YOkQw6BY
OA=$!sZ5CK0%cP6h)@z;q4im7F4QV!G~>F%-y!*DNY?O2<y7La{!Py}b^)})E-YN>%o^s6;KK*Jrp;@P2(-udUsf9_nP5|rOKR@r9djL5a_|~0sa3Xc2Ji<;
lB3o*;Yebt>oSqL4_QE8H<HrmLJ8N<mx%|~9}=lyr&3Y1V~0BZUabd)JaQaw?IN|{_v+QylpVh+T@V-MYUq|>Jh{zjJu!epd96Qqq|>I@P<mhr=8?%t6w;^R
X4}RDRJj0C8H*pNi0pwjgH>Wm-YA@+wsVf+%8_vSF&d7E-8?ASDx%7fH@*V72>rr!-K++7e>iA{Gv%qd=WJ<?YQr=PnNw1wV5ank!h<gBgR~H26mL#MEKaam
!qwX1{l6gJw&k=)vm5Vz$%J>YrybC@1O@a0a>C#SY-WhKFyRNztC~LtqWi$q3G|m(P3nFcbf0!Ux<Tx|bb{p5-w*9PV|J8iT{H{$3XQI=guc-4Y$C$ZeXU-9
6ma(glC#o&--+XFg%%)B9HAGa#N;f!R8sU83WH)=u+ab=LLEz6mIgx;a6-EP4hCPD<5Tlf8R1f_jkOb7KBbrWFVz4e#c}@+Rd~q%f+k3CR~$f(oi-QMABVFv
Q{(Pv;UK!{6%tLXOyMYQlR@sk+gm2N)UHyWvZoiFh=O0%_fp+cvoQS)U}oD&Q{t^~ERn-MAPpcGy(|RMxSS8*1Y+qjbLZ?IpoKRoh<VFbLz~~gt{dAPKS!tw
r$QhuLB%p_y7N+p&NFHf5J=rYyg44>9t^=~aH?$}DhkW3QP+GVaY6x)7|}RX0~s}TJDQ8EWbz)u4#FE0rE%RK8NXgcvOG%T-!3O#>k8Pc<Hn4f_#zv~a#o5C
3<6@u<AmhATAbt+Gx6b=B&>9jMdaGmZK~s-ctjRVrDJ_fK`~EO#IiTD7wDG7MEn-S?)u`L_JJ+Cig^Uk=)<zQg<Om}Pe-kUxDNCct1jUY&iC*TJ~5I<^J<HW
vjVcf4<VxfDtg;6@|bgQr$6Sqr1x^5Xn3wm_$lk>oayimHxp$fiJGrZG3^r#MmkxJhrPd^8bT;`4#0}jcKjep17e6kI3^;@h^`_@!S(QRn#Av-K9{fWRjXUj
kH?+5*u80hEbOB4dN@@eS2hgZ*;2wQMzO<7xYg_pi^PVp(5{7rl8}b>@_wnboRxj)uS=yv0xHwbo8^u;#*7ZT<{O?|s=+Cj-3863=!naDSG5Eg+H3?h<2q>z
4^*{&RP;_~VKh5mpU`tXPArUB(t0Kcj<(<OtO}zPs%ztN;2ey_9xI91+3E~BG}xo2z#Ha=lYiqHdIv<$1nM*S8$hF=c@tT!kY_`BIQrl>nis5ptH!p=C<P6u
CgB-KuI^DLAdC_qhj4F*o>Q6+JN^z*m2D5D8;<b(`YjYUryr2MSf%ofszcu!ZZlk4rUX4j=!4-<rALXfY>*^x`1RB8sgLjx$a{C%xgrDfoqpyJm(b}eM#xsF
X#B5(Y%rn$s|!Remeqt>S;dl?5Uo1SH0_faCF@~dC_F+qyf>LVl@^ax&)XIYK(%jK>}?Qk7QsHpw1IqT7*=i0^qV?vZ?`}!Q7+gU4*Q6aM&uVK07AG<)Y_VK
b7R9Yt{Zy25V{C4`SMT3z6qW+y%BHp<Jg1Y&RICo;ipz}4MIbBX%d(-Q}*ue=6M>J^F<blm}8$bh;0-)TP@WD>swcEVI+exJ5Wl3iBY*}6i3{S7HrnARmMzK
uLlGQwm@80^nVh~)F9IZ-()YPD)WYghaeY~r;8n(%qKMfS7BjexpV|#@pXcbS@&9Ei<htK*_wJ1PzSRyl^V$t2=i$wERhDIaKE>xycZL#;L7SJ9-a6kPv0Vo
LHcCPhC!)za<boRhI$CCd5HKcN|c=HzcU!#M)sIdmqr4<T>MREG>cdx`4y4<vtw3^Q8H2~kfdV)RmR0oyVj5vc>QGzc<;ejUDT}@*TDUC!9d^pfq&YFl2C4b
PA(oPj5)S}JGgCn0}0ZJl$S9zVOP>CKJ3$>0!%8c1<1wax&yNQ!I1QbbIJ3802E8OXFSoW0BR$7r9n_na(QrzYemyEmu-%F5^tjM0x1d}sv}tyMEP)M0U}r>
+PA5t?o1#3vK(|_T<G&Iyjyun^^gK_tyt=ms|9f>2A7ZG6>i~Pd|PniIfcViE9d&<_*{^K6gQJvDVPg#UeL(mn#KI<IyT-wZAo0nl&%$pBRo_2bF|Rx!CyrA
P-)U}VV00u;_glwr+p>JX?rG-elHzqnbEmAi2{+Ul#C-?j{GkRo2i^=#|mO;{?&_BMok8%yTzK%tZrkFcJq8t`x*jlb@pjvm94t-MAK~+9xRWqvxs+0z9wov
>LwQLmE9gmbsoZRI?yYujHFp=aTGzmeHhE`4R)JZ3^5SCw#eG&!IKE_cZQVg97kk{p#Ni=P3!rngVMybT;tK2=$79|G5E>lsB?i!s%p1)a7c$4)JhcgP8oz%
Iw(tX<t{$Q43XIufi?OcovkZZNzhW8Os#;dPjr|pqbG>>(U|7j08fM3Ou`TXH^-TpzlTsP)Lg~p(JOsnVd&0q7LZCJzc0GE3Jv5ULfju}+%}DM!<EKtaEwr=
kjmyE0%z~dO99F%=0yaFf&sNVkF#5Pna>0hfh_m@&cuV;-<XQU*fy2HL-t9(WOwAp*<OaUmOz(1QB?=JcQv3!KY*Egqqnn$PK462fus~T?KQJSYvv*R0qHxc
+)9?TfsmrDhK^>ZKO+{_xGm-!bx^|A-#0`O!>y7=Uq9KXo2=)@@up-RDN|xV#zHBLYy--^q*6%Xik0Qt8&TuIyI;sV0+mALtU4fD*p+3x=?ubMYguNseNH6?
J#~6vEX2SMYM1zk&F%)DoJ+JPaR8Ohx@zN^@i9&upQ&a*&L1{Cl#DF@iS_tR^YqmN5)9B|qrNOJ3aQGXd?Hot{&>bSmj5!A^@V!*)IOnq@o&(Z7Hg5|<nF=U
Q)b{zBTdLd{-T+{T-lm4ptb@XumLy?t_<Tp$-|qQF$F&gP{ceCH<J;6(JoOMbF}=Xv!8Qj&Bcn%2{}8PNgwRqtaD};`65Ff?+;03h44gAlxJwnNyfTmfhlgt
;fL)~=eH)JZv%2<exBTiF0TxWf<i@0TowM(P{b>Ga-fIWfQNK9z@#LX0*xYUx@CL-G4y<?X?vR7{41$(wU{_eq;Z!76II%_RG#bV`ZzMylPEoSX;k0^$uzJ<
3#7|NM)zM>d4B>*kUX)mNX&-P+NfS87%hqCW@71n`5VIY^)xya%$F!W42c7x;>iLB)eq6t*Ck0Jgx_+^n!%?60c6e8-ENV`{)$m(5k6<M83Fl4Sx((Drw4aN
R5+nsY77wqMb@b1;1vP_ZTrbM%pN0o<8XPhjNYGGX-enQ1sl?Dy`;BITW%~2r9Zwyg6(&9U&>vj<C#*DuY>V@wm>rwC`EU8Cnvo%+7UORGvJ|@cei9Cr&K(~
tPUqF){5E*>h)c2+0XrlezN+%YBSR>sm6X^cUw^O`o4zV->2#|dyK~(MB*8G#dQ2r2h2T9A3!$WAd*|m?U_mVf-OWWdlMKX7XL?>-r5!!{!cY4R|7^N%XGF4
62TE^n=_&>`GWzgML$tCk*ripoSJEj%5X2|G0F|OD2RLI`<==py4}568^^p~%I7G-#4bJSs^?K9**<c~oBNiLzO5)`{0xJl#Mjg5FVLzP)LKgm>u+FL@#EKn
@-Cl~t`kuG*wo;jE{R$KxM^+rx$;R7j93;y7ytL?L*NcOMPNh31gO*?6K+DyQcosgg8Cfy`8qy6!VX1fCx@w+Qz5KH(&vf&-bdOGl|&*>lQNi1&DaBY#jl<N
JHZ%m=h>X{<+v*#;6uEl-?K9j<{w>n@tZIVAe=l${WJ5A9OaSQ*oAteRqzlA$KM9dsaB#G_XzoWe>S{B+#_zw1c%oYO0QB&Y;jSQ#V~i;lDYaCUEBoo{s54$
yhoji(bj8*5vRUVx-x!+XE_{ATELIJPJ96v%n(-HROKXh?)?XLhzt|TrO5u>Vxb)ui~|*>MbhmS4Ts2v(ZU7dEZql2sQ3T2EG`hl1w};5%hmb3UP=x0-*?|W
4pLxMf!!tyO+XAgD8a<Ccp;UuZBVy-nkFY|l=Sovk2;@ln1xCP0!%0O4@rF<<Zg}f@qbVKIy8G$6O#3WRr>C~=vOmTIlxj`{gZ_Zt(ZOuX11-?MFf$#9sk&S
EH>f>^K$Q_h-WlR1rd_y^H!DA&0aWV59Vf(;PI(093y$+BCo=6Z&PtQR=%&DCja=^A9QvW_+lSX*NRap87)g20(KhN2blOXvSHBf0BM;eR$~G7gok@;@@fMh
E8`!;)=eVb`R|Rkv-~6g{nd*WpWKkK43Htc9Ldaovf{n@%0KYZ-c;b$EA@AG&SRv_7&ZP0GN6~Rp5Qtl?1eVDGsEkT@t)WE+W;aWR+ah{3<`A%x?dh^A`cvT
#lgB>#wQhAq8A|Y0h=hEyLb|i!nq?#KWXl3AVI3;(k#Jog0FFkw5>oYlE|;%sZN-uQ|HXyP}qm<5#giLIaCxArWyzVC-{PoPrXam-6j}tewNtx%m`;|S~9xg
%~qRE)ak_aQq*r%S76{sI8aPcm%Wp3Debz4Z}&?cP*KvZ`+*u6#9OSO?Hi-)Km{?5%cX6~^p2w&(N#?hf$zP2>p(OToI8v2f!FLz)33aVnafHB8Ey1lV0jm+
9?gAkKKej5Po{vNJnGM@x1XF!qVKXzToP9+FeQh%q)euYng==0G4!IPCoeWTGAQeWP}Mdg^B>hTqtpo*!F}KMM7wrW7ce!CTC#|opHc}F`^a;B9GFQMlZ`t#
Rx*^0YQJu6@ScaH>2XECv79ar-k_ncb=2O7x_o+!FztBoPg49%=>l-!!x+0L#Ws5oMiRSW$+j=?^xJnf+?q!)H@OLcF$G#>dbS~pULK=7c$wZ2kdZQ|JA1i<
%bQ?Z54~LWf)uRQGeIoFGPHxgff2uVF^z$`eausWK&AomPAxA{h#~ECqk;pzMc4I@gQOYS4w9QsbifUbk@1uwf;3c|#jlG1j#lZJl1BcPl0R2m5ztdBWVQ}9
ToD%zi{(5DFjNE2!)OoUJWq($@%BI`Lh?lL53nz!3W>|PHZ(Nc=))oy49&NU;)beaHI3A?=*uwpAXYtPO4}$`Kh7GS!fFIK#YZMk!WOZj_8RBOXp>>XWV0h3
5)~RHD|z5uuLv*`6NHxBO|4M31@gkCf#@n#k1!!y8WF{;)AMM}^*}g*UL^!`eY&R<Frq};Fl3q<xMIGqJN^@dxL%&RRWlS?69u|0HXqLWxZ-EeLp{5<VUZad
wHP}j%bFG<tKKo!MWn%KR-rfq^n>4#Z!EJdJq*y^OWL?w@V8<;U)UN<NuwR(o-b51^RhrvEO{9z@YbBuS6zqjy~4ye+_dAH7ZH-?zxpX^XNGp2X3h06AfD&<
s92UZ1$N%kT2d67>Td$a0kKaqlwL#&yJ>fBgG{>I!V&n3nu?ij7c|fcGd-Cc*M`t)EIC%_qfZFj^om|Yj&MBdR^t`=R-q!se@8j2?bVK|h525l_4LmmrQn?5
TEbV$tewh_wKe;=?*u?^h5_O5ACy`2TS)vxCjsM8v^=>DIs4w?0mCv`_$YN8(p=jM!KENsW3Fr7SrDUW_8F(peb}L+v;n4_-6`-_?tFybix-7qVyS<0UV#|k
Z<u#1zP0ZfIqfq40pb5@8QUHAR>Qnav<$bE+(u|OAAAT$_M|Kn*`_=ooLyKJ1j+;zyX*xM<L|5SZ7MQe<a0@aA`;sukhv8PNKl8PskUp86K{z$qL>pCw%SM|
l80${fT6B@@iRd+oR4wAy8#EHFTO;sT80Tx&OIR(r&v&!cBt|1>En|9*^})7iMx42lGBpC5vd9klB>FHT{D_8%usp0!O;krwsUmPIRn*&FeV&oc){HYkd4r<
9ld@@p6&!8uJ9h>6EFUDImN4k?FvuvrH2D?>1&u}GxI<+l*qSl%@Dz37Vn|a%yyjzX9~+R0x3Vx#%7y3vZwqD8epW}Lmq~?L8eoqGK3s{S%u-Mx_px=*J^kj
z!#&z`)t8#a7%no1COZUVnZ{n(Ic1%D=ht&+Co06XN{$$3s}^QmSKALx;hfvGuYEbV&Yz6y@7u8hmRB$@T3>AdXSP&a8|(Ha-D&u=bLSc&!y*Fsgz%J7abG{
9Q9agUaYKfvMA;F%0_JM<B5<aR;O;&HBtpiHLFU_Z~GO!jcM`0(Uogq3s3H}PAA>3xv4sCD!DTL){=kw-4hcgkaoH*{*q|?ZR~4{&Y9;#CZ)f>c9J4(h7`;F
3heVWq4UX-E%)h*iUCFE=!!~WJ%Eisw~Tu-^>N2hLZk%0;d^G!7_~K<QppQLNKQ9nfnhfm)E*V;-|<sKp9K48Xfp3lopxoh-EnI2Q@TyAuY~?7&j@+9+sCSg
?E~!B-YADM<36+UciJ8+9DgRCHOMFPkK3YI#dBz(-8d8SKL_Si)wcK!;(5z#^s^=rODX)I0+o+f`J|_pWK0tj@_S`lmW;&BdV5O$e0>afQyB^}8umgf8wdw@
6ul2}!GecaLHw1hr)g$FS6$D8r@2K1;&hfpBx!0?HikRu^}&*Z3^@j4?X?Ffz8)Jwu-Q<Id5XDrGXe~{qBTw)e-n06tw)l9a}-u-tGg9F{+W=`k4t~rYyk@a
-M{^Y!6-!YS*A8!%f1I6Pra*;@n0cTpikjp&n(m^pEVL{+$HyA+k?C(w6ep6c@q{sw!B_zxzpV=+>R&6_#+$DWtqzZk3!*51_T=VW2a;9Y|UD~O2;R$!Th&<
N=f5+nU%iHp_L5#Y+gJ=q#`ES4Z-v)#BGs3(UfcdEtkcADf~^zFgw*Q5JwbY!|JUhUs|L#Dfz6U&oiiAxqZ(W5I${RJMM*23N!R8n*7t~J$+C+NJMK$??M$_
ZxiN(6?VsA{M`zNX+_V7hw}bhw(=R|3gQ8~O=EYG7DeD#&RL4#M+gUMqHn|ORp-hyUsky$a9VP@OO#^y34PGxIMQ9D!WsyOt<ypSC<FN^L37d|`}Fvv@G-Ba
w#n1mIjYY0(M~2ud0I{&4u$rD?L*EF#CNAS<ZdJ8nWc7SLgwJ9$QKBZ_W&eGsP5gb?49#(9FCDwS1Qg^TVS{x1GKi~5B=UD038D5^JZ&S)0`m8UYc0c;1u~u
h<xmp&#|9g^}a+s1+T3L?*1p+36r3?2;V$v*<m;Ng=;8mV|SP?Th14if%9Hm>Vs=L4g~>AJLmLA_uEnD8zzBsbFf?Vbp4yrjf9SBF`KbQBwTnHe965VU;>z(
aq`vL766L&c-8$~EgCK?h+!sSZ!J;lz@*oOI6PRSvPP#QR^^}`W&#tDSYt9E+^;A#d6ZJ5B*%Z7|DX|tKgY>-7%q1gdL+X&JBV6J-5$|2G`FVl&iv$D77TXq
NwjP}4cvLCRUt0Bf|}3?+g-vV5TEJ$x^0%31qm-e>`;{9HcS?j!Z>^tnyP&`_Rw&dC%ckqfUQ-O`U5b%n9)dI&20{XQxF=hZ1%rF<uIzBq)b<%d?Sud31IA6
FU}hE9t(I*Z+81!MkEZ^_QF)c;85%Va=DyWD?R3pNPUsVsDc<c$};+~4Bm=oK--|r<RwVea2*uOk2Z~kP6N;wG9d+zi%&CvlT`QJrCe1}AwR^&E8hpNbcvi#
a5d_<Iqi2Q6$C`$1Rd(1KjqqOPxLa~L)q@Jh$G`t2$8N5F|!szK8xINe+=~s8O|x+e3`;UR%dG5EcW1$=zpl&1c2HAS=acDT9xb)oM^ir+eTPO&VS9rtIvGv
`yj4-j?-<+?D1c2s2Jj4eIx0l+phZVcsZ8%O{8)({?xX<D~5mz(w7fF#YzrI-mk#E2g9G?H#P?xw$<Wb^%PECF<g>Vj>}sjSX7cqS@YWw5>y3Mbx<<aD-Mo|
dSfEY*C55Y2Yx{^lHELUp2q~*=y0ZIOS>7k<VsrQjnqBHVD~)is_U^6pF;NpF(XiV;?3<F=yTX5+(Bhwi2dqz!saWe1)bfOO|jvE<1AT_hJBNCvfw2#EGB&#
Jp=yZSVK?vZulBb?IYRqZH$BZL15+V+ksB1Rqskse3-7pO&6ryKBfjl_l<hZ_G-+(!5UmC44Im_rb_8y#jFd*&8!`|pH1AlWsHRC3BLgky<}p3X%v#;c4tOf
pooa`HeVc6(JVm;<hG^^ku#59_!_><P=NY_xgWwV{7R(3_LGkP;`!xawmy+jn)AoSS9G{$cz(+Fjfp7Nn`g6kU`ebPP|L(y%<U2-v;|tH&{#Z`u^L-Q5_Jaz
=O=Krm%(7D5)Sc`C1Ef^y3X>?<C>AEZjtW$5-L(kXa&wxbrWz^+_q^KT|E%H5^{ouujq25Q-B2{ftMPYpAw&UTr^xTmu<hV{c34oZz0f?t3^5BFIExVZwN&o
CK0pU>icGG-LRvSi3NW!hB3;;cJV~3=x|ia-~;-vQir11^ekI+hgljqqx4^!3X^qW>n}nE3d%H-l-9Kj{=Hvzhbq9;@68$f@Z99$ZtGETrj55vfY=0sBzQC~
bvM+Hc(#RAJhmZa_r1I*B6fs{<P_DS+fW79#`Cic?!yURD}0E~z<dX=vsJ{t29sqf&enI7HijFz8gP{jsL5C4<h4!~udzM^71d;>*@pg6zDTMXdhbz^aC*?z
ieD#5asCUS8$Z{T4lkS450A^5Et0UbsH1;<^ZBDq&&0`ndndOsIc0urF$6xnHb9~$FJ$w;PJ5|(%C5!G#nx%V4g6o_Mcsxdo|H2}vJ3h-D(2juxfla`sL0*g
HyWEZMq-Ds111eekO#K4e!~=ESK5vunGeODiitRvx-)74R|XsSn5+K{P(EFp##yC%e{tk}uo77;MIGne09V{W&#}BL@Qh$bW<0IPx3Zd|P;+yOp@)d(M8zfL
WPsJ%b`uZX%GSML-@bd$NJm*6Wa#S&6KTRYFq{g_CJL%!>hhPMf|ir6LqC}CxYlc&us@l=+0y}kZ+A?dzYp;!z_^j_>&l^&3!|whatUbVh+A(TXc3oQn`!$_
yor9{=dDXY7?ZqucX68qCLIh8;!qbUd!o$%#s2_1G!ueRA#a@`BLc|m8B00^?s<tNCU)c<_-F_+NQb4uAfcLpxTWoL8m+&!d)7Lg-co%0qEKCm9CVjW(G~(5
*P`vNMI?FU3%Q?UEre_)W1Wi+d0O66gbyAIe9eak{qOnk2IO<SVPCrgZ8#~2I9yzm+7RA7>mYq5$bgsI5v>7BwmR6ZEkwE^X{4_r!EhtpQY=(v*OQDpv{zUC
<r6M!?)Y5RN@o3ZC$$H;l4>%6D*!5ahH#N?fku0(R3!~tL~fB2XUGCSE?lKA-5d#+ZA%_z>__f?Idxr;XPv5rlsRs2s~t5ddQ}}Z#s1I>@p93|1))cNo49ue
=0w}kox}4{bZ^EMrF6%-OaUj-1T*DObI5{P?k)({Zjol9d;)#(&QSGs_^`QB#>?Sq=YMr*c8ZbY6IGgEBelkG%06yq^<9I@Y6XI3T;onx6t%Sn4!^*~SQf>+
k%w|m{je+G_&5jiguTIExt{80M$aO<sYS^$HfuYI7i6=*G>CBZJ?wZ|$|VQVouO57Vtz~<UDrdABa><PU+gP1@j&R)n$8Le{Jny;VjsoHmWkElBsK?H`9FL|
I_OI%EtJVz>8iQ9J=DLk{5@m9VD!LDHaD{n&_V@Jb`s5i%~9~GdW5MLx#pC8%Gij>e~1{7;5FObg!kL(2$))Fdf2I3$Yx1qq#bd-7gzJzk$-HBhhB2O27hc7
X4Ob1eAl~R;kr+UEV~;F%|rr9N*aZIE8=f_WdVTz#&3(?0dS=h4O<HMc$&mGhTF1I>m7Nb8_$+V9p?;`qmyivHgjC6l3?Wb*tioa(RIoIOQABfhkpW{;^Ok-
ylTjbmSA~V8USuSj+qTZm*XDuEyWkd5>^0FWG8=gD6Wo6sKB|=swV2@^JBGqUAqr6#<uHtiR5D;MoP{GP>^c|tzgbliD0Y-dDCyTY2GPL%K&b(r0#3HK8&VB
wjp&bfW^k$8$ItOgVj^XT;FO_;CApQ{(!~~J$LyeLDp7?2eDnLz8XKbb<-cKX%qF$$e8oh@i_UsHj#SAJYlnhrwgBS;oRXz_~-Cq%1r&6D{l%w$Pso75r<h<
7YM}~Gh)*3c^h@wGwXpk!!(fYR#KpODj<xA3k7??dXkE)0WSCU0v3c~-_|kCJvA(gc9_`H5>5De0ssMAK9eVo5>CIMR1A=i!N9eq|6Z7ImPl4yYU@6kdd>l;
2$t5>2!nZ+WrtBQHwlfs*vj5b9n<w^_BZ_#%^7dALmPmYj(smI#G|)VqE>a}n)ol|VaU@2pfq0&eoK2V(<MYwMK6Z*2?DD~foS^0>Xy?mR^KAgD1{2K9_|?5
H0JfQTP=&Jd7113u0IQT%_IW+HHNYH!CZSy0AgGX)h&j-laVkGP_aAVhji(u7^0f8bXaxF42AvQb-vLV9?_1ZS>MJMcZFg5XD$*0XmE8?>_#Rbw?lFPjmVJ&
yI4lMLT@-b^5_}{+8&8k?;g@0aQhpd*pC3FGp{syp#R0DJu57GRQz44BJvi`*mw$|T73fDN^<oU`L^4JJJ^CK2MlQJ-}4S$%!7*!55cBsH0fEj@r9wt7-CTh
BWmL}DT&mwxJuS=QXGGX|90@Ko}#3*KQ@;oRO!xckDCoeAs^bY<IovakF5&@Cn%1kv0sNxpZ!@dPTj<^`TLm0a2V?`AP2At<NyhSWnx|EboQXu)pBnAiZA0O
5Q1^v`L2l73K5H$SbUKQdzM9{&)DWi3>}qXm@mR6Wb)&FF95v$*FH?1eFN2A_wKSm<blQ{g{{#jIq20@<U+W(;q?L~5S{KP4f--ULQ$sThStC^^#<FT+b_r{
`A8Jb-OtrHB1XQcjHLl6t4S4{L;Zb+FR$7kpn-E~$P2bP7d-*>0_+L^jl~rxSbG=lpV{dsCopwY@XXTZ{L8=#2!t6yAN<3A;uLF%Jdag_+Woc;H1Br|q1Y^b
IHry&owrHgwp2O5@QgtksuQDwx9G?*fbEnYBb4^TK(A;^1=mxP>QWN9lB%?|rNXs9Cc#gJb;S|khUfbDSR8|WX5Od1yfed%p%RN6JGn;PFDF-6X0c|vmF`!}
fjqOOM+r6}*BG+}GF})O!066L{kmIJLF)<0B5`{xEqYm%=92Jo1BH<mGPNr@?F0c-Kyg5Hn3<IK)I~c_G@3^0-y|w;aAYm~k`9ERH~pUmEHbKy7h?l)h+@m$
T_YLc8`3*N1)I5YR}~@A(J|+RxQW-7b*495;e3)!%kZrH<|5_68z$f^SXzAO|546J=!?p^<u{KbFkuLrqP7tABu9*;on#s*rYT*=^$7iz@vHsOO#PCad-Zg+
X?$_^0u8)Y9~e=L$+*uYCGVIWhmRk0y3IRUk;S+cr$$%Jn#l)d^L-x9paq*XWKA4{5nb**9PgiA3-yJpGd&!3*4&mP);@@RT(I_^d=&c|XUAOc6amPgd+1*a
R|2}*T&1%YO^n|i92lOO9}DG4PKLhk9|m?tI%5H69!D>CR4v$w<5rXz85NLaVge^Qc2lQnv8}2~2|*zTJ9>V`0V(tHrBqd$#=_)1UGEw@?Oxh$i&r;8T@pIP
zB6WYIg8CGMCE89j<&;k(cIGC$;jD{k>jOG<Ho<7mx*Pi33*>K?w>Rq=)e_o<!`Blsp4=ca7z2aeE*gmKn=a_851@00Mn#do+*CPX7Jgmpbo<3zph_nRXQIN
fja+qFme%~49C|)$TCOhr@$5Hdz#AIuu1m;718Aj(M+D}@{F9_JL#-hmYe}>*s*1=`&s+zo}+1*s0Jw$89Y1?pil{tbQA5UH^P#Jd1vQiTup!i8``n#Z&gh5
(0+7{yM@MVIB5xgj-zxXDQxEo9=?Jo;>|gJ%RP&2N+WgLW=MKl>F`%kEKzJrnRm}Ujo1}$1I^v**sRs<3!{69{=!{#w9{YOJ1?c|5_2PLl0G~mU9}etzu>~(
N1loPUNiH{S4Xj@Gen{%V73R;g|Hm2`b7_Sf4SAejah46qsUVlz9+fT2?xb<KNeVpbpPD}mb!A2_hDj0RtD|`#}3GdX>C5@twP{80o8paYzLex*T3)h!;?{T
I)`oF<*zaM+9tp@lu#R``Z2hqlVz2V@%-WFleYF9Hc`g`lF<n*VTI^TUS_dVjnHTv(1e;`bqLx$aD7neGAe<A7m^Jns&w~*=rRX<%`S*Qx&Je7r+d4yfPXA%
<EVar7!BF1m;6wlu~w5U>v!sqD(6(rm@CZO!SC^jTON@MtIGi6;U7gmIh+Ud^mDJ?4|6>jinRCv^j7?4n-iUZX*6zZQX~S#vTL&}mN$Weq!ei2T$0}MNyZ&a
kO}X!@y~a06ZE^H5tm1GtyZjyyR(njn~okNYG=%4{Wu)sQ$>qqtVOAp;2oK_3t%TeOH)|3dBej$gy+IKMbaK}mnr)4%>&Ere{y1)D#yEM3`p$pfbkbAdVnTV
1Q}VtR?DOL(}Y$EJ3}4k4ftY{7<L0l^p(tC;{QncjM;mBc4g~<Rl|2mBU@Xme9&j26>`Tez3LD_EAldWmrSSPe6;7BhIb`3m6XL}CD#W8Nl_D=3+adYlf}}a
-@?mO_a+te{|~TQ)g);Q)KYX6+0+A87)_LaJc!>34AhNOEBFnNc({Pz9iUC6D5{W%E$fV_91N9#&4tVzr8wvKy*t`=fqQy$vi4-l!Kfi;5c@XVqBVpY$!C&-
EizFg>ZFiv{6bzzViIS@7ik7&|DP9{i93yKEmAU>?`jD)Ti-Y2m_0YJ%%qT2W4HWGuuBtTG&mcx1no~r<H#!AZ35vUlgU&lV=y;1lB6Uz-3qfnVts*ZAVLWU
_7#s$5IB>xPTksFecl#C7v-zAHFN$@u10mWvVN5WZQiLRA(2uY!(1yW>AN%T?-StGX1;*v{Q=){D%c`S<6!{Jg0x=+vqQcRErk7iP9pFhV+BZ$;mJA?aqu{G
I(iY;3)k1#mf8BaVyJ@5n1JBaH<5l$OBZ9K)pGTv`jqwueD5p;o`Bx;ZhAh1p}8!BgxKXftTKf)pKX$aEpj5$dcLL_5R{V;pvcsfX<*H#1A^JG|9_NA4Tr2o
mOPth4aKRA>hE0K50M2f2)R5%fH`{fR8X~gI6{F!rHTH$a0V`aw&*R4#H|ZES!?I;bWuT!Zm1tLcJ3T3f?Fh|PTRGVRYDUnC;VRo@B?*<i9ic7M6x}N7C{1n
ZSTku*q{SYr!B}eOS;Om!Q7x4<>m+Q)Q^$k5Gf#s+E18re<;0?zZZ=C)T(3#@BiI%9{%h!xus3XdW{R#JP3qqb^XH>Ir<*q(5yqjYbrxM=~bEO)+glW6sXFP
P;0%M=KbSFuarlx8lo^6H^QgHV1H}yG$`ZywkI$N_h{<UV{h+>4Szv?50ig_?cDVvltmxTdsropeKtr7!v68XB*3v}$a<DKq?=VtiPb$5huuA8`b270n~f4{
ChPV^fRBJ0GLtZDU`FpY(<P-}++Xy+3Q0y37C2a`*O;`m+}($!;~9JD6)#{8^jSg20y33}?M{1^&g#%<gZ&vvmDh{_B@P!tmiCkuD;3gF0Dp`D7)&fgp1B-P
f-_(9O;#SPs_Y$$6E-!O=qNF(woVST8eGuVe3UAWnCshQ{%Cp#oexT9G-DjRN9DO`w#7U$)S>j)R)QB}lm`bmizDeXN9LW^uhmnA#x4!F9n>jCfd`sbn@DUo
dy?M@g^+G9C`yn+br;<f&K|DQdW?@pt=8Kt`d+()TUc9538gZvca21JErK_@xJRjnTW#n5rh0#!{9$8jf9sa<8*XHHuc?qSa0;flVH103fyDatzK8Q*?6|`B
(k6pZ5iwnY#bZU)!o@D9Y(B&jg=eoGrF8<tXcUV_;j0V((ZpIt|J2l*>w!U5S|5N*h|#_mXJ*!|n@*>{^vtpSy*r!Rvh;6wQkbhZkET1jbQ1CEvks1MmN&zs
#j13nrAv>>rvEt=z6tyk566lx826q2epJ&_Ct?JsI-Krj1hGxLjxr|Ek{#{UmU%z-D82|p28)~brCR&4rBwT8u^Ogn!<g0)cMiK|#2$r;Ix2XfgBHT&LMaQT
e>_rna_qI@z;Veporh%5QB+)ttdr|pv$+Wb(4&-z1z#4TvMfxqk_-{;2fe4`WHN3K_g2S_rXS1|0SK^Fyw6aMA0jCF6b7s>2#8zi34!+lE!^Z)T_~DP`-RaH
Eji~XE={uO|DK~}-MTQH<Xsa4Atx_{<SaZ~mBuk--Z~}(yJ*agAo)BBE*-NOf2Z6ma5xP_ha$%Szm15htDNz5yd{yB?=0lwO>CR2J<?9Xmu&<kd~25J`0zAj
b;0M;C1&6_DODirkIm!7AK*W{dwY9@qm~SPux@ZdL|2AILD-SAIcb~`|8zgMTDz|o0Tis+#k~KNi`2UQWNqM0^<iL+++LuD+-|?{-Umd<5#px@Rhjru$Yy;t
AEMt-mdG68=KHT;>n!DcF`czY0T=R)Y1!M9V!vlpdMy^|s_l1nopwC8dP!E8)2yXbh_6Lge?2<g;sr^xldd!HFLm@RRw8j`tMgUh?hP%mS7+P8L=;63k`v6W
BoejnUK7`78z7%=zNAv$HOMDCq2F5PmH1VfpU%0EpbX~Jlkay0(OWc%5*sw$#f1|F+4S^g3(MICDs>VO#n3G5G3(TbLnFUzkY-r->3d77oLIf34@6DPu0{!A
b-MM&qHX&D7wy_A24X$iHZNtoO}<on_)<)FDwk4qc%7*VIHmA;TNP#~@B!h&^K`j|D(Fcl5xgA5vUVWfoeKbD0)KIxW;GeC`q)q6TiA5hzEI6@-=Fc6`+)O_
0+S<>4BpN3ciIb{+iOH*a7eRp62Bt#8DKS=rEM+1L?viva!6-FofmJ2o4-^ghr~zNR&T+$S(XN1S)<N6f%+udei=((UxucZ^Qao#QW@vI#B4Yke=QR>Xg~x2
?9!nH8MY=E0&QmDs0}K3T}mz#mG;W-)UW{W+lZ{U?qutuDIQ6EBz0h!WlE=fLcrc06N#<D8@Ju060sHIg$bv7*ry&~ACW8Ul+-W&AG*ZOwvVb-NG1a(=7VAm
*d|?XveR&E-OowX#ZI8D)-hePv)=ae3dp9KRqc}Q$3U6o1x8|+VXE>)^gtc%c3kj(x-It1dlQUBc6oD(Kvp45CcWV};xDn@LapHXyz=dTV7wEdxe`6~f2o!h
Nk+K*ht^Ws2jBs6zVEA!Ad3D3q5Xi8?8GC*8kpM|uu~oeI0T-adExiWbLSWoO<rgajCG`R`~oMLn4vTZYOUmMVKhiLq~7IRM*`5I1vFaZ1{QK@l*%1^U~lS*
afIkD+He<m9%;<+P|vTa_Dbsl)||=uoDN3mYVaI;xAGUUqac$Mq`ZwM8DI|mX97<&#=?i)_D^Rr4dLtyAXAtAYdk}BiBiAibKCXl=5C^W2Vc<wQYgdCqTrhN
!9gEOJv`+(&AA)t!zz?#>#(TK-Db==(F2OsWR}rB@KD+e$xnV6aEe!aq#jGBLjjc4xjs8r_3U^+Q8i4}z&5x!xt-R`&MD&w0y&4amy74wBAmJUCxPAjG_ZRN
Tor<oEnQM83QNh~YiE-%yKj7G5v16e))mKJcG-@c&<mmK!(CE!KL~vSx^U21>y?6AGJ-tN>r}h>`|kca99>&$sfVKrd&-hGLC#0FHvh+b?j(mAXy<Q(ELik-
C&4uQIoWowT{<O+fo&Ew-nqAiTIjxgSc25c=-i`2u#*Eh-XZ&djs`Bs?HpZk+>k7tK{_3I!^p0P`_Q=Y;H#i_`RohjL}KJ(v}ICKtZ_qr?Y)io0qMWrg%cxm
Z@k2b^QIg}3)l%t^zlL#%2hQjG%#~O%|f94)fbrdT~9i|)Z&MOdU&gI4vruwL^=D_h9&L5bx85b&*V(G?)z&S>BdiJrWfJkC#^RRZu>xSE#XG;8>nqIrqRGJ
47XXn;x$CYr1*U)RhrG%*Fv7(=d_rBwQxmH8#40klPkikOnW3n&Ok7=oarFJkMu#fo;K~=&18FO!&S3EP%v@0<TpPFQ)Pn;k~C+liOTgCcV;>!6Oi`+6sdWY
bL9Xg4i;-sqg#v<YD!eKr{xYG<9~cs>K8pVAvf=oVtSov#&mOyBy(bthMmB|*~wX)SZtS<6<1FBeQop@ghiASCgwa~;C`~DD5Q2x&HAf@Xg2v+*jFG^jb?aY
B?zblh6Q6`x7(Kc`+OY~&wA7JPA?(`itzOpoC<$FPUfM>QhBnD0eg^2?*UB7(B^JRl5)n;vIqdli)8jL))maIpi#Wa-!lynGjP+|E!Q$xLkm*6b48hPo%m4E
GtfDjXAkO%xUb(@bj*G~d3c;>hK-19H+2a6l^Aue;EILRqTMAI;YEQRJHAm#)A6<DJHp?nJf3=s+kapJXI|$ezv3V5IXOqMaX@iinr@A@mI*8*hjVx*GI8XS
lgVTxC}h}QNH@wCrfpoB{=aDp=p3PK#nVh;8jv8CW9o=VJeU5~x&L{2{PFdjr`uyeltnmmEemGZDyqnkvs{Sp^^^rMgMfzjV>C7m^$*`ABpaHV)kr>}zjg7p
&ojK^qx#RCMC)cZu$ALbQ7-QH_<bCLI;R$_<=}BMO#VgWSg~P12gN}6x1C58_ijiK_NH!u^Ze9YcmV%%u?*}U+|n8t9Jk8#h$5!`%G+GnJ;+o6iAdBwlK-Lv
DyZ$FW&q{(w0=^7&YV<V@>Fl)bIERzuQ`~xhty}P=DLm}AfWfPITyI_w&Na<jg7dY79D1jT~4aJuh+8q7I%lpF`5zC8hM4YjFJ=w$J$fA6Sp7<$vwba3;flO
WK?(6EiA)2&sSggC7-<GSot=@<`sL#dce}bV_MuZ%8>(-r#!9itLZbjcd8M3KHAieQ@y3*(XZf?t6)Q%>uV6=Lvlq0RYoxfkLxGgGx}Ug<{WlqufINwIbST}
Y0+z)yZIQOu(M48<YRKToRVV8*<D6vzca9{&E1G+farjLY0lhNkhoqo=;bP>1#d9#t$Vhv28TV|V%$F}EvG4$KiCi8pLapwq(WH`7<p|6bGu-I(z)VwQS0|$
+<Y8;jDr<q`$tM|Ed#yA>#>Y)D2RM4!U0-<MHTxKz8L10vAU59QIbsz>L=tu`0=*=?|P7F?_CxP_voO>-F<i3H~g8~KImx3@nKY0(Xox9a6Qm&N1{Hk8kYrr
`M|<vTm^zMFF#r*Qdi0t`LGA6sgKct2A0)m5GNE4qe}<aW-up?H!S;_Ioh;3vZHxXCUYA=bHzvH*Eixc)Iyk{az_mHdBTAK;1oV+iU*=P(r!7!^aXtt<|}`C
o3OG`dAHUwWNX@h-)9Wn&NrFe9N{=a0{L9vwkU=Cy|pu40fK%PYOvO0U12%fL8l^-HFl3(O{Anf7kl9;uc{K%X792T{(5A(yN+@2D(hEivCw;IgLTx1g~`3~
ao$k=Lk0I(FikoAXkf7I`ras5%ww1Cuc8Lp-4=kHiJN97eJ!vA5Z?&lemh-vb{u#MgUV-fjlrM{qGeMF_u~IvceS;KdgvqmdzDB0qjBk*u4HK<zLt}SG0{UH
dK!g~;~P3RA^{n9tYZaY65P0u#0z(aYcB|l&Cr%WYNV{9szAtMcoYjX*$eVD<p;*&v&<mZ(kaM#Xz_D;&yk?sNQ=H&-8uhfBz7~+#7Cw;<TQ~8Wa3|T&34A(
^W2DD7?JaTkAl7<EM=Dxmu#vv8R~q05IdYgL_5tLhbL<j(e>!{H>cnn^-os0Pyo3~2VQ82OM=nzik^aWgFgxxz1#hgZuTd*ZjWtk$kQ+fY^&!)l&{lm&(_x5
5^bbag@Q}@fh8?yk6*go#(5(%S?ICX(<ZW(Vw3Hb7LWk_`*FbXIJ^%%Q+qt~+jYt)xaLzFFflSWxHbpxEpjy=Wg`VvXvYr5o`np`8c&6}H&@V#q+e=g3X~(Z
Z1NSo)YV<@Py*^!FB=p&a?!GZ^(yGp2O$y+wWjXGd&H&{!TT1s1o*2u{_qY+VIfKJbD!_*(|@d3fpy%z?GFh(>km?c?97<^zeyGld7)Nad*(kR-|G-x=xQGU
kBibJFBW2c^w>Z*9C7Be*S69c-^)ArylMN*ES>5#ryRc4wqYs=p2H%X5`uC@9}nb-FwF>xPF4Z6Y;9`T<=F+1X;Nr4>6GLsHs^WVHYyt8V*8jdYrlsMxDZa~
l`xVL6N$>yJ14@T`)a@NyXiwiGkj7-N$fL~;OwZ{{qyVp=xwrwuW<}YF8{FQX}ZY6fphk<B=D21^!tGa3?m``)0iv`sOf2>2R)rTWauh?!ChYT!k$Ohby%So
7DGo##-U~pX!ZrspsU-|=dQ*6Nt>P&?8XlW{P{gKQd(`14}51O_UgBp*Q~dia%5}sm>Yz{2H+8TEY6n)!}|S+D3$cWw3TkfVjMBo*cNdo9`x?1A!Tp2j5miT
mUvdbkA6AM$BNsqOt6}r)~5_3kMWWXKAV{i%d7>Gihm3=kcGj<fyRn1Yv#zzGVzhNU03U1I2Gd_C4z(mg=&J4a6WBRXT6>#j*3PcQ}=%2M<Pg}iS1NrIyL7?
<e+|+PrXyjr7z*@gVz##w_m^X2DdhzVpV<8NDX8PMjQc8&-WRBMrrhQ4Nynx1;b+m#Q4_oAw|Snug1{mQA}n~<6-0@;$q1@^zRXq=x?Sq+PdoOSeBFgyP*z9
_TheVInvy?q&nNb0p7!=)?n=UMF7Sw6_ur#u&Z(!&;4$<1BmjAYu^h=%O{$;zIZ1UR{OJ1kLq#g!<Vkmn8`!s6s0GwC51qf@7GtstFw79s<c=gsuqQ%z+7QT
3y$dFj)jWrB`qQss(HVg)%GQ+cUeUvZq<-|<`BgW+4sbz^JKrywq;c5USz>1B=X6oo~av5y$>Wx@~;<n5Uy9l#HBfTu7dF~B>G%K#}1~TY&ByJOYIWY9X#J^
`zMstYVEwO^tu>Rx|SOcSpNL#G_F>t-`dA@&8k!bJrs|ebCQ3cxNDh3rWxPeNmmm8CLgB#6X+gPA?Z9dlR}=qd7+Aov-TZioc@n~Bn{nqJFf>t9}|(xgWL_+
SQNLhwaE2IID5-ZfP>>~%xUGZJS2l~DAcFcYe6V`E7wQtP>rW{>SPZ;+LrL!iy1G;g+F$`kN209D|?_g7MPCi%w>Udb&{++-5pIHrBcrWTFS~<mIgqWOs9H2
3wnS^IK2!8dq-&x*k<Hi*pj9Dn75=n(Vs|=Oh_+sg$E7ytSE5VRUr`#h$e;*48L}Dl6bNNncKRAt`72N!?$m}!lWG1FRlniu~B#2&~*2EyKgis+2_(S9$_1h
ClC2})qQ#;<5SFmkDjc>^%|Rq55VboGjZ?4|1wpP<rYTB;dO+F3opHIe|kR#S#)NIcxq_Vp_Zt3wraK&9Y|5MA>t4NiO*&m7BU|^FT<rSK-Mo_v$C?bW<KoF
l9;ZsF*kr<ui#N<XdcBpG2)ceU<4`R37`>ntGC#8Ffk<N4)cMf;Z<r2m~s$q2`|LSR3#DKG9b|$_b4hgG=IB-cRaPNPM&y*Ea{lSKZllE^v#M%RwvZ)m^Gz-
vplH*h-PuSe`)!^5<GD^Y}x?Q?#+LGB5YQ}r7i4V2@s0D*cH)tm;q@-P3ANFZl@u|FY-W~)<yCH!w9SZi))zH6J*&ksGvYp`s)d8;Nl}%LW|1q2_(u%wcRrE
3EOTAq$B&){&zJ?K~?MVDAY$%k3zWMQ%JGyvmz`84)L9Y5yv{aJ>VU1qRL4WIS8rifC%+VZ)Msaj*I})Jk9cyx5ZzD=!zTw1J)_Vlow*g-Y3#4w$BqlKu#hN
6Dc|6Fk2AyzylODpi~JS(!&tdAt&1{&S4APmbma-(kchrR?J|WkMwQnG1NL?`$}e*>Ij)Fh=o_*RQ)iI;Qn~?>M{R>idO!lB(@~GW~|%=w+>(Em4Tx$pPHm0
naC69o;C!16C7~&jFo7Z^Rl)FkWX-jFgk8v^+V*mIX6-}(IO7lS=3x|(m}j*fMNP__5|u;;vtx`J6=*kx)*C*&q>I7kirQSMs^>&4SiXk=R!sU)O?iLKNn9U
?{esHqoJJqzms>Rm1NZ^Sh7+5G5no+GzO6t3L!(wsNl7Q=Nkgr%=zJ;UYYQ?3y_$e(}6mu!ZwW9n*}mp;0e&q9MPrMFXWODN?vCeHmc^sABo6ZD!wi#zFP}l
mdkWvuEm0PLuapmI}3r9m&p`YgC=@#KuMAg5c^EyeZ*$Ox{xz=sXf2qpkJV6WhUfB0|G@caB>R(RqmU5{C#M^4~V%CP3^sHohWT<J*UT&#5>s*?g`vrh$hRM
A;h<HJeIRzIect+(v;+Vi}m_<vUsGY6ETQ<oK|u=Bz{R<FH((^Zd7ZgzWV{<d^?2+GB)y%4ai|GSiVP%OhIU70$^zu9La3hVx(u1Vu$;X*3aLV%B=D6n=vIT
I7lg2%TiOt`s`-@)`u%Qj|q4%bFFSIDB;0&#QP?flf4|!Dmh`KfZI*pd3I+z8K_rqydK!zr2q~~bV2XWT*Eg()k&vH;VxTd_5)i<d1U2%U^;}io={$scVY=M
I(GWcbtOXK!3OgS;cc6bLLO%fY2ReN70Fx}Q^DKXv8*2LIdGwMG(zg>=}L`N@zk?J6F5zF<3@hi0pZCu>{<G$C9h_58em`_58<GNpW|8>xvh)h0Ii=iU*DD2
W*e@*5M-rPRpdCc`-j$pc0ZXoLVM3QW6zJcpF0_iERyKQNH4FbiPej{MW28~yx-H_-gfB-DE5k1gz?AL#0_;CqN*}RAcOeZe=NH-l8QAPBEIp~)D}SE;SEFK
d8bbiNYmiwxuUmGtj;KO)7>^Bp(cf8!F`;GKJ3=gRLvm_F%$3!4f=0g)+^={3jaB0D+VKai^5LF89f`;KZ{dWX)2c{X;y*|2R0BE$mf){Z=3z~&JE;_`Sk_2
IlCKQ0g2YcSQGqiLWUEgEF#8{eSkEXyNvE^CCeeOq#P5dH)Gm$gjweU!8OgheqXuzRP=e51-akUg#*4^psJQ>qw-eeZAX!A!9F81Y4xv~`!7t=X8ESW{G-uv
W1@!+!LvIn8?Yr<rC%m-5}+k^tcleh5zh;81D9Q-7fZ6oS{ZWIcm|jg(^YJX^)>>~W1{Vy8|;5~_HB%FRwFOs{39{Z7R^EMNO4j*zZ)lQmHxIgf5D*X#6EY?
#!2CcO+P`SQ)%(*YED?_<3@^txS=0B84Umw7R2j&EDel8ps~qiTI8ICQ%iZ>oASrl<==puR}h+G@kr~pS|3-T-zRF9E4*Uc_E*~^t=b~D3I7qDj73fiIB4_~
KN@Fl7(pfBcG^S0lmuf03|d4t(AiZJ54m(WOG3paht8CA(GIQETtlUFV=~Jvxu@PBnwqPnYh(Wwap{k;_jVW<rFMuHR-VpNpS>16AkkOrtBSztLJ5sv%dEod
XG`JV=B^_{`$9=FgHrklp0eZY8}laJIvEh+H*4HaKRbOMBy=p=$eIgWHyTgG*j(e@vz{$?Nj@B+c)QCRinsPPIwhF~tD|S4NBM5O#oNfrbBG$wkCfg45~@cf
U_IZ2&v?()DCIpf1vqwGBobu$QIP}d5nPQm_JPqR<dV!gN43Z~R|T;N@klzP5hnwG#UbHXZ)~`x%1l4ewX;x5Jb@x`Og&zI459CkcGFO!V^!A$*xHGEC2K|j
Is|6E57LH?zSz%O`rqYmVyRM68a!<iwZ^pG3shuX2J-wO8-4AQ<Fp=HuAdems;WHrtviHI(9-Xmb(~JXjD!`>69;)oolemX6eue|P2@G*Pb}cnW43Ocd>Kv(
xutB`bUEOEUp=##FA9xk>v4I9=g52F)WMOIV50}V{CgSivpFuGxXSr026?H>wI<~F#<mc?mFwKtSychh+xQmM*}=<j2g&->80jdSLs6)_Wvjb3RU`u)$8?VD
<zeeRhD6GMCggC#B%MHMd2YL3Be=n#E-KJ}a2o|&|K7m9`dP?m&!=q9jrfs=9fVjCccPv!BWv;aS=(W){YGd$%!A0Exm-D3VqtE%!a10n29{+W1V;F{k*G))
_)7V)o_d%=N|tpK54qjTtm%OoaK3@cU6Rno!#rOC6K3DkrN6g>o0Vjz7Mqfv%GZzBm9v7qDNMeAY5E(b)DPVD`pLHr@5G)kS)sX7&!-lsy^)j@t2@9Al1F&(
dyBRP<SgX_4{98}ibU5T8>|2A-4f_DMb4Ga1ZF1A35^JM8e+w^S|{v7h|<eSQT$#;#{a}~RaUJVm#!E%o)kHJk!AQ^*RFw$4g48%ldR4-f9N6{f0T*9Ov~@x
dd$rhPsl-~u=qUrx3``r3%TJy^cM$r^1o&B5O{WuH@E$d^LWtIc3}T^z6~{8nhfpXxP^|;0J=c2S=V4j=RanB2gL#Rq~X<vmjF%&cTarbf4zn*qTzAtyN1JO
f)w_<Vrax|QJkFCh5qb_vdfjL*y!|h_i|R9CY9ClI{SZR<Z_WvKj++fxfx_mEkQR~9($%JA2tQEM2RL{(ClZ}6`mNgwN0jm&gsLbPz4zYWg8dpZwty+ut~{w
h2-W~+ffctelDGIW?oP=pQ!c-cjYp2IL1uu-dg-1h=<ZV_Lp{|3^G;n9C3_$j1qtB9`e=uD~p`$txpryMbu<{zR~y&76gx8^(U)VG^li#Q&i1uHAc=}iPjV#
ZS^Wpa?Lqqb)I_$L*VR94Kr+94KdrA)xf&ykU9)tS+kc!LS~;I*G}@mCt}dSCfTn0ci<={ULIBeB*h)+pKqmdDl3D-H0#-k*#$3Is>jPK&;|A(W4S35>((i4
AMNl0Z}Hx)&{b4H-ac7lEh}eOU{jCe+^_X*CG=dbEAqNhD{2||tTi|nP2ksFWmwq4^E#QjNu698)_s0x+o#TWs!?02Ka0g|Q3DwLkTnpff{huz+5CHL>pWr7
r~VX%Wy)n*6+VGMbe8RrN+KRiFm`+$_c=bN%C;trc8U;%?8#hb^-NUZB06xd-da<<Dsn9xmc68o(ZGRZz?TCUmqR{t12Jq<;3KRVAt>0;9CPk{Uz=-T!?9t?
XS?3CBA0;aFXIwXL#CApew@E{eUdfx6z#ML^e@!3JtJ#0m=X?NiT|Z!r)27C2y&|L<<4X6`Pe1Ur3ok<Asd;w#D!|*b)8D>uuLPN-Qw3Igfm>1hBa?0Hgp_L
NU7ZFm<_}pagy_T<?ARbdNxxR)VCmT^|Qpf_tXwtMJ5ssJ@_(Wh?hR<f5S{fHG~r{nj@PdGl@Vs?tO7tk#-%>kpW-UnBGlastp(sTKx*Yk5YOuX!o&OPMNJP
^I3B)d0|w~XqR!efObJqB_lAD4?BXn9Wzk*z1eZZQReAD++|5VsYSf5C&K1X5G~>Hr?0xbiiiYfs7x~${P{_cX~oInO8vS(UKkngPT{AJ2+9F%PZ+w%!l8Ao
j0pbK4^>!6`moKa)~5D{7+&i9Z|}TBaTc-!3Rc8d$7&`VtbR%lQ3n)!A+Q?oTMlt{vU8f#KE7ALPUT#iMF(DNoh-09f`Z92I~*T33ReO>g($gW)I*MIdk8pi
&!nH6hu%cVwBi|lHG_4W>#D+6;C*o#tmXn_+f88QQVe_&M)YmZ)figyepEMT-eaKOazSO`gKSB~$|>KbMEa=2N@*tXiCht|j$Mt*P<84gT&Y$zKS~TI5A1BW
cNWn`8nV7CxbHU1zxq4&>O5SeH%U_WXa)od-$ctH*005<NeiE7Ff9$^vJlN8K!6fDd+*nhqLmACkUX-aGi&Qz3(!zo`cNibQ~!V5jT?ug#m4psF-7`Y(7#oI
F_RfH4%IwUMXS`C#Y~}1i%XW-7(Q@Q@N{qiOB;)SHmf=pFx~AP)!%$bl-gX@f0k=~ZdpPV>m-U__w%@T^&)SbFX?c&j=UO-gP&^5>zYKa^VEk)J75))?TDR5
3)pq5i1>~yIe_Ac?-^a_11zu1eV{9a>Q=(@@Yqo=pCdZ;T|x#^`Oi@AGn#BvGi_n4ZJSaoa6u+F(FWKv<q1SD1DffKaPm-uVE*nXl9AU>a(z%>zG1460!g`0
XRv0cfNiC)f{FQT%;1r^$4`)J6YyUdI~R-k83l9d)n0<F^jV93Kcs^NRiPu``r0JfFjrbN^SffJtEDx51djJWXhS5kbeM$6D#X7MW>yFbr4{Q(J2ibA^Kr7N
)gyr|6u|WKOVD^AI_(PsU5m}HlS>Iy)f6sk&zG({Xg!uJVF*0fZgwe78R3p(Pw7CUB_x6u2Ct!3Hx@sb+)JMWfdCKyDc<Bm5u0+B!{RO*D(|v_Z|SmRJQXi5
Xu9?l+k-R6-KM!KATels(pH~%@=<;Du!&WKK$PZUsVzD8v{8^Jv`6{HEtz{sFw!g9vy@&j<;k$A9~0M$BJV~uB@kaO&MRW`IfK4ag3#K7#OZlky;=wRGfm{9
Lbp4=SMsbMloykfaSP>#>UH^ti$&suMhjk9Gta%#Gu^9vVobOq{q6hQ_vgvKG~Rl7Dmyn-&f>Vqhs7-MfA<FtM~n3I+~K(hNSI4|V(V&U1y_eteQW?!jcpzV
7sm(k0yzb%snjv+KyCTT&+%{Zv`Sk8#w(_qP5}uFl<*<%0<p!3w8D+^fS^EEWv{c0`N!biiB>B{qoi#RgD4i`v@ht?vYW1f)Q2J|#eqEbt1`m6Z+PRzYESOC
y>BOmQG6RXrj3^0M@Ss-fP>4pV(~&?>>ptm$ogAOEG#JI*MyH5$jRkfO+f9%LR8T|sl7i;bY0gbSU{gJBrk7PA_<u<Cf1;DgH*uxdX$IEF|?`}!k97*>T%?C
gA4WEUb&z@B1#p}&D@bB#barDE~);_<BKQm=))^Z@M^%Lw$lvYhR;cPD1b*r9Y~JS&+E0+0FAUHT7(IhRiRgT4C3wN+uvZ9lmb1Zu!HO~<SZKzd}3eQXa7*<
e`Y7vb`9G#0F`<=CrcC|tQt|cxcdG266*Z)z-M`60IU@SC`AJDg+I$z&sMbLc5en6z@e;nMS0WFcAd9l&2-ikIWKfa)&}69_zgmv`cKNKMgN?ga}HS7N4C-b
Rito<&Gghr5vrwSqPXt(7!N~{iTpLe@m=%aqw9)AP<&fEvuz-3!epVm!5@L-HChHqckC~<>)b=`Oi-$>!FqSaa4thAU!A8c?*Hc@qZ|jiW7yV)%3^ezAabla
_*hPMZnIsTaKvGwZ<a;VQkJBWdlj=FV=?Pj+g;koUP|x}AzaON*nv7%PC8Jm7lrP>LqUZUs3-`k@@g6kpWw?f9(-L7X&&)Myr@I_-U$_;+enc+@d7tGG})Vb
OMDnCJ)x-MN7q0<h9O0uW4*JSnsSCXu@U!GKX&l7e*}*o3Dga3TO?nlBGT;%66Ncuql(ch@#B&R*mEQI>?gZOEE(h!Zp(`~P!^@~A>+AcF9Z~~ug`a3%=av{
p#=N)L`827XO!N$M)bs!`^oO%rd}I!(5SRcx@@R;dA9Fi+cXC$s+D}pDW28e#ws}Ta8(AJ_%IBqr<q9C$d4#eM3jQU&`C3*5=S)vd+lumt~qtZSpwsIxK6&b
%a(U=5D)nxde6EBi>E3w9_0#uZ`mXPR~!8{yQ0pqB)90pp_i)-89($@8mAuL2a1x-obB?!%HW$m5z?1|bAg@=NB$bXQtY%58Hltpuk2e0j(Y2b_881T0@3>N
LQ>o|U^mHH_ca>Vm*d0Iv=>P83`6d&UjIp<7pVnUvdT6ULcZ8)hq68u8}r?JlB)3;S$dq1%@cD0DuTP$?5Kqv+C5^{(G&8%zBetSyLRiPKIZ@!$x83y1dCHf
f$A(Zg~@*U2i}lDJmX-2`(T{mW%A@(A1A3U!!C>%81Lj*a;E1`$dmZNtPlkKMRA2c*`nxzh|-6j63`2$si?C-3tSNH5hrzVro#2IvGMhsY}QBHZdGd4<laeS
C?E6H4tx!=Wz@-OVeez%sIYg22La6o3n;&=zP_UN55<>Abyik74!}Bv4ew~iRZKd_t6!cdBy8n`;(p>r0bI}CsaH=Z?U5s<P6CWOVn*;lE5QG*KDdXQ&sguV
wjbCnF^FCfw{+p}XUYoZBPwR$Rsa6(0Pf?&#bjBJ{qw>Og`&U8UW}@nulH#mhvx$ANk@!ewMWK-65VqPES5DUEM0`0B#8YrEJVDOibxeVzv)0v*HbqY%ZT$P
ci$m?(C^+b@;3<;w$P{?IE>sK=rZV$SD94yMZ0-ll~G#&NWdV`p3iC+;<3=k<a4b}8^Ve2NXTj5ogQ$DI-PC?+p1Xq+i0FPA;c5t#a(x`p7=s?$kgFBUJDVk
jsid4>?;}tUILHv&uwBYd-+KyF`awH#V(Q1@yi)VefBRy5%+gD&50>MvTr&l!^zfY=F@O{Jh}II7%olTPn}WIG&;{BExZj~+=Z=MsgXwN6A2p8`tWLA!*Qo>
+QAR7yiI5499oUgA_U$$HM`f`Ze$IEpYQ*oFYRLLQxdWe>A`5$OT4iQ$Tzz1y5j~fJa9DvVD$e-a8hteys{>w)aUrw4E*c@n&24bZW3jj>$OI{XBy63faN62
-4B}SIv<o3%^JU~<s#@99!bvKP(;Pkb{4Ls_f>l4%{_##iaaC1BoEjl$Nq6Zdv;4R=eU5#E6*hhJOfg4-_$P0-Hjkys1OWHvjJHQD(@wGw_9ebg=!44;X4C_
mL}7`O{A3U)H$!KRMM%TT$KSwXwk>7*TN?Gv-mUjouF)qBg-R<O*e;Ti4E#4b^*5qjq{pu>DvzqPHLeQ)_f91L3i8B{VS|!Q+9vF73mB!>n!7&lA3%Kn7<J2
v@YlO0xC!<UtuO$EZGrkrp4|&OrNrEXDp!TLOED4;|RGtW%$Rm{%5YA%J1}b7$qo9&!D@2yKFxdTV@qu3rCWY6lDXq)}lD8A98%0+G0JM3bt9@ZAZ?P!T6^+
#amWul$f)athqY=O~EH^D%kLzs*~+sTqCe;pZC#b;<r1g12&irWvL*U1L?+~Xk{5EsIf`J1SI-9*^M?;-Ea5(06hzN7TGX`BB@(n=k$4Gj^&#KCbw6$7WoW;
a<UmhExKa%;*4~eF4^*Zn`Qsn!cSWmBva*!ng11SA~>KMaD2;;>gP83)-l+6-P(IB33p*GEJuykmcQmk9j_#MVz~saQRE55D-VUhgvcuO(L2ST1A?tS7z@}?
#()VrI65&dzB~(wTDK^Sy-0dn%~|)@OX}`D6o+TeY1SQfdEHOaIX8ESex!Esw>GuSKJpOSI_xZ#zCvslG+&)P2eS=@wQW|AM7`CAzW7!dNBvE=IEmurD5T7u
qgdT#7JB4+Y=?$&HVW0x8-!S|VV}2$2~<?ljd9lgm=S!glwEpyb-lu&gKZHGez?m6Dtb@et^|@URFDiY@g;55h>{TvceE7fRZZjLZEuNi%lb+TtoaE2_-Q}x
C`HJLE@fK1yB-0cHY{5{#S#MX={ulJVzK19oC&z&mgB=5Q0Q(W*AFFtVI@(oaDao;2~ayhWnpJelCC94T0QN;+$Wil+T;^b1FkvN{`R%Du4N@O;0<!CPm%?(
lk|+Hi#9o-eoM$scq?ninq*NdzIMIYa(m0?80XrnJ9F(3=~k`E1<)XPD^xXgiATdWXd99IFeh?L*aP3K9)oPX9?{$=?SC*=DPSwHym&hQ5E&JVC=FthO?~!>
tiRO+FVIvIBntYAs!?{bx3$&tz>5m&kX+P*n;>FxPzaofR^l-q?F@wbtmv_I)%J{UC>^-(mQQLAYTvl9n`l{n-O;4gOJIS){*yb-jCt5kDrRLS8HBgfJnRI#
dFhoE!Qu6II6#@IoMl}WL*8%^a+qhqevv{ha9jA|KHr&o3mPH1LsXgIBwtFIa3^ksyu8HOdv&EjZvl8R$?}(lcKz4L&LHvZ%m<-XRTBCPYcyeSgePFwl2<DX
!9m`}xv%4OW#e`TvvQg*_SpwYQU#4AS)-!~c3hH2fcLEIJRk*)94#cIp*-Ysf!HP>6l~S^4tbih$K%wq3DJ)LeWf!vyf5K(22scCLQH|xN`^mpgFI)rnN@pB
!J4!QV$Zd)`faLv){*Fs`M^lV?`!IW!IV?nrekGTO8uhFG&hm-$p>$i-X}7R<oQM2BG))S(|-C<Gn4~17^IzMN?2S*qD*y2b)buPbJF8a>sQ2V2!UDv5Pxig
wBCGd`hP?#^Qlig2UQ2UpZZSTtOD|}nLuG6Yo9iEKeTFEW)An?Kfv))VGS0IsvEy&nRR8dDZ#5rBC;dB4P93YDi;Osn;^PDShxr$7&lA484d5y?Fh7dsEoaw
(dvQ0!6<HT0>^JJ^5I=|!=g-KZbFoL2L(Z9OQpF633x{Eg~=(*1=kL_xY0v;Z+;>YHP}j%LQ{)oNx`Z7O+Z^kcR@m2cPJ~ZU$WSub1l-pw{~fK)KN}GNC+g!
cOV-sT+jYIAYr5$^0qVU4)XcGNzy`TlM;=Fn0ja$3>#*z?JY|jeuCN7IFlmZj6)S)x;x3tJ5|6@T8iCJYmaKkz&<;r@O8HaVRZSys^1B?+A7w55hL9UXIA)!
;K>v>jW-*g=*oC0(~S5Z5rd6c$N+(;HxnTX?eHFER*4*m&PnzCy6xS!b`a<$akq!n$^79<Exa$t%R&PjgEb%Fbfj#eN?HK&PW*WFKjk1+wnif-Ha~zp{}QHn
EwoCDjb~ZZp35vW<uI$@)Z|LF8Rs5O`yWw;*k8qK;Fdf0c7#sC{wL?r#E`DdRc&Z3qB2?(TAPNu(!sj7ZdhnJV7u{NvH!*JxWXsG4Mr8cSrQ#u+I#k>8xF6E
8$E99Cnk2<`gkgL2a5&RfpjKs<g3#?!XNHM2!^1}*VwEF$&q*f0_S}Qq=^Z-nrC0|h&9oiU8eU!8yvk{p&b3SNtrdnroC`X0PH2^t7u4XWC5nscE`qtG)vlB
-Ag!`JZi780WBzE&=RLO?i&%0ZRqNEy{;sdOt*bqw@mTDGZ_3)75vg)kz@H4@{X=r@PI8%+9#WE1hysA@)~!#(41-_@Db0qTqgdm96dw*WX!!q`3#K~?UU5!
{Jldtjm#4*6!>GEsb}oM)BSU&2U9%*VbmuQ(r7}dzS{WuGO7*(KMA}fzm71=Mvnhk1SfOvxLVyjoc3APL0p6!4R|U}DMy@H`9{e(bthWrZ5(we#EE&+%ATXc
BDRY)AjXx0Dl?(gKsDmMQ&t_SwtJsS%#EmXG9L-S8pbHEj!qFywP(<<-&cgSeVmuqJN+mc(GE4`<VwR&))5pT=0#l~*`_apl*Z{O|F4AQ-;6K(ZFJD~j=k6^
Ml>kW?^@)1*nv;TNy`Jji=L9+r2s3L|4@)1OfS!%E%lE5G%R!c2tGmOQyf|U+-Rq&VyqaCV$^3CjWYROZ8DUeOdD--UV9KOJ787}e@RvG=JniWOPCb1ROVfO
_5XfY6hEzIw6do2gZI`u+ItY(qd5sZaC_9Q<l2~6OiBtg{~+>Jmyeq3FM=FJ9SE#s50Sqddc7+mgp^6Q_NGe${coGc5PKX#{WdEIHpbvI%#j}|3dQ>QN#|d<
fK4Kej8sFI&)M(^Hfh#J;J)AnVgin3;5I8Fq?_gT^~l@>30uKTq&Y&{6}L5h+zD54$ejuF94ffh6*#=QE_a&cFLpfl&HSUzvmN2UW5zH2ER({hYwJ(+=Ah%>
8ug1L)rS@mc*<SvOx7fygrO!h<=0NMkN`}a)B&cs)0-zy4{Oft%PZFO9ZA_+&L<6AX(kYTgk#21JPa1D2)^dSu7peG5E6!7e5BJ4hV^1<lHl*W>TpC-nu4&s
2lMl()umBT=&4E-=7R!6&|^F2-KPw8-j3gYQISS1g!TWW#g~cs8f2J;^z$V8&n7c8hU)yj0fU{Gv&4wplvRo;Prg)JL)(d{HU;D)c7ZR<F&Vgn%S1lOqI0k2
kG(v%3x~wF>?&N#Bp6Rs%qQyXU}kL!)#6RL>j%InS#X4`_%E&#&8*y<1|IHbJ&Uw@tDm|{ov#llO6SGs@SCX?o}z`@J$>ImL`K}r-t+nY5eOwvJLZ2NQ;WzF
udEO8_Y}TBFvV8(6kSaG)6DlR;a1|iU?1(m+am@X+Ta^Wfq^$Dx8j{=E5eLwo?VmjGNCg*;%Lqif~rM7XJ8~<1VM=rMJ9NsS?T)0NKbX{XV05X-F<Yhtp){N
{SPc7(v&izSP_M1z*`x8cA<Z-i?$n_Sc!kN%UFyIlLRiUdn0>ryaA@pw}CmHHj+K3nDsp10v{Emv^7K60$WDp#xM_lDhvD^j{1y<P<9o&5q7qe9ITk*r2J+-
b_BBkwTrGrqvgKF=bE*w?wL=NfulK;!1-F0_rqScSmX~XP$X&z%!L3*|5qT$?7anlN0o?MTTI`(8Zr~MyBCo}$|N$Ggg&33`-;AYBeu0=y+~e!aH!+CvX%($
JPQO_w0E>&348ALfj4;l4_+oa^xE+It3Rnml5c<z>XkW{OcCSkAlDT=K;OI*|7#X2!~)qsHAQTOWOP9EzKa+i-F$WtQeDSTWOMc0BY|5-vq{jWi7BaP6FzUe
!CNiB4>M;0+>e#_2DV=B$RrKaI^?cuG)lim>fY2FLZ*TTL+D_r25y7zC@ij{PI^rls0#RDsb55=4&M&Mi!ue1_A9JVlSi48$QID@kJ`&V0yW#r{BRoetQ6eW
WzpA>B5SYo+c$EOWu;n#2R~>oc@trT<TxvJtkN)j{Pxj-o49_^cC=cX?iNo3sXOT(w6|(oqP|eJtF6O)p-FX;sik+qg+=y_D}nV^aQA>Yv#o?@bDcQNN}}ua
FJsB3o=l>`;6K;yic8DM?al^CG#2x^ya$^@BkFOe`)*z3EPnTe!{)vZ)5#p@l<jO&Rh_G`&R1(vkI!z!3T>e^ax{3s<*KP$8-%p0a?-TxVgsD702{=e+g;lw
7WGxp+ABxep*9HgnGJonIPr%0iH;8fcD4v5m*JJX6x%r#6!JqI90tYjE?YvMR-qbG723btQBxXQd+@IUBLrDOXEfHtnf<$+x{1lkt_J;b_xn;HtAm^M>CN0I
bFoD3M3ZkUO0P}EPAV!_hg#WaffKa=W?ZhNsc)0-V0#35K|Icoo;*Urp|aY3zOy4Anm$B@SzqgbDE*hAbNb?kgBz`~nOIKh`JT?rk+}-5waF!$upT4=r2UOT
<>F&bT1&^8jVc)8D~K1^<(<qjAq1jLNk2n+xA4Q`Jn<MRL$}<OJ+dcUi0(KG3$jg5YE?*f@-ZE!?h0ri49-Ir>P!#ScAhyCw9jNM(PrQ~iankaO6K{W-Ck8r
;>R1awcEHcYRZ5$0hUcu)u)&zj_ViTrtDsH!!2R~ksA1jk`!2#{ym<Q*Uy!k3+=V(#z?t_Jk$z^I9XNG1uUv=Na^(n1QAs>yo^HaBwtImaw^zO=~X%}!5BtW
7)2vv5Er85)}lKJo{fWdG0rTAsN9;RB&SG=qM!BZM;sd$-rf!5ORd6==r$|kMs+SFZGewdX>psi7BF}<?uUU(LAcIz=oHnwYT6i*W_UzI-hVoIQIWu|j2<L)
n<ol$za=O-&lmCOAz8aI2O-XLUVkho9_e5ARTIn5^IVwNRO-OmpBwEtG5OFbx7Z!5L}j@@FppQhW++d|3wtil{8BRIwAh&;JH;#(6G5Yt+*41wov0xq{RzPJ
hD!!@(LU#QI=f%fLi9|AV3+qnBreitx<g=VPe1LE*5)wSXaiSMRxb6B^lvUq;Ek|Mj>;IV5?Q`7Nf9u6{n&1r>!-g5SuUDscB0SyELtonG*lo1z4OAM$=1w=
FkRClAE%WR01^ZCDhMd<@(`fnf>ijd7qs-kx0ykOsvU``^?{ZA;DbCzQ|>dfqD%;Ma|haAv}PB)hM*=1KIi=(&@r%_8ud_RQ|b)?z69vMSB!(~!_bI|Bm3IF
WDZlR;fz*Xkgy^wP7%w$otm>K+Iesn1(L+Q=QjaRq%~ULc#j(|D>XsXHSBwzdK#A(5ut;SW=(oFZG$k7D)<|m?PHdJu_I)*hOcxU3f|b=Y0LapdD_NrQTIb%
R`Rrnc;BNKj8@4FH{_pd%Q}(Vj#0(J{w3#fKn#tSfFkC@|DU`0>!Q3Z&J1EMT(rIi1NH~Reg{eDk)IV+U3a4PGW&}?8i)`}QW}+C?Oz%E*0n)sv0TMh$uyro
O~t?~OOM57^|pQ|o_h&dU|b<Z%Ht>aDSXIzX3$ZY*x*n4U7Pr-GSR~WD?M^22$m4p-=bdaRe1KLItB@gss58XX6tivPq25$x=lR9M7I^%5YJhZYt~T_rdUmH
)2Mi8v$k_IQi@Xj{fS!Akq~fT7ZPvGx`=Zyasr(d5Clp0(`$~_ec2YOQTx_gs6d`PZbuM&xxZBwg;x{xk#M@qP6Mi`!YTI#9F9kz?H}R^=P=xTs^7I+9ZXkC
c<yhN6Dyn1xUOM7m87XgeeG@Yabb>w^nmX0%(+bYZZt&@d$7Dg3%h+;qQx$4w!z%L&0`8M(Ww`*FuF@&v9XnUIMJUjm0Vy%VHY;H0NGM<CoZ*1R1~F<X@rl4
7n)e1wh24bV#aL5_?hO8P6n7iIZXtdzKI+4t-+c8`@yA>V>O3h5@1|d%8S2yfZOnlIv}U{NvQa`j_tCLNCneq=m_|Bj|2C(SwG?Ac_PBPm&PuTo5B<wM|w_v
6Vqn?5CZym#DlTlac<doW*d6_(!h=1)4}SL3`UBKN%27Zmwygjryq^eQ<N{6mnu+RZGWc>s5o!SgQr|P8aT57-I`=`4s^rzfBWHvgHRDbVRVv<#EIhrL|WFD
Gb6E^xwA}J(c&U`wtD;^8K}CJM#^DV)>MM#4W*N;jB4#?f+5RT3Fqrlx)QKZ`4(Qni-~@+)bk0-6IJZhHYrfH-DFmcK_y_Z73@PTXvMKnZ#MJh)^>&M2MPc-
Vr*T8hl=2jyepI@?N=yF7ZI>Aou}gz2HTZ-LGu|@%gM<VwinyQ1Mzbxq*Y|&jA^cxxyqqJ=VS-xl=RNM)7yPkkHuQN0ya;f5)9Xoa2%yUunOME9nJ0Ooj-jE
MPoHxC$^q^@0lee^j#i%o0P^Cl*{nX>lz<t{MVNF@ot%bosX-o(;lKhrY&1nbmQHG?K|%D$F-n<{Y&j;;Jw+^p(WC5H$d5ao4_Upe6QufXv*I;CO`0g#8#Vt
AY-yPcG0SGc1$2Q#FIR5B@)QLk|w%6abjd<DTQeXoZKhYI<!>PUUBM72>y(<oR?-Z(Ck#aFye9FEDR#lDxy*ihd-aEshAcat_3gbIvr#?F#;aJ+kIi%kujkU
4E&43L3L<a(d?c4(CX<XPP!<R8??~0pGB!VE&vx~Upa3yP@Dx2&h0rS-KWLlwaAC12e35v9jkH^G=X1EzJ4r6rbPF?w0<62`iwq33p~+-T3Y;WCdB+f0kCEp
CUfaV;B&5M0)r6pTe%E>L7!Um#~BJ!nT8ol!$yhILea8tf;{X)?DHK`sk58WTyIte<YJRmXepIC%!V!^a$Zx99z`uaT>{N*(kUR+_i}#6hJjk(yUA^|J{|`D
7{N^)N&Aa|q4<8ciTVf{U2_YhW6+;2F922Uu6-0f!Azu~O9{Gg`+K5FgKC%A^vpBqGWARXEbFe+$%*1!nu=CCl>-j5fHU2mpZD{|4e?YCr|;z=gN&U${J?S`
Bq&-=US2M?K7jA_d`>^ncgV<zVVj-nvAv4mr~Q$=eyF6rca>*!6QN*0f_t(>uvZyW#nyXA1E_n~Z8f9Uj81Iq(u<fL>3rW+*QE9ownw@)93q~sW0DPCB&9*>
QNz;BrlMCPLhNqE_N8X)8QO?=OQ(0)fj7mK(J;4{C11#7{ff1<<QU`B`1ZkO&5usF#q&XcugASv*no^z1A4vr%wBQ2wQcs>2|9STGd@G%o+RiynN2HJ#Z;0J
R?1-Dqa#oh?ZBWIYzb$BIRXp7odzQ}DsxTsh3;nGx+hwpKZ1t%-8~|ZMeTdvvR;8_1X_a%f-b9PmX)TVOT$pM5&|meEXyV9*sMAutk~lb<c!hWKjw?cKgY5Q
%Q>v*FeG*X@r_|BwW9I_WT)s!RVwg(c*D1yeZUPwyz$zF)7H+*^GPF09@MGk;)tzO>4p#&J=XIPXJSuKz)F2{KfT%9IS9t27Z0#&^29ZyJu2cAK4H=6?e1{!
|C;lEOht5hadxnLsQOb$WgOvGPT>?|hyV1QPKePW-&;D(v-}g(t1yiJLL`wu!w8~LbU6q`UGlOW;3OVK#H{FcHAE;-<|6Vw+xL_}r-pN_DD*OJOb?USG`-BL
uL(EoS3JBos&%Zwrm7xV%jtggD-F0gf;g(;5JO4z=$?%z(-X)@sK))I;O|+I15wsqp(4v6$oVk$z;w_8cK$NSnp&5ZeUT%&jAUA9L6xMJ38_5yT@rIbY=&_M
%%kv~1!4xrJ$WPke|Y#b&+XL3*GN=%8wwtMOw9d3<Ucu@gN+Hj38BTYZl$%EU!S#n@3avot$=}`Zqtnx-P~!VNR!@rP7Y5=wGBS2FUtD1Uvr6Ji*`1SZBu?D
-uqZtQpauwGAJ2pnl-D3Wx5nHBPlMuzsw!qU{>MK;Dl;zski>SYzvx}?w=cIF6EB^V;bd#MymFqEcqsy&Bl7VD1<m7ATKsm2{U4NM7wCegKsE6;2x-)yFJZV
?y47cUh8;anXIT0>`TSbU)NB|KO@*h*g80Go~3F8Dp9JjI=mb9-ru;4UTQXF&RAj*s=3<XSC^)Z%#YGXMgN~V%Q)d<dJixnDpk6+M5~%oYBJqCtD0Pdu&G0K
25p^BC5!;OF;)DB2Bv<DN&sRA4&vP342U5h1#6oG4}Y(V7`zO-20rkvY&ey_ofrii$BpelD|a%G?ceL1N!x&!YTW9y6kJTuHF(}P!0W8C5=<`&0n<hDLCo@G
LVEf&**sdYpLP6g4{6Tuf3ExBTi-=?`uD(@o4KA<<iUbwTE`Qh5euc<M!Wu;a%6hO@u^4@D0iepSIswZ8`A2ThX_fL?ZWvITYtZoyk{%2l!AEpWgc^sS&5mM
;0lIGca;=&dP^RN@Ph18sip}zT6kl>OW5uXovjH^wzBCjTOZT!EOZewq#1^KRY0)*?Br}m{BNCs{BklDd8_ms%jw#cbvG$evzlN%-`u~<yYpb&+L+e?b_!-o
9OxE4FFDB3kyQK~1M~f0KiUtN%kHfatpvpHL%AZf6WmRAq6V(AscDNSY@fVg@*_&0%~)y2eN}eU@&#V5p&>?<I~=!5kn}KhHjHPPp|t^8o8W<6?<S2muWeAJ
aDE$NdBgirk7O!iVs#rmJ%+srJ)9Ox2Onbqsn9m?PF+m#MwK+X8+`!Srgu)~PJwUjc^}wS57l^W@~+}oM_4S1Ve1`stY1ptyAIwX+?`@}f+62wT;Xc9;OWyv
{St3EOuDsyK;3*?(X`NWW7W?K{(65wI#+xKrJ+<mq@i$}&Fv#RQ6`Il3t*5Jmf0~JJai%Xk`>bEX~#t<y8!r4-59NxBudhNo+s^e5+xVeS-!@86UmCln$-99
BJfMJRL&cCLO;k6l)JhYr(0ePXSU<6^z9K_ZJmc?S+I^)-})I3!hkeH$$XWh*Pv=woMu;FUSZ9JB^=g1`*Kq<aI&R#7)^-uOE+g}HKfRQIw$Jba7Lxy2m%u}
cI+mwnGo@G3rE;10(?XUR3fMHDIeCNbIxVM-PD0UEze4e0%OjPgInsF;*Rn$iZJg77?o)vb_*KZ=K|{OP@^a2Gvu2X`~hnk4|}&VF1{u2xMtcK^2_pH(jIE3
cmQTiL8Q^Uw4D!8Y{DY^pWczVsiSbtEl$q~NJuXL;x(!ZHI8$oU>{7@0qvCq`%1GjQIk?2o(C!0Ixc7Tw+xxi2I^}WsJPA@7YY3kXD4`#x+gKl*2q9`QEJ27
tV1+<4tMb15;8+=I30bzU$1lqXSGAQgGhCf2{mGkN=c|Y;A3GNq*{Mz-nZyKogeiYgtTgFnnOycuv?9_Y&3ovxX8F7)lF8P^uZ)|C-f$$y|Sg`i!-ZBeZ11k
nciWqy2bj@Jk=5<P#VMtLIcYCTv-*&V{`RDP6ZF_+qUDvS)cN4!Lv`Rt7k)b2t5D1>L8}13+$?FORzqF<L%2?0Q~His~xCM`Uhmc@Id|?%T=C)hBdmZs&MiG
#)7=H|0N;Tt3QjWul-{f3>WtivWap{7?Vo2J@V?4qcsv*3hx|g!VFvSpP%flQrXI7hH^JLU}0CbKEF1lI5ysCD!n8cWBO<o$nzhcu%k2bp_wsUN{@ZnD3X$=
Ql2*_&{vU5XB)aVW`zJ<NPLpq(^wTY=}f%yhA{VUTGNjnumqh0oQon+?NjTmnmT`1fTldO*}uANqm#a)0;(}ow1+bZPckQ~NGkV<BnkTt@%s@x2W+1jmIKn(
T$PgPwo@;>oc(8+662Xjuw)??asUIxCOLU}gID5gyUG(@$k&VldS!thQvWqKM>vkMXhT&DR#TwKKDK*Dr<Zmn`58o5q-EJDVOS@-HMm-zre>qg<;C6he*cy>
tg6F*e<xXOq_IDb!YyS4ec|rYK>rK$+tuvKEER*q@BuAbvS>D#>UFKTiUz|Qk8)Cg;5g!9{v2Gf#IK!(1w1w=ne%PdE}2q`K;zE`L<w<*B(no$txHcKU+?kI
W430VB`3ndj^*Uw5$rHx<0TyqTcdu|ravbY$1UdWc~z%j8MNhrr>EN|NN{f%;r@!q{U7%qG#gArRaEUNe)@P@a?}_Zx|{Q7Fcb>KagT+jIpA%*>kR&Ba}w$e
R3WWej9-T1qOgbiexb&V7pzCeViXiv;&Hv1XSp)Yq&I~`T7H}dq>lkEP=^D8yLQn6tD0Ha)!_*^1p*Y#7reqwd3M)K4BZP{w)NW$+eB+Q`h!BI(YklGI}=fd
EiJn_(3gyA`$I-`0KB%J5)RWf3=Q}}h-b11P1j!kdgVhGJzkGTJ}$U#VYvvyRW#(e*W@w*__V6%tS3mJet;7MEW`IYGEWw7)rL+^-43MPY#iR5CuzKfRC}i&
8(bm;sGhE0y-*%}S2in{c#t~{HaJvI91=yZHysYS?IE_F#p!lkH1qYPGF_zNt1WI(IxlS|p&{$5wHss_*4%aRv&#@{O@L~NufI4(i{ySzD;=A&5RMHX&xg~K
J}^Mye^;2&V=Iut+Pky2j4n*8k2ub%<Q4%WbGak|0$vKi;V#DT?<wLakteO?Fn=A-*9)Z{K0yLcGyuL}-raMVNek-|Ut<O7PNYve-P!|CUvI^(Z~R&2hrbzY
LLIt@!y2Mj(Uro1n2>61=y8iXN{R{;TK^%2N?N#YZyVIC1yh42%v@Kao)qmS48n>)&NzlEbuvuXNX1_j0OBb%9wJLONx0`lpz}^K=ZC7RcT$O3@-jCEH_ZKV
4z99l%`S8C2B+#Yhb4rl47Fb*H+T19oMp45$a&j2$*uVuL>15TMvV4_HcflK;k>izyylz5?1>l9S7+jSVWMEd0fJrfxJa)U>R#4?WGDIt53PDOk~%@6exCkA
r7ou;;nNCt8n2#pk!i!y*CJIF2ZzibA&}K$%rE@s|F*Ki|IhzHA%AkDY%bCNbX-@>oqaKoivzT|GTatRaYDnyyK=5dL-q3$+nirJB|<C?(>Lrjhcih@bdDmN
9_8=+-W@;Nw-7`v6~Ah~(+jJ_EvPr`hjH5MA=m<M)NqDC6g)m1KgUeFPX$4mx_r~WXd!rr#BT-1m3Y^uMU`WhhV1QS8xO*8M+aX_Q07syBbXL{@&>bSQva}T
xSs*Ppb%<yD3H&Y0Z)>jEVyesPVGK~i@IMkXQY4IAmNTe7W^?ybG7dMJ`~-Wa!B8*Kg*fJYpy$x6PQ_`*;!)f`3+0+h!MpAhGs{ap!ji<l6PkK;q`GjcL6cB
^?2c}%@(Brq^<VZzUJuwoEa)A;M58K^gfBqT{<+o^RQ5e$9G^d{>fK}NBzAVapx-CMA$z!AgiSv`Vnik7Jy6Vb9>huFh86#FKp}9P)T~wE(`p_$@f<pUS;3a
7JyhhhEv{vGO{3$WXPJxV+jnyicZ*zW4x^QTg+PX1Hg00EE~^F$ep#?z@Yja59(>CzPUFGT_+c+VwE3TZ(6?m=cF|dABTjdNz&r)nw+(0jgP~E<mvJSJ_n~$
^*MmC{Oo%gB=`GgPxMsHp8|z4GBT*4Fj|%ON1NamYYtGO$RE>fs$1W$$YStL)Qbq3;MF+Co~UBQF(XriIi6CU78orFk}($zdZUR}ox<3krOF07tp0=^BR}i2
#=r{2*s5c+-;k7^R|Jy;NOktZLMG884k@!0v_l(<o0S|ylfXf7AleKS&xW%0!gbFqPj_%UrX@`uuP%ZiUhWZANNtJ<lis@(f=2E-A^ZT88eAqd?TOTYjQ-TB
1>cSPqeIIMW8dh6_#K2GpM~!FFBJ0hC$sFkt?<8R4SH+ZyaKGZmxtn%wOLoeHmxOl#9Vn1Kg#4Qu7N%V;~qAn-aS3*?jj`UbN1<cZ4n0!Odr_GAFnE@YXAh@
2~|ryR42x_Bn?w$mKdA`d^(uh*f8uEdqR7O)j(82LyQS08H`S)h{nGwz(Y;I!fJ>E)vWW)Vp*&XKg;;U(T9^I+k>t<J(x*x-4Pmj01@O`d=|3O#B&vWEebt-
X2^@6&Ow~nF59VCk_GtMugSF>{gS|J@V$m!0;*;C^@sgyZf`!LY231V$jG%CxeEep0V_|6asKybC0&-5CeU8iwfv%2)K?$a$i%stP4=LA^u)ArKnS&C7683;
0%ib>dZuWvTa*q%?XhSFW=r|fBM0Q3U%#S4qe%ri=P+}XQtxf=4G79;2q8|5<z8#`?K2>x5Jix9%ltzq8ZG(qrzdi>os}Yfgm<mVp-2wn#aX8$-m5c`54snV
QHB<$k>{Ihh{T(7;KUaC&ABeuVz^WNaD9zZf;V6D4e5Y@K=w3~lw??|<?FLoj_=1Ds)TI+8gSJtD41|P^f-Dla*QSC!izZ#QY6J|VN<oR1r-|D!NISP9*--(
vH=FFVm$Mfz}(rXdeJdABP_6bGBtnDF{ySuDp&V<rIkG`6WpNc9ZMvu#D&X0pB9;kH?l?}Pn(VP#rx?53`nL1F-VlFS^c=|zYGOm&o_gxxSw<D;hgQx52`Pp
mB11Rt4{}GL!*%99CV`%asjjV!5B@2j=-ML6Yth$kR@f;U<y)le-fKo1SJ68XF#LO!lkW5fF~1Ak(-x#?4?wM_TO)Kpv;Fh@#NbDIkygaBOOHXB#0n{R(fEs
B^aQdLD}A&`NLn8CTwrr5kc?`yu4I}pzp(iXHw-;CDIt4^kS7==$5D$jfpQz-4tZk6<*?<;|tAkm!l4QF+dxBjt?pH3c@$E=}aWOrwp%A^JNHvm)roIL(tck
&-h>}E`}6VNBPP8nZy0%L09L22B0@8RaAa-u0$rwil)WIkk|jhMgf~%YS<!r4VIBUCG=n!4!*DJ|JCE+y#}H$Qn$ucRFQ{cERhlJyMYRMHwJVX7ZFMzZx4HJ
h9pXFPM(ym4Xjh59`OxbG!Sfbmw-HemdP;5QJ02C?JKp+r-Rx7olHh0a(HpzD~^zY)jkA%mL*0!CMd|(+fpA(r46g1k$xh~`#XVYMz-@b-?A7cZDp#`U_k&h
90n0#w{8!9nDKPax>{wOsLi1pWJJiz=A(|z`+9zdzHeFFt{DPC#j)oP_{x`SrCn@w{i#v0cR7a3kB*5Y#8QK8($IbPe>$mEIRBet`JmhwOh}0w`M@gOszf}3
eR2W2*7qZ`o*ZA;;g{qD&SpH5Q)q3^R|;cb1BPOyZ@f!zmUq;SCIVpMsj`9K`j9M)gk-ZK*#|X)nIdRLn4~$YxU4ryMkUoyK<igcTWI=MM=~eDViNq6FJq!N
^ba>lVf^!9M_8X<WXPL-g|&C?GZieD7^5*h`jIPLrsp0Ia9TU0z9`c`g4IcMMQ^J^F2TcRa*KOa)frg(>n+q#lv0IY7s(I)g?8+Xg;@Z~R^2?D<#&P0IZ&3|
{Q?=y!!>I6)9?<$T`ktYSi&dtETpEXezC!l{<UE)0<SDtQ_gG)3jHkT;fuy-N$cy;pE5<PFnAU_5y9|CZ#&>`8$RlC83iw7StaOwFr98bVKw-FcsZ9&zEH|@
&-T&3;c>jjThN}jc+;@#rLeQXhes!D{9ulKW~+Y)pl4d|WSQy$Z5Qg{ePCi<1rE6wT9+nEauE9=D!^f`Y!c{4tXFt6kkaC6Kw?Z=S#?cb=;$Q990c}Fsb{;i
hLeQ5HX;Dhr!VY}hs##}x*NBF;MSQ^rOk$+T!$&mb$2`Q0DBHz@A&8J*lNltSI2CdPFe;i+RDI2sry@vtw3J6<y~?qxmS)<88A?inceV!O1$-OFUn~Z1HZMy
V~2`7Hu551womOA$qncUBY2Qsk>)}IuFOmjnho!dD@k36DYrrdZ_3iO!zy<i8?%-yf1s6rI7D{2F~Z9R6HwlbjW`P5r1H?j-Ip1WL=E_6N76(&if24B8&hx+
b*H^BF#*EprIe`9kpFNzxvwsK*(a}j7#s^!Dn5u^IOkEZqi*CMT&Me5*LECL3t#TXMiK)g>sFl1u_7-orI|`)($L?Gd}6rO*gixN{twZ#Po(vV8;1(i<!&h?
&q`{X48?1It0Gt!D#gq&nUbij{1vlI?*C=|v;>qrwZb`23Ues;IAuZU2*V?;YVB7Wq09JkU+$38-1gdAoe<;;)g`VcW!h(rZ`-5>8if?87qL}1bZb0<FhQXL
a_b}C=eCRnplqq(uW>L)b0u2e-u!-KhXv2Y11Bu`G8-XPIz~n$O_*@$V(ag}TLL}(4Ed?@Mx(bbQTTJ7!)XjBpPD%?{z-BT2aT}%^~2q8fVkbp870bTq-iAa
`1WZa*nlb*-`%CBVwNw|5Lb_(p_?Csg8AWjX#x$s^bd{!eOvJm2iEUH5o+CY;bS<@E1p;5$97F%<W#G(wMlO3l}SYXiyQ?4KzDJM)ReZmsiI9M2<Z+3gBb?j
qg5Bva`AzcYKAYf=q1^Qum|U-o*V#(C!$J7$t=6~3kmMjDF4QC_N%D~j)sxs4Opr2yMB^57v4Q0t3-WL`BZ9<*F9Bxb{>%)x|Tv^*e&8XA5B(EnUU}kgToSi
0U`O)>-XOLB^s@6`Ok5%mOX0)a2x*m*W@Kpor|cX6WyO>I;_61`F_ovEXljIt+zHTDuLI18CA(v?ELREx7_ww>K0<ece8Eiq3vIC5k$;6Xr86zgoyEor!e=o
Oi~%kx1bUydUtIAlpUx+4=}NMs1TG%zH=FIxia(6khYGJCqq&TFe+s%KW@rA1VwcJUoU}eVvObsfTnarNY0(yiXX7VdRAf%<-n0l#VhJhU?x$fdY^x<g0QQ5
{T0_MtQTHOa<0s-<{NC9x#(AnIN6M&#z9&T{#VNAvY)*8XB^uZ))BTe+6Y*D=tcR|P@+SQuCuUi+KG14&-sm2sXfAx5gQe_l@Wd_aXTwIJyDfk&tn*Een-XB
C8W(P>`=4xRKJMwzNA3R#yAOJfl%65nbk}9R3=W0{^532@9HD{wIQcobk-ElPyFe;#KD4_hz)#jR|%5<aopCMggYZa#3`%_<@~QfxaA4U!II#wzgF5`1Xfb<
D&}dSj!edl7o>aQ#}l*67{-|i|LoCPJpwocBlBgj*nJFJ0vAced&j(HWXwR18<5?tel`z!UX{xA3!p>)3wwUYTW15xslMw6PHd%ak`8q)@eRvAJDil_N9M>{
3p~kQ)o4z29k5VU(cj|WCFA*%i0E8caQ*B+G|2jLyA}KYkB`A|?M2URFx)KdeaM=rZK<s?=S7rCdG9kjZL84$n!keR6vp7N`RX`D5TK_}%szfHS=P#}&0q6<
e}^6`<*)&Ly}&9PT55@KEnxbA%dzy%rU>if44jBD6CfSJ6r(%?bkBHv`AY)TY;Yo=hN*RFdOvN<&`FzyE$|=TU<<0-!X$4lB=3dL2#1%~DWCt6XnBLd|7;OT
TjF#gxly50@6jG_0|eYU0aK#bBvkIqZoPMZx9vWePi{-{lhtG)eZh4jjOT8t%CSU`8asw!4*a?tRlcP^aW`z`IUgVa$VeN?Qtg2D>Y~T&v?XC!%_;J1^}DOc
JVy$h8(nIs7tKs=DNBg3ZePD}7jKmIdp-pzLvrNVQz<|Bb0D_jV&-X>M7bqVYnHS;B4NZ7CX!llFE@CZlE)OiNGA&e0keS8G6Q`Xp}m#xj76{~J#bQWaL4(8
Jj^ehrc00Q=POe{pTdm-SYsLGW+}o`k4qtY8jnru((e2J>3QG&Mb!u(SAVG&EnC%90>K(3WI=c;($8%*W-~qNnq3y`8R>Buz3?SUG61uH$|RpJ8ly?zd?iNI
m%&=B$&X$8-bYoJ>gbpAo5j1-1pq%5H(Q9X!|flYL*YVxe(4M+*^0Xf1F(HYV!SLYJ?H28WaK7)ex*XxLKZ_a2OBn=yJsM1_q8vd?AbCV_0W>dQ`2-@!+}<R
ZZv6EVmn&9u0mU6-Cy{zfO~C;JA!<x9W6?~cek}WsCO<DgECRdvDV`WAwnv?z6)c7icH|R=JPUBugxQ=0_U-9M9)NPYceDNKqzznMga_or~q&_aF+DeJJmY&
4wBqCAH_}@RO+PNlMFcLI=%xTXX05`O7*44FI_g;Nc7q+>q;w`_a$$vPV&UL0makSO>8u2LARo_ZltBT)F(Uzt-D8D+%57CI2a;J;90&+Puxr=Z`TU(v&N*~
{9~l-vyu3hPFMLO8_AMl8&QZ0OJSKqyeVFOJ@zhh#zo0lmfUBq0^PK;R&U4M`HEM@$RNvV=(22tf4~fyQWxCGHe`xb{?9O340USW7!B@HyNOt`G$v6ODBd0z
L<QEAF89L5VLj3$m)sB5F;GcQ;b5mscO83g6`DYxUYmO`%PswkXYYo+!JBp^DUN&vM}Yvn{LfWM4~OK0N{G}K7^vc#MG@pH?Ry+jVSQl<w!n1IeIlGRTi})V
&IHisC+SIdHD?mEqPd^YxYRW`aoUT#2nh6E)-V@FIg;QXAA=xBYV^%z>~!)GX|lEFoI{wyL#9l%__CxxUEl(iY6NM!*TBq`*q}x$b&AYoFCK=`{l59Q8?!RD
R&ZAHtzu+iwRqQ=9xJo<!@F)dJdeC8+9McJ!`%jsi?pbk!;8-%8W@*~t3$zefCZBEsqWY_Zz($*zJccM@(npUzW1Z}to7SD=(@AQFz}r8o$Eob5O6>{$*h8o
SgzL$95j@EYKhnE1p9RRX0GGQ2M<E39=mr{y?C2dW|6Q|PxN#v0w%8li&i*-4B-7g6GiG4=ZFBdx8hMbs_!uJr!Z$RKoRRiT{|l`zd#SbvEEr3pCvb6@_5#f
4j6t;kv;Y25JKVF`J$vyvS@o|$W9Qz4^}mip_V}ItE_A@RnQrP)1q|o2+^dX#%_@)%o8dt>vo8>2w9^kzkW1EpmS^^)22^AxQI@vE`qbddqI}|QuKN0w5xSv
>PB7$r1h=F^8M^BDw{tcWQh^^kSoS!1!VPf!m;M|ZD$qejpI&`)5EjdfP&2Z4z&|!<4S-Lz_07eeUj5$k5!LLZ75VVA!0KKBghC$Qw1wB0yIt?*b((M`In%S
CW}VbWg+)HU1DSXPs8&zaGlig-D`R<E!w6M8RYE~to?ezHBcN1@5@nf#mgnj-!?{Y>Yj)3)S2~Iu1K0IK<;$g_bskk*Gad^jt&Dh*gSeDVdYhYN&1_$s&Ufq
uC99sl4_%<vnf*P38z0#x4x<jzh=4?(>NQwFJ|)HUxyL1IVRyc`S9)X6&Bz-sxtnu1}6WS4QmDw>0Ucpi7?Tjue9nym^z@{iV6uV-@A=nEcIKR?CzR92$vS9
gA#KrP#KYmJpk6CMKAxQ?ctgK^+t$e3=hml)Q~HFL8s^q_MKuCfOcSS0DT0Hjr3Ta|9<%#n_?4BDwC7j^DVBQu5-NG?(PTWp8D55Xm|UnaG5WO`!nz~2QgLw
@{teI5}(EVN@~|r8x&|uM-8PjGQtNx{r|?*=l`-e87}caBMCPg1l0*SqEs$YpZz7BfR4WO#~LAs@(%TEZ}kv=P*q^<;2l-!Z-2)@U72%JRcuSOG#UAzI8XPA
!9GX0(7sBmNSYv!j!y#`Z4wKJ|Ni>W7*H2nAfBInD0ow`N8GyN$xy;Dpn};hEf92DYT3R(7Cl6EVu+K(DX*=P_@~WGX$(fl7rBF=%0ReJHYqIj&)G*wavX@Q
>@cp!=Zu<%3=^jyRWXx$^F-M%3V#;GRx<}n4LW0JMZ<IT_k%<3<d|YdG8LOD!)^o`B1MGqbg6KQelgD3U-^>{^M&uhy*3&Wb%FrUK&!}ilEDqYV2@{>-Co9_
nEwN?k$<agBuduHr;!MN`bRYzv&_}vLJXOy5&<+|syWy~7|*_%ezSO_rg0`^RN>vD#s#KBs#kV1uJeDFrD)H^jcYOAwH7lx{uT`nm!d5jteWtWz})BSxnJR(
6TZz%sj+SLvi=)bCT8c63V8ZaG3_TC_SB?-h0h+{PS)GLZefOCW}Swlo+V^0^S|_p;djrl5_7-3kv7?=#jq>l65WTTL{_eT6M8F-sA>Na*ahmR_FLj2l)NLp
A=aEv*#`OS3PrYV5zdPilZqF|ba~3c?i?6sHPhd6f46brX%bR#BP~I%9D=atYu>rpi0nasEp7Ta{2(U06cp^MpBCj8F2%6-zV!gml)V9TodHZ@lJq{Zx!wUq
&spz|UCi`mM13n9x~ej;IqLo0Oji}XLrXP=WA;anL|fXxgRrA@`2hqk#Rtx$V)Y&!C$&Bl(yM2o5in^!CNjp|Aj4Et()YiI>X7X*Nqn?cv^}X*qsbqsBJFnj
YVqN7Dq-isgA@=;yf=(u=;h)t^3$BEsc;@hAiO_Y=p=63ncOi<u?!|*+TiPMf`|CPjq2h?MBEf=2dmiV%n)ySCrP%K#MN@^Rst^W;dasj#+)glh*7>KzJBF5
i-ISMt2f({;CH+jRX(A;6NYc>uF2|)QdL@zvdJr0>B#!hbeKAfv^3zh7!bb>M3V&7JNRCBx%DGwO@N+f7Sf*N%F8c6hZ53YABC)Fmu@EOCRTj_)L+SI$r{$+
ZQrbzm%1heg#UvKFQBDRPm+=(2?r93NI#sh9Z|**s{aF@OCaYSb_n?wij9Xw!ySs%#ODu5)5W`A<>mzf7&avSrZQ4k$xXKg?LiRr4q&!Bpw^vlR#LRWpcA2u
88H2Akt??r4W^Lry5&BfXoeO*z6rUv>1FK)2_5NDIRx0wf!rJ-3hs4+sXn%_XTZKPK6lXXmql`m|2O@C+w*Rnrd*vD)x9m!RD+$5iNc-p%4R5raJKti)1QB(
!vlF(XY>?2axWyWvzbSdj~-aG*qO{QK{JTAC)gSJ04NNb$$ngcRVHXBcOA^!nne7XMy4Vtj@Kh)InS7La$hf2@>Aa_;o4+<XJXl?_^qzK+itsdpGCFDNgb3w
>$gAdM=#8?H&`aao$QBfwz1UdU^j~D(48$pezTYnzFD*Jx>6RL#?IKvArRoCb(5w0|3|{!_&P#L4hL30GITRT*z1%Mtm<gZ^wN{Jc<~`)lXg6Xpv3E+P-0>+
-k{X^Z<f~Yx4G`=7kQ7gTRe#KNb<zM-(i=qQ&btGOlc=q<s#rrK%!#qTI>3g$xGq+S=wEvp&fD*nf+3lUm0I0@E#o;HF{O~>mA>YQ6=-u9YK70j6Yeg9%yZX
@{U8Dxh-z-om2DLwW^3k1x_m=4UocLBFbU)0=oy>j%CPV?N%k!#x7fW>*==d_s>yg4y$NUyJswC{(#bI=jtPo?dm4K#U+?BXX%gA$27J)FSDou05=I|SwhjK
I{=-~Xach!M6?Dp<noSut#}OEJQGhNd*Gd=x4Tgdj)d(44cTl=${n-&^l3*&l;n9DZwR#V&WTz_+3~4LCFlqjn8hP5@}~-y4}zJMk1T@1jHxIH!Kj)pNh!)s
L+Y%jxt^MQW|tIGKy6PuHuCA1(8jbTiL+Xer6L0LRr4m)NOg+55pocCAUqP!0nLqXmHSyDxh>|s*iRPUS5*KY$A{q*o$$edSOu|5^y;?lA1IB692GeOGsk{@
_j}G#{s32h+CBruis=mTh4;5w2TmvSkb3&#9v(l6e<My+`EHZONhS?ogY?D_3o1rW1wU`2E$>XYLa>;uFe?kz_tfI5q3sYrA2t!}L4Z!GNRKMZ$gkan_3183
rn3L|H1EhuxMhvxAKFzV!;P<?<~K3n6DEvKMd?x0EXGh7Dnr|G_fX<4Q@Ti8@30UcYErCfnvx2Vda7_OUAfBp=H($w=e~LlBI&Yd<lr!$enA}s7hi(8y8r#q
fqHF1^_>SGDj1tdP$D@zE(@DHY@kgdgH)>E503m3HD8u9SN-I1hM>`3CTzyH&TeU&yFeeba(g$J<;!bKwDGB~!rmHFxsW)|(zNu61IUpHVR=SAD{YTuVHx4w
#wNlMq4;0tHERj?MR$izQzh3-xLPtQHMHG!3Tv{#Bf_6>ysmT6R3BLLt(xLYcj?4%I9AJshz<}L*LkhX7%4Q0#0yiCdR-@r{e@ibhCk&Df4i^*-2ld-#@k1(
sc0>4`}Q0;Q9O~5WjGO!aBZ#QGQR#S-D(tmco$xD$j7ZoGlk^D;nOyq+uCMQCImEr8Kr)nKWg{3I3lZ8M{XU5W!ewj@4lnTH^dq{g{$lNj2bWnxsKuS$Fc}M
mZZ-3AWd0|%%;_G0UE`|6kZf(8q@mtzvG9G#v=SoK#Kd|JmVz|*WEF0Cg70k_JUPB@MN|+U_hVtlx;?PJ&*D*nc>=E(~dQ?%?@cB5k{s%!42<TJiAR_Pg_OE
rEDV3e^b`c{=+f#_fN2$&B0Kz$Y$;%#&vB53%M1BDfOaPPs)Z-h#|NawX9sP@pv5+AgwkC(?l4w%P_5Cil|v~Nwz&K#oWPGdK=cGmc~Y3;2IqCX1Ax=F&|GU
@5i38kg}6`nG9|{J92vz$%{=#V9s;y&|_4H6@1m5bYAk0(=6DSH>-}eE~V%eOZ}27rP3d3<g8XrnHeiiU`Qq;2q~0oJj#49<tGaOO7$Xn+uY2g3qH)iWcR}=
5h-@7q=Cp(wjyZA>PVHbkpO;G4Wc9xXD9=EXxjTEwIX;oc<Z4#?|GWjP1(z9AL$uwf8rw`Otyuac{#&r9g8v$67X?PFVzs+iUnxha<x1vRAV&G`}y}PHbAKY
n()9t3~GXNy~3Wo-^%d~hy=0V;zE{S#mXs1?~S&m)K!NZ6Z`PB*kXbHsHh~Apv^Vw$d6c-pLP~owmQZrI%{7)70=>M!e9O;4Cs;7yIwvc`rSi3PFkNl$UVLK
)Uf-dBHVBxAH08;z<fMM>qUKbOT|QYn0%bAzhk0pz2c5;QlytA_srmX8k&(m+N(cw#}nN3-aFmbS;_L+pP94bEA?gzRIVs?sJuWKbhZ>Q?UT&&gv_&-?>3Ld
c4i;Uj;3cfdVNH-S_7^>=lh{p3*SkAVGKX!*(gi4)LTZ0X-4Ntl7(b91{anA_!cy%LF-xV1wVz^uzcSwSUfgMi`rjw0!!kbZyj`f;M^9w0oGGH6vNcepZvAm
ZoLzQ8y6n}OSlcjON}GOL?k5D7wP@8*Ag=d{`-D<$5tG&3Qk-~NF6$s?$boqsK!y)5|AcsUM5m=v4F6GknY92PdNGl>>+m8E|g3G)1*muIo#DVT(;)FKO7Ja
U&O)S<{45KG%nyh7>Uxm)b*-4*pwW>wn5%liTn!}gW=tsqy()ySb*3TUWTSwyrR!v)u6xfRWe_hYTJ<*a~K`_hC=sFq0EZ&(s#taZx0B~fy$x(6qNnfU9LPG
+&j#|tay(c)Uy!hS2x-cS|8bAaGvg0@-TWrfqg2Rmb*P22NyvpDUDp}9IdDk9$u$kVh?|T;7R3+j;d~LBwWBf&9oWCtBC|+`hkYg4#2mbJa`ZCX!ndPnrM^c
7cv`+dg8gq_A;8k-5hVE%j&<-;Ax2g(r;i46v3<bB?Rvy%h(1149IiM>o~(QxZh&>qgLF6(yHiNv)0jMu<4$HbXL+zepI2VCP|f?*gyTH+b0+*u{V-Q>h<}7
X~w`ET{N066l9}trlgDeDO+UrP$4PQq>A2%q(UfBR%2mHH65N#8@?4;E`P9BS-odEhzl^uAA{e1RD)DLNRh2H_XlMza)_b9JNIHwHHUOz$n&q4>BYOJU3ZM9
&%_W+<a+huW2Q%L$G<pI2||+t?Ti2X`XR|Kxk=qpBHXsgo9D4)yCw6bLmyo!>l$%G_54K722pf|Mb&zg9ZhakWKO3`z2uskGgb?S=}r=}R&CL}OrH&)x84=a
>y>HTbox@^^LSoW-4ek$eEP(&igO3^6fb6CT$uMJH99V~zWP6l=f$s(y(F>R18do54EBKn&fS+Ir4i{opbVB9jEkdn0s`ekQ~~>e?gD2jSRIsMh$*1|<I-mU
m<(<*YcXjBpOYpNw`shuh-sJS_TM4mxNPgj%)eu;P@wevK0$3AWUt{HzSP`Uz<*hbn3(`vOC#Dz*z3i2KyXFltuc0+Ll5iF+qJ<88L{UiSYQ2p%Mzg%ioa$}
;vpT`xc)(@|GjwSivAtyfE*~H+UXfTWISRL59{boc!IUQOIm(}2OL%-OuL2~wqwvtz>GWv*YszZ3{v6r_D|aue~u20Wc%lixBJ))0*_0td9_7Vd=q=jSlE5&
PT*lh#fvxjFPf+IOXqQJDek|f*f(^{aL#Uq;)UdJjYJL7njW)UX+HmvG$d>0*s+PLZ!pha(=#WzUY{2U<dD1t7CQT3OD~o5JeLic6`jVutOCwNE$TwuM+Gb^
$MC=viG=`WzBBWrz_r}7@j7Y@ArqacMoUr2sOKcdl1=0W>f&4~bu4$btG_neLCR$6TQwfsrEd7@8kIlg{1>NNhq)U7O(CbG{_=A|lcgShSNlGZuG&>usP?%J
NEFqYRuT%1qsQq{LhK<?)jk|L%m!bA#EPakpTJP)i$I`h3*~KRjKop^ME=VA=FVvjg{fGZkw?G#&#dBk#EqJFm1&>8Lx+WP<ppe!A0R4P*0)PBYJLr3&L>wL
Hf4eqifvWnjN@Fw%|9N-cwLu3Dnr=ETIkQE<?n!<_JkcAV`_Pz@I*0^DY;dm+@9-nU>Z~y?qcFg3*rq4^`)n|ZgJBFk=afM!79!1FlvT<Fcm1A{YbUxNEXR@
9+{*Fon~QxL6Wqx1{#x8po%(H{S0}I1sS9-hVXIgbk|4kHO-s#{>3%uWjKVVn9uPz@UFCtGE?9CD6dk;$UgMqm6<s9;<2}9Ga>SS+f=6&*xLDP`6-3SBG-R?
KRfRK48HgG^XNPJ(6m&kODuu9j=kA#39fkVuQJG#R54*MWfdhq7R~Rppp_J3`W!(q>Nl!ufRC%On$G>d39F73dR6`8t7+uy3FDGLNB>hNzY$#&t*^q+@tIX~
aOdYKkQHjjd9<4aDDdO35SWhcihScM30@$o;T)H29%!#{6S$&d{*x^$fDZ88IHHK49yOZbA>1Lo^O4p4```KEXx<r|Q7QQcpm881Add#w(58(Ujv*j0P(LXa
q4Z^^S{z*SS8pm10Ib4IN=5LFK?qfT<HJJ$ImToD9NZoxmJJ7ZJ+>sYtt09*^dH4FhQTO_hX3%WY+|E5p0GC(DU`yaR?$gx)aG_*NBofc;4IECG}LKRr{XN+
JJ2DaH&SJHSF&Vix|_qn%-eb00MznIWte1N(=sgYDAy`#tP79tB|h##`jLJOPPQ0P4_8dA!<_?@aAV{ehBEB}6-7|AFdzqwOP#C4Qzqx#2!lCX+aDUv_ultT
eu)A{A?e%L8pcUCyMWU|Z9I3@I8enz7nk(1=K;8_HFHCSH_N!itbyme@|%-mtmYbW6o(zclX+_FdlNR_GmoFT`uRga>dzMK%5V-F*%+p^>;Ab2TgF7DzH`pv
sSbtUopDg<>yWhNfN!3O9z3hd%EqTzGqh$rC0o8I`@Kp^`R;=NO$A~s8hs>aE&9BnSvJSNQ&RitBK|kN6*LO({%U6;pLUHB(({6J&S?KSNT7|3J+%59$YAm_
S!JD+&yP{EySIQ5G+eG-MdrWq$7@%&^c^q2N$<>tuRTugw#i#jXaRPU#*t?LLv~ARQ;xmuCJyf}Z1)xbq`N4ZQa%KDLLnO8UCoFRvxM$^MdqLcq;XAqDV87t
%x!qg{KkrORXMvwOSBzP5P&sPqnF9XCz@~fJp7b>4E~n^&(v!XHrWn6YU_Xvdip~@JR-5|>UYk5qrx8sIRbti1ZaAr9M%O+BcdQ315O^ROZoztL06JG&FFVG
oNv3vvTrZKj6~UM;j;U<wo?VOQng!=xLQ!NM~t`9{~CKlOS+O9N9-|yMR6qz_$KsUqPd$rw8^W@*D$^wKG14xsmLJa(X%|18kaxP_@GQF2%zAauxXG?eOX;b
b|z{HnhAN^<~+d9J{)giW7iwO(s<(fj1UgEVXJjBDEf}7au4W@O1O}cMg<;4Bzz>97SXN_0TK;5s#;aY2!~%k@j$$>LgN-<b{V87>DJw+`5QLPRd6sh4ZleK
oKO02y6uH7m#&oPVc_R(gg&7Hy3~0V+dWn1BV+f59tsc+TS4*@gTz?qY-ua24q;1wBd}6N=v5-7a6ho5%9^Y5w~i-2Tij@#>If40DVTZRUCY#=y1&8$3amVy
OYIcxS;sqa(?a}+>U_F{ZM%+h8X;3Ve6J$WHou633-3z#XF7as;DQVFI7^K)e>8=oBw$vU)aM_!9z6wyD%6C7gelWJ+MVw7X<hqm`8VUgtP}6KfyoEnMU)V7
U?_{VpShtgTC}?_6(@x>hp4srr7hh|vYPRUO#g_fTREu11(m&YW0q?XtE<7|u5`TZl-w|w+hP!v^kRh08=V_FIn!<>)>hM6<Mirgc$t{-vO1XOwD69LO^Ipx
Vc;-YfF?Oie^KGu>}u|DP%1dAvQtXz_$YIz=xYRfAN@DySs3(LdSYO7Q3@tUvec3XT4^3BWw5R-Zz-jS?2qB8iy*D@kPa0@2H3@BXUV^jww#Z7z_}cV#I;wU
oD0I!<y>dm59lDQSG)2>9<0ysnS-cmhzRN8vJhga8V>}o>XdVX?L*@P6ml8FZTv`(W>7CGqn{_{UWUqR+Z+;EWGBEyix-Q4>YfZTWXBhi2JtC_rU;s0XQGz^
!!cd~KWrm4>`kTTIz3r%-=nZ_{z;%~C@*%b`3SA_E%&*YM&q<lN~P!P^VnmWRQ7vS^d8|+JR5%Xe*l>cKD)rK<#mI}(t}Cd@O~LYD@ZqAD~COydeX4ojyes|
R2+={Y^Up$+ob%-oio`Cso!QT58QF)!1cj`@NOqfw|5zSDoD!0{kA08SN;KAP?(xYy1pZlUCRWLy{Y3A4Y04~_;5F&>O3a@zFWI@^n%V)DL8!6_1V*rd+xNZ
>dWkY&=f!iZyUi{+F*du$cD(HDVL$8SMj`T{!LHi#s8^5smULBasJf{PxAln{LPQeg`cM3Kj)JEjTPI5L??q~1!I{3F`Aw!d%b%ixV<I(=8i3#r1+Ue=^&e7
Uv#f8FZTBj1<+=SXuzQJ27qf{$g8>>GE6z!#Jv48G~a`zi5oPZFWMXVsEapJTz@Cl(`Vh13=?F){KH=o*sj31yhigQ`+o?p3>YyEsVu7pN*~U1#CgVo1`N^P
SX6oi?}z!Kq?y?MXOgm6qKk3fef<h<tduqds4YREV~B=)0kj%<77pj+JYeqA3$ti9V{?$GcG)AFP_FnWWlfBn)n`Qxq_=<fW)-uKLL8*W8-~b&l3-MCJ-mL}
i&AHA4Lkiv>Tw$MvDD8JNTqs=Kj=y8NfHizt$G!eDTy2g$QTv|>KhaGAY}<S8Zn3I4Y*5uq*p?I1Yb7Na}gKeYh+w^tqJ}j?dv4#UW%i~GG-9Xpd!{u#{T)0
>?$(N7G90pJKC1|8`#v$e;~8bXM#uI*b&XFR?{`Xkk?>M$%|JG3`r`w7<&R#ZdN*3s$utrBB{<u>%mBLzWrDz2eI^1$bH(y_cnqvspl8GxH&ZTW%-%`r93HG
Ez&H4xM<ugjaYH8f6>x!Z3T*Mq>>-7*dY@s;@}CF!&8hgg9`7$g?cCRWBr=mo9jVN*m6H8FVM8v(G7Mh!|08+&f5w0sTY{(B^v;JF-JpV&R=viEUXLJ@;_D`
9}qw}t9XmUIkv@fD7Sbo&C#$hjl}Xevc3WWZ46MdomU#mGFO)qUuzG<*GCpR_+YEuf#8PW1^v1W*@l|@uP@cOwq)58L;EWdT8-7Yw6jAuA8w8GR#|+dw>mJ9
4-~CF{<@a5iB6TCkRKJ7lOZ%M$ZFWJk${tjGp{r23L<9s=fn~Vy_!qgE?>l?j4?ZZtfF`BHZ@kNXm(gl^5A3TgfP_`(&=MmHFfMp5pkqAQ5|2z65E$w8%uY%
V8x%pNB7E_z4|J436NFGl5hByR^Wo7{P|MLu!UK6Jm}Gwv%W)q`gezXI)(NozA{$~Uq5ti70hwJM0D_%;mc)F2%{FHVV>ngQWL8$sI=Kxtx+ePAkK8y-P+4;
{C)lruwmkv{bq66qdyTJc8Ve7koI~)h3Ri$Cu5w4Vz`o{z<ffi!$43dMP>a<$B1kR_YEu}wb$<kONsidy6f-e;=Tm_c%<ma9eR<7C;olEMO@|C7KW30HBzyj
3%5PTtwntr{g?b`WZWxVT|S3EqcvX%A#+K}06I+uM-)Qb+SkYREcHieZVgy!0qFi=MzHv?CrP$wuxsIJLVrmP>JMf~k%uSwS?w@tL?i`Gm`XaZ7tb4M%W2($
E}R`G!7~FoSZv;We?=StgMGU3kpH*!=x}+yoZ0E6dfVwoZhf~AyDA!kgqz@5#!f^}szY)38jz1Qgn+&DpeRve@K9?}W`KMub?gI)uEM+VN!vnTw{@s7gf#SS
nH&vH%fYET=!XM@HM7=L2m?M|?L^xHrDIT3(cqkZJitwkxbV%~^DJg6#f}x>+3>d*NtChq`RBn$uQ#UoIjxF_e1$L>=z@#zq&jGGtUzSI^RJ$5HszW<g0wH@
Viv5(s;iyId#sCIvwH3;k-ht6VO95$m>V}VODn%7*QDCWxgwq=-V%zBoZ21Uh_g0``1?CODxeY(=bBGb8|wCN>IQMABW=FC*)X91u<-tXC_@%1HUVwlMfGb+
O>$G({mk=HhHegRwPEal>|^#SIZlssPvQ~le9D)HBofF+p!`gK)864zW4;9{ICZW2B|BlXY@s2iI^H?z;z5ZUbYZ*F=HFZDJdd<JdDytSz7~-Lm(m&#B<iLQ
v)putF@NBq^H3D6>xGoFvI;9}><NOMtjZ25DE=pcB)8uCY|A=TUPoNV)%TQ=H7r9y#l(*K%t@d$aoGt|obc2`1_?rmiMsRsMk6X7%9oo5JlfePTUvk~hUeZU
H(~g%mI%R3eQ(J3Y%AT2mjpx}Krk+938SF`@3R(`hp7Ir>4s)*%Bt)81$v^L&AlXI>1fE&vMiXc&11rBWXc5f*<LlLdwo$qrcD!bMSgZ>(T+FR7a?7vGV<%0
EVHyNw;evVz(UP$Vw@b(xlf`_=E5INw%qn*e=tnY?~mILo(4juP`DAh&q#ckDK;$SJ&AXeqHjCxPUn32&*Y}}ealWE)<V9-ijEP%pYf#uH}1|ln|tBhjU7#~
Y8V3tC&AUET)h-@IHW5N(|s2%9S4@qne1B%0CYVqC%s2_>lBPg5gsC&jk*8vliwZX+E*H@c67SW$n-X+cW<VKrdojQ&bOh)!e3@3w{Kr5;y!}8RDI3ky~C>!
nycT60d4PhxsOhS(sL%s!KV{b)gNQe`{FIt!I+WGuQH!h%0#}8uP|6oHHVNy1bHSwC1jf(lI;@d!IwT#_2i)`QJz%{n~-L5pCvU$F?lKE$S#01oK8tkV5tnw
Y!{w)2HX+;gLl?4=LQnJ<}&_K_$Ck@4pcgc<0rlZVkWA&P67b`?^rtBjK|fTfWtCewO8)-Na^mHra|>SpqlD}0syxmh_YA!g342mLE{<O0pTXr!811o+lD(F
_e~e0uBNmp9ruN1`V-D5K;PS~==hTl%cMLohp$KGlzoCD+36J3;x8O4dl7J~S$pM$6i#tg8{{5Y6-{BjevIA89#0i`=VhwxS}VA>X$jN$8chSF_5$pFf>_-X
h+w*tcE)Q_3NpFs)x3!(A1>-m05v$JkkBRe(UzvpH*VSsK9O!Bx3OD7v&{@k97%N0JBod&FaU-L!gVo_9j>elZkD`ZO~qqarKgUj5<cJYMrM5W%RfN=Uoo2I
5o7Hxke41sc1i3BvD>dy?yRKyF7P%;%jP(D?5VJR`Q6@ryq}NxYXBJ8P$Hqy6u??LUtL`do(OKqbCIKppCm=zs47xMA@==Cs}Mb)NPA*0p$(^PRn7f}+UU}T
CI(K~S|PQ&UJz#>Gwp6-LP(8JSeyI3c@_Mt;9^pQWTAqweW-uroAf;TTT?pzi^n-DUXpkm@Cz>QQj)kGUDc5niI5o7CwO5tTm%Fzt13ZhX>f5txza73<CiCa
Dx)z(*t(#{-{$c%iV>162uig;C6pE$Cpn_`-Y&yuPS9-}6GgHLo;oD-23B1VOw(@0N_9f_R)<ajnF~Qm|Ggx&hPaPZRllFbCU8vnQ+Td-E<I7#S;0!Z_&$wM
xdfOw|7sL_`PyQ!e$3A#)OKVuXaxV;_9^n&taUTm@Y?mz^+E)T>Mrscmw<X{b#gJcN|sNheY*S(^Mer#gwWQ(Z}$a0fBJo>yZhDjV`N;lbfOGQYxH}dxHni2
@{MYYQaxSBs!jHCeWF!B;ejwIaPU}#og7-Hn_WNR-|i;izwVZPUZ~Q0IeYerqdCmvEiy4;yohx12$(gzvB=dLGzDx9Y!kUgW%fSf=8SsgECq_|7+UE`R&2EM
5AIR6!@;+&9&j|C))h!N>`1T8JH^*XeYiT3&m?VkM9poDXkZ|zEKgQ51hxd*&KJm%f_LXaI4&2`Yug)tGD*19Acp|4@rygU7b+<pjJJ8hv7F(32i+GZ^F5Xy
mI2}d%)6~?JRBS0weyN^*}d)Dm&ugMpiOl%TYtzTuk5SxMc@aDyplK{o?r?ht*p+wbiJ>Tcjx*0UWlHehZ3&(SeGoQ45Cx)hyix0ox|}xpa(xo*JfVm-J5@8
`ifyD30TxItIBmIJrBtZT9pFlGgm7-ocOEOm@esIw{X7kS{-<=<Al=(n=zMbj@wK=h>c7#>5DcNE0Hhp<H)fAisRtXMD%!r|GfxIFACc!ySr5AVxYRM29ys4
NQi+7VVA8;7oRBzPjwh7x({5XUu*oKF=c-?=JDK4YMF(A`;rvYU5E00`m6s7%><{?0%}`|d6ueUCoG?^qzBw*!MUk_OJE25rzS!$AC~C0Om}ZMhWqb-m|k1m
uHI5c*yt(#cIE!1Uv7|bRfto{@A2|XS%*n5XLvj#+OX8$eg{o5&UF`nplonkIYR(l0_J#h1OBfOl0_U`1&!`DcZ#f0d|?!~474K+nBl#i-w5K!{N4do|BYX5
<4vn0*6HDf4`J-#G18G(-|i$7#CH6)wa2O?(Ak_&hm2w~<SraWl~qGmbU5K?G3x%N*3|e7I+STX40$?8R!`0WpQ<(0m_s%T3P=M+f7Zo3r@Ps8uM?9uNYfzY
qvZj+-7(7U_%nLR5JRzSWJEiz2}|J-OxG!%p0@LQK`aRo_zGFV?!ToNP@mY_zdzi;JqgeM9#5x{=oiY@KZovO>Yc?aIZVu{)1;5<Ph<<fQYkqEd8Uy1kl13f
aw&_Ni-B<}djuB!<&6ILx=r3;cX6&JQMD@-)yudbpQ9IUn|iuMF6WZ4k^~&KoY0cksJMGMuG}jX4~Nmo93H^K(f9Df%#N3~%f1VA)Y4%i3NTqpXZt&^z<zi@
w1{mmv49&G-rGW%3lZ>7QsT-amE_gHz2+g^7QCCir?39L?#qx;aY7V-*KticA{b<GRZC~b(?c!(g`p)aL_KX3u8{3No_94Q!0Rpdx&{J+-y*DuOear@>+VqY
gtaIH??{>bhi<eU!$}6ptL18UB06HG0~ov45{V`Ge!v*l$78-`KWme*(HttJQJ7Y)XM{5@xoEtSTK@Du!1>1EYQXn<x*EWyW$WR|G9}Jq{zma)<F*}M0&-h9
EILe&4)F~yrG+W({3>3E*=8lLR~yZbD-6~6$M<n|ndK&f3${Uc_K=?_=bgLsxIQ&7=Cw&{9@HWox7RmKk$%`}2kh}Uj<Xx|N?+!N2#vZ+WaQ7`k&cjRR7r(M
#M8{#;1c79-pNgJ@U$e~qFT%_t{JqkeW)oh6EtD_iHzSBCY5L&-I$)$n&A)_`92XlM(DMU`~XkohfutMrh8(RnhF+-@ew~@sgdj>qD9MR6R@gmZ<~UzgNGrI
7{qIDHMH|=jdB2-WlE;MC?VOq(;up2HE+dEV1ll%NvIMd+@fPl1)t5N@Um?~`4CQBDnlh&fU@Mi`EZTWo-aOQD>hROL$b5-83G@rd+q_x(l^;E!DZwyJk#li
0Rm}iz!BY~HsZT({R<r&F$jI4!mn!@TQ9Rg(HTi++!!bPt^7cb2S@AtV#iLB;c78Rf_1lg#cXY{{DMeI0XhEE)M@Lih(E;wTPSy(Jb6nXrader^$?KGR%STu
+mIl94qhK(_euW!NRETu+7n+X2wo_rnFPrBFaA<5q&~J2L}q*j8n9mundo^ux{Z2%*2chHv0?)BfrFT?<-bG;I3Naj#T?@d_sL)^ueL};5d(?)?1Osb#Xmg(
GF7V{y7yO#StldNKIHK6e~L+ER(2=p?1Xb6t7vp_0O=;iLM1bnDVqWj&V!Ua$5zAS_&FVB@uQ8XCy%xL-etr530*Pu`iGC;oZXKe7eF=dt;m<!C$atdUFL>p
0tY33<5bQF3q*;Yqcep-u=dG_;Y_}MX)oBz+`L>=Z7{7f&E)_N>17+1M*1^Mmmu86rNDai(PcKyAbfdfC6$2`+0!`Xo`|a(KQik3h3(U;F89Xw-y8L3)qR+Q
lu2m<_Z8+8KcRCfq=2Y~FI!$TPg>GuHVhtt*Y0vluliKE1xn3Fsj6t)0=pPZ;~N;wotNfy2DN$-!iafO)mfl}JjsSO!dS3cvz3#DU5&S*@%Yw-CO^@ADyy>Y
d{lvGM&oVeG;dc|z~T7ABY?e($~Qw<5MM=g+`B=`OB56}VY*XV7$<i>k6g$vhKO=*ggQ|x+UmN2mzjt`il(niT=vAF*of{GvUpX_dgg9IDnzXtgYsm%bwIRy
SBBdu>&T=xWxwf5>C_zZSRGWs0U^`f<X%E-AC}TEc&~IG#tIV1GVXJ8Yw1$zYH-4@2kvGU{mdJ%wlD4PS&7$ylP&i6u5}pfjMsFZ3Bb3u9g(%QA}{X|aUDQz
dDLN*;2s><<uZ5BemY)<LjUX~tF*OY(P{-4&nB3X;gmuU{u8`X(af{tEHUIZTalLbIGAEDd_{TCbX}ktywt&tEw@muTK==7Bqdct@Gj(!FUBOC7@Uca(CA^W
faO<!#6EPd5B_Dr@{aKZB+mxloU~rxR9%?#XQ;Jg;-9!ksQ-mD=+F`xww{^1cz6s1INQwOKnW~!`2JylU%eI0@qenOp%&S(o4KV-qqrozy8pFe8rmI70X}ey
Oqhp_!W*+XX0f-A7GT%Yaa0LRh>kn+abveAO*#!I9VNfNTueXXz@VJ=<z7;)#s5|;V)LLqv{8I<ey~;67{{A#Lx!hzC1i{J^E|jlEq~>s@LwEvALFhbf$dIH
fGWNzN|E#5y>1_;Xp8}BRMqDif!9UJfFBHYTLwpDl;hWTmE$*-ID-{^o4QN3pY{f;5Ax(ui^}-B-MFS!vs(9l;&|cTQ@uhp5n45=(5qvxRUbm2y%(1($OhyI
^J#i8al?y5{r7wzN*5W8DlfYI^taw7IfjSzLP%rZn|LbX48O|9Pqcrx-39e_2V2wGC5o-RyIX%Ns9aW~Esy$J2aU5Ch}z*nLvK+2--2uHLU`8;sisHBnK4V`
Ny3rwpc)8RZ<!@~GVpEE*L?G94Vv>TdeLM#tK{|a>iCX;KwuRKr@{ZB-E;&hZPo6EG7RStZBFR*d@6r$BH4tcMrm<`S>+TrDfQ%FIGw_8H?$4uTPA~pg}_vg
tLX}4s42XxpFWm98qxpf0v%(4pTh@@L(|G4PsEmT&G)TeH;|qjKQNcoW}c&2cR2kD0=F=~V*Ej55e9re@+IE<8U~7HlMA$oco`ZFYei5lqsJ^<b`1l8#EB*e
7w)#PT)oBj$4h|NMQ~A;=`ieV;&~twrV!^z=#sW@C{;HLkpCE?-EcGgP7=IGdfm2>c8b$hG&_6TwL}`W|B0;z_9-`^uNAI)>=Q_K=zuPS=xO*u2M<dBDfu6V
MJwEj^Wnw3;9-`=b!Zm*n>+v>vzDWb_MHELU}3FT4`DOKd1jLPg}@GDky7n-yi7wBGJqy2n+<D6NrFeJn^To46&9jP*{xc{aGG_r>(ggAP>&hZqJJ#O8xrw{
97fF0qyE|I2uq>_zR1ZSId?E!MRF3%>1SlAr&;j2@(tKFjD?!exv*2oF<UhGUGntD5ey9*%gHMvwF)f0<ZZWJSq5WcuvJby3TLrH;oWkM?Q?$LjxH2Q_HDrH
7n!r~-<J(<g{u<RC8A5CJK3-)at<pidO{#|b;_rFdYzc+AxaADAQx{pC}B``ODGJ6E1yIo0lMH6HF$*i$65|*1%~&@;WIhq;JW0njYkU4v>#I=gP7x2Mv6@B
PjdhnPCpRDt)aJe1WNwu0fh$6CAoK6{XM<1G#WJPpELqnL|$uvHDUPuRm9vKLtS8Hi|(|Z?VZ+E_MSpv3(MaER1yD>?{{{D7~N~ZTtJY@2K;Sgey_NZePp1y
AHTKlIs+G=A9&-<6xMK<FD%X5a!M|#SWArYiU>~rU6>(HRTj}>C!b~`D*tIvJccWkX!OIJ$jDdO>j~5WP9BgiM1emK#EB7<UGPmurrC_<_LSqjYtoLw1|som
_gGP5@b=hX9=QK2xYpRSSm4shP8c*b$pz2U<#!j<M%;+?6(J71E&xH93n;vEuZRJO_M-3-zhvRNUbScGP$<w93t{8)<z!`I$ZWlVImd{a00000PqtE02o%Zr
00HsA2BD!GW@aLnvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/treino_gpu.py')

In [ ]:
# deteccao_r5.py (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/deteccao_r5.py', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;HhLdL|p(8h(bq-BQwKy4>SE$m+hy=@aif<Hp(x6uSQedqEo0ysxn_N%4%ns$4kyqM%f{Q8hcU>P?Zi%CUFlOyx%q6
<KX*wM+ojI<~Y~xyJd>0y@RxmSTd(@rP|(<Z&AW9kOu<*`m$&O2jikfTq)qU_QHJ33#ExTKx0|5N}g)zl%2yPeM6db1rHNsI-mrdPrT-;d!y(=YrJcz+y*!w
Go?}tH3*U4f$lf5pLAQ&F~+woghcn(c!afk)qCLDb8w3g5a?TRrFTITtwYOmI7kOm^dyqD1h%wpB}ziG?iX+}!<z|ShRx$F9oG}Yu_U<*D&dLB3Ob{-qPIlX
kjKO*m<)%M4M+?iLhwhA39*{Vhe%n+K%W54*|oj}3Hbl}_5;D7PtG^<?QZ0q8UD5L;6}XUhq0QV1+6y6qpCr@)s}DBpi)F+C*RJo?xK~guj%O6aw{daAw%UF
1cjkme4KiN-~90;+?nuL5dYh~hCj1-h&5^GK6@#>lpOn_&(EF*yky2J7UG8!0ujcBaL4lm@|yCWaSY7?1!83Qsg`QdzH5#&A=qTw4C@!&7ZJ8wi~~-0m9PQ(
VaGx~*j!d<dn{Rby9i|DN|z7w`R<d$u!!y=EZys)#m{}<<olU!m89dixB$UdihD?XOLv`w6I#JYs;=n-`0IG7NiR~4Uxy)oGwkhx=(~@~@^BKf41M>hFYMBY
7#)^x3H>nVNl^4vFffg9WA_0vOCwz+o+|IZL@ckrM&GCW_VoR!b2x~;4fhYpF+~&-t{BS}c{QL?-|yUDZ#^ZS&X$mcI{A&paXwn@jNn5u{0h@7FVd7YK6GtN
u_?u^ikF)v>$&Xp<!+np(m`JQ-xb&uA-VU;;P_~3xUU|x*`BP!^VA3<6^gr7fhilp5112e^=FH#xicXTo@Ak~vX1d7;^3ry=b&`B^a5X@e1M6bo{G-)xE+h$
g_NH5S*ZxzkQOX6ZZ~&v1E*g!DEVuYKCEnP1Y(w2^vB77H!w)ga(fK}?`r?zl@X>5ARl>^@OUs9VqC(XbAW&I;j9yAji0MSE#M*<{Tlm^npSw=8W2WsEvElL
3y{EZj2!R1X+pmJ%xzbsC|8LUtHV!U_S(sd@z-{6Z!Gr!^Macl!GW6a_Hef+#vKU|UHGAxi3*{Edm}Ns%5;M2A=VFxnpQo?-GGZd*=-)I(dqHBPSGzq`FhAx
7_yXe9EAhZ`6JZj(c!rc9~{aH19TLCajB?l#lLnXO@yVK`u0<4Lz_H6GRR^lE4Wnx)kMVilUrE3kaBs=7Mo_v6a{Q&zT|nc<SZ~Usu^d~%S&9azTjrvcX+I;
E8klnQ$sMTNBR8yH>JS6DPc?~y|UzvO?WlAq;`TB2!myy9-W{aZuPUMU2o?`>C_GvAtWa|z6!<J{WU9|(Bzu&9Jpb}p=%ivlf0lsI}`G{dS~xqDYF!(IAH^)
V(>WOa58eB3G!B#HRC5XR~w0~JV;FhP=h=p!_+F|XC5ylV@e{xUx1y%K-OSzSW8$zzKLq$L3Roro~U{Z64BUuAL-dIA9_>g)>zvVTY|NkkUffJ0&BK1R$HFF
hRF?Tq0Ds%n}wDF`7ov2&uOR6uV$>QY&&U$aetI7&4{t-_5xXy0R^7;kt}1BmlmvUd~BBp;YX)PeV-_DWxaHL-?}PB%&kprWE4C7ari>d34cvjxuGoHxhBkd
OwfQ(8r;W(U}st=ZYwL#aU}*!h3ssnB2A7IRsx?lJ0BXLTQ07M58Irjizu;mWCe}h=^1q#Qetma-&smnzCx(D_Mg3<Dsf#eXn1-CeJi}v2}0&@+vO5GjL&RA
+Dik=sV%Cj5ObK<eC8t~{@I1;tYWrY=+{{vyUWZW%Y{n1ZgdPH=HfhpEX5j1KLY}2GaCx4{%QW8y?93GbCBe`^B2G>?_h}hZte=NDI5UT1WL8zr3tyyDfDB5
i*~0uS*)H4{}!?m_De-8VTO}h+X`nl<zknB8->nRa>0m7koOPFmG)iQGOR}K4I+T{0}~LYbTyvKb<nkhh_|DT5x8~L)~6W91xf=5uwP9C@dRiYhgp#$Q+RDq
*#U~<1pgIF#)SnGXu#8Ir0QK1ywfI!o3gm=DQ?fl8&X7cJ~CXuZn}NDg5?`kL2ncA>Ae8j^5z#ykKEs*XJD0&vE0kGa!|Sg1FDczah?EJ;q-XJGv~MrIdYV`
Kv@b`?!m#65!>AUiLbSGtZn}ML|=t2YD&__apg^fy^mco-^Ro*wtQ1>x`W$qxO7-<ig!`Yxy5=WIXbytVblGta?5O&AL41O>JBI<e|Gb>(@)l4c`!IrwfVk>
VoChtD#GNCKu|=3Kl<7NIhpvsFgwmaAv6@(4xD;iC-b@vdY7sxLsagY(G?Z1>b3Ba#ew4bh-a#a>VQw3HleG<F0l1$I??FlabFk23Qv7Edf^&?NZy45kM0&_
anSPMGayik^{nHF+@x0d_Fh+JG?Jon$lMCTLH_B6IY$pC<ZH9h>Lx`K=~Heg%;6RKj(V+l2k-I-^A-FlJV&vijo77+oM|BGeeFiwn!piur8is(*m7G{RexXf
Sz!t{?1B~h+XUvplgE1{Cv6*d%O8Swt_j+3HVDQeh*6p<q?btobbn=fE*l0v)8Y26S8kcT;x27HQJCcQhTs0sCR8;(htO=5^NNJjzr(2<0oa|C{SQNPAV|qm
n=ggA4_F5G5H3@e3=wnDI^koCt?2&EoRef|6lH8cb`c=aVF_Rn^e({4m%aB*3OP029Mk8VGkcn&6ouw-;x9<=%WdtLA5~#+Y})~4_O;A&m65oMsii4E$(W*9
HSPuNp2TeNB<QXDEVWAM&HXs6R=wVi{M9T)tiBN#y}3#oFw-gCq)u}4%t+*<_WVA6jowMh_3Q7?|B80rckD+t1$b!x(I@yFjEH6SwQJFxOmPYjF)6Hlti}-D
88FVEfQ6ar`b_QA@q5OxAn)clLjZ{d(6Z+sT3g>WIPlQcw2q3ReqRm|GulYPgV>P9aAWa`?mHbvv5S#K4xaQRJ#6X~2NUGeKS+<AGMODa!z@4&#$F0nkC`30
%qh~~hPodXn@Zmjk_)i}GT9%slw(Zg*pF{+&HilMgp@djeRAja<_Xb^Rt^=m&pxoFI@dy-Yxj-EW1k&|JJrpWxhcDtMw=Rmx=;G)mez4R6BAQL4V{|Qh2~@$
a>y&}Ghkk>NtYr7ws1K^XQj2q?-OF7s{HVy+EvLLgE^pJF(|YL(UC48#Ejg3HrA9D=1lmU;)$+QH%FQ%uX4dtjHhDuk;?@;EZ*uP!u1LP^4Yj6`<6$pJV&03
ty4h2G`^=95D6kM@&ClB$A$A~A~s09sM(ywroya0w59`X-l#lRc|@k9K&FIXXut*4{yPKMyR70p0_h@_X~?%%XAAzaR}_5BZ_L+p4zeyLECsGYSZhO49DC?(
`~j0X%{}B*crSw?N&{WQC*eBRUcSCJTyHTXWJ#W~s?Wh2yvP_20?kkXD8*eblU)h&$JnBO-7tbmY|1vJ%<EQ6;wf^$uW^KEMP)@)^FL?qF;<(aM34fJL+iO>
$3lpw`c+*jZ5-1%`D;3fBCZT({OL3SO5-z9u#tE5ut@p%p99(IC&ESsy3~;&i0fq*5Y*^c&5NqBPPPZh)<|xaAEv5us?9?2@>XM2P4nu;j0y3fNF-wS#-tBE
ZW7*5l%BVMW>qmhx~^kdJ(byJuU>W9XPk^18m2;*)&X59MF*S8cDM++(iZ3|Q-D7}F|=v^Ui6@SG!14Wp`^2J*?b}MptV#%=^U4qfB~I8C)yVpK=DGoWKx{H
`o$Q8=TviM8O{Oe9&!lp>k`5zZ!0uiPF!h2M=^BaxLyyBc^N7{6ph2v&!(t)IGp=(@ySUPw*!?oReurls`W1IPM!6LWNN!0(`vb0m!%I)H&-e5hv^S|4jhxx
)-vbaCduf&wN#NZEC@}G_^~?+ri!4!DzQI8&?#%&hgBZ@6bpDu)a)mu_+)5tKL=PReD)9{i{d?C$qwJm$&}S20G%xWh`<cTs<Oa7t9WugA7y5;$qn)@u+juU
mbZVC6KQ~8k`!>6_0Kl|5x_nEz>wD5OzgVvxnM^hwhY+G%jAr)UD!$}KP>{J(9(y(clHDFVsgrd*|`Om^&s6Ps!J?jeKE|!h!~rrfyox$@ybviAbt=j{^bA9
>$xfrlA=bu)453#raoO>bxG#0%JYsbiYncT7BR*;qIVy`hGyNJs_;PuMWZN4Hmr0Nb_#0rZ3)M0Y9m`?nFLvT0Q@qCk48v*H^GcpTXN|sLIT$NNj%9Nj~*#|
5Ui<|`eBV~6t<)IjBtz=bi6xKYyUv3T-0eC5Ymt<NcIg=jcBobD0NKKHi0j5GWuVa80{g)Worp|u>SF9i93n#w=uM+$^f>8Af{KbiPU>vL#tH|V`UOpo%fAn
$)>cFDE(Qc?X?sd4U&YG2I-VF2v2LtOlm7o#7Rm1$9+?5g_Nz_7+%(aj1o?u+@fe#6o9Qq1sX@beLO6?(Krx7J~)N)kFjkFlQ5%M1@6NPOzn`(PJ^AkFt+8H
VZ$C->w8s;*ZF@W%gPQYg@SA%rUsJ(nKnh}pQLx<cT^Jxoz4}<1P6&d;FDS`EzRYhqYn))Xtx(t!=qH@mq!eoHF%7oF+xW}XBrof4IFv0M%zFVmHRgfh7cB{
=;}MNI86Cb*ETu8xx44Ho3j*SehXt{#F%9>3MF(D=lF;_RA<=4G^mlI85O<tkKj%)Pz$c>+igX`bUSD&n7Hnq4WO|Z`gonNio)aHN69u3Az-fbZ+_&eI#7k-
n@~gMZ9HdI9ZZ8aoEfA#srBzH*)(r?YBlbMI*xY+WY%tLC8;yMg8ZFPd&siqo)a*@JL*^_pyP1;QDyB_K9j9u>U4zOcFj<Zq!OZbqFc6$TZ;53g*0my-GYOv
4=ck@v7^FL5oOVBa8P==*??r|Ol#dD8fp~$3;;~)n5}U}m$0q(hNo|?LbODw%wvsV(?C0JLFpM7fE}wp7<rBZ2`*KAEEG~hlpNTXFBx52JePnM=nAB+Fci~u
BWzf}ONe066^O*$)3CqK3t<?RhCIf?2!392AuzysrHGVdVpTQwn1WmAW^Hs?P{iZuv-HFrh5WGw<CtBeey5paD?lN@g~lGLf6`I+Cj_?H5=4^N5<-Fw3Az73
G=_~%F4WmhZaQ6IB~Z2J83$w-@~y~<wX@t^ad<GGG=E60rFaOy!BxZn2U$53_)t!}ZKatqYnMOG6@bOWONb|hZX4fFpJ<g6XRDeV<!v-n!{(f34U0t9s@1aY
>vD#xR54}N_}NAbaVnG%!Hh&cu8t2KXuY@1WziDb*Di|#O!2%PWKgyM9VTf;g-9w5?#jd<i|C-?Y(;pxp{~+#3Dxf%xdSycufNR9tj9c831G0{vyuy3oE|I>
as-ex49L^977f`9Wk&fT3hYntL~qSLW&WCzb&=_ccxrxt)00IJ7GumEmQt$e`FVu$(P%LwxE#~HE=#B1J2E@df(njBT%}YeHsd6gt!~9Kd2i@`JN91;>n7BX
Z=}ap4b{%_(jiwA%3qpz5n7}mTT#sO)-Wx9_L8F1-PF%&v=~m?$3nBo{MjSvm+DY3LPPz9I1_Gv%!v2m6e6n19HEeQB~UY4=M2Z%z}N9|)aVU4?h^52sCyDX
mWWNu6^1#IAoPE~%Gz6-aL-!F)x+TsM0pJg0cX)4l~#rzZe=kOpHK`#^an-1k7WXt;U)LQC1^`QQ(q?T3G97Yi+mEsIyr4#^2>Jt#KiMSOP1_#DhV``YUF=i
pazF&)pSODI@1zhjL&+@M!{&Eb;H-Rq>+XXOtkVM%q$|o<y4<0^nvO&<xHjDK|bx-6p3NbH{>o94%7{5fCrRtio_Zf5~4pOMTPcnMJ1-F(O05c77C>-%V>pH
fJO%qHKr<t(G<78g#SE?Zwb3jqmEvt3vYN>3yBH^I{?YMXIGDlw%6F?D`5&vJ12&T&Lt67#@@is(FJC{>FyJ#%UwNjVjRF&X|97A(|SqB6b*9VmUk1$ijmo4
ND8ud3KNOzxQMEX1B_CJ5RQSV8~V$HAHEMTIywTXoj(|1kX#?dY4K+7LL%KN`SSSD8I5k(WDG9AD_dqSC80-Bn5jRH#XQi+1gQRYlFAdBZ}=#t>p$R>4$Bma
-rNDM)*;@ym=Ed(cK$o{pYVt=pxGKHNlO5srE9Ai(g(4XV%&mPP{9cV6$I2(55a2+_*kYWfZ(8z8N*~DcTr1%fq>t^SmbtE{`Yhu)<q8=s5Z=4I(WhE#tqv=
p0~$5|M-WZB~GY)+~{%g;0g{Fi;rkvhFsI=(v`NwjMhS^cGp>~*3)M3Z`^JdM2%M;%T+lSps8TS`Sf_}iHV8_sLSVw3vChh>c!bjQzP|s>c9Sf#N~qGBQ&$)
7(S$TeAP{`bMNF`eryWC%l5;nmj*)&0r3~BgOtvymE4rmYxU>7`qmjZ=xeq=sW@ez5cHX{GGxnAvN2ix;_tcZ97@@S4G+UO#&J|Ydlf<sSk>gjg6xZmb}M>X
{QC%SGkwWWT;pT+6e076??BABet_HnXH85tPX1S6Y_@)`sFklm%9!}CUv`+bND?mz(B*`!@X-P}e{`b3QZhg2;UzqzOc;WYCUGbbwQ^%uzh#P?ZX~8hy{k!z
C^9KwWNu}tFOb%c6X+eNZ671gOJDz&-2Hil>a88LwVETm_yoGKi-9S*Cse<B6g;%5$$iUzH0_V0!sS}HLN(8SrfR2k)uZqWxl|rCPwZ{^beZ_TtzUe~KS1Ch
<^nSFxd6Ba_!gKrEOHeDTKWkoLkvJiS2VtZ{KL28(50>=Kc{Ofvn|ZoPVDUuwU_OB-iQGVyyM7WE1pFH<GiJOib{rkthWM94_jkd72kuDPRw~&{8e|#(AR$g
;HuaP_jE{=fPILB5~_>M9=$0TV5hU+Tzu6xCTzRws+RF7rZtH$GLAa<qSRk*7HO;^bfJjE6+@;gCWOw$XCJgdP?mQ_ZhYj2>&Eq<jO}=^braR4-rzJZ!bJkN
VKff|SrU`bBb(asStbw*x{5o@Z+3E`5t_{g&ylUB?3DgMiLW1$cC(P>d<xn8+W2u=H%_GF6B1vIU}87#vCnZY?h#2X!}BH%@THC5B!fo}Wd&z*jGu9<)(=$9
VwR>`46HZW+fcd9$@I!i+x>=}p*N3M`ldf2xrlguryq#N0fV(Fe|jTJ&V@vySDcU40l&a_J7jX1tTB%n<SQhu!Y4=$XAoRUi{Q;DNryKc0!>B3YwXLsYVRVP
K~D&&2Qd}UFs-J<gA}&^w?i(4R4p4;P{+%EWXvE-!XChR7o7B3Du;KJIi37`8}TN+=?1ZivFM|Fp4Rt{uGA#@S|F^j)9o7D&dCK5hU=f8BvDc_bi|{*m2)=G
ZRshoQ9r4|8F5wJo%~!UXnOVgk;sg}rEflZS1Kx^?^sQ<<}%sAsVh|D^t0V~^_Q1nZHd;(8Bd7q7dBz`F-o;lQzJ-O>30=r3qIO9J*+4dp-~IUPvbaOi4vCP
{a%e90B`!*zOAxX#G~RtWVw;1FmV7V!Gr#~Dds$@z+Ul5SwIQElE|aqHMoXvJXPrN%=jfpc13;y3<qUA{n~UOIBAr&Cm72Mw&8m;ivZ<q`mS)TC0KleH}Aob
5JiOr?PCl()n9SMI48tDyy%;al;$)x7?hc{_g|`*5l4u>vOC?J^y4D&nDMR$z=&RkXX@uky;7Nu{!HK_!^0W@wDG0<C?;?U{BIm~xTL8zF#KZ@;)kwP$bn`2
<F1(3c)_tp^NgC_l~|NH*Rq+m1#}tZuW9N@6qH4|APOIDXVC5Iw|kzUQwcM4?0~YjzVfw8Q8oBZGppOj+a+JerYGS<Z89E+4#Tg+RL`DJxHD-ZXSN7*FW0}Z
<Zqz?Yq%}7=_Xm=`szmZHh!oq>|bI#)@<-efpch0qY!rYyfI-oSPN5=SQhZiK;7c)8hdi}nI0*OwMYlR#RAWL<4D1#GNOza4;3H?x}ZsxIIaK<O9w#0?L8Rn
2smBSh&012jFy37K|~dgc~MlZ2_~Md9!!a%wz`({?&Y#@kbnD=9lK#bXQ(V$p2=Vbo9kGzb_2Rs1=bIjUJC{GnA<IDG1a`g__nvKqO=&!6aWn>e4*M5H0&Uv
&M$!577s@YvDui%q^<Y1fbJgnOMy8bL>7OARG~(SQH#iHZ6xD)qm#ZM?6@XCT^OfQJ{p__slu#{>J1-y`hnL*Dvxc=>;*`iS$6tA;cj;hx-FcV>XA>bVgU0$
l!z4vpxkuz@}B=h+C-+$kXUHfA&GvA&hjP`z?$jC|M5SYpQcQakJtTl`>DN5=#!3l#ef6X&>&H-4OZr7*>n<ZKsxYSFl?vHCE?J<gu~!kWi}W1FSI(t=_lW;
ov5+mAauB7Tp|J*uAJXnYhuCF?$JTECeF+2ctPS4X#1h$gyq!v?_C5|Avf?a3u3l6GWN`1Wq;iIJU<eZV0NUIF_F{~7tR;WjU5f){Vi<I0IEHADO06z`op~?
4{cj|_o2|I^Na2-XsQ!k(15}`WXV=wwdi*%)+OgVp6nukzb^RCLCR5bcTH=G+8iC-%pYEdPp-iEhgN$J5x>E%m6u@jbvLHsP_g*7_F@OnsaJw1kF|;okVQvd
j#-^h*4sfKL*bmv`fWJir9x0avRf-S#*M6{AAYLQGQp;wL)GEknPlAa4*yBQ^j~1U&aD0va$%L@ZwX=+AIyun)Eo%1KE3D$Z!*|sf6(`@V2Ki6LoBm8J1)B+
TLQLe9sv%>tsd>anc?y=$nI38JK8f499#w!$_$B_4vd6?BvG_A#%8$f(U;P`vJTgCc8uK+AI=%IraJTf%iUsg&QpN3UDi!1gm%js@;}eOgK|ns1u`nptGOV;
=c~NlJs|m%J#gko!#|bj2+=n8AM-g_fSoEEU&?lUsdf^7C|yD{`};%2>y(RCQ58KAH69S~+jT)EUCS87&lam?OnssagLU|Nb`{TEHB$|nS{{iWA=zubbwf3$
Vl(`@VTQM}&*e*NU8u`v0FJGkHd>jlWdG1A;-A64$c=JZY77TdLQZGT%s6Mk;3=|&T1s$IN;vS9meXuiJtXD%+v`@Q1|OdTpWM0bhM0k#!2d!g$g3sd)RQRC
QJ<3)()gH3vbZ}7{$um$nQ;Ewc;P}64J=8&GEet&Dz0h1jc|1PjO0YCd)7$yl|y#jmGB}wGa>D850X6|$9xX(FI58=CL?7qsCoq}9#F5N&fB5%7moRNaa5iL
eIg)DQ78jrz8O{ZH4Do@ucD>R?+{zRCR4(zkLC8Z)$>6gdh~WytWcfgno3w+*DaO`raW&sO|+<A^Rm`$xfqeICH%)4_+qKff_Yjh<-jUVkr0WlUhwOmD2bh!
4Kt69+xu!+N?t@<TZx#l=MZ^5-AxDxd2EdX$BEVPoK$H*a$^i{ev2o%yEX{*4$+?KuA0C&U~~;73DS1k@E1O-<dmH0epFF28`y(UTQ@;M+8!YjM+}^ha}iyC
eqTOLC}SRw$t7*&%~d)TkAJUEU^-h}zAOMhi_?6!9<-H^qFoZ!r4y5JoVplR#ll$ii(HG6$FbWiLwDkU%;_S*W~UIyL0=(EY@8R6IQzL}#c!Di(Rv3E9ilpn
Q$9)ZJ;JNC$4@EzSX-Ec7iY{3!mB^I?n~GixtJ0F<?dobYBaJiuiF=M<QqAM*c5#+*zX|3-lE_h{v8e2Xgl**9!4F*f=^Io-y{^j&_+gYurSK1<jnTDFwXCw
U!!F#=z)dT8?*sDpoiKVdLQ2nT%<iW6o>7Mv&JFnW89`vWy*yRRIp2@4E&@_V5f(VA(2p>t)auN;6@~V8GH>M*C|0Di-fkJ_O;oj%V3j;oOqbyf3&r4EYI7m
-9I8bqF0pxS2d72c&@Oi-P{R+G8ZxOUF8eCz}>t`lvsPu!@y}en%ai*1@IG-S8N#mv`LrCYV;g+3feIZrN`94v`r?S#WN~gI5Z28i=YGw$#h%py1}Vu!Pd}y
hG`7vZ$cOzFr4CjMIaXXyF%Bf4a#|5;=6x?fh7*JV?Z)JJ`Z<x=~xBKj=D~!ECzbPQ!?eo=41tpfP#XE2%6?5S5eYWVQ&PuS&ixL#V~iv8D53oiZ>BbU8aN*
eQawbwzGe7FNdNypWg{^2D*ayZiglYe?St`xOL*#c=<&R0J&NLTo(<gLjF*RU8-@X_lUgWlTZF2a+f@l|E`_WL%+>}8T$ES4JqW{nqE_>&ESsF8Tm7cXPYUo
=8+S57=db*By&#(sX0&LLbw!Df|Rv+riOQ|Jr^%|$Bwsi_$NcRf=`s*scBMEP8@Lb6b(uEI3b;}(F5MnYzoz|8?=HCsML37upjw83u~G-9nIklsO#=26Bzk?
oEU%AdX6O}(cRU8V1V=KNlMhmG~;e^!>Gh&wvQ!xS1Wl3i0a?tr!GRVCh~aiul;!1(<8vsJ-=>^dEvw)&h{oiY*hz*no&Sy(}<!$9>l<$PnP(PjU8-@d@OF9
KV0xcMz9i_czK-<YI|6xQ{2g|@ii@hw5OOiES3AF;Mi?B;I3<D1D5lTn})yG_~7eSXNS?rTzQrE9`d~5%S#MDBmxPwD~xdkjNsGe9PjDN*bChi1r1*X>dPme
bjVT6^~QDY)Xjih)rW|^O|Xp;%>1gPcl%;3AJb38z`JjTt0}Zam`%1Pc?D{2?C9{k>dE<{HQU^@SJ?uDy)Xm%v3)K)0JpeKg7L-C|Ik9M-@HjsWP1_d>(Rtt
pl0w@5PMp-IcdM?o${Yp3@mX+@FN4OjyD8*gn|0%$3q`Ix;MeBw%-p7%{iLE2LQ@C2K1pE6ms@rc=obsNiF7+(Yn{R-#b#mu-&$a>FF}&2lE3@SktWyY-T)>
`uB%vJlNA{A*I3&O7?sE<_aZl812Is-lrt?jrNbM;ePh%GRb6Y1F1-F6Q0$<ulwJelHM`d)(BI4ELafnd1(}BER2p3{)m-7PFh=ui4IAnieDgB9nkEoA#zBE
KF@&_fwP)086hZaibAfJMR{{I08Rsv?BLs$s)6;f|5%`^e^y~5Z`FDF%Ck-R*@~q>`3(*1_0p6aRNOpr=-VwPcz}&csPIcy?%XIqLJ7<y_Ipg2nZ`BX*3rqv
x^GC?aN}5$5y#P(=Cb;(6tK52QWhj?PmnI4%=A+{v=}~C+W(GEH<S6i)=Oyq@8hpmpg_v2Y^N6*laIC|ru79D#(0tm^;btO&|9CJwxF2J|4Lsed#u%3d)ze!
%}ryV5BhjLMs2^9xf+GI%s~7|r~}paU#$ek5!)|TpC;bS@%l;n{brjebgkiUHg-{Yz9tft6jAZD&FL-=Sd=$FD436C$I-*RC8EAJgz#q6$Zo=~B5X9RC*C)&
^`v2B26JY5FDt6d(SRIU-^1k0-3Aa5+dG&~3P3j;PzwajN>u6#40LHhhSGiBW=J31NzxkU;V@d-WBkVDOq3-YpV+#XaZ6t_EhKn4H1!G*jbwdPfc)8Gb8jO8
B~L9a^vS^TZXAeVj^qGPM@|Jx5T}={Xko?)tlUMWBT#QZ2IgD-#X7h!%#Qzi?In#)M`Ft1IrX3iqoei51#AjZX88om3G}(fG5_d{{&FgS!q~!m2Lg&lk0VL^
eqT6>y1Ph&Vxhl*)z!lvsRB*&fGpH;bNRf8wi^uU{X@?4oLWL)`FkKW`U2`PGFwXdSsT!rNIE-gH%K+Z>xKyy#5nZ5pIzeY=~R<D4jotNv8)y}3KS35U+Wes
)VlI`$;s3du|D<G)VFU>r!|Xs^=K=h6!OaEaezvtRVtHLhfeTeWC3TD@YdK0ThTXS^lNNb{K2T*J?KY&n_~YC%I<h1!Zt5_Kt5c#At`#S#=^c@ZSVi;|CB4i
!~K0|q^XrU$9HUc0%S_~vw5-hCdi>vSK?=7J&brQy`Hb*Baax*z}FoI2~*eWQ2sht`5cC8NnlURYN5p#w<iqqpLy%evA2V$!+W-dGr3?Ah!BEyMb${?>kvmr
ZT^-NUwix!q`8|g;<tDbBM}B5$$(4N9)ig)m4;OtNB)-iDm$Mq{)W;m-~KS4{F4*g!%YDT+rto+v(t4$9p3h<b7_EDnF*dFiiCfxIatO|Z9g@wN!hunBV>l;
q-PbLtq_Hc$nfpnPEk=iZ!rF`By<Fbmi<Cw#eZdZ397_hGdHLFp&6bFJUIX0&0s#28xXsILSvrUF9eJ7Amhq9aP5p9>URKb+L{Dj$LwwM)5d)+0gi%34|6OY
3@KtbR0YO0(kb!a<sCD{5`=6x3*mdX&$|OTMy!YK=qPHz@^19ghk5<zF}%({FixU)#Uj#!BDxXlq?cfUo6%5C3;e`>Uk#XOjIVf>WTJ%kq}eX`jZw_PX^tRu
(s9JqXI=tP;BtN|$>QwuyafC%=`_b*ZkAZ93ZeA*|KmQEHqmH`Ss!KbW*sa3m}35dloRRbk$f15%|RTl-l&%OJJp$6LrdyEi-uZ&JFcpH1f_#>uI&=c9WQQ}
ZC;>dls#LO-Fu`!=nzhmQ|hNCk~_ZM3-dytmj3NQx2%(@tX@l>*<P}4dU21sk3Mn0U0xkHzG6CQXWEGrvrF_%MMC-xVh9>+7hX#9hUf0V8&tANLqerrw3@+;
&E-sM&5w-kuwl%Lir=tRckN@^e9RwT2CbQUcJCIDw4UzTV~6%FVe^tM!q8GmxJfpLiCNgPfkuix9(vUfUt<<E&TgBEo@f`US~|HD)j7jC8{54q5sQY48S6Cj
0gx>E^sl+NOv{yGqFL!kby(d<qg^wC^bLsA!q2QvSr*;g;Xu}44Z5L_G01f9qhMGnb34LToTR8kIw~KL{Y2ikgqnQ4Us2~F3^UTD;C@($ZPO9Fx9y^fkn#k9
X&H)YIvybea>D@`#fkHE3Z&KhINQoi#cg6pI~8ZO80fV)s1P}fLQphdA|TG(MROnpnCF6Y_%RGD8UQDR3SP<^nE!ZWyp%Y?8G?Dk$3F<&;$-kE<#oom9O6Po
Ybd_e$pC^TerhI_Q&~M)JafCCaXw~f=x!R`dd!(eoHp~ityJGpz@DtupugKaBj!rf4rP<2qVrsAA(1h3%O+d{MNy?6njls3^GR3oRm)}iN6efL0ukK|8)f;W
px=Q;T6cbPrQ_}pj&e3B4XUcN=A=RJ$c`nlV6T{&D;_dmyTQLU>RCJM@C+bRt0_FP=Y-Sp*cm@30ahSBwZ>0pEsaobK~c#J^I)-wVwyf(OY8ZA8IDs3KAJQr
raOZV#}Z|De-eHSaTzwGs<wvp=wT?2fB104i?+ZJj4&3_s=#pEQd}(TiNe2`769>yBr6eYV{+-jqmKmxTkK^%UBv`jIvGEyimCV^6;{->jlvbtiDN|6w<V$6
8r}2h&iHuQhz(G-(lmIRQ7J6iuST$;E3TW8jBXH!oCfs8r`;7?vX-;7r`%t+2B>jypY##is!U#;wBz2oS;dZttJ!W!*t{X%os%WGS0e(54kba|unse*wAU|S
`7&V067JpBo6_I9;VL`F{??2JA|Zqnyzwp{QTe17N_r$R+~IhguV>;zl5V2S72|bP{F!*_KU_*}yQ=7msg;Fk{oE!)aB6v{eP(u^F+I4M^3?(j#*#N^RzM_O
UvApFpaW_Cm~G@|W*mbyZIzUoTQ!b47==tUw`3BK+Dw*5x0f?p=;|Q(D7GICx$_5O6Wd6QM)g>iW6Dm2j@zNfK;6`61lv4&X4o5E`331{W)59<=8_X$roZ89
!`k=7cr`eB8(yoYjS;HUgebBa@Cl_eND7ewijWy>{591vPXjdAr@BBu&YbTjg*+X(UV7w)wm2lAQ0El7ctHnIKeb2Ds!E~^<*Z7Ogt}&hrhm*LDX}i<>w`B|
5db~ae>w<S?p!(Y^d!plT9Np?wS>O$m0E#K74ng{u*M0EKG_0lhTjBbRk)3<s=^$;>vwkKhpE9C=1u$&yz#Dz<@PToMJG{95L!1o`uJUGsy2TV0z|#wR{Q%p
Bz1Zn4xJL~<bIwJI%Xs><LT-a0SSquLv<}v6T7<Tf_LqCmZ!HY7>!d?9%EGWx1(cGXgQJ4Gf8N+Tr=H(ET=*daTrNg<L3FxJk601Yh@vbJLCD`!pos_VY*={
r>#u=f>3YjkNS!JrsI#js}^VBd3yjXpRn)4r|TZ>XmU(TqEak^{S!{oT434OCZKn@Btn8aP;|5-a)qqt3DSFSUBs+I`xI}k$$A#(jhYQC1b8oF1nYiZ%kn&;
ctKefl2#y5OF0fwkS^<hPmTY0Q6LR&C*gtLwR*i8v-;W!Vcm2`su@79E>qqD(i%(mkEe^S)`5XAX+IXQ{G!cXGR0<L=YYX8jt3QSNZc8ck;J3OO9Jh<&Rx~F
B7=9qQ&6vyRA-S^nSoR4c^2OX-m5($hlalmfCXKu76kAxjeD$YQo)Neau+LO()^NFO>&gTD#P{s)vkAES8VJ_zb}vPUV}hUZNhXM5x^h2vjJqYaHVVeR$<t4
=xJ&<>{LtuE`==A0p!lhBVZhi@NV?63@M!PaSPIT5o*Pz#U_()kKW8%?#jQ6^|$vZ1VY808_|H~wbjykt>vNHWniis2n)=uOg;gg{u~IDzUkLguAPpy?Z8&$
4vj=73aL}to{aLH$M3lGx%SRa_FBN{2SNv<ABmFyDN|KB-VPCSwx%14^8EmLY#XbP%O94q452*EXt;f^HZ9y|IK!*~F4E5=igmS~;>wSb`=sRYl(a`tF4iQ_
8u$0SmS6$3$jKPMb3QcIm#MU1cp(VUTb_L(XY#l-aFm6=9v8i!(=>l4dL*3)69sxfa2<5Tm5&xLW;|LjZ@O_(z;^tmq|HU?z$I2K>aHtIbIZwZ72_Y`iv&^L
y2%OW)&7dMUsJI)cXghb7X~Z8HtRLII{R_qg(z`xT?13FT0n|*fA(vsTRJ9`XB|a6=^s`hNHV|p?|x2)t)c4oW|9vPc=(IFuWd4D$-WyQomSbs<S^5Eqm;9n
`M3Ef0c$#E!X8WxoeeV!iZmCpFfcPZEZ=P&oEY<7H}C<#TY;esIwI8dz!(=Dxr<blj2}m$Sk0PeTdDNV{Y4~wKMAQ`W)R<LJasfhsA<y6CuyMUA;(`WbBq#z
Y{>LTOm$COCFB}Y-`ZyPKc~}20>&&x$_=UdC^!ERj#+~VJIKflpa-F~|NU#>VA~E7s6YJ9<3(-L*}EWVtUrCNMYyBsxD)zHxW7TeZj%IEpCmG%pQ;E&6l~~Q
`=&D`sqanq<@Upd#Mm95XRLny)^RW@1t6n{o}*2rab&;f`!9<o!s}2!r0(kGbm!NKi=vSTTSYf&r@0grhA+>ap$b|iLwAefp-VkQWgO6uHC$kHCU)h=DY&s^
pIQ7=&c8t*+z0X6sQ7lkHM0b&rut8fMhcIQLTpq9i*Z-<bE)ICqMEtK&Ek$+!gMhfd{>~JZ1fnxoedY&FAPhnmJvKQV;|R&%i)2A2L8^+fBb!03)1!YsTUZD
Dbu6gdz*O&Y9Yu|fJ#u%*vZ0=F+6vcxS&RqymBA_cj<6xYs4aN4n^`xZ0~GC>qp#Xv)?NwvC7*7Xl*`T!7xFWNzN<3){bN%K!vzo!<ldJjLEikTqyoBjTFNa
PY=-{x2#Lu^jt6)cI%_~fI+14gNvPu;xpgElo&%YO0be)LBzKDhYad%nz)TKm@w0?C<S@WT1p;DL(ppE>Ty|+IsUnzuL{~z>yLl>(X=_D<HNL0vo}ew-R?!T
>e=G=f+-#4VY1rR8#jDyY{Ph5Mijx|(7)YJtApCXU9AFYY=9jtFF6J*Q=z86ZjVj}0ku;#Ue5;XB5u7~D$-gLBnJv0(B}dwaTeUzJZt|<cajB!hPtu6TwDwo
clJ;*X@sRkgJ*URo3CNcrVXzr&95sB(u>3fv?<KemOP6<f^4$7x+;=lW*86Vi#jH3=%LaS1H<k6*x@=X4s>@oR#^R%i?zp*#`kNnl=&5r=*FmDXiY+P7SYTm
R!ZKF{Cwq!wOi2&Om8qM0QONi!uy7j-Gcmugz7;HVxQ6~>1Ha)H93j`a%7&{p_&Th{(kwzwLZuU0_`1QJO)`n-tpshExEH45C~liLAzH~zO6st-lq?X>!6+7
Cpi(3thTTZlX*{+fB>Nz$4{2j2aX(Y1VLd)RZMH0CB`TJ%(%L;f%IKiGXy4_ieVkGsn2ih82)8iV^2n&I(%&r1*?#zLbd|;Gh0yCnVME;9XK134K3FG8MbeB
2-^Vef;(2(+l=X6;{;_&=9C8J_U7n*|3if@l&B>-4uw>y12R&LQA;`vZ1*VTDf+(H(8-GS3k)c$8<D$AXp))*A5I_8);BZd?^X^@+^1ghX<O2Ww_Flvi`hGz
eqsv*SN}BYVEMxL6fFtJwK-;l+`e`r$cAG25PAObq;UU%kikz7DbZAXcsx#hR;ids^R=?OGU?%jUuL_h`{zq{SUL4ZtN)^EXEq{wke=c-rBvx%>1>)RHH+2y
>uIlP^u}whO=URX4`6uQ&c(X@ir~gZDe}$C4_7dN<YsA0d@H!O4h(m==3jDMT3=43jvBzTXg4WE#C3PRbjyw6u__F3{@eippwT>X)A9EotnmdiGJSwEbImDI
06+wWD5jM-IaOC^RxC~=*12*{oHRLsRs(8Q@%y*ZDvgq4Ss<Q3se)TK85jWcNcA&rl@>;M<c&rZo%0l8+I0@FGI}o5wPf<N6DtQJ5uzDzN0Kg7_+kf#ZeH>j
&|EU=!yzDBm(ioWq1ZJTCl?JA)8-8Hi%|eO;;j<xNLXrsQCFi?zqpx$@K~V)dBm7$QxHe37q-!fY&V4+vM~7=8828C?SxhelOLE~Sc1D9s9B>ePsB4?Itl=S
WY$zJmVN-6B_;e`CJH+Z;)Zm5VIxC?inn922*)8BRAnEt3wrFFXjQXRrpu;{4Rrl2dBP4^yvX!8!%T(^+M#K-oGR3py%OOFPu{vfDpM#q&yh$K_%XQf(KslW
OV7EAg*1^RGoc5`%?;(Wum`2F5fnwT*5>Or-@1PLg$6N8*9~z7iSRh9Kl#K^SssTzrI?NJ#1)Zw+n2Z{f=fWK7A!3C3?^?bNCAr=5VaI}<1~U?EH$?uvxldA
%;##i60!El(sBrJ8A{4GHUP@|WI?uF)qF;iqu8J5?+s*#<`B2%C>TV(LO9*;rnNFf{&$d6Sb{&9?@4I0eQ>#fn3FJ=-N@&;`OyP%=0*`3u3bz-gT2>V@5L(r
6Txdco*Aly65lX*)s)APktZpv50uN>;ihb7^b4Ki#%iUm9-fA8nQ`@1DA{8#VHR;_FPGT^|H(+<la9?iXDwk;BNROX=IwHrsBKH_;evpdA49_~F7EZ8Zan%%
k`Ks1q~^fxy`gyJBjK4cwM!T2wUR@U*lbJQE2>d*`j8iIQG+sJo#meGCLDC#o5aU%Y{_I8i;o2iy(=WE@CC83DA>W_msGw2gE3-;af#33Gk?9M8xWO)6Kz|7
zJ){eRhY;m?BPNyzZCA-#>hp)iV*bUYewKIV3d=RPYzSml?xs&p~aXL!oD_eUQdsY>6N#b+3WG$rz00A<Ltn4e2+!gXXITZTnAu<bbZdN;m>xzY8QUN&GBJS
K_-!2R;`xp;%}^HJRqtjxViFAcfaa#8ytTD`1uh7#yzzFIFn(n*Y+~JMxutIditf%6>Wq{F1#eHU8?|IGp_(Vqk>W5krMM9q|Gq%Z)bG@W8{}FV@<6RMCfU^
CVJ;ZR5LG!(E7L`dUbdWgd=Rt`C7c4X~A$ciae?~hxmhi;Ej2o99s*pVs8BE3}S#t%MKyfD6+Hax8m~84i7I`=Rchj8*(j)shfi+<@h>Ha-L+%rS@mhSbt6Q
sP+!)#e_};oL=b#he=RhM-WiYb=<+VyjhA+qa0X$>rvVoqVR8+4eV|#yq%k<!rP_V@^Uci!|+mUJUUqK^(R~JzaRK<s`qPu-ZYy{qUflH*HJnAk?)7!=K1V!
0UUW^_oNh#vrR8lylx`_sZoAvs-LGa#%<*kgZjQUYDtY+6JSDM*@R+(D<@@PPghVf1vJ<-ESZI#bJ9m81eg}1Ke>RpU<jM5I`3Ni)_$&sT!=~b_GIB(E{aNk
$to;;!U1otPiy2h&{cxHJ0Shd(rrmY#gM>i-=R-Gp;>8_shuH|{|)51uhrzQ`v0}>UiBC8<|CiLeH{QGQJkK6O}#g~v=ql1U5$4LNid7t8Ky2v<ldZwZD77)
14l&xp8PSzbu^Hmi8LR?t9t1%Un}7;TCR9Es)WY6tb6=k6)0zsR5RJ0|MFX#T}C4pp^p=J+5H&HhY{X@){pCO0GO_5_R@lix#>WU^R$l>moGnUe*<SBq&&F-
#h>bHL+B*o8^8_1+HwsWp+_oe)QGSv=kk!nrq1bCIX~rBA7(>7`2GPVB=((3%X%<_t!j$NAQy-H9S^TRe}x)B!AWvIu*dg<yN1;M>nu#!KR@fn6xFRCVym8N
g)-45JS+D)9=?n^lZia+h$i@&7T|~U3>{+3J63t--8CE3{hkq+^;T_DX<|6d8=6&5u}7Q|2^<$65R$N;A1vwDt6N&DIK}N<#i9oDtd21kn-yI;2>$aOo~Vf$
S~C@38prtQNtw2CXnWB)+i}R$6f6{mlpr2KNtm@Y)I@jyV>cdhaZxh#00H1|<<bHGMjcskvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/deteccao_r5.py')

In [ ]:
# listas/conferencia_r5.json (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/conferencia_r5.json', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5-~!G8h+P0Zf`9=dMrM9yQU)ur3Ekve`7t=II+0Y5_<jf6z7xUw6At9e6Jb$7d(tQ-TtZoYcEf~iawj%bNb2cUPrAqA
7&sKD!rUsxEb^f#PPJ}I?2z7Zo35%(yuqGHjS!kCgh5~QaIMws{uco{`c~CSxyyNBJGl{YvEM6~D3P-9e5!V|R41`GQ<mJEA*U`4Re-v78^$XwzWr_=K|3(0
Gn(H2IbO#+<CfmJ4MFJ#t?Sef@%e$L!UL%CX=;B`t5IUb7~>oqACHyn|7|j*^=fXuX3pY(*vQ29$D~$jy23;l*djp@@!>L!y^9LPZM%)AE1Bk)=g3j(CI1M_
gb1?P{Da`QQnHWUYJ3cdW6gX8)fD%@9K0I|YkgvGAVUmqrHGV9eFeW-BSkJ9v+sm3s{rFT9;8*;>No+e>g6nhJ=G}+8`P1Q(XQL&EUw%J3&fn*s1uSzN0=x>
Fn`$c_G^9vqFUBbSJ{ej`_cKcK2CsJf8kt_Jk*t+DNan4JPAMmJ~@T-CtVAL00E=}&jkPgPQjfnvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/conferencia_r5.json')

In [ ]:
# listas/divisao_r4_flavoria.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_flavoria.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;!SNenOy)f8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7v-DGDY8X~*_v0-{=i^@umI~GHRNPa@`5dWMqjsoLnzp;lUd7Ji_mA)I0;G(WU`yfBq{}ZaufwqIWAMf7Qh>!WAa=OY
+>LTd77ID0ipvQ2TQZ4Xg|G~xnjstD*T&Y(KBc<;S@<Qs9U7X@p5cDxn>~aJ<5*U+=Fmd<`SQL*sRepR9+$*n*W8psfRPrkCKdNIrhQ?HzMoA8xQ+5wl0@;2
?a9e{Pk{c=5n=P=Rnd-yNNouIVpA{xrho;Y_Cvi+FrdpgrblY;QH_I!Fj@khlcwf)S;bumsVm;TNmjwTB2w@g<G=RZL4xkI?vq$JsSO`oN2rhW7`QW9--?&f
qoYFghxI~O^%eM*;4v8fUxwjwl_s={Hxyh3<SsPjT|kNg+`kZisyYxF($x41=E$gzbcg!uRESu&#r|PNsqWhV6&X5KC^+0#bPK>poZc$y$ImBdRvV2HQs@@j
vH~>Js_-Iq-gVLoGDj8MSLkdlSQU7x@Te#bn;qjqaX2qN97**TirM%M-m`isj4JE%f1!N`HrS$lJj4>hb^{oRCuxSU?o(!Lo3nnj&abd)Q*MJzV6CyYbiA2*
6aC7g**#P)O*LXZV56cjBLVQk_-|<M;FZl(f<<5sxnI#7_&KN4T@-Gv<xJ=IA1@_F-5Ewfx!@<tw7768EX8bic0^14+D@o#qe2g=WM3liL@cwO=Kot=#(_X?
K;jWG?muZaj<0)WrP@}TWd|`a_&9okoi?6fJILF|VV|!PRKx-jke(@04542%HgIYm6NT<_oJ@E6VOF|8RnSOj{c3cAIfNQGjL_j^W-!Cm<vKpsv^(kn-N?zZ
0g&%#gZo0xK){6?yEfw|1OU+M3VoCD@}<g>=Vim}&Uh~E+jfE_4!JhYvP#es%>)n0iOg3To+3u@N>qhTS=d6>N_tD905GXAki7oWM6*2rbc!_xyTu)k*?72e
uJC(FF{K_1&c9C>!+7qEb0x?D)9d4S@Rd{yeN(5`*}sy4jf~z7q`pIb5~jFq2k)XPthqBHd5E5EJMu@{H=OvI2)2d!@gX&^VQTV{W%J8?uyw?$j-lS>kNqeZ
5z~~TJ_^#-%GEbSE>|6e7=#`i&2QOQB-$-kVC-bR*3rY{%_#9`0PwI7?LDWfhB;ucm$!zZi8L{o6Vus|RL_$4)SkuEiuGYVo{rVA*^9Gt+{J@8x}@ir4@u<J
YVO?<d+SMFx6;(n{M4jnIoZY_fbT1dsn;g`zVhf2xBJgW-{F9*^ZXCA4Rfr+Ax{9}FBUT_cRuNM=EwKs)C~+zM}vUz?J6FHh}A+k6o`i?6VYb6y$MyB=x#nq
^(#FA(BJkQcjHvIR&Rb8-*7{HgdtRiGj4Cvl2tc@4VdWj+9Q{X5zyI=6|LWgg%vepws!pH>Bi53gII+P1cE+A&lLL&-EPx^7xLNWF5hKw*pbs>EW4<2=G4{U
0;cefFv}mfRV)|{sx2S3s6ZK|EkwO0iyY9boi(HUbpXS7XgF(Nf8mZDugM_`S$V!`i}SUvUmC?gfNY-fpv4P0YEa)I*xaEg_XfRYTWQc0LTKII3o{<yLW=)W
p+hA^Bf{k!qv=^!H=+AAk3fW3xTvPZoL7M&e;K>o3!?O(P2!t+lZ}%+8T&1PvZ5+M#V^{wWse@eV2sKa_4zU`-AHhrK<#KJ+Ro_<qGMVx+dfVX=#}XeIQy_(
#;vK!xPqOL&5@UYcYPFSjT9{81_2{N=9bK#qU_81uB6KS8!xwBn}JkD&UgZ&@p3-tN6f6VN!IC&uKDSD{vtrWb%6$c#|58-kA1fSmuOQkF;)0gby~<alA2)b
mHlJF4*Da=KN&I*QhB;Iz>i!VzfMVR<QCrb$@WqJ7S8m94u*N4*!9Iajew)kEQ%XxWeDyw>j93!%XVti^-3Bm=@-y~dW08i`jJcDU{qo=@g=;|&HD~T1PjWX
w-8Nb9vuWCjHGiMU$!e5W!n>J+5A5?3zl2^6}mD>2VMTyCU86=I1b)c>e+#?@O>A_d{Sm}b-Ip^o0#JHEk}#cF49zS_#?vMkOUQv=Toc@NAs?I!)!Cc<tMV+
pC079HkrQ<+N*dlLr(|#2=|T;%4GIq_vVB-lPwj5Jgt4P!5dD{4Wogfkj7|R@-sv=as_AQjAWs$tMePRNt4c7U|M!p8ij#^5UP3EY?X<fGfsV^@_S3Wdohzg
$WL3Qe~-ySbU0R0$o0wFtzRkQnXH}N|1A6}Rz{t;^p%CJR7nEc6CuFzO%BEJcOT}}xTNKO0MP=8m#pqq{hK<rctxfctj*iBeRoCcb)M7=9#XHep9$<4Puh>`
l*eTK%q}*sHgpg{cc}KI)C$;Ji8>w}3pTM79szg4QIh)61A+cmZ4W_|%ElV3+0DZwV_hmP1?7(ce!)!;#(V|(_`{k8Pn{hmpIj_i1JX}qWehQZVD!_1z#brB
FKBqS^{HCmcui7#o5h&U=FdmyOlinPV|&|O#Rb#|1G1~x|98Yxj}5cg={4f;GS}}$P|1qfJlh0aXl>q&Qls<XR0P7kS9jgE0ag_lP*(O|M|5#PD$&3zPx76#
07wyADPTE7A!QpSep4dT=XkN+xO8&HeCQ4gmj~WeC6e@jNoaeWM~&<frj$Agr@*fsI*$J$#c2tlG@ddTDh)CcK<;KTwpQ10Q;@xP3CEo$(nBXghkG76fCis^
tvoYl;Fd0f5z+D6KK^znR_o%6_ktK`?zzX?-TAt`4I_Y)hVmRYA|!nZ8&)-F7%tx;8yKxl6aL{%*0LGPxWoTw+8Rg)RqKi6n5{;TFBOA0!O!afI0<!?pTEbG
v`+adqsi@pxE~c;|M*r*nvi_i(DqJZ$_4J1t%WO!EG>&`DTMUI)>E|W`xEMQREE5F`gg#uTpkH}=otb|mG)KAc>XvnW((i$kI5uv35e&p0f3sQ=M01Rh)y>X
glwOkj>vfn;*S@4apc<QM+E`v*b9d1X|?g*^LZT8-i^0lkcCvn{XBL5QJnpfjv+rX+iPAeyY)p>-&K{ZU@*u-TCk{7(X-In&`4a@avEC97Uwk(qH%E$RsQqx
(Zrr_8KxiJ?NDZg;66Fxkznka^6IQfg)B`RkKIfPsP-6Sj9<&{4|ggm*qtND%TJ|SV0%=<P{66XRKVuU0>)42L#OPcW5dYrmGKOqTVrP){52j@pbL=?f~Bl;
jnEd?kU$y}TJ!GZV11{GA_^UQ&0tUH?A=f(yh!>&;Jf;%?lX@cEQ}h>o9_EO<Y4Kv{qdf!Fu+^BS*G_9{eHSaRbP>jID{paUIeJGH~;6Lts&dnvYc5ZJ1i>+
kHsq`Mf7&-q$FU06#@RnK7H>NXwxSq`ww5j>r^pqrptn{z3cR=T@e~(YJSfA>Y2Wd+5D!j`?~bELYJegM?HX6cd@W+@ljmFTa+D=yfrb?@p;P-hd>~-vz!Lx
=sCz(+H>gsm*{yF2Tgg)%KKCN(jA;bzs}I7wwEdO`hPkrTmE)gV=s=G$xzKi?P4nvSXG%stz8r<IA+8Jd~CvU+F1^12_HE%Pw}&aK#zuzCU=B~D1&@-EZk_&
915@n+3$+~iuOQ0e-R*kAiyzX)DPYI(QNva8LTC88t$nuX{T!5XRXhygQlrD7=9}N!grkcvxJE`<GZ<dI3t!AP{v=VRMZwb(d_;1a)RfW1{8TzZ24R{UOOn7
{))a$w?=|xLr0Dcvw5&eFhu;Ge{Y80Op4mUlgXFT(Ez=`PyG)(DysaNCOm+Dj3~H&z^z2JYx~UzNyEb9J@pat$Wx+9&f&W_YE{gVvfL4fXgPGDij~oB{-^RO
dz}cv)h!Ear3gSZjV<q&or+<ys8zDNNUXHEx*v|zy=`4%vC=`epkAY#oAd38#@-UJ5OuTNEzxtPUc!J>TJ8&p+#$&!c8L2Zp3MBb&#fidt|&%}Z819(*3`ex
E1OdioPQGZWV!6(_o|lrWvv!e7IKqN0za06_I(kkPq!(v<{<%p-%Z`-c$`u2)ob;iwzqHtm=dc`xwaX~t%4K`XO|A77z=<J#zLniX4JR;=_WtLPLJK0))|ma
?u(1ynFo)=R=QX1uSs5h4MFppperx;_rXOKMUm*cTn#|WV%naHBt_obcU%bQ<C>^3Ic3LQLkbZvJEziMJi_#5madBKLOTAP`?}=-S7`W#cPhox>`9(k6czYf
-i75R8K?0QWf$ETywGKyII;~a*MdoucF#y4rP1R>g6o)@l!oOK%_OM<Msa6&Z_bS)hO9jwmfVEWrr-HVkIUbolu&Yf>We=?%M%3>!9H-MfEJ%BALJ%h-D?0q
yoy>V{j$pE+JxQ&#)-e}%V`sPgoKN19BE?k4ZVEm8xLxjm@fAqa77HJTPHy%8_IpJi95K}W<@$gQjQzyL8}?G5Q49H$LtI_O^$?-s&q|h1}Nfc95t9!sQ|GR
Jq>V=lhbA&$Tyc+@^qdxrXOpxsGW0+r({3N%%f<ZuZpk2|5lk&_*%0zfj}}2wnDtld2FGORJ!1Ir8-XK^|KLi<J;dVE|(<%5d>t5`-AK(Tgw1a0|~A%?Mvzo
Y1Q+HX<73M!s6jJ)lSYjz`ui64X_(bvgstcy2f<C1>Wn$^?8k&J8g?%^mwCy<vfNH*Zw~(1WGIFdZR<$T2L?TTtE19@kluRLG_}Z6bgEZ2C&JuXUN1%sUc>n
_wGmpcf(7O7pO&k=DnS-#WHx0?nwU*4QC_)BiL#s7ENlj&;Vr2h~&#4RK1Zs{PtAx%#7bOPS#{VQ-a7$E6DU?Z!myVv$<b6saJt*L-%4{!KKI8=Y}p_=n?pO
twq+F56>Y8TwQLi(&(~sJ3@cucJ;nwqhz8bSUI@iABi|-xaTh@uI40+N$W-FpL^}P?%~=91-~(W0BJKH;p9L(xGgNOTE0RxLCE)PTi|8;Y%?W<Lo>C>!%<6g
3M1HJk<Qy>`WZYBUmmMbH-igXn9W|`Y*!T8XnloZ4e>-?im!(+Tx2_HnX9X|(O*y_Ts+@J#q2cX#-93V)S2A97M!d*<oMXRD92|r3RAOn92b+Jb-sKWFSdtI
?UT&2AFfV;@BBqyL=-)jj+rO*Orj07Sl|66&M~{+-bH%eA8TFZ=xNHkJhx|Mw)ZA!aFVXRf{s<bvavbiqVPCP)Cvgrpv)@s_UG^3a|0+#uf6l|h%SqLkBg+f
*>tUEIz0Jad>iDc7NTJN=DhJ870!jpUqP`$A+saLoY8_rw?=NaH%ptjTY3RBH=dl4T3oP%tC7+^Ci$IguvrX<TC^M<X|znt=UzzU`kgp_8mkQ)0jR*!av<SL
w|8vuyEn%N@gKwKt}qO5dBg*B!kw*&ZYw7gQ2y^Da0%VZlG!?!TZH$AI_@q{bqy3p{YpMQz!@+%4kD%pD1+%ScWCt{YCz^=z*82xkGRXLK;o#UOkDxY+FD+O
;3R7X*wZIEzu-oPe?laq|FTzFH#n%CHvJ<F*6{&P(f>ljHoL3-@h{L@ZByUr*d0TC7Pb~O?9h*hH$KC4Ba%0usK@a1i9*(QdB4mo%UtFx<kQI(wGwXHgUbl!
6t4T@y#E3?XJ@;6u^x)vsWXO9aAAzy)%t(DthGTtRzPbrD&;G<e)@$r1vOx-fHa|__8vJ(q{^tqM2as}s)a1B`<?xzDK|zH2It2H>gdOPGF{|$P6Ai*A;dh@
glDcHg<<yy+zOLdYCfScOb4i|#+<`jR}`ii7I@<wA?T6Awuo*^=Mg6>`vHxx52t?2_z?kr+Vys&icDZ5=05L;s1V63$U39PwTp<S=tA1t4F3ysq|Bv+oeF9X
wuy9yk{|;Zhn_SBo^#j(7vM(tnzi>e`7j&XZ6#Vrh-YhfPA{_rU0|c(OK&jswL$kyEC9Nx=mLj&`C2&b#BL6`|NQE3J~M0BK1CPh$gL4#3a*E%)AyPX)&pmz
qsyRLynE)=1v&Fp5LDh%ZC`wz7{=1>ix;be-Lmu}T()O~>w-3ehcvj$!ad<;z+Vq5A=YwKV4x1Z&M{evIg`6Y<^185p3?Aq^rW<3T$Ydkr}t{fsX4^9#GKi>
<gEElgQM?NsOK7jYcOtkGI-&Wn%7hllNXnKNCcJ%>3$sZ9M#n2QOQ%2TXooZ4L_uMjenUNd>)Qof@tcmwz4ok{-;&-El4n^P0}8^L)wZaf%F*uIqS7>r6zL<
l@G{fR`t7UxVH$AoI=+q8y$oa^1$owCWu1xbdadZqhrMOJfksUnxNKV0HX7w7f&T-@!wi6Z&a2C<u`rI5#Xp}sLK_J8d~xlFtGWRY|x$M@gGy&_KK>evWo^B
HVA};vx|=^diqH=B@v0-W9-?uEZBh5i61+AFJFCr{i2~f?}prp+n2B77qbfoEtezbFVx!Dz~d2SAX)8LqQa||25)ji1vQc>@zz$n6vsi&l%#E?TgQy;N3lWL
@ipBf=)}#z_$51X>Ev_#2(kcTNnwjJ0N&M=l+Ifb{?=`V0J5JwER3}V?Jlo?-{_}SQI<8i&oW)OC{(cqUi_c1hF@L$<p6jNa)ThyM`;&I@-*M5IHJ%rMw<zJ
ZvOM?nB(Ul`%Yag?NkN*_#6=1)Mju%uOH2Z`#vUx>A?Gk5jQ~ZUt=F2PB!o7+<=&G3OO}--knxN;u36EV^HdC-U!B#j;kQIig!@i@WukUgYZUwxcl?wm`u84
UU(;k@_br0^F2UFt|hxPZv;C$YPCS4jod$cddv9+EdZ&a<f*6twl#alxXg3dE*Tqe5KG5APaBF6E*4nUu6qniH!t#;92pRdH5}Ua+O0P8sTV%a>-^Wy)v03e
$f^j?m~W8xtdD&7Erwwnk_erla43nK@gIqa8(GQrqN76j6wYW^#^CGP2jW&YRB>%{iwz9No?U9GP<<8-(!J>gpataJ@W7u#nvY~gFirFZ_}r(()+-q(QxG!$
O`W=t&(ToA6@h)3Id0DGY!77C)^UuezDwUl*hPngHzs0$BjtRJmBvISQyQjb4Hgq}=<)Qgqh~Sg8JSbQ=aTIqv>Gh32&HfED$HVSNDh$AB4mN1A;Ov%N4?Ln
I+JRd7=cVeX{=!>Gd4lVUF`Sm|FZ@%-`v+;?lK?sAt5eiLmOG0pvKQ?9X68GFXe>fq9vH(Kc1Uxb+>!xTZUcyz;^MsimXA}0^DNX49>VG=ng1$SkM`y;e^uD
xBU@X;cR#T<L-T+-EEJj6V}_sO_TG-k1zFSF>$dP*nR57Erq`Q@sgbY7~$(@IACE(8UEW7?HyUf02Uz@P*uaQG}|AsZGUg0N(I2UsI;B>;I6GpniVGIzWGJk
5=LZVLTS#Eb4Kz7c88ac`L=F%<DD-UT+IFAt@Z1vvEWRzjg;^Pfh{R#+a1N`bp0qhwbfoc?K0E*gBHJtPLaHUcgzH-OL{df!MKqSyfh{PYCM5&jxuYS*(sN1
u3fauj|pNrnyG4bde@3wk3~y$bCcda9P;emk5O~ih<h~XJrP9aCqZ;tq}XsIfhu=W-ZY7sD7eCo<Z!q6iBVS?${+2EYT=yYx4LJV_(>x$ik7Oy{iQ!q-2JoV
GhR%=$Ay|D`=4U&(OyHP1Cl#zmQgIC1y>69Uu#D`Z1yq{C)*BF3v#$J%CONpAb>!{ddHLOnY@*e0n^1>!Vx(XEna+G`k8n9r?rd&I2BnSq^ctf^DD=R?$qso
5*azFQyC?2-XQ+di{dBIuYs${y;3014{M6NKQ?<7Kkqc#OkbN~wB#q)3~Zn+DXbo{G5j~H7Yu9>FjB$#WUIZ!WP{p^#PEV&&?PE4AB%1+dyJY@`={`&=vB);
;P|e{*?LHMdAb!@Xg`4QMizC92eeyRIeJpQbOrTPU-vB@P>DPt6pn~Qcq+9{zg?I2m(_i_a>W{Au-v>j^X%R~A=FswdD)-T*Ll67m90}o1&J!v0+T9^n7J$J
PI;dy7jF_WVc6y%khVdGd-EtRv}`MA+38g7J*ukd<&LDf>gEmouk&tc4%X+@A*sqOs&Ek^JM`zTu}|z@@W1A041Es2<fX>ciAbq}$vIANh!{DNKh}5Z!D$#l
heYijl4dmtKH&+0=!FRC&Wvr}yZZ1CgY{~wU^+j+D6Z`JcV2s55!i-4QoP?#5EbjcwBwv^pic=v-T+VHqjxSkz6e=rxZ2Z2=681+CwlxuLcS1{@Ri@yyTh0i
0h{TmiotG3heI#bYv`8{B&8;y5l_!O#Y!ZK#OGgwAkDcc)Mq9ictt&-kG>(3zp88$Np?BfU3{Gxrczwn3s`b&+uH#D{g#Q`L7Izkvu30$Gh1q_6FWi1{VBj0
I)9F$0pop;E*w$UnwGs0W~=4Ea2?TH)X0>;#_aM<N}m{qU?9|;s_YJ;YeIXhiW6+0`97K0oF!*%m)%C6j6Wi#*{-?e2v^Y`sBnY(><<lAy7LhrPNM3Dek_ve
<D}U)^oi0{lf*Q!fd&|OD3Rih+Cr$LlBorJ4WajI!9#Cd*!jS$bPEWuh|;CFEheCD?94RQm#fOra61Di28gp)NG3!m4L;7cwvIPjaV`GeM6)%9ArdL;wU{<`
fTtbk^p3y^X;gb&`=CKN&6}1nF%oGhQJe!e@S0AtFV&ow`~#^=BI$74MY;tu)w^kXTJF8wkk*@q&Gxk+#}&xibw6#ek`bv1dkf<fB(cUuus}mawDmsa#RR;>
?mE&xm~vgKS1UVzu4==Wfv|>$O72Qkq-(?&!L>=RQzG+@5e@Q;^R3$fP_gsomLvC&OhxUge$PQ%WEFQbC76cXubhG@KhRr<S1(pA535nl)>~&yP7~ZrK!S>^
Xg+=vN6Rz1gy5dLeY7#kX(XAM4i{hM$s~VlRjRazH?FDTo3@3v+TZbU(o4Kjb&O&%mnE+ewf)>OVh%6IGP`z=;TCV|$UNm#=%2?=Nhi6jkZw}T)H1+lVP2Jd
UVz{$IlLf@JITyb$8s@O5fgsl`viWOrD(mH^L#PCp9<!TAYT;(iLB4<ESy_O$&h*0(9M^Uq*ZH#@CKDV)CC^<>Hio@cRtO<+IlU^sRW_`7RLIu0ca9qixAby
^k5;JbZ_@ys509TXg%bxNC?LM{h5!tNk?$uP+$%$6r$*_o?UKn1-Vt-659I<{-7r2ar-kRJ2(SRKqNOe*HCU9>hT1|FAdr96N_dOzXaZaxF;JSKu~7t?$xwh
(7a;p7q$Gkx;+H2>t&2D-`(izIkgn*T?>EkoWG|qQm-|Nvcy(7B5ek@Fp~OzgNV1ARy-c{8jZLXGF*+}hD%Hot2``%4Ccea_#)dW5sQr{!Ha-Zzs?^7KK6;l
dWehol470LM0j@RrXbjwPk3bVSvolXih)G9En89yRQD%NC__Wu?-lvLY~9?08RO)(C`LOdLGfF*E!d>|7D`%|vC`wq_Y^dni|_$Vr)YNs(=QojOuG#uX69B+
P;ye|wy1^uJ$V+0<=aS;6@d<CRGHxHUj&6#m=n65T4OCPwQjInY-1cTELk1%uEb@+VCSV2)sM?n4?u^@Z?I@mX4$4SmQA(2dm^AO0t5Kv^g8UaZ~{k%`RhaN
^f5tgh>f@SW(C3YbI?8`TXG<ku-TX`qx1}(RvUXkzSF-vPY)7dm$RigCLVbwsCXAFi_uzt{4!LVBv9i;)gi*<9eo61v+@uhm`?0GSV9Z-GKM)~2&#kWuH#XU
7UhXKEj|2bH(GZ5iP8i)jCg)wIwT^-u2v%NYu`hg8&9U(boa>EzfMSxTEFw_5(!Dhcg_0`kU(I)0qxU|uwS5ps`LF|7G4b^nI3<qIFqf3Bs-hY7^&CI=2e03
?4Q*P0&>I{huQd-W(LN;@n&>71Ci%p0cZj?bVhtgk`|R7i>cdmZS5k~UGV^}s#GF2>*njxA^2JD35zX*;0T@7nO@D&pkC0GuQA`cNF65SwOA5>0s|oP(mX1F
LEkb!l>MKmZKHF9{_Xo{WZQPw40t@X;9W5t$3!}vC_IAoBEd~OJnKQc#7F+DF--;*J-HuVJ##7hq>7Y~vd(rH{SyWf&+q@WtrA=kT#5|T$rT$U33GrO+H{^~
06}#H<&Qyh>CFoROlBwj%qX(1X4EdQ5r^Aq8h|Jyaj6}F$YU%dLCUvcT|ve_R%T3_rcQ}Tz=A+m`>xsa6)2+JpQHy{TjpaR8U(~)WFBc0I*o=A!;XQi$uYad
x5iBkLT&(f#)f|{`Cf7Mp-k>$W$od(V<H=EL<!%OlX3bw!n@HupsWxpOikkgp5ku3TiojNP_R%;GM7sO{&3+BOD=H3@1aEOygrL0<XKUSO5xXPypo){06Hol
f_zToXiKgE)g!)9&?cKtwre>?(#wawP}*cs(wwE|E;!?ODOZJ-wc=lPwy&i*Y^sgO28txCz625};!$)7Qr>#lk5I6m6tJXwj-!gz;a+M3k-cal-6I1*xjL9X
C@jnqSC9pGwURAU0m(fFd)2Y6sg;fEnW3>a#$%Lganw}LSB<dSUxogCYOidptrdn+8S1l18;<`o8{O-Ox#e<An{(gmhk<TQCsle)hJ+jacL(UeqZ)E+`(--$
md|(DDskx+=(Vnqdc9aGIjj5VYHUt---&KL*^zCPAqH;W88jbx5wn|N1Cq!?miv%@U8Tbe!i0MYG^e5Mt33gHKuPKclw=S_BjqSAxnTiwzM|{!pY`n|lzZ?G
8MCl7^PXIxdXeXac}z^9@Fzq2R<rLMPEkJwK)6u1mkAE|);K+vs)8GXCbZX0;s8sa?b<`MLTi_-VNCUNd3HS<&9JN-SY29`Wf3z5SM8{90;z-+Z;ec+ad9}0
;bk*{nx}J`GiW3NiSK59I~4CqNxt}>P*J?HDRsXX7ca$Ib2Sfw%{Bnmcd^XLNKRJe7B*f`a*-uky0P`^AwIL+y<BC4b~JK$P9nr=%vcw!zJRTEq<e%ri0r#_
V5X54OE_OL1LH9yqCk0dsP}3JSLh>tWe<L#Msy3vSED?JV`kzs&;(epyS27mwGb{be_Kgef9RNu_<Il6QiC>Tk2x|mesI1?c~v%+9?Zs<snNHLKLU)M`1oLt
zFJocZqeQ%A!AYVRj+)_4@HK)ImLZ}cdmck_rTvDnaP>B$Y4_idsn8WBkCo%m5>_^W{^*=+o6&xx4LHD8#)xa>!IsN5IhoxgvqmlqR~6VQp`U#g;i7*QPPqu
{>Gu08Ds8gQK!EPlY07B;Ss*N^SLABov0_hMOm_1XDDosQJ>>)o}oAiN>k)>^ChAVbVch2M4KRB6D^)`@(Fr@u!E?r$ma4HyxR*^HBbH}%?KEQcS%g2MM<Ck
BMe8J7K7`Uy!qz-^LAg6;KMy^pv7hCisY+$p?g>a%ICnZxr;U7Lrvtzur#&5zt{LjMIcNOs#H{;3@a3;&vm8Tf#^<YgH3{9kPem&!FshJL4Huon-W#tltyAJ
=HJ{b=w!V@Tg}C9bMzb2>5(%yC|IWvoad;T9M>V8FZhRNBy{h&?b3lVm&1<$QOr+3=#{z`LDjA9P$RfxP#iqsc)jSYFzo1a{~3|)xJkq$%|4jF2=p8e{=#oK
8ba=vx-*-^#W0GU@Vi>6Xp#X&`$C2(<y&4k-kL~5U=vGLlbG^>%iz3x*P+cAByjyD+8xHid7Suw2g=Bh7~;KJ8OvI`iq408&=Vc{4qQM<&|rBI3l(&F9^m%;
oH&#da)LoDEQHhampvd>rV%)Ja+aplXj?#|dT&emv#~xs<7WidSt_doe?pbLrr|VUm}GORs{5GPni&1DS?T)=c~ah$7k87JveII6(7&j{KttG7iBYqpZ;Y(m
Go*&B1bReru14u26x+wrOOGLoOF_F|b3T<sFT)kGkSh9~S0wkwES=7A%?1UOu`}UL#S(`S-khh=<Ji>BAyYnvQT44|r|8{h>0IWQA_L->Z+HyGPBMTB)o!K#
{1DeRfQ8Eat28cINQSjs#vJ(*l+zV?2I|o}qB(|fRw4>zFmXMm8%B{#BSQ>B!HDp`A4ggsL&)d7HtUM-q4=gwq;j(95pRF#ku|I3=jehcnBU<V&T040q@yt^
?AxqpO_<?g0i(?aDDP+ID_z36pCNi}v8Q=DExYL2gM)OG_$}Wmg`>P&dhlQSDs_Ozt-<ZdE{ZnfwP30Yo#$Pu*9NK|=w?vJ81B8)@ZO`LK&;G4$N`X*Ru{kR
ZB{=>%}yf?-6>#C)=_WV03-N}vFC~;_<lEJxda@hV9Sh$^9uR3BhP{8txb(XdsoLiz&@Yin>V^5I_(8ih$mWC5R)Lxrl^Y1ZQ2acL35oeuVIKjk@HH3`pv?A
=AH5mmOj1qj}6xzWmO4J|IJ?XDs)}r%Zo(}Y6XVIm`<oIIlC6ota+q}U>_AesLd}_fSw^XQku!0FS8ch!n^^&M|@XEE?$=1)>Fn(34PZCU6pr}(^4R+|4~fb
LYWtA75sVMcX;vh6qsND_0O*7SO*F+xSgOjMy6W|C<O~w_N_uRi3mAPr=kA4=>lg;rZ|>jt~7#@NT!*XNvBB6sDn}ktJ=Uul-ecNL=^9cZIw7(RITEt%*=A0
c)$oolouuA40Sia7p*dMjG06ivR&cj(W*#-skdgs8}^nM?23Ew$nIGKt#=b*-psASV#&{y$!;#z{`Sl?Kgv@lzTZuRD6Dt4yP0nW_M!EcYNy8uj{*BeQlm8@
!Vf=>M$63P&dX@VS(!MMjia4ei$)Mu>l>;ku!(zH!D&dIJ9H^(1~sr28yt2YBo=kfnL7QuL2G_t`?Gqe$GkSi0Ct6QWaucu4UUx%dfv1&v`=APrW?lWBB`5d
D|=X<L!lWeR{O=K6yf@nr<of?raatU<n8df<^$Cc^95W3X-t+}mv0Gab911BkXO4zmn4U%pP{ZqEQAAFaj|H1V|I9-(-0+oEsgNqStj^SB9L}dkVNDu&fI90
m~RS{{UzWD*VnA&Q0pc1eXF|XOhX%}@mq!gEOs3OR<%<WLcB4|GJ+v^v7eVC);*X(5KEqRLhyHSUOz88bXwiU$p~nCAQ3f=UEV?KRm*xN4ilLmn*z58Gi#Kh
=cL#HdO#j~K9{h*IxjD%^-B>HJ4^4aUwOk;Ae&!tFG_eZU<NQ|_w&nqK%}|B%P2;rdNU1<(b_b*{d6qGdlPjwm^F>Hasu4y`(<`><&rbD$Xjn`p+8MX+;D#+
jL%kcSF{`2QnL;mGx<AGx1kcnJZ}A*%IR!D@)XeR3qSm%xfEgB=slLmXWP3Bpk)*F;(e9>kV6n~jx{V-5o<Wj_ApFJI5+E{?X7m~QY&H@Q$%!XJCu<3DYo?W
jSp2HT7w@~;b+SO7j>qd$ir%TT^ues!TDlohui1bHaze*f%a9F(I6PvbZ|ex@$;h~0#T(#FfVV-hMo7z1r2O8yH-Mgya(pZANX^hy&Qq~EC`p*^grYxnI;^)
BZ`axwzI^Mf5ebVQQ_tIrtZ1AseHm#mb#fh<tdD;wM}(PG3n%cDA7-sW?VY>uG633+JhLWChC&CI527LhFpXzVS|@RfI~q#M!yNDkJP%NZ0@xgs%t!h*hu<(
F1;srf{v#>PC8Biuep&-WCNmVxWv!us4XoTDNaEEmc7d&-Nzg2`U!$rxR5bj8X|c@h<azmFuxBiNpJh2<ZP<v+|V_!jP3W@2~G5e3%1QYJIryXpa$d2vigTK
(rFk=j~UySE0G5l?hqjn|4j!-NcZ!ohpV3}C4!5MkvBLc9!byMcG%5sAXibNbPN5G){x*^u>oCk%vdK25*ZCRQJhT`ATHPMKR>|l`l2#^Kp;<)!K7QpyqKBp
#dJZ;-)VxJKUK}<f@0Y6iFoPi#41!1{Fwt`gZ@DGJk7FMG=)~RQ*m0t@^Y>w2}%5uCXi`jF+Z3L@<v;47>Hzn`Z|_(a`+IjMg!vXvTJLtm0EEtyV8FmEs`fa
c^VV*&hnn|>;twP#zn~6ml9JAwObC;(&1J!MPdqN{npvF?_H1H<J|#=?7q4cvxys>oX%cSN8Q6G=1}58oJ%GQb-#8vDAFqjUjkv8**#J!8Rztk3t5UAF$jAh
b1_j&T_*k>fw0CfoxK+BPd`Is^#}i^+ZgK?zV!iDk*4=E$^&%`u0f0=7xxTqWz}MZ!|??tH5*fJ&YF2Hs7ab@qsq={AlDQ{2KG^D<7IOq=q!iWs;QHwJRJx*
Sf~(lxfROtW4~-+Lr^|F=op`~%f(>XJS8W{YB-gQ(3W@fMTLKlt`xu0THRf%{P?y2F&p=Hkp5LGsDpIKWQwI6G7Ixazzd^K{xA1eO*OWymc3q>Ol>TZ-rmJW
Q{~b4%|X=}%jY#4Y$gDq1N0C`Lu^v;RN{CCQgUz}Ogi-W1pf*$a*bS~SE17kAE@FRu8pOfT&lk*o;c9)kC2?fFmvdCLdtHv9O6aVvemap<0BU*bKh>qfJJRf
{L8BPOt|D@_s9J1zBbO15G<LM&`l5K7TtxXlwK5E{z|<k0Q2N{91}py#Hk}E{TLMXy&5F?R^GMg4A@b2<>#`i0S;kh<ttB$SedY&Y?yccGTdMX>x_q=W@#XQ
QyuSk6-g?anm1*`Qvo(~*Q8x87!ffzxYCC=g#syOTtWqXzqTi2^Rz)L><aG52x5g$_R>qGz}Xms=w#P*;gTeMf?s~pA&_I%Vh9>t6N|{fo&JG$jhHp)L=y{=
+FSrlHA`qr+6g*h`{I;9;jtzx=sBiE$Z;OT#W!(|QY&@Q4U_TJskNL#5{Udpe&UA;hlc-Kp$}P4(IyI>uD_e+otJOZ+tuu6_D1Q_-o8ybY+8k{2-tA4hct9V
!{w!B3~nnE&OdprQuXa)$ntNOS<l?>{8u4PR>H%!LREK;sN3bNYpTX!Hzk+qc$6W-N*`$Wn)3=>%q22yO4Hyp3|Plt*cTQsm(fybn5$?j0vj<!k8PuXTc#I$
p_d-1t_d&|sMzqV&#-#p;&p5_G23rX&HuZjc<N)Cf;5l7xi8e{K_D(tx%R7lLOJjTkrHez)WgKtfPh|V+O1LkyBHc!@OVM`6$f~w1^DrBdw)LX`9vT!Iy(NC
@RguhYMeWcP%iB=d#415Mn%WwEm++7t_=*3@2@3Ff7Xvmvy<b{Zb;;dQ&&$^x#y5xCr5kSjXM@nV~0WG<8^bw1uV1bH(sXSU|~0GDw~^aG6%#y8{F4Dxn*tM
FsBU~g!VT=h+Kv<2>wYT)iym^!jYuXboWE{Jp1POSF!Q%CvL3a)d=2}vs$7n?X7Mf!DaezE<rj>GE36p254ia;}xjtJcJaVr??JTz7Uq?|KR=TkALbcSV*C#
`w~1>WCZ(Yve%v;+nZ3FcZJSmx|AavlvFgrxe(0hxLL{w<p5r~)oeBTMha2Awl-)iyLaCgz^t)4De*iI4m5m68Mf1}rdOcMSgmrsYLJ>GI`@cyU2y887T34|
Ah<PR;Ut0w$F4HJ@AEky4qyps>yq(v2d7(^IaMRNJ><7hj8n$KI`OV>f0@Fq$hdEHZ7dm~)cyA;oR{zpzS3-<5~q0RQss#n^3COojz)9fpypdiO?C>(ZYHft
=A@*V)rj8*pBUOrqG)-faVIcfC07f8-jDLE+zl3;sZ$WGc53Hf{WLe7fdaS+b;i1O0gQjtEG<vG!4EC=bE?$oL@HTP>nIEIaky~q!r*&eR$av{qJw|e9;6?0
Y5Xa-cyWPWZ_ZNs+`3Udl<HGI6~WE2Is^_>XN_(u3?W+|Tzadu?CYb*tix497@D%HuBA}}jX7@WiMV;0&e!z4<jBNfA2AGaPP6^nu3`OLK(G8zAQEk5lQG%;
Hhz_V9*G71)RP!bRKLa)Z9|VRi$5^uv1iL0`>l!EfCw{X#IY?0R8<MfP`<Qtn$@auiM^EE_0YTh+7Eu=dDb|pzLRIf-XH-?mYFBh;M|yeFWZ(&nh!>bj%4<s
B(cGMCa@-wsxBf3uA$2&#V7tWTFOk^%?F=gjYf^}QGzNi@w0#2K)foE{2<;~TCJ`r8gga7*#g*c8&IGXz<-^#r8c}wgq)VBF%;2a%-K1g(3222x+BiK#~Ck6
DED4tKS}+`%Ea4yd#Fv^FKlWC7!qKyF~+`#PTdw}ZPL3D>#7Z*dYv@a+VD9NYXod{0q=-@)gi&fD;y9z1#Ue76B~$-%0+6`IqAd)zbn;!isWC^;Y{f3Nc0H<
)IzG6Ua{-A>)X}Gt=j>@l>xsbWZfVg->bkP&&d0jGbUHDQw6F0!kr+RAPR8n-uPqlT|;4|na|y_loL*aBm*Ih#QC3nH_QiXi=ijrxc$xAi1{!;%pq@IQrmV!
3inLZeh&S)M;amVv0;B-+ref^Ff&%G_e4FYn@EO}gT?@Q+-NW_s}$b~M_mfAT^T$@z)q-e3dm*sxEV?kc}RqCcJCrmek7}_e?Y!(s|o4(@O@--^uRsVGsSzx
(1H{TVIxZ`Q5o`yL1&%TwS0Uc^BnjA=nxvg{I0K5VN>ecbRQQ%VHYPegYwB;*%$jl+HBwthgpD^1;eSF&imqp+D!R+BmeMee3!2W;+k0sA{pfL>Qx8yx#W7o
5;K96A-Mu<T@W7ft#zc0<V^JOukICAcP+i@wc3PLI4b6s73}KtRpA9jNuEa6e7S!f`0M=(fZSuDj8ntNEg@XCzN(-Jp`OeJanify@CNTaVzSHu5iDO<Y5$3K
6p*g=hq10mo3USju6F<Ev2^v;eE#>3y=B~&Gt&XmAzs3YNRf8Hxlx*f^|b3pfDu7);yHLA^3euKc(F~6_o@jeLyk=Omctqr7_!M;ml;bd)=3P@z~6>lt!M;C
4O!MB&Aoc*j+wj&FMuohagtQJBE=pjrPG9yz2<1#A6pM^jz8*3W$lbZlQ2y@9N)U~zA7?Np%jiKIiaoBC@DTuRqWb-`trqVfbKLn@JSrJ<}t$n!@6Ihn-~%b
uo#<DB5mxw3d%6>v<FmSm%3gLx>&H28AX=<uL5LEQSN(e{kM<m03a~BKgS<vs$}Cey_$on_;0?;qx!REvu+Q?0$&(C7Prz60eN%-#Uh;}&m*aAN_-O{LY9`Q
B>lXif_2F-_qq>Y0V3e-da3KTR&F0Ai;F!7FByHKNYTSQ!t@uLpFV$&*V}t6p266$oP<`n^mo{l2!{ZsEKJt~rr_zDh5s>6n5~TyTHW~bB^ql*v5)O@P(pRv
E!0ct^iI;53%>tXd7mAg6Q-5T`^jrvjF6GfPnZ`SLjnNSSE8N=uM;f*#f2~aO?Vcd&FgNx0S*h>f$YsVgYw4+c5KBM2du#$)S9CrKLNRW#`E8BS%6sNZ$rS5
&-oy^QAer?KkKI29FYlQg>;B)9x3(G^?k8IsR}p=gASEsvSoH>nx@7p-3RMQMboQvxb1<Td$=qXZraWJqM_(ofztg3{dp^4&&J7&4XFzVLMI~eLp*y&^ECkc
eh8};n|@ErBov&X;ow_PG$1!#K3vY=qI8d)Z5NMUr(BMme#Q)ee?{98Mt;Of)-G@9<{xmCZGB21tn>zI7XI5pZS{3vOW?3Fg}qCT(ej;1n?VsqKFc^xL?5jZ
?dylRhv6r0caUseKn;K^M#6L8s-XTXxc$4(ji5=7nNI-_32g~}()<XN!Npw!?ir6piX<63b7)FEi8<sS?>1MC5d=(0hILCw0?TUN1!;=X<nD+?ooKp~KT`_6
h9cr6_Nm*adaACHQ2}sAv9Lf}dA1S&00000&(l&U%<+_Q00FgY?wSbzBcN7rvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_flavoria.csv')

In [ ]:
# listas/divisao_r4_foodbd.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_foodbd.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;b@8pg<SwL8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7v+CfF>~C;QQrosS;_$%kDv#IJFx}pY=2av<3Kn?467oD(2m?AaT%IkSj$p)f0Z(RW(fyS$dd%?6jQ~$NV33~0IxWbj
Lft)LN`t1>n_Cu`l4#)`ZKhRed-N2?^`j-b{z%pEmw^dmYAnzfj8u6-5vXMJ6>p@#HoTw2b^N*CrQ**|CEl$kDl2di2^gl?5ysnmeO)`VsFsGQbw++W77vch
_(~h2PU=C>bsPn%g1tB0TEKGfD<b<e4x+nMy}K=nkOs_pYug&768%|5hJnEAqgXh4m_d9a#tyRbCKizitOuMt$OQuYWjiuLj+_4Q_~@Q3ltnEGJfAlT1`MM<
Ss@;Z;2hK=Q71xhnh+lkM`7-X|G=?emJ9USUs=Y!`n~-jQk{3&?i~TGK;W6(B1m4%suo9jb^VNL!A@6Y*#pti53b*rqf^!zp)n|r&*<VcjsljlH#PCnh0MEd
i@j2jGMQ_$iXbF`0khrc({)p5#ZQ#Vvo7~*b~=Z|9miE8;T*NZMNHm?2Ne`<8jjw6zQ=(1dzfksex{cWet&iU<lvx2evidaJq>X<c73Z|G!W<g&~I!xlE-(d
VANHex&4OZ_|)U*tT1}e+KA25D5Q_1XcOrr9LNoXltOC(X@+P?m~vy1*}vR#4JFc{HqNtjxz%{kw3-WLs4kavL7ZO0nRPfX0TDI>gcX6IH7LfVRuhR)WM`|e
W{939vy#$mqA1&fg_vr51ex4~BWlDI3Aisx<S$a121>~?!>N|LsTla=pvM;bLH!;ms|OCkhFgCMzls;A@MTY?a5Ia(M$pTMskvN;rSLeN><zEIM`vZdD`W#H
9wp&HB%Sg3n#+EDh>Ik;%XLsJYZ{zx(LJ%$1A`@y>=$BBj74kon#X??YXC&x@5@jw?ZD03g|2tAW1&>e87W^y%OgiJcKm>XoI;)*`nuR$H~&0S6)KQ-gcvR}
c(Rm-Ec4h*z?OxdOeA;U6LSne2l=FuT!UKb0lLCuIpk+f(9e6t5?;cG*+Ze)U6cI}e&ri83Ej>K9WJl%?BU!(Rs_{uAZNF2#*4+v`SvpLhfbzw@CmCZ<g_IX
LjJ$ua@Bjx)<zt+Rt~Ehn^&zLF|;d0sjj;IhF|AI76)$ZjP8$o%jq@*r+1zW`DogXc?n2M=6e!3>9Q1FLLaj|!o|jn38ge*Y78YkPf09{V4fWrZZPQJy`5q!
i6<+ct>XX~IJ9FO#yKMwe}Gd&aS_nDyj_@LCrRZ7wpvwikLJl)<Jri?_LDt;7XSM4NRICFk*9OiBH0^zI?!YIe(N;!+$=>rRdP=wx5MKq-R1zfkW#2{?GGy;
&*~g{@?p0m_wy)nzNv%uwHKIctbC$5-a@}Vqx0>pr39+bSh2o@vJuEXLpBtYY7OO$#EXSu<4#k0WH6!bTHJ$4YtOjurk}5UG!xIp%$LUjkI&T12D!nO98aCh
pMtgP8B!d?NAP{Bb;<B|KiadcPPP=igc&{948u0xBMOZ6Air2LQ;6qGj45Y8P0~5e)xw}Wb@r4d5@pX5IsY-=js$*rJT!2)&IyOZ49=@+3VeZ$JERow(_O&~
4e4bUGB>jj#&2O6PF+h-!QabH3`1oMLF!G!jmqn!BE)Eq4>1s3q>T>N;`Z~D;t@|W37pANKAHq^DQeT!fi!lokJlE*bwKaiNq;LiKk=iqmSn5-{rv(tTkC8|
Be*7*I8$w+g1`ml|L!IITkf&U?qnR1ZE4y$p&wXJqNIEJ6NNy1j{y7T9zwc-b++YiY?_!lP6=KDm=ZPqSI$10b_w?7`*p1qm+B{8QlFd_+5`uW_w05|=bu)6
IbiAUc0quXu92y}wgIPrzbFPTO4`fa>sH56{8LyIK8pvWQV7Fe8ae2|<m<riVbggl7!`Tm0JzcYkjq8R>Li2i-Gg2h7JHHwzmA75C%W{9lI|XS;GU_{)gu8<
@T3xDTAQed+i*95c#gfuKh6OSfd|6W?MTU`cqhOi4w5oLA>@+^-~?|CHtC8$4W=>iQD=xqoEoa6-|4Mp3;U7tFJoKm!`pI6<;C<yK)$CO;R+4WRDk}9T;IfA
CFt{iov%c~XJuN{cbU@97<r&^n5!ak&0?y_|3*1thE(leFp*Z|LQVz5bfJ+WfPt$K39@phFJE#b(%``5{;ygXLx%4yFOQrSy|I`-R2#WG`ndA9`PE{KlWXOK
fCBe$e2aE$yL<vgF;0vvp13!${`4{SoYv%U#@G{PhV^ptscJUuI8Z<%U_4JN3gqhqWt|~CCuIk0I0>892}hI-@ed7M7SI~M<$0*ebz7`IWGy|w`tB?tqi+c<
NV3>2V6VKyFn_8*B@p?-j7+WV=Lm-d@3M#!Eg+htI&>wW`M?WsRS*y{H7C;jHSwUipk*l;pU*(~o#KZpyrE_!@S6=#8CJw<hqj9+Pls$jFeK+fmaIcdvo26!
;G(cv<AElU!J(%jzk*A$&%m<h_*y^$Msi|+n#Kfi52AYTm1S~47G3TWDRS-AmhbH914~Fa1A@hdc14K8cNrZqi3~kQ^lVP=W$&=pKA>u?2?&r(;XtZoW&_}9
67;livPVt$>J5*17~o1GCSzXo862?s%u(L)-sL%<VJ|2+CkRlA)-yuBz;%H%HIY`=_GL&@6EE%n+@+&K7`=w*AkEUx&nFl(RU96Gw_hVKS8KrX=E`Lwfy@&M
;PbCFkQn@Qu{V!S=W@me6q7OCqu4(bn0K!~Y_DeXYyn1sw$ljDbr)pMMAAa78SvC|2Bni$-{5a8lknwpPe$93t-H>ZQDt|MC(_jE9kA3*n^A8Pm2LHnH#evq
6Ng%>Cr#lF&F*E^luh&yIU*0mIKdu!c$t>&En-}r>GcD(i}p64y1EHpu}~th$nWhzg4vMG9M_G%8d*F*LQhl;T{Fs3ZA3{RXKJ?B0ML6=G|^0D9U!@$I+5K!
q*^~}k<%BEC=Ob2D2&wh7hMB-**XnBz_V8$U1uFNi7ZZ;S~aubxjW8uI>wz4_e1NpL2O}2B8jWPV0$6DTz|a)00000U;Q%XiQb3G00E&Bi_rxDykRl@vBYQl
0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_foodbd.csv')

In [ ]:
# listas/divisao_r4_lefood.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_lefood.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;GF3VkX-;W8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7wR^Z9<JW#Oex0-F69)mXKZ=wHqQ@7FPl+n8r+<lOj6-HXU%@EHSZlR1OGgQ7u=ipFZ-N)3U5&haau$1*Tl>2pRLDAw
{;Bw9$?h^~2gF>EaCZD83@v(sEYd2RfM2)p26HSA>GCIV>TOZu1sCwkpN(l#wCg{gVCV|*Zq&<L78E6u*rpwFO*!P;KYJ&4JoxH%z<!ouRIA1zQj$cgwm2$}
!^axxSZnvF1vLG!#wl2ELCD39fq1<6D+PPA$5q^D1q3(1K#^gMaY?Ij(eJY6R|mViVStAu#Dp%Z2||T#n*H(<|3+9T)?{uJoOEjq!Z*Nl@7HMG@21;+9={nZ
;_Vv1nz%Ps>8$+*qUg0}0(0{i_)qC=J1!2mQ<i&NNHy#<5qnSz4t$7L{@jaQ`xWSZ*1h6r6~yxD><SfWk!?&1DbyUTz~H!U3YBDqH&)^=4PkBV4F(6m8n|l*
@|hsWTu5I2vF6-ROVhNiyFStv#C3P~lPW3~9VxBK%|>{u`xBp<9PNSuEk4bqQGy@9+hA^C$P{7e+@$p9hsS-3UB-YJ%YhTPO>7~$M?T~+8u~rqWRWCJc>#IL
8>)_y-nEVpdm^#7(*w>zNcvG+c(-YN<KI?URvjbT+9tJ{ij~(LSG9nN)66%gLgp+db4*l|zlfJjRJsltwj4&@&;uZjgV6OpX%(%R^({r7Icy|*Q)M*Es&ZLp
;1Z|)xfpJGsS#PuAL<(oq7O5kobD(~Y}+o+40UKw&qS`Q8QkldX>J_sY(w}aWas_g)%+e{A}D--mgjC%NvJi!aP^4y65{}84ndQxS?)ng_xL|wbT;+Bo;{}a
OjQ%@wD`SOTk=&7^Mi`9ytXF|kW<;VS5-8#CuxSne_D*!p0R^`oQwudB|>G=e_*MH#MCj8H+P?@X17f<Q#+CK9=mLVl>wQb9@62D(4|#eBpyc6`CtiQ7S|^m
Zt)^-41l_vD+2bEIv3;b{$XrrN2y-`iE@pP@W{h7=y#yJ4rJrb@u1;r6Y-DAMnvSyQ|s;xOXXicis)2kx1cK93WM)0wXyhVbq9|dLdF3N{F(;1mq1>}tcxyh
Nf>;lZHQNi7yLfMh$Hairf%-O+MNpLU6LceUpSaUCFi@N;2GIM<~}86NkH6DC$LtX{2%}JQJ17t#zHYlyd>#9qbhhby2lPP7SU$_E5LC2L}ADvI@5X8b}p}w
0yh6~u-=9rr$5foaZFW7*yo@ESJ@=j=s`MRhh!1B!J<iqgPzG|<eU~BLZBLCi2nLm;Q$3Va3(nGTP1yH+=}fYK#xC;lan2O*l#%o=UfPs4a#|X!$zbPS427l
EYL?yOq};KiygtXu4oO|m(be5>f9dtCHcpop8N(>eavOi_5$~N(thF&v|GtNn;iT7w;7}`AF`PamiO$mJa-rO-$_Y09U_TRA+V|nn|?@G9O89W*e%Cr;6J|~
K^`=<7R~NB>I~%RC#_N11Ru-4@zHbV7*jjNHw=~s0KMCN)M$p|)9f@B`{<`xs|PTO?H9i&OzJ2qp$nXxh_kKc8fmT09_sb0$^y4(&a3k2#Pk!`MCF{R<P|G0
hecIunLORT@}G_jGTd_@NdRXuNSLy7e`!jWB3E#p$H#_+XznN$s~|>_hZrPjqy5uja-Z$NYBD(n24-cu8yohTdaWaP!e$>``HQ@(!6ab)%kZ~f{3AWy*8&wg
DPn!w0G;C{wX|gZj?GGSK|5%o>}#&W{8G`+1<-yX%0{ZhT$CBL6G<r*rcGGIaP&lyS`c~8{1y=IpZR}iU(xS6M`x~2&Jxa{-CJQQ%V6ZV1bEZnw{adI?1sN%
IrbeB!q%fQ^br*wEwY8=#h5pc<-vOWFfGzJUonQP9OB101xfGZ_nlm2Zg46}l(zhXKp;jYB7jE~ZcU;oe5yEg!1<LtkWEx9-S&`*nW1JS69axEkaz2?q=F|E
{v)grB1%{x<bg@3Do+H1F~L}LatrHHH{bJ3xSW0t<vw%cKO;+>!DK43l7D@LG4vvFSV82+kxt<_J_GycC^nTDt*&h`aby2D0koS<13v$)UVZ{|XbGc|p5$zO
`{v)?^Fl7;y<=Es*RuQx<cEKZD{UFA*vT_dIN;vR8FJVnoB+Ie4m04y6F+OW*TRrqf$M_xs?#Soix9jPywBnjP}|CkeUwzoE6HQ2!x<gFnzEnODWu{WWukW$
3&vu3eL#}jbDWmunuP-#cdMwWu&r4Bv)}?eR8rglT#BLAYMG_{N<V4XFGxWRZUng*C1Q?5Yt*)7>y->7AzrpKF^_p~J%6V^IpnmB1ZPKMk`!exREJ`qkaW7L
uUg`x7mikNo(#?PZ_dl~fIuqQAPjh)h(}+KzQdfFR^`?E+&<kM7EX4K7eqghZ5kTO23-Vc3K%LK?xjo_!vd8s#es80!_omIbs5{Aa8-)be(8zo=b4`rslCZN
%dmfVZ$N1H6OiYhpD9Yt?fn(b#yX{R{C^7-VC!P*8n5RI%u*0(DUHKFTApYtg+LqR7+*(dv3iz~C`Sc!OV5nucaVS=rq4mg#REHde^plnwy{&amMQZ8M581j
a0<=xINesWo_g!uFy0JjX=ph%hgAR`Z_J%3b*z=J!j<LghBp)v!Yu`ih4=nsdLcSa=`cXg0>N<`Jr(`X1QyKX`a~{U83UcOX+1(bp17apHvVYBI9BL6d!KoY
7khr6$*>l!5yN0G9CF=|*28ti_4PGHcqA+XPUi}cM2LF3pP<5!%!%cF7Hc<0QozoG<d@n=4DZiK>6#07K*!jvpX<%}uVb)gUSO8=r>!V_i&=F{AH?{9MUDK|
pV3)o)kI^?WSSq6kg%i}uj;}?FCPnL=w98IgQakYD#tNk%!nSjH-_qVe}^%^R_pW<HmeKfD~2J|O_5`CqO_@%+H{hqQ5o*}%tbUf#R@fHUlF=eoEHGodg`&_
4&uhPk(|Rsp=S%O6x(ml>7cJv+j$vHVf!T{``yR<{)v`*OHzuPyrMKW>rje32E&I5jSJc&CQZ-WZh>+dV76~Q%K5tb9kew+U==$X=#0OQLw}?$UrGdZUc-+>
`$rWt|M8&2L~hB=tN2_&QxM;(Z#`Z|_YG>p7(WpE1b?g&YK3v9l;k*CXXo?uy&VsFp@vvt#Owwtc*0T2A=<k+Gu5Qz_OU<I(_TD(a}3s`DmD$*d8U6k6TYgg
`BeKKI5<C!25C6_Dy~E1U58tRBDhnlgpNOY_11xqK;fQQdpDSIJ>FZ0EW`6oPQ8hljl(Jr^+{aoz`;<w!kgGPV8$V8ML(z$_EWPPgW1=gK*#x~u+&f4*>Zkb
6wSVeyJG)ZT*ipe^FqJ#IW&B~W7gUmWQ%@Oz@wSyarv%o$#pix5*J9O6E~Vk7L?{lw<IFNO=KiKB6PBYPM`<Ma}1l7!40P-hcC+VHph<p=El+5YJ!SOHJmbI
lZESpPs&-f<PksSHr#ftCurZlB%uQ5g~~gJgLaP3X#xESW<eWc5|Rb!dYAa(BK)nw_oN-X0Mvuh4<Aj+73#p_itkDFhAu<$L!)Pqw7w7}U0eg$D@SZs2e>v;
{q$2LZ*|O1ELXTcm~|sLh?vTf2iEd0jaF3sGAS&JbOMx=sZ(2-y1Po`&<-ZElCa5Y+~%JNOeCJRYw1j=(l_+r56@c1Q~nASK_!n}MaZeDJ959A=Iu`&m6r+h
qhQS4jU-tiz1!o>5&eZ0zG5E2CD4U*NYEHjA82o1FU$!R@4H1EZ@Jc`NMgCFCz&)l#}!4Wz|%`D{yq(2k0CIDV*9@GPEC7LJuT-@9ekRWWLqL9O^Zm4xmTw^
bOcB_TmXqg6c>X~O+7fO%&v>!%uW3Cz<3-;D;j89<;R621Et3zWRK7;WnNS_)XA!ty^Zwn&x==<2)8%mr(!#7dts*7^$aqGtCPOOaWm;ptHs!zcrpQqC@1v^
#G>RDrd4T}xT?|>r4bs_cuL#^yIN?9_$%5*;)WD`EKOuX0RWEQj({hRb!nU+zkYHUp`IGWn7OGWHx$#jrd@dGpu;-5@72g`Ex+$!W4Ct*U*NJ*zo=xCS9o$U
Q5)8=mp0OKtiqUp^$)M^wY;DWu2T6#;V8ZP@QNxVB=+E1TyJd9m*9Jvk<wY)cw@1jr|g3}*BQ7Og9Fz#ywu_i!3<YLyS?()Ns=O>Ci0iM4M-ts3yeU(oTUuz
K4CE6oo_f3Cv~0&bNx$WH^5G7&U0MyaKG4&T@#$E5Tq)P9U+9tf`m*OJT8$y^82yaO3^8&+Y2f&pbel={_q|xw{Aa&zdPh*%vp?A3AF_<=KLaqJW&XeN|*Ud
<wbJCF4;ocRa(iJe+4F9txrTA8W(%t%ML$bK}Y)gFDx)e4i;L8`#adtN!jKA4L4t4S2-I#00FEU>bU{{nx(+|vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_lefood.csv')

In [ ]:
# listas/divisao_r4_mensa.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_mensa.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;4Fp$&|LsB8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7wY>kRM9dNEoAvshJ9cQ8otbPUZ&_o%o&?Z>B{wbQgN{wP;Bsxs3myQ1bEj~v8<}<0zJzr9YohB6&doFyX{6r&knfRV
VdP7k?cDR1Bj-}IvNfkoVFwZ&=hLc|o_Cq>@~;jhD|Sl^1C~RA`R%bl0dgRTv{L!K_Y{HoAXJO;y-H}fZw=i!OlRO1F%qX1d{;lZ9$PmL$q{aoz)+<;=y(k0
)kc9L0u-;$U(Iy;2IoLPC%i)0nvaT%6QW)ve|!KOb<}=<pJ{;)<HwUao29_)shEX|5nn3iV*Fsmh|~mfFh3nYvYCAxVZDHg?>1>>Md@3~(`P}6_Ohj`<<Eb?
y<oka>@C%0cJj-NEYV_g4R_H&EP2}WERPpGwfss;U-*cLz{==^UR^*|-*;th7ip4l<_U-B;xSJQ(Ne!2d=r_WD47PLnLKUHtPCnAfJ5#gtNN#2ek*2We3}kE
!=glfmxoB0_B<ibMs3SyFefaLapi6fb$#-AEwe{*XOpIhaf;P+l8%F5#q;(>frMG5Ie6L|XMy48)`lMn;yl^IVAoaaIr}*QSE@yV78-9EP5qsvvdnGK8M%C(
c@P}&WpdCIm09X?lWV4UauFs^Jww}bKF3QgKiOWmosR3k{M@1m-~82K@S+r_CbcmjC05NIC0NDDhnbJ6$ibLh?vqYxCHRN}=h#=r{O~HEQ$qB?+C)|Bhq&L)
+vzfQG>wcFBtdWsbdhRNI6jbpP?6OYmveKjLPK0Pxp{=yO4VY6zJ$58hlBcNpn4c}6ons<H;oSJ$i}idb60>#2FX-VD9J9Tyj?B;sYw*nP5>Dt00Ha;hgkps
=^M~_vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_mensa.csv')

In [ ]:
# listas/externo_flavoria.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_flavoria.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5=d{0^)m;EG8cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppbIM9w`%CkyKXG=WZ3duqfbQ1z_g`
N2W}#Ui%oQe%&YTzlOL=Ph%`|Zq2Dm3%W2wqX%~!;@OW`NchuDTF`BJCUr*XV$#Qcu;Z?5h8Ah#NLWAV<TMBJ1(15_b2}_{&uDS+uP@I%DZe9%%k*?sd9Mcc
?qGL`Xt@s42wKQO5(W(R2o0gr&g%~-{E|Gg#r;haP)P@^Iyc+UEJI2T*SKP~a6ZO~#(VQSLBe6(Og}Z;2iQW)IbJl~9aDhwjADehrDV!{3*Io`co;w8PY{{p
p6Q9Z6@DYfuP@xEihMHfFTwm~@-qDgyPlg$GjJ^Jz6*dmGM{zu7-gfQHs{J&`jbx2V6L!HCu(&YH8$UqsPoavKkF1B;^irZ<uF;4Mj<lFYB3eAbB~<9JbU#Z
5#MkEB~+p@rn(y})~uOVUJ9`0B0n-Hi&nVvyGT!sI)5l-AcuEEuLJn{OE><CFPd%k=*ApwXx}Qt;wgc1{pM;!*I5I!kr+8CoOBm;Er1?;RbACx4kW54sVU)S
ekbdESzz*sSOk{Q0DsQCQe{LF!&)?e&1hKr|2N-Adm%%I@sy3+z%utA;aU8X4KHs3GjR0Y$jPSHCM|Mcwmw2vg?Bu@LDj))(W6}z5lCQPE$Iap|0qfy!D0jt
1B02O{4b`6Oy6*49UtnWekNUa^TRc+F)R{pWUA$#qk)M*Arj4rUFJMH%RTXNL!0M`B`kW2r-fiMLgk%ewh3<DyDrtj6uYi@Z=@A7?4$wZ_`Dp!@?})RNB1ek
fV|da_BLQ%`l1O>KJlG**Tkn&+?3&{o$ZE*i{y!EO7cEN0U7mF_OTYuXssL9toqTIv~Nc$j2;HU%OG3xDEuCEu|v8n+uHX3^d)TGCCc_M_4IR~=R9t(;jSJe
9ODMlp6B8TlFy};{<VtrB<njKZE$R#fVqUzb1LMLbf>9Qap{UBqbF`*GXkRqKO;I9Q>`2L0%BR99w+!eUoM47+l?l;_@qFM!>~(Q=<*xfMgof0y40D)X=me$
KpmpY5EtpMt6v%s5XfEVv~S;FE)4OBMr4~(_@Z2ASjiUjsrjW^kNexn17Km@4O{<jn;$t&oUsoUtTTgSaK2sN_tsDbH1>&5Rq+9JjRpAO#f75^GJ|L6ta<=f
2}LoPw0!A<>jp6odc?~=wE$q-6a1{Ms9JDnbbOBfct)#qe$azl_@K^-CO5xGt{X40L-c4=s>_i$IZx|L$k1r)A+J*84}8%g8d>z&(a$(-gyO`rhSJ3v5DE_C
ne8Y)EfR5AS(Zn_t%?RsO?(Y0<OSbC-po4ak?_0y8ON$m4>Ax)=2bVp&UeBnu~7~?2h&;L1s`*SxJ=NZ1(4<*wE5e{DGGdLZ)U<L>}M?EWhhfIep{!|YTj2M
g>Y?jj16)+^ZB*CMi{C1R~W}t^a2s5ITwE3g^Z|2FwZP8-Er;kj;Ab=J-bpR!jD4%l6=m24gH0(qiY1=4P$9_Y6q2N)ro3RCD)dk7jzI+*On)kI4&PYQ%vTo
llgMAg)5nhRX})mIe7D$OWqyK_b(9Fq{X8xr)V!y2ptkY3i{j_@aWu>fJ3f!DZRiZUd;{|#M;pzG1FGdq4se)%Cj{>GVFZw+TCjD&pFYLok2syP`tSp#6#9^
__o1MD_ugIY4KNFY-Koxi-_T~Oj=g{OsE6G#!_owerrJqp1rCw@e0$)LH9r8COtHn9fq^ciRIu(Ru*8%zgHtIJNT@xf1+ihz+uZea5`Y;-omSe85A%QF5$xd
l>~BbrxPdMi7bJiZTp5F28&<N_C@>-ii*bw$#HK#XoSwnyWyz1F>b>L6jkmGvZ(7s);wmcpUJ1UL-%~;6&<o*aJ-O4&Xksa%!=9wW_d3VRKPVaqg(pk+7!J)
tz6dH7|MV?@vZYm?>!-8bPsv^S;{H%l)`Etrp8J~Uoqo2ad8VJ-(jan;OW%Pateoeq@qJb%%=onH0Ubd$Ji$mknJ;4PyNC>oSc{A7IA&ZXQnzK7@ut&jaZxP
0!rdl8Hey%a94aad;3Qo?ER|)4HZTwpVyiZvqz4^b|W;%#wy+W5&`Xn0=sr<P39(FgIs`-(ZZ9F@pvLr<=a@3e@SZ}&uX}16c5v}o5*_*QUTo6(Z@HhVv)^S
9e?shQbvrFf+}^{XP7B!B_OJBJUfjcp+=4Z%;HzDx2!ik({UT!L(;|etj!Q6!SM;KzU&qkaEQQaO);u^mrRs*dfj1OPAK1T&ht?eL{N4$*>5zSuUO+n0ekZI
5WH;}7+Wg#?TxjUeplWVcac3sCN=tZKFXy!e;=@xB=`L!M;IWiu`ETwMbg}w0&s?HGdl}kChNF#x7O?<(T3q7Rg_rJkiq9eI(f^%BdP@foM==H+bn#S-GdcO
{|F@_sSP@%ZDe1oSL8$H<Ey}DSqV|kY+rloC8w3(*#4k>&4h#Kht>h$2qe<?@lKWbCoX>;Cm}}5VG@G7veAKg`{A}uSW`xwNww){Nwf(5!1=H%@vpN+wee1>
BGu|3!UV-aTz2f#xg%}&Y5J17$_FRSIo!<kN`ufu;+vB5?FW&o!gIC1-4ts7D+b^I$;^>V75BpG3_vD6hf%LxQQ8Y%hHQ3_bRIgIrU<S<_S9Qk@CHvI(FiAQ
Duv3x^KR5$_VPLya=U>giagbnqk^e<^i;^5!ihVYBhQm*;7@NmxlBlY1(skvaZUuU!JX==)2C1f>A604wj)8oF`4*p^XelB6sIVV*1MhNyF_9af%rx-qZY=&
<X4MptlWynHfn(t<2MZbCmkvDQQbR$<>}KJzqly<@X|(cgy)$_nbgibIVM3^P^t*1(nHgpOoZ)i3sj3f5#md5h$6bHRdnH1bhVae0tul2t|D>#ppB&3gS&co
Ba!jJM^J}HniS}z&uQdP0+1ntu&p@nvbXpu`mKHRCS}4k(rgCx^NIh5K`fc?*acVlIp<W$Ov{FQVM)Frh`}NGpWg&L`Jty-N_I6bKQZg{eq7}K&X_N^BczOF
Znq793Twr<iF-)e*+lK^Gr!})ZffdAwcEOmMhnd{P)Cr~nf&)I@1&j|!XG|iYSj4p5f=HhTCXkI%g<^yD++I~iu3~41zX52@23Rby`Z?C&pWeB{f}B<YJe}T
Y}B0$-Aaz4=%FEYwusO#fw$cG+?7Jf${hZ$c+6n7SfxLs^B%UCcaIkHl(TAcCU?jUfFU?0?qLn?3E-=&^)N9pHnugrB1<q8_qw)2@{f4=HRI7pzB3d!knOnD
9Af_HMN>$|SJ@m3>|w~BR|`jVV!h*+SsUfDQCs3gDY9f(&8-Fv|M$+=uJatHB>w7ze9Aol;h9RV?f_W5qpfRf$o|yF+s&lnJ;yn{R5jIk>eeh0uBUT8(iSV{
Ahh^9MhrYXPm{Z>{%^wwtD=lxn3GC6FiRu6x|uQ`8GyVjOsGs;m2zqqC->!R%p2~%TF~WgHONPP>!+|zQE?@;wSwYq1fT4j3R)T|^O^ib;?jgA3^M_VV3s>f
8bqqep2P8k@=4H;Xu!4?6W?@c@Nv*Tuxx@Sqw>sy7vc(}2LC2Q35&~sbf@l7Hq{Hkyf3FzrHO<N&!?oZs4__n?6UZ;a$-sl1)U8m_k>3+A4Wf!DGJ37^?{&K
l1&~Zm){>spD)SlK|ZH^rPT1(61o&i3QoI&Yq0fk4|6=!lg{|HMIf8_aVuachpfQSqx{&Z38gEH>wEaHmE+ORQ<QDTR!(rqx_ekMDF08X<lUBqfVmiTklx4r
wPL;Ylrqp<{-YX~kTFoi<TO_k>#Or=LK>3fo?Ah=THpCAyfwp`nsK1!>~bWet67&J$PkpmWSq0AF?PV}!b8A;?uEm~b9T-TF`{z3V6J$nbvBJlJ}_~lF#1va
(oj}ZaGo2)6w}!M9Z&kF4%aMN;+}y0)RRzw%@i^t32hzSQFD{~G$t!<aCta!DIE3w-iu8d)_y~5q?S)=Ix}a==RH?j%LrKUX-Xg7ax~m&%Iw8dd2DI<eLXc>
*ahdE5YCIQ)hXM|<~b>P&Ca+ld_M+5rPpO{l2KqLl0a3r<id@Kw=V<<VQdW%2sL7T`Fk9+cn-H=ARDOUzk5&n=WZWNl;(3qMWw3~ejsqM7&hSZ8_HPSunJ*b
$`YWe&Yr2}1otO&obtyoQP!$y>czXW(`(k{ktQ)m0yELQop*{<RZv?%Qd&s%ZG~vq!mz^qA3i=4u<C(9pKTL1G0vK*+6?HeNDE0B_vIb!{o}_VVkrTLA19m}
DN0ye%sw7MIT04n9R9FMvO`t;P{F!MSctO31>d;B5llbk+=UWjs6d^n;CEzmJL|9+!QgMfw_1=$oQcl@E1qvab~4${tXP<;ku9X}VTMI?LrI>x{cUG1Rx|q6
PRc9fg_RxCk7ynOa~xm2w^4P}8+(_4G0VBjp{Mn>ypO!Xd`;4DTkd~Np!G~noZ2~HscO;PM3W*&i+=t)4n9L<>+Y3Wi)pJ%!vU8h&mdO|m&B)k1PM=z8Um3~
TE_qz?!A16L7@Jxv>szZa)M9BKQO2$6I<5%4f1t5^`c=~Zqs&vh`!;x*Kr;YMJmEd!b@gMET1mP&8+nW^KuW{mZL7yqZ<}DVS+Io?lPS7?usFOzWn!O=h6Za
R2N*ag{|WBiVPeBy9F&M8^qW-6&t)jdem=Km&$L7P46S<$9hXs;r5WIpiJJgm}KdwgKUnQjJI9U5Y4zejw-6|c<RFG4*fSm&q9=j9z0xVH)~_{h};>*JOQgD
D~f9v4Ok$rD53oSn5PdoZ)2gp=PPQp@HPx{mK)o=!dBf_8+5&A`8Z89UH~>s2SGFrSbScTt}S38A?+ts9tmEGWRqh|QRDY1e7MU7(EX?MVl-IV*zd1}FM#RS
Q?9x;ur`jPy~5#Fpi&j@gQu!<u+yA<j0AzMS?@Z@gRMcj7x+cjo!4-81i>m!B))EzX@{3n(P`bz;4!cgy4fU8mrZip=HXTQ1VS$B4Ep;~lm??-X>w@mBZfw9
-94p5EHFqJrjkJJ1I0W)mMOt(B`oY9+J815l+W6dt|y*il4RnkEr(O}x2U$aF{#eyxmfX8Cw7G}&mbj8su#EQq_S@^JXdY@Lsftl9B(1<Mfv>k+0{iC#9g`M
O11;zyvNHTqK1Y?A=orB^iD!qm#*|v+G65u>HQ%<cTDM*6#xmtBzoLCrtw>B;&s;f3%;dJLX3Och`M<>6_%pl8JOobOLB`vOfjJ=tws8~>gp|@COYj1arAV%
=tmfVRBc|od#fNov{RP8MWQh8yu%HWmRl{!u<8Ik9&YnWEIdC{o!3qiI_qIwb2;5BLRkS6-ZJBY4F|pya2|e6B8>Sfw12z=r|K1>Rm<q*C3Yrq&mw83<-=Wx
F)^Y~4WoWLXh>|_Jwy0erjLZlbEfKyF(tJgD6*e1b1br}QjIm$C!SLhXcvRu^FtIFDo5i@$U%;SZy+-n$N6BxNL=8@9=GAD=&*0OpkF84xy3{Xq4DCx<*6r3
2V+#l0tGcQBkC0Egtp%6OXa8s&}O&v9t=f5H($z*{=g;KX6+n)yb)9YJ{V;HAti9sHoW=8FGY0~bhrnU`-o?cB(9^!t4Gjp5C2luf<l}g_s@PG1I%DBnmwob
sqE462`ky8Eo4e{5t$#vR*1fn?z&WGOP_76gvsJ0N|OO3l|Z_FNyB&${E}l=CqY8LNfpgofXxu$2wxm&vfS1)TO0qwPoB`+$#~1;l72`inx2v?%Uj$ohRh<{
mFVQN<p%GaQ~c;!B|KMfw)JCl;=te!86kwj^!9$#|5$0Lm|*5x2sYn^f<tTE2Q4D#QyU2j_R!nXpE-tXa-sF>yD&fElZa!{x&+-CyiHQsryiMx++(sus#ZVS
x!5dl({q*J#bRc^-Dm7$om3AFqmcyWpv!?nTF(1Z9vz%ofy3Z9TLmTkHHz$2)urnTc3ISzuP#qn+U65O-pVukVf6=5h8z=p)#*yYe%;H@I@w4qoGjwpU>`BQ
OjX{ldX~ecd%+&Y#7UT#I42D^5s&~dFqQrbNhHS2w-Qwut0`T5UhztQdfElyAHj9<5Yk`mt%jy=)rnJ;pDVht8c>iyQyeSh4H+x*3C%>I3wZxgFR3w0Ke6r(
40NoS`wX4O9dz6I^3T}1g69E*EImBH)R^Vd+6|L#Qxe)cj1HDWzJzIO(|XDc<jN!KV8=9nnCRR&<Dt&(psD-6IV+sy7@&#)uw~93x-dE1q_E~9E!Y&sXe+X_
UQ`u|woMdI;Ymb%j2PjUxfsB%)drnKiS=tETU^hgGd&5$qEA#)+f}OcyD;1Mq4$h$v)QuB!nxc?W;Q%0LCPu>t|!V!9+j^(j?JPP`+|LAd--|1YH<GadV4qF
i+9ZRcdE2xk)<922a#5d%jS3H%R!J}@IiF?VBnP9?jVrj6Um2U5`zBa^eIR-H|mr2+0yhEqqCqNmva1{ajIKpI5kihvdf4R<o>G3YR$_B;-H};TXoRb3eHE_
$QMqeKF~Z*f9sjs;n4%A*;Nl7`ZkQtc9=YS>dgWJ9wk#tDCTPlSOJe5aw*&%DBZD6Ld#c#uTV(J|6pH~+yI}s3|kmj5I5%jdGp`RN9zU8h=Ygf6&WQxI|&i(
-feGN6W}AIgD(UE@wl{sbZOUpLF=cxa{CnIwip1UFT+1kX3#rM1Ubd8&D~hWmM7?0k6OrK*-|WD=w~iuPmrR@<uow@Q#*2MF3_9&@cz<?@@?qcG%UXSaL|c~
*WGcB@VQ=K^-Sc%7s#>*{z4MH3K;i_x?R?fqU@f=QH7CH(;1yeQ1!50R1EenwS0GC0J&t?LS>E3k#Td}eZrH<tKQ6T&}s-?4wXRAj5BdQ6a<}IP&-zAD(iXu
Lz)6o&?B(tYbJtTYDi!&>r1yJK?}#!!0+}cK~Jia)lHge0}K=NwNZ{nW*ZHC+@l3lB`t{QE9a3VN|9Yk;3q%+4r`Ahd7!N~2N0o+qqH*)5PbG|*%}A3H)h}r
i7yIf(h<i7dUlpmx;YSyMfLSHhBPezf7Hc=jKsf(2wF==NIX*Atkpp+UnY24s8Dr<$`$^_J}%&-dqmYw+jJso07`+5Xd1TCX(5%{wfA|8;g+~0bD_mJ4_gCP
7D2W1+(zBkCF0i4!AT%hqh_gs2d7m5(IK=ukaYbvv7zDf$G<&AcVn5oGu4GGo%C%1p5%-q%%HlAW#9ugVviNZTY=YouOg9`x3*UD5UFp~a}@=jxop?@njpnk
4~QrmClC+===mdVV;~9K%gal8cT)l~CwWlM*UI_zIWl=WgCx{5(_(N;r)TZUm!;fT6nb0fjEMs-p3#EDjFT|;8dHxq$KPq1JHN@P{K!%or-@jtLOF2)3=<$!
kbC;?1|W+6SRy}s<4$d5X_=2DGY*OX@&x|qbt$v&MD8uEF=-(A>>~3~VIZtFawZRJM~g%Ur@G@eD3El_BpbXjJIbobz=s_THt5R>W)}WcwdR;0FUBTK2X{rp
|MG^twUH`qwy_qjw1ith92f>1AagKH&;3rL=1Jf<J3;W`#wDHyVL7#4x*pKRKIZ6p2jk(h-t&;iYdAr*FvjF|Jf;58aj}Quk4X-fDo2*BLnCzY9ZjK*b4a3D
-Mb3NuX|joQ(#Xo;yGGCQ_Y5ZxUB=)RxU>5JOHS^3=gOE6(CJ~3DCQ=IH)qKn;`yuh=G-UWIg?Qp+E<pNT&L68ghpm_7FP%i_X(^q8x{6$R~=FXRfs_On1{4
3HzTA@o0`%wVA7pEa|1?l4pm)-uG}8X0H?FqXS5jrQ-WV#fecWE+HNe*jVIGcji;hJvt@OacA9TZgm-dbzgrjE4g$JEAhZ9X{i5m!iZ!8Jz7?*fhuKcdV1L!
HDRh;*tomlY^2@4Vd^+N%Ee%&Zgzk^>7Vq-TBhZGj9C3ocfD^wkwp6%ELmU^&ndaeoq`VO*VLH=@qZSBs~Z6hG*JRP+oc4-m=TuD_@};R%MUdphfSY;(Kdfy
<tTc~xu*oKS>5KIi~5hfObz0rU7J-9)A0{-@;nrwu89D<=AoXppGtRhL#i%-Hxej8XI<YvV>^ZYg)6rrhKBg~@|rwqRSf`twlrg0+QqyPgmZj}^POkmJC8DK
M@7sPz2@-hpthCidIK5+!c14IKvY327;~sQ=g`}Vf!sc8w+G0MS{pT!fC}5jkcZ`3Hs9$il?)h9stVF3#;I)!S^(t`mnl}OavV5=FCxD%WQ>Zv(v}r@(hg3}
5ZIjfFGWLJk?>;X7WX3}oc-+aL9I}w+!l|@ziMtrV3k5)e<&0911Z$k<Z4mbfGYEB^1A}rzStL~hTYU2G_Oyk<&nDO+S6-OEEMj!ms2o!;AEkr6AO-Gu<2#v
OG;7$Fty=W_X_x<yod=NY6xA&HwI~S04zN)_wG^b2C#Xu{E9jx4xEyNM@g>X-n3*M2{Z5A5^DH}w@1tQGHVFX-jq`%)5%-k?+KX1!k03YeHWm^LeAydFH_G9
xXSmKWHVJ}_mDjHck+{WsmedEg5xI?CaD-PX<yTURZLAkYl7PD%-5(I-Kzp0-hmeyJqz%w36mnQbaxZZTOf+1lmyy4ooCg)tC4{)2G)kTnaUhK*dY<1W>llu
rSPQmD8WOS*C70OJ7N5D0NqOs9HX>zo%QoCTR)zMH$){6F-{Uhgb_yWK@-S%)j<xjlui|HTdr?$m|JgM$8PlrBcuNu+R`|A*yogzB57_OF`6*M5XKvQX-WtW
I(8cPW&~$r{qg?qraC$0sA76xmCsd0J#vzK=ESRQ@3we$1uID=@9*@Zz$gj;hI@~inA5Q2fUu!qx7)aL2@ulZ;!?d(MqZlFY!>((R4d5xth+pplalxer|UST
pNY~C{n(L&r!hU*$mM02NEUQo6u?^#q2^dS{1c2ru<u`UsA*6?r+&sY5;l{$r(2zbV+(<dX6QceitnYyimRQ93>>h;b$E7>xjYmB6NcbSS?SZ)fTIt8fdAdR
8od^ewFvX1rRSx)xtmoJ;4#2tALS>;d3x6fRlUmlLvltat@%NeT_vgJ@XS^VD|*7Eq1}BC`G*MeFxA)~-Vd@yxF2vK%?)<<2KX4yM1FFNh0Am=sX5j@mdbWM
;h8eL8Ps6ZN%~W&<(&MKffj4beFRs~!PuGAJmQwVse^;yu7w9LLl-|2t1eqjL+`eRrG%;60xDonVd3r97Omcxe#_jbZr5`ifRZRhfQKiVZ-%|ik%l!M|J11e
wIQYj$@mRh7`GvwHv=sk)O1Xi5Ilq)*mF5+Qe2|Jom$${u8B;BVCl4oeTTI;G3pc{jP2dLP0c-(I$JZf$R_xO5&_B@2Lb=(^yh+SlcR5vR6*%nkPrCo!x$?V
l5?hri?}qg$ou3?mjPDJ%K#Og^+DNSoO%TD5BDn57CYHXK;sdqFSGmFjWGqeeZeVt4RCS+N7i{!%K_$(A7$rrv7I&nBV4pr-eBp7>;YEeCa<J#CShhIo0%*x
ZhBLYom*dboIy73g-49@7_uB@8j>Qq8^}f_T{pItK6o<ffkj>_q#o56s#J(xK?5Jp*)F8jDPzc*?XdRV=X)S(9krX`+^Gy+X){EK3f>-w_6X+TZ~9j$bbTWY
$P8?MtMg|pxh10h^#_z>M+ZJUYyU)Go^$uIbo&zA)M=0$_qgU_A})dux;$<J+q|~s8oyx5p^-(sNoHybRCo3}WD?MmUzEvTmKeemLozzCER@a(1~25b;rL(6
@YF}2kjG8zKxaOlTQ<i|WkX<KP6o}RQzZiBXckjL=vz!U{MV<v?@%zF({w9DKw9Nt{`zM|xDpC=ZC_lz>nB4l@Z9XWYwDrs7cf`6B64|>%Dh}Aq|ayBejQa%
k-W;d4p`+n{xS#S-cCM06}C=L8&)+tvFM%BPIUQ5<e{CWNL?i{3Q<OlCW|!%%v<-rmY)@WH^z2nNHvjXlxEi-k8G6i#X$b=BogsZpBx<Ks8$P)FaocC$0F5?
8(a_&)O|zy+#MpIyd|uj1DT#yoK`NqqXV&Fk{Db3I4FlT3NR2wo<a@$C}lG-YQ>o-ucUJT5kh%2sNQE}C*MyDS3Dj-LHzSczjfjf^U}&8(q~t5OJR&=Gn6Kg
%sPOCEIs^+Xc>u$QFP1jZF{bV{!kSU{I<7Vr60A}G0p@JiBkKI8A%ac(O_)3(qa-!pLfWe$6Fy){G83Csd&wCrW6v`QL%^_Vwnk!FMgV%-e!qk0EiqxdGhe7
2Hi4ska^21ONG_|q5tc64`;BqH(SwR7=0=lE5A7a00rk{|C$}sh`20zx9&gp+&AEHx$YAVPR?_IP;$O9>91tCLCL1mfca@`;&g7vW7&f6oNsQJ;zD<b%rleY
rmF1e0;-6)Rm+1HR)$XzaduKBgBe7hz0e9@8vwUd-z7ZBYeS<u0;ZfEpyU?tD~WygFALmZUWO4^<#Q!ZLbBJxEtlAQjIIA$#laz`sd8qgtF`Ch@qEgjYn&TW
$16l(p?<dSeLpm2{L=W=<>9A^vTqgHvSgZ=FT#W>GLP;pH%cUwljs*Cik`?L0$D1pLfm1j-dCe(9kr84r|!murKf6OL>uyX0~J}c*z^qyzE*mY5&s=U)TNT9
FTr3~MeMoQR?M8BKrZe#^sW8}#kELruV0t!Ou$_cY;OQcUx0%^j9zeyF6Mx3hooysP-;jLufl!-6F*)*kcvK{Gy|%c5BrsHU6@4P@3qufIwj&FKKgt1qno_D
?FUl}-tFLbFhWjbfqB_3P3l$zL9&0q?+;E%a8uRCXybFSA+0b~Npk^m(qj`gUt6UU+9_w$FR&jpya+-U*&Pg4ZCL7<s7M_>AtQQ>FU;sz01bD#K(3fwuZ#^E
#<KocOZRo|NrLKBeQV(~L$1Z4w~%L5F5;GIoHWwVJNwTAFpQE38X$&uH>~SJ1UAtCxi$iTWKYZyl^fB}gKfGx@1rsZ$vbISu|voE6U$yeWn%1#DA2dP7t$Ux
Ewl1@q%0H4G|@0@LpcLhqFS(g24ZvARum3Jf(A@-M#?@p3CY?=FYRyriJ}4gVPb<t_{89)1@$R{)WxS`S<+f@Ar9`etP(hc>MXMnn;>&lj6`0y&H&Ct0W*X%
TDsYS+8F8pAY(JxwVZcAXnLfac%i^$WZ`wAy;<`isg!RiB#Im#NB53=({|RHSPeh)7XltPiVe3FL`FG=f=1P0&7$h^qe~%5VOnG2<OM9T!DgECR-i=0$IoiD
ld1$*l6I@K$0-t}+NvDb*MnuphnZ|J9-d0+oLJ_+nj6mog}uq}Q0=ZH1C6GvF>|YxUoGx_2%A(_7S7E7B$!z4h$>Hv6<r^sh_*qd?z0Il*`A%~p4ISbq7+T*
(~ZZqLbF;OkSYQkkd(j6Afsg237i_Tm+)EdLZ8aJK{z|3c;u(oS|h6V_wF&oL>xsMQS8(oZ9G%9>SkNjWsuzeDAu1`W*t64UxG@u_&V)Vvk64aN}cOI%X?_O
4h&z6)^+3>MSh1CKH<h1lh1@o^KI<tuEdQyA62kF-;F7H_t3}@?77Jrc|tG?PH`6fBx)cRt{pFr4gmJWNW-?gI>|)LSZ06te3H%0>^IReN_Jz0mPDKYV7G2n
jc=sR$=QWU2%Bff!v$vbtvP~+iRlyvfFxV_%=nrC3k<RK0b;VVoQ349C}JyCwoxnD-4$~-bZqTPq_2!UcqS2bAFwDoB&I)uT-?fBo@Z9;03ks|ZBW&-+3Lq&
SY&;D6SZY{^SsvT*})vbl~GO#Dnp>WzJ|j|P-H&UjG(QlHMxtMYMVks?lU3?G)ghNS<A%>C}5x_UCHgddtJZ-#u!(|d7+sx))p^b-MnXk&|}XwlG-aMw-k%g
G!~iF{2Y{MqmZmC{}uX3c|u8WLu<PT)rv$8CO@}He(gQY8SgS{ch(Ufg;3Kmhf^km{|)Dne$fW{iU0@pm>JI}lR+i}p`!iF3F$_28k5|fik9xsuA5MloxfDI
if=X7V~k+RBI9x)=*EQNXwd^oKhypAuE@c4qWqR3S`fZAB%sd@k2Vm5(rwi%!7p}VH|(@|MQD2k-N)bC5Jtawcdgs0k)kfy+19&jOhQgr+|$R06Ue%yN(MeR
-fFD{RO0kBSu|IaCRn<%tjO=X3ePmW2cTMcuRmqzo^v1=o(;d$+T?MCKCy}2^u1qhKa`TR->fbtK+z|YNpDnEv)yNIP?3&r3fGT9cSVk_&_T=Tf`C=%7y)Yn
vqi8v<Y@L=1scPM*(I)aGmT<~kfGSVymu$WsYmX@10#6?;7un{1rwaBr{pj;5GLb`MA?QmpPXMw)QsN@)c=mP8xBY@I_>F%P7HW1+;yVBNyeh12l|0K)p+?*
j=TdhOYYpp|NG5}f@wM+fGZsFIpG!E0s>2GSMSfr)@KGh%+e%ZLJC|s$x+ecNCXv00#+R8A3L=%fXb~@2pmz(^d`0N4x8CjrBT<Wb~yRl%FW%_RlSD6+GX(a
IyEf`g&Y-Q6U6A<^4D0NHM+u1IiixTdjG7TORU$k?M3r--BU}3(G<XDVmT@K3qlupdSI7^u4JNpL55G<zW7N6o+b2Gm7{MiXS5H?o;l@0b-?4YJwu*AmDO)O
VXW8Jk%ildqh1rMCPIEX_Y$GW@;xM6TMrd3zT7<^egg4jA!s60Rnv@J^}Em^8k}XqI?fKp;>x#NuIhMw-YI2NG1;}~#MBwW(<kA-aAILkwQQ1d?xI<4NMElb
PBy*FbxhZ=7*;Hhqquc)RBd|)C=50U`SyWefEiI31f6XQZd>uAk5juQAKOshmk+2bKF?gN9}}nFuRD12C6;6m?2rLKj=Im^R3ND^Ys57CsmT`5Xl0{UJ{vdZ
adTfrO7gf>aAc(i@X->2Ldjn^Fumoe9TBi1kn%y*2t&hr+>aF}7DAoRT(|dsn{xlnlD|X7D8U`H4{d+q>=3|MKXiu8HF%t*XBw;T4VHspJ^eOh&ruF-(~}mb
5{QOowCy-84(^Q?qo!mWSX6|qV&rs)`l%{K2HOnQ@`C{pD~kb7K(VfO+<4L|WGeQoYFL2tyAxhcaUxsI&){gkcL$}qI2!|*tR9H<Q?2a_fjI_=y{G&W7KqhM
QVI!6{GJb!!GJ#P8OJbQze$?p=eg12xx?xGXXeDfK_Sk9F2o=T-z6vq7W<7<+9|{bE6|;DZtILc8%RJw_|{<f$80H%D1fq&epiNG)M;?JxL&MpCMu*NSg*gF
qex+YGu=@;5)78E1HkX0Ks_=vPt^8AD`9ulv;&0(kfY^SM*hZ{!hg~gL~!L8gC=`G)af%5ZNUPWAg}V0)bIfxO83d`RUXVd=Y5tJxS2gv@Ycu&p;!e{9~z(S
ujTsIFy#%Y;xYldmlaIic?JV;IC;rsdQ*D^lv@&2zdmu<s52s$p{(eefl&GNe5=LC99Rg~W56R5_MW)1J-sB6Fyg(chOt%$3xKPMsTkbTyW(=MaI)naG{e0~
o}pG6(&l*6W^(KCsimL~nyeJZ828WR^{2FG&z!}N?cg6X-RKY9phueNKzh{~7^kCA?z|^MEOHBct{CY!J_6qyZ?a{&v**UU1PqXRU!iv-G(Of*up`!F(%{~H
40j@852BX+g|Dx^C=ob>{`osnccFi09|Z_+-yE~Z6aD!d6<WK7p%OlAw9Sce^92K}J^41^pL7XbC;l3dD<6Nh#!vhFz>S&XB>GH>#I@m{z8biOE*%??LI*P^
ZgzqQfPq`<AJ$+dj$<d!PWn3RdFN#0cOOnY`Lq=Ar5p}!OE>X(&gDjk*;!oc5#pXPjIl6y#jSRqD{|fDUBojP9xBPQN*8gdR*aJd-tA8BnqGxM$27i*)0)vd
m^{hB$(@)QWUxAm_J^m~MSF)CcZ0W)qvQ3LstfHoHajJj8sRm|x;wMx3w9&PX(d&|I}+ku^}g8%EVhXL+F)Z<X39~%Ums`}V+G;93r+~VoC^Ua4>7)1{B?W%
^sryl_8D0h)F-EL3M|5-I<*w+`cIIzYB6txJSPt6nm3Ozsuvw7+}^xoJ{pV_-H=ldZW((0f^H`&6e;UKO2?g>bY#<e0zghr>rb;uSBHYey8~gM@3+D-G56X9
C^H@q%rwI#A!$$R7#CoV0GA+rO1$WzUL1t0j5{E~yqUZWMHPi}V{YIApla#qzXfayN+5XHg<y_V+0GwZb^d-|g^IhtjrbyKI0|{Y;NI;<yUN@aE~hf1u0elT
p5i_=1zY11wIDg>72<0JT&l|X%@;}$n?wUo%5sB|u;Xe{hqP<{Da&#xjRGwmnCJdS6o}JuTZc~>3Iff9VZ+YzyP@*dR~%$DCaOG<blTzxME!Lhj>4BtC+yKO
;4zgY&(`$wi=Tvx(HquVqF89kDZ}A-t!UsB?*y8L+HQOP-qxiEHa0!sW*73O9Yi5OMYwcD1qR@lf9eY}+sH`Qr9MpPpRpTl*3@6Pfx*2rSP(9#f-#1{;NbjV
N9D$Ms`|{_gF)_>yR%Clcr(-=+0O&z@@XE8YyrSLhd6v$Qam<wQE_+sXv*&GOlyGwkjF-;Rf3DpCj1Q8f!(l{$iwh$;b5ZhCD=5T(3)Cvqb(A(kK+tlcJ^RN
=R6ButKa3(ycz(Ut=REYdFe;`uFa{AmyjdCe3tGy!A2KIMdcxplEYGFkB1cqPXqz9m|t2%1-RwiBlXX1@>Q0F{3$A+JgALJy&jVaKo%%4k`?PAOsY`nrs2RB
+~oZUgI!oE_;m;qrd*a(L(#}p#=<fyLoTYd*+j{n5z6uy(g^BFVqI$k*iutf;WW__t;B)uhE4JUL|-arXuMi*iGbl4;NHSGs{F!J|Jfy_n@{<Z-VZzb8l?s<
yz1xi6&YR^2&uCJ+MX-=N%GVTNb}g%<}qu_Kz5-ryZ`{<FHAE%`hGAFJg}Qrl}9*o&Sg2Qm#0iS`vS*u5V6rMnmplb|ES<U7IO%`i#d#{_U0tvpuaRh#?#H)
gikWqt2a}tzL8x*x{Tvd7YTCpgvd|9f5!}QUq%?uiQB4@>XBqt)<Pv~kM5aSdG552S*yA2jnm3i)-14a-KHxFR2U&WX9V+wQog#ftwO|K!yx#*w74$(WR(b`
26V$0t7xK?wJP2h*-<i!ScRn&2Tnn$uo@K2abA#$Gh$h}MB0J<SZT2k#^lzTux5f44n7wvel}3bNCk%k$J>u(tInCYzL{MM<905}j3fyInctQiC5mC`<GLFI
COC1Hy7J1WwQ##qK6l<gf$f+WM{Vqsa>aY?Od*u<H)+J#>9Vt9z<Lq_`~S|RZkNJ!VS!?Yu1qzzaS>X$tNC8((&nJIo@+eGL~Y&$hMNM!9{LzcSWXsq73Q@1
9yu{{6({9s9lsO6H7lo5Wh7O#jEQZJ$FU^p|7@^OoM(HU$;ZvluHRTODkZ$rw>%hiu_s6sguXFyCBp0s+bk*GfF2fvg1mx*3-Ca+VMi$YRQ&fI+fT)CnLqi)
lvs@R8H`SB$D)>AUQLH;6EYnkzxD>XNVM@eZ|Ht!za9lms~b9oEjR5YzaMKOb+qo!BAcWp?=K4l>44Aj9CrB1Bgcx*l>I&i+`EMGA-eq_cZVJ*PN(W1^pYkO
Z;H*2mFihNPQlP&(xg+QANVrF%_Mi97p+DFX>nGA_L8meR@6L)Z1mCYq^xS~p?B$=7)*!$_1@|9Tb8%2+B&G~@V`jd1h5I9UvXQ*b(5O4BH4{?Bz<_tBo>!7
e-WZB%G$m)nE;E2(L+suQ(P2D1Fc_azs-1U8h#NofOV@RG#yV8wp_CCFwCW0w3@)L(9RQpTfS+aL@FYvKANn-2-F3E5Xzg3V1u@+p|C+JU$&)M-zz02Xk5~v
dP{gapR(c8(t671>yv_HfIT=pQN<^F_VqPB5aAR$j!mx9|6*wBA#TS`e(K&@a1yLKy}(iD%6!MDuVMzm^M%f6wT9WyAI0`uJm85uZbJ5FB}RzEMvYGwB>LRE
HJxZd(Hn4lmJpHfsJa1sY?{zIwO5@vgwP!Z`)T>oK2xnaBb5sQnX*}tTa)Br(acsS&uxi>tYd`$V{wm4Xwdck|9Dn-a^U07L>x#O12GW0|LvJ?-Cf%!#sms#
=Iwl#pw|vwUzGn2Bp;<UEh_FNtd2XB67_h$uO#D+#qDFZPa$P4Wy>_n!9s+Hcv4W)i0Au49V$-z)WuxVVh@z#Kl#Q>AMtJ$FvECkt#N^==35vk`?mP5??WJR
ib*Q=N6@bU)hZ_aPOL$j?xSXmMNQNP{;51w&{%1ce4i1fnHxR&BiyhHPV*p<B*OVK5>_}C4%-e?E0qo|n)t^4;j$H@4e4|3By3hubTnuZjWGC%s}#y|(w}+l
w;Wna!>Io6`a9@32TxRIO?5|tv!+g+9fWoiJgV^_(*p6^XfeTTM`+!FsuqHz9GdLM^8B5#)z;X6UdhUQ**Y3}Ha*C1pUV{vxT6)FIEDoH`8uzs5fJIKn2@4y
EffCLMxn|p;18z0ngNWSXDlhQs;Sa>!1xYP5ycU~_b*4Bprvz#3)jFlGk1wxzdX-Vn_tWNpW6B>t?|H#Ut-3<#qYtYjrS~z{nats$Smq&gV07*{H}Ubt>r?^
A<?&Wau+7cITzj;65-tB0ZDOR3ppN*`}U9a*W{6NrP4YvxG$j-8mp5C6<$gcbEeRorLX&MDKkXup(Bm#E5DLO%)Ar2dn7?)*aY%>XWIK?L5>iA!E1U#Cvyk_
DTakVEA+zBtDkM3taoYGXfxEm3G)*TMW=yipWsCc5almqEx8N2#R$)0B}h7ONx6Q&F`;8mB%3kSdEro$Ltl44n13%CP2|x(d6wfG{sQ06NB;w?w#EV>B>3U%
Kd=64-I<PL@bHH^+y~DXq#Y0Eg5*Nxap#b0S2yfF$&mW;+f>7*Rw5?AwcEzDjVnqa6p4+gWD%^zKqth?m&kLW4-c?8Enz5NNQa}Lf~3*l!aiv8eU6O-g}9!t
D2BTf>e11l1nr^Mm{>*0`k}=X`dGn+s!tg%jI!fx<C}Q@);EcD5i^%@bVn`enhgQ$MwZF0Af?Dn&iI*?F+(h!gfVepK%F~A58t4uoc<)EIkfk^i?fz~f{?e2
Au?VFrsiJECz=%fvDYeTAuuq6FqE`H4>8N+aFB3l^W!m`Q%Xv?zMfi#S*T!KAQDU7O`~qS;8z$<uDk>lajgr^pt}d}<gZ0sJJb(mUimebZ%H0MQrHS$4HJDi
Q}H2n%oJN89#O1UuY0I<UZp6{-gt)kW?lzIIN2D;z5W0XnY?vbc>Mp~t;~tRfu&7d#kn;_DJv!<U#LgqB4o!V%qk2nL4U&;qY)~;#==zdEK!F5<sE5&dd{#|
<LF8tK1BLb`t|5uOHes45zUC=pz%uv_*O4W;-vx=SSYuL$MtR!^xq2Hy?925fTYy=@#cjcTzAeMwlNx(-|Vw{XSj?KPiq-r>=?GMpoT5<JLg4vX{(7+oFW=e
pN05vY}zKzVUN@&qCE_7c?a-V`uc2F_bvBtX+$o1-u68*(bHiP$dit6jdK6N0=&JwTOwFgH~a`4YZwkGxEV*)jF8$r<+oQR2?yiGnT!IFuRPbe$Nhz>8$0E<
X6rYK`afFSo~G)yY>%%AG7>qZts6d<a!m4hWm|00!o%7hMrQ3p^;uGJ_{J$Ll=bsn`Z9X~O9j=n{#v3J!eN#pRPE@g;N%)5aOT5A4a{J$2?pP)^&c9^wk#t0
RWspn+F#$Q>&XO=0x6c^;--sLK7ZzOc#`#W%bQu>+OR2cb$g-ijQNVNMDHoIw)u*xjP*Q?!4(!kYk69T?r`-2U|3%!As}NS2d8t%xl$w~K5JaAHyk3Nv`xW{
%o{jTO$)w@Xv8nu=8XsE`3pxi$ZVzZf#QzNVDqy1R{ZgRjJ_RYKwO3lfSANgUI9nObq`zzRZP6DOIcB?OanZY!huP8kr3QlG4Sg)wBV8j4;|$Owes-bz|7@m
5IG4GR}c6D@$T8V4wOMkP9L56D${bFXz7BaqHAL}r&86#RpQqP>t<+$iV<}{HF}t-*GxE<&^He+U)xGEeS5fxPs}>VqeIgQG54{nk5e8W=kRDhQ9FBuLNW%J
XpJ-Sk#`V9p?`ojRLBF*L)IkSpaxZYm9Z5+Y<x3?jpXJ?s@;UZNBdh_xFT*dkem)4&_gP3Ya=k2>3JZb=`{x-#lhyAqjQfxuT3)x>6yhoj5jw%)ugyHbubv&
>Q*9U+%3^@8TPl2G*BBI0VhN&j_^#n23$}Tj``E`7v^Y`{TR{h#L4XlnD1_@(BHqTiM2Vd_XwL{q@k5ORyKED@|jMsi0mDMz+#;sc{pRXtb;N~(vHoWT{(#m
Okf`JrsQSoa-HHk=Ge5xW?)N&0>c1k8-1viFvA_9`mP;WIGkLcv(2qz?dWFYI+W11lZ1%@(Eg=!_EK?w030ol8=VPPQHfiM@>Kf3H+9&@7DnP5lAg~=j*)3q
M!i`bms|PveV^xZw!XUTS$Pr(!}*G0aVzi;-CGlY3}2mwo>Ddj6*REeOwR0TtveinfvHf)Yz@p#8xEY6lGn4>o}g9!P(i*Y$8Er1BItS?%G;v%##7pWHY1#_
$fJ3tndw>A2eKHV8xrtY{q(2$fcyO|9cqfU$Ih{0vf>ICpzLHFzb^u~@7d(^2Yh?Gf&{J5Y4M;q`Ft>A50kh<Gr1KG{g6igNz%}cJ!ISxYOz?vQ24PU{pw%0
U1mPH(gMuG&5j=7u7l-q5P0H@ya2r`Xpf3vtW@i&vu~cvbszVtt9`AXaJPnG2CE=*PDvju{?AOx`m!`BBtxPaT&zr!*8ov2Ve%7%OL>|E{rUGBL5gdE(n7b#
n0=o2PQ?mcA>sVH%8fSdfLzv8%FF?^l9(6AxHSU}Vb?_5aFV@0%wGgAejx}=CjeWdcN=Wj*h8-$@>kn@A2b;TvGh;C0s_EptZA1f_y&-N(yZ2pex5;UcAREi
A>kOeE5zE8F+{0=YFQXU*(dNYz@v;=agwl2({%-W)Xji3^{6cv;bP>@+kAbfK=^OgL#vCWyR~TMA)tn(JBQ1slz)B<{h_aY_Hi}VF-_F=MLZ3$=o)BTv}Qgf
YIk&^^*P(y%i~-?VFQ?)ek57M{+{Q(7g7SI1o8{t6tcv96;~EwuP$#q-%Eai0hnux*E|wXnP)^uNUWVDBy`%?J!)#K9g7dkImE2Vc2c)u3$s2WJX)whcYq%C
P{Ae>GI;pIP7xT3_A;`hxJTONFtzc|*xN{4=dv~AdNu1xgmP;O>T2`~JF(NJ|J}IRe|<Wu_C-DsSUDvZ;OCfry)XEhRmw-=g)}1l3fAZ49DL%@Gg-d?sNv;R
S5Caal}Yl27zs#qKl2<HtD>AMstOGnQ)=mve*eXLsE>bW|CuOMbIJXp^l&RwZ5P{zLL>gZF(iD-SBvWhm|iL1+$CkBR&O9@CnZO{Izq%i^q3<ej5P6F;2QJ0
wFa?U8C(HqU?D}QVa7o(`T%k?*Tf6St8F)JOBtP>QVrkTUnB&`&(qt$jBz6vyBW?&{Lf@^<0GUSu{Hv5(eY8Bw}F{A_K={g(L-_Y(^D`h7tZ1|+g4<Bh7|dN
b<kOXfhlB+nDYZ%)3znkL3Nqx2*LL8S>6LQ2fC!#?_r9)`iS)uj}SCVhs*HE3Q@B&q*1YgE>jHYufe3>L9R4?#9fKXcexLknEH4`GIpoN>QGl6-e$(JSk$){
eZ%!f1VtS*8hK$LdqNxi(tyvpeSw5;F9R(g{B708JAyM_5S6{+w7RZOKR=!##g<Xc`yh}CfcP%O=3|Kx*tkCk^!H09(h7#qEqG$Mu$5f3e13RZDRL(>m#4(x
w<1aSG}OO$R>0%RRCF9i^XkGBv!d`PS~j0IgNb<e7|~xYAejAMv3$VD=u_bOR_GUU)C{-3f+{aQw@j9@FtZb{#prAivQDBpdgYL3R`vy8Xy=_RHv3i_u$mAX
ON-#X9@UlNJ~s+MTkm`8JhxQSk@R6T+&!Wzw`}g7t`~SCa|+LVh6ZJ~&H6Tg4%EmZC=2^6|9+PUFeY@UTw>Ci+F3A?r#V^iaWG@yGxxx2L_To@tF7XV1)fa2
I74~0k9fC=U2VCwv-nDk*zbahiYXn>_k%~!`$mhM`!d^%*W~if#v7R~&+p&OX7WUxp;)}uHIBLvI+T*Y{Zky<d4|EUgJsPLMMw<)M;=z1hDq<f)BMMyDSz%Q
2x@8=w4(nlnR0FXYGjKaFN-av_Bm@pIy5Me27MSm1aM9k0<AJ1<mskpH&$i}*IdqbtSO6=R-;aii*~yh>iHJIgiZnPu~|`-hWUN!$7xH0gsrA0EH81nrC;W;
FJ6)gce0Rg-jZwUfX@Jb)e>*Scib>wVLQu@$^aa2$~`A<P^JiV1HplHAoFjCF;6Z1Z|=R`a>!uL;B<<@KOn2?@NqwkYB+b^$=NzQ*PDA3WDU|WCzteGhjnU5
!z5{dh)6J5T8c1YISy;jm{82^X+OcM3Z-dVaUgVBSIKFvdOJQsxInt+8Dwr#u-u3P1s49_zpb`t9CT3B1!w6$lK+Ah;A$e^?-BWTQ?U&(1Ur|0;zWxirLgTx
HRDLPmfeXO6BW(k!_5)*tz6O5ml+VvykE(SfD4n^Ax^Lk;tX{QC7z2`p-hS+X6sVRHTq29T2iCcc<AAPHr`wO(v8bYxV^?IA}oYIYt^>%FJ3Ulrdd8}Z09z-
1A~Oiv&`4cU&tJD+bUf-T8JAKzg}1+*7rWTpD|4z+vZU8N1~>uub}r4pSRL-NfCsPF4o};{$P%0+ZAz(UTZ_=+Uf>o=V`u(|5!9?sKU0b8`gG<6j0YS5C-m*
5@zT1E5;7L)*99O-!n32qZ26rs)}aSdSB>6;1>Lb=s`Emx-7|bt(ha$gXj87#u#MGs|Zfk?Gq77yIocr>h=hqSOZnBz|@au;Xi~ofi5L5rCl!b15c`1WQ}kw
(*4zyD%1OoFhc6xX)aFw636~o>J6Qwo3aWSaGcxXX%20;ITN`%DVbTP+-O*AAS@~(WXwT{G@k|=BlcG`X>;-rXUg{5)Ha=^9w-G=PI&2lgB^i~E(zV|Iwsed
l=S=o6jp1;ePyv3_oSu0X!fE~#`n_GhlJ9+p=6&oWTC`w{z;RL*gEhzF>?|iY1wVNgwj_RxGH8_iL&L?8mYRUw;iFW%;iZ6_hh?LQk^|76iw{rHc-XDULl*U
ep5O=1X7X~cY@y9Osj@HZsD-%&oA9f^oG#_g#09)_qz$zi&$@+IWpJszDCT9TSHyc{h!84NZuV9ynsO+sz>okssnV$hX)L3N-q2V{*~^>`1=k2>Bg}OQ|f5t
H>;esp^2$-N$oDE)(H@TP+k)!2p^#bgHwp%{xILJ41H?r8Xv3ut9j}*)ezco>F#|7$16PTbqr0!!&Q2;s(g>`R4K~aah`1*9>hXj{}p%iYYv-gIjAoTQJ*nO
GOpt940AIl{O*r{4dNOKVB5dFG2c<`R@l8o4!y2!itOxaG$Visv)@37%qDd6>6!Vv!$T5l5AyGB47)<%se}mDlF>ghfOcSn;gezu`S}6Vir@Ih{Q65WJn<v-
-u~lgHwZ_wV0^w7L`1%&epTSZP{0u&x}tSLN!$nhWzSZU#o$k+s@xazQ<&2-ONn4}`corBfCD+ePOs4*wi`f=3MG1kbUJF>Yr%F2ws!1r)5-61DQL%}@eCg5
KOU~;x`kzJLr>-%CX8FC!dkc#^W_VYQyUzW%$YD&IZKR$1~8ZW1$2G*n2Unh+gI`VH5jO-7Q@0^ySHp|>g%)zzvB0%elwD{r(&M|@npv=a#DYw=9i?|G!ucq
GCe#@1+EQL!59QgG~4b@PFnr*WXP2VMzGo^mms9-qI?&P?GkxLCUWQ#776f+(xlCfe-RhTNJoznT^;_|M<p~=M_oSdTPLbbvtJ`JOwbxFe2}=~EDhNN)9ahQ
OzmOW;_vH%0(XTG$@LZNEaO$$@^yXLl%onK?kDXlh(5=Na>#J){O+@>@hTT$Wz0-CkJ`J`@?v^XkLx1ZHo{20b6vP;M6y8*X8u?I|8|XgQTC{7>;E#^KWZ>l
lktwxUQt<)F`l70!~77PKe6)xBE?O!C%BM9bt8cy?fYTnoLM<E{m%kmMIr(4w|*fpG83LE!D3kJRZ~8=(KKNx_cO?XcfUpO7~f#h_D-QmzZKpLwwgTZ9`R*<
=#m>N-%d`we*@kq10Kiq#YL?9O1m4_5#K2PjNER1vQO0Y>N@7vxvyjzlMq&aQo;QvDv24%smY~Z{T{tGrw3vHkFva1`6jo=+X>TxOTl&^Q0+Ebj%$&tiT5Uv
8)&7*gko&$2&`)b@GoxJI0noyQV^Py@=!^2F31@#X~6*mH`MoPzg4WbPi<JBI`NB{*a3)v<=*}AAmXxr__aYcV@k}cPYCx$Er)4MJAAH|vh9UsmjO}d=R4m(
?lUuVgY^HzgTD)mYP>+&S9C0lH5+iQ+gKQtExX{2T2t4YwF-n5D-DDD5}1s$8Cv=2<$R@=41WCx1yM7&2ncp&7?*R)pz0~V2U}SIoC(fczkM#N%K7r2`rqbw
neJWP?8v*+2ZIMS%;Ug@-r{|V`k1@;18c9co$nA+Sq<w>$c%qn`d9j<F-BjEcvWb7B2_Bp1yAcCNvl(NO-AaoT*ug;Pb}cb=I(TJ>~Uqcyd2;>I0sWhRe=Iu
6pKJ_!7HrxZAeTN88pP}=A^P6{z&fnN6%QQk&;M&Wl_-V!>HLmj`-mbJTzQl_Q43QnvAul^3X8Gf`7ak;m+`<a_(rI#TiV1fLph=R<Xo<U0@gWR0!0UuXyp*
^Yp4doi;=8@&tmoT2SWL)K4EnBXJyTA$@NiklI7Ax@#$UG+e9*&jW;Zq<I{#J*a^QI+L7oU%WW;q7}qxpnLYON}NiZB{^`lTnCo*{OG2edX--9*vpw`499lr
o@g%<owhr+oO(_qDX<r{w)NsYL_s8vy#NZV9FJ5uIaT-pbT2K&lI_HGfS;zQOROE!Ewn14>?)cz^gnd0Nb8rG)07L_7t|UQ=X?%8H%@|im6P?hKMM?lj-B{4
m9B=cM=<z~IzEQL)e!sFV>v-SI##UpdXWok3MvR3B*WC@XPy0?Jn#Q|lwxrCzR<K{Bc)aJ0}fU7Y)^=Du*3MuQIUn$&Jq?}B6zURPF8EL@-F`xwk?6pqVP8T
FlDq<ot1v|0Xm2bcP;2t?4`e;V7KqYNVj?WHH(>n73JB203uVFgs96Y)<K%kwm$Q*5Qb=mvohC7xLtdH)bd{$UiB$`qTK-8{ED*Cnmo6<&RE-3KiCE#bH&TX
e){{nw#=B&x(}5;J6+v{VR!kqF&(QT1XY-b1VjqX7s?l?s{T)8;0Zs0w94rt_Vn5f9>*g+lp@gFa9xU+ss-ws2A@voMI7f;xgpy^Z3m-yW<6&?PQiPW1-DMA
(P=e^E(RSp_BQK@aurExN000UESiuE@0@GvGGRi5;Qe)7IDzyU$!KexK8IkQ?Lw-<Hf{}U#N({ZcaukNHPn7xaQv`P%FR&FB7VV_m=VW`q!*WPsa43kFCWg_
W(C|Sr8c{aEW?*0v5wFN?m{IvRn++Vi_1@f@pA_=&n}EV!g@%ewW$U|W2N6)oE8bwsHI~<I~!^Ad)+wy^HYs6Wz}T=7c^>ViU_uGuSZWoTm>X%Y?k(DC?-%d
bRFH!Wo}Z-$iE%B6=Qqa`Qh@%HuMy53w5?7=imhP+Op~}F3{N$>`TNYP<2hOfV-^lIxFr?f%aRR0XTBM(*ibhzfnp|tgWC-Q#dBI;_w3cMV>7Eyt->aR{MZY
#LKj4cDif<l+1Rhtl?HE3ba$^LlmWqC#MMcwT2_r+`v)+;t(W=Tcz3Gx?qpDWu(X5gI7*M7-3!#=I5PlUI!HpKD1Z`&iwQtMfp$`azRId{fyG(`*PT})xDD6
+6v&B1?qUJT!SyMb(Yge;rISFXC?B~NMn|I(LB-viKa~>8yTfUXTB=l4Pq+`R9TJqA2us$UUUrxv7u1{MMH7wm&{e?eQ5Mk%q&+N_6K(-PC)PliG%FoQ8OVf
8LEp;DybyEH4yQQ!|+DOLyEq6s0P?i?O@sKxI|>GOMwE+U?abda%#xz7Q?aUrt7Mem>BWO!tw>7(Ebq^>s-_geS~O5IaI8bnWOvhs{ob1AiER~u)I-LBuTdM
%D7!|Q(G%lkmv6f+b%azTVAjB;d6<?zrw+TqUHTtr|>(l2VJJ`evj$NX_*zxV2_)|f8aC8uOcx?Oy_^YKq@v@{1c-mN!2K}1w}xWvAGP*_1fe7CJQk@q;Fff
n}Q}4kE+I7O<vYztZ_~ei|ZRu+ibU$i-;*!N@hW|kKU&$+1La_&E4Kjh)OEpg>;0<IC;^gMHY#Am8!0muRi(WpQH@XYdc&1wv?v0n+>hsSpBJ4X=2yq$VZsw
A8bJMAe0dpTujbA0=sbM-6lvmlW9#(AF6JMkW!6k2OXhobJLG={drMSz{V^k_5VK;8fI`-Q2gfvvu(7$qK498^m02x>VZNL72ft5=&0HKPu%f{m-ehBa#ZV7
zM!wN{S{eGue_w}UdoPDt;|S+##EE6Oev5O=o&v(B1uZxS7)(!58P_);lDr<vQqEy{E4zYL_K`eYNipli`3oX-%vClwRK1Gw**QK?QdpcJl?VV9ArbE3H>Zr
Lb88x-7zUR+s>|%@|5Lp%J506wbY$(3*qu@xe{+!#8!v81jnmC+i6nDBt&wQ7_VgkwG<r3`Yf=y1+jVr3#kl)2~_lWdR+yCquOhAaQ_|bM#3X`$Z+-u0|l+g
7Pt*lZ<oSuG*5O0+jywEp~^>(QN^K`!?0X<A7uCa^1Z9}X@FJdWN?GN38i3Kg`yFN?eSp^O6QSb0yMEmGH%i2jh;%@u_X*KsYt`>>GUH#l8;*I%DxFu{ah<e
Hynd4zS=^|dIc6jzYzl=OV#*J;(5Nm?M0h4z!;;Q9vRWK8JdL9|07u$G<#T*Hra;%`^uX^AS2&7IY$@dhlW>+l#CHkBLVV}*}Cx`SXJy=O@yR$20&&e8o)0g
4}Vzq?J^@hd$P#e0ZUu2cv@CI0omJdgkoxWi`r2)$FPe5BYyhtvq1Q|oChSL*8f7(h%kJI_cocW-FYre*08SX&8f~7Ipo?Y>%n((%1zGTKYi2~<zOa|fC$V_
(q;J0Cs2rjOiuCPX$owMo4Z@$FEZ&lw;Ri*_`J>1QeNi0Hbp0j(eUUr#O}jiEh<?GtO&~Gwy<))Gh=4u$I}+0vDrS_7j>vLHc~Q$!&o=sn2^sENKq1IEy9|)
&}GA7wC`ztTI&sQv+ZMm9M#HB*@2g!%!y&y6S$$CRd6{C)`|t$b>M@il3P~vv^e?gRBVb@e|*9o)qgd=JvGY_UJlqpypM@_%zig?jao#yh=nxlSdgto#UGDZ
Dz#n2A2a4o1rZo{Lje(RQ*4$9aML_>;M9)D{=Sa;#LB9oo08;bJZrpaLVmh^0aXithzy7HrJ>Jwbj29LG2${#PzTO;aG4&jXHu2gvF~%mZb{Ch06v$eY8cJ{
_xnaV(OJEZEb=7Ne~zwIiU^>IFdJV&CHPJfbFXDuA&0&6afi++a5@<Y9utFT-)tov0g;6MChP4%sCI|3uCpU(t&Y?}G7>Jp%{c9H`&Y7$+jG73@KV}Mp}0%I
(#^5!Rw$O=oHq6?l4dE8&}$i~1^J|m>%P`ncbV|9Knc3**K#Q_m>iQF?>LhhqAKA-9*4nJU0<79&OE_(_F3ZP=LVz=tL#a*GzMWj$kBTyydR)11;#BB+8foH
qFH9Kf)MFM!G=i;eD-o?wjQ3#vuJEcUpyqvjCEe&G9Yy0gNuxh(tDRyh2`m>5mZDViNqbuYWHnLC^2g<nW!M+UWeWO7GF!%#brKV63_>s{=(S2b0~$+mWBWL
CfSzPGZ;w^Mw%E@&KCSW@9Th*uyu7^?>u@<;fc}=?)1Sx))$v8rnR}#^)rYKigspy|JMT?VN-^@SZ1tGDm7+GtDc$|XZW8eX@?TM&z=RPW-(*v&yG@!agO2_
kVPq@#b9?NC1oI=2e`O<P_zD$MGo?=hM1)#!?5h%L6*!;5F~My_hX%rlc!!y8OT!@Qr$uXbexR#E5YmPAUPQaPAoB4GS1KX+Dp4K#~p!+Y$-k1u5dR9-8bfA
dR@jnTZOqJ&1mA!q{n6B`-cYghH8t_aTLFvOE&GPTaUmerYIyK65-j{v=u+srqoe4HV|Jbt(h!!S0BMup?HpRqImMS^v@id2A2LjhWQtC0aa-K+y6MjJX%%i
pVc}cke%xWCSEyKmBq!m4#^_HF2yfc#Xaot>KLxd^yI*7ZL}t)wP#g{AI11G&mb}$(n|c^`KYKOM{B5(DLzp2k=6C?B+zJl<!&9iKDT}r`$d0i3Z$&$$y4ia
A{$I4IFVrbJ<ot#_HB8&LY_2n(`wTkxM-cfngM{uVdvDfDoauhF`U8Y1?KRchQm3xCsXDT@6_a*r~0BGV5P^U@8C2B?+v%tKL9lmlss<9s*n2{C)+l#gj~*M
7mE@Hgv|F-P4{pt6BVU~^;xSJ;!RM)G-(aVbi#oMTv8!Z1>gcsDuqK>9i{}HN!eL0GzWA<$16gIBBQoB#TcDTouGmHZS}b?uck&4nPUqrN53kRp^s3bWUZRH
9eDqVT8XLy5L;%Jgv!q+b1j;+V8Jbc)Pn9XXR8D6|1)80$s}^TJ(wSnk~y;S)26nl1fO~Q`dKW{XV^~WPV=1S0S;7zY$68?vPuz5pR};@i#l|UQP~3-QotaW
wet$~ZZ+1EoMzbIx}%tUTQxD=5tNHZP<b9cnDC(xq^~%*JHscC5a;kFVf!8R87#rn1Bs{rMX^*r95g1p1DUP3^Y^q9ikKy!+s5Z|BGIrGYV?Uu#hx2u#Xow9
f>U{L1Gb;Z;{J=LwaE8ns%&=U$5b>30BRK$oP0KbYv{Mcpu}&yP^|zHs)_XzGRI5Ky3LP8rXG_@a6=b9&5Y-=N=D$vr=+qfNx~hlmR%L<(fek}eT1ODVclE?
M)lf~QjCR(3;ha{h)6AM`QL|3T1;*JW8N-tXV{2xwnJzav<abDcN#&Hd2d5yslYW`BS$O4v|7OH`IuC&t(DPnsZ4)c^+>GLIuF|9yXX3^LPP1U`?7!fIv4ah
*P>}LnJ^gwD=vj-d%uu(a6)Sgw=gS&Gjr|puLw&qTO|ih&F(m7X>y|g8JZa+Ty!famV4O5S0Kx~t!tr}g6GwC+O{x+7(udq5@InFEXdoUWTO|`%wwwxuygKJ
hf7u58VOy@+$mktw>0w35X^q$GjRxAtLEUTLoe0l!_N*7vn}Vs)hKOD#86~p|9hMk{-@dqUbgEN^Cw1AOM}41$k`bY&trM?mu5j!p3~|MEvDhMbT}q|!u=lx
hM&zc!gAA)aT7dAu+C^j<9cg;i1sRBYIx60-j(Tu&1D~z{YLVS^jvV~`g{GufR*PkZcy6CtX<Oa3n$)`Z#^J*cw}-fuyr)+=Xo)ED*MCPyd3mZ<;hK<aGiy8
9H52x0}S933QJwOer6CcMAWqkD#TC?*mAUISz}E<ly`hhSjmdTAxg+VD%V}>5fX1(pr3R`N~Z>uD<2?8rmhIBq%C2G@z%KKQrt?zy2D+6g`{&XP+uNrFpka#
g4Ur5C^a?pTgup-EA%@rYpT&G@=P{%SdC#xuP3lRiJ}U(!3-1xjZQWUPL6O(Bl919lYvy7b4{#P;G?Uw4wEZa4|1q`r-ie2KQ{|s)6;XxOb$UQS!UC_K2@;d
A&#;(%i}58Nh;0GKqK8!zlF-wXeFx+t?BwxiEU%zp0DKNs7e_Fq|o8W@|H0c7(qF?r@qG_MFBW6Ep{o$mz5``Fyuo$xMF4s?TLT3_{}CEhkrAs&j4Jfv{=zW
Tyzq7%2kj{u~En&YRGaHBsxoWvgg$xiSLq2g8Vy|&1uPX6iox-K;hHkysiCs(j{ZY02J_6*neOogVIdQV=wp0!i$Kw2eLW`X&XsnE%$Ml!e=~aRH=6@#yp1=
5XxIDLq?Z!I~}sC_QnunYd<x+CvX7N3lMQnI{UAW(5(S8`L9rtVXRYz+V8n)kXkDhV<if+-kOb@u+#~mbXGWef`fBUPM&)i6@Jd)h{Z^qtvO6@d@r4`zhlr7
HSe6-$?_zcm~pWr>p-DS?EvzVBQ=-bWU353FK#JIKy?l45-D(Vv{g2Wg!otiO!##m>eVjdtry@T#SJ>cP=G{=>T!GlD0De<Pg;!x!Ff#!s_6IjIF?0|P97dZ
;n6M6q$eoWwhFSVkEBv(`$Trpm@&zU^1?);z2P1bmQhm?oH#Gr10p{W<?GR`u0BO`f29%@F=mSY%|y+e?K@Fa6%oe{QzXS~L|8yXK^kX+q_Pe!zd4hL5dYJ*
gNcNnqL}ska~33`?a=NbtwaRt0!mXG{}7#{i*l6X40u749(xj|R`dq(s;T>oO+bntX#L!-+!~2vG|~jH{^Q1C+=g86Q(op=jY4g_L-qA+eUa4?^fu4ReeT9K
4VVW9LXSJrGhz$uh1DvhH4SA>HYtT;Ab{^JJ=Mbni8RQeiy)BaarmDj?4Z|$-cbP}?BO6?MbBAsRiGu$07?`yag?8%&**lSyoBso(7cTOe@aDgmyqBY>ubeH
q}U|_sqY<TBS6sQ5{VT-Y-$iUG|$_sKQZ<}mNrY=U}V<l<=Tz#oJ_N{wVY5BjkWBR;}LbmE)~X5-J@eW(q)&1iCj4g@guC+Xq0ZG?%h&OM9N*o@qAhjq-|%b
LM(2{Z(5Y{5Rv(+d#~6GCxz*_3VLy`wCbK!)HrbuC_es;DLasZKZoX2>Dv%W9T+fhJ{V{OuiuYDIYd^dKQO;UHBmnv&7P{VIc(DIS4zl}p;!;0V)1b(iPZa)
g5R*L>~GcyBx5L#m^M|4QJyOV708G5G{LSfw)$r}>916OU9;RJ(p!U$U>%j+rz9`)ordS)_pC3u6;kcENGA@J)epJDaE&)}>@b~MFuKEjEUjtWo!SmdqBEO9
q#P1z8S4j*hGdJ!r7G46Vm9e(y7VDbF{ux!5lQ|rpjhxG@<CWu0Aj$XrZ%OwI!d)Xmc>(ls<_iUzte08=?3|SG?hVtYexhz`WL52U%D5%@^=kcq*MNX`DxU(
OH)@}?^3qVx-s&nNOONE+-dbf84y<e<b3L(<1E@zg2<=8NC;)XYHR-wlCUamsC}{-*xR7oF!QaAyvpmG-pT|fG1k6{@o&W?r}_oH6U?hZw7wBE3=V)dkk;J~
K=9M_W_V3h|Ji2P)mu>M8q!q1)4~PN5?FZH4OGJb!)b=FNKadOU8lh>MR;}1eTvn#xg;7GL0%QAU$>xf`VR0JG*{}r6#G`Bnz6r8d{~!KVcUx`i2yRKA~HiS
-VWs(D@*mwU66x6yW@DHY0!*x*nbYc>J9-~4VERJ9?KKOuGMsnQhKYIUK(m+;J(G<O4?G5%o!)lU5j00CH$acHyFIWjMpO<uQ&$f7M;UH(_%g!O_)M}f>}2e
877KdRqcijva44;T9iJosZyv%8+>&TU7%qz=MI!hl|VI~r#0NxPO;!{KJ?j(1K({$By$PaVm{x4D2SK37p#}xlj-i8YRRFv!ho(+e^mQ~vLETn3L(6peRUYD
5CTLXBikJ9iwKPdMxM)Q1Juqv1E9k6F6}ik25*4V0LeoYE{uTVe%w|BBh7FzS_9&(@K?wZW%O&r(qhkT!6z7rZZX6T5D1lLpf$n%@o^Q}lI9CIHm-y4_s`K2
^W(PWNRv6~4Vn3SpP^dB!2OE(EkML6>`vJaOS+Wm;Ehl}b{z@BNZmtIF3AZ$apE9xN?jdlK+a9`p%rSU&oNL{_9DdD%;#%(tk=7<;$s|zBT0rg@ruBJ0p}Ey
>gVjhpKIpg<2qdKTDgzT5=8;+tdOYf!{m<<I$E{5Agbu>vQE2x-2NnKy?ugl95&KLphe5LpXEZ9BpcJ3*9xK1(6HflK{h($>B@JfTcWlGOtHZ}0iXv{NKAb0
b`nRcv6t5Ef4t{sx?|vEo7^X+^bbtY`TOnboe`GBasGbu0&|ZNowU!|L4UewkidUlAc1j{<Pi|LDIU^AxwdrWaS4+3>Wt}{PEskAs^)Ur-6gZAjq)^u*^&du
5e9N$th0F61WNp&$4QV9*l1Sa#b~hHt*g*dYx_lfR)<jU#gU+gvrlGmTzbVTY0g094VCHi1!%IX4amH;66cjKeuy=<`caez-lh9kuz}to4^|0NhDE;SL@gv+
m3|ApBBC3Um!OM!=ljI|cVJF?(%koPLW7I0xIC~FV-8wmp-|c!k>#~tWj=6aLtKIkKCC3u7owwm93qnMuA{+fx4Ks3@t1}{5;Ms>u;y|*f{50|M8F-8!x<KG
Fx-W^%u?h0&COt_eiFV7Fmxc}r4=h7-g0Rxf^+U+<k%^JI>w?cTQ}FGum|fV6{Yqe#P43sS2|t>2&eUYO6B54%j8`DE<pZ~n6)x~;h==&oD(!-pVN9y$Js>j
E=2k-O`T{6K8CM)&L(m1HIUy@KhduR9p3TZK+*xxN35h8;MF!$(P3MZ(70A>*OxV3aAiSyRUd;$;z04Fy@t~E&1lq!j;Tx8ZfZ7+Y&RR#Tlq@4?aF>ic|_I#
+iRqQZ6fAW+5t-C&fe|oD0yaR=DACeG08ks_A;;{$_h_Zay%BA79~ZS#KY@@+)<dQGlzYLT7RaXwQ;;cUmZX3dSe3*dG!!%)cRVWsdOv>$x2+)y<~v_l{f{J
HPy^Ol$IlcEASP5hD4iCeX<3mE(t4WrzOCL;%5-}9n0(4DNf=&zXq5FXZNxo#X2Ncw1nrr%&t`Lwz!f+XG$Txv!Sv7#X-2ZNb@M<Zqqr=F1ZTvUI~y*o@O)Z
a}X>ThtxKemlB6-MJRc6AF%&xurER>ZCOLnJ^}H6e8yMFVr7%YtTNOFFr#{Dn1iBAWSIwJ*E9Izm*gt{wW(LR1qYQ_pSp8+CgBtlVG@imwXR9-<SaO{(^09j
G)`BkHNF1r8jD;E+H`s-P{9HtgZc+X>9d#h^3qiPz=hW2h$qV{SvM@}@;^V!h3=aQ1Z0?Tg_mu@6jvuM6Y`V?eJ4<2INyivEsSCyUwh_giLDzK5Xj)1KsB&;
5PZodAU1e~vi9&PA+f#`@<k3i(stt<a}&^WkDCsK!VYTRE*x^g>KVgHxZ++<KM16(mx64>Au?S0+E%<n^5-&zKNe||C<wYUtf>!Azxsa^Nc$WdLaUs<@X?66
DK=jTu&B-z!(XmbP8YkL6SU~95_#Ub6jHr1AVg?krNfB)ZZv|i6B}9^5`(OV)oW$RqbYw!GkAh)g9Vy}f26MSbjgg&WsLOL6yNt$9ZBv|HW|9kUER^4*epCk
P4IeO8fjM0r4FSL-?iuOF`yW4=lgZ4gc#HA^e#mrj~~yLWMsT*hSyl&b(FQ~MP4U&>sUvafpJ)h%|{&bO|g5g_j@?;;Z6K*=rVza)dupYGN$e1v?%iUmu4}e
Doz#)V=qs3J7d5Usq=|EBLf3>VJL;gxP(A_3)&K`vutAFG1M1fzl+V(g3CbbBSc7o_qt2%>TGF4$v$U?<a@XXdsq_SlWGU${q3uDcLA6-6<Ej7J-eOmqVA%9
!W8eD^BoWO=oXAl96Tc*(dN#S<T|3z$&`|qZB^Me!cN+#9GVs6@CThk?f>~c_CmAuGY+N5{eh~*`mC)E(wrkeDy4W;M}Kj#Is6y{vIj?N!7#RRbthTK5SGiB
ZU-^^sW>TLI<+kpp$+!!n${%tvQJ&Opvr0L9Br@`j3*G_A-dp=GPP^_x9sA8X}QQ`;9fo2VJ`{tW@_nnneaht8M>1-VcvY|pQm+x=YHngLx#;4#iWjn=R1<N
ihW$q!X%m)j2UDEMRFIsu>?(=`%KqL3D!VC!Fy<Pcb>~Zi_xaQ$h;8I%fZi==VKvIal5;o33?;4X8(#psqG(DQ!%vN)hPaJrT!<!E0@$R%s#{<dY)TT4@iB4
sePn0jIk)=_v#C3TgX_(wlHI#?d#*j-=s-<dGSdHqw!QS_T#(YH80%4V3KW#r=4;=C1qQd@DZdbPtLHI#>e|eh61@m_2ylN9aavnlFo@nupn@y9(A5~PpX%~
00u})lPe-;{LZ~aWM^#>6rHaQZX#SMi--~mqG}(fu@TzUsp3`kg}Fq8a6oD2UaeX07|(?9^N&P85R;$kgpg006_;owsF$=&tUk5IPi9Vvbroq*g3t*o_!76-
g_}^{LIj!b<xc;0dtuFX!uoS9QrIyX%8uEHnVUy0{R)pd7@=(OAY!B_)K7Ykt0g}_k@<&X+Wrl()2F3X-)T<MlU6r(;oaaW>4P#k`H!h*-hP?(Z>qq0v>WR=
yiHjPs~w*}^Xbfo$-St><v9EjSo1neQF_4v_^V2nGfZ;*)jq8dMqe`zK>;oCcJ>NZ%lQ>0mVbWqhEZYM0Gd!IaU(E9zGg=Y;)K7e(Vmv0RvdQqy2q^Ht5!O-
=lI?`t6hS??B4n`Wu3{~s8anKYjKEYT0{t)&9wNh8F<0dvr@jx*Nu}LGQJ7EvM@~Qh)bN-)-a>S?y$KPZ)K@@lfqND%24HYgF>7n@JPI-_L%EZc}+!{$Ch55
0gmWZ#-!&>r#}-559X90nKW89dsN6q(9$MGyk%|&YF06;$2p=X>gygkX(VnDLk~eDCy-Lj;u9X*(nfH?Fw1p}SutVQ%ar%J0>5Y2Y|{P7rnOtiU2Z#%e@EwD
=?$syar9V}UCSH@5%K~f<m5SbMM188;s4;V{MNk<XmV*|wHbklcY<xD#x^GRGVrSr06pE1leZ6;@+Wm*R%}+5DZ~ijwO~YGxelrnV8Q~8ERY?y5P)^(rI618
WJv4UH|s1t(VNMiM8t?kpeoWr+Ew6Pj{$Vf{FC`W?lr(@Gk<p|?%+P*Vh@S_qMImPZTn{0(Xc}JM7esJ5W+vujhv+aSkhuvkiw-}j3o!Tkmr9S;q)wqbL5Q>
o3rsFxrtqEys?7Y0s4iilycr#H81iFdH;lk3>+eV=ujYib^xlBGRx>ybiALKe!q?C013fw$T;?>*7_bI%Kth>v4&qP11MjB-raxB*ui)oV3hc*t~^R?zflcr
V0MrL<U?y3mVWsZ&Z1(IS$<{%%e#q~nTY=U?#w3nHB-noXEx9aHH3OkCGejJ!)*U-`0I6q#4L6xdi`VND$CmvKv!8Eel<13eiKPj9((AIas%SM!Oh7n4Np4d
_pdyR`kvu>Q{=C6AmaGXZ``{S7W)O73D8)*i1#$85c@jn$9bOZB33oWhuYQ(m&`)AC!I;69=;B;?@E#swzpa`=7+!_t~Uo<9@?92B=4+IAWBR!K>Cl60%Qd{
BY<q1`!-{!^6Ckf`oa7zKm_Chij&(6TdxUKD|Q5ZVnJ}QB`F+k|Hev?;QGnZo2fV8S#=pTZ{CsRzymmZJz~dw&<_KE77#YeD;7(dsrr!EpT|)BbK}hYXxq~m
RwFIN3@R%6i}P~<&a1(yV;H48u`=EB1GXuHrsE>grs$14uwRXL*O=Q7#$Bt!8*AnT@rRj^+($?a9m}sr29GCO#!frCLl35?mEp+7$8mC(m}*|g34X0s7F#fc
jb3LY%>TD?X8+<5zN>xGJhw}2#1ZPe_fd>RTX~WrhZTY9J%{NuT!XdafJSukwa`BmZ|rOEW1#=G8)bR6-}@-s`3Yztu%JtBMu6^;v{3tjvly5F4GXh?d7xiR
L~?``eoMsaqeHrhC>uTlnPNLmIC|fQuP~@T7HjkcKtd0?N@Wv-wTKT4<3FzeeZBxMxgKgB!M2La+LKsV-em2O$@V+ltA)%<V{XD$5iJF9V0QWyeh|b(8k<XV
KCCHmMTx0v7zkd>ES{VT?e`@2XDgt&66B_I4Dqn%QG#%V)jOT9R;e?0U~?hv{$srjH)q0oT(AGA;3m{`goO7Qr{AG^vO0c}6uUR&zt&!76+fSM-99;Yvi_<G
8OubQmH6~xS{dsA+kuWM44V9#$!J7xfe$TLd}psL&#tsTxd;KpEIQ@U^i~khGI^xi$)cn|0wx%edFJH1d{qS;BcM*O_kp#y1hHDbyW;l{)MtL@rBk;?9!j+N
AbIpQ8+J8OMz{K$w^ku_rCd8mjyFM?fYkC25_6O^Yf&9;-rLc<x-T3QNnsuYirx(k_&v|YF!g2KVHksl_2qX|6o=&-_Q;>rZ5P)*2gk?;Hg<gJ8BpM~*WlTV
8jW_Ne|N~&ZIm)&`8Im^6Tuw<d*(b<YUx4R9>S1pmeGYNB<G@qf6JXSjNAMJkLGgE-r(m$fNpv`O329YnPJh9xs)AEM*}@f-@Fpl{npO26$DpCaD(V;+}P>@
P>FA}oj1~tQJf>|GSUFk-TYs^MWM<!up7=`3k{x5Md}<O3kt#h=N+(=ky8&V%+tL(BR3;iU?_NO?R^3)X-<sB)O-6o$bR|NaY34aFjfgle-tU8hqYXW;^??x
wn>&+$%_#ok7aztMSz-`7cM_R(3`sE2TMc?*pMQmY=w|E=O%g4)KOn{*P02#msDnj15p;$5`2#uH@rj(f8_lBnQR0S3hGEqrJCgp^~?l1dYl)4E&gxq9vlM%
g+QABi_ktj{ZT<bp1WN_1pj&rNdV89Q?r3i;cApPLmP<Zx7}f!;cR9;X%o>581Y03wxRv2dy#et!XzL&>bSms9=WK0)W9#q1jGq~Vc!!k(O!(96{7@GU0~O<
he3lK&0zdMq@mdbno-AJz}J4Kq*{WT?xOluniJyj74vyOtU}UBlp^t-ab-pa-9!Y)wrK|P;`}BCAmkY5sA>>tSgUCr@Kyh*zpaG8<jdMwj=J;)84TX7y&(S2
BRHVoxV~tV`ZmxKwyPFOZ54B0Mjijt<JDD+s8Qv}&)5<uSI5cdhq9M)GjfZj59cMXSWZ`sB?w?K1&#)LQgV;mwm*}94!}~s;N|=q{25UFg3fmW%;?F3O+^Z5
{|jld4(qtEX<855Hf4rgUu}Z?*MdV_Omer~Q~L5`%mdo%u{T&Aet>^2YbtcYKY{fO(Y|x-U{fl^+>vc|X3*R{+sSbK)#Zau2^t2Rv+Prck5q^1;XM!&nao2d
-L{7#D)!<TZ==@7Mws=WlOm1&)VoM3*78bp(fVOZ07IHine_{!Y(i|RseXf4Y$+GsZ@WbD!&xgj7T8AU0!2NEsZ+8GTZ>Om_@i!UcD-BS@X;F6EA%eYZE2pE
GpQqv+GKIyR;)H#@1?;7X~ufo*b4x|D4sHC4VX94<=d`AmYQxJGpVbY(wDT&lez~%^bX=E><@irLZULwxB83C&02h<(#`gJq)4)`)0eqc$L|fcH(}c3<z+UW
2r8_>*UOPP6I>d#qEiICt|CyH9i2!7si|Bw31C+lq_2w!Hz}RPFrFx%N~b<?Rs}e|2D%%!)?DR$x;gr+IjCxezsn9?7le-Fo@BwTF!d#0>W<{OJSviIR7qYY
Dg9`NA3pT#*ppF!*OIOdoBdFq&k&%_J9jOeQh$YzEqk{Ah9AOFRu&ODbh%3t3k8if&#<%1nrdL*H5NYwNfEfg2Etf)=-&FuC=O3fm&zbTx1S7^{Iou>-g$m0
`8*>+)LrXN-1fD(ykxd+bepe1lf1H~_`8-fRCeuFyKEcn3bhwW=Q?26QCK57=H@NdKr~(W#9lT$!Le!#KmsUS6%qAOGD_?kD>hl_;+wW^(o#imc#>EEAJ6KY
*fsE4D5^XtRk^fep+km--bwaefEbywDsPYWliQd6aGxDwacrHI4ukIM4%xqn#K8f(DJtWzh)!zC^@4sdK~x%Sh2-XFZKy_3$Vs&mD><g4w;Uj|tL2OI1QVpD
VI$RZQ-}n#-+Cb~b@_6kc^3P7g?ri7rCAt`pdG`|PvWnp=C60%CH?uVYu&2L+nZ~ECX2rz(U~<Pk@!>_)$4jXh0uCtrrHdwnCL(d`Z0e(rWm?Tu{We=AM5^u
f}(EGJgA}P$oIgkG1}7fC-Ew3)-NFaVc9U~bbY^KD@f<xVoCnO38}V>>z{gyzyEVA>N&+@X@yrAm!P1Gke&74jV5vQz7pk29aZ?+T>!zV{4Q6yq%+OX*fj(j
!+ny5D9r8{oG^7CxdgG_bK#V6y<M)Z6kNDV$HWhRzcp@Cg-IVUfPy^=wMQWBMXD|qRH0+#=2qAsF4NOS5#9=lpw5P(9e>zrOTcK5Urd=y&|OZ~tz{4vajZgs
Vk~)!x3j?QQy>5T21dTo)^85vpJ_q*uT}BCAhQhz0+fV)k`gDnHEq1au=RFt@sN#Nlz8@wL*Ka-C}$5(Z;tkwU}=MVz1PquL*Dk?IY*1$zH_jxyHX_+<^b8R
=2|?ai`~(Ua?xIKfM<})+Zf&c@q(=DNT995yq0!)9H4n!st)>l8LqOgXMeDFaRls{ogdS#g7an601qSGvWc3*Tzmj6iS8^)e(|c2>>(k9{aoZ7sIUN67+nMv
v3X`|(h63WCRI}guS#>%mlSYS&q{=tI1tQb8n&V#cs0!`K`etDi#^i6POB}*HznW01mZ3)9-XzmyP56Mm2c7QJemFMgZRg?{Ogb*FP1h?f^yD-pwNrcjnL|M
)tKgG)zT_P4I4+sdm@dds-B9`U{ME{N;OMX8uU9D%39zN5?sO!3g@zxsyjL(cx<96brFL4k5Oxd0MtLg;`o1Xt2)XF;~L-(fFiiKfOy1sV^1NT8>^iN33P$M
OV&9Uj*X><5HKZ3M(Lu<59u+ZmI5#O7=O)F)2yn-_LG+4w6zeNe;Cw;Bh3$5#^O(r)R{346{)W1BJ}%}QN~fMh|L+=6`w`A`y@*x!+G#!SaDNq1^q8K+ViVa
rqp)&`UL+-h?Eo=%6Gg=3|{-7LeR)yO))*Jz;h%ASbYU$jTi#-c6}@~*{Uu8-=sq^7E}Haq~%h=sX{8iJZgJQ%u4c*Kk0|Rx}vDG5OLIyZlI=3U92Bs#{prI
k_Dh-#S)5u<cvVofcrdp{6`P!t+0n@XmqRMo5+VUI7=FO7pKe>p}(R@CG5Dov#vE~GPQ@X)m5CPJ8(HWfFU%5IWRyV(xmhKtdv`~nop)e)**-T70f*#K%Fur
`QI+d0)LzSV$~59o^$~%d4y@2%7;b|EqFl)n7}T<gk5Ku$W2WaTW8{|U12@LOX(53pUirH?|U8e<;y>2st~^)ZX@QV+Zw0S{u88!C)k#BR@X9VXbgelX&Imn
s&DC!Ti#tZ+CmOJOVsn<SsZmnRid5DrqCxc3H=8x2g~pC%U3NrBFZ;bBq-f&ldm^gHX8rX<4jX=B{mkN(kWN01KR?PMs|-=Bpm&(HBS0nuhW-#Yugs7bAh{X
;T>-e(D|Y%@EkgPOmY9F3j4(01rzWm1t}s#LF!f=(@d2pl?RI@0#JTk6!Cm%U%}e2tLr$VJ$enV75dID%=WUZ9mrPtZ<dDFX`vRDyhcx91j1|+fZ><FdOYal
gh>&I^(X}v<w9ktA#++*)~MTg3$ls0T*>13D}00hh%sApV2jG&9D1~NJy#Km+2g~36ljGe`5#R#8UP+DzOYK#QuJX*Zzw-3nTx$7nhf9wfJ9r8V5QwwBqj%z
jhtyjCB%KQxSb{Z*0=CU?@Vn<_R-$Edmj<M$SYbfoM#}=D<LcS{3E9-U}Is`F$O78ix(XF1jXyB>f*)#wGVn`<k^B6(e#6)@}EQRCYp5XcDuA_3neCxZENja
bG5%{H2yqsHf%2cbJMH8mi%dstlC0Cj`SYu<w9Xx%6yV>tS(YLnGja%WNs1s<AdPU87yblhCwrC0DQ5QJO#Ks8cy=mYHwhp5p4AS{SK)~=CxY5Yctp<E!k^!
WHNP|J^9bLH~K$MTrSL<wPIYSZ{(M)ldLF33{(sNy;mmIj`Jw}=oi0KdG}B}wz^uhAWwQ1eUdarx1Z7^UVw0$F;^S4vXs2bmb;U0uKXDj!*GDntdWY(EOu61
uh6fNII@I!S-hEhq7l5x)?RjT&`nxTYlTu2;CzuL;n-Z2s0>Ug5DxH6G#>PG)79`1yg9RSc8=$6RrpqA0>&!WH)C}IZc{t|A)5}(ghV@v&Fu)KWxBIq7vx@o
-?PqjKodZALa7~M^TJ+^woSWfl=dCUhl;ue3Hm)!&p(8uv8DD5-bhF!1t6|h>&YnS4C@6q71*HoAhR$Clbah(iudrkk?@Yvr<b|mCu9*0k_{2diJ4Pc4tL|q
W`ZSUPbh_1M$AU6qDIR|F5!?C8{BaGc|1tm<WP+`mwI|8lDIN_X`gGaIb!a=c$~VOo-NN|xK&YQLqmnw45nW|8htJOZy9qu5Zs4#`H3;i=AQ9)5J$$z8s`XU
ojVn5?5R`7W6SH`l@xm$fl}4(vq#-e$D7ZNDj+E;4zrUj50MNO$duUawuq%SUx67@28=!R*bbUFAM*u1YgO!H6CxtW)Cn(SWyGW9(pO_btn-K~BHsWp=oUzR
*lK8LL{59SpBrN?L$hP$3uo#kPL%~qhUr48;yFK=f*#gQ8Nx$FR+-lb+*W}(M6HzL@!Z4w;fw58t^kbA_!k@&-~9UmMVTMIOAB)q2%hiFzk=MQ&=8+2epG%)
tkb3*b}J+?p^9$jT3;FHtXW$8nt;RE(O>?MXF#H3%svq)Cfr+?*rz><pCw|9LJjes&{c^P%~YBK=_G?EHP+6bg^FaGrCTa5PI63kLQdR-E7FMgfoyo^{dp10
jgV-ulz;8Y(3>laZFjlME*kYjc;@@(nTxIr#4WTr6v<-fbHl{MV||D{Sv7_IOkcD+e#~A2l|@El(uzEfj^w|1AjNrv8YjVL0@$({0OoU#^+JLfx<LbXv)sJ8
;7a+k>f52oz$&lj#TW|c)(VXhA)6{^;^&4=8z1z(Xf=GSqZ;O2hF~p9XMNaE+3_~C1=En((px6X9}ul>kwHwIVPXd|e-CMf`Ab01vJi-Gl>c{gn%P<DMcsqC
G0r9y`0xXIVeT!v&5#de@}=#yGBqO6<^NML$lJxWn%_toDx_EM-dPaD8xukN@4OC)A%vC~DIGaKFHJvQ9pLm3`T%^o+hm;nCRi^6;Z%c05JNGUjtd^9Pm1&L
a*JnES)Pk6XW|7gdM+(3JNch{-^!eoe#|<PRd^q>(~&lFv+XGb2=1}s3NcUX0gqQ-siugR<rzOtkd=)sifF#S(CYunO{;6p9fM=EkFAvsiwYSF{e3&>YpC&Q
y!F$lhQJH;w~}O0p)HB0<<W3Dl1W5_v&H=HA~(@J%)grOGtM5Gq7L%PI<shmmm7SV-W!RL&0xL~m^^ufp|poy`}{{9n=4o>7HPSCN`KeU>Ibv{;84#2&~QT4
;JBRnn{rg49M<t??I%9$Cnikip1fl2I}aW`ZO)97T_G7u*#~GFaY=Xx>@E~uI?$;R${beI-7;7syKD*9&*OGfGL%EQ=D$xm+tAf(3)6NB4T2QxEoNuWg#N)@
5nGbc@`TP_gDukrhj&g&P7zf8C{J$r4{CgX?beW~Ab#Dc^sB7}{GDDfE!gc<<58e}aAI5X;R{Poo)Ry3S@3$qg{v+qJZSfx8*fnU7O(-ECi>zU-OyRVgm8R2
tLMcjD{Nh@bL`;<vKb<0o)39c&Fae}@x;jh3-FpJUIoqXJK)+E&&~%yEC;aS<0}*a|MYKvA_GIK&-y`#6PoC4Ln-wM_=WGWkF)_NmS<c1iEtyC9EMjiw`gfq
>{4}1tw4&cP0#6TV=3^RfvV~mM(wKu3Tka4^4g}>&j|(`Tfj|+A`6%I&rbc1KF}afR1xEr%yA?`Gd7uNRp#F&<*XfHyB`_o*BA`y?bPWUG685)gYu@k0veB%
+k$G0E+QqKn*QdMF&GNKd&}mOmMaihZeIi`mXd<w;u4AKNLJKn62p_RN)rJi5Bba|-sp82J~6qVe`5gpK{<g`Um8At5(_qMg=N;9+HmvH8k$uY8Z5IC9R=Fu
aSYGp3S-$mcp3xW-bb3*xQ2f%%l7G+T^zr%r;U1q_WClBx_ko<VE2bJrb*Q-p415q!~%I!#eWN64B)!*z!a(#wuyUnwI4{xEL)%Mv~5;cKcCAfsE-~Vi}1%$
DypbaV6OCal_eg}4C}k8Q|e2;E9l6%6eGOZK{IU!td{@=ES%Sw_{gE%@niD_&#Xa=YTuM;S~64|MME9zBN=NlC@@CPV}IHS1O}!4jLY%DHLDk&FJW02_*YU)
*Jv6CfS$`GICWLx_=8P|hRKb1)OxyoW}GMYLK|Nq*9M_pc_UN9==hYs{3E{a`NPoqDKacNH*j|GERq>Ql!MKs6kBPjol+Lq^~G=dPezrH@AsCZI1sn>$3~5H
f<LBn7`eu?i`xZ)SSf%cNECm(D;S8MWd<|sfg>HYVJQudC9v4GoL8I|xgyOuuJ93P<lHyUEZaq8!LLM9SG)p;BK{-!9$!<*{KlENG|#dba1~WdHven+J3jO{
J|^?V!s0JobGOTDVDBUjR2dnIU=$P^dampv9>;;tx$I2O-h3=0{m-krrSh}_loxQe-n15WEDv-IGj{Ijz-hdb*2)W55O1t8kQ$Uq;m#m%lK{xp8m6$(=F$Zm
!%5b|zE-9RTOD;$fhut-d}7wCex^hpUo~9BO@6~7@Oq}-Y~2?wwdmW|acgud+;P0PI1a7IZ_jPNOkmW)gPYtgKt5{NO`Xkgjr4VXdU|wyj<T9#2AZb0jfGUW
04WPp0^IGEMu*g8mR<NILpUEroq&qVs;?G9mi)5wg~Yv?Wm_7*TrrwLr{o;kc72w;j!;`3*{y5#4ZQfGiOWtNQe>j$Xyr~=L8u_hR}@vWkUac1tRCV;tkM4W
lh5H&g@mO7H=sjyY^jq5PUT{F=mO}HopVp!!i<@O(ckWeUbE&9f|f5?Tasum3^K;my=53_I$2Eu#fw}#K*F4ggVK?C?uZ+{6Efp(W=1J$YbMX8UlJN#!N<lO
9!$P{$coTe!~Kg2HRAed;T!_uL|tztN2pAczyf&nMaD(S{*_$<i%+0~ml`f0b14W_R|Di=vQ?rEZ0Cd<3_Q>wVKkD6yhfB9qD6oQ7fz*=&oq+l;E-%bvj7r3
kAQ(-9>#8clsCN(>AQf^CeE2aqp(R<Rg*d4txFFGhwFUEhE1H~9!PdsPr&jufyO-);GAIn+TdN+x1+8-uL`sAh&H<Xz=5~G&+1Mda6#|>^N@dne45d*gMx6f
PEjKWR9_oBQgB&j;$d`OEZP-|Nf!`8#*zPD3~@HCU_Dt5h*ex$_nB8uBlDtRsNC|Y&d^<*s_gaPvk->t3Z;zJ;;Wce*QED$n4D0%V}iZWlIF4dbllzm>OU#0
!<a7DxPs$V$2Z>222x%IhdrT{OK$x$2>zvb1N^-FB52BiGbkng!^>owIk!4VTdNhbf?3Z-qnxO|fdu1s0=EQ^Z9nXawMAPiWA<snVq_6wNag9d%(t@ILq!G_
;C(d1Hioz5mnMYt5mjBr!8PC;m$Z@kUt1C(p!=6Anvg9I(cfe;rPb{>kgb18>{a(?Bg7hXBrQHjCLs1)esgT`joA<0$SkvrKe0O8jPKwD0N7cN1B>jU_HYzm
@ev6j6}aBNxAxJw<`fm{MutmTrq{`Az_v00Vl}Kz3CEt$k~n%)>S#vkN~(6V8U1N42s1QP98CM0ZKN`P^Xn}Lc^!Ok6p;BfEo4j}ReSP<n4)+D-30{{e+GZ=
qg>pQs&f+@lO*e(JNA&M(-UZw7LXOMZu-UvrAXt)G(V%SX^{hN{>;?;<>zCkL@~Dur8>j?9Gv6I-EA?^bp8Q8C)I5v)A2ngK`}%1-lY9DqYXg=hu#nJ>GSG4
?EK&r?1u?n;U3HmT*RVn5^=r|+~6WAx(*yNmaMxaz`J2|+xvnv+5mP<{9wi+erxLjIV3reW@b<xf7TO&=lr)_l==yOwBxat1V0E%Pj|XC90()IzN@Q@u804c
KTgO^12p)vSw6#S%Z}}b<OI=cSu5U;0p{IYgCQNMI*Yxyac4T3{Cf9t^GuVoI87XYMpz=_M_Y>hh&VcN)#ma}Z4u6Bu9W_hov|~BsH9lJB$-3l6<>6!_G+(U
=Y@CgS|%FY-!Zk`qNQ_*Pnh!b=R}rB+1IW`A<ZPajH+=IsL5(U-e;xG)NrW2W1mS}5YwYkf3UjU##7+u(3!nzH^cW^;GK|U+NY!mI<TMt2GjutAD59_l#QhI
L2<v6VI9C0b6}PJA*Aq_aS)KXSs45OJNU{F72avkn!l|TIPvlC64enhP(*>fI9PG~YX}i-1}mrN*}m8Rd(^`sWpJQCUQU_EX|&wbq9I;Dh;XoS(tcJb1jfml
Y2x~2#4)E$x~uBh^kAFB&2fwdHId^5L-XU{%j=Uduoh4qJ-DAeUCVY4`C<7nMMnzQKcbat?E8RrFVC#Trhl{E!J(cUTCC5|Y%|uC*Zz><WFm-FxG=iDk$Q(@
`Kv#fT=;}%J9+i(1Ts+L&Ons6^Ucu^vhc3*&owKa7Ixe?Ey#n(AL}@L842*_$)B8*0*9*{Vv+~w?^Wp4G}^;T<xG<91^522IpKzxY&uKk(8$RA8I7pgABhJ>
!f#+n$|18G>$&wERVXkrDSH2@SY%p+-rVJhF0Mc9*8gYq2RfUU*L5~!v}_$;=%E<E)#_F9>q!#kgAyPXq{Z1S-K+~2vOF>-&mo_hyp;IJ-aG)rv)*_TR)nX!
_~a=ZPTvUxfFVhJlkVuLRj~fORJw`J2jLy~T)!nP_%X89!xoKeAnH`qrHo*Ps1AhtGIP4q>34#my<^MSw86m?QEqlr4+GdD_lY-<tu=IhX}?3zoK%BMDjj=~
T!cr?Jnr-fT)NNBp1|!+1NR$BS&7lxssH@O30d^Wb?dc4!QIL7xx2?DpXHvr@xK18fS{D{DO3@B`=>xgfuZSKfT)#GZy{aWnv*ewbH|3JZ1LfScrDL?8~tQZ
$U4Vj33jt{w^UJ=SZ0+a+JSzL&|hk?nmU(QX<p)TxXZPWMf{NH^{*s1ErwaZ1g(|s;GL~Y(Qd8YNYSyOj2GPqXQ3DU<(^CrUCm+%BbNN`u|d~T!<PrCV@DL%
2V0DihozhDU-C-RI7{fX7R;shQTqF=e32W!cAZLSTwF7=V7OEsW{bq37LgHdoB)-hr*FF_)S=3Jqsk!U#5zZ19V!%ro)HqFh~F)WZG^~*8&1mK^r-nZDo5YQ
KN(r8GfvdOLxr1A;q6Tf5MH~W6!s5+G2KyAH$l30r;20EL_hh3)i(0Ef)Kk?47$kKV#c5)7S5GREj9&H>@7!@cT&2rL}kDjP^&v~R<d&==(5j|L&Z7Pk3QHk
%F`2u#7AVf9y@mEs~zmQE|;*$g?W{X{Wkz1RedH(F71GUBc7g>@{!QV5h@PBU)D-41Md>`dD55%3UXXp#iOqDv$AvQ$>tEzcG0)00C?tYOL<&gHa||s%;K?l
{%Qs^$XsapqL?ni+d!in=asI$-1XNpPLW7D;jGx)+rvRKx~e;0L-{rq2F^3yHW{ZzK*h`{tAOLL4dLmlZW|O0k0HM{`^6ayB^dUJ*75<X?(xiuZ!}pEz*9`X
S((ec`+1`fBzUKcI%k=>9OlwQ@OLQM6@2mSu9qT)>|VH{$XGa^)$bTZIn<xFJ$rfTQO!OOcw!Q5$K#~0y3q$>i{U20^ZPxKdK?x8FTN1?%rU+{T?=@drGHCz
k{&M=-I#({WCv_&C<LFl$}XmctMttebH_lpWOdj0sZATGGw@$n;0c!VIhLCxtLjq3VgBbtA3)!Lh;erHCAL?KCLd`%#MXF4i!Yd@C)0!6mguw30K{Roe`4uT
X2-*^P*{|UVwzKVFOQ7os-!z3KN41Sq8nIRI&UORK-n_4s)I(g(98|KI@GXu%+~HRzT+M)hKa6x`7fno!~mz>^;+gY(8>4LIz7lwGG6C+Ff(_v><@*O;Zr<l
6E6i0Dn89r0rX-^v@e;LSrrH*p7Ta$3Wt4X9M|OBkr;LVH*`&Zk*0oTYp2F8$Jw~Fzv1hd&@m(WHFHyOAqH9s#^*c2F*y&B<^}X1K%e&D*rE!FMh*hA&^yM}
)EW<c>ZlZFoXmJ|1d6XGM74q_mJ)&iTRA$&UGd-|yo4JutzAOT@gcDSgj1O#gkHXsj9@V@2RhtiGZSq5&ZZNO>tQH~Fn@SQr)B4WoP-c-WxYr<Gy=<F1K~YR
DmY2d(iSb@{D(nY5&EosbFzBJBBSA7u_c?m7|(i4A7r>UI}u@sVemyQTno0#e)_~LKez6Owsxn5mOy#W6f*6HsJjFY>gS3R=nvw8v>8_x+rVl-z#2`{*{n8@
ZjrQl5;lMH3a|qr&+3;VC-#Qf;-S!rxyRu3sM5fw_CsE>N1-K5p$Q6tay$r#Pr!W$bbRsI;nadLC{T&yziN!sY&8Jx!-AyFMA<sbM&bkt+xY@rCN@(k<N}sH
2p!%)Kvf29;iz7(3Y%ju_7w{ODlEIX7(EJ2IBuP~h>3pqsQA&2(Apa~np{G`cZO7<XhiAA`9GHEJQN{*Y9MS7sN%=y%yC)1-n|UYvfXk*SI!m&Uk!uh;PAK2
XrUd1|Gw`B4sjx3B}JMHhBP%JD#k0@1jft<y$!o7C?m{dzA{SsF|pQqLC4$Ywp~Da7y(nHxCJVU0MAreH@TuTv@-~tmn*lrsQEyW(00q~$dDkS=FOe{Uw0VG
3Gpdq(}zMF(S;zp%xV3Td)ZP5+N-L~AvhMgsMjp%?EK%dPIifCd-ZoKloR)3Rwr&p2npO>ISmFf3tZEABsHb*XGbzPAO~fQ%S&)eQa67c!d)}~mMUD=cp9EI
0A$bU7M4{=(vuw{v8K;WF%vAcJLm!E#xrIq56yhwW%#Gi<e?&3E5;DG@f1Bh<+$@Zp+ewLu0yLwfQoRp(64Td@Zse|h)xKmr;B`t<(sAxP8hTdoI#-T8Qh?@
%<gM7{A-h0Bq2(;`UYMaG?7ONRFcKqmr+7;O59KVZzAwvzcn_>?ImQRuEV^l%WAOZSIDh4t<2%{B&4};r;8&(etZ2x8o$ngXfF=%Gdt8HIc}r(z*s(LdcA1v
xyRj^ECJ;9mQ~l{JkpQiGb}U{r4=kMvn;V(y@Ys`Ow4Uzi?E2v(FEi)3bVWkRZ^NjURQ2<#ND{8Ft<Dq@gtLJhHSG)w`R!(8E~VD-4Vjgw=4YVe5I1goYmR?
xAe!l+(9J=37=p?08D-pZBV^heiW{9n?H=x2-h)CfQY(T)|w>PNe&tWfadUuupmj^@zEpzUK$emTmf$H6G_=49R`AdoK=Lf#07qpLFMbJmp#3<)-p%}ohO!}
9$0L#N+QPOhsH$%c5m&hoWr>1@WGWJE`iN-oKEIz9nXK&EEbyU%u-m6<!Qz`l<SRH;|>F^ZJo`-t~G;vJ-H7|#s2LwnRKP7&SXjK?S7_7iPLmKE?FDlC?U72
dgeCYQoNUf%*1JP6?C105xclr4`^f<s_Xs0mF|qc5`!iD>(`g&2earl^ejAe^x|-#g)7yvQY!3Pa386PEF0ClQJRb{jXQ=AE<4*pPSo|e$y#>N${<Q~0qbX|
IZoqOZ7=|kjD0Nh`#c{}JtS{YG)KP7+q*A*d1USjsJITsn)T?A5#A1Py>ZzTkIpW9J@$~cz$!JBESK+3K1N2A=|dd8MNJRKQ1-mrH#?Kbh?rkQ|MC>*Zb+Y0
^(&f8_kjt3G{313T{U@(h}I_3C3N?XqwC~1W1NhlSXbP6z$3De(*XFhNZ`J~7PS;3E3e?k-hZsT!5NFx{=&Hzy`;gs*7_brP5k$8+}}%Usw*GUJ66|8uHQ~5
y*y5GPlS-LA&=V~XI0~PE6m106-)~~_WbdKd#;+}l&UGLiqLq{Av&|eIq%FuvGV+ZFI3nm!t=*fDjoGqvemUBkeduxd1;Erx=X7Tf3YN~z-5`#rpvP%1hEL(
?x}{Uf{ZY(OgHW>cLbyN)BvO4vou`?WEx5?;0%zcB;;`1=F&bXw{-xHoHRSN6m-Ll?2Q5QrUQ^L@+p`?A&~wDcn$@XPVv!}M{9?A97-wG^F>OFhUu_*badTT
7C|0!kxwKb@jE>etg<n`ULtUj)oqhtvLBpZsXhTU00=r|<drmhAYDRFZxm-kfx|;F14hy~n1iz7hfc+gj<*f7q&P>e&m}Sq^GQ`QR8kVzNWk-^vv#|$RN)ot
=PJsz62!1Pi6~ywu%FUuMX~@&`HP^vLM%&SwUgO|tQrF0lg4C#{_G>`ySnu`GvCqD36e9E@RlZBU4|t@aPC?&VbH0pj*1h3B%_*S(g-OooTjPLOPdP+ZL|(4
y%j1_lCD*z2d&(jUu7ar7>c5pgL5?qHEF+^TiW{vZQCO<8Z2bc+3OsB@GtnUJQQv)G$V6FAsV*_(sdirnvta3R(bq(E+m_s6u-rTZUCRS!**n#ltc0_FH1Wl
@NwBhN8AjyCwmvvUjEFr<r{uP{PYeSz1QIj;0H1M7A|Yh`cxxmqcW@7-#Qy@j&AE4@h@XrEx{VS$tlG#sgDne6(O)ys~bjcf>15aXy}U1dNzCh7vD#fb2LjH
GM)oo{-?1A3(rY(KXK2U2&Ge}TL9Ydw(dJb2ezO;o?V{ROq~yr+0{lMnf7*Kz^o1O)&aJn(9=p2I%PeiK@QIPde%b^We;As&#hv3H)U{B$Z9%dG+GuSoLM>T
eH|<^(zXyv?Yrhb4}yh$>kiKAnaHPW9a=6w^Z+oS@|~5d7OSv>Ub_=!26W_+6eSp|FyjLggG8gq#@%ntxlx_rnHZ<5L`kUS9R(uh0qwvg*)X+;u#YsqXp80p
V~IN26JqV1xhjur9kp7ovya{ZaIDTW0o$a@?o0VJ)Mf)D1oonJL&Jx|^WHVy-eG4zw!av;2(iu)EyPmMi8ea1&a(JUw=VDTKq6Qq34qfI;$R*Z>1IB^q?Wxt
b8LW=Z}y^)fdarij>tT9QWf^?Ro8;QUmr0UA$*1bm7M`WF|l^GEQ1N8nu>AEol*s6Zm)f|;#sADILNxT`|8NZ5wJ4^&ehcm#J)EpT|12*uFc$~`))cM*XrH6
(=U8WusikP(oEO{HR4KiNcIAkn78qEA?zG1jpCC0P-RTdOysnk=>v8NjP7hMh247%=`{hJfwVIzJ%}f_=dB53!T!-*wnA^VBKM^DRrn5=Bo~YG1I8oQ^e^?X
)Yq<o?8jaj3Bo-^?I2X^*MPRo5Vt-WX$D#-eeP6|PD`2J<NP67YE!D6KF3Ry?db6WuC!t<x#gu#j&hwhDSl1Yre|45S#jiwN9W+dPXRI2%<VP4p5+YG^F!n~
h-iEC%hY<EqNN#-f>vBBBZcaMn)Z+>aj9}Zb(}^iUUC=T(rXaD^-vUsKNwaX8KhIx^r`)_+DUD3RNsSBfI!`dgG|b0<c%#a$w@t*N9$MgA-rYo9e)biL&?dQ
^rSj6%l`mFDjE>3a-W}$aR-+US|iQ%-^IJy;HC2@1s=dWK8ft2TQqE;)99#aG|}DKEH5a#Ocd<JBJhSZg9&UnoF^oK4{5_*lcy(i`^~|u0sKbG00B8b&SmNE
Wj&QvSzP~EkB+!36m#d~0q`MERut0hAPMPyPX}tlD9ZCALIBP9=MKlrmUnsu6i=XaK=U;{m2jGzs*tXtgQ24B9k*csgg)FWB5d8gDkZ!?_Fo8|p0&e|bKpp*
|L>};hQzn%l6Yl>!MyzcTbijcOMKIEA_==#A@CX`oDU}L2oJ(7CczuJ&2+XpF#za$W|0rYGzC{L2frZe%LfH(Y4H8oHQHU*K$`c3%hkusBAEpR><|H}bG;F5
ZbuC=?>blZA=RnP>1S|I;{zO~-MZHv-~xK{4I3ovFe-9l`1$59R+5-s#$NCzgZNFO(e=u>@lu(AOMW-vv2~FW#I+w}^_mdiC14+bM&AXHAx0!01k1`ahkzZ5
`iOSJ2_F;k>87%L7@MHA?mkich>H!qv6_K)Zp{auFm-1A=uEHY1{6mb*A|IwFaD10&Eglj{llrmCK$<_P9=d|(0jU$8=4vuOvu8N+Rqh1CHuc)ml!20uZ2K3
GcS28tdS=@bkF%`M;5ZVpZ$gk#$<aRm}Bof6X?A^i;uAgIyqNCZQ7lHE4l9<sVFfe4N~z@Q+MGEZ(Zr@T|r<1boD!uO0?(oO_=|Ko%%G;(-^m8zu<I)oR*3h
_|y$}>*w$)yUbEx7SK+M25JIu|DW>tG-#C5JZNYHMB!~+wCwOx3Dy6q4ooF~nDz*M$lqJIiBj0;6%P}xh_xZdjaF4|m5xgO6khz8$2q>E*(PYoK0cOnmidz|
Jn6v11-&4@-z@gvv7NMfFHMyaCRAS~ho9xc>EWr^RM4P~QUWVd)<?w&0><6SRoF($^uA^Ax0A?$t;jgFUCOc1Xxpe-p4?J#lc!^|FG06m)Qx}Breg}5RM2F!
!T#Kib*bk)Y|7Q2eJShCBZ9H%#$~5xBIjCuZGhRexn^R}S7BtNp1t&lTQUpLjMYw`adv~~L-UJ{Sq(I;b*GD&iFv~=2SB5_ay-<AfA|m0Mqc^V_QOgV1dV19
pkcp)G(DW$m06Y8u9~CIR?FBm_@{q#cC0=y{|m}Y&?cBT8ZS0H4jxo~FFi}KEEzzVc<J6IEyxKTLX_MDqQEHKv=T+NMTII~sn;@=v0jcJGIWFw{8B^*G=#t@
d~G-|&j`FM%c0DMME^yqXvq#&vD<tuLtoDO7GBW~XTXhDtu{3NNM5g7p05_2&mKO{KC32Y9lifo-TCq~fpmbmyw<Gl!H5g_O)%+Do*x7*0@{jGMhg&E+<LKS
#^D5ww3T*pOs#&q@9g(KDkHy}9?@MeAv*>@ya!q`!oI(#aVTzesrC>6-W_`g=KMnu{EO|clFh6x1*Bk<+q4%~JIE5*CF3koRE~(9t31v+n(Q8QHx-V{8@UZ?
d&l*smAe(h=8jbG(<9-gjzS&26l4J;o~e#g0Ca+{Hmaqm*B?Q>anXsmvK~aHL!rO33~-(dt!|Tvz|pR#vc}@tR>>T9#KAf*PXll6ihaZiV!~}8d+JOrx@i8a
UZOTEBc4e`yAauXwUmvRI^4mz;o4v(e=$5X{cqJ0KVJXXfDuAq-8>nAK0lrs+#C1_W5Kx5CaP7eqS|#AuZk_9bYgo8Itax|FW&o*iUyb>D5KO`5d}$*>f<7%
aiIm>Z^qrUx1GEg7xN$|Z}7l68dMInR01_)lntR}N}+<Py=niqQ+=Dd#YK<!<Ns}-EvE`)cWVA?d2}PyWD`m%d5<zeaCiP~ONH)R$B3&7P(p4RU#<Fi7ja3o
0VpKPy=QS%|Hzu+F*(rqCz<=q0!Opi0$5O0mp<)LYq0S|rpGEa|Ls9Se-A~#;fe!ARRjp_Q77=-W^vF@Aj2*h{YEG&^WCFkvXwUlu&y%f#SIND`^Y;RQhVSx
4DfC+3_}R!y6mu{r<Q_xRS*AGz`;9+gJIzk$JA<C9G~Dz3pwsE46?(ZNF8K>?o^L}#$C3d)`QDUeP^&Yt6^1nt9yEDn*`%mzeH#~r7b;r|AmYv)V?^`v<t%W
b+{2Jh%%ZmmUZ*AJXXxUBY92-a4oE0$+OXYoW3+LPBA)kk8L%FarBOz0Y9imm*ERfdr<|`;u85cHZtM=NmZr0doRfyanb8y?;f0l50YhFwG_j8&#|d7#D(8W
W55gx<{6$Ug#2TBd!(*?ZUAzJv3_2unTm6=t(Ur6#y^uc=jLhk!wys*fBF1a($4S3JuR_*qFb-M%39L6K{iwrZ%XD@)NHv0f|BsOOsq_2ev-larVVXsVuDEr
)&ik`SsH1Tmht<wa>ByY`Q~{JSxWegn{ZcQGVflbq5po^<Xk>NojIBwd3KTJIE<@iwku_W(WWs?H`dWHKF4?GODH_3DQSPS<Z1SHO|>54l7?}{xi=JL$7=*z
(e6*$YTw4~vUx#p+W=|Q9BVZbqqIZIG0Q=Pkh*m79hmH-QsA%|3^&qAHMRS;;uF&e8JUpP-2-8OM16A82Ovi8C^A={A1;%;YpxUt95$?kCTR@?5zyIYaTH-Q
kev|yPC#zKA>&V^hNHV8L{JT*nS8cyW2C@dz~?$pbfk6r$$!>$HOScq7lF}(>#PCD(S}%+V9z!DLnmP43vs^qQ`&T@{z5v(V|Ho~<X6RbblV*D9U_KoUJcFX
Jc^6((L6xoy@45WWh@0SB!&%U;!zB)E0(bE#31-(4{8o&U<8w1AP>9G5q2Q1&yQf1F&%?^j0KcH=(XvM+j#6!tmmdRTfokb-w~!EGpLf@A-Eb|<c2ZC`0YE&
qA@jQUM6IWym$q{7VE@P?D40xbH}QG3P#Nb=YyV4=-CYJMYun@i*7=*1LuO8enTnY98utPudp&_OAcZZ9AP|5GtZnPl#x{B9qmcc4R;;!mfxgdI%`8opn<dm
Q>ivLMQ1P<iCs$E4DlVhe6t4do38(qIm9SF68Wz8CjS0g?sBaoYOKbZjy_lqSm+XlSIVxc@B-l+rAN~UE4J`!YacBibgp|2<-J*r)Wkzj%j6&LKaIUsQS2p4
_u8z_MS(a?Zimj~L12^Orf6Sx-$ir{|6!*=S{Xdt+d(nZtjR}P0j|H`UtGr}U)-%5W<Z;}!GW?BO;v%?<3EO;X9)d>EM6gfLD<Wnm8_k{cwXyaifKgevu(2P
%E&FdX;EUmVT>RT6F@MP)13}qw^H*+k%I2zpKqY%-NFvdxZWc{Rw1dgs~t+43@8Bapv!<a=L5ep%5tt2cvp$=`Gmr(_Hm41qN6@sJ8oLw20|)KojOMF!YNAr
75zI(=DYvfXZy;$lNM4}$uF}m*<|3;x*B33dP84XvxsH+H6=NG|Jlyzr0Ro3ZcLfJbXMo5w2hu?=Lg1>q=d<>zXfc5s}Jq_BEqfkG_qP)8!W$+rM41x3Q}3z
R51n3$^HJH*1F^9;#kQ=AIf1r+BJE4_C6j4JJck3@Tr8dH?ck*$)yA62!G|d5VgKV6!Ezi=lOu$V)Uc2cBf-r9#)k0yqb?0a(=a%DHz{GZ6K*g;L|JvYn&qI
q$?VI0EQWoNa!V6n;K-}=tKe(SIox-svM;OlO^(ZWW)QAjg1+rEM)CXeFNm+m7sNLqt^X5g=U&mC94l2iC}d~&U(OV)FlSIkX<9OUbQWa#N>gMt9G3=qLRfB
|Izm;7J#^Jac@jhUHm4XZoDQGi#<NX-Uum52{mTPk*RH>$w6w}3Cl{Ru6x^~$DBH&-wn^!_L*cgq~=G4|9BFnmqQ<vrmBBAp34j>Q3+PwHN%o|RX2T-u^-as
Tmi%0B!BC{x-dB6lDK<cRhPn%!K#|Hh$+O0SjC#DR?FQGJNR*y=07xRsA^JHgUNqp%{ky1i}DDNW)zeVt_s~F;&h+Dv?+^k{67^!{iHmqtMjlf1QQAUzT+yG
e(^(+UdyA2wfZ)0RQg8RrBrn8IcIV}zD277DOL2Img~Mk;8zrk!hiTe4^f*6VGVXC0=SE#?e|z8p4Nok=ZC};anC09Eq@w|sWq)vG)pDNYABaP8t$uN)&kev
sb$B4IA_ltXahX&qUzoH>*5}Hs#{+6hDiyxceqpHXv%(3?4p$*X9$q@$~?HiP6S$x?xh50@Z;kB;uXE^olY*65=U_r)Et}Sg8_9wlyIPgyms%*gjq986))+j
#+;pAxZ1b03Iy;sj0fai9hirSr<|_(Z24}xx8Nb7y3la4!^yhtj!B`hrndB1@--o600000edQ?f{6*|(00Hs20>J4W313-GvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_flavoria.csv')

In [ ]:
# listas/externo_foodbd.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_foodbd.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5=D0$tWL*F<8cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppcnnU{7bdih4?B9z!pNF=Kdc1QJqa
xWK6Whw&<}@YIHDAeHvXrRW(w-f};A42_z-F65CPD_ogkB+bKD^WIPF%jvh}IM^VEBeE$7sYlS}t-##sfmO>;1n!>-pPx`r#jY%ezg}kf4m8aY>oxP^cIE$Y
=I46#_|a8F)Dh(Y&6*y#T@BXo+Uu-`I&wZ(&tuUi_p}4wA2WeVNI1N{qZvsKv@>4AFmGN!>`pBOZeaM>F8cuFhf=754Wb|&8x~YnuQvd?+J0pqk;&4BaK^8W
eMp3NLqwY`HUk&f<02t>t&PkJW77&8iB3xz<tl47Qf7iIOU6G!U-eAl1I1!~9S@@0E%-O{1|c(QDsj092~+M!9PrT?F4JSmTIsgTb!|qfQCC7~!nF!X({1fd
t8-ymdWOFe@z;0<apA#E*&Ejd%LSRV@^cX&w+@Wzj`dg~=yKj7=cfN;8{zpcBR+$QUjwJxVd4lKRj9M_3V$)V^wEKcG^6t3f0O*V>nmR9B6m}=RNpaX<{B~U
0Nlt5cX@9E^(>hS88S>Ftlq8<Ch;S}qOJZ#qDd8-wXM4{w<Oam7T>`h1TsEn_X8X7GcQ`Q)O+f~@4uj8xrzEc(EUc?^$OCBWg)Cgl8a`$!CAt1??VY#I<K32
7K>jO#I9=u1D8KSXGRtmYTU;B5$je>yZy^dS?k?AiAGtt$!);PmMsz#%r?&rdQ23-)*IAl?gD#2!l~&WTBxnN%hAwSLN?T>hl8=y49T=~*u4O6RPQYDu-UP$
<0jk@%UE|YyCsh3?%zY@Um1DAK`bzAlynSkqj~Q6lT($d(ZGU_K}}nz<s?blJTD_)Fg7^PZYkYAQyF#`Joygm(a`NkEXPFcoaw=F401MUax29qs~S!A1!rF_
+UzBeeZBa%w7t-lR!}PJm4ed%{W(?3Yh6tV=@K%oFgHOr)&35LWvw}pOpPgRZ<Ic$4RS=uWOgV`)B<<Y;wU>6YOGY^JtEQ!djExGv|<8GUQ<?vrZqr<?!1wR
JgT?}S_;YcuFka01X9&6<NOHd2h2X0(bJW9VY#ZX+rR-WIvX%NDG#z%MiDUCYW0a|)-rq>S6!RHphlC8rgXHku9UNx(pjsi4oZ@MaW;0q9a_(p9&kooVyDCw
tR4G5m4;+(4p-f>QQ;gaMo5x^&D93jplXF%yhzDC#?=6OA_fd2vD^tCZgIt0GswS}r(_vqk25I^@1V0H8<~QF{2nDmP+U?(Sco8-JZ_(OcOB!?;c&n8Rpo?Y
0;syj-=zkbIW#c2#~P&8-2As0n+EVzII|GSTU-<3b`a7!tXoOrnR^bk?-V;CYhWoi*(GnhT{RzdIK$LaMY~XDyQm5p(fttLwn^mZ`c_-1vro<2CzXuF5@>oF
eCcG?qkcrNknah;4gv83R9ZG5Cd!#ElV@S?fn&llt%TqJ+cZ`Z4#67@v7sYL>DaflF9uU`hFt3kg*6JC8vxSA@sB94g0)A{D>;}*iY*gmB!I2YlKxDo`bYvP
V_wvwv!d`n7qM4f&8~)t3G4UvJq#VAVAuT<zg4i?u6j4xqdYD500}P-vxi^+ArPDPZG030tb+BU1~E21ChTz*w&%_X<pSaCL{%vYbr07fVTcL1pRM?E1lhzg
7jBZ09I#25Q%aYAhd651nuj`^fN;i(8?7oZIpNvJV1KBwQvcWZa+Ba{kfi$<50e-M9H&MEQ3`>_wv>E@Sa6)yk43Mm%$~`_D9;f&z+Gu2qcu;Er>7O%kDI02
MkXEdSX@?H$@OobDIGR7+xsd+=n$$1OOe|q`|^T^0v>yv=x)B3u5n!`hTfqkYTIEcFGi7Sz;>4Junc}FRsvKsC$1o`F+6CLniCrJ-7xRg*K$Y+3(~3*f38ou
Ih+*8@ca)#O{BU6`Cb6|42oVe=rAg1iM0Y;_}N17xexy3Janr@h$-2)NhdKTeM7V-s0i@ub_7fwFA?Ex0$`bxRndpINqDYFK9PL)SJ^B||L#&KWM*M`20G8&
F^tMgVK)C%i3gg*+lf@%V}-}q|K7+V&czk6?QqPWiWl4fFyyVeI4n_HY33|;P2?HtlN~*}II`mnS#zOEEdQl&5P8zk00JdN99lFNCXt$l%GQx<rlPDd(=!^3
3SoiAa6xc+R{TUom7nh-Xz!)NbEqn(zl55HxR>)kh3|t;&lb}_!G-@HR_p=o%fFHbvc651B=?CA9i?+;>7%yWP5z$i;ArBx2?cs`+y7wKhp&Ssp{kkz74#Bs
34ku-?=#1}7c}eP_6Kv9t6Y8%DlI_jcFhZhbs-TbHj&7cZYN{LP97(oXLcjX1*lh-S6bka93jY&$@d`MIA;JsT|>lb$nl&zp0a_QB2;zc94EZ;)jdA?@%{zX
&kqWRMQ@>h(IIJUL%{ve<^n~J));%zQ=alG^uS{_`W8}pr|I_>IE1fDh+@JE@S27=P7<D@{Pl;0<Mp{E1W&hZj|Acvleov+D=Q5m)gx`8E`|pXDD`(E!-Lx;
Lr%oAmISveSy%HeYKQW{B56G!iqWf1M5w+Xrk`6)DJ+ExtQ$7T37xQ^C==I4g>H-c!^kP+Z}Lq{X6>=LM1>Y<3q#1#+@@+(;4-)WJC!wA?rtAUZ4g+PEcS1E
l?6<OxB9ilZzU!kTB|zx2gX0-+F=Zu1Is<rP<zfoPlumW?d+f*qi$6oa-;P7N}aWKIJWzdn{dPxMjbbOQAmQuOU+We7qNMNj8!4{SFGeNTyWd;o470}j8>MM
>qBX6ihgjPBn)HX`twP!m)lE85kP#cr-)Ssd(Q`|&r!jw22|g5vYM2h@nh3!OBt=Yt1-A~kAXcS*zH(;wb~p@V?_Ow6)9DjiTY(1pt)VJ$Ab0hf&|1p&{Bmj
*{6Ma9^cps3~EycyVxtt;~5~e%3bAS;s7Q3QL?^70S4@UJc1&g`6euQV>A9~-qYi6IO}JRr#c+Ct@!GTjQ~%h)1E%kw61sX)`;Iz8Kxhk$;*e;rsc*>E^|;O
=^MbML=(05Ip~4CUiL`2kqwr``kw9iQk_)2LBcYXnkGPiK66NS4c;hZF+8c9R69CE)6j8trcv#UlA~q#uUe`Ix5J3%+bDg-(3kR2l7C${MwuO?v!w@5UQ-MN
J7D_lP?uk%Jd2&&i6Ap@i2`$+mTj8IxB-{Vm`uQ~d!c){(|)bDflzv;nkg>ZHg5H15s1<U0SDdzK4BQ{&cWN^)K?<`E=TJb$TQRq#B`<U^r)eyqC1yb+=DPx
5o`Qhb{jCH!wLY2Es&86^4!Kn@|c$VeQ1&X24#>)k+xcq%5#+v?cwO2Og}OBb4+J_AKU>jSy}g*A6ew=3`_7Nz+ET9$szoQ_a}H%?9Q3;8i%^-wGZ8f?6MDG
tF9~gTD&LbjM4-}0&bNe-TYbfzT<;rq$QoDHOA2h?94t|b}J?~M0}D1n=%F>OkQT;Sy4n6raoJQBs5AydL1Pklz`(i<?eK8PicA5_OGYo_)rDkU(@K>vBr6L
Udi&Fc&$51!LQw$`q{Pll(9{v3pmgtncj0VR86Zp`mEc~=5AU3P@ZzVq3;M6)0;yznPO&$KfI-MqS+^>$L{WMI~PP@@`Tf(!J725)YL}cA)V;){JDpvKN_>1
)u8!qG#BjPx}-yc^WF$B@K|DeRjpd)wqhEIBJ5d@{`@Uh7tj)alas?2QOymH>l-h34Ka|IE(yWv82Z-y#oZ+MpDZZ{YX=UT<_CxJ12x*MN?1<$S>Mn}eCHpJ
Yk|j3T&<UDbkw&Z_=Klbqg&#H?k`urx<~QjKjLqKHPn*H(xR;cg6$$RQwb7pe@7B^qlKS$IU&TL<x2}sb4D7iLiT$-6?Sg9kVfkC2D4y9WmC#(91fo(q(I3i
34+tAzckq~`YJ`^!e$oZBzdZi{Yh!>*DY%CT|uR?h|FvVXYdCd{^ONnHn3Eg>r3VUHIpNCR}_e~WbEwGIK>jDSc0mcp1{G3#1||QViq}WHt`UyUV;N58Ff<I
0gHYS8AFiTppQxIQk^|L;$$2runU$`uMzcU240^i;RR@_2JjNDlQb?F7}1u@GjDOCR<~%P*2@ZMbMo|jb-t?et;?jY*l1AA-mADqh03kGy*f?y?3?zM*KMiN
MX~lr_P+Y@#=!<-7LG2jFzl0@<yD0NSvg-(R9j0wSN|gB*{AGc`wME_xM^^+!Iea7y&!Vl&GIN`g2qEL=$2X7dT@d!MdPS&Hq@ystVZoBSv3d`(pvF_Hu=&2
C8Nq~Wrwm0|EyaQCi{n=XK}*<kkqzoKtZMODM48-=+aw<8KzE|_i0PJahU};5L&Y{oBcES>xnokFJJ+OqZGGJH6+3(kcbk%XfVA^6U?x)C^;Kl|ERZjI#?Y`
YXCJ&#D#ZsdUI60;jrK&gsq<D(FHC;hY9Vzc(EEncE((oSUQ*L?K3ehUTYvoh!w^}Ri(C?Z@mCL?Im~eyO5%(X6zaX+yQ?dWd$X2#%O9wi&bIIHq3Zs2Y}0K
k46@<(Wm3VQ8=4(RJIfwaf&v3M@@6(7Db(T@hprn^dV20<Zll1^m%-}O^Aqoh)_-C*q1b1%>v@26C;Y?)p{o;xSE{CIenIC6|xkuWXIrR3ZQ~bfa*=p!CvmM
#^9`kjv%7q01pci6_)w$(nGTv3Dp$W2Lr~}I5TjQD6@sey!N7@wDts~KGn=Daxtrmck&+p-JI^chLtaos_3)&^_C2$yai9s6{7YW9Nd%q8>ml1MF;PWCJW-V
R!6q7zWT_UmBdhn;na_5%xnOtaYFYqrp43E6PsqS&+5~I{noB6DSO-Z#{r2l`BYcZOJcVnxz})#*R&Zn+6d=x6;4|(5il1H-`uxlo&wG`bn1%_QopNf+a?pW
#aVrz&)f;U-(6=oOmY`ia2YDc_uM~-ntQ;OzRMb_UCW-f4Jfb8fF4fOMoGZJe(0+UeIGazWq=*eVcGmtq8I@#ro_^wP&L$#T$0|FVrxAxo@f{0cc1B5Im>*A
b9<^knl)dzDOU0p&SXjvSYENVbjhKhBr?1yk|}YTeMY2-8!cMA_#!J-%N33HUUbaZx0ZB4p@R@@%yNJqopUwTEd4sAI}D1mm^x-w^`mW9<0=S4q-6y(Z>%Lq
oKpiGwGcCZ>}gfYORmiQYzTT+sYIbwJ)L{rsLQ0Z1G=HzLB^yJoj#e5?=|U^=bOMx(UW4W9|}v`u1I8O+Jh})BOJM~6|_9Q@r+mg2(kThnh$Icp+(%p_A$u#
oVOASuzgxwLoAlbQ5_JHtJPw|YjU(E4p}{uAEm#&SmnZszeX#kEO72N@p55#I%_~Y0xd=G*tu9GgSrHR-5icyT|g@2O>}QT(e2o0nJ&lH`=&Np*P70ahzR=u
Es2_{#>=$vAS%>uSE=iD%7gS;Th8+K5yMP%^CkyU-?_{^u{Xh<PKlyrWw*|j&o(!Z04J~zh^b|V!@y;YgXCvJ^yevr0E-qWmFYS(kTk4h2Y!(@LwZPAFFFgM
6v;p>^EsiZDu~8KO6CGwiA;q)jx<oLa=&heXx9lzmT&0FSrs0D05-p{$ncA@|0PIhg@7744UY8H-5W@p$ub9YB53vRT7DJp1yt6#S46W9)i8DSu7kq{?ix^3
nNk-^S)VRVbB_voax0-WG6IuMwGJjbio3>&B!gQ6Q8hTd!a2#NH;{0pz7CpUV$5r0%JKXW=3-0$Fcg6?O3(&Y>5u|m?d^b4ve79q?o9mwza0o^n<v&^*QxVM
8!``w5?ps_=w5oJUNt`J=-~z1&NT_1q08EfOQjPp62)Ak3;gEZd<|+L3aREE+Pae<Z9qzhj?6md+Av-4x*a-!%S6c&Y4DfA>2a3g#Kl8hekC9GXTZa*RT6*d
Aw=FfIpoGtFjE<CF~;4X+4W6|ho@#W<={9_tFU?inZB9L%^4YdF)Dw_i<CqnD7f0;fef9-N8tTmQ5En{tTh_hR_H$*S*!`%a%~;)g=BqM=2A>+bDyOm@ANS9
dt%h<MUWD4J`jr*_$PD}nn&VMDjSbyT(ZncB8u+k3<A++^%ji_R&idkeWLWAJUk=mW|XV75vs5F+(sFNH-RonkW?!HvieT$W4RX1#Njng7NWAD;As_4+p`Wr
r%7G?G2zJ_v?spa7L|qtuYT2q-_TINyy5}zHIwrXu)9#i$w$EYILib8l(%pEP`WXaA;~6~{6Tlr`#{#!8~eGNttMY2dT!*);p5#XU37NCU+8?jRh#Rh?JW0m
9sKeQ@u{SfDMcfd&X6xKr)Sy^%;e~wgZJDQ{mgw%Vb!%?tshfvZ-g&_$jMd3BDND=AI&>{A2~#-k@43j&GKtNS8q6)|Na530I&k<tRB<^sK=>H>T>`QCwIdv
{$5DbLua1gd1BYuU3JwXuHg*0{o6>h1MdGpZF>%|Pil&let*B7=8W%0ZN5HgHLq~h(hHQHs|$71$;83rwfj&p8BtakH~?Ao72;J1cl`{+GE#xOc<>@$eA6SW
Xz1)|c|xI_WBV@?gaz~mqui^Qk0L$c3n!d85DiAh1|T|GgMHg#wa5w4%tE6^5q~+3U>QdXn{UZsn2yJtU|@CvYkKzs47ZReYedy$wrmrc1B#*y;AIYz{z<1g
xt+kM(4v8J!UW}nS8|`%v297z=J*_>%{{g&YLV!c2+>BDBsCAbh^EI*6YL6lupa*k-6(6L^eP{Ks;=Kdni^XFFyDlV$pPRq0qTUm)(oIFdQ`kU0wum&ghwhf
u!h7(mBp%_x^dmE`I3aB0<y>|UkpafC$OK66H(SxR0Pmpu~gnx^U~$DFrP6M%&-P1K`c7`4oA_)9JNUC3N_(6Cp)b=0_V?jB;AE6%(3KCMG>473W3q)25>H=
n2TWsFr8#^*dD%f@;nH%6q`K&Ha0d^y_<D$L{cUPq{z}c)|)q_>xOb5S+OglJ7%+w`pFl*V&dRL$EwI5#<A$hjc#i^8YgX*o{#sy%o=Dr8K8`>HaeuH<{LrS
WHcT;Nq%J--RpT78q{F@ws{(9tRHo~;Dm#2E2p~K&59&~y1}7H{81Pr^{&v0pfB{XkcU#jx2Bgd<z>j6T7@70l^_s)1daY{@E2Bfr5v0txoE~hGe^##|J0=L
h~pu}(mcOeAMbkk>{!|2FsLohaSK-&GJA(N;m%RnX+U6BnAAX(Q++e^J3tK!dg~mwsyeXUD>7xFh&M<3ECCdSI(8qOzWn^S$o_}lFqtZ5q-c~q?;A|X=khcA
g8o>MrD!WGomcO~zS%g{hD{|(^4?NT!+z|m)+Y+6((DCA`)D6f<n&kp<9AQvMbcJX8nAFtH_?5wi%}3CGlM@fkF=blQH8h_oS5cNzZPXrgcn+**ot%K3nlGT
y75J!5Dx2c86PCMYN}IeVcb8KxR+TeJ6mOHV|6PwYI3L-*1ZS~sdb^?wkvRyXN+6nhL=)a>7!lATUCY9C%!mjZw^h=?W2?H0*jAoy(cOJLxjrF;=(yt%@&4J
*osevc?QQd)x0@j$M~s~|BaaaoTFG&v?I_dC|mu{Lao%l;VwEX1cJ&EcwSnk6@*1~?1_W~z8qKu41M$^@v;lSR>uvOK=x<nxzQsS{5WM|dpV(Sq>SGb57^F5
^JotP5bE=&#3CJieyg9Q+o6<GaI?_PH+Yl-{1L6@$748K-$giZap+88C`%U65qB1v?*MFz)e$CCXy^5JD-|BooT<Uo8*97WJw--sNNayo>gJEgBbA_*h#t4G
D|t?5hFkL$oO~tykz=a3LvuYXs?{?$0!-Z+$0KxfRxVwM@(bsx@H+A!&p%jwGF^N;>9C5aZ}ehe#KFw;iLmcl-nvJ%skXa*!@q{SH<fhj0Ryzd(HRlS9u9Kv
+F|p&7}zTQeC_-d67<aDE8R*rTz(nVvy@eVAi`bRZI%BK^|F#=up%0_z`B<UcpPKZb{eG*%p>6-|4Ch#YMxwL|E6eCQsHS7M%MHO%cJNgYnRcs6mNncFNVb_
PZIyjFH6J*mQTf(;<gle?-R5eu2VO#1Pt_vWt>G8iQ^S|r@377N8+<Glu2_AuY*9gnhv@FeX()kh;){t#vab4Ni06h`GLiX1R`eY^Iy(^eQ)0+y~>G_Rq9mj
DgL8L@aZiiZ*=Vb^r9463=%z@^8G-G%61S7DFCum{yqxD{etCoK`9q+VKVl-!0!lTt_EWbcr(8J>pV$-i=gGAt_>!k)&fa?TEmuBj&EI`kPoQ7W`RIY_g+%!
CT{V)%)cZ&PV3wtPg=)9)|5opBJF19e-)&0g!*H(5N~{L7!r$Lbg<wXij8_cFC1)l7}u5xA7xuVwqLNHGevolR@5%Cr6T_^?$#1f3MQ1oxXk;NIN6CLbrp3T
KN7lw#6Xnbm@9`zUDBSTV%<wW5iHSivRuF}5L<gkkZ348c%SWnU@n`qL93A>8!{=-!TAZCp%vtn!h>oD%Z)qTKEoO65b?nmBi|!?i@n5Lm>4^Bd=Fnv_cy3l
YtkrJIKpBZqTZ~Ba_mf+PzP)l$B}T*aW7Q&N+BmI(>R&nI%WRe@nPpFxFGXK0fjxg*Kf!vAAW}<7Ua_U+?sm1ntLm~NTXrq)wBokAi3k7<m}TIqtXQ~-(rxc
(;_J^tM1S@m~m&ooff?=zTK9Gc6@1%OljQ-r3k#(O(!`i8(Zezkqg1jA)U1}9jQpFr7i7qMe1dkZCjoU*B@`GzRe#MiKKr^07t8|d^F?#x{h9<EeaR<eZUm}
R34=}JX92Q1N^H<8(F=g?uT$OXdOFd7<3uh#L>W^`ukM}225xivesFm$}F$8hN2+&tz~sCVC%#S0OkmZfpRuI(+o{5Mt!S=Z))?Y82ELYr`Zx=>J;mRpHjbc
^oKaf3$yIEx95)aIRhqDqYw~=PaU?Y*`vYz4B=8UHz`xmvHzgW7_%@GFMtqj7Or#Mz5K2hT(+u4gCcQtJ8{TJOt?p4oOxwSc{Z0~g;A?<AVrW`&Q$V85gh)o
Y#}SlE`CA%YwUN`<7+gcn@q#R!oFKqjTtK+UPibgo~J38EeBLoCsWi-1lt!L;YDCgPoK8oprP*8sjFc8*-FD>l7fv!jk8w@x5Vh5=KHd#>7<$*PFwu(BAym3
;@vdNGUF@PgH>z}K=0HSHn?S^99)GXb8<g=zV{!86Us9s>v|nS&NC>M>OVFVQGQ#XnaA-`fpj;vKkgiS*ycAslFTO#h~@~%s{#q7MtWVbTFUFIDUxM2%d6O)
xCKMZv#5O>fN9P64z!~bzKA;M!#@NTlp;TY$z2%h7ygBzRA|~T+Tjl}jDttVqLE8U4%2yfC((Ph8I5RwNcz%US%oM~{1cMoNas9k>_$TynX*U#!DE-_OhK7w
UshA{PYaQ!=*(mPB(g3G%hKsP9J~%SXG5;uPkE)Vq8+CD!;qt3yA9={)<-g5RUL)zNx5sAfPvvUrc12L9w4|)S;1%f#@6Tmz451bz3z#AipKl-)7P+W|7raJ
8ecCjNhy={p`N{oE}Kf7{(%e-;K|N?2S<T#_cLMyhHqkG9P080dTMJhWAm2omAZ%?43JwcCTuDV!VJ6YrEBL2{>dH#1bXS@2{k#qXWlqNN)8(U^7&hZPwq?N
(kQ}5y?^U|2o>-P>M}}^CK83gY2%cIVgFhN{2mViqO_(%<1F2`W~<4g6B`HE+={H~l9$~;YaTh5^^NAs5tJ;8yuBYtnBKgEve4>P-b?S5g-rU}+TaK|-37Z{
5&I16F2NP1YnmEN8`jrVBkI<>JPS)@trFpnh==r2B`l`&hXM||R}cwqeuz4(W-?=hW{`06`ge7fK4yRA!md^;k{f><bZW<r9+lu$y?>h_2Mw7%%~24P+T%mg
IJPcudk40u|KvpPg1dL`HJ|GH*-XT8O55CWD~Kso)1jYPqbdcgfcLCoG_Tmszs(lV`b8HgmW^-tO9bCAJMHKd&b+*n#2HBeWc7;^*V~G9YrQL^ClIv&d9DwG
Gfbv}7shQoZ;@{)78ZjDkXh3CH$0=!%OLM9(#S{oNY{I7{f0Y**6%DE9W`gg@n{hhJwpWR!naOz0nS~ozodB2DRdQx!ZOCxifIU*;c#HSOOy*9$m_WrtgJN|
Mw2ZdRakd}&q8Pdf|*!5<Q8~!cV3;BzmvBF2p)!IDmV^30`&2&L^Bo~a<GT#+Cvm#gbVpDAP+H-M{5AI9<7M+>>(>yz&>&rN5g=;(fC7kEvBK_n2Z)h_wALV
JW6|r{uTL)h4rj}xSlddUNo?5JLUjq-(dy%XSWRI3zHYMRz%mh-h2Ra)nfX+!wC8N#XHtM)5`&_zm!=QmES_YOX=|%E<diGRZ5r_(uH+08`>StQgsDeADPw(
8j3V$3RDz!dwoD2-8tM<r9&~<6mh7DOz#1+uja-u=%?P@;TX*P!JmB_g1_YeSMR*=c%qQ*K&n7LTqa%PhcP-cbDW^tdJQ)PMTo>GC;xHF^SlTRwCa_xy@3j}
Rt>U*?t2$Riki9qrGH?8n`Y+M8G<?mM0_}a;6W5~w}45hRi}00zw%SAmRMs@QN8<s>&EBaWM#+=!Z(G`Harut!NZZnvRKqfb_mMPx6nk?5Tr@p;OXS6yO0Ot
y6&sW$!<mhT_dd}ZUErn5(A!Poro^{Jw@&X>6UJd=`3nm;@4U&-lOy1hO4`8tbL!bZQr`gBJY&`MY%zXYMrC(qC=f2s+DRmw0&kTrks{_Y99zv7RPEjp~!<k
d$u;`S0$ABr!BOLjnoR0;Ni(+TDs(H6OV+99-bK@0QJw7o+6)j{FmqRH^h`X0<~C-33bPI%ybaS7lC@!k6Crem<@Q5jI^D97CpqQ`dfHmCZyp@#XDM)+-e))
65xoOV2oE{b1NZuuBILzysgX+>jI&j1T!|;K~6Q1`_@wqp>xo35TzRT>L<k<TI)*mbub*<;z0vM^38TnaUfQ&;0Dg%o)ooXM#-QcD(rI3lXJAhb&lh58h#)-
56v?vCJWl5KEw>01*Mrg2G_191qaTMS7Emvj>_cS=X5)X_)|gJcgGdPub-ssk-6a`I@sU!hpx|@xOeTmktLhz-@BLp7ed=7#UzM{%{n2rP>MK_fSx5;r$(`%
^RkfELZ=K5!kf2LK1RbTFK*>W+MIk2$l9mOiT?umC=C+X=jThMtqpoU6*G;YEA8cZXMAPV5oY{`mCW-KB(#3~|B`ou(89GT25geE4G~1dNyyuS?7)B&Hf+89
DmWwTH^qO^?wY!~9f^z34bgStH}~0pV*H9lO#W^Y$Eg;fphbY{jEzxFsgs+LEO$NLRe(bCRb|bdCB-2x%)pZ-_Q}2qt=8Bq2x$yZ2M%J=5}X0b>D4nyu2wc4
$W6E&w)KucPpbb8o=ss|BuG%gMAX-|&N!L8{Ndo{hpEw>1O!wS_petM;yi#<Nc#I(r|R`O*ok4aR&eBRUv`;p<<ZTqF3gdJnG#|AuqlVma>blwu*-M`p0sy;
Koqq?ubnL&maS)!<WNy1y76Dh^tPC0BU=cW-@EyAGdF$2j|6z0Q^f1otGafV?40082JR&9A5lk_6%ZDYB~d^Kn@ktRYZ%!3&q8+QO=|^PetDCG)N#$ncON0g
v<_s}Z<n>mexzEooIa)3XSLposuTZt%~6yoeH6FesaaOKbB$cv*RsbP4TKbQd)$D<`#Ohk#$zdG;<<oCI%t(8X~vi+!;Ra%h&kuy90Ox3qY-=Iq+Evao|Am_
Fcnb$G9Sn|2PS$d=*hEblrc&b!CD;-JwSq{>Bk#{LM8xt`SL~<>t+gx&_g7s4X@8P3z#0K7AjmlRaH}M46jXPxa-ko|M+5azN2HrvSp;HtCA~Qs*-=OiTL{Z
FN;^4obyufZ}>!>F%~yD641jY=!y_E3=SQ|7U--{fwH}bDa=KqbP-j@Bd8%W&ma`;;z&|5KA&#*Xdzgm$;DC708NPQ%*u+F4S$R~;8@{-artaSQg?n|;Hm$y
L55kWF83e!uQvO^3qspoQ8eoLl&`I^`rJT**xnKfm9}a*_E;-}GLVuh=V03kpj7|C;#CyAGGALl*|R2V6C_>c11%Wnfps06mROo5pdM+|a3~UruM(ojG!_$y
5UIT5aE7{l^Jcg2)Zu-A@>@EyMZGfZO^lER6LD&4dbidw&nL4Z&9Ee&%88Vj@yGzl^FT0CLS^btw|E|NcIWm(Z{T892#U)hKEPEj98rmuN;^$1V(@XL;?yp5
?asW$MLEvA>xQv%G*diYxuTU?<A5ED$;bc6Ypd`?srw(VS59MBL}p3W(ypu=zezuu1g;W-0eQkbGQzBcyLSL42B*@;HZ)uL4Qr03zwehgF=YJcw3}2I(%lg1
AEO88Wo9fF5HVk&L3;SxJtn<0QT`4TdQFJdq3;8y4ngYKd}Aq_Rn#_zl@s=e8PQrPLdJ~f$W5xtNY=gurK;vwK8OzIAPls=y#b@0jV)`P1y1r@xwwJRW4n2e
c@y$q>y-5s7oWXf{U34tRh!jS#P*gq+}yOebG5h=9byDybE5HJ(0iOuL7~V%X+bg+d1&Vru2j`-GPzlNoukkCvYGqt_jbA{OIj^$9A-`kh%9WA3a+e6ukPj<
ZbcZ=3?q27M-f0V>F}kEtpt8$pwyg@AvU=(6yp%ikKGIY>~R?Fg*jo>AgBlC5yEHxD<O86!0FxzV4pEni#O*r9vK!U%Vic3V(gM@3@eAFmKZrg*az=pSyhpJ
S(7s0L4Yx&_(O){dyIWj_BX^%X$k0cR(wLss_Hw4q3sr!R@K0;C8H_9UjdEzoHi%yxQu2WlgO|+#|Q+&1DTrCA$snKZpk$y4GT#7L`gHmlg-Jf9fAdeTcPTy
KX!O7BIoak-@Qzhm`a6SHPili9#^T^lpC7B%qLU=Rwv85V~@xxA7t&AE!dcLXJrN7=wB|2d=OTTqRtSxUL+z2{ZEZWAdSyPI2?RM*GCfPM&_D6B3RDMlEq?5
NO(ilJa<cP#U^fe>L!xuh{lvRmVrAI3WUWvb+T}4)>(wsu-sZDSFc9E0_TF%cf)*n6E|+oDm4lZCINgNF_QqDCOFLnYUEy?uhzebyuIHN9qDe%V0R4?QeS~)
0b^qIokKV_rs?Mr$DDb6Q%6@29ad3vxmAu_S>;=2DC60mY*gLdNgnJY`b6D+Jfm>T{C>qlII8+EE4p^JcBFYu3zqlze5?{QhILBvZp;Tn-cjpGY@P0a@Hul2
no1nXfn`@ra|*f$k0k^E>GPv7+0bqk4|O-+USq1TS?;G^@d&jaFCuk^-s0u_;+(#-57P};xW!<DUriU9f&`fL&nG}5=lNLj_!AU~9FG+cHpUWAOIZU0|K;7s
uaK`(=<6uj2vC)b#3#_T_gPI7DJv1xqRw>r&O5O`IxVlQjVbZjQg&=8k>sDQQu)`w)%c2fWbToAK$M$U&_?-=Er{E?cXDFaj1k{1T_q9lRGr?@#yRHz@MBza
_84W-!Rx(0Y`3K!Y^Y`Xrd?3~Y>9d)3<#PFR?OY-XEuy?PcN@A*FY|Mz^E;%cxDJ!Rz4oNYi)pzTO<$ak%Gxjo3@m*#h_2EIfgKg)%*SVj-Vpx<!@}PhpOLM
QdH1`lR&BVPs1z^s;H&ldN|U9xmNeHxF~DYE;02bJ^4WRQ)`x{+Rc2n)Slu*+oS<2!C%Ty=?v1&L!)lIg_Dy|RXZRp6@N3W(vL~Fi37wmAzUcqk+0atY>DWu
a)^FXz(u!xHVimWMRWF#+>6-_p~p^x8{qMd1KlC$*01ltWnp?(e28iXfh_uq)2E)O_H^I?)`#4j7LI8v!lp9Xp}8YGk2fh@`%WEEw-3p{jbizR(_61Q&r56M
nkHW^GaOx2cKM%QwxP`apFZW)z2X&?t6tn$$koI@&HouccTP$2++ob2D_k9>&H-GfN_s6pDph6h^6u(Ig3926M@CDS4Sy*R1&(7yHB`A^FQKd-E-kuZ^1THL
LICv}LOyJufHRJ-1D^Cce1VN08nI427|OiERVr!#aUwW}`|DJhwDJjocH^C8u5~}w7)-a_ooGTZE6D?Kluc^-?G$hq>G;}XY|y4i%oP&f^yM<^=)Wk^rLw7J
m%2Mvdz&U@QfL-fGBk=lCzJQT3|@9MVMp}nJp_`=h0Dbyzm7|&Ua0uR3A11$HTj}waG{cG)dIrmOVM$)uS3!JR~_{ybcmTt=LkM83=cdS)8&qs7F@#~?-VGN
&VKh;zr1L`%hp{B2Dk;nXlMUAjTO;L_eh1=J1rIyY)I?|g!N%^wmoM!<;M?c#uH*PL-00@>Mt8R&2`M=t5zYa6aa>N?&DO%GL`8P1|g}@`qi+fZSkWQOyag9
$q9_`&eeKYKT419-I7wXQwbn?w-ST!QEZ7XhZev8E>0ja{1@Y$i~#{*l*e@Mg(*>-sU|g`G@!8tuJglziAg}^53C~O``tcNmR+fs`&LjA4;naD?R9@O{13b&
4U46jY)60PG6x_4b7Pnb+JVH!o)#Z5omlcCx(|x_9?sLb%JJ^ESsaU-Np>I=IVZU-Tu@yjx{|%<h~27BFB*f)Y7v(?n(?;9YXG$N&hSrcTL#tm+_!{9L7%Y>
8gh&^A@;H~DYc%+0keLhAlxf`8qnaT{uY#Z49AG#VsnI^n5@nbed4#`eIIauSBb`Y6-LvFUFO>7z$zcc_!?auX9Ch-rL>OTBWiU`(oH>H1E#B|2{;`G_XOfx
CV<*rVP^*$+O6<?4^W5v=jf~l9JTPvp0u$m3CLi?voX?sjRp;e&ybywapF<P{vMqYo)|e)2!uGsBiOFUZ$jX%0`IVlc@b1mbNr1b#w2^o*Q>dk9ri!0(*5=J
rNdXd8{g~^bR4r)%LQI&9^(Isg>UmBW;I@)Jz$X<5nrjvrzgo87t%OH_rd0Gg2!wC=;oM9{h<WR&otVO%4Lfyeji(fx7qt&U|vLY{gvc=4~4N`CHnoAg25ai
G-(OLD$zYvB_C&^ikII2^1u*fV@T`UuTK5gHot0JJ?2|vG)VCV%Sz-HnCyvdINddhE-fEH!n1>-lXO6{k6qs_$j@mNe1X@^=5&otwiCe8lkY~(QK|MZuKP^b
Mp*mMx4~15gtoYh+x5ej+6;Q#3ArE&(#A9>2xO*U-IS^?i@phsE2SRcs+o@S{%hGP^mYjO-4(9ESwP30qkgfa5b6&p@;Pe*?e)t~T*a~|&}sR7M*cKwIe1iX
DKY?^h+_d_cDQi|>UAVO#?`~}A4*LTqCuyCH4tk4$Za<?I-x^i#h~55YGO5rOIdmCND*g>iY>{8&~x&lfLhOqk5<8GaHG4DQ5P@`1b<k<B)7L}Q7b5MyPJ|C
d#7j|U{2VUCXI*TM<%z$UdvP&<%QaGa)sLT8SDl0N7eV}_mZF%2Ztk;2yDN7-eQZ5jsjy)v@=u!{ls6<lK3t)_?+W`b;N!~eh*Ox*6IttiQ9u#YmCF39%(sl
p9QaXM3EYx_U~xatL|cktPbYv;thFw2tnUH!?df>J=wR5e)mtDTe(h;ENQpdSYsv4%3-P&bMq{pR@7B_PHa%-;-wgtO;UwTC0YPfN!fH;9=@kJ#LGrH==e)>
ZGjTZJ5FnbumH4L8a}r3y7F?B&%Q*^C_+^p^;6=8J6BcM@qnKj1p0M*X}8)lszDg;c4+gm_+MtG6S+>S%1NfVKoST;903~)-q5;|FSs(+rT|HROEY~z*mgAA
dhvJY8nu5WHWk^>$X*3c1?ht!s%Q1wM#^d(iA7#~<6EwON(~n>bH6c%gqn`|J{n&h*MwcT>KMteD6?+oCXdw*jVUX(z^v3-kN_zkl|=93Lt!`<<03K*fw@AQ
IJCP9LI-QEG(>#+MW7cM6HEpr3E(MfVZQTmuq7J!Z9D-{1IuX>rN4j}u`q^<rY#)6)bRZigj!?nT<`Nnc`3B6GzJm~TpgMRGo7z<NNJ%>)iiZ3CUkCN%jwm1
%~vz9x^|d&%Yv?9>1uXRciqL=K=r_PqHcdu>MRzbCao2yc_6VsB&QWG!H}Z<_X{^F&}tgPzV1re!&KsjD_jT(JZoclBM~mFNhtA6h9;@6TT6^B*Oyn!u-1JI
9n?!~i7282F#k*nfZG#NY6nF&9#9;1DHQum9AZ&kkO4{=VqgaDSzze<kbju%D}>8rIjJl?Lhi+FA9nCpsLL1l$GTiPB1q0<D<rtR@g+?lNL83yweLM-OQ-BX
Yhq$E>qe#wGtwJ_V_>s$+gxkXfD^R^jF!D6hoW5<l@qSq->t(Vp?hXz;zHxqhC%K3T}p}};1d9zvWh~eKr||PJ*rgU2O*8vZy7CTZ?z}adRdHB9##8$yPy@e
O8vB7KOIDlnV_|C-Jnjb&%9S0s8$a4VebMZmQJ)lWTa3l(1y*2V&d60ReeXQ6A{rDKznn6R3Lf`z;^EY@>4I_1p`GfHoZ_8JNJQ8F!#W``)G^jCBK!>yzA<Y
zu{T*mU$u_Ilfn@Yrb0_xFg(U;%~20{rmQxQp=(Hx#aiiX11ANTF+@Ec0<DrqEoaOFY2VwuH1Ef?Dn0v<pd;_STAr*ZQA;ePOwEo|L?6NogUHJVD*r|P*mh+
IbQEb3-e?+lhg>%()Hznt^d6|GmmpkqDo_`-mY#W6)zeB#l_^+*}r!5^Do$b8DT8H*~o9{834L=E%h`m=VT7`%jJXa8Ht6IiP1HW;{V;pUP5IFSM*eiYSt8?
?%%in;>$~PJ|Xd8t<q$(hR@Zm<+AmR;1A9@s!KWxw|;BM;Y~S3h?@mOw*6UFe}eUH_P$YGf3~_JC#t#|_u|V^UuxXNSSkrBDTW>6+Vob-Tf@wrG}T693a%5*
5sv7yPp&Khot(Cgfbq(=bVBXbed2U+Rjx#tke=NZ$K0*E*JQM{4y&s123;bP=HkkXm{9{o>pzD1rl-}GdZW?apE=a4R>nQDecW#LX;v`S$f8mDJ1Ubc$2+>T
|8c%Rma#;hMJaq?!X3fKw<>A?{G!Q99TXs_hw6~Jw{2-fOO6`<a@FW^%2E6yu}jyBqYxarW&iOxoyGp+90~{d63i$SUuR4Y4rKD@@Ij`r@Ufhgr<5WTnnYeU
27;jg#k9o%y9_ovHGciW=uD<|_PhZ)vt*dHAi%=b_*7~Lbr_>LW3;`AP<6h<IZ96lrd+dEWNoWI;#pFN2hpH-d_5%rzOyXMr(1a)?z})mfal`QB>XU3i&swG
++oy3i|jdJa0*A{*a`AWt;`bIJxm0{t<TL{Hu!!1^%WD@ZxTRU`>>n{t2$Ej|60VSv0TkZ5$(g^uqBBRFeu^Me;;kigB`U;nY;2aGc%{HfvB$UuH!&k!*Nny
Z8j41=7*}6G0wxvW^KPgqgT8^5I4|u$o}L^0XU!2_UorrD{axEV-lo&>?f_M+3Wow*j(oyi91$81Al$U-Zk08wJ5ZU>Me9Z<<hj;0gw{OffCqipPVMuNp)G;
oP=3Jg<Zm=qggxZ&Y9=j^p$!?UeufEXcx$nBa@+Ou(tKCG>#Cx`jTo<+<CaDyxH6gNn$9jiw)gGMRSdXRzV)6*8bC0xUejbdTP@%?+;xQA{7@9a@nd313aPo
Cq(1{49GwbkesUak4S^%&T*J4APAC*nX5Tu0u|2UjiEG=4W?K7uYF76y<HmQY%ndW;enLzc#E-Gb{fi}GsU$y@KktRk*WAjV3kN7uaXC64bm<a$$>%J|M(1y
rnlQ9r{U)#r9=Ln%_C#b`@Vqy8!is}nrUp+i*t<<<kEi7OZ%svXi4|=4hy#oxN}yO+5j<MSHW)kgg99NG3~_)u%O@Adm3_Ue^ZRc%~_(tL`fmVa$X~>$7LsY
5dfb>W4lu3F9EAP_+nSHa$e>;V{dx;g#Ee^_%B7(x;Vv%8|`QKs0q`pT)1y~J*Gl3dpDM~tFpl*{CG79fSdR=(yi7_9P0`&X=sMBeouW_RR-$jpiSdMrl;;U
y0q&Wn6kY3@M2a)*9pXEN;0`OwhiI=CJ{KRfjyM>&iS4}{}3cFA_cTLLS-lKT8MC+PurDSKmFU-)viAD$?)oTyW8HI0f{eI3Yh?%iJ|Doq)ulND96gky|o~b
XbWZkOm@x5m8*Hs+$1g3MO4ddJn~CzQz-X?b&-_W=(bP4*HT^3RtVx%qKWc!momKbcT~GF-b-m3yxCcZT!~R!-rZn64orqJC4`}2an~n~m?sR|yMCt=Tnbd{
MAsg%0}_;JV^ygM*!F{Pr9Tb5U*^}v5YWr=mh04ELUb35lV=?iD&W01MDZu4h{yA<<LrAz){wO{EwI6==@I|Z#Z1|tRZvmnrbh#*s^~W%Q<e%;X+`a%jC;(q
uNTHk(^42{H&r`=o3$9$;(n!SIn$CsnTv7=b_{h#B85I^7nP0_W)QTCIzzH~T_R)mDyqCPX)Dqormtn3I9EPnl=g9J;i8sDaCK`vwp&3SQw?)C%_<V6q#QNb
6F+8l7o&B+!3fDfP&bU@8-u?(&bA^tOWUMwdLjEmB0Mjm{Q|!X8Qr^z5`Ckn>W^x`T%s7NOH<Q5M(JciANt6NQV|jUgq~8PpyOCkS&e1F-Mk*vH2%M#(Lxjb
Y?=e2bEb4RlnXBN|B%^J!cEg>(_eVTP(rol*A><}Q+dBUHp?Geh2X7}U(>*0d{%J5jE5aW^*8Jct@EgA$a2FiNs)Issf(Sl{;pLC(c#0C<6~crx61~Y8wfrX
2-t64<xx3P_?bA>p|AM{1e3G{{iX|~g=ZSziPhhf|FSJHH3<QjqETmpmQ~6apHU+QbbV+%>$S=*9hx;1eVWPWTmRcCcDicS_Re?0fUBs*v#Bm!Pm)j2^NW=j
0C1fXGq@9G_VYd8q5kul8G-vjb&*f&58S^Kl~x*ki=6~%a4nICEEfxbTQOb?M+G^((gEr-3rLjRuw@zyp2~&g@3wSsZ>?xl7pqMOeakTH4(L-i7JbLEEMl7%
VzCazTOR;Vwu~0taD>amV{RrqiUs`XaJa`})92NEm&wp@*|l$lZF@f@YZn5$+WQGQIgW%6S{J6_<80$0YpoyqdtWRv%{n@a5B3mIIkQ!7v~|`#F}0_)+6XJ7
P#+<kVbx|`y3IV4p@%?;2T4|lFG3i(405frC=PW8O&0_wm;VDllZ+2G_IRwuN5(Vehm$qmMV>VRtVo(7FR&H6ip}jh*w^)MwAFiJU1*DcnB|F&(bT9qPl1oe
?r1Q;joR%u=%GnFv}p3N!4Ef@tE-5T(_=;i>4J1${DT5dm$0N1eVAi#AC*e!q_z-bEN&G;-EJ_cfoe&$(Zn{Tv!xbJ`uziT71YaT%xqX%5SWT=b?K52naWc}
v?v_Z{oD5V4^}?6^2!gTVPpZ|(m}_plXMG)F`4o-3R`n1XX;VsxapD)pH6My=?wm(Vxe>Wn!D?hT}4z-JfX54HE30^oRB7qeW}dTq@XRTr+MIs71|~#2$m!*
s0BRz>k|8L>CqhwI4va?lv-#RTFa*_rQuWl<n(xi&6XCt-z90I93~~{oL>_0KstgCad`<pysshEC&FL>5FPA->88^wL(YDR`lOBh3O%^O--JA@j59_*Ejq`k
ars2ww`Yy*6lsN?;-quGf6I_>IE6W>9_rv2NDiES1Tu$|eq30-W<5$EYllk$5RV|H%sh+_K(@Vie4AyMGMAZP)exgm4q)alU1Z6TD{4l5qV`3nD_E3*WP$M<
I8*qB_;XoYR-7qr*iDi`R9y80VbBq~J(Kn-Q%L*IKK9J->o8^Yh(#rI`82@<azeH0HESemv4@S42h!e=rD{PBXYx_X<&|mSSjts@&C`10-GPV<=Da=8bbSWu
;RY580m8)vh%MMne^-xt$Zsw2h6a|sm|MLtX2M@hF5H@iAgmW_l}%Sss=zcWbuaH6o`mhj{b`j(E2FYbiP?LSK+*0hR$FhHl&~M|b>|w^$P&%8^&pazF^+kF
j6%fiATbu^u3%`Q)>2vJ<Sy##Zx;$qLb6BYy(BEyB$hUC9D({(pk>CQrszwpFOchrjKz%0xPOz4GKrMGH45Q)Za`=+&;_y;A~1c_U{)a}$If)UE9&wXpr-o7
u?jrX;6@5ToV4ho)K%)iJ9#gcLO^tRl6q<60%=Nr{0%?w4{hVU=<roshP(3Ma@uf(hgSDa4t{9}zk8S)Y<3PN<}Phng`6$?R506=kNs;NvBo8~^Y+8F(6Uhe
nc;v>^Zr4!XUKe8@XPf2gr?rdJT6aVF4NX*m`QbPj~ddPgPylcBBf%CY8Na;@#CsQ&e><)=)MGHKw^mJ&at+=rx|JzTwa}RK0R*3|7P?ua3<#ij6b{(6tXKz
8#MM+#b5FI<?rQWEYmwcBumBaA)gItly53)Jey~?5OlgT`{I0(%q$EUe?cXpVSyGqNrPz2GbuQsH_J-#vf_3BH@X6A;?)h_i*&QSpBA-%!gKpO{10K#M&TPn
)u>S`GxproozYpkd@1#B*rnixJw_Nv;Q%!gEiS4G-N!j#>EjcL*Ms+z{k#x?uMIW<V~Bg;mw{5zQq#p6Pzi5)XV9+fvch+Zw^R9%)MHN*=O(Bdz@K3g1|iSD
=1O@mD-#E4a7hzQ+&;)rvElgQ66f&C4Ne;?vUMIHQ7XC|wP6UJrmqea?}P4xC9>}D_pr}&i8@!4e4asfMUt_Jy2h2_1k{+(FqligiPec5Ii1#XS6&n<6y;E#
UMN*yshU1%%v-92HN2F?`z-=XmXSd?@6Te!%+?2A4l>;LP3<hP^&A%Fb%WeboABI3OE~N+zM9nI=eK}vfgu?6&$=(Fz_N*^%sbdc&=XvnvS4Z3p+*-~0lY&Q
p!)#KAGMllARWxR6Oz!8SKB6&0667grT<`*kQjkU2ZrLLxzuQQYDF?JfZS$UiDuI){@4I9tkpL_Zvw%Yx!kOp5GF+AdiAGv1<bB6TmR_W&~BTPg3h<%0UseP
43DVCSy1Ds=1rMqgSi-N0_#jcY|T-;8pIsu;3y{<*z+Os^SYAFMQ`3_zbhSg;YT}F&p8>HqPr;^omB&neAwy@>v!oS=%>Fshz_oo8zvR&&&oUBHKhZ=Xr@16
!V;Y^-M#n@B;WTL!MDoc?qJLG2WdCl#yUlk%HA(y9-2>kW^>P((81{@iFB$pMfY^gh8E!QJ#A{z=%b*LseKxo1-k{Yp&^EK5Bk6tWqDFs;P*ut3M{?u!LZCJ
-x4O!E)#VMBn)))*-*e!5c&s)Qz03K%ub>%^)x!|buR`uQ2N~wU1GqG6^%GPpsYdxinOLQNEfJNfZzzhh8pLpD4lB&T(+0id*wG%NCMQcO|?B|`VNL#mi!R;
ENC?G%%#^J^e~YCzy}LAmR_)g!CW`H(xtS0DK(>@C{ZF2Br#1NwNf#`bo8q35Nj?YJyd3>InzY{qF1p}2l7Tsv{K1-X+InHDl(!;F-P%O!VgtGsDD$0gb;jB
rr-KPqeGLX7|tP<Sl|FOYY+Dq2stadZPy9-zQ(-U1S8#6rekjD>lgeq4R?mKG{jFn{0}UUBhM!im>*bGcW^Ey`(U0e*nhw&j!Dh?6dHVz{*|)`o;<4-(uSX8
59@vycb?#@_mlsArjy3(Dc#9)z%E11d^8}V1A_s2yEQua0c4^`($`gnWqPbTE_FeNv9GM*(KP8k)GQNRJT{8BcL<VLTjiX2y!ClTBxbPee?5R=Iint8F238O
Ev-^zl?{fUew4Z3_FBb?e!*bBgj5${{CDP>LgkSHvYA$Z&y#MiJu3xRRR&k{$|U=E4O5d!vFReZZG^frYi7f>rjxUrQbVv$bbLuH#Yj+Pp_&I);H8Wx#CalE
(>$XhK-*J565k~9doUY>TP-*}pZQMr*{xLA)1BThCg<aEO%q1lv#JQod31gsuYKEeGW4ydfX_>8iX8?O(l%Y=oU0#ikA^)+(BoAGag~<_*dfXgW5(v>V;7|J
%fZu3DxO~@#fF=B3ixQz`HgmN?gYj#>aXvq5IcabWMA)nT894`0fATU27+EYMs;&Io0X?myE4c2ZZfE&ATIo+^4z8zSo^g;sw@6%0Yi+?Cb14DWf>jK<0BDx
@HYXX_8jAZBpr7$U2yetaV{qq1|RHESu@Vl<OwsBaPlhSM`oKb@MSQmLF8Jrjj8F{ILTDWepL=lZ-Ynw`fHVGv4`W=sJwyVjLlqM&Rl)!<cRlX0V^rE3|X9U
n`yH>w8GdyXaM(Y!wXZgt(D>&JDMyO)SeHKC-SGWaV>KHDG6ob0JGl8sT*wDokbtW5Fh^~ocv0c(mN<bcu*JW22T6Lkx}-p4q5V`ndSJy%ar=Dzs<v!o6+}R
Xxa3}6i`aoQDbqBBjY+lus((Nr4r$V=U&GNKwRwb!dyoe1H0FJKTID3Y%ny3tcv)LvHW=<_*i-~z92~Q_@<1KJFw|=n8b(fN%i=T^OK3dYZr`|+HuE?kpM6s
oz4YE$T_gR!xM*tlWRi6?W<_^OK=l?@&)W7bC*fyFw@(RJp^55kt!-n@0;iP^kO@k6_9V>avHB*T-f6v<=p_M-k6A#>-j=KK<o~yB*Q<*rs@n#geDPMZ%=Q5
!FN0&=sp;Rq3wXI9*pLiz;T=8pK!>rZx0F(v>RDp>zx8mB$7BZ`w#iARu>e~tnGFN|2=m&1&g6UG5_YGO1#+)2NRp>6bJ}ICq7BaC}zT?%TwLX`1&!)w6be4
sJ0l;dPydmW|WPE>vNY9GCBJo{tWMWn<|X5wsQ{)i0<TZy>0s7|0P6j5W8)6Z!UcR`o&T;HatL#6x49c|N2josDIyl7Kwl)ml%7|!5u`pgseQg$LRQ*2flXB
@dIOf^ze{~pG|G*fg;2SG9ap|iH2chWAv>-AE?#<U;4-O%sx0l*qB!+nzS1j=Tp8h!j1Yk&(?H0>A-HY;u@)ECBYi0s2daEF3gJ+bk80zbG_=~PbDnagwGRi
OjF<5qO{v?T@epP8b>xJ#U3}2a4Q7cP)D4L;mmn`CT@r(5x=f^N<jPtI1oMC29MfGt;;NE;UXThgq5SV#uAFWzL=XIb~_wV5feXZ{e#O71P0?|yhrSs6<>4J
Kq4M$$xJ;6YkbRk7gI6&mHVnGuiNF*uH)_q(~!EPVU@$bn91*Ks(=@}Ps1YP3q36c#Zynm-wZm{J(d=1OJZ}DsC8#aD#IH}I1WV~FhQil)dQh*`;kWvL+JSR
`)Gtf1b)#V)M<W_+Xr$B!FU6$f9@ufPg}L8C>P~uovXQ6uBCzVi~MZ~wM^R8#QGO=ym5^(d=x~dTGz>9rI}mEU0{UxeIYHt{v-%_?KryG?3U>#NdhUDZ<U(6
oQQo5O*F+4Aj<^{_*ba=0;vRdD<dQuSG2y_@+X+*yU(pdVT-b$+F7n#p&IRxh`sNE+;(K0Q!tbeD3v;lTIK2+<yxV1)s+~AsSoS|r3?bk@Ox~JAZUxosyMGe
4iEu@R*)L4tS2=j9IE2;Qr7RU7`BP@!{HL@;EK}Gmt=J)ZDaDyx|VCTmxIbQXlmkMjRIY`X#CRBmonCv$aJN9mUS~jzBy;dRGtt$ce2&S53a~~ZKjZDzKn|D
=Ak}e_b9`7qh10FtZ1EuviRWV*R|SUk+7N_ebXd->GFqLB5hg}G69+DG1BX6r&+;>HB|L$2J^s>M&zah&e6C7*EVla3n<CfFo+x$VgfK<hVIu%tt6ehD|-3x
qzI{mUPGNC2gXJhcNI*Suq=JaH_&el1)R&Jo22PD(`J>rm|bgxkE3Xx)DTZ3i>d4F3X?I=lueb=kv`d-6^!r*gED)3&sVGC!I>qW9(ZRpd*|&>pfUGp;&a2Q
U@|!rQzT_8%f&UVj5q|rVnMv&GcNQ_ToO*SyS^xSf|m)k0rMq!o$+t|(|4La<i3{G#wY+oOrH0(ac~B#zqNn*EkNpTuDOGxQ^yi+q&(U_!IuOjMOXr|IUZPZ
SX{8nYU%FUrGtur(^18yW=8L!ntNWx8t;REG+8(+C6Cepx#x}(yF(#sYuq(T=(+s@$P~Af%&9(z5VUejT(*LrNl0*OK4a6nYl{~7loT;o1i3_29<J<|Iy<?d
8MvGT*RoB%nF>gB1|)iTn*7qm>A&*P<e?=b(9T6l<wss_ajy&ur&sA#S2A`7f<hUd15nht=f<3Ae5gh{#-CgLML7ZjcA1oB&guHScVQ`+F1~sqy4JH?)t`cA
G_N?38mS?lA`YFee+?tf5i%aW?=xc04yP{81pz)=5wx_K9mTICuCr&eaL2%NL-5z9@&ORSr*CU<!0du*vtnlTzj{Tr5gk3%{;0N=unSKM;8}#?U@eK<CkTg0
bM%2b!&kyl)tIP2qKmt50Xn1vK2eiKtUS7I2jDz8q#RHR-F5Uy>SRgXRdA9kZ%El(Cx!ygd0(kCAz!0q^FAhjr9x+eiY^ecieNm{wy#A7?n<;$HEt}lUd6|f
F`=YANOiff*L;b-zYazW>)-te;>2AeLculry&maKmh>xxBEr0HfpwoFuPc>ok^EB;xTBv>W)rsdfD+te&3I&~EAp4V#)mS*bu6faM7XvSL;x%7UbYwR-63cq
cL1)T+jI}8B~Xsib<M>Nd!|EHI4}efXR+3gv9fDKi%%nY(Q&mEfR1Akg#;SZ(*SbcF)U{&H!bc&ihx;o1t4B2?~*RsaCu9-tHNf}dIOL@2nw!?pk?anuSB%s
Pu-^_YLbj!foTw4z#79$@FSG0v3>XIrM4wH1@pl09cCGV2dYV=jr}ub$<d#v=upBKANm%aLEU1tX{**MJN%+jIt02ajJ?I(7Wx#L5OlTnL`&~2KibhqALRh_
ClnWl^W57leOSQ~4PD5gL@7S#uh-;4*_()D`g9*ZATh{PH%I+n&qravvso2~w*n|&3a}vmW~N~LZOTkaHlYt{$#gTXHA1{+VMF#q4#8oAsN<^xij%DJk`|?w
AP(c7h27~rrKZ+iWvO{L`UXlpmN%5x%|BpRLx4eJOPt(OB7{0A=X^iC)HdQPw%$QJo|khj9wq3`*ByxDLZ*m9CIo5?@5TrTS&F5raaa~~M~^cL(FYzw5}_;{
n6jo+v$ZM*{9gv9PW&o7Y<<aPb8Y>4TQWNUY@M0lz-6Wwsj=jR0PiEYivLHB?-|9Drxg;c0b+ywybk8-{7J9FuRTIn%s-#WVU#TKjWptY{-oiqr^Sr>)8q4@
a~h+B`0&&lFfK?n<2&xHp4p`3xUwueOr2a}k)DX|!#3XZ!9<B@l>p~h82^@})sB(0p0tXl&+DR#00rerG$*G6NWUjz*S0z%{a)bNC)~P}!t4gP+x(?YBI2am
S*|{B$tA>eWSXQ3oTV3DwZV8zjYTFN7X_>KnbxzZZ?cJej#c9?HN{?kr`M}m!$_}}3dB8J!1jk2Nc<qz?puq+TWvSlq?X8~ee*Tgrb7S6Cj4rSB!#jAlEdfx
q|$isDIF8_)gCDh@z<oR28i5TLH7Zj6od{=3cLNTQQ=?<T|HyoCrjD5&7>~kJ&}Ubp9j_O6#+~)QiDtd@fqq#+R8hGi@loo!{dw#SX_m8-z+9)0Q-knblQCH
_wrU2zV4kI%h+CMElC<IG(N#;#Uxzvn;7E4ztXXs;0Dc31+qN_$)H9Th2kf|K`e8AUQ7wTr>rRLz4K6uBn33?4YTV9GE5jAW^h#6Rl(j+AY1wk*?@b<JB=$v
Pjun>B)o0>n4AK7zC6hFSKjPT-q^bQ9n7CPIiX3KNwA--1PU>b++b7imv|jl4?$seek`g@c8G-qk~;>>{I=!0Jtd7}{6jq^H}(r~ZSl9_tE_>CPume(6P<h9
RsS2q1)6%zgWm;?TB)~QGeK7Bw=rxnxH)NanCjQfcig1u&a~XyW(A{Ha)=tM;QG-7QWaDImqYo&c1DCW0niC!>ow6~6{oq%Keulp+CbS#0AHHESI>Bc@7@WE
sh?MRwCtx5vyDa3USgN!DAz~bYOVCu?t=j}>XWa{8K=$)N%w9;a)T1O^XRB#<?N=3R|x8UFkZlMt=;EfPkPEq@a`&EPual^ONLVZq4N)_xEe7zZflt&82zf8
RSg}_F(MP%6>~lUuDvJMi4W;Jj9apBGA)TLf(%R2t?RNNbQ@~@ot!%djiV)J4ja>DM}nSa!jG+V=3TE9Fy^I?0Y%tvY-z~A6>^uOZjlWe3~B-%%q|BJ*{`z<
BnrtR5-}cEA4CO9io5Ai)wM0Oznp}eyus*(T!!YD8y1(IpT|_LiuY=PQgNo?_`k5S;~S#rkozW4^SL)~v=1WTwsJdITy1eQq{h~H;WeC7IX>2&doZLY6vzB~
Ox+&8jw$g0sNg?{76%bymn`FkJ#JN?R#0)eWkdA##YP>1`r^azA_7FLvHQF&)P;%eWdSX%HJK{U&nu}1_*#C<_}bqn9A&rj+|ZRkRUdAt?ZJslKoJS^8yhzv
6eSE@2)#`r$lY~vTdY{A%8G4M)+#S!u>>y1i)crw*OC4z_}E#9ROXB$dXz{5ozcp4*4dbb!#s#(Us>ySiY;#_gxYR45qQt*AH(JEGD$nuYhL5Ds0;Skk=USI
?(PvOZ^U>7&Ax(@Km;r`Ay^3FjQ1@kwbSbuyYZ&#vkBRWmYnBskZ+Iu4qf>?GoG?<vXruH%c+_7HdY2cZGZi!A30S=v?85Tj8z8>U(|DbJ|ce+3}qtp#Xj2w
v2F9R3h-VY&rCpbIjM*zTqs9y&R}x2rxzCTN9)W}Y7FY}%p&r0)SL=~>o2jy0(62UdlWSJU)aKTxi5_@ae7<dVp13NZm3Q@SjkVfkjmLG=!A`}29AU7%Sqyo
)Wq-hzujtbA-Xn}oYU81yQ&gSoXUDjg`DnfgjKYP^zZRvBRDe!QJMmi=r<(LHo(dO1_>z^kGQ-T?r-zKG(N|>Wpg<#R`CVX{lD#C_FO039+!;E&e*NSMd3CP
w7(H0;c1#@F2HzXTg_F}qDK~BX$1+^OyOk_eyS$tW=Qugsg7hh_?wMfcA+CMr$Te&j#pa^coW0}mMY(qJFk<V%YVkq2Y%8;)6=ix%`Xid0{4sL%xRM_FH(8o
-p0VQw~*qqUpt>oY2~^m8D!R&Qb|P+UP7OHDNK{z*$f5(ZYY2iXft*Q`<4p<q>1WgI*b+i`t-b}Z+$khTD_}HPOUVF;Sz6f@tE}J<G~&w9zuiUsnotpNM9iq
Ls_DZCP$I6pQiLiM5h3Mgnc`Xd12==t95<&rC9z0qW?`Q_|v~?M*kPVe(&KBq3q_Iu9dS4V+H*p!Bb0}Oxw8m>u(HuSt|QeCjCPABnbKRibspn=p=jE*xU>G
bPV7|$22~FdH46<9WcjD$yb;B%tqgY=#t2NaJ+s$XgQQ1(klZ9``)(bNp>+f_y3?+ZF)lskT)By)<fS|?*-Wq$#!xC72DGWt?nO--we#5y{_90t*v0XP3(1n
;EM60frbp{AC|T`uFjd|Vmo{v7>{40zLk89JosH7UlInIpC~l*&O6_4H3^USp%@*ImY`g>>GG&ognTq^x1_i-E6_5BUq%VcT^V)K#v(&%0TDB%ufWjPAx?5;
H?Ql{hs~ivtTnYO`|=lhUTW<jTAp^WMtwzDi<I~q?*5;{hkk372&O2FIVY*<cJnXDlZ*l##>-UitfGo><BqRVlY0l$R00J``<jZx&QW40sJ*!p6`92<r1~f%
eqs3`Uz1oDA332k?S9RU(hZe6#VxG@DeMq15r45Z#Y%LH38<f3_n0%?waO<{WNR0e(`BFnbd?1K2E>~y(ug<lqp5qy_HQ`HNES==-nE>|i-X?W27S66C4DzW
7EBodSUSk8VaxgnomM80wz2$d-G>$&Za?6)@!Z1YA<~Uc<*m1z&jCHj@&;6;g4(`uNc)24nQhGo(Z{7!i=Hvlaw()ol9?^c+*`xes!O#M{2z;dWss3<BB*aw
XpBQ7(j0fW5?}V5#-tfU_Ly3|vTPca;WMUAW)f+EtWV>2tQLkiGOl$<QuYY}j1bt?rI1|tU#gWIV<M0pNz_CwJp*93=IT4e57a#ndMh*Aa8(AITPnRqTZ{98
-cJa5D_YW-;TDU=-+ih28%e{Ta$r-uZ8f(B;x?sN_xoITmgB`Wm|>R$<!e0!!Mo8sp;9mdvkI~AG}^wXy)7O;wrUe@e{7qS%IE&bG+6{#MkI1W=ckeo&=-$Z
^CyA{CZv}-3N~ok5B8q#pgkO9OCjm#_O-H}EbTq_X)oC)9^D4@fEdR&DdgrR7|6jqGzjSiQ9lZ62LYZ_D{S^lLxZ4;6<XhuL~|ngD-UWqw+>xOWpk!Dj|e-d
vT_r1KTdO*9vX=gAv$i!H8dPp>}HDYw@Rc$ogK>yna4WheW&L~B4U{I0ugWMN*(&ddn{vu=tw|k0hXOl$!~=Cm>zu%n8N&*nYUJ#ylQj-j7!GGEYnB@dP~`b
JxEQ{`qU$jpb6U>t$CB6BOEXGasiW&;f|O(h~|Tok3Z9W@SRIN?F+Zl6J7ldz;kLAXtDIhE#`q)QBwFds!5ANACFfe?o`X4{-JKb;I00(<)y0^qwID{;+onN
pC2eqnIzAbQxFq?L3wM@+O+Z7tD3cZ(|-{rNT}gffi?WI=6Xdsp^SsI<smXBtDR4+xC<iVhT6eRM#Rbu0(8n(j~WvUvZC~t-h{Ds87<)Dpa5kjzaWswk!gH1
2qP-wyU=ncKA&{G2+<_DNM99L(&l$~BrY@1F&r{Ds^DMMm54SCE!=URTp?>;f}fgqrd`hi4gp106+NuQ)#dWn87cJ|8h{_DV9Q?Q!P{w?_6vs_qWs3mQr{d+
6xD8w!K~C5%rH8|9bopg)$qtva_TeCCdMW)MVkqa<8A_y<hkYhND`=~GMhWW3g+r1<I$x=p&l1(X9OHGq_+(0JBAFBMuZ=(5YXwyDi#Tw_(B<KsX=}3cxEHL
;?H{wYlOavCQNR`xr)f53a?@AgH8yd_KqCBkO<CKGmh1MB4dlXu5@R@mV~wb@5#-MG(`jGC=>xCmCwskPNr9bTDHf_BivBE65t+w<sH%Gyzs=;L8w$07P%@)
+q-^LIRhZ2*->N)V(H{qvHVvQVxh!ytU9-hhS2o?{jK#cjX-C3y3{yhu_z+CKy+5D5R$XHC>;PN`isVRjvlRc67UPApESkbZC?z}Zi&{0XoBZ?+9Oz<_Y1$3
8C#|$h3t#a4%Wzf4woljOvv?019VzgrEANF=Ax}(!X-bJ&%eJaioJjTxzm5B-*s&cjh<oYsGIs8d6(mY;v+sGZzZA3DuKTi@?v_-yrMZ(nq+s*9Z{;qi~}XZ
CdPl18;&o$HgbRa`W(VmZZtV@f&lVMfc4|~2}sWQ2p`wG`09GwHz^K(yFw_Q0+>vP4*C^ZFdaL2QadQR(iW_s=7Q*huPB!phijY~yUZvGu4+(YqHij^LhBeL
CFEFW`=gL!A+1MgM2v4Ce4%ZeXp%TX)6PEQ_|4;@3XQlmktM}EBEsZFvV-Z^l>1yVK$2_s5bzF0;?;AtP)#<fd10zDtni6IN?kSPkF{*<L`93TBNm@A{G414
Xqb*59r7|VB7IO%QG4WjTc>78-#M&M>uA}#tWT3G6C;sMH@B)Ws?{GYWR%{hFmBwl6D|35>;dR&2m~^xwT`~{|LU690hwjzJXl*T?$GA&j;l^DiQT8l9{Xu`
1h2sKpBOd-@%@mz_CR(BZRRgI`~~lz1YMR`W(DwkD1oS6YLvFrek&#<G!s!^{j{-v$I6Pz1DS<yipwGIZUt|oM_1!U8=#{Nsr>0lhSiv@)vC&#BXGeSn>yXl
`Aes8JG6qcE68W4Z83zFo8DbJT4^*iod7V&Zz&T`9?*+2$!TpTZ_Py1SwnUN{pbmozmi`WEgo;Xk%R6t92xe@pDl?efNGzrw00w@_l*d~__+dqa+iV0-i^7P
x;Pj(qQ+c<5J3YLr!K&r2#OZ;Lx?JD=+eA1NNx);cBN1ln&30~v6$@y3#6@@fH+M~Q+~27o>>84PiEWnDsO7G<jZ}TOcAw2e+5I)>h)-#w$rZkxW2qahnKV&
zj>7`jj9`tI;fNxWO+TL4LS4_@W@mX(RaXbm2@|r{G)AE7@s*U$EkKAmG)_f&%?TCYZ5Oo)kE0iNfNbpWv|{VxJgBZAGxZ2zcYDsFd%sx9)`vWKSoHeICQD=
4Dp2a)d}&;`y~yN2~JGhaO76Wq2>sS4NKb7sNTb=d#5YUa(2lQMW9g$=k(O(A;$N9%cNQp>kCGQhA;diMr)!(LOP;NVPsUc_fZ5DgJp)WJa_CV1r?&D4_QgW
S?;shZRzR1facl(WKwBQ*a;uB>`kC+q-Z<)q@{OdRXZr$%9})p8^7{l)c&<$R;|o1)u8p0$lhTl;lnI#gDfvnnf%*S(b{oi;!t&OM7QX+iW({OFpb@|dOqG&
BiQQ!!ZXzLVRho%_T6`KJ1U@OiS`)A4M;Ht$=PhKseqK`@y<sC5xVlC#v}{(wmdB51d~KJYFWF1PY-p+>+_hy6qk2WRp1G_q{7Y?b3V<dvaM)R^kZc+rJ^5e
istY9!g|E94D5PEI|VuP|5;v8a3h_}VyhbIp;3skapOL;UHp%EF)l)73K1==Jyag%)3<Nm|2lHORM;e#LA91+Hi`Ubbo{Xa+B6c<O<6s(#jp~&ghNRyRGcZ0
*R7gDASN5oFCEc}@$oB{h}#4gFyCyWvFl(^#8l#PXK5^?W|c#91MSzKnr>hzQdt5VAz3qu2Oc<AP7|+F99;(;VBiD(XDFt5j)H)aVP`GjdCdZxV{FKWmFRlu
8M#hy=H8!=5GV@OG`^xhWq0z5Od?rTxKgJW<6M1c40b~<HS-&kJghmM(#fU0MVW|w;a{_$?T9kie88S<46tDyKfgMQEj26IQ)LnE1PNg)lHyrzpM3krZZ#;a
*=6O?Ln&b21$2IGEY<g9flrZ)r&EI_3F1EGUs(rDUq8Ud2uZ(>pTGgfBB4A_oh1Rr%37a?Z5~w^W5gTHDAWLV5ykhybzwYR+=qXFRjL#NFTH>Vd2}exT-PT&
btv=L9#smP^vyeR#q{}YL}ZA@&#DyEwN0x}Sv)_MHJ*<+Nlx_v-4J!+&}8iR4r*;Q4-WQL^fBGJ>aWK_&3Tvw?$y4(D(@zMV1U+$7yC_9ajswu<p)rB9ViA{
{gtt#QC<%)CR5xbXC0LbJ1o_)E>Z*GL_B{^&M8B;1)HBDKos!6aOK?QePiod_75(SnB+CT8kgO*5+dTuPOmdNRO7QH{@-0BT~;+lANfow3)ur{;9XQ)Xclo>
+|!|%FijT@St%mon`@l8qvQXpp69P`MfgKQJ)4xCmG00W!8|n}P-AvnKH_EZm~VQTQE23T@u77UJnqM*@|-rTr4W$JO^b(_*z?%>LKTd*dF#EQ{&GR!mywAS
mW0QjxFv9?`q$B`%f_tF)`r0h3aDm~XJv(9r^fC?Z#BlnR5Eyt(u%3dsB}$_57QA1sCpf@W^H*}r5e?jZTjJCtq0&RWD^=5SgwOvYMoSXl}dvSj-L>O`rvyu
Irw13dr79Mc-(eqjksmP_u5HzvK2#98-4@%tujNV;$DKP!}?V2koEpD&O}7OWmD6A8Sm<6!1QI}CE9l_{JTX<G^piWVak+;qY1X$iD5;MB1z9WlYcQC|Cs+#
m$@DU0Cwl_XiA|yi|NHvu&Ww(5?}<f7;W(?=*}>)7w~a^HQ7L{)Sj!Fil|_qUtDDaJg`^>oX%Il`+Pl#5*7OPpu;F*vA6!X=Oyf4pv8yxJmdvf!Zq8+41Xx+
$HBpR$CG<W@4LP+A>UL;^H=A@!OPd(nua5=RBlR(3PzH#Q_~|QzfMZ~&|`-+WSft2okz+=P7{!kz^iL;vZ5kE5>EVH>8S?}yf#tDH1eYdgYZM^5B;#s`5_up
lyhkwJFZFVwhA4{%%}<vj4AsNR0?5^ayq0|a>cPs@~hvv(&RFX)60bH=X1KsQuGdT3a-=Ow8Y$vQhC(qq<E5&B46UOK&I09K2{QVvO03Z&uN@^Wuk`gA@+HU
{!-xo!h$W1UQ;dgw4qB+_AhD7R*XCUuAuZuhbGKOw`Zo=sqlFPie3<-^=Od}lECLxU#JG>iWO9pc|}=MVi<7pj<+Lg2EBH|s%wo7Xu69*O6~BR4-40F`ef0+
!upBlkM(9sxjP?^l5pu0R@^U0DvR~Utr_@7D@aNM1pFQuPzSACE8z%WiaF9uUr|Ot3FT!*UXF>zVHPnNh-#d(COA@A%6pNl*2+m4<!kCk*Q3}XR}AChHW30)
c?ZUTYe}F_R9nP0B2EN2Rrq-K3o|p}4r7>vXQjPG@9aBmhqVJ9=Gf}LT`AD?V`pGW3^DJ2gBuRzIQib&<R$N$VNsZUPQV~N?>iET*koUIxKE6wQMx7mh355M
SCR*phhq7#5nYKdZ|6z*tPRjaq*=#I<3T<TVBPH`saEmJC-YQLrE>LH6Skfr(F1K*9pi$<>Ek2Ht3v`zvD4JeN07%pTdgb1eQY+9l|27^JR+x5PY@;A^_U*M
n(n=5%<<4YX?Z}~6B-*^QQ4<UAiAHn$@BSds|HKb@SMZ+bWP-BD?HkMp0r_4*FX-1r1wAH&h^C-q(73ek%^yGs`S6iQ?m1>1!}pG$WRwc7;yaFhlV5G+;>PX
mmq{!$my8C(KwfDX`a^F?y~o-lq&CF{<L*$|97X_rj(?6?NHnATN0**1sjRJsy3l(G;@bAbTe5qd`r4Rtov8d+K|j2-mUGhWA5b^9X!1C%+=~*v{=kH3I6U;
jk=sa#wtm@PlB@wU4ts<c`{N}7MO$BRXiQa#mA$>oEmY6GEv02=||N1SavpTu<9Wj@2^o<3s8~Rb$@)@w`>WP(va+oIA3Z7fa2)|ZsG0t0lsk^0yiGbRs49I
fDb>d%fHe528lahu!skA7L#(Eo*o=HawLltvNaYm70L?rD``vD?9b$p_!@Rln&L117Rd|H_iXj~P2QPliuqe>*$awi0eci_Ae88gMLy(tVF8Q3v(ki<$YKrt
EXHjZAz?R^eGAbR*t=khjg2zxo^bDD18`A5fJTuw*@4wJJe61<U+`=L9no#MO(tdkg4Re!rP|G-8wD|`&>Y;?|5z~i&itz^g<{nxb&`syjr-yShKPPi5pry_
iXx(YBo;7NV=-(@ugKjc+hpmr6AcTLuGd@9;Ws>662s>8G2J{F^WK&eJOA^6P<Xui3D*CI5T<irbVoE0!aP8`5(qP1A6!o*ds=yx`SEd4VZlVF{_kk$eFbo?
`UG(~y`%`Y5(+fy0ZVp0eD@91gej6xHqqZms9g%1t$`dKssm?^pMY&qk8}Gl0orQ6f(}bwe#mAv69aRU=O%yew)op}<?L*l_*-PXYNR$<2IQ+Vh#bQ%5=f8d
&-A>lA*YKqqRv1@ADc30sWTQU<nbmg%zd;hK`#u%pGFSP%bBdT;F5(f#nV9(2?f6dE6F<K8T^hk$zH|pvHVb8L(!hy2=#>yE-g}3Cp<BU;cKqLTjUqxjDc4;
CWMF*lQ9WN?M8B>CSH$i3&w)KsWK8^1*2Ced?0*jD@>y{mw|Kze|N9ZCBs#ad1PCU`;@lyg>wnkcPG^vfEx9sKb%~3tS*4GyVm}`h>I`T?3XL$eT&#7o;2LA
%xzzVOIk?Bich3BsU`s5TML=KB9SY(fc7pmLx#>_@=c`Xu~0AErPN4c`pYc<c(e{qdmDAP&HYSYDw}Zi=t=uVMlt}RgoG_Q)l;0`ZCC-vU6|<1Vny1)tSjFS
iSl}Q*Tqogp2`Xj(REP<p1JnbX8WKYPV;=2_M;8}IAj<PbSv7CH+A`&M01?doEqH?hi=k0g|E$99dBP*(?7d}cK3_Ua}W8N3QFxmP;O&+&qnq(wyW8Xz?>KX
#mE4IjMOQQ=I|S~3t&2L_TalJ*rbM;^I`Mt1MPy|B%9e-==6---1c4q*;!;3PJTm=gR$v0l*RNOqn6=2+^JwgZ2~WaTxF(>w@s+YbELu1V_O{eY?>YRv8s7P
BfFMAzeTODHY9Il_e0hlZaEf(xlUbNI!_O9%kHU+&dv6rm9$f*Ehi!@r`ptz#BXGelBq9F`RXTD*10<}Xy~bij<Glw@~OSy-uMP05I}XZi*lyRyR4(MsJs!R
?u^QMq@;-FsmWp!{Kjp-X<4&b5RFiawd$OVrUvLX2=e7Xpy1=+>n+9e41l#SyTrZ9EsSfDG!->WcfxI0mO>6q@HO<=fUo(&_U@U>c%?}7ovjh;>BMa4+c@l4
UhpF1<B44rZzr?x{ty4QRW2EJbkN0uA!$ax1IKDgh*iUCo!No3rCINIV_cJ^q>sb3fW2h2zu{sDbeY_lgrvgDBLD;D{Pg5R+8E=ZsAA?d@UKxT2;KL45)c@q
mfHcQfVl(2Z^c{%Rji);Tbq1itt7i1s2q+1?xz)9(=TV5+Q|@!&aetx^-F}Wl7TAR5a-4Ij86Sdz=PPY1R$y~Ri!h8Y43UbR42pZa_k|S{V4}WWN)P<Ei7Kb
m7;9U1?O+BSdO0MmOnlU?Nf>wbr4U$=Udg%F-35HnS~y8;siM|B2jer&4OJ!uJA5By05^i%#5D-Vqx+y{peV0cer@XK$R8#-P^>ug2&+J$M<j~+er8PDwDKc
)~nt;Q#i}p@zwFU;YDQ@GR&_mB4je+l3RM$&X~5F$guf)P?5!~39k2OA)02rZUkb%nHDeR)G85;bG^-T`D`{OgXT4zY0c3?k&FFAf(__X@81x58lx~ofp3Fq
e_PhaGUE+8+y7+M=RYar<gC_Ay2u}<9-6p;6}_e0pLcM}^80dLV5ax|>%rSu|Iz<AgR^y|k<J^@HIMrWE8;BPmIB?`JA`FWw~%gL?bxgq$D=i{CTb@T#pXW<
R<D|kxdkD2S01{SytJP^4JmeVrzB+{2ul|6UN8SH)|xno;<JD8jb_FN><+=G(dB)xHe+6*8+<AUaL(-o{(Q2tVG9(D2YZ$gq1RPJDWTN_5>%ribZ`jIIEL9q
+UI^yam(BY^71>_Cb2AGQhc#fcwTd6cp!F)b@nY?9Th{a`LucWM~8XWW#iA78Z~8Gp~54vMA=%Vl(EE>`4d!rEx{=NHbP1qwkfs!uBscGiDeY<QMHV_Vy7G7
gA+s?<N28GAlmrA4wAlZT-sIo{BR}#MRAhqEHv<PGsk~|KJf4zz%jX(Y{BKtKS&_kTPGJ#gQV*#jL^RL<}*JhlXvax=P>ze@iW7+w)L8<yb|c$s0z=hh_SMn
o{^&dQ`FXso%lS%r5ya8Vn&PVD2qebi&n|Bhd_&lk-;JB`);8dL>7Eg4}E|wXbgS9`T!NRc2~>-#}D8O7XjK&Ohj#k=ZDmv>wceS_d$EBiyhvSi_Y%9K~(W|
sxmj_Ag#r_3K$kl9lgf%C&EOxD}fg8yeq4b%%nXoP&0FS5Arumzh4WZg{^)R@+}A+8X@~IDkE7&C1s^V;(V4X!(!xWUQIJcS922|M42R4KZKyB0=GT8<oK@G
=r#G24Bg=VKn%crOshIFd&vWCU=)t4SkdEV10D&tPkkM$gn+3uMJ>Gr$*{9<_L1POneLya?5A3%_nUV3^PFgt>z|g6%tFM!^gj|$y<G`kRxtOlcaxqzU}=z$
GOQ8Qf7Er7$Xs`0I$Y=&hC6)po1Cptcepdul0r>4@S~=|T4}2X1$YNkWSN9jswZ|*W2|Gto}e3Dc+Lnm`w4bK#5~)@a!nH3&&7n@VsAna_7q#KTMMiUI#&6o
CS4I&Hy;(=3Nz<X_K=J^!}b`Gn{7d>8Pq{wTf?5Us-7(FdmbBFvC+o)Qnzl{Sc~4DeTOaBIH!vm<`9b0dQ0}o0SqV%0Hi+kJZT~36s0&S)#-L}H%H>@e-TZ3
Cwg#|f#C@lyp4f+dT(p3!Vn*h9JALYr$dhKO~d0_*(%Yk`vT0^Htm$q8lXT{&x#RpOr_XnXI%j7n6EiD+tQ3$ZXrzOy{+~N1jo`g%}7X=&5L^&ne2qJDP0jg
G@BNIjv-FQlaA(b{HM+q3z;||p$<MmbmNeIb~R~vi`5fj&NQ-XeQ*U0%0qWlD)<@8ap9x-W=1g|N~*yW-4CyY7KeQRbTM6-*{KmL=aX1N#o10YClOZ;8d8{9
91N74!v<6u!_k(8r)5`o4Gm$b#U1>e3sRRN=*>a<ID8!SVgy!h0$+&oDBQ9J|JeIG@%D3Bv*u+x$?b1LDZs8kFrwAa;qygIfK#|e??z#6sBQhnd10U{|5DSH
o|6uy-SjI+ZOiC8rN)PCv>nE}@_^f`c~@Qc0<wr9s`ee&Li%*dZ=E(Akp0cZf?&D$#x0wPd(_Vv_Y?h%au?jO$&$PA)vv-H&c0;`@VhYtQ4c#d?!!}^i3PwN
8DTa=>C7xfKL=}@_bZfL%e|tmi*poq!RVE1#TYq#CvU4_kDzwPC!tJt#Q5IAccQuaAEj)AQp%<A?cpzIn+P&7Ynst?(bw(%ZLKJkIG6=15*&SFmV}u*9fo^2
Bo_U5_FWI#Q3WSYGxMlAM4{8GhBWpl=H-X2U5>n5lUL$c2<5R?zmpJFNPX0(z#jjpp$J1Y3AAZo1iu1pFhZ@WHgvDO%xNjLV?Qgi#p2jIDJi*O+X>C<Eiulc
L-~axg@Bp_R|UPx9Cd4R>A5z47BPcvEs;!1;KebvslvP4rc*m97j1?({Q9NnI?e;BM-EADifcp0>_*#1{ZQoU^Ew@v!<9R^LdIO6jrI9hPvS-Jz`w|TP~5oI
187oKXyr!GP{x)c*qbj-9Gy(NNk=NQ?x~}JjhyYw3w}QQ;YsqX0CR?QGbzE=Dgzx?ecA>LgwYgk?9*<+S^nHGIVAdl=(Z#Nq)Rh}h!HD`VqGQ}9|(|eA9@O<
b9dwbpj)mht>tDj%JAlC82$yt^N;nGwRhs3ibEtIgU!<CzQjE}$B`xHa;k{9Yo+s7DUk|GtVcRY+$v*nP4%G9pxyD~3OD;;4EhaZ*iq3C1z{2ACy^Wa)!XHX
hyx$zOS977%OUfn=KX@Hq&IO}*K6cYj3a%Di#=r;dn>+C5wGT-Nws=^@4e`q#PdZyfh@^O+XUmIN@_Yy+~YK+Z2)@`KDqx}m|pAA#9o+NDmY<&U`JJ1;uD@P
7g0{Tsj(Qzc^YP~*`OHIh)82EqpoTF9j=Q4rJh>!{hNI@)b6Zb#;Oad0S3~)k;s+prD~tcu4aor`SayQukV6yXBZBN&>{@F_vf9nVvugyhkpF{A`>-WHPr{X
QDPZ6_b$vOXlv%ALMIRfr?kr9YqZ?{NRj+NK9KZ3A}7zxCXBO-8F(DYPRovyMcDIb&Mej&HhIlY%j(OLXC}#Pp+d{i;k-(&%rIipLg-1?TDm%R4x8+w5Bv*F
Skw`X*x8EO#aFAdDk)QQc-G&Rv~bWVu5=S~IlNOt7fVm1A?DPz!zB-hta$2q$trTS%w1<_QTIlN#a~kNu6xSStw%>891hlv@U@869v7dSfD;InAR_Tg>hR9=
G7B(z)=270g6y1T(qv2$*yD$@?q+?SnWAtxJzEy4Cp-lUNbS^TGL-?L&g>n{wYf?Jkko1-Uj`hMxfoj0EQ1#0bo}q#k{g=G<HU%blhb6-WIx*#9{7qR8aDS>
!%EdSMBMmG6#(zleJx69Izk0$`4*Kb;8fw~@ZJz$X$V=rUI)y|>Id~9kq5nts~g3@5bHj8&le{DCH2Nh#IB@paM|yv9aYZ}I9`T_6NqaW#3iqU?1<U8+}g&F
G7}Vc5TZ*fsIaE}sWop9F(W89y#RXR6MBvj17#gafjk9Id{=<W*<O0sZ&zIWkuYAnWjuPOq`X6F&!BiVD<KKPlX@D~Oz!kE8C#<ku#k~yIvXoAJJ(cY&LkO3
sM~ItbjBMZoMf>hX(jwN4-)vq|HsbKjeY^l7;aJT(LOvkb61Y5e}(r-SVU43HwgJiOpsNeQaGf~NzI*CWKfYGNdrzX9a|aTm8Jk3nUlzyjg}9snYfKe3xfh0
tMO8EaXJyk*QmUZ8Hrp792l^{V(A@TZ8<lK7a1Xn?I2?LfoEdxMYlf39RUcwrerkgdNOX=i^c2bVZK7m4!jh{rUQIH-@}q@O-mNL9N?3g8!SwCcUL~Ossh&&
8(ZPa%C~w6RLo#>D6oJy?ytPTzCafJx3N+a6TQV*2}oy&E{OpfwyC}>>>_I6eDUVC*fy{?{c4}4aQC^8p>YPpn%}^2&i+$|NJS=9lYKW_UE_3eye_}+%&eoc
3@pNXT+WrQk|B+__nZtJeS=Q0gQg$Qk9E7dzB2Pw+XzR!ZiB&l!Amwm=?TxyyP1f|6F@(R(c>yD&-|MiTK{$PW14sM_l~nT4AcsIAK6zP9*>TF9?CLKyr*sl
`tP`B+Zr0hC~~0x#tBJ5{R_dWb~C=I(_imu!Dv$mRr3bG!A^33P=N1&9vJI?q%&EdX)b|qpUAFV0mSjLQg({d;cvz?@Hz6uBtnqkanH+S*dFWVqxRx)hJKdf
-)HT`5+4BiU1xYkGRdBlqgbx06U}Q?*8VcGSViGk{W4{0;E|ol0W%gHa#2H~dxdPrx^|u1WNveBsd#h4WUs>!=GOaKSqB4gM?ef|a6?%8{h{EQ3zf|kR?6D8
Q<r~q*;N3KPTVDm0kBWcr&;r#T_UzFhTw??9Jn%FiK7*YQ@wY1#C~cNX@TO>Y&ehq3qmwI<{&VXS;KIxX3FXd^Xy+4Y^xmP=&dhZvcYQmIP@|);_gO4*_tyv
rvb>1aU<(-R`rZX)}{&P&6P6QlPU;VwcZQ{5ZoRy1USJle4EG~JT^Aa#;CfBxVcA#O5olx!u5iWH`4dkDHKaP@8XXXhQA5$OS=f4yv3>JT$0(sx5!pS;TOwk
$8;ymzAJM$c&A^cwOCP7_+9b*9ZR6u1Wa(9UCs#-dL~>olmC;wgoln}kdLEu7qfRS;BXTTIpa!dTRaI!bf-xC=NjBJp+ldW8tC)~h9`Ifo-finx?D1(%GYfj
$kkbk%Ub4l2eC%kwTHK5CWs(SLeWNL-}S1WP2bO0BoN$HX%Tvk;vk9s%Y}Z(K`9*Dtl1Njbnm&tlSYT75d({l=-gS_Ryk{dCh|}?FHGGx4HUvn7>xRFz1SA9
DaVi@e|;v^wte4=dzGj6)gIQQq!tW{UpKqb1ZP#d-D9^KEX~2w#)dnEonSDa%x&ws36pdG9)*i_0b2P#kn~J*@@<PNU+*vqrb#0-1xS>`!AM)a!DC|J38oQl
r+8}O>94Q0@t}%R7oASIn`+lM4pb<#1Y#x8PUu#<tmx{XMH$)`)R4u~38lJ$ssUMwnHiw7anUD+djd<aGp7V5F=;I11yoA+jd!qUo{o|m6v4WXWi{RHO{%Y?
+GfwVh+uO+u04(vl1sV;#7$d3*Z-1_HVEtNnS4wCv>PMRexw&xF&n{!ff<NfdEkZ9!{J$92boKLDLhg>@=YulT(dD+`YFRjvTZyG$mm?dZ%=(nYrU=9mpD;K
=PxtO2IGkX>ko3SoIN>5r?r=UO$kbw%*@HAp@RsRl07B#)K{_Gl<;%mW}Ex#BsnlJP`f)9Y@tZTnU$*fzl}?{)THqfI~PrzFSH+}l>_8^<YFbAT3qm^9QuGT
%qEBWg`Vf(pX;1D424Xz_sPGi5sZK-+b)UNg8;v*yz_FE<SEe7W2gSsH5;BS8l4M}p^Q|K@X{d#o4EUW+dT#RO=BMv;;Eo<{i(ZzC5q|Xj8in)YNb@)za1xc
^+OSX=*+>K$uB($i)IgCFx<V?WCg_meyrYmyWfr2E$ceU<?(5=!ivqwV0+v8RE)r!IdecL+_I0tNNDS;&pH39N*-J;*ev{oI1y*NekpO#RXHMQL+tE6fOw<F
*8W2l;ucHqPtt_5PQYx#+Ub+d++dJ)n+pFlwyEn3m3Dlfd4s=$im;9771zv-VbOia5^POn+HWZ#mERe=SrP;Obe01c84ToKAOFm-p9AzIxZKYfe1@Lc9~^re
Cv7+0m_%^EqD!Tko&Z{fZPer2zVx6grjA-3S~e=sNHPFZl${TwmMHEq0RFer#3=dFK+b})_6Ye{s-AI}tD`qyzoiFi=hftj$;r0_x^mE<(guk^JSXEK77&*2
3Kds`$pf7lCIS~oy)!bc?V$K5m+_k=#MK42$WtLX_=)(xSrCzOW$YTbv}8WL{hb2#?kaNn{7y^FQVZhY8Eeo&GUbm~ItPoGJHGm;4K)+ga3p+ls<ki0YNtsA
i#%oNDHV<=1;4X-d6@h-CdFXA#=AL?`4CLifetYq7uN=+9(xK{&mQ1On`1eCh?h<bo1N!%RuQS8khIp_mS-yLPV!NohbtvkE5SZvqOkMSm(VC9kW<8W;37yQ
Vy=E>d~6#T2!5w}q~6Yemw&=ll*@VXu59lt`3>F=$hy+DdpW&`FI#!J4*FE4wcZPP<)#{UWnllcBwqp`U-zEwd7}327&Q3${S@*T>8)4r$5;zGuR*>t&I?l<
=NMf=NWNft2jI9|MCWn4KbnH{<J8Hf>#qfPh^(mWWRI33Bvd7v%;D!~jsfw-;kRz=FU()%9`KyKX@6@8!yNo}b8L8J+1B7|iWere*b5_k&(j!l`<nW#+T3ZW
?YiKotcVcp!st7<yfhIH0Ix}akZ&XfG5BhiVP(*pP4JD+%i{T8kkt&XC5HxSVWg&ZT?bpAH-tLWTjVRc$zC(TBoh`IgeD^+Zb%#Se-2OIs>`?qZvAx8Me3TC
<GDkzxCn?P2G7po%Ht%@*AN^j*&3)|n7;d5si?fLF#&<mvTH#_5$F9*EgkE)u+%w>f_yK(!gSI-k$95+fd25r#Uw~BQC@B@xa#=n|HA|ZxPr&PjhTn+xt~J(
AW8jbFKT{5#{-DT-}CVX0;z65?3Z<;N`WOO8OsuNMu>|e*$L7m%t@4|dqVk!fYc#Har$IgtsIk2$n1wCRp{u`rk%W*l02$8La}s&NPH9<H1%vN^llU%!4O?{
Qy&_zyVsO6-b}@+?l5IYP35G5Ue`lS<Q$Uh<8OxBu-@XsDh>V)ETP*LalFo@Zy~-ufE$QTlM;n$=M5B0%STl;V-CcmeV?g?NSp=H%UJ;9MRM-HjQKIHQg4SF
_t?u#pO4+4xXp)ZgYF=`xOd?v-uFL4#z8&Bg|H8OYDx?<R_wP)gVC{Z!lpzDS+=s0B{vU^ZP<>K+G7(Ah!dZzT_s0{C<#gZ#u2`}+;51g(y~y7Z2f|4T5}7-
ELZnyasC#n4I^#a@p=<$@Lwz|-}tLT=mD@**>Kw}v-!a&U>YC$6An&{xI%99{hM}h15sAm!`6k8+ve4!B+Ulf$KHRuF1I+wqin<kResx&a-GDzams-MJ2!5;
ftwoIoy5w)#sme8r|_9-O7WaNjI;29_+~V`PTYlAL^?iaZq@h(c8cM8B&&3u%Y*1$&lHL6)cQhDTG<AaoS%>Vl>+p}P(&cw{(77Qy0b){=Palt&4ZT!z_ke`
DT9!bH=iOP-Q?>Ka?w%X-E&CY23L?%2I>DA=s4<XC1<+^KC_)YjFZp38X8@6Ktn|*u_t*tC2ojbYR-+_rc|3>evJfO_pct*Qa*y&1t^3$ERx=@g-q8F1}W*1
(AQRK0HO3A=7@Q8m`nmuB`VE1f`%c`w{XtP@xIfm$YFe#h2{4lv~{xvJzkHgW$b*zaYuBms^aTTC)hwc@%^FwN8n!ahlC1#89RYy;3zwR#C%G~0FWwLQ74fg
%K=ZF{Yu%zlq2U+l}WUfz_PK!&iMHJa^h1Eo9qD#<AJKyzpLE^>}hVRNq$Bb<3nRfy0^)dd{~=)xX7F`(3raXkqftHby_ptHGa732Eqn6_vm$em$a}1RGcAz
drBy}YsPkX>Dawlhh7-`^d(xzlb_};T^gCc%iP(Dra~$x&h%F(eLS4MXw$o=m))-lcu3C_B@%4@wnGn2atEKq`dIrC`uFY&p$Ter(QPHOiV#OC=%$k84gH41
q>B*)7SeIZg77SS&pB4zjE?RZ9guZea!Ob%Q{2>ri)9pjFgj|tR|`i+9sNBzIl9c(BhFAMUGg8El4u{S6H|F8mqxO|Zi%RA4v)%(y?BUFt?zWU?X=KFwQ$VI
kB%mkyy%L9WQ_Or*<X^jyI9HJHG?S<CfEeAdZ6xHK@<~|^8(Xh4Zcts0R7db8)zr6S1^L>#I<|t+qxwgjS>l!E!#ytj3(J8ckf&%cm-}ypU3@M-&MzorMN=;
c!SwU1M+3m3De_;#DGW>AM~SvEO1i`p4Ur^9Y3e+x+h(7GbGKi2>2(mUSi#GaPXyoVBwii7TA=0WP#Iw=%*Yk1dx3jW@u-ReiQ2I$dlAMaNGUD{05yug3GEK
`;2h&<^Q|ZB&nZ_&<|?<GO`T%xt+oGD32N31LC_t2b`Z}6?DS2oD*$g8+=)HZuK{ZD*1c0Q{5*W<SxLn;T_0ql6CDr;9}$kd8il~(9h*>Z`QtbBQ4>+tSOxF
uRG@#@}`h#+(p>)LvC_!N=<2Pxc!l^<XR}5XNqsD;tuz|9YSUd7AefSFPxkVEsI?FbpR?$A58!+mVPp6HC|x>i6i=H0TGj8?KFW;v72av81!-@SKm(#Bd1zY
66?9Hu9;*)!j_<llT1l2BEBv~`1hWT+jEQ@=0EWKke+^MK<p10<fR~oQnR$Ba0oF)=PwFlO$@@S4U8w#bbjeQ^2%t`cpoyAwr_R2zXoTX*AdM8w0KKvCUJKB
L3Z_A;Lamz_J-Mf-`#UsbdwY*Xyg?XQfP&3>87!`cpt=eh7o5bX){iw5<gFshM;^i9^*-hysy(P6AtBy%#T^FH318&&nI-wV`kKdQSULvji?GFEvM3(6V|(0
{Q{rmR*Q<B$>y9BN-NkMOV&z0r~mbVDJFj@<Z>`)W`YyKz7-PaYQ%@R%R`}-N}E+plc+6-%)Smo@GYRZ)ZN($R1<HBCUOa5X94fI0-hfximKe?IT{K0mOqUu
oC|r2a)oLVYnemBw*8NXX@HCU{?-hyV+#loD#`U@5n`>?D@di59=t#fa4Zx8@cCmSYmR95Z}@=Fg!9;&Tnw`xMa%)b{`&U(VJ7|W2QhZ;;o_C+BcDaN<ES&+
mlf_H%Q*$3T?osy>$7tWvw`+~gu;L@H9r7yATKwmO$kiX?ur!BPzwW+GJGjS+ZhjI>5bQ_+>;{i;$MWf8UF4{bJ?0_8YE0X+E{=B_cqV(66*}h=DjWvZ}qa6
ERr*MsrD%kQCdo(*ooDRG227Ax5&BVFlwO{iJo*qXq)9=-#de&0$&Qz629XAJ9L@db(lyXA|HvJC$?X?4q?mbn{WQJUem@V{i)M#N8(ADcGC<KV92g>4)JT|
&lf{AA>_pqNmTsn9ep(S?|GZ3I*?^pc93bm@mKYRkSU3b+c=uzP|ua)o@X`mX)dSK$bFVJ;PxUypd-F2I{O~GVcF>y`X~wvO-<G)G{^2z<~p`~so=i6VJRuY
`#=A^`D-PK=P>f<UmI+nZoW26IeFhEud<?q24`pO0{<~FgV<D%zu(J=CZwmKw_1*h`$x~X09R0PwHe+qsC<cjcFKAAen3~^1Z-9(-Jy^q)+wF|d8<xQ40rpa
ARdoU*p+nk>js5;Y0q$i5tpF8Y5wbdz`)V0RS8=~q4f$i*bM;7i_AT?%ZQ{Sw`3SsJ{$NJy)@i@oyr&Dq66~m8lNPRVIW>G^Up=_aKB`-!CyO9Pg2=L%L{vR
c8CDKHzH1qA{?KPnSi+ctYA7!!kLF}uQEY_UZF}k@`*4H02#NfkZ$%z;*4Q*3wy7zRqB2yowCkr8ENgBbA&VOAG6J28FK%2efz4(>hW!fBtW<?>|234wr!XJ
!Y&#C?5yzH+W{l+XopHfgZy_P)Y$9aV&OtA0dt2u;l+w!<9eL-3!hi!L@<Nz*kctIC{$EmG(JG2dxIqVT~yKINfJ2{CkqOJ!(G$`yi7rCo@%W+)};pygaNL{
5hV&Xr|8F?n;Tby@V#24U0Kds#pej}Y9PQN{Tz+gRhYN-MF`4FDT#vhUI<#;uC5|R*^#Y6q2-OQ&ralipDQqka;W581COw;sNhHTEu)mC*G#Sf+$Y^wumIJ(
9#6CqeASS!4;s?4+mI|LjKTkA0p+VxqA<?*(6NLYyJ5EsSy^~QUAzCYGgn0}@&E=u#;%kr)mx{X{9x<?2dE~^<pwN92)$5Q89c#_|J;v08s(=WqUBSLclQ39
4z#ozz~ZD0Ydv(BIYIoiqJU#t(uWPau^*%Y&iHTMcT<mRs>A1ug}*1N*FOtPcg}0+lyBD*)ZpHQ1_$TDRW$gcOn8FP)NXD5)**|@0bsVVvf!wfa~-zVFO}NJ
34Oa{N0>-Y$bzKu8=9J2+zKDOX*Ozjq)l5*kNVG`P>f;)6LMP&8XQH9Nxx9tVMsoQ26nDbRb)xO2?eHpY=Yxy=O8~GXu}m`5d*+LtmSlaH=Kgn?w%6FwOz3G
khzI}NM7$>JB)r5a?$Tg>$Uz1zDG+8ZWRe0ueCQs2_CP^tUr~BzQ3`&5UC5(`^nC*x8tIdCuS6-oOh57r@u5U+ax1EslcdGf|^oRa+qn8@(zI=h~N~FpSRbm
Hk=xOCC^m_zKOEn-Hmq+b0>!;zDv}3kQBVXJ>7@*sCj{Ns!nelc-o7_%PlYPw=sGVe;_X}AP^oH?x@gSwz8xjWk3fRKI(M1PpAD>;4Sr+!NGhjSX4ofl{6_N
YA1Xr>&?~*J4bpVvg;G+)L0Kp@e;193ZhZ?^N0|wN8b}B61n!~TrVY!MQa?ht?E}v`eQKChY#-DhbPTivVLv)<kwXnM4ZY~sJvsSnPD1-VSgQ*?ke_Jp55aY
VBT4+3!g}|kslg{Nc9Lof~R^82L{GhESnUf=!Ce7tW>rux7-&`<!3aZF!U3OlFF<gM~xKFA4!#j47u-fxnA&>5K6Pjt;D(G#AqUMFWn9c54vnHRivM3-0TRU
j$WaQ`W!|a>E65{X5tIPU<cpCp2NE87IN-thw-OI4Nn<xNF!V*6^%55r%~}n^l!>f&IA0|IZzJK@acGHS)QqTsxF|n_US*wpUzmy^{ze@%7bF+#Gvu}9K6WR
qizz+0GeSRz_Guh9+UJlnd)3Z3)$J{*YADMMTvr&d54AyrQuKZkgBK#Vhz{n;L-X!d1^aX5&KQtToXoM?zcHT4h>Q7IUFa~D6Z~f1XfYbQfW(d3ZYF5{mS32
k$CMcocLKRX0zZ(399#PZc0x$*Cd-5b>5+Ny%jm|S*u;B`iXhtRx&<@M+@XwXtoV}?hEqZX3BN-8ad;O0c4%)^H;?SYDzIRA-zI^f^$l)U#_4ZA8$g|V2Qa>
3fv-Xpa-~ImZv1$1KTH^r_uytS-=yL$FS9ZDjqi1xMV0K;8)kq?a^F5KKML}?b;!h>^K@0j3Fs8Oy4E>`0p3XMYnFOHX3v^_iHI{x>WJ-^ChK?dvTuYlER|V
As_FM(@`p0N5J9@(xT(n5aH$Kgsvx*H44zFTa{SHNbJJ8Ab;&`L#mg)A)-D6Ig+?bVKaW}#ti89>`(cIkcW-cw+cWw@#EJZOWwLk0*XT+Vwg%r&qGTt`?!Nr
lIWN|3*Dw_jJeR1ueC+HZq`o!SJwo{JAbZ#UH3{HI`jaO)rgtSJ&zl)Dq=GdOs$jp`=CR^$=VFAQpWgHFV2zQESH18?+Y_)mK2MqxC~;om@1T-pBn=JMdx0e
OpTjAr>FS;)T(7Pj`tgWL+PwQeai{d1VG~rMklE9r^Ye7R9+|xfIysuDx#0f)jzU%mj|I@W5OHQp57h0s*fZrbA^MG_#HW*FPxNx;KF77)fZ%}w)-^#LnPVU
)}uB;BBx$#axCRD%)-^}syYJAv%8%o%vKHL$-+2y{;-^d_0Wa^GS$!cB%st5;%=F!7mipx!y)rVoWV~ta#&^n??7Vw@&K`wW5DTByGMDX`ZSd?x-bup(+_F_
*tF%^_P0RkG1PBN{KUFpYX9LfAhjBDD7$=@Sc`2>(63UxlVy4=0?Kv5b7w~>fctV;x?n1(?AfAQNJ8BY3>Dm}yhG&$54`s<HtX_a)4yu`&RN>B5nL*BrAx4|
QAkGidL{ryTDV=Ig%KUl9J_iT)8&-)z>~#a=<U;SB@}g`hzDLr={i^m@v%_s``gKffop%!HLo&p=V(y5awF_yKuW$Zky$qRVekX?&(YE^d}_{_&yu_4BTJ(F
o0Qr8U+((#Ol!e8oNIE#+P;V8Uhv~LZhVT^mSocSA^Fy72CiNVf<)43_AeWS+U{8^xm|-m&ZL0Xn?|^2*P#~L=8k0sWV<_s+Z4Fcl>01k3f57QGXx@eQDzl{
wEC;M4?6wlx)=A_j;r0OU<#~XS2Hn{d$@a;0J{uX21?HUAJ>!aetK+m?pX<^q(69#RWF!cVMHRjE$|6{nWHMGG@{GB9S$>=r06;fL~V-W17jFJ6`$_&=E?8%
sRX&Ya9Eb{{;|b&Q}3b)KhdaIS0YmYpga0-b>=_E2GTA6FbYzbHJ;uYd-^`P95X7zkZC|FUJ3t>Mn;6^TdwWQO6yUSRCi(Z;Nic1da{j?dKJ2H-Md>Scg=Y-
d(LiOM)tx4`~WkOb~5I^$}132BCFe;s>AhyPF@66dIy=4YJq@LfmK95cvZ6?FF=%gciL5u#1Lu14^5df!0UlN1I-de7eRC@ON|OTkOi(&hF%$y8mb$_3>ntl
`Ro#kcmsnEnbYP+H=LLUrP=ovWRah2!52S70;LS`6R=6>P&u-9D*nXc7FT2F4Yt3oailhW-jCuYtzi~OcCJ!W;v;(CE{MvC=!YiZ;Ei@j>aX3VwXue)h2m|#
&=GRtN?#t_Cho0{=}N~BnKfu+X#Q}txIl7&CVpO~w1j5xFX4glQOq&hxss#~WdV(IyXL*D93&^W`Zz@Jk{VRPiFtn0RzsUV__^H8QfgeDZh>o>L)Pp6W?P2C
zpw`BrChcpos&h(vgt`C%2ZMu&r{o_azp@BD$}2ic-y!!tWA!iN@WhhJ<|zv|HJ53WB^KaM&*Co<;)q;x+#5J=b)aybePQjWAMnDq@yAk`0(7mw@hf~(LJIA
w7_j}9#*BCM}zIRl=nKXnH~E98?(j6-gQ9j)G%roX5{qA$RlxwXnAPA8Z_Xh)Z;7fu)ipk#5jJP(!F9pixr62Gs3+Hj6d8i*nAw?S>U<FSeb(faFt&9`(aUn
O*gIp^pxSVF@u!>E{_q|b3bAQ6njGCR5Rqj_0%IUYRq<zt{)`Gpy6cLaV4*G4tFKnUTR973Fg0HeN>IOw|+@AHG*Z}69fW-k${yq&O%#mpPB~D=rbHCc>|{W
FTrIE>oL{*_<Q}B0qU1hpfYmg!(W1%&7G^J!YZ`9Eca6EdELCxdw46ytJ!rD(BT6vShu3R86h;+rrF)5Db;_pfLA0`4M-ErO~`kMknG`1ot;&9?03V=xhf3$
CJ5rLe7AhQu5qmlN)<{pu1u>kknpRF10n2S4dt;olV+9q$A)zHw|TJAaXqLTo+D_D{b7ApaOp@jhsJyW7_A}Ysq>O}=YEbi^V|YcJ$d5`(0bYGN%H~8RO|}J
CeUchT#Po|!OkI(?Bf3yON#SyQSjGlBgf2Wv@E$&5bhc;V%WP`+Tb4<Lj-;3_HHlFSQkiRonLQ?Kr5>~k}FtFP=0rmgEVFM+E>)WUVc#wKBzDXkaiT8GJ;5G
q`5~=cwEFrn`U2jZN1gIjx*EK@jWR&6>B)P%_F4|U++tI;CbO6mY<nrkcYcsmhq}jd-)uebxiBUK;evfE`jYIX6(>$sP;Qq5(}xu5G(#Ac9vq#ijx%TAv7-U
Fu0~UM7H@y_92X5*U*Uh@hreK-<+klp4>nKkE32+S#<2#uct0(W^MH{ZufBFu*uHW%^1U<Bu2SsG)Kg#Kmx&MuE6e9E51GHOWAR!JHe>GQNk?I9iX(mS|NqE
SabY)QONtrN)Fba-n{K4ELyhvn0yJ7gKso7kOP(ka`5d1il>+;jxj(IkPUUmJ9F2UQ^iCH_t|hHtymLIs{eCbBvG;}e>t&zIVXU2D*3#rIefCKyB=|6gF>nD
YOv><=qQ&t>FF5uj?b<)zUuJsFep2|=Vy;`)l`t)W(OIhhWNw$dK>A7^X#j#&^jwii5CvS83j_Be<H;U5~Qy1$W@Tm-4cbcWcfJ@I52?tUw`#&oP{IImE%rE
iPC}LY?95JY(L}w&i`f*)zLQQ<ZGQDs+*!#_(wR*V}a2L@svMcVr1bWQv9*WK~na#;YLS)7Lg~qsC3S@%SQ0m?PH(J?o1J1GFb9}e%%t%S~xR&cki9c*S`ld
W3fW3$hJ_r%ub&5OSIWtjYoPYvKCBLK5*ceb9FV;V1i4TJ1u`egV&rJMosptH>CXbWoRJbJnA#8m?zGm?^#&greXBuwYs3;fwl?OLt&0r9NTmpK0ry;_3G6h
<w(hwqy80YDpRtuLZeoolzf8LjFDrt)*vXIa<iqNPF$R_K;j!|C0Q>D-S+}newnQ6`XfVLsPoU!MsNMm9c^P`710(PITqA{ze2kF2kB|zlKVoc;S`c?c9fuS
2+vCGmr~zg@C1!heG-aYjUQiH(;GgC@%zKt)2;Nbd80DXUG*RUgaCl;{&0g32U2K?UmPh#7J)*jf$?N*$pw17JnG|ixs{-Fh^v`(iC-2N>`MOIhxZyl>_E~o
tn?1_xP)IevKi)1)(55NAM60BcORCUGLuF*y)haICTITU7&PpcM^r94u!Rdl75f?6xN?2X4=p(XK4Wqq;J~hl*iXZM53ZvuI#sn~2vrn)!J*qDI<%{<63#bL
qzVtJgvYUHOijmvZ)_`mzNFmi>R&X~Ww>{;LcdtbE~|aiaeCX6r75iTwEled`4TLBoljcBj8jFhHCu8ogH`R3v0fp*h5b6G<dG@o)Z`)|U-QVsOtUf7Dbac7
Mo-3a*4D5kwybxEB`ZtTICu+$6DcSz^ECoIQ0#lJ)?r=L2g3xztWHyTJ7!kdoJneCLklswKQJ1!P4mmSwoUqDnX65{zD>*(yVW7K7Z6=;5x>(&O$k8EtTYs^
{6-bGWJmD;oNMQa)z1P0IiK4&#27Qiq4!4Kug;m6Oi9C(Vz5Jz^2~E!JkX*Y4!SW5wicGUt{deA_3qFY%)I;ACoKe)guKO8g&7<*5fv2z+NOk$Im|#qrw#cf
EOx6EdOm*g8rkZt9$W2~AUw~%duY+!pO36rK3a<v5?|I-XsxBfeu0{>g@1a0YTN(`Sc@I;r|_*P3ru}aAc41wyA7);R!|9<g0<iYky>CP2Vo0fQ)6?pRTA55
&dQ?U9%!(9X6)f~VEOfa*`Z1_^pJ`}q3Ie9by5>E+v?!pj&Kr{fls8XoBiv)nPAh&Rp4}mNqh+c3A=jneRw6ci;il5i0zO&Q-QA}1RwS@ocOLSw`R<7MD}JH
@&*9tkfaE*GIYG#WtL`zoNGw?U#yjf?4;z^dXIFU*V?8sD7+hFEZ>4Dp9d#*VccIF<#1v{tz>Ba2AAe(?Dbkg$xt;U+muS1Gb*AmyYS0gBO<K~S;G%9o`Z-(
+r&poBy02n$r6J}?K8<fBeM#FlS_gsIw!rD>*JR5LX))0hP^XXqrm*+d_s?js`1{c!&kf(+8v$e@)(b8o4aJrWS^rTpbUv|X~&kPH5>FH+Z}V<Y$Ru|PeXOS
8hPnfzU15q6lpu~GU{CH3Q_U3!~{K7$+>=D3c{cr9*rL}Mhh{5`x)yYNGjsz!rxhAeG~{8T@jCOX%GoPAChe8rMgjFv=fMgxN82<WEeQel<mgR7ya{3-PndK
ejRj$*M@i%?P;DH!!YV5p+6!Uxw(}fOt|=WJ<~$|fmay~Zgsl=V;{@ZT_E9Qz}u%atY#Mk*V@v*!9!N5;to+_Rl|-h<)Qk_e9j5(q{8d`^R&Pcb9$VE4aV)i
&|&+f_}tQ=%?+E81xnhg%V8!#0j&Uv5Cx}id^bN`+XF;oN=#KPRK=0WE!ovyDsqeG<GRpJ+y19sFmK1u?%dQJ323=Y@u}0iSJieLLg5gUASnG@J(`TL6Z50A
bFu&f+X3C-g2Ive+|k+;Swft}m!${+Ag!-|yd7M;smQ><IOjDYCjP<BDtyVF$--}Ue1&S9SwRtI4BBNT^dBH7doD%lP|v$w0`~ru&-iulM?7JSMo>Z3J~A?Q
SE0Mn=qxy_(Q?t@G#PO&CLS>^iWh#3$cN#o{wuaBFSv=zJ&R{)zj2n?{a4q0Oyz<YT?XKghwtXT+%Sz8mV&|kwnDvudzpDwg9R-HTZh-?&)!Pfnlz0`<fn%9
dk8!&8ao%ddjWibKv$e|kJQZD+Jz<9tz{3mg8UzOE-`R&NL%&N(^ZTLtzjc{qNWhwHEZJC9w}OzH=Mh!?SsY7V#=XmjNw1yRPV1uK7>}h0W1G+yGIx|W#W9k
>zKengP1eS0nJ?6=bE@Lzc1V&AXgTV1rg_yKLicki@15BLCS(J3!+s2oeYseC||Vy+vGt4%cpBNTby0rjC}vg^OwoK<owudY2#}ZPG)IZ3*i;uOGqy??@q}n
Uds5#W&KP0Bx8VSBBb!9MMd%h%NfQFQq6%UA$?g*st~`33>RO2Cg$kolK&<ar(qV0nc-_wT#m!cKP*qsl`aAxFqyenOTU#Q&D~iQwN6-gP!*-o;StiUr0VT~
2Cv2>EwjqXN5tZwFvNv70x;X|xils6UYooI1sfUW<o{o&I%v1U6aEfASH4~*>O>;?8)ly;CVggL4Poq#fyxH3LpkR)S)WgdTO7+SKH;8cyT>CfG4OL?j~L$R
7fG~8To49hjVee6Lt%Go)uKWZiKAKuM_X)#Vd_o$U#_&#DI<ZgpM!2*$UJAng6w=;;+%=hw)RnS5ZVxJi~m^`$Ibw+eJ-&sXN#U=sSEIBbU}FgDr|TYkeWpa
npFMe$^hf4WKYuGN2SOIGj*hqa(GY-f7GxA+dG}N-IFqH@<dLP_TWP&5x^pAr3*)W$=WU(Xtf`{JH++zE<O6%(wgCAt1uj~`^nu71VIBIq$v{iK>Da~mx0vz
yCbe$1y=do(3tMiw0mVcVhW}-&qA8FlVQv<HxGb^h-(K~jtG@6MoXYEv$t?l@kgntM)u>f1S+squG@{Q`ie;@C#Rfi&&zrxe-+bDb8CdF3;nywSzegm-eg!s
yLvgfyew>Js&ze+I;zMWy~$&VQmbJ2AOvhHI^-Crti~5@q;^>&md*>ZORZ%G)++19NX<Adj_uQ(Ymj^oY??(=dT5!RLfa~M9%T`Z>TqwpI4QG{o=<J?cIOw<
vwlu~xU{!EPagI#c(W=u{~o8?g0lvR)p8i%F3L}ZmFUPXyL+ML`7_4txUpxxD2t4v2OHk1d@(~pt>@Hn%mSDDamDg~tLVFS*844-cc2SVUeZF95e!kh65nc0
`}82i^=^IG5HQ=IO2GN?`zZBgBMqBX2uVV+&DEHl<^dh*{8j<-$3#Mkn5o-NfNR9ml>yYyMGtBt6E9ZX-{N~?Yw`pq%~EfmxC4dr1l0W&(S}4CJPXg`a?30(
yoHIgyi_1Te8TvFE7+D@$2(%Y@^JLnDwwv)FL`hVVerMvsSm#evanx))t<ro7@0GlCYXo_N~Ng<3YNNSJbX7Y>hJ2c#w^FBxw8WD6v*|`Ac_J?vscl|hq$8B
O#lB}isNo;1{^n|*_PEm7s9^Z{bhgb!B|IL;4;(rjXL^33OF9Q10Hw!CTV5gKe<ht3l~G`G^%m<h&2*=U<!w?>^Un#Sp821aKrr*>>vDKxbyZ!^f27geH|X3
E6%;3l9Wo*Ip~dej7}cw7QV(rDN45esriBKX0Yo-eH2bb*-7XNrIV{gH_UU!)~yG2-4qvN)ACJbWu_SjDO66D7B8iknyO4gr8TtP4}^<bJIDRa&o-^fg@31v
160DRE9^WyklTYE?J#yAtf8b~BK}DJ?F~;wypFNWV0lr)elXOPtS`X8)Bitid_Jiw6SB8jN^SWiF+WkC__?6G`D2VvPD(*mUhEb>fJ3<wW&lIrFuY?vMQS0g
&rc>$QPNjuSb`$CJqd_AVRy!X%`;gcwxaHkkqD8f^T+NrR*qx{yV!4}F&DF*&X33rSO<z83Q$HdOFiwI=&Bexi;-iT6>`6&sdnF6J`;^}y=2zLBXJ*r!|dBR
Y-s;gZTjp8$7!QcB<*^$O0=wWr1+SNP?FndDR{ow)4}lp+&)EtWUj%LWp^i4n+`*3lhT-Pp3@;oFB)pjn%=;Poxw<_7S`*Cu7Z*RXwfJ87@hB$Msk%*rSO(x
mFTF(twrnIY6KLo$2^M=Qz*>RUaGj#S<k)iwf{(TqhUnQdH&*|Z!=mjURy?3J<)}4bbRlV(9M6A-qGA&nVf=EDDm>wBP5hP_|9iR%rqg*ho{m-iXysqW<zKG
>#?+9Zr92bdXHx8)cM3Lqp!)-+k{yK>*`Ffi&va+a7a0KrCjjs#9O-yKH)M-3fnz|M?M&Nc!zR#C2n9ZhPJDHR9Zk9fUog~dM8+=VwDUI&oy+nqg<O@x5Hed
M^H+nv}$Z-!H()yM<ex`&_TVuZ<a_su*UPmF+fAH#G|QnF-_1dwdhc%I=7{p@_9C_hSdv^IuBbcs-D9ihqWKmE~p6qf*}JgFAvG-!8T3xRHB<BXdUXz6amH!
XPe7@nn1t5soem53RD&7R>0IW3BIM<@g=Q;e^qFe`c}j?M>csck>I5AsC)s^1_qpUcY<e2af~ZJEB-e{8}MiHv>fQ9rzc9I*j-L6)IDRp?qChc@z%A5H~pO_
P{;SPgCHshAx@JR+-ChhxxKJaj%Ja6+hocvWyxgaJ1;XEV2A8a+@T<N$snG9b=&IExb>2D&2Z5697#|nY=Pc2C7Qb^@&I38py{2#1j(XmFnyl*$awGfI>h?N
dPF&N&sq+;a#bHa?n`7Zwua;J^;|A9KH-CKv$=IhXZYaref6nW{!Y2mCqAE#o5qy`!E2(?sl^1dJcoZoHNUzWM|qOI=Z8d1a*5r+*WUEO42qI#-0a_Ta^W$Q
1<B7G(Ox#el&sqIsXxSI#FgM-@7)@npH<s%--^k*P9gX14UWKDJ?OEXF_p1oYw(fwuc{o5AE0YgYT4L58EH!erNR-`+isUF1`Z)blY8THr;(A$dbt~@Zlx+6
ULAxp5Eq-5$RHdacZg!S95+J)!D>=Q3>pmWD{-E%#HQ=p;SpLBCEwQQ{9!#L(4zK|_Bx)~^<}_M0+laT)zC=9n6CyuNf()+-q+h>ql-h3WvsyX;mJ}#IJ{{r
plpPTOblQ~2nP|CmDarLLfw)HU!tom%r<~W**c9F4%h%x1qxvX+Ve=*>OK{6vsXqu4sZLBb95|e5N$6Jq+@_fRqzv-Gwg&bVcfI$ZB5RHz-*x|G_d|*1e^6@
@;8o9`<TP8NE0p|&4MnbS^2iyBSEHtcM9QLu!3(uyfAXGJY@1b&!j$-Z+A<q?wr@UI|yZKURmDlC1aC(kp{C_QI{rKoQ@3H&_yThfE9cT;!D!Nv%f(+?iYFk
+5KLGB@~!E+}2gcmfX;jb*!3<b_Eoo&|C(^Pr_7i;aQgc(<I${k!p%pY1B0gU!KSi^W$mn1U4$xf$*EzM$zgzVFDInjx~Id-`Xn`$VcVg{7x#{hYSG5gG)^*
zl(-5o5GFVD#4H*Hm&IKU_eCGA_y!y$n>|DFr#%iv$8+CEkDwNP>X1jow@HEkfD--B`0KbfXj<{!fXf9=21ArGgifQ#ZEN<6yZfnWo7r78=MDsvy)379qP0}
3@s4JP$5;mq^wP4naX)-Gw}eLP@qzfgGU=fuw(pQ<j?HIxkQ^zb+d~d))!SnPZ{<fGKlF*<;u?1B+UiWVod4@3f5ilZwNO?z$+yBXV~zU3Pi4B(ZXCebiuRs
ndDgl;VQPfx}tx$xK0`>#a5f?W|4o%x^KYKAQeW)iOt=$6vY+V4BldoCH;JL+ZFZhSCs~);cU!!%)ql1^NSOMkj%HL8I)lo9377qu{Z*4qHxzB+*97x0=_Rd
p-@!KA!+WGcA6nyNbgnc5?1S0oAFFZfZb+YIi%vL`B8_!!T}uBhA&3!%jd@Mu6YU5kDmET?34@n3(-kqsTyeENs`@|I0_$n=0ecmRHcZaXl)!R-VNWpnfFqC
1+Jm1stU$-W1jIbXdIuc^`Ra*^9=A5xNqA{N7DE+V<~ip!2TNb0q5^@D<1%qEIy_ctMAZ#&E|n&3Ue)$pVtVmtaWn412qRezz5&#-tGbI?~3ZptT(!-*lzxP
KX$fe3zAG<_2~O3<0ybgk>K<O6{m&lb$xJj?}EH#p3vT(wD!sIYX6P1jj$pbI}hZ&VqB0Wz@3Gzk`D>SZmMpZG$xrNdz)}}JW5NQKpsyixK&@#p;Y#~CHV+d
$Hp+TQQaG(k{n?-uTeo*b)~A<Hilj!OFvsRDrTpUlC~ELRsFIjFZ|d$sby&H6drHm<YI>6C*-Cn`k%4+z!Of#N(Zv-&$e3ksL`GxEF_=NiC=-Mi6(7&wAN92
R?dt*df{GMrvg)L7+f%&*a<%wkEidQtT@EzTQA(&mmeR)m7O(~JGH4M_WV}+;4j9uEw)51J_4H4j@~<4b4;QDo=X^}1sdVL*!se0kvqJEs{;Q057~Hn@SDW0
q$L^_^f2|(c+hj1B$pL8rD5KLdL=MILK@0{6g{E8j|qSil5OH=|63NGM{;Gd6E^03(bA$jiD)A#tbo;od4(G#e_iBkZw?J>)9rtpmyxOmnsJm;&WZ+oYF59-
aCao0DMswLm2ZJ?nePqVjFpVsc1xAUPsCe8=EF{;`>NW(Ir%%ha}O#>j|)N?<DYgE$-0(sp1AJD?V!b&U8IT2ck%rgx$m^?5lfFh>*xlr{Ts)q?2_jHk-yQ9
{j@82V;k&Ot(yY%ajr+hn!3a3FvHT^At(&<dkurUs%th_wE(DC^ez|)8f+L-dD7)w7c!}F8?&qmLysD9!vbqF?h#-*qB%?3*bSf|_RRIVm`>c1wzMA}n!5^N
c6&ztC&R5jOv<1-*nw(X(fFHn?rmaW8b{;phVP1Aepwz`!h9)YkUX|6pvShDU;B$m>w+6P=GPqq8C>V&`Z+Pb-VyTn{4G0>C}*o%oiSyISd+5&VeUR>Gw(C}
ZRl0%fUWNX$qIaYe9yghk433>T4J>XzLBf8VJ;K7h(W(&C0t6@ZpjW{B~D|J=y7wmQ%DkVuVb4}-+=+DO3wfEhBB;riM)Ncj&1Sh<5)d5Nn)Gbf$ip;TUoHi
CQHb{?(|Ab1e;(W&J5BPSzfKx&oVw3BK<>oi<#_eeK}~j^VpegD<lFcf45`R&%PExXA?>Xx@|rXhUoIuhds|N&n@zyv{Xr*Zv0d3#7(28PvWocdbInt1y^D$
O#+=B<-<6T2W*R%B#VvrJwh~xPCiYT=*R(&bT~rP;@_u;sw<kj%1zyFnz}%n4U6NO1e+l3*@mue!+g&(O1f1iVygwYJH)zLltaeTo=|<@a-ClP1rJ2OLCgfO
@cXb*^76DB?jmM~-_uNrA<oN|4~ixMdS)S1tYhxF5ySbGHEaRTzDqkOtaPkfsJqGoYbB1Z8*Xg!ac=y;#YxZ6aSjAmoECbRhjeY&hqoWc@^)_C7eyB4?p973
@vMo*7L{)pz#p&KPHxY<nD2@d7=iPOk)O0Lhr&%PzdOxzXr;S1Dc@_&6U~-QG+)C8hR+fc)WI5DD-$$A1$(UmyjwI{Ag4$-{Bi?^Z^(RKr*_<hAyBd45tV~p
1-(Oxy_fwTQQKZMs2CD(SmBQlAuFU7^Nl;~1y$(YNOH_TitnO-V{nvDcDolmnuR#?m)Fb~+fS;D9vRScisMU}{+wJF;!3Z_%bXL1o7$5;IIje(>}&laK+}1R
qRsJ22VWT?e5kTJ3A(9L2wYA3PgUdwWcD`uNFeQfAE;R+QgD>|0Q^_S!b>2o>=}UQD}*7w2b7&n%H~$gDiEF25<5HR92Ty{=h8=^&G-Yt-4^oBQOjQky4|bU
dZHLyKgd9DKffja1tzZB)c}!GKLZxg)hS`@?0`yY5@HosJ=Pes#W&7Q-OAkv+=#U>4<<QgPV(#e$5&Ws3Fnvq{vvq{SI(IjxjHbjvKkT$%wQko(X97!+Pa-W
>2+Leh1m20kag-)P;!g+JKLp)Dk>JWgB01E=BQyg$zbp!*68%gds!66%>;ZKBA7h=GUvLcWWIm@Iev8{WH0~#=Kc1H>;P<$00Ds20>khcdC1O8vBYQl0ssI2
00dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_foodbd.csv')

In [ ]:
# listas/externo_lefood.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_lefood.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<JXN`bzJ~58cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppcnnU{7bdih4?B9z!pNF=Kdc1PQ{j
khJT|E4KRKjFNUmimR`l^{%Nzi4CIj3Bjhg_*Tv2VeY<iML9W`cFNRMLkE<>-IR9vIAMD3jO(u34+3mEFoodfu6C14HvnZSzk+>!CDj!t2ML*@Xb3giH&9~m
QoV0RcX^R)4=3q(+3DD5k!dT@q?J2MM>>8k!Lh)J0LHf8;&3!@Z>LVT95$Lv?tYU#t9ZaC;hC@F%i;S&K9VDE+ffB-b`ohyc+)XrUb&zc^u;O?)yJs3H9oo{
4O$IJAuvi4|Dm`U|MBtlStsl@HCa4Eo~$yrN$W@UX--%mAW`jj9huojooOA#*g4Il&o=+?^LB^42}?M$21KtL)+5^ynxR*-yzLFg8rSZ0$BUs1-y6t&dfOB;
r-`QjV*qKirPpI!1paV$6Rdx@xIXRtn3??3GDvj53tHF+&D$hMGg7&C-npAgC}36Mlwz=_g4Byq%6~Q^b~BxGf6qb5Qh6e*YgSvOABn*ENP%{lveOz|6NLuR
CR^o)`pi*<%RIwf{c#;H!UOASD3_f_Zn0~lEz*+l=u9z}%|-T0x6SYZ$p$=<8n_)kW`&N_P!GY%81owTLlHt@Kjrh<kSP^7vUUaToC+<F!^Y^SZd35RoJ4D#
@@8t+3V5sCO0|y}pNQQ%8DgHA7T9=4(%7RbgCW9%R#6k6kPEC`h9+ZIgG$q2kJaP5(YzU7vz!nf7xHLd1CwyVJE98BaXJ6-airQz0&x_Nn+bYc10=<@MIuNq
to39|2!CY$ub*Xk@kDokNqE*8LbcU*HV4u8C7W=pS`s0!!If3TvMhHSQ=RrUn$9}Ku5|2sQze9($X6%<EV=6LGOQiLKJqGR{yFKj{6uZcvk5R;b4z8jOSaUh
7C73^Q5^Q_wc9Gk_4_mrAZpVEd-|N(;<s7lHa`xIO4?gRs~5gyEjYj}GFdrv**0|l5gNFE?Sc`=F*YtbAkPRB<)$lZpk5v@SLMx&iL*#TARKGh+bgAE!6c)$
AN=IOh5}w+-(}c=OsuwCiVoKHfX0_gB|syeAWm90LRXY9jhfaT$tQDU@^#3Umt<zwcxopeB0*tw>f6*dNGNt^_)KDpy*1w}ejSwEh!fytKw;a^GOgV-Bg-Kd
@4H`p_m#yhsuAaDPflpv!};@`dPb3fA$fF#`J_^6SD2H^kCA7a@VmNFW3t0Uwz#Hm4$iYIFZK179QZDS`>;B9I2k}qE)H5v(4b2O-g%E%<>V8lboGobDl_=b
YN24o0+xne^OfB6Tv*$UZ7$8;scSi0O*s{*8`cxyvu_a%ya;KuhwX@lq5P(L@6A^?k^))(;Tc6;v{<K?KZ5vw3fQHC%n{uK(IR|wR_rap<oh?Q>$$2uFpLr~
WxIN0<YF4FOzdX{;9?)Y>E?QpbMY)0qvrbmY3KWWKFi;|ylglW);UeCL1<3Dah=ylYH!=u@fo72^9wYWt$L-vzv!q<P+HQ|nkaD%kb6FAxpQLrW^oMX&nP01
J+<5(vm!cDUJPxxV-?lIoEmls0KaL-UqTv6#QLmpwbTSx_M|TUQ<;x#|Hsv^Vtv)GN`3pG@{C$5uYQJmo@+&1fS>gk+M%i~(In6713BqR3i-aE@tE<u2XQa?
q^!wuusQahgj`Z!@vej{bz4ovGnF3f>Juvp(B}x}K5LE#CD}#+?%$<4eR}J2*1RwzJPl-*+E`1Rp1{&~5<&hgnOZ+_{>Jk(0V6N{g^b?cu%&RSzUwPazXcU}
S@W4CaG%PEint)r&}Lpw^eYnSSo6@KUi(n2($k{hHQQtDI>J|r5|*S*peiry{B=A8?ht8qAx+rM9RQI`ukYQdnJsO4a}Wqa6oNqdHeq#|l_Xx~P^MtRO5-@=
*QxLbfdB4=&@gWqN(KHZo!`YQXVhUecSA}<&q?cufthj_U=0zkFquzmOyO`wc;~ZXy<(Hew^CRR67dR@k&}gf26jc>cR7D$c*|ki3{7FK+dnGO@=Dg@wUS~J
m*f&GH7EEANE-@%uInodA35}2eQeshn=)p6j}(6tCMdSPV7(*vHLPVIcK#m4ryDj31fWHd)U~qAxs9le-zS^ZZmhH)kFr9+7_}=u((bE4XJ;G15C3wS6}6@g
yIjBb+*Z?e{Iia$wU3ACH0Al(L-%;)tZ|a>cAE9kYvLW_3PkG-TM|c<GWy5~V2RmNe<qGvYFgaa{PvWh-lDCMeDvb8kmfsrDr4P7yGs3f?x7ubxGIgqR2`YP
zXkC&Pcs$<(Kj4i;FcgsOtm(5MVh6-XZo2f(*;L1$XWb-tg?U|5lIEoL-`D8bSoJNf=A;Q)@ia8eKR4)t&KQ2$3$}SSDetuJZC+3q$LrcHr;qD{`x#-PwkER
l#uXed<j)aKMpX-o5c?7p7@>*6erT?@s-giZR7_vP_87<Yo6rxB9vx#^Bm)w+F~f}E!@J<d`s4QuVm%O7&rE5eCF9;5Ym-j>33h1x$g8WC_p6I{CUhD%ZU&e
NGGbhS7{k$C`{rru3;sv`!=8_RmfAdL~lk6ga&iygg`b1yO2=0qL`|FkrSNtwxId!P`)g!aMgC`!JprJjVha*EF|@PLscO!JXBVw;j1F$<u;ajsAvuYfa|+n
@&~}t82EDV#R=YWo$|kt9Jc+OWhT5H4Gk9kjAMA5Kf&FEO=2?HQz)Q!ORl#A#ub3>654H!6eCpoq`5v$GqX3LrP8zNd5!k<5Z0FghiGPd0C)D~z<VxYIu3F{
qQ2dZG*ABZdt9NHOGw8kC4jg*Yf;sUU2qUQ$|Bc8wc<KvuIGidlE&N(b7=d)WlzmpEB_<BuXxKD0L`OMemOf&PgAVDQQw)acyJvU#HGu@vWLA%BU~w^BN}}=
&xbaH0-g#SCS=>pWP?l^pY##16B&MYzv>o;Sl7=O8}h9Z0e?L2!4yah`IjVVtGdg2kdfM)y?k0n>O=eHbOx8rlnNeDk5y;rj*DD=_}W+_8hB#ZqFqWQ5^*+`
;0CG>%6AntaEK8%6*UyL@9BQw_-dMA(8VaU?~E|o)~3lX9P*7^7Y>;><HY&-%henmc4bJ=DCpsuKq3!NovqKo_W`(L8;^yC*YbF@CWFSmJX$L(?V##N(`9kt
tFKv-)9W|?zo!OF<BUNPi2GEGG&hdYARhQGp5o}(W07(!3H%X0b8Q~~0T6mkHq}2kz%7x`w;?*djehvB@g=styuEC~i&)*nJs2HuYGJB(3)71rcbarS8{Z;w
h<SdhPiMEgvnA5n3JzFv56k#(d^c?&REC{KOZv^vA=-x9B&Pq>i!>~}d0QH_Dlu?s^Rf0`(bzJ(pRqHnw~*k5$i3{QEx>%VuVP{-EBwM}D??q+LBT|xfC=O|
LQC8hBJkAiJT&m{iS&9TCLBDtg-M9G_Ux<R7k<Ib@n1A0C_qr3zMTtE0)OabTj}@LStEJG-gWw1q^{{D%E2tzR+}`06$yB2XX@dS686LAiX@=jKBNc%eg#1T
NdeRrl_`3ZmkAPu76N@`qVito)rSs9p<{}9N_9&H2YmID7t*I@9?O{BsPTttu}YgO#<!mPluh1w$&$j9%gp7oa_Xq#cs+hI2x$&zGQ;6mYv-e4#ghkK=$&3&
Dx_a|Ey;Rw`V!EGH=l|jl}j7+%RFqN51U!4^&I_593S3*<Z{c_#)D~>HeJOs3Bh5K&yD(lXbVN3rQ)&vr?sK3a&r7Z-)EMV_G5JM(x|))u@A4FZy@{A8)6m5
g*VTZ`v)0)*KTTuLB8x+{O_y;SjZ_vlo~m^jkKCDl#aQ&6x@Fg)qUcJ>Osx8)ZLCfYr6{jX~!97NtA4Sj5N0fm9g;F+p}#Ml>5h3y}9K+G3NG^Hm}=9raDm1
`8q^nz#lk#H(S83Hnhof_cI<SAZo!%a>yqj&e4#D)MMpMSMzi27V33&sBRC|Q+_b78%zR%IuXWtSblMJuBt~WD@LHkP(50Ci+Dr2D_!$!0i^Jy3UKeBC{A(B
Qe^cGI=70eY0eePQ>Ze@29LXgpfPW2Mcnn)+mgn5zO~<|X?$aq<~4v7DHxpR_aj?mM@}N8g$DNq%~xLVDRv6FLie`}G)mW&u>2ud!vd7T47QKnqG5a2f5f1H
E;Fda>MK!CC`J-c;(&RUxs}Vb%S4!Uhyu*!Yqx`{K~91se71Cv<Yi^KY$2!kt(AI&Dbim)$!QC1rI#z|gc{+f#aesOtXZU5J;+!%yFeA;he6IahBfZMvM^Sy
KTZH&3w*iTBY=1mAQ_jMFu}Gw4-NeFjXFBtH@E~g&T=N7*EH+*f`-=gX_EgLUq*ZE`hCPIZZ&<76;ICxFYXL-l@EBjgTFEbttBBsw{SuA#{y^f^x>b$Un7W*
xH?G$FSE&JXuLq81hb~aJ*)5h+1HK0o!;#GH=GVlBw=&a%D{IjG-&TYjE9fk@*dJO27`%ryV^6|qhYx?tC-U=d<`b%lbBzHi2W7(-?Ks%oZt@TS&u%y2i4VU
f2jzz_*3lgp!nE$<%qs;gm7qXJ?X+I##du;8-AK$rt{2{rNcElpT^p0aBT^unp#|(;ZJ6{NfM@F3(YWNS-NGl?2!gSU#0b|M{#1yo%qO`$4&BrZxA`0Z6|X2
KB1j)?yWDuNE8@oqi(q!p5e_;=kqwa79(3{m<UUAo~6*M!o=n^j|U9Z`X$=lY`Y%ls-ikdV<P55d9f_l&N(8-p==jRb0kIm-VQT%?5Q{@4QUBivmiy;*X%@}
_SyCOF8qRMQ|_qGRRHuNaTxhWJM|s#z7{F}9Pw+B{-zJJ9a)_)l5o}t$>L8qL~ALk6pOBpyWYgLP)Bh2^*zXb#svkz;qjz08O}L*;TW^LSx5<Nz({R)OBoAV
#Ti!A)+jGcgX~OLu9NOE{*uUCS1M2US}=2<{62QbiDj6tQdIA$!OrTX^==j9`|g$#{_HhKx32rRxGwVGCNN&67-Z5MPV5(|<pWM4;8#K)XoSb9xYbk$48I$*
&$E|v?A{Q}fpZ|DYB<49EVIvhtrcpMB|0uF)WY=`7pSU4mP>@45?{rt`b>O;_|*Kmm5YI}g%okbsH=jW|H1pbQ1n~@1R&mT9qFQ`L8lS7!>HVVii|+@b>I@&
fjAwEeN3sh3ZwAoun;e_iq|H)E$w9?pSu}NPeV|^N)q_U;>H8urx1<sW1b(QbWNpCRl3kgLGER#tRY`+%P|H&9t}BXA!YcAt`cE5kX)q_rpX;GZ!upB5>Q$c
c3g&d>&%lThatY~TyoFFg!!XCzkG-{F!GMvqxWz4W0Et?eb6h}dS_w^>#G|dBuInBb0_9ZMW@0<PrzFlu+ZEU%p<xS8^kRWyc`Gvq{QxVA9-XS-Va>)4Y&fm
<~obrcVkk!3Ct&iURu*Q3JR;&ns}$)IB`}`2X{K1TLqO?Ay1=bkto!1*7l*GQp0zXcVz634#C4}AHe41sw?;)ySzu<^rf^evRaYk($sGJ#g$-K^kxn$tc@Zv
aVZ!`Z;eV&RVu-JxL;=Ut5BR2c}6+=aja|4CVXPKd+N<tHR9mI=>g~^NpXOmo1+P#Y+YyvZi()kzA@NCMYW<0v6%!191lOeyk}uNRT;QHte<YtrxLDBV>>k1
jjdQw?v?D*MuSEh$4%YLcj2AO*L#Ob1O8igSRSQiLL?`Zq>LldZeAbQ@EextE;%sfJw!?B?+mr2b1g}Y6bWwcNxjS88_5Yv7BssT@yoSXm+X==Lc7^XbVWxA
priYyzT>RC4R9}b4^cjj$}OcM{hbZ~wx2i<!Xr$wuZo*d=YbBR4i-t!s#fHi(}GyXBM-D1`o_DC;~MKvp0@((RWjY_FB#u9Ud0{JfiGQYw?9vXOPzAcxNQF~
CbKWjf!s{1Nb^5?u2l1$yM0+xTBsUo3obw?vfh+=vrhu_GN~$pq9oUY_)DtFxy1cuiiPvClh<ub>)`>zsY$dhzRIhu<3_N~;e;}plN~MeAtPCccxqH2w)g8+
uCycP`VL$~O<>!b2^N^oho*>As6-nolv=5ii5PB!5d6LI)1E6hqTUaM1^oApw5H7Pk8xgp_?wz|7-Ppe(ddj^-$^Wr(W<1%X3=ZQrm*fWTTp=@A~pndQ+#n1
MoL9=L;*L3UtoHys<B8-1lvTM1ry-Vin_^hw<3o-xA%pHq+&3VXugvu+TrPDM?L_S;0R+BFgJSsbaLFAr57i;x)V8RJC%sH-K0}l2*z1dlZ3&1Llvm8Td}#{
E+}gOm!MWx2&h<HHz86rkd^D%@2#A<d{|Z2w%k29nPBIZcU|^skr0c{+BuxXm(tDp(4^KP%DNfj`;<;XIF^8zUt5C_)A}QxbvS=U@;Y?gXkrl=*Zm6x6l!9U
)<K1Cjj`(CBXm*u@(p-eqo!+eD)kQoDs3R`e-+F~UV9T$=N}3w(Npe`9{pel%mj2sk6JESRQLego~D8qpeMGK0sUPqP0tdErfFSaQKkvnO4q45GpVUcf3TYT
b|)!Alvsl?FO-<_8vUiXLrOWhhr3+#NO!a?$Q~8nC@9Z3qyJz@lUHDVauu}e5$2yiY6$C3ht5|<t9@S?KfD{sTzFTq_{n_DUhPT0GS7c!NVbd&ID@7i8Y!i3
WH!yRiYa>=G4WH;Db;Uc?Gg(-dQ(@}76L0APBuN-N>t;ljuK@k0ZdA~KkPZIc`r(uDR!_lCP76TyAdU&ou$glw&aJ#Ko0jvA?+A{Qf6tBY;ab+JM|{kSu<Y6
aUi9cnI)QwrdQ%Js2zmgT{T*7k)_)q;#nXhE|Pb)#wh(}T~l?~MA=TMtrk307}7Hz(bO9xH8}z)mbA1yk8?=78@^^)RevwFUG4H$UOO*;1>^nB`M*r<VPJDo
d$6C#bh2^*q=ea=9@tQH^C0Ab8PtjC&(kWnfv}k5#b>Ph_tdotdI%0i-b1>;Sbhm7FL}nA@DPcF)7qHJ&w0!D%IP3~6w~z&@Xwc3Wj_Nu0P4C3E}TA9R~jNS
&<|&))kKp+@TOxHDNXBlN;>I(Gfw|tzs=>RdN;?Yc{=-4v9O1ZFF*B+z*8<G_M#Exvo)~X)5t0pI5#PKIu<BU`WSzdX^^V$taV}u+Pq9?=8839U$u*kBr2#x
+Bvv>z5N=3j%%FFZeVa9)G=m{x{WrDZH0f1%IR(k5!a#a-iU{yJ%9Jc(qV^KOMj~&=H5Cc3iFAK6AC~{aPCg2hzaRaG`;SlD>A)q8%UhIVWwOm9AQtx=NFrg
+(z~zP~4uk+3-r0`T~j8(cmA-NVC%gUAXINd$~vR<&YPy>k#`7{jPP^s(~b^#sy7i0dnlTlu@14gk{akp;2p?NupbJ;Qa3O{YZ%edYPR!b>LH?#<@&RxpxH@
ZN=;&uc#T!AwcpQy{6g2gRa{pjvCqw*0*yS>{iY)lId$sH&mB|%a$Z-A)P;RR=n{g2!_TL#}DYTLLRFDw17nlCW>D?caqdP+SSGrxHBfB@qzQ6IM-XY6Q?`i
cG`-*NiSV6NSrVWwEaMTPV3tv<?CN4LI2lMro2}P{BMnG0<40?@FaL1vIhq3Ro9}|Fe{qzT>i>Ex);m$UtRVR0-l@-jS|7m0#UkiFxh|vT!R}&NDEH`UsIOl
kh2wr5&1j-{3ug&Rl;$2a&M|dZ!_wCP)*p!r7W%aKYDpPUG@7u;Y4j~au}b;5H^qanCUZFsKv$p<ZH-NL6}crTMn&?P-LCY;O|FUVoJ5>uWo>U?XMz2?tLCh
T%uhb@fY^qV%@b!ro$k^EO~Zb?~7^Q?9FuFsE*yk+<fU(3-kg2BBngHxTA)o+qNO(Q`7}ZKsXIb$)4+X7cG&FpjU2*NaB$7j$mbsTQTcH<q7_G7+xzWTvWlr
@uLa7r4<nU9|mGR0y7Xyc+AP$tKS(7+kOE53q7_#lC@A`iz_IM#O68MG8<(==XQ?A=adcFWU#UmsQil;4EE3Du(}5O;3v+&m^_-!aV<$gXP|ZC6sAZ_!0t@X
eQOI8sT<H0D5wAZ_5*b=a-=UHXW@K%a190e(BtDino2i_wo@6(2r`auZ8nE$u5T{XH?2~%YmJO%l>K%0HQs~Y?a};;DfUGRZZ#G$e;L0Sl2H1CGN`md-nu?h
p8KLgi4+I-1m(*7fV)g0uz|h)2Y`nasWxL>^z~#%ien}Sx|=DzELsQ7>yVxXkmS29AscqBxtO){2g>oAG14JKQ*H{<_~fp;hM9N@RQf2|lgX{NRo`#aftSAT
pC(uEWcJ_gk6XU>*Z57mf#}CdxI(0da{a}zNj#Kbs<x#wbA2h=#mS`vN9=m({jRP@hGv#PWUaMWS8VL#a=MY_PV~DNJ6-@*by?WJm{PsvLR61%0l4|4yB2qY
5<aA_=3XbiaAfEhl`(pwhDV%Fh~9cs@#TX-L1?kSv!2vZjRuqIWX`*I+sTq!aebm>Uhtz~;A%2lbEih4rcl>JivSe_Sx)Yvo3Q<t2ZVH0H-oBR5K6=CgQ$=l
1hr|{U+URi-@Z(}b+ZHdXU2cvK*3a33`6BsSq9}kS@qFCO-G2U8OP8B)C&!piQ?Glra|CO9y3>LPNjq_J|)SrExx7vDS#;58zoXTPL@FqMxCQaeBMmbe$G1J
e7gJV%~LibgKv&y{`a7^#*8r}aiV>J!{B_I!I^}_WU390$|8qOHfAM7PEl9eTRe;097@-*f(FH26PO!K)a?aXx#ftz=&34mJGKZ??S7Luuo9uHy?(ci=-;dj
0aGplvQe+P_NWjqG3<nWr2y9T7Cdz_F4gphY_=b*KAeUKvCLOVX@%S_iKF)a_B!>v)0BAfMFa%0fb%-?7kxOt4_`2oo@4o@7Hw+>dy_1|VNw_hv-z*zP*&>_
LWuAj@SYIQT~~BlVLyO56Akm7ce<QQ^%ph!ED89U4T}>b5+DEYAq+dlpxOexi@mOG>M-kX!>xdEqg3TAsufTB^(%d&*DCGE0nc`DjKluIm*LjJ3CX9ab`^AR
{h7Z#BDtC3D7v-Elq&uy^w8_mN#L^(oo69vV_B<-k@f+g&<7oF!UQUltAv_yQ;V&sdZfsW{e;(enfRJCV;G8s8T#f7i$|H-l>y1EoCxK~n~Qm1jx|dGBUt+C
=)c@#E-|Ha23lzHUI6^T1@MLo!aQVeoB8gMY;zKGG{$JD6onmVZtQU{qEO-zk5%3WD*1_(g$HudGs|)va$3kE2@QXaiOT>ON}hN4amS~}zi~_#?NQI}?#ce0
MY4w31O8k{*mrj7cR&p<btpO&0%=6421YS4l!%_t{Js!askF9k`dgTXi*8Q7fHH5MAEcT`Ed3bykre8T#M2bk{tqyfhxUH-4E8kQoe|f1X0EYmm^=@cDxYmL
gYEI{QV1u)9m<28VNl*QWm~m8{i888v14FNMVk;<<PDhxxFdfZv53&=Y5$dK4)KV+0o>bQ%D_C8HbnAF9U_(jM!f56%H<Ql&Vd`pAUMom$nWFrJ*Pp`*G?)a
RqY;ZV6R{Q0fxA)k{zrCdZA}vtt21lLqRoH0k&(T1Pi*WcQ(d@$-4Z(SYaIlW=Wt8Vwf2-+8^RxKpOS%=vMKl@Z&_S$`Lq%9GC0Ls%$)=d>rrblN1qBkHr;o
Kya85wFyU?+p#87p(Na%g7gVGfJT^^LqSU@SUmUC8p!N=RXfo%nlvukyy#w)Zj-xn6*K6Y8YMBm$>|7zCOG|ntpx}=S0O-*7kBtRn%f$!UC6q8L>goAQPuLh
!HFc`n}xjvn0|S@zt9oW9eowARoF_9VZ8k79IW{#{lhxD`b3ASvS6qBfIm^Hr1FPYNUU=-=7yCL_f4%)x^oViN`M%EmB?fQJ!adqNC^%_BAk`_Fhl*__4fuu
X)jk8hq#Ob@b&v%zjPXN0`T#LwMosmi=xUE8iM*aa}lwzk7Ddq+N&W!yIfB7(!a&EFTXEC-tlo9eZ-Wl%iKhO@co4+M8p>VIQ7R8!LVtVLehxU`@G5sHwfE9
?ocU~>h3WAg29oc-xIK_dQZ=ea)Q(LzPCaMPUsj2E_UVk{LiCU%E~WjPS|GKU?J8zo;;oKjsB#jn^bR2&13$Ug7V*<M6SzQr_N-Yhvm`up)7aFYgW0X0F6wt
bInp;_h+m53^WxA7<-3o6g8YzfMdd=_USLqG##XZ#ZEJ^Q)WMdSEnT#{Oh3n$xQk+|39~KsY-5(%+*Tcg5t`1?|*8-Be5|4arYY4@U-PHWPS;dd;4f_v0LYN
4<5K@_&Q8PZ>M(nb*gjwXzkTBgXoJ;PgDO<#hJJ`tZfuht+>b(5xi(l*Gg@QZE;li2#uwmTEWF++gm~2)jy>(!6;UD=feyJif<xsS8+lKWZmqP3dvB?BmA?z
)^)~_;{HAgoU15|Fb^vacEWs#8n%B$WPKVo&MO0u9dE8&^L0=IcsJe+m*XCZ-G=6MP4;$L8UobPuJO9v5!U9dFuJz|bW^!RD=&Bf6+^hYY3h`<U;i9ah($|*
DRD<E>@|f6PW9_eY0T24?KHKP=DR7*F&X6NJJG5`2fPY%&#<tKk0PPdNFSStC(>d5wo5bs%Xm4`FSQYzp3FvSvha}3o5V8Yqsj#GC5(G<ckN+ercNgI0JT8I
7Y@MK!w0*+CjN*`7gV&OqYNl!_Xs$wn!dQlz7#4!BwWj6+BralrJDuCQ}3o3Ny6UHepDvpW!2-&`F*=R;3V6MZ^1|oAH(7-SRcF?f8zkmjL(0YmjA34-q1o{
OrUcx+qD?<$k@Mm*|-t63JI2pvahP?E+rwn!#7ill@o6%yw+}HAS!&zb?+NYv>o4fl4*`IL#Rbj;+CrhVtMXbiZzc*>Cal0KStwr)*Jr;(WBGzDfVNXniUjr
WD*_=paSwH7`o3bpWCq=qe7xo;<_PM)%Jk!DRxECNS_&}{1ykU<qx0Tzh6aJ$e6z!@5x8(%OvsQ=ys7UjCiVxD_j}v4NnrGz7`EBjlR=**<4xz3ucuv$Ns^?
LhByvGslFa$H%HtO9~^UD^5;m&T<4uVrd##Q9QAjQC2Prnc+x%DfRbzP$c^%%dQ@vyNPD?N?WX+qd~-j*ByEmr!R~cV~(Kl1Ae{upj=G?6)PU9e8#0aZ(34!
X8qNdG!)%)soeoz20!m~!Fma(W#pLdt8E|L26~!VHs*}$LVmvChF+8dRsR)oSj!kg6F-%!$aLn_O<%(qBivtE`<n0`(teh>D(tFQRqdJg4wIwE7tD&e_4;_y
4BQX+tQE1*I?m+oFEsit+5;+4Ufjl%W##k7f=IEq3O;`O^x5;vtpf}~pGCQ=J`T|?x*oVxjmyLETs~%|iDy}+0g0I<AqPFiGG!O&{Lz@7Iw)YGHIg;$)gjWh
066OLgCdQvXQu5R?W9d#OS^CkA!dIb8)MmtEL*D{;^%5{(IFZJo3oSd#daAvpCs#589|}!9;JZkBN=))vJJ_Xd!>-FrM|l9!57anB`2qf3bMq0K2*Pc-vO<a
W$7(1>sv#e^#Gl){sn(XmRY-MsL;H<u4BaA7}Z+WA@||zgY(UwpHhFh|3LItXWakPc$#Nm-na0@Py&RI854*F7j_f~vDgG8`-MAH=oD<1=SSRz?RM>xiZtx2
Eq`H(tib+0V1&!psXztVK%;N3#cS&4W`|B-!_IqwXZa#P4#yQDPvGU21k5eoSf`D$6g%Z?bBk8|h4Z-=(UrC2J|cwlq|Oj{&#{`Ql@3+BZY#3n<x1gxcL`nV
`|(#TQMBwKD+qZcP=F}W8!lEA5~q1_0~=93$DHP}z+KVzU3Az2$f8<l4@xeTZAUvGBRV<}x7(SeqJ^p!AoRn^1f_Y^$3vu5<8*PUC*(HentWkE0%1x0|2=}^
e1|z})z~j9@)9bpB5e#<IU&zDG=kACo?S8ivDGG$H(V)D{1<+kJchBiOZTztbVfA(Sz6f#$5=ERTb3kDG_jm9pl6<F*Jy4!F|xfQRb`4^doZRqC#5*-{WCEV
(%yqDr%7A{W&km;8Kc1@7FMRH(pvguaZtb_Rc+G#Gs+b}Lz?m-6kFscKEB4ZY+0+~({MSj1zury=Ey$@+0Cq}0rXF{n3T2~(uDeY1C<nj)y0;Z>`Vc4`J}U4
rse*ay0$Sm+^*Z~*SfA}HgX@VHX9V9ZB-1Zc0)oX9qg0wyH};KfiA-*p^g_Qy5}o}5DStwd!i#S*5vOuX$Zy*yE$Mj%+Omydvvb~xuj$;uZq#LFn=X4F=TJf
{PDUH4+wkUOX7{K8I6)i=X@8m{u(cWOOBZINFBbe^r%yQ<sF9Sl^*TC^ELEN*);`n6`u8nfyV9$p+d`u`}8T4V+QE%(0;9VsH*9CJ=IUV;%ssGh-yQZjK@$|
MlmC{%(UI#IgRe9-|tv`qIqyW8)0whTza)nGMP7&Vk$U97=y@qGEuu~9khO|Xitoc_PPY~sGDqxv>x!(DoZae07aJ>(*WlTc+I4XYbLOM@se7Alp((9zeQzl
C?B!}EXF~j<-OdDqx{ET40bPrrKZj?6>Hp*^<Z<0?hj#JT^j%so8gJ$84Eh6c*ic%T-KR;Hrre+LtFB0EJGOuaF7xR0Red+SG!d0$2`*FYZ5N^>4Js%&CGm!
$qhjUG~br<H5cFa6bTP`k1kJAzq(0yM2cITq?-`N0^^hB8*DX)7)KokQUhyBDI5S)&V<WE+6FyZr0+mN@91=7(^Q6n1=-P5D92FrLsSY2w@FDjG<xVQYC6GD
=KV&`)^Of^{T(Sy5)t140Yg@C{Wm))dqJ|j#IK>pDLs+2PhysqNOMzLcpTFAZRq+r+#6LVUnLhV@lz@uM{97Ddj93ppfa4$<%o$|k~Z59!~oYpXq-zsV)B|o
)lEcOfN6a(ZD&-;l;tlEjErYlbJ!F$z<o|)XX6!T5BT!4+8HX(xWHWGpq_%9ZALiJ0O{Lb`3$C%fndSRP8L)_4uY4)Go?Qp@(d}AJ@;oPJ3fOc{wc^QN6qd|
k)<~1SV9aPh^Z=oXaJAr@BzLiIUAu5<9s{5^eT-6v&n9?RL^{OR*p9Eax8x5cq>HT54<UG;G(Oq-Ez1ZBAqNB)>+>_*6@EDt_;SfJ~E49ZtOtUlPK5_=Mz@~
n6Zft6okgbYL=h?F<_OZcvk}*NB$0GVmJxb3icf&Ic0hTXx{X4(FkgchZaVtoyNw|rKY;VSPBKpvAWeX>YtEidPD2xJ0@Kp=aoHjrf1Rx0^pA*flX-pV84(9
XnSa^{oCo_BQTqYb#rJY9|_^RjDcD}#0$}F>cC))5p&?LPHu4Y5+<8v2QkeO>b%WpX8q0|-G9rtt-Hzn7jj*I=ZiXr^@u^jN(;K=JqPL*aJIar1=WKTq26Ir
ko4%$$eva8lTl)9@k>d?BQ6CmGYd!{HRm?$v?+3s@d>KMx?Xdw(?6u5aT>j<(68tk<h1cwO~fKHbJ?aMR?FtCkpT*<<-P-v?NZgY?=4but90Fr^k(B`AzP3w
_0W;vLFwJx7X@Xv6ssv@78a##GPJ$@E|F5O0yBQ&0Svr$(4%5BeHDl+82Hdw6jI`5-|q6v5zYVes0y|=v2X-gcJ6aWlf~>;GedK3R^dwcgXW6>;Q{!EMX}vp
$O}rW>RBIVIN*l0=#ukxN2fV7R6dK4IeB6DT@0a+E1fNNGx6jRgIosQFPf7n(K``XV1Itn1Wppzj?(!Ko2!iu9Nggo8@{pt;a7=C+F*29yZP>$P4@HCi6%=v
wE8f6E?<tH_JmiWXofr#cxkR!UFkP@O<**tgJuQ&kA+yCwGF?sv6togTdZBMAaf~S&$(rMk5yLtYISs|rF2*=vGuHx0<F4SjmiS?d40+Rd%?an3g+R3u*%Au
0B|@HDY;W3CH@Vs0m%2Yf5z04On1D7=H?|Nm(WtBA6r1yWjH^jhjl_mVlw4v5e7<X|7ws^!Cq9C$H5x`F>dPmIKpyw!P1Nv1F`E%+fAnbDS<R6A8TZA>oiX&
L{YrGX%dj#RAkrBowRf%ylS6|iT}bTaajiPpTrg7N!QF!BR&6CtuCmR*%{IQK!lYMYJ0dqZFe}XOsWbd8Q0^SuF*tw4B$^_bK1tkEtcHdpnF}_0PQ-XdPJ|e
ac_<4c$t9z$BSXUZ#JHWFt2r+xR2}yq3Ns=K)}?|w4Iizw%S7As{N>iU?`^acL>O*WWGSePs?E9m^qgGQ3@k4z14nCV72K%YBYN2#YtKs8`_Cwb2OOnCyrr;
{(0WyNuQ`rO+S05`b<*I@rst<d)^0CV_zmzi<%rk2fNUMj|0%(5V~qMtB;3}A>fCS3st|0m(td$OGs7qGQuE~=d0?7<hRZAVQ^=>GVSXNV;bsd`se9XFtn?l
*-RPMH@uv(1np6fu6Q0#9J#Xc&ZU0jPt@ph8kvBPPYMlDoqy1+Sb#GNB$+Btkd#4B$C~yVHtFtpXid4KMo9LrGo99-n<wd6=(Fw0cRcI{I2fC``W6Q!K>4s;
<$c7VOrQ`EjD~X<(9T?wsoDWu0~yQ@OmR%q>oVs_44!rcdt%2W2dPP!JX=>zJZ6$u%4$Gk><~gjuy5=bZNCvaO{wHU2!v8$F#DeudpvfpHl-S;KrS|SWV6Gp
Sv!em$0f#|Nt1;N!uI*~O>gH>a^<g?f`~}e)Lqx0DJRg^RaFy>6;3r*1<M9R*?v9<Nb4Ww7wkF83gK4STApu4^YBH#n+csXyX|>v7p{t<t*3He2?gJC9PoEe
i$(;=<QB^uZ5eJ6z{oh(@+gMcP!~TufahuXTN3$5uH^NAHo-!1I)BD5kP4mp;Fjv#Bx@wjDHC3t^D$RpeTJOPWOs3E%3pvr7J)g*POI?T25tsg)daXxVyEOf
`O_?pb`Jsj0iJd>O=$PJ%G2wU3pBr<iG}4ZN4PzI!3r{-q7)1J#?NFrS37-qB3N6jys{RF0ft}(=XZpOX1yY8{bj-bmI>u}&bny(DbS>3d5T)NUBy+S$}o}b
*Tin;^+U-OKnKutyl!_FPpWzo#tIbEr00DGOE=@{BPiq$C_Xc&=2=nu;DHfk>J)?DXuF-Zw4$bBK(X~xclm<h1FW}imj|Mpz@};G#L^$D4qK_@*fyp)J{y;x
kPGb9&ez6&uM}>|7-A=*OXlNJ9EU2YzzXZfj0=;N_5T$`p*~tUt<<z=>lnoSzWZp5kw-n%8h?zdoMOF0>2e9d?B~8N;J9N?%+_#^;8Xps>nj~G0FXmu_h6Gb
mUof1yE!oSV_LDu%SP-PAs+cCWQR>Pu(2m1;qF#R^IUZ)M%JrF{!f@38U!)d3M6yY&biDEN1I-n0Qp%3;Yn@BO)ebP@$z2;=S*4JzyQ2cjIy^cKheF{s1~8|
0Mm8IHubTQ<jj=F=-z&2;7UOBM}am<7T<D!&a8`Bz>qF2ujRqwj4^E6t>G!2b|bp*K+1lZv2&p*e<n250xo<Z3`I>nZ}qYP_RqZs1rrqnI;S{`5|+wNDq5{E
Zsh4L;1VxGQJh^b0A6m50sc{L)my{-In(F;*sN?bqJFg79KS6xeV?H=9uR<^mg|7KXla*qI#m(dRE|!2b11gjAI8~@N1}tCNCS7>qv5qOnu?!d8L&-Q0M^L+
8qz+uj^#u>_%w<9m5-+XR{pFIN^b0h?$@c(JdMGD*HPcGAd28WM98qn2QF(tHF{3BNK5+)S*%N7-!gFQO00-e8i=>o96jiMg2g=J!D#qJ9s<(B`0kxPmU==M
mhhutOE|lB)oUd?_R*z7<R>%DE_gb&3?s?F01^&PAOEpiY-AyK?!0o9>+z*{(k9V)@o(=hF}5xkjV{S#Dj26LkTa-YV5blF`=^hv(r|3xO&D|4G$3Xf7z66+
dbHGv?M>{+)+5F+6|(5xwwLqSvSh(vyl9xwgF(hz#9#FkSQA&`-sr-CQ)mqneJ;(N>A8ev`q3xR-P{<;_IyhYaC6DRV6SR*d!LHqy`7tv!gg|-s~Ir)`_B^-
R45G006tZGKXO)HjcVKj?$B;8Nje&QNulgXg*gr-rzuF62&1srYjd9jT{)2a2wVO=360cQT5CtezP|^1OpF6TPR$Vl5Yb$|S1sf<qO|RS3}N3yL60fhm0G%?
jM9RyPc8@UbssCBWOBlJvFMKlp8UxgnB`{jL+jqe>YXTa<_gT4BV9Vqp`I6zsrXh&Fi^Qw&v}#Le<2)Z7hrkg33RjzpEG;*HhYmiIj7EwZ(mLqJ81~Bc`r~;
@|EppBBmeCjaIENJOwn(Ky=gvWF)4PIu^)Ni%NrO_;%URH3p=htO6!~!5%rwrN_nH-UvfHKT8O59b;YLrz0w{TTX3-TS-2ZMe+dNsFx?C94V5+;)I?YK7d+2
ft9`#t>@|5TeOSGZ(7^=kLa*i=UX!_1ylY==D}gPI9`CBTW1QQ%4h#b&EEuKoFkQ$LEUhfs4h_20xd{*o>|H1qz;)T<5{0ltHtO?@((21ZMq7hRab%Rbe~5I
WgZglOY%GdLTVK`pED7HbrAxu#?f`b_isPkp5mtbKKblDDe1=P|5)}*?I+LhJPN>~o?h&BHc%%~4V1R}wl^YHJ(|^U`JH?~ycSUD>h0A4)LTuf<OhY6akKsF
1kog6-hqtjN!G&1T0Uv915!&GnmXO2nz|XD%=@{6)`_GtoA=xpj_4E<u*r_x_^9AO9e;A1unCbfP@vBTtA7QK@zvSsj#jkd3D#4|ok}Bho1W*KW!%=;Zu?rD
cU?vyyh4o6C5@QXcamGW-<uYU&yPr;xC8H{>J3bFUSLoo^X>?p{=&v6Ct##l<$#Wv?BENN<@D$@2~taELqlhn0V~)vhAA?GfLzA=f&k7S))}17tC0`>Sa)}>
mfYl4u^N$rW)@1FHvlOQ`^!~N8IS6LB7{^I`Y9dX`{!Mohf@Z4A~oRntV1IS5436ZifVM{5$U_80@n4Q0(;dfn~!aN98q7VFvs)U4zHVIz_)GBi<#H?^PA=6
OgN1UbS?z{D320lO6Nm^3E+a%uSIv^dfvYxk4|ycEi3RO(C2=#j%Tl{*tsmLYCqF>c0-zu_nIF4IU&sH)o&%V9RnXy(7AC|D=MiD={&zJ@Z%g0vJ$QJ)gObe
_aKZc0{?%+t`TgfP$&B|^)$~gkQ*HO_eM^UFF{BacnStVSjGT9p(C!&LN9J#o2djcT?<nXMw}%eT86-UQM1JjcGp@4=$-sow5_^e?%6(z_pWdd*^~63B9E<j
P11Py{m{7Nj>~tR@7~s>R!C@m``*mq5;WJau_4K$ivT^WT_qMY58;|%7|y%102Mu$(XM!9<I+*O`Y?dylhl?-T?l_>1%&{jrZvLRFNqdL_02k`665`rmdau7
nKG3%QKWM^HA_?rn5-?TFe2)2<eU5iHZvQSBaz#Z@C1`!jshD$E{cjnK8p*JC7@|pBlI&DupE@eg@B%OUE7TtKkMCy*0hKMBhBl0JVkz<QxXvdaHDy#CNIx~
bs*h&DJyQIn1zvmDM*m2{!Uz=TlRu;@ctaKKysI%-n<hf$N($q?#U><@-%zt>g`=I+AZGqKPz~_hpR3xv&ww%uG|b|<xYo<!2*mY@g#5pzxbRluf@<oaujZO
z7q{qFey2j?nQ%y(@N15D6nP2XgFnF^w7SeC<kGt(JIHCW-FN@gJnqm*2W|wxXhoG529*7K=bGtmj{roFxg~hTU#6gTB+xC^xhLTV=7R^IJQen<!pJW<rA`a
av~@xlRnizNQf0?vDUjcPo1}73RcR4w(2DO`^sf1d)-mozoQ)q2B+Fb@;yAJx-d7m-z?evVO%dd+w_lXY#xde-e8oZ9?kw5eF`f3<6H0}c)<k#ptNY&ruEa+
AIkdY7;u&$rRU;F>1=<WssPv0&Pv8*Q<FZhmaKDLrT0)+*FT6A`M#x@1hpq!;gRh-6OPRUEM29P8P^}h*n7V{2<8SbT}X|pt!sul5y$V=E>%l!sbbC4l-S^Z
C3l`fK$kDE6^kp*`R;x=4dq51bHK<NW#Cp+gKIB(Wi)aP&odNbn#OiUm$CLP0X7oADw37Vx|TX4g#W4~_KT7KXJv(9+(t<cgYHyfRDBh}x}}>oCbn|cp^~Wa
BSSApnSy!#%xf*^!SZK#$~wAP=8}@9G9qIiA~snOXuwbu9K31lwK_q(c%J7IQ;jEPiYON>ct6gfE5Ol_?)nFrda9xo=eC9&Ahh>7(v(G@B8ezv5y=bkcNafd
5#pssfM@K}w%gzxog7wfzj%!W*PS%>Rq$@AUYL`uD{qV9Y#d}2JDhIKb_%?-T@Dw$PNtDog*wBRR{FQ8ByTnI+aCvvx3-;Qq*vFM@&{;Y`p!W>N!HY$hJaB=
K@#G#i6Gvj*C5(fswAm~A8h(KWE5a2scLv1RFHTJOeKqj2L6r<vuO@Bvc*~Z1=EBL8UPW0d3m<j9lRbKoJO>C(DM<*+OA?MC&b8vjys@1Mt&P!w!a)qhTuIy
vw5^KK)j~h)o9TX8)5QYKcfZ5Mf%2>%**B*CTJ;D2Kx2tQ7HR@S_6JhzK2PGBvXyh&prytX=@HKcE4E%3SM5cA`x`}cNxw=axD1Hl+vXp92nGaVBX?y;_nXS
Nqk|kIE`&2%zY^(4^Le-1&Z<BEPj`kyun0t*ktjT-1k@zOV<RQK>9IL#_5}N$N?OjS6B}ZuZ+)Cv0B$zuWkf_!DV^J#xC3cR{8UpEH5^RhnJoR;Wi$p#ljX1
=~g$@(vF1E&&S+7*u|XgH*JZbKAU_0)FpBjG~sc@qym#aHQ9zfy*yCDYTzQ~LZ311weQa=;I9Xwr1G*__>5?YYis$nNDkzc{V5X37@4ZRPIO0Y@67yXkR$+y
%K*6P%)XTPNV;~(xhFK5xf(t6dRnsv-~b(`3e}OQ_AQhVt(xJlr1$-zKcN_f;x(LJ>br`(pmn-ZEIy%muW7e`tBmBiq(D`pmx{?UA1;_))*OF@wTWd(g<nqQ
jWRkfL>_5T+_c0S++%T(IXFOAR830LX^k`$usOzMy9G;so}I4ce_nXHYi^){i4m@ADPM3(J#tqaC}+l>aaX{a7o#xj#Uth!>O*->iEg`57gbf9Pc*#u;CI5?
n{sKcz2lpCqeOb;zsF=SRmr>Jp1N_4x+`_l1u`D0MXd?dy~X+%m7FYJF53p|fMaS0lVj`yX5AT|pyiarxh(ErFZdoj=mBqe%e9_#ZJv~dadQy3brY-c!9)8p
X{h*F1QXvZv-OAosx2^%%hlw`B%;**JFVXwJFI%w{!?)>_vbP_uUf6~QX^T&5cmt&a5*NEx2j;ro&E%Xyd2TF(nN^M-WaKVKPH6>XPkQ$?tXG{HVT@Nb)5wt
z}Ypxgwt^i3G}HQ&iqh_x2CYP_1t6!u@4S46_YQ1Cn{}_&}+ZUJG~G?Y77sG?ft5j0{3_Bgg^mdWW}6a32)dKuV%Y8ca9*SR^F20hIuFSfpV=XA`?0*7qz^3
TJI=DU87Po=bN{`_5kI>)o^(w*(p_=)bebSxDwgis-mA9z3S}0h=NM`kV$V#h<%FbszJmKTq*~H(X>gF{Su$>0KPV}ADTf-K3RkTvo+95%NY{CMJc`9jnmad
O`4wlEUr%>Xu;1N0<z)n822>VXI+%eVlPmn!R?Da0^<JwVWS0Hb1`A^=`!%c`-!(4G@&dGNw;$4ErFd0_Ke<8ivoofY`1Gkj%Qo7ey`M`O*Pxs`1}RJF;^M}
sE(Jrt>Bs<N%T;5&H3Y4a%Awz10*<ZG!M^qK2E`;M$5<{)?(E|*Nl{@tfa*ZCG?wL$T_~hwyH`*aw383rpVVlI*~dy$L1^li{NryV=FCQySI!ZR?&4gbs;m|
ICv{lzsC~-y^xbRCF1-el7f(5VTvyN8;$tunOLy@%k(UUTP{Kdy%G@>Q{99jfF}v^LjS(tjH-ASE~t#~QO58Phzg2MhX81Q65o)gk;~8|M6SWQ%ijG|(kZNt
F^(rHyp4oF4DuV;qdC;2z-B8rVV8AMWJI9z!j)^&E6@~fasXrVjq4h^J3Ua_m5;(Me%hKgGqhDyf@2gq)SN#mPr!ss87)DR6Bk{Y?NsctpghRXB~aTni(DBi
U<vaQGXq(-n_C8_&a174e>XY{+S~hV)8H}0p(|Py_r}A51ZGMoP+<6ep<lAPTTIovR8%<)mxuhq!K4Nk`Hj<U>DrCQlkd&a*(OZx#A>FK$x3Iyvq#i`j{Sm-
-91$JG~S{;v3}*qOQB!%9Fy{zo|R3D%%O`yMc+r)LBNt>fVfSVYmD?HN9q^q&<1#R0s)?Kh78Oq)$KvY@4W!}Y?HWr-y;4__^Gt2%s)FY2u8iTdz$}^;4-?r
6UwNZ$7#^^4R%9+)B0t+fHO}!5*p)vAG?CM&WSwy!K|Kj`H1tL$rcRA%fe6b87FWUAhu<BqmlcOK08<QgE>(G%qtlo1TmL(?<xUlVn3cFj;0$vjhnm@@g}`Y
%v*#yENRsyggIni)(Csz4JA-y>O(mdYQCgG0aAc!=fW$fHGAaiJNU!&DDo4gq`i`yAW*LMMINE+@{75^#5S3_lpC^DC$kR@BPTu;f};?-#rT4FQcGmmi%meu
bO)-%ToG*eblDH~p$?Or+09frntcS%IIlWxKUyGxFYw&N5g8mXA5bmHP(iueZDbdi0)RC{G5;DNP{L_gk&K{jV}>~llVkD*L^HeSkg1wkY}3t8PJu4(GX|bF
E~ed9{?b?Fxp$T6FYRA;D_>-#GgbNT+|$rT8_sWc4X(Kz1`Lqjqm}^Ms5BqiPHf^Q<%o`rJf;?FdG7^Gx`2b*)38L8K_<)Hj+);kFE;FII<8<8rBP{Z3mWGX
J9ivE2?n}aR6`t!8w>U3?@8xVdOAj5!QGN?wL<6yRR+$RstathD}f|`1@T~Zn-^$K9qCM4h)x3IC)s9~c;&t)Tukp0kE%U+U$d?RS8RdAi;+Z#uGXdGV?MN<
!dJS3ugoq_Q2px_bV5NP6<(CSGvzop8_bV}%VDnB$fxm%T1lT}EUs8wb>D2@m#w~+@BcaR4Ai-^e{N<PLh|1q2zq^cY#u3r)DemEbRWHm2mA|4s2_N6nOWuK
xTO5vsfzM^6xh8g730{u0>+WVO@e0Wp@`c0I@6L<H%!iL>ccKyoB#(tFg<o9x%#em`?s(x&*hpYyCjEw&#s~WugziO_Tt-xo*?<&=Lr`T`94!6w-gw<J=S06
8ws0C!!93E^mCgqf9+g;ytlQU|1JXsPm53xxYEHh{fo4+S3|7$`6&<x<IUzpEj+a&QGudt;`Ni8JMXJmrQxEwDP<d~*T`u^T62C172_IuofYA2x^iYQR<LcX
w36B9wuTG<^>`ODpbt6H<_q;uJ%a?%rdpwzY61(Y*ASajEzFMaBvQM`JGUSucPm7@68M*<(<sC@+|(Hi7bj4|mq{X`xVzgvm}ljOJ58`3GVLs;irN)C9|c58
4e6L0K--2jh#ui6k#Ny!b>V`GAc}N&^-9F5OB?9RZi>nQ6vHEiZyU7{y$pecS^@YOPU$4y3;S752z@J3kai``vnd=8%V)@SH*O~2AF-H~ZaMJn8v~8;j3k+Y
UKZC~-J6v#Q*O}USarVgm4#7U8`_q&pjYbb6m37Wmlxv<imV;1pC7E6E{I0cNpvn8$R5Df(Ek^Zjy<e55`h^*TrN|LzT+Uot+HD)FIv$z%H7wA5PTT$kXfRH
^aLVmS0cZOn*c7KjPYxi%8LJcOoDwdpXu;`E*Prh&$zWm<ph&~E#?VCC$*-h!fEPja1v_pMS26!&#?1K?<xHw%7E-Xt8V^vZ!m5spsCBK&p3Ej7cfn>!STRp
vZPC`g#6nK-}A)fOXBat^96F*mXxmv|NJ<}%vV8^#Zbb}Mrqh=GZTj?%))(7*I>wQYc-i5HJ?LX$w#VN%KEngMUa$cANTPT=iE#NR<MxKg*KAfInX2C`-pmJ
Iu-L)$!`3A@zK{`tI!YC7i?CX+^YXJ4HXL&@21<gL+A1fOE1Qj>r`Nd(8#QFOtJrH*!>=KQ;S)F;x0AVb>}Z-_eM6AWxVT7Hh8{qC%*ZevhcI;9vERJ2wm`B
Ms`{6%Nk7lRy{xPx(hboffnsFG$@u0=@QOY$fxb;R*@amN`=807bRQ$s*W_j+AL#JUh4bGtQ=N0x}}z7>&?@AFpAoi^LO0jS#F1_a@GymKMb6ioAu`WCq7$B
y(>hA?yZ$Fp9)CF9G4joXRpwTN1AY&vuZd=$5MdP?OnHcYhGQ)GdDUX__d5eYo%nMcT)4n9t?5YT#F_#g~~5Vlj4VUADk6;=IGhfeK1xVj|vDK7N6r(^AkhR
xCARAODxSpbDV-z@wlsj{x9{3qt=jAn-s}l;C}k5xU5cNV-GRkvz?(tjZ=vDeBlWw*8E1L(nXLZ!xUJgC^2nMqIZVT;C%g0%=+Eyo`{;_gPUB?YrLB@oLM1*
5)oG5Rd>aJz_V?!?!1BnB4rjGLm{QwpcLx{*~@&B%RS$0Mlt?Rv+=mr5W_F*NJ>2W**5MOMO!!NKzN}?#dzDn3*!71ROj+qKch^^JeN@f19+`vKb=t?NeTxo
+SJb86E8^FXpE&t?+C*P6~Khzg)Rwme(;8_T&-ltbO(#}17LQ{eW%F+af<k+&maT!u17fCC|p7*o&@dMJz9uJ$QNy*F~ODVB&XzZcI4Nxd)9b4I9zu3tc$lm
QZ;%-HRV)uM6~pMtmClbjiXjD3tF5!Ov9C>haI!e(RzwMyYpq)C(Wips7&tsCk1U7m5&J9$%j^d{zTDh5ktkAV&b{1%=h|Jw|5s+Gd`No75W&-@JLj&v{cAM
vF@Y;y}-e@Rt}(;6A;zaX$CSErySYSLHcK7DNO>bcIAZ30JfaR61+~fsJxs5Q|Bg0j7JWUXksW)W<0x!<EgK{yN?~1r}DT85q0b&);2l#`#>YcR?xa?y<dM7
ZH+q2Z9HCc9%3+KggB(DN*h6><r2}vINiV%*7+)$ji~$RF7<HG9qB<9(Xm&tBod27u!3_|21*&*Q>No+1@AHzixx~~7-S&j#F+QhRha^@Qzjkr>yI+ksO{;V
U-|+VB=60yOYAHIkbRV}@O3-1lCLeZm@93lHs(*DsQ5(*lLuh-FB|qpz)WtySS>Fq1C(8iAw-Uv5z~}GRY&4r8>nR>72yH7wL!-n5&yxdYBuFwec!j?3sy>0
4)12QJsp;Y=1PaZAHMJIG0-K&Xt;1P0ztPV7posMx*j{tU`JcPSrIa!o0svy1gO2uF1W!XwS(&f5nk1G^#+q%kb|57n(G&|j(N*b$&QyaS?gaVXKzvUy>+jL
BKl$e%7R;I_ic5H8=tXViL8Ygu;TkPky%2$$iuos>hB78aX;VzSbkdh9>8<$0q5p?YbJM{I<AW7LrQR5C!5RZ<neM35T1S?d(tyQ5;WHbDgDGK!rNXhAW3Gg
aLe`}PKO`mzZkDMCY^e53H?;xbYhfyCUAcqeWnK&Jh;8cn4*&f#+dD;)#Jm-xLy&<p?d*Y4}#(Hcen-a0*zeh60j((2y+n|E5w_3u=m0`7h<upBHcAnM$7{q
-@+Z5zQ50?$uSw*@SIJet;UvN<P7h|dn+=r`93EY%wD@wmc11Pl4Ixgh8hlql$754ctC#)4*55|+4TqJd>>^9lTQVLWg5<>RPM5pce_1~1`^L%=qnqwNf98%
5mqyQnYH$Iumum8_8*m`k;Srrg+xIrgRKE&r2`c(I+Slu@a{DMyf3D;ET5E^^<Q&eefwX(!Hc`>4|92YB_3{rrcN3|u(uiGkgfzCV}yGXi59~)4y-$$MwXwW
x)2>ltje0q(|--yUOZk_`NQu%WOuY&*HYr@P8CM2&kjaj)qW9Pb@k*DZL(teFT9cc((Eb7<-JN8xg+e|5VS=8NZ4pxBjbGmDse7wFI+3;_r#etkF(?}4w^nT
K2F2fThmGuOtd*3OcX;u_y-r-#PAv^2RUjY)Nvr{-ZA|eV(p!u2S1Vdb0I^^I|rmlSO)}<gp(h~DDpaKmJA}vUvElm$9q4^g>>mDlC4Tu1M2o>8}98g(J`IB
2VME*ZlD$n@RyNpZun~<n_zz-zlE(JW(oG?z7g7WI>EfTPo6(7jlM)YS2u)73l~s%YlafB+DYH$!-EY^KA*q7vxbIu%os2nJgTWGv^Uqv>YVDBy{G2av;CE0
{ya6ic;iITP-<<pnk<YNfzGBpr}2e^PWI)-+5hJ~w?QEtoPBX!40)PEp|X4O!QY<OOMC--vRrtQ1RjMl!9zrR8E~SdU4&UdToJ%yjm~h&(M&N7F?9a<gd01Y
jy^X0oqBX)*7K4s9kh!vu-BzYqVB~vX(Od5ak<7gMn8~aFZ5Qe@=dVT$f1T#{c0u<KtT#=X^4QPs`(l##73|Y0vjFcLKHZm(DC4ae^2xBI;4+aY4+ZCs7k`2
12|Bm=(ulJUtaSay+=Av8mp&V8%dUb<N9_pp53%scZswDni8}B0g=scIf+H;O>!#+OGT%b4k4dUmwGZ`B$wU(@c^2$h}nM@hEzGPb;uF{AaQri<j`$eQ3fAO
c7ECC%p)}gzlk#;;Br0uMYk18tz(Hbq-~Z5Db-4Qr->Zk@uV3)w+Uj0TMMGB(<n?ezZGwBYu)d+&eH=l!1SZ8sI7pz)szebBfqWaNO(b4IM)UJhkchWIkC|p
W>$yTHetBucyVNY&ydg86fmpwnQHq*ppo@@8wh<im`6QPBBFtzqWFOn>-y2DzZ0mK1K{oyd;rj^<oxd(pfUI51ZuQujDA-A>CUnHsa`ix3uDkIS(F<vQ3)r2
IYAGv47ml(Mj!3ANK?<htee68iyej|st;C)VQ7117JW1-4?MQ;r7`R#5T(di$zVR4=xz;|?QYuAZMeV&OOz9-<Q+gF+NAph*p6-6=bhv(v!`W62GuIbc92Hc
Gda;w^7i9NW8eE<XK206npiEE8%pLOy$nKJznJ7J<%ttbj5=;)SJGOsh+~1fp0S!{&Iu_iE!KD|bHHN{S$kbe<!hz^A)qx4DD1;FPf`HMScr&K9(SgorDT0S
t7(T%FA$?;o(R9wf6awSG<Sv)Kx`vG!H=UWepxk9`H6{%2HU(Xu#&g21`LM(Ju5=!7leK8IRthR<Xb~v5dxle;UlcM5f_PNRX6?1ryW<EFvBvBV8RhebGY+p
hV;9SI;XwiEuSNUtfd`4Mb~|+rfEz3_?Z^XxJYw>?92K~|7#<bX%a*8c6W*K|H@{RnnVn1$_B!3UA8qclmW!l;3=QJ5@~<ok&_Wwu|k$%Lc%1hp8NT1y#W7l
#M0uHR$)^fd!Y|Orx}vHag<V3nx#XYvuYz8OJv>+2D<L&Fcaf+&eaCtm-pgB-Vyi!vsq@IX}lc-IrV1CUr$`vLR}TkOy25P&J|eYuR(Db(dA@%+{tfH^|K^~
p&eFva4;^Qtlx?Q=0Gnu@pKVaExRD;9tcFtDq~iq^qVA*KP63mFl!>Sc85-^Y~SY*8)6;6`y`H`5rB9pR%e<`w*(<ufe!CNfn}!4C%167^v5I0AdfME^%Ox@
Yivxyb|aAG`X4s2cPp*%>D;m6#FOP!)_}@N-t^iu+F!kUeJ`Clfw9h&vNBw{_3k>x!)33=Hlgec@B<%p3*R|W$$ly=Nw4Ol|1h*0;6eXCb7*Zi8`rQYX~9|!
wpHYnw}^==`l9j=^M1{CM@chU9g^VC;V7{H$e$Aa_C+3%2vuGglDv~CgZF~85^t9-j+>ZA#mfZmpNDGL7i<^Qf_B)RmCyv!;Mrc8;||bX1fqJ>5vjxR&GQ4Q
v3I;25gfGW9E}(lWhpK)u?(rk+f1WMD4_M;P8^zFG7m#Pg~%{S#f;tnQ&9Ex&3ip!xD(d1E*jztL}I?MoLuab_jC-EagSsOvspa`VJ9{(u}v!4;_iM?n_!MD
{;+LHUrDl%_xc2qVbram*A<PEFrsXfJ+8@7KsY2~?U+gsv0&eU5hsX5rdx?5?q0n^#BhprK}^A`WenCUsKg~ct9XJ-5p`DrltFjCrNct>Gfr-hr9vjBmvi@{
JVN$O`q@w3&@)CN>srTpB{MJ-hGQkl^991JpKRv^{*vNJRzP|Wr}ak<3D9DIVrd7eWLR@#O_PBb!cX*I%-YM>rLM-;`^Id>>rgs$X?j4&S~4xkpY1V%`*wUC
A*1tecxBWCrO72`E8GdJmmStK3M<T1u=N0I2YXa8Jc?PI)IlsDXfcj{Ox54Ln}@Bn(bKCrx-?_wjCR^qw_-P!jyyqj^oQ+s#q-B2Bg?M}b-}D&=q#=B5ToF1
Oz45XTlXnnLUb3pk_%$QdtW@eeQW_bnd#@aNUS(kfTw#98rQ2_Zd6zk`!(+FSI|mhvD-ROLRu|vrTKsYB%3wmQ98@kFP_x?%12N2()H`{nOC>tUM3^1wMGQs
*9($q0{R9;U{=o$@bo0s+@dJ_(3J{NL*(ViNTS^%v6LUo%Q>#g<T4B`{_5$1R0PL%fCtxXp<nQ4!QM=E<lv!7tTR(*0KPqB5se*o?r;`B+g%5}6ELhljymzL
J<f9={xg^#s^;u7pa%++ilc{G#5{Tq=tE8PxJOQ@(64#_Z{LJEdg;|gdv5?~oD%QT2J;S5AgxtZ>L^P>`GS!m@W8)=_`Vmfb9c>F^v4*qZ2;OinK@}Q5G?5&
G8!ci<@+dsPokG)BDCZUB|oZ8sNM(j3I1c=EFSiQey@1vb=s6)+#{Duxexa}Yi4g=(Sk3368Jzi?(!Qu%2OV@f=4tPhc8~&zdYn}j-c#i!Atlc%96htDHQ#$
)Bq@n?$ZUO$pmy~9&JLCU!bG@d>Lun;0xrd#WEnmgJClQ0luria}osA%Y}oW7Xz%jx{Ft0jr5m4Y!?8FY6rqTP$OLG0X$7%xvAKY@c@#tS?YK0Us~uobxmV0
aO+TQeWm^jw|3a{5l;ihQxfMO;?RdPqVzVPwUoF^(R}yg=S=(K?;JUwPbzC0U)#~8rBG<e?xU#Q=Sn+S`w}t+k{nhr&`=w-m{h@bAUNMuFaao2j%3B>MC?R0
rPUWj5_6xBAw?2@7ueqE(YGWp#EV%RQq~=LtgD!|Y^VtxQ_)U?1ocM6JQa^LLAR^_HP~fIRNuT5xFf+<?2&A}(Kpu#6bXoJeQ=%dI7IMp8d7jr2iZC(RLr3B
x)%fgB`{C~Oy9!@C%ujz^EFtnj{1Fi=fVA-hZxe-ZHu<3;xhVf7$%S97ICl|LN@7IIu6_FYWr+IOYG^lLS8u$#yX1pQ4b&m2dhqcfb*4T*w*7-)PxiLw-p=w
@0e$>9wJm>;75N(ihquD?~dkt<-54YiH)}K+Yq7#Xf&SF?6ze+wWor<*|7Z#XrjYl&9h8+v&sT}3C$5X2an(EWJ+@xnhBN1$i+ft*RaTMcibKPcmm;%tk5%v
z<bl1!34#BDsUJl0+XAy!~m%1@>wx1Jf;lFF34l;u&{436sS)O@f4=S@4TL_H?E1i*dT!UU7nN#*o|lq>u~b>E#OPIhkLVTBjbZEF=}LssK2*E6?uyjxznr*
!mU71Z%Pxfa@|&Am*2|$f5l?w-_4F1t4UbuKD_an=Z#@5Hu+P#DKc(D7tnNOE%IoUw=d7eq#u4nkt{`87j84n(jx-M$`1{dFeWiSZt0RJD+1T0-u#2*OOG3-
*GIaWc|2n2Icc6E2m@9%{ol9k_4JNKm*AC@Y!7U5$4Xj~74(}aCfTR3gxq7G5JH#kRl%=`{4Ge$*^$eOI})S`Q~G672^>Rf0Kyqa#P$;<KR_|Nz(b33faF%+
adFwt=^@=04OA7)!j&}e0<cI}{&$}4jhNcC^lU|&i&7#2I778S@~X2L=Xq_xPMvL@h3KY`Jw3Z{WJ<!3DUkw?BEF2ibsNO^k9VX0ctRgLX^adyS-Hpvmi=Lg
1V)3MifC5X_>yR_ATugL{MER7WdScw%W|q~+k~z-lRehX>e>nsx&wR$0I4hL8FFyJ&%Gj)jQvSweU0z?I{i}m8TzZ*poTB#c-|*op5dcn3n$v{$O+)a!b@;c
1e3Kp?@IZm_(r$Hzb!&8h$INV`7lNjO-2`UpMd)_`qp~tCLXP-T*t4%zcVRsvuGp3BYs?4Qno87UAJN=f*j22SNHj=Y{!1bt(%cA?GM$)p$y~a-}fY<&DLJj
jr#|+_HLcZb4Nk4lpBDQjXB`9(FoR`&aG1|k#?V?#9u^(sjt=w5_1$%<UZ!>|5^&RTCsA)NpDfNZp#aE@s|1XB;(lFcTDcPivho7xXJ{0XF6%d)t24GpIk^y
cHcXNm^C7ZDWbdw->EFp#)kaUDg6`-^j7kc?e*Iml$YnHp1a@BCM9ISS-Z~CFgf~vcaYc>P)JEei}jO!L>^ToYmYVBkakP1G_k);@`8+@p@Qtqzlk)M>^SBo
^QBZr?w!a?_#EF!v|cD7S|@lw&P)_K?ptmvcDT-~w>JX#mj%MQ)7;+wgo}J$WnXSyG6q-`>aT$3eQ$bSV~p3yfLwxIo8{f|;&P1`<sfdEYrNkn(VB<NB{wiS
pHXAiT3xewtr>+s%p19cKDhGBlOC9vc<A9Y@V0|BHQ&#-0c;+?TCPyaQ9JNV5B9vvD-9XcL#Q9Mnq+Xpx99r<IRUMhQ8_GPiWVc|MgN+}`50k~ZA{P+Zio+W
U_yOI1`>9J$en2=MEER+>mBeh_cA2@;am-uM`9ESv<bZK9Bc+W_xdF-TY3eyvxb{XukNBPmq04mu44xiQpW8@#h2(BzC!NZ&nRoWlwfv#QPcVils~3Kf&0Bf
w8*c7eaZAyGS*qVVIOrWZ!BP+B*Dfj&|Rzpa93>(by$1dCZoIwkQ6`^Y&yJv!s`_;H-dqh-~IJL%3Ho*U<T1@7CTeKaT>^Y5qQwMEN~=M-T9|{*p*$Ac1Opj
;Ud>>B2Kk#PoO=mN|TZd*IacJi`lWEeP+0rw#Dfv#Hh8XXdn8Y=6N}ShOn&NzfVA4qFH^gASi0j#@pKg?<2sjD4@c;03q{(FR*1LGi_!(lSit=CW+T)t<#{K
lo>R7!-P{(->0*#exuL6%NF-V?ckbPtdVuXucT9c`Vn(CuIW7856lH9x8Ox=XJAJO6zM-p+<qm_p#fzf^PemU$ADSv%Cwp4BDxa~Qv-z08HCeA{gngWtQ^Hb
uBX*9u?Cv}hUf@3M_`=VXV;b>jms7$xWnBDvXY&6!jXv5_P87@q1%>3(jK6UVMBwMaT*2AwOUJDni2)JGftuJ9Lha2P`ES#1AS)~pTe4Y1R|#`@PthtJ^yF%
bM7x`pP}iIY|NVIQRJG%!p-NDnyX4mwUd2={&nzlNbg;J&jUPMf~pV!xX%^);8F9xU9F2rI8o{#ldmyl5n+=5MiW=m@iLz_ac^q7iKV%QoY0EHrpV>0+iINh
0-aN%A!?MpI;a<{_t)x34!{z03*L7#kQGr+gGQmQSu3bRLI<n_`*d)GuMXVHl~dLB?XxhBq;GJfW7KA&>TG?d!H^6a2m+B-tqwUm`j@sp1G%Y!-jImjsO%ql
8*hc;^{|lZZJj-y#+PKd8z+ql<Gz-b048F2dK*Cs2GUWq8m0mlDtMvqv+ND&^OQ^ax3>a^pzP!YE7;{P#1NEu$wt0k7bnG@uR*?9`U>lVM==PsEJT8^fRCxZ
i4v%u%B@~apQXFe)+UEpTfT~gsr|o$GK~k-<6&B4%f*W1iE0L6QPAY*k4>!KL<a<SV(Mh3jX~ipt`bz6XDihyqbDybcpHZ$b@;M#=1l%lN(89}_-7=jKH@V-
@7eoaeT1Uo*xBE*DAh5@@-=u3gep^#xv_zymTjUvtPnkok;qL3RLes~@Kvhwb(ywZVpyV4yeH-0ffjKUoZn`m!CpRGI{u8;cVPp!r?Si5yXBnb;F?L~t#LYz
`sBw#GzZ6y+3S(dkjOs_jTPp*Z%y4bO`n-Ri0gd^5?qV?W0qkGack|45#hL7VFY6`{Na%@hF#4(>w=KT(cH+)fMQ4pY~|DQYk`|kWVNs;C+K1u`l?7*g|HTZ
jjY=-e)*4bnj})1BX7E{m0O+{hkP&LiJe{%g?>;|d)Ub?)3UXP4LD5!Im0|tX%FMK4E8er5oyf8d6HyFrh1<}K)xBYlLR|%$d_stI=>6}&x);`?TzUU%51X&
CJo*hT^(4s<kS3yh`TBb<!&S1mGFrRViSV@6+`a~lXM7_26)9iReLyXIsE-1-C|`K&47r~$Tr@tbLfqj13#)1Zg#nxN}}n8DCKTlcKoYXIr7vJj8G<sN^IsG
kRMzaG-1P%IbN@C?NDeh8XStSTNsKWES{ZvxckQvlMxHYunOc9t#vReD_d@UGlDNWXcA^Fuc^9MWdhM4!kU9#1nl!S9((#@WhjqBG+Sr@00000^oY%;v5SW4
00EJ?0gkT^2A7ZGvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_lefood.csv')

In [ ]:
# listas/externo_mensa.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_mensa.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;Skdi9$f%38cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppcnnU{7bdih4?B9z!pNF=Kdc1PQ|U
QJ#v&wd0IhSXHo?ycwa@(Ot2zx7U?<NkXM5n9#DM+4D4$p&uYl1ZgtI>*ILLUdZ&}+!n*kJ6Nso+Gnx73$TxhAy?mVijvJp?=iXBe|JZH=@;2sB$~B3FBRCz
LN}+vs;NJb(%ip%S$_;G7Yzp#Zm%HtN_l>GkbI+RmNcC!+DGNl9dTX50#MrR#YW=bqpX5L(#C(DKe~`ugB>%)(Z_s2ZoAmlMwPDqHWHM#$ol>=f+FnbcwNJ7
3R_B=)oZdRZTyRt%Ti9mJP^|LGmBBJOjbM>pduOEa(qaeLaG=LN?}~XE;CFGD0*R6^!j|ORSBD9r@m{dq+8(zrg_!fmZnM&KG}iQFE=kk>~AJZ_?#0VKbrq=
MRK%VC%rE_tgIYC|EZ#A*J-K#wTjlke4FRv`954dfX_L43mGwNngRq>u~c>vu>e|Qe;rSM!XR7P3)mvx6Bd_b<<*%e0$61A?&3Bzt%@)nwXb^+J*El8mydR3
B(Tfh01MHc>JJ2xYuMb5Tm{}i)fzD?kBh+$9dl2Maeyiji6o(tH&UYbINnUZp=RoX8Bs9IoTuB#$<CNVhFmerzf=7g4aWd&+Pw&dPg5Ag*g!zq+cS*U1*NWF
_}nN#6+l8ETVRHuaE?aaXcr=;Vc7_<nAW6rL>oT1_&TUb5~l$RUz!Kjv-40}ncaJJ*Ln61JRa(dhcm^eI--J~;ZERcF(NBvd-}N@6f90#bb)MdW1;?sjSEF3
IRxu6FiZ-;isfGvIPDw|T@JHA2bjd&2M5XgN*2MAzgF1~KHd&F;UP`_6J&_RJ1yW^Eoh!E&q7vT_Eo=`e^>)m;d!RJ84LE&a|!3$G#IHzY_<vo4y8#oz-(>g
@*La~TAhK5FqeZ(ag?Z*wA_5hB(0(>i+8+>pMOJN3#7py-(%=-!qfZH^#w-C)wd>|3W5ZQX+L4Hv*gpc$}n;?<>6%8j|sZIJq_DPyBd4#HiRPk!MNa8qnuY@
&6wb$Rk`UFmyFIff->#4L>HSwNdw}TDR5`;?O`b++|ujkXs#m_pzXK1(n^{fKW!&%FS0V+u2B)bcM-$pYQlc%&$S<uytTtOdIS}f=cf%y)oQ-b;W$7>=?={4
flry&%vun00~MHwaFMG8{~McjfX7~18aR<bs-~DthOl4Vdosc-C@X*?fmp0gA4ADILi>Vk*+8&zxV_Qkbs3ctM(f++HQ9cC6;65;fC#Do<BNh%@0QFk<~1fi
M~z2oN3s4KInS;C%&H|ydx}->$wZN=ld0c!*&HAu|BI0^2!<(87V+bGT2rc1eLfV4M)p-`?`Ul@0s}6^S{Xsr4rW$osM=Ji!}aXy$`ki;*WBC9sHvmQ%`UgF
DaMblxDZ>*h_ZQ7q`;g?^^o=8x4%J5?^gm_Q_1g38<P$9mMM+!fviVVWndJL<DC062xll{(J7$LIu=+5b`fZozgJNZrVn;BMzT;=UWkOB5wIk`R+0l0uC#`(
sL=B{>j);k%M0?a?nm%2k~6+<j+UB<)SBhOw{^&_C7b9fYdH1uwAF?x*OS&2S+#X9{Oyz3>z^h+(b@XQplU?#3ycF5s}=am+TA6%Zq^h;ODq*-(k&JN0zA4Z
xD_|)lazvmS4?on8<rf|p9=KO<F=aUzZ$R@!SUQaiZlWQ3B@v|gvr`nCyR-p5s5!Uu;ll@;@odR#zh!-Px`ANxA08UX}JKRo6@~!v#tO_o50uNBug4I%m=0(
P?$u1y9)`jW7}s)Y;a6)#fR0UMHJw1ODmqap@$mEDm0fUWr&@L5u$I>RZ4ytf?&@q(buPv_g+Hq7u!RLP2$mn>vu^nPjo<m(sjNSUaR8L&!|6=Ts4(WsZ>&h
+0)^Z^+E`wbC>^T!odx##RJ?bo9>|&!AEdM>Q?SA!4S{OdZo0t0;AmRf~v_Zk;afn3ujHL=wjv^!AM8JmBIYOC<#@%93+8_zh{2qaq<r3p>gVYUUXHhk+_W#
q)GuzQxU}WJn;T*2Ld#oKlk!gBtd;DwX&U_ss5L?U%=;E4QiUJSMy-EPXT`$*{LYo^_}-uj4e&-6XEJR0bkfSyBXEv-Hc5!R*CQ*;$AS4K<#mHnOSW9XFW`<
ia8jdJ>3q}Kb1FW?EAkL(yfP{Bd*CK-H3PB0o=TUv<={yEnO%B7jvF*Ljz-B>xi&#sqLvUhXVQjDp!?@zvouHYVH3kX1ez?I)6OxGcF+Dkq|Apm02yvRSWM6
uKbEren2M;%u(#s^No8|tx2(6yI+a$^%=U?o}-M6UgS%|a^{}5%ibR^h2l+~3?Ea?(9(qeFjK=Z%(VM=`HkM3EOj@^R3wtCMD2!;K9*4*I(ouBmelRSpuvY-
6?&E%@dw#7+}oaE>V-ypnTiy_gNCObq13N>T|^ujZZu^*Q0TW7W)4o;_?*#@hFp!qs&m4Cd2mK-2KB;Tkbp>NU1H)1cCV6m0L`8K$#)sf{wBhNZ)>%Oqhjl#
zFmlUT9DuoKTj?)<s!vM_~YY&VUl}gGN`UidUuf2yQmx_8y@Y}I|;MJBN=|eQ5hC*3?5}#tV-E{MgQBt2QAAVnR&1XaG=W_K*ZABKWsoRnk!~TNDOqpce14I
%AG^m>&IVx3#Em|Ttdtmu{o^p+ldzj2=)J@I2l`qa5IL$wMHq|PxZt)N4d`cpVC6Md92)(`SbU37v9{EF0Sl?V*_6tBWcW;Pm}4XNg3cCu{IlhY*3t%9W){;
4IPL3WZ}X|ilYf45Rw3WN6CGRMadqZBPoO^Gi;#54?s8;f>$6Jcha-ZCJm3$eYK&rMr@d*VJ{hZ(;7K{QkmQzH$-uF16VUv#Gye@dW|9usw4jZG7|ZQqL?X$
qe5RLJ3&^+pa;0Tr;3&EHDF!2@u@WeN4R7>df+gK(Ru+@3VPT*ks}A(2p6l8jy#1*qndUwpjDWu|E@8Ds%rYZ%v$H`ebcX!d<~pIop$LY<%O~HL!Xxx1W*a_
fM)1~{*P<?I^#1|ZABLg=?b6CNhWu>43Dp?p4&_*gcq33POo3y)h6X4JbPu|W~Yu(d}cG8c8G>;u~&nU#W3A_?h;fGM_f2MtJzAJK3Yj8YO(f#*%473{H^=t
r5woN6bw@?<Aa42AJ2JwKUT))CQM7@ZizcO`+-DB&&W5jPiU@FNZ!Be@S53NMTQ-JlL151vcYf)$XoFCKd6iGJP*d!zK4b&n%YRub&ru>^g|k-R>NA~(Q*Bc
m&^kYRxC93)%<K!I{an}!usxy>8;JW%5h);ILK6wEnTLGUD2Al!HghL?I6nRwrcF<ooG%qR+9m}wR}OXAgRnM{?+baJH*bc83+)m8C^m6FJ|c$0tIWGC?Oc4
yKS`4lUIjAu|SmwKPtOw;a_=gvunFqk&6tJZo%uhmg`rhK5cnn;p7(+^7+7Z{S=V+QLMKL&JGaejh(cPQ3A=}@rz$R2RN8<d8c1@iCQUd9l=N-;gQ=)T<RR_
go_s$SAn(-{g)Yp2)m!Ih2t|fEM#Z`>2pc>Bm6`*5k$?9p3V>U_OZv{V*rWv*oMJyQ56z{0Po)Q7xB^%E>Mzabz{{iO`x@^w3Z_t_<%!}!^x_R8=(8iTOC`6
YI{sd8>O9~zNDpPo#t<=pvmNRm)kqtb9*F^U5%>?+k@~FZ}6NMCJasNeZoPK-?0F9fXJu{lhS$%J!8pC*?#VqL)Nl~m0!-F^*CFRoKvdq6au}t!2Nc%PwhA(
REUHhGvAAd*lnVaso_F>27oD{>ytDkScyEZBwrXMSv;M}P)^2i6aIdhhk1WJKg%xdmW{8sBZJ<39$yI~MO4piyZ)a^VT0A(CC*-(OF!Rvopz!?`wmIf^jx|e
q&bB1UCg$_!&Uqs8+?0?2`-FbI4Isb`9eT}h7*}@ay&gPRe7|+;v$DR6(7?nC@|V_xhDyc;Ol|QA94&HE>uC}ODOYWR(ggNfGBQ@SQlV+9HsrwIWivXF(1*(
d9~h(e~%Eq8x-rH45&xc(t6gG<Cu7%l%#FhAzl_CiEQV*MC)uIgL-NCYlU(hO|v0IKTvOUqbGQjJH#oBtujL@%6&ZMYI1A52C}J!6YRQmvK+N5z#4g{AM$2S
Y16E#*pXyV#k4{o4ecag&#W>`i*Wuw&dy7q3!!!>nr;1*DSB8keIaRKr>`(*rOwC>n3|>TT`Q`P_3%H3X&`2ksv#hWn%JFMMK0H@M(#a#Jqp-B;n5#>k}+l;
dfaifBo>5$7Ba<;vLQoOnNCRqg0iZlN7JaZ6|5-?+89k3+MySHC;Tj~`hj9dTuo)U`h!y;x52IbtTtK*i)wZj4$G-mQbvzMCg{lo%<x7HzOdZ<m;}v$##Cid
=6m@1qq(rn<4|cjzE?CwqP9T?0n@<JY_YnS`JVK^@o<u@nGfN-@Xv+&URs4z7~8+sTa#s=oT9<%Qdj_pc*XHjpdFg3oXLR;P<svoth1@A^sx?^CKD{$B^9$N
doI+Q89cXBR=E)jb`G~{3k-j(<*lCEqH5&X0=dU4-H*Sh5ja<f8;e%3@Qw>%B#0puFdukPKD-h3aLEyD#>oMk0uaG^i=o88hse;SRz#9<a^apeaA~0QVyY?)
a%US!m$riw`Qm7&bs#6l8I$ES=>nlqgx|XfJ<Qanq<nTGTan&){4@MdoSELRW>|abagVr}^Q*U}Az|4A8_)~`ArwKhfqE)u$ql(!@8qp@!Jzf}2eqyk#{;ap
D#u4ft<D(&5CTWrN!}G5gvbI~7#GNhMsTV3Xq9EHXhP=T;23hR`~JQfML|NO;KO;!u%a-y-yo6#<SR#3OQTTCW{2!F%<6sBiXhzex25|^%5HhcXNfomUO(^1
6wI2VliIEGWrEwjf>UN(eY`xha_C=6X)`dBnTl1$e>(j4dEUV{`Zj}%tjmCpBC3dStls2WDWFk*a6z$;2ZjXCe0gRVj>w)ISM2sNHg>Mu94K>+!(7ceP2GYR
n(Q1KRfF;157C=ZZa!*_G99-4qufgn)aL`=cJpj-)kvC@_iP>8@JnBdjX=#}fav*~7K~U?lI6${yB<Iqdg3;&&M&r=9bi_RB@GgjeL<KjjdPRq2Ya6{QOsMz
oI}3ATNEl|r92M$T^sQ=GSN$J;ei#y`}mH>e*O7(<=kWVPJJ3k-0#cxnLrLtYlYipX)i+WlHo15$v#;C{6Cg^ik;CY+?<%$<dcwMx#z}vTM}%TeGDajQiMsi
z`_#xzAhUCn4%3NC)3ZpI7~0X^wDO-(}j>6{&b#>q8HQS6aCFnF0<SM|K&S}AZMh3hXdAOy1XgKpC{|oz3>7NfhAFb8Q_m!dqW}aY(ZA4?xfJ@Bln{h5dZ)H
YP>1$o|liW00Fup)S(0bH$>&uvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_mensa.csv')

In [ ]:
# listas/procedencia_r4.json (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/procedencia_r4.json', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5-~_t@3|#;{f`9=c<Fp)p>rCx{$66ocxA48)4nzoP9DCX6`0NDs?Tl6E^p9u1wQHc;BXPHKLy7}Jp4u&1v%#^Kb0k%6
iO)dymlR5q(EsYY=0)wNVX~Vb@n?nW_<@N64aEOUwp2o&e+v=6+06BaN6_2tvludfaq-ou^Ed`I<)DW&4MGymJ-_tGt1maNv<L&y_3Bq$NbS<nG&})9M4RlJ
%fYTsg=wAs+xxi!Tx7Sgcc3s&i$^#Ed8*7+Qu#D$Zj|Vl0&Y2o^Cj#!71p+F<t^&HN0U~MpbuobjBA_tc9&6{Db)+d=z-qyhuSl%YT+XQ=mwJFLW?a}K_lY|
qSwdJT!b6yNs&UG7y$+6x*+VE(uEi02^Gi?5q9}q$){yvLnq7?SlGS=&P!_QY)@A+{t>1!%E@9>3<|%CYS)hHu&_IacqaDx4z@ODx_4+%AX;Q%%-+1x_8EkO
JZS#$$RQxbvfuUkG=sj~$2uN#3`|%VQzD#x$rB%+Xlx?|HgdiVKKGLmgDT!pFQ|^&c^MdL)k~_$Hge6^#pOt3p?65(nbT!)M<!jRyUj~Q15el4uLF~h$6>I@
vw^Q<Q}&6ZE}~k7_k#Nv@M&?Xe~iHO`yXI`C~nu=eHKLLO2uiXS|;99SnsGkKSwCH35Do`o|NO=Mtyt!j^J(Y3VgWFrk?8ntBB^<4kRt_00F23ya@mRGb`_#
vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/procedencia_r4.json')

In [ ]:
# listas/ref_r6_prato.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/ref_r6_prato.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;l0g$|6Kqw8cNL}xSP$f%+<@`qVenIyA@A}Bd&^EP5}<II>laObrWm3&=J04vdTS5z)<rt-fpe<h`wBBT~$IP-|pJ^
J}Hzx<hk6unR8J(^o6=|cN;%u!xNGGbG=|0^Dn*$X4QbB=8!$?NBSSsv<E^_r{3{e0BJJ+A%-q`pPdD&(M$VC!W9c1Sma~*Db|{seL29JjVJR_+idNzmu=)s
<v$^(ch=l)cx@orGa&W&&%8kex$OAWMp2B<s^JE>Uwa8)SJ0pZgU+xGtX&cl$-uNKu8o;q2c{Q2fDe+oQJU2u`=fF9go7o5P4{bj6$#{LQmf>WaCO1CM4a^|
?#&Ms>e<6@#iAwXF4NqKt>`x>n0w|#IUTuPJ0Ao39ONJo$b2y4m6Z7_P1RJf*$vAkaf2Zml(!#0cq46$_l+80uQmEc7My)(1RaG3{I!q)NQYD8?6D-Ne?`zi
?DqrUy(2GGLxN-gS`<&?De+S!e+K6F1kxs5?CskDhA(-etQ>~+g*P~B$g4JvuiWcoM3@YM?@jQhz~T$~VY8X}mLJwMU5)*}s-z2}?+4ST9S`?5#ciEM*n%X;
!6X)HP2#>an%U|J5^K=^U&rMC1tP;98yXD49QQ0UEOD?zJ_pwmb-wHTfd_vknt+GkCcT3{tRtH~HGb*%D#r^;oR)R0fNssjT$QVKE3v<KmS=K_j=6c?5_J1`
_hH?qb}pAwHpzWn`z{wjP`P`dq2j@9e{%!)wDsm(gS3-><PoYk;-fWmsClzE^PD<QRmk2=)nK5tdzE*<Y|0KcSsngaO$`_huRXq8Tz~5vi*>v9!H)9;)zUaj
tzGL?$Y_!akB1kgesz|y{03(~DIO{SW9&+c@+}eX08^{t4&Pd{THW)Iw+Zv}s$CstC<tJeq=;m3{jcAOCy&I`H9@o1$hhOuZo#=s|7DiVuW{zEVhG2HG}FfN
EI1^vXK1J79@EV%FGjwg>#BHdVkc1<anVjPeaG4&nsqLY%o{jukkP$}|D4bE+F}$w@4z)Z)dpm7^qqXbhY<(VX)Y;Hl4r(8sKhku%SqlP3u+y2B?nwFdxd|I
DP{7`jcO|*T%$1Bi4kRX;(D6w<jxV$`}zY&YgZPX0|ga(76IGt-@EKa3%o{l1FwIBc|liwiuFLd(8^YaQK52shKn~C0;wkWu#Yojd2=3xwtme7CMkrLL;Y3~
H2z~3fF4f)ZKy6dV6M|v?hSKQTu0*{O`a^}J>0ij@*%$pCuWEUVJaq`Q#v;=ux~HV_Ph|F=p4x=*t)3qLC=TDbGs@Pjxknc2EOuM$(P4|NT8~}mK&~@dN&ge
mcn^JZ?saq^sBr`tg^XiC~)ycX+v=VnxBf-{4&1$D+{i4)QAi5y4XPn0i~W+Yik33OsDwA>lW~}<XHo90i?+R>!j`}&GW8W{AAxG7!v2%u9@6k4NEy2kvp_W
A`KcjL!5)sO@UFYQPx+E60q-Pv8&|;N&7%)IJ_<EgM|D~8|~8v|Gwl=ye!{WqA=jAeWe>`Y|Z$$rX<Kg-Yf-o-~IK${TRWNq5vv5Ei+kupx$|xKdPrUlWhs#
_l8-T;igK9@$|a9U)%HT#uZYSS3f+|DF51WwA9uYf{^KNaOc^Opj-G%T};uehCf&SX`Gv2{ed)dU=<a$ScT3*#3*x&;-0e{rQOvMrjR4K;v+6zbvY?8HL%9~
G}pXb$IU4my3=WRA@>l>&;X4|FlPJCvQrMM(Yy_6sw0xf)11;vJ}Otp)eczm9ZoIHrVlE?8uV(3Iwjtl^SK01NJQ!*`AK_=7SbGqhL@apl@H`1;uV^~@4-jU
Ur<yRD{uzZ8z7I&3UF}E1(SgAYDD0GFFYir$P0{b+)v9HkU}gDN1!tTv7um7S`b9@zfWib=>A5n#@Tv1qo)IavL1+SWOpLC86IU5v4j6*(s5ZX<#YRiISL}k
Lzi6hG+(-aO3qb=M4)I&uWHkt90$!~lXk-EuvWj4lXpOq309@23tKu#d#3si8%$p2i!Ei^0u%~&%R)!4UiU8FUNNi67Rd~e#P9HU@ge=<LI2Z@k~o0!WJ~oM
L8W?x(%b9(7}&kSZ{S4G^HWJ@U9A@Lu`AW9gd7$ML}(87-46*gJkv(+dm`#^l<)2kkb(1eLW4)#toNcJlZP0LuM^|z>f{McX;P2hC0O>8Nn1G26ym84FL;F3
L<}_GS(EQUpA6s@#xMM1^)nUk{$jAL=dKP3Ub-hIXm~llX=IvphQb#h*h+44#TuRrR4&L8G4px0rS9<RSkXgfw}g*1bO73oCl+21L@*v!QJ6#r-xcR*1vD<Z
q$$_A3SDY5UlK8WAqS1yeGMp2bp#I^5hk88BDp(Vpsyh5q)%JwSXP;GSXB~t<KoW3M)Eix(2Qq$jKx~?R1yjI53YTMR6D{9sKdt}mh41rb?{R65U>`t(_x2h
0e*gN013Of_3@f1ulT7;mwj+6CzymacICfp_(JKGhmPde!+u_lbaXb)fgXp;`Baw(3!epSoB6aR7iF<QC}6oysok-bz*QC~S%l$9n(`2&sEZBVrxZfD62S}@
F_+a<H-EcTi_1-RXUtmd?Iega@seMxNW-xU0sh`m84OZU%ao0MR(1#69H#k5#xJ_duzo^}=^}P-mu021b_9<yo3`V4#cVhbJLZ1Z%bNk0$=?z)V(dP;H)SAM
IxQo`&;f~mP(N=xS4-WHo}A5)nK!Dq&f^TZqV=4e{N)d~#RgGPLgvce!%PIjnzi7>7M=9VKVg88H^SJsuT!squJ)w0N~LzBU0<#l$nDvS)+Oe|^U1_5HD=tC
dRI`j<RdXBgjFBBzU&Ti9<brok_~$=;Ge$`N6zR4s^tCZ_0^sUSnLea=vX%}sD1%)pC-CQvtC!#blz{6)25=5K5FQfgg91)^KvBu!6Tnw`+Vku0{YM^o37yF
`1I?!Ww=2P&g;rXMGfDqlrC=nU~d(Nw8LjSG*h9yhKQB8v_eX%m(>n`lFEJ$Pta4g<p7OijMhNPz4H*Cb*1JdY*p!L+2`QBskCKdyg1}E1AlM0%*AKN?5NdU
-HOaUoW5FtMEzW$%Pa34dmxeS*RydG@n!t?!a9HBB=P5xOVM4)@a*L4T&Bo!vUFP!+>RwIWM2@$>GA26hKmW(!*Io2nqDfN6=w=z4vU(Z&MCi4d<Boa0UyO}
o(2sy+hk89MF3;dxa*#tVfi2}e*7;pq0KyFEXdh7YuRkRVV<z2&v+;-&j4x7oclTSDqo#@#1_iaJUGWMTAt@}S2EUJO31OMF(WLFfG+IGRK_d2i4u4??)~QM
%%z%nV+wS73$baGzjX^kg5;)db-i%XMVqC$ZIn6Jop(k_`rww`wT~N4%U0-CRF#6Yx6VuCJpeZ<U1k-f?^-uMFZ&=-#}C06tw*&~+ak$!|I19HCu8!;KG(if
qgj>rj2xie)epx~o%(B1flj7t?+F^$QUE(8P*?dn=V&eh=>hp87TuT425G2UImp0#4dIBLQAbHm46Bsvtb?ZIkXf<|)igqII%)0v#s)oE1pwo~OMG4YZm9ae
Lurjg6(Nux-@VI9$i8N77F4a3un9xp(*9Icc`<8Jeg7w)CXfymX7G7htNpP%z6ziUh3N}L8T|cnny`n*NLxT-X0CNXE&&q$OrVt3U<MRu#QC96)!L5U6+y%5
9zOEF82~lLVH5PidMmYW(bNBM3%={2QHApRjq#fbIxcp87IRN4+U-umxku1ZM-rJ4R~|KgBf+DBpI1p-wvpplcXS~vgF%YZ&zLcxRPb_z>mwgB?y%7<`5Zhp
Q@(gOBi;L#f`NIrp?!tInB)Hy??%gOh8JdgdIOh?{BE=7BO$Uw1s?q=p^|3Iq53x#2D1X}IMSjK)%~k*{5XWWd@y>&RBJV4o*eGhOfOPB;A*O+LvAJ&Iv)^U
y(n)Bd;eP2l-~O}{VaF)qT+hK55$j7P%9su)7gZa-ZwZMpmA%?a5PZeW1)>3Tn6!Czj0#sUD@_KTkbkcKkt(dT0G1;Or2EULEO}a1{lqBz$Pw}GiC)zepBDZ
A#jU_6a=MvJJx;epFz|!lPGi&5G<daI%{1j+N6@}N;y-!X|$aoVNCi4f@Rz7&@3*F-N%)0f|ST*Fb!HSLBUUF<uPtmb2`<!F4OCd@56lnbP+;I*hOKPrPum!
d<LgQW)VHK!Nnzu3*s~QdJotFDiV6xx8{nqh^J3Fu-H%bH*jG1dnq$a#CfDt=s}z!r$}rZKgnr~&vbRUu9$mJ-Zd)3EUc5+BaR<ctB9UI&CSGw(w0Thp-VOP
wR-6%X+uG9<{c4ty*4@t+tA<I?&2TeqP*9>PsS6;r-(<-Cy?ceF8ls5%O8^e&~RiIF^vUGS47tH)5v%)I}W97y+D#HVEwcebzc<2%}jMtDL{K<lp1~Mkv|iL
!FG8WPxH$)!zjEDrE73@@V&c5Q^Hb{(U)2=K+t`j@;e(+I<6V}kLcPP$QxEWSYD`Yqq}V~4+&GO*0V~E<t%HI4JC<QsQprW?m6PnKjAGm#Jg(Aoeu&66;+Z>
Lt;i@4-$5m;#D6`3o(&s{P_1uP*yp{fj_2`(N$aeIl?L=GHRfv>xcl6^SyiSV*Mf7zD!sFx^XOSplT%j&DU<3$EW$w!9z)8G@1~KuzEK?TQW9zAj#vj9|Mo;
>z}AJ(u0a5P+lc`88nu0C&pX@uCXs4?eW7<w)c1XhkEWzA|A^2-oLGETWp3W@+IsH(s*1X6ucTVxf541@b7?*I4gnGi+Bole>IgCckhXXCt!eDlg}6&*h;6d
D>WbplF9iKOt8?~lU@;yy*_pm8)Avz$e8&92``fo<Oj=laJ@5h`FwNMk2c`$r^$t*`eD+ECT2A?>a6LU&(pmZ2WgI+#fnxwSL2v*qK3SA<WlOq^p{zWti`JF
qE|?OZ~O(9!r?J`kZA!77ZNeaj_v=rx8LbNM5P3I-sO_)4a^FT?8P)P32K>7`1127?@|N;MT{ff(F~80ZP8f%Lm*$yW(Dnno`Jm3LY4b?(X|mxfztqu7N-A*
8+2WXm^@tWO@VNpoKzNKj0-W>qN#B{Ksi6<{<`w<a&Q`A1@2Gc(AiILiHn2h{gr4@J{(7{9icU3T_g(@z?FSf$kd+6)fKuB(1hN1yJ%TG0^({4jUa+JKBWTS
Eg48&W>Con41yD7M?f|mi?e$E(w-*9d<^|+5pZVgk{X$*=5Hv;3o(v2XzS`>eldY3<wMYLk-Mm4A^`JgZLVj#0G`Gpab^6-XDp3#<_kj~id5PEJ^T`X^LW))
b9riLUcAA_ag|rH#^r`#f0?@9LWNh+nQDi&>dzve8YUVJWEo+9yN9?xmG~84^mnScQ}a&^4U`y2t2?$!0aSeohkk>tUplKD4sp#M^L<C<2nf#5Mx@Czqp^LP
@aT<|9=%1UAWQ?XU#pt?Ys8X-A{(m%pki(Msw~R?yu&WCQ9Gp0yfJUe)EbfMqp;6WL8~HBHhukY0dnY1mkNT#5@#QY&PYKxyK(l5R3JgPWMZS4kCyin2tVmG
u3n#Nr7Jty+g^DXq2^!q+Y8P?9Fk=4n{+{sIapr?+JUqpJ2V9|&l(su^(8Yy=eZ?Y%%mYH1s%cJF(q|mSoHj-^?f4yGz;CwtXA^F^5j${d4jlj7g7AJI{O&y
Sgf~qlX1WGFS1&&^F~=t0p&trWbD|-AsYiZMx&r>B{*?vjQWk2AXg~&UA?>54$KyG<jos6IK=bh*KmP)S2TguB%N&8W$Bz;RS;vqAYz<~MF$eLy0P3J#;#k&
WEjg(;tvS6SmF8igADJ!oU{1Z|E<y8J6^UlojHBh_Guq?;_y=R4!DJ&5y?|;rJa!p$Wrpe2_2sT!EJ#$850hOJ(xU_W?l%XG(am>a=r-~&N7CZc@Y@NdK63h
ImF6QadIagcqWcke8Hc*Ar;xKq;FumKSzf8uVkRO^c2Y^t*KUokHj0RdnF1Ad^gNgiZh$+<f?dc3-LGCZDB30mq$k+3f0AWzyQ<BG6X6XbCy(}7^gQ{5CXw5
!Y!QfN^7)wiUv#5(RnXng0VHnsf0ij?C;=^NLmHs5V3f?!DvlCYxl~p@Bi_TrM=tC9dRXFvT*ZK@Pcn2Ut3hgn~`roekP?JqzvuFE!VyqwB!L%eV8!=D~L<j
*6^)uRj7f$b|E^feI3$dYXVdzMiD|H%j@xlbLzc$_C71}%No;Q{RmtTbp~C}Qowf7-yN1{D9lfLj0)7N7>Xv@e|9{$l8j-{VJP|c6*T7eheo3_8x#p9M1xtD
(5TCWJcnPqoFF~SC_%f0Wq(3XOwC+^YE{ePJSo2P(qi5s?kO@@(L`x!LgksU=jEGrI*iy{_<!eX$RE&(x0eP1Hq9pq<ra6ROP>zuUX?pShL}?bDG*>p#o#iN
TJK_3V4w@ThbVnte{vozIWW_{aj_3S0KKUsRU|bEnDl}!$x0AblXlOK5aKvVcAaQln*s=jtitV}F}ymfv?q|MfojpN>P~hG_L3`niJTMb+r<^m@nXEz*-Yxx
tYOESK=yqVd(^Hn5MxEB943})yGz`C>AEvbO&}qESU>mqwnt0FTXDA8Gz4|b!Xwd_>gxQ2)1NB_jJKS@|7EQCkjt55*(v9-*S?SK+f9|os;`V>q0`{`lh=_2
69$n!1_^)043w16zfwXoABu#WHX;md;x!HG2bd)^g(S{rozu>P-RMawU;bHHOp;`9iM0G*D`)Vyt|}F`wo_p9CG)EAO+Y+mO`o5;yv;%zBI0S~g^w;z*g<yN
r5h$Zm)l4!W>-etMPjy6WAla~+}K8z%RTs;z+MK20jBd;37KO?Cb*Ep6~7MG2AtDw2#3lZ8<maDcz67F*A{PCB-2H3lVRE#!^joGl2(r_Ia-7Li92Y$sM;4*
S{HCNB@qa8{C$gRoN8RG!p(z{je#H1T$p_9qGRHA@!=&{X{%Gr_>VrrlN8o6Z{Ey_n0W27ltrUoH<C;-`K5hU!{mr9WE%)^o4UIj_S}9<9KGJ$Kp)<~MT><W
rf*LpW_?-0iZ6!H%b_OSe<a<4ur`MEyLM(NCV{UB=-P&!mIh~)=Lf>lHyi-}Ck0()ePU766E#=(B!tS7(>5`b$GHpp+zz@kyIM5H>R1KE;<FK=W6S2iNVfy&
*p!&|E5Ih!=X*jH?G%rK)Cwu9Ea%O`x#Ce)TpsWkQwd+ct2f>@npkpNeAlaeviX?3S#<{a2M~Ru!U03(H_~eVL}T-1rjf!U8U&W<IU&F05|Ri=D##W{yi@qF
g(Q!CpliTlnOF;d7<IYe81&vriQA7GtMsHi0ZFzZ-8Km4yajOMyGMgT=AQ#;Ux1Ty0Q07vU>Fq7=9ijBC;E0XVVK;YS5vLj)f<gj^6E(XKs7-~ML4))PHrhw
rHmnn(qecZUvLIlIBLkCqtg`Fii<&pO3Ee8wbdw4MPezpvTIq^n@sKFYuXdfaD<SzMY5xoP;!Yawn~|YXcK;`enf=Z$nI9H<-MbDE&%?82Gr<It@ih!WnF##
uw6pU*b~K$JCA)g{|)HqAf_v>u=~En*6kmJeqoE*k*%AWpK_btn;!2<vV**Xr9|sq%$5;?;Gt)(nUX*~@QL24dleXr85|o`;1wmk$a??~*b;CF;MD=lKgrXD
T~#!;kxIrw`%=gBq`#}<i*U_Rv7L6P@Y^*fUq4%;{VO0uxGf*wc?A<$m2>9FB%%{0cCGb@&~8)JU(QybHM9h~5n7N<R0XrIPfbYd#K04&*1q|Scp7{ZBnjFf
)rXe@<Yy%is7y{%1xbr<ij8Y`tl`~2B5^f)Qgb_q<IQxpezt#gYoy0lkM|gs6W4@spby+oXy3L%6(!E|Sg{dz>W@R#QHF`@k(7<a?_!>%K_>g_HGxD}bjMcT
OU@M)s$D@)J36`3;uhTPK26{wh68zsaU&_qibMg$`2R9)IYJzdlFZ0nav~LmWVNJUVBtIDqJowLf3Q|LfJYWyY>y`8Ps%?4OEn=V%k$C%`U{;%47+Pis5%Nn
-LDst%m#=?WRKIirBiJ$1Wp4b*du-_{h+rrdyy9KE7Yb2R9DZU&8s^bLx&F0g!|gJjz!w!Er94Oj8wEn1FF=<2izv5TRWxjnSVJgGelXE(x4$o#=3h$B=pfH
3$z6NXaElzp2lCXC-%&Aa?Qlv9fl+W%!;2^O7?w!!t>kic`@4;pS{t4-`^r{kR6I$bhB9J#RrDbr1f&s(QH~L79{0*FEPzBUPw3MPyAm>WK_hprnFtnZ%5-H
d#y};W=XfX(?t}jVzixSG1yz^c*d960w<Nq1&s^H)r0W=$+DI%OJg<Rp(;u%fhJF&Rn~7VgWxG>z)8Qxcvo{PBEz{}s5{0}Ld9G>HEwge-e0O5C5AG`u9&|6
rJRjX)aB}wik3yhy<A7lSJu5Cu6GHP0Bm!4_T?FRdMhBn5%oNLWtV@gFLD+tu_6mcSHj-<<xx*dJg1WDvIHh^R3#h+b6=?s1DPHTQW(h266bBS<i|U2O&LXz
!0YZPN4PoUfZ{Y+al4Ufop~>R+UVb~z-*LKt6-X&rOMe2UURX90C82hg*?$9<ST7sa6@USIp|a`xX7|yoUh54VkxbS4ZSCf(|>Z?K$KElSO6$~J#n7%wx=_q
F^dg9g6q9brx`<mTbzDOtZIu$aoXQB42jUg)b@z~U(zTxm|rhrEeQlK3gLDL2te5~W5mGrLReQ|bgJ=);mg|5B=GBlJQEBqyQ2I^k^g|q^Dms^kS%fg%9MK-
$7=vz`M9yl|7zJ-=y+W2DcR9y;ch|yb3m=!=c|}zNMhJY5LNzygVV2ttUO^ns<p^wg4`v-%zqeOb~YFb`H~+pHfrkGbst{sA`v0hHw=(w5>`kHNrxJ^jYkH3
AGNfxrl3WWsK2Knl0M<0uS!hH(-SMBG0X8p@I%(huo%Q}{S7aHVSB^7KWu0Po1T1BsZPx<9X9yec_2jiDstYVq4hC$rFi%Wy#NWl{w60;O*fUU%uT*g1!8Q^
;FH)Qocvk_;Tg|0gNpW`7G<3vAEbaa^0N#{fF(8t566QQ%<~%%jQSS|hG12dQLo~ceq7^ju#J^m;4GJEH#j~Y5rh?Fkc6MJTAnc8&Pf*0JM))GX)6f9-lZH%
eWSL0gKfh=`9^txysJ>~k!VA=SRB2a%fcfMtD=GZ0I$KX!NUpGOYq3m#6guQ-MCb-w7olcJP<AeWnSG}=}2Z|e)B)jkLng=xE!^P-bX&e1?n6qtuAbul2x;<
5>5@RR?t{l;we^c0FV8&IBi`MY0~WPYh)0p?yUFB)P)bJ-J8WoAXBHM)PXK7)xk)T9GNrb)+?Z;5!eIKd=>cqUGLekELdfZ7zK#}TY3KuW31pkF@5y|8#9Yl
lEt3XTVzS6@2KEgiY6gb#Jp`(f)%29Z9r2=nU)@a2gd5(PQA{X<Xq(h<_I(N8m8B28npE_cCb{e4gVdxHn%jDaLlz8$zCP4o$Z}1tf*wsMkF_t7Uf$$vY%0x
Y2|XW20rjTH&}l3><!WTLEm+&5m;q9c@}gBKUngVO$JcUUFOnK0p^tN`Tc`Oxi4C@ixd}TkWBc8>w`{T)(FtFsfvMGT>sV@V<=Nt(MdA|jDx;4gFKBni)nX3
o8hGA?7GCUE6F6EtA}tnjsp!4(ekWM=T$Cnrd%TOWG1?fx53opc9WTeAbN;QO8O3yus!VpjD)a<$47o=)li;azw_>cH{;X*sf)9XdYN0AF>Z?--is19le+tS
;3k28dkfR9jVz62*WVf1-#j$1xJoA~GDaLiym!K5pigFc|8T5zupjx{?PDsDcOk{nZ~LtAgNU0wRBma7jn-tT<0TKBtkYXOUom|TK@h$EYR-IQg5(u?SpmV9
JEYv^^Q|||<H0zj2?kz`@rhD2Qdcvd4m^xMhZ5GH6%v4BKHc7!P8_<n?|-+B7>fs_1zt!3)i^^=X76X3Z*0)!LcEJ7ncNc13Am(}%9_8vBr9M9H3+!<K8@l7
{NF9*DX^)l^0+}<^}PlpIuZ<QvQx@d%_J-c^Pu_W3G$e94r)uXN((}@cH#bWZ_XkG0hhtUctrU;(Hs?o1_uQJK#{i~H^jarvcteW5$xCCj_lc*KjjI%i~DVN
-ENV3%t}B=WZ@FOWR915rbxU!v;W?xoB|**N{+--gze$Mk<UTWi{JfA_BvwI>}oBs{`ru{`K;w^zuZQWWf~}#eNn0zJ>q3<8>t;#OUE@$Yu({>>(8*>bZL@f
|LXVh2IL7t<2#cA8fPbpqelold@{Wj#PT}_-lc*uh{HNV6zx^hgYT!trw;}ik9$H;XO3}CyMekG*}gbC_XADC59nxRY1*$vp6dFBp;z%xmpE)i9)?Vk;qsQe
+lD4eTj0Q<G6^K7ImjO3|3Nw}>U_E<JyJ?@{$onNgA}8?Bwiz>#6RH}AH;#bY}+JyQv+Wh_&1e(rgUc`PdEeXgW?HpL0UzXEH}B|cp}g))s@Ub@+Zt%4*yB^
x-LhyYuNo2swhgj`2FdYsH%v$Di-~lVnRczMNclvCVDMPXWeWQi0;kBU#8h(Vf0F#f4~Ipd~uW*Ps8htSmuLq^KUBln6}%%QIJNO=c~q!KkFF{)n+DRYy}Xj
PY;)Tr->x0wnOSUP%MKm6?-{}AP@RO)`EOY8E9;-Mh`j~8rH<hH!qr0fE)Gzo;51bttm5g=<fhN=wU0YYl`e#YMzo3+4SRx%Lxr6QsgOMHdwIj*V&Uyoz#8f
PB#<AA|R`CN$?@Rz~pNsAb`oBYSb{bvjn|Jb>TohrkKr}yO0Pa=nV}MnQ6258}S11`KF9svo}qlE80?uuO)eg7}x3*F$r;9Y9zEWTefyMlxwIQ)?Aqb3F4)n
$?XywkP|iq7p()Ii_5eA`f23QAGG&F`-(X|RJ#0KPQxRJKm6c^ni8erHw9%Ucn^-YE9e<)ZT?XMV-+P9bYAq^0%uzf0c>Se2zrr&CK#y6T4A%e?M^@s7O-11
S5EP@)oMp4?!&f`IP7OceA`t647ZpdMC}si2AhNEUKR_GY0`ci;eEw($J+VsEAX6+?>6&G(p`g++e|(ZJ*<ZwBJ90a;ap>>n(<0d;JN(+?KY`JHENtzbX$`1
W?9E-ODj_Cgbo=XQlwJ&Md24YQ8Y1Q1+q2)`oGFd0SzYDmhX^&E8=sk238j-1%aopYOLq$TsRdsp%gH6#n1v1l1|HPh))1y#*H!+>K5eItNqFzT6h!x9ND+j
SFRH;K|Bf3AAoRx3q5l3&{$M9{qyRb5whc!-D}dP6Uu#)=Od6uvrYh~@jh9g2XiqWzTF`LG)+(M%;`?vaS+K*MoIZ7;U%jLfGHs;<Th?D!nUCark!;>?4o*T
<LINQ)I$>1lk&Wy4sq$~_qy3HcejWiWH9PnFT{ML0j(wlFX;?Fa;|-_1s)25UTFCqQHnk35}+M{v_9LjCf5?H0AS;^Qc3hpc5?~jiUqIKf<K#P$j;T2;^G#P
xL!l<D`VSc?oAvBG%zrNm13wCpz{^spq#s2`|T|hPcW1CHA@^d&`c4LSLpAer^F1_=ToH)z;Hov{|gQ>poS3!?N<LQz8tqHM}94i!(ufa_ePXJc8ZeViQ8O;
<qOU_`Dr;Vml!M1gl)qxi=K}eAhYtu@)yLk7K|t<6^a00G2r50>P1pgkLn{-tE9>~SUmq?WMuticR?22cw&jKu>`hjgRcISDvPg|u~>K{Y8=!5Phk_#uZd;)
p`fAboAFoYVtwk#-?7V@mF6tr8l%zXOdiqh05>_O!jBtn&qg_E6|99C{Z%Y-6%or0mOXv-6`W~V?joG#7n4~l;L)rD5+iImL*#z8!tX4f08`~gAyq|hFfteS
W&$S21KeQ$e1yX~XV%PRk<Pc#7FcPE{(e(xLgeo=Q3H?yHxeFR;ZNM+<Jadeg8l|94Idh!ix+GS@}wP*-|@@zQNwP(xF<hi!uA@aLyS|GJI^7=r5>X<7EUss
BPI}Gtc}TZJpA29Fje7HMI8N1XrYs}ffT9QHi-)_)*ob$s72!3PLnzH#rUqgN`YB7O$<^+V=X78lK&!_Bx3d=&yLu*n$v4fHq`h05$h>y7S;jp`Aoq@7sK^E
9|gz-p{x724q`fGBOV=G+E(C$H@Xs9Fi4A4Fta+PdsN+1#mEXC-n?jx&#Y`OmRy(G5yA)NeD|8cFEx255C=T~?+eDQ5rNDoI(kML=3smJXyS!|#7dK4U#($-
)_FuDL7~Q$(jYsp!iH`XGE;(w=iU`eY37nPS&45Oi-*1B8a2W?*$C};6Rp5q@m<G}juvlycdS#%S2R@`IStN+LdtPdP|XS)u6W+q&;nP98fE*|jLuyd=Un8o
Bm>PNatL9Ek8622#ivACtlraX4%0!NT$ei_!Bw8ma67sfbH#Z0)6=Q+S1YY9jm2!af8D^_I3^qOnoRGCy33v_ss_!0Y&9O;kGcHpsqFYi2OY@P+no(O!=}Ap
A^80_9!!wY38R{)aO3SM@B~i<^4#}@w9!0`XbdUdXv*QK&QdWoC#<^XA8NvnZiP^eK-)c`k)Cd9ca+06!1>_U^^V77RE5D{K9hm>B@4(K7k$p%r7;`(mz%Q?
SMQ4qM{lRcQw!1M#9D<anjy_unwwG4u#p)+)A~<x-2Wfe19MY(`9MG9W~vWG?@u=-%dZ%rI+{KPpd6q@SIx$(w)aX<Emt{bJJ&6CN^(Fbcy?UKeZ4Hkrx}LD
+n2mTn<+?*!_~_|)eCT!mGU*U-5G(d8X2<xy3Nqlz;sbYREOha?L;}g(4}y9Fy`JrkuhBxo>rKuEo%dAV|y<vj}Dkn4xZ)m(Zl^(mSN?92spXSks8b-K{pG8
&3nI-HuX^|2apO@%5YC|Ib<VrP$I-<avKL;1<f8Ra#sb7z>1BZb350GK1?MnJ-cE-=%7L4Y%6h21Aaq~uLTI;$Mi;QooE>7-_JDL>}aU9s*Q#n^lTvg4AVWe
&`a$&!;qL>vR!re4hAPWuF(Eo<~~xs;vG!t0Rutx$KU&9G5M6E8Y7|v#8|KVPQPEh$<Pd!VDPk|`e?&vwwpO}?34QA)$My4SBj~o7JIx;f@#^POvkr0U|XM&
h|a&4vhqCl0_PDKYuKI8k~~pdiZxkX4xy;AzOG{F0>wibP1ZqjIb3zk>TO6mkbY+9Rh*;jt`#_sU61V8CW$!=qdAY`EHBiD@gLSy+_rW*CESM{0P~@b_TQ5t
WXt-|!H9Wsv_Ddynl#vCcCY(CG75cmfv;1B#wscHEN}2&<2kj_>#jGLN=rb-?n+t+HD-b~62EFAT=~f`PqWVCTK2Sfw>UBgAq7cmScDghZPh%svJlQgg7G66
DwVFru&TV+^MldDvdw2t7S?}<qRyH5{mf-_C~RgzX}&QVb5U0EBG7kZN7^xc>>x2ZKr8)Y-RMB%u)QG}T&^J`;&L_2ktQb^=A{7%n%`mNPOa}ES8ZDj9I$^5
xn-jko{=QUBrNmdEA?JodJob)hWF%$wM122n^tnD4YhGlie71#GRdYD;6L%M<@|m~aHPuWZ$uQ@4(q1ZPLr2LM<>@0UThp<h8L=I2t}V%2$<wOT{h!{dg?!M
5_vO2ck8W)BDhn=fvdcr^WacBzRM&6**J`QVEMw^1)n2jsD`2`i5)&#&dTCl6;!b`!*jsD-hWZUUJewE?83!dE?)6_fDjJ!T<viTR@6{L0YVz#zHTb2uTviY
CrQ7>sIf#QtlUEw=hW}Dd}KQrfpq)<KTS300%VX1eN7SjIY|g<IyZl8g=**woG$o(usq-7I{)5<%%gGs^jz;5+FCKJ3m64B+Mgf0K7N_kd_d>kFq*9{WzrkH
uPZyaZji`$5glZnYQiYR-c=0M7m>Od<G#n@6m5Guc&@D!jSBSR2YT=?<io*!TZ)#?Dk3&`J@!V3t2w@4>XKD^K&hd`S}<)or0a@-wFU{_QFLYH!&mrY0i%Yf
m+;(e)*v{{&Qb7r;DL%5qnJbKb=Uf`45i<pM8KDpU|yE(n2k8yt^8qI0@iUdWw`JyVQf)T?jY~??UM!4omR%K{SJtg17|p-Ism?F-Wh`#ILH)9X{Sk6BRtf&
R?1^5rK^mzMucUWTQQ(QCTe~N9lRJs0i!%3V{0=gfz9oI>7HJYPG~WlA{7a>=;4<b_cd}SnGyg5^Y80)F)%<zdJ!dv@gUt5sZHIAjVf%xR|Hut;v8~ExXo1b
Yf|An(MLx>MHoHAD8hi!$wcK~dVn#GBk2K&>Yxj|_ELQKqK&|Sq6o+T@oml#sWaSG^_cbusGQ2%<L;2SVCBkjppMkt=Lw9jV@tTvXBGqAKcaR9#kkA_kFE0%
1#>OK=n2pV#Njq?Mmr{o_jrzAPgoIZb-nCg%ng23X}W!!#%i9JWuh&W_!|BPxwl5EKCfdR6My)(+1?+<lvMt@r+k@+IU=8T3F*2O&>VKG9x1^knFLcUB{>MP
)%BG`{qvJ|NWSGP@W~g_yv<87cYSw*mm{6(3~15(k{nbxqCABM6&KXyX&4o_iGHfLO{{NtzzfyZ%N`RnZ3WYWk*S2XUMmZP)gZ4BjqsdLG2#*>;O;QRULeo6
g+mK1Z$4EbW+hgk!vy|3J9|NseGsvw=F<!gygj=XCBMja1$?S$PRxx@BWmqRAz&}nus0Wqf3^`;qq=RFZ<$toUc%GfKN<Vw8S!8;F<fvH!&b~AqRW(dDP)z|
RKc?Tv?u_j4?W*sI5zal5ACDcPHy--98esO^yzO}jS=~@oV)tfLa~^4M-hM*A;H=UxlmBe>QsHBRJ-l75*Uw|JRWhBk>~IafV*zP$PAkEL%WRTVXp%z(3w>k
9h?|e#E%;rk~vjvk&g#nJS!fR8Z)3XUqLL$+~;x7WY1y9lbwtos7w1cm}!E<%S~V-r-eM$!1gIQsGR6{)x5K%%zop~lpFR`8SlXO_&d(kB%q@7gB(dD6BmOZ
F>qvyzEi8p5jr!e$u6-QOk(xme+aNs+JU->0~a9KrWe8HW?xjDW0R8Le&C=Z8b#oZl7q@~VZBA&{BS3|fq6ZLgM0<%mX;txL}FZfyYpR`&fzxT-E8!u#G5}Y
I<MQPHA~6d3rkBd=E+f0AU@whteO=F=v#+H7af684|SE=$Sad!tg1u}5${IA!zJ{L!P3$-lm?@+r-9%V;~L!}8pMV9`?dCmS0ZX03qDLoT$?Il-~Ap>7okN(
a)6ru!)dRx&si>}XG(t2fCk%Ay!RC#G_}aobu&X`#2rY3bVm4ShNh6OdoU>c>s}nY^~E4ffFl;Yf*cpy_|q7g;^G}Y#O-B9toI;Q^PBTwhk4-C@yNJ?=Eqkx
U|nO(2Zl47#&GzRb!(yjr3mU%>K?m#1VR|N`A3$JvOnZ=-dwH5amZ`qCsAm((`bQG^Pc=7g@1AtS)@e0SFG>Vb)imygmC*LRdvH`O8Ha2QK=og^46y+nSv3W
yelU1hP9HVJcIWeoJJLQ!WE)9tJ%_mFSs@9@;9D7s4zjeXkBI#C!EUat+BKQO7%n+Z$qY$YtJph*`S$fjWfx3WEafEd0TOt14|ji9Vu02BgdU=P`lB@HoWh|
8e-Pz!&B$&k^r<Ga}^jCwAdclg#6YL_Q4m8pm>Pe%{VBA2<dy)7KSp2vht7edMtp$cMJ^oeYLfk$o9Wl%ePvZ{kD+u<O6o4Dy@2{a1R|;{(9R5=#5a(x(DCZ
1dz>_bcJs$vk#c>?BaDu_4bI0(v<$bXXG>d#f~J^b7|FKi^oSBSAJwuG-_3RoZCs#$;+peeJd{T`-tzbpCqvWnO=V-!K5(OKMAHhU6H?h2<8(8IAerfr{TLf
4oAb6^smYNWiH0~H<!w;=BM-?jP3Jf)vTOY0#|964REppH}pQGlfKm<b<~$4LlC9xiN&&_OCm}G5qOjW)j+JA0qd3xQZMs%x@73yR%5#ASi<moAHRHJSDZwW
C<%_ewU>L{lq)V}JrCPGJgs`;DP?cURbt~-P<$JXot+3pgpSD`L?41(K|g|POqP2wrLCL`<jVG?9MKvfddHi3?8sVaFA%v?n#+D*xCs!X)8QiJTgmSej*$Za
^K91y@D;@g&|(#V1g^8HWP5%Jr!|N7d7N<le5T7>&|1EUEL9_UYVm;6|8goRO{A9uFhW(Vbl+hW0q_Z4wr+TI4O9VCrE>MwyyPo25P5a9GY5>0Lz)gnV`=}P
MXTfXUdEV(zF_YVR3C3QYY!2raGc^lTgkblKGnDPU(t|f^*{Y^K=BamW5K+isRLBx2Bj?rMGgm(OEO2z;R}}l^R`xHNg4OHRuzI3ceCVR&S7|0DEjy^GVy!f
j9`sy$e`B%N5J%{dC?qx48KxXs@*r*#k%9CZ>t^ER296x+O)#)UQK5x{y-QO28RH@vZLiLb<s0ldXDuK7{bK-n~xW~cU_AF9a-?1_RxBXSKf;(L1gtt#O4Gy
SgQxv2mkU&`Pi}<C3=b?xV~E_#v(pyb)G1@;+JM};f1b|H=B7KE^C3dqMFI0r|?vidQ#<<Ut=+GGLAX0GXC>%Aer1XR(8@Tq9`>!92DeD4G}4Z+a_Bu5@r>$
4cT}ELQ%TenOHuMfO_Q%4v2Ym)~J&cOU(v11`gQr4Gg#ewRTgR?WTIbkpW|zjw7I9mDF>GA#O;*nB52m3Rerb7dm)8*Vluv70CvNpGQvRvsK}p_E<Z%tDSTz
YG`<2=19r3>{269ykb{enT&3IAGVI`Jnm(S;Lm7(q_P-XWjt*x(8URDP&zdCH*-3Ikx@O_>CzZq+lj5zxIy+Zs7Owvp!ttOL&{Apj!aO1L5sg&2(6Il8T$hr
>b?=6PE%_VNo~GbS@s0~(tQKhBQ^3atPkX!I6qHWwJ-7Zm?^<-PE_93e-G#F3DKs&)(06Xn2oIO!$Zxq36Lb%C1d*wD*eeUVN)x9pU+%>9qq&i!yI;1H*3B5
GO9nXKD5g`tx_aEOBx9DQ{!geDU@zTMMWeoo`LV1GsfX-D|d#Ai9V;jXgS9uOVOGg#zcA)0WQ8RL{|{^Dxj47!M2?q>AB#-g<?Hx$}&n>9wr<+_51`QE&nwg
)N$Y2$kKbiS#g;?cBKW~^U$8u#DNa7)9%h4tVmO$8=YqGjQ0(7A{^^@VTLcy8eANia3?I1QafYEZj*s3nd<y>l|gT4h@LCw#>;2Ly8xU0n$IFyO|Z4=*nsQo
!jRnPv;{1M?@d*SGRe7AlXGKF$XuP3KKC~-SW?^SHIK>>m251<x~S0-mU+1tlhmG4xYhDo8VQY;OaUK`fY(1&twr&=WvjYocA+2W@X_HMfibp$31s>vQb~;I
4ght|Hh5A3hRxSqwWe0+;&~tqMofv(_E~_j>w|+vld9u7pB?KRN(lt|AB{Lk40P4KQv44ijx#smcnQl`L`un+w2!JMK7VZccK3n8-p8$j*c?NpWk_ZHYMRVr
!pfG>DNS3S0SEL$x6G}vU}mDxRu~|MBJT=^fm#AkBztJg>ifQU6V<*t*?cb!iL2Y_PdXyfT5*`+Fziu7WzmPS9!B`kH^Hp%R-=mUE=<gIBxba?#FB9ronJZ1
u)pmN)lm!|=w`(jswq0({b6Hfv;{JStYBZA)=@W+;gRS50Zuw7y<#lX{Q%?Jgp2p)p(z69(GSj;8HC@c!!(5)6lg?iz#}W|Pe2G!)f)fk*AX8a4N}N2=<&T{
){Gwsf{h}EX{&?@1&!cLrVE*M^*z?hC!STVhs@qq|8|qI!Yy{L8gCYO<rGOh;4qD`>VEU3!fu?jIO2o`+9}(eAg%Ugth1$$MO-HNGUh}n`av7g2({iF_w|Y>
F_3P>ct6q#eIE`u$_a{SE)Xi^uz?c;%bw@=SHG$jlXvjDBtiAVxZ~<2kAB`~(GM)<=!q-Ee_v_wC_kq)?c+vi-y3&7JGI)53f{HH60hWuO)>c&U6|#Z+pt1X
8sD#S(QH-M3gkLzZ@z_$i~-h_5(4<sQ9{jaBEGIu52VK-0$Jv4;4JtOwRtgKXM;Ds0de6Ioku56lw`f(q$}e65Py^gZR1}D{S3!^J!=XIN4FvEX4YXuQ3a<L
`m=f=`hMqU>X{^ryn1yQqr;_tD?EI>F$z3-$>3X8?$v@-2_Z2D$&wZc^KR^rU%<FV;jYZGj%kgjv}ul=dd4gn_^FT1@UCQ+kU@~Vt+SIVUTua4QQpw0i`!LK
kA)lHsOk~geWO^z_v*D<RiuQx)b~92p4vUpf9&fBl=#K7g-oA1EC0DZLDTvP2APfo>%oCG{(lUwrPES>ZW#jU80i;iX)SZ1ffQw_4iWh|4_)D;F8<3jkz<vd
%-6ZgtxPCCP9VW;W@$U$f-Ul=b<YV%_wsyYZZ2F|TjS`;IVYxjP51tE$bm`}!Y&VWA!G);oyL<YkE-dE@J8?ZbHk$a2WHGRwb-)9@-y4W+Aq$9xd0^t&nr1#
F!E=bkPjGyI+Ao);T}M=uEYT+k7+xgy*6wCly(m$PFo9G(m9jG;hvU39wweMIbo;-CRvy6GOUjq`lyel<%LB_@MiHqhOHwbKJ^=USkhVPs)7N#9>TAfG(g|N
1tJ3CNQxr8m}U&m!+p{%+Zt_?Ug+=sKEXl#a=P5s!|A17OhAs;Re!1k2y-tZVY6=?dGL0EYuR*;fp6+N?6mVTc~8xkhq`C71w`S+(=PkPw2|w6G^I;YT!pjv
?K^`l8@?UlW@^h{ONz)L(%oJybEiYZ8;d-8MV1~(L;d2^GcRbxHZ7MF{e%CW`Rsn$%;A?;v25vPgMG(@Gm_43?6UNb7bQ%_89!#%<y~`IxY#veR40v*`mV#M
31bDgqOdO$WGug+^dL>4=2s*gyYMZTL~|^IrCevjb+s&iPFccnN%8KQ=@cOlfP(zkd4jB}JLLpsw@3m9rG>uKjkl!@zu`ktz9GCw7z2%na{VVDh3tdys}ICS
XsaoDG`<-e5=BieFKyOPWKy6nT$Hoft#Ol>{B&w8^&apHKgeSUL-k*v3PX#Tp#fW-Mb`f;t(i9Y*HX#z6PTQ=waD9?)`j|c^ABsUl8fN*<K=Ry*<vAZ)fHxZ
Q6bGcj$PH`iovWqpH?<;3AT_J(5m2SsX0=-ukn-mpeaZ|se%Vq^}pfUnACy@JcSrc>UV?#)G(a|oluDZKqR`e3C~Zb{w49&QIoabbrHHZ1X;S!%fNn2G^oJu
#VynD9n9mXKCGZh>p(a3=N1;KBf#QlF>x-XQKHqe?0T<Mq!P%FiDI=D^1QDusgEB#F7~Ysgo1NJo!-n(<2m_v0O2}SeBU&HYeU~brgVUOqr&h8a}$_!wY|u7
YmA0Ep;aNswx$`uZNfDm-guwk6*#i|qcUlXaW=)3HE~yJkP>vCv8SL-fSCM()q*64AOcQwQ3C7dmXr(m@XC(okw7;>;Yp$(9JsBtM}>|wz&dk?cP44NW3HzP
9e4N8;1X<b!E@3fG63WD1w-{r!!63iSK-G+ko=QPWrIT0uNSoQIpV)|XNh6QU_hUK1X|PM7k!u0LSIMIhQ7wbVE^0*$K|I_8e;<SySHx?M8iCwoKcTfoSR~M
#>KQe+#85h%qi<PhOKFrC;k0i^Yf?Tw``LJv%KReRill9t0Gk%nU`ytjethtXI~JG8)BA5A#H0n{;jN7{?Ozpf6Ht;il$Rhrx{FeGP~Rrf&TZBAfOcdbWr<V
B~xz?Bl;O%uL)|Cm!oGgi}p|lE1q)o0YPLxB#VUWCA@D6$}$@lEWD6vh|k|R3&{a%|AKf;YB0M{Z6ELoV73PTz_ZqR=n{(4$tn@~a0E^BEHmWOk)9|1<cGy9
YNd`&kn!L_Kx&3fvH!)<v>W#cRO~N@#aN{Bd|3HteEb2H4X1*iZ^0ec@6V1*a;sxco6>8}nh%WJawDb>MzOFV64tTm4x$T(e~;i*V2R3Kiq0scuPsX*v#AFu
$4288f8HbIE03{@s@DN>D#e&1xzhz^G)=Zl4H)kh!l{kYgBQ?Jl6f2d8v5gGket_c7&XmNna<+r<W{>|DG!y$R#nAh;n^Cqgz>92Is&42fCfEvAw{KvbpyYj
6rUoaU6{simFh9rU?>XVk1Td1QwSUc3T;WukNkEiQk9pBSVh94D5$Yi7)nA6UX3)b_s+i01^Z$!&=_7#?v`}n9HEycI^xgWr2ywJfj{=O$Cimlm9x{ai#&~v
5eCRU(2^m`!Rj=~b2F%r9M?Wi$(mqi>T(w+9w3OcfV#HO@BbjYg5;4E8IW}|27_12XwNGYm`shu1XDISxZY;K!9)^5{(Z?m*tDPa$U@dSh0y8=cqR`43(ee+
x(C*Z&5y7du0V}j)Z@|ASf{M<QrZ}vUTf=%?g%B@i`J5y0J*X)v?I}TSM;W%I3f#lPQon_vef$l?0+JMkIW8&`3iSJFLN69-7Rz}%}eYdXtew}OM$ekl7(6m
T@iGrzIRG5`phkgkd&+92mC<s%2oQd<%1pQ2M@3PO>U&HY^&uUN!?NGzLaSAlW7{F4rZ=w;E>PDTV|Ewz0F|>xFmL4JF1j*tzMsD7?VXh;czpeoNoC3jPxip
D<vf9t0!|kQ{v9fdJ)_Z7pEf!{@$R(tYzJ?{nP%loiXjAkMu^MT(9<GLD0dGTyvoWqANi6gzIR_(CE`t2=WzlhjnN>kH8<Q4v%CHaYj!n;~b?0XWTK}h*wm#
!F>ZT+mwN0p1kq;GRb<vSm7LGQ!biv!0crfLuBcjuqjXhfVp&`Tqu3B-<T!wy0)~Yp5_%3@f$+%FE<pn5$FErIf2n+vAV_CyHEgu&{g;2m^`QNz>x}2LJ#|H
2!0Th;^_jIH;6-0)9n*P=1Hkz|9roPr%U=og=?ob12f*sP#DU)a7)hS&R|MU+Zv_>zyKjV5driUNV#5w@=%1%Lm?TB>2^mz#Icsjr&k-+ecVpKDe(!#6Pyap
IpCK6^|`nWvfmVOIb)tgL>I~#lw0uovr?~7sutk~eD#1aTSG|<^^EYTIUW${$4sys52vA+B!wAh{V>EKJ^G26sIAff3ZuY6A>r`J5GXRMN$eNS1HXYok?Djd
VhG4AW{|`GH)RH(bZyMpgxsjs1N@tnUr#sAfayv^@PZYB=Sh6X!ZDaqfQ9k=`(CKY*uj}DxTp~IL#yNqa=LbgYU_KC4-N4@G=(1$rp|7t9=s+PiD%Jw8P$b>
1XDWpIcGDxBc(U~0$!H&LZU00`Xn40!A$OR1Q%e&LUoW)SSu`{0p4c|ctE)_)Y|q$?njWpP40RFOz|tNx{nPbT8pyeA|7zYCXFt@wmP~p(E%bC^dhtEzW0^R
fb^N)w>E>wKC2#zRelB+Q1xuF$ED6;^WY;S_fp<e%+HW{P!biiTtt|x;h2~`IJjg`>3V>v69q_cFFf<%mtDgRC)GBy;5k&s=nsU0;@HGW&n9XH>MhRnKp&jD
ty3A~u6u$ZK`(L9<-}|t5%D2IaMJTOo+d@AXY`-x&uClho_8^HW2^5dTn2#(C_J3Viy0@L-k9|1+>M;zK1*7Iqn|L)h^Tk`GUiAm7QRttOpPY0D5-+xcUutO
UM+_4*v1Px`o>s)O2uF*i$-&Ka$aRxQlGfsrwQ)R>(c;eyu%jGX-SV~Gk;WB!#j)kD|CMKgsJ0)E3YM&_A9Z3{oTZSVa`?qyae}-Z3SZz_$H>IHvioS_PjW7
w=-bMoF=$Tnrvr!gLia3-zt^He%mRz+ANBs9cxPa)V}yNn0!dZ+<M|-8WQg2ia96-RWXV@l=}tU%rXU0h|BFT`jL29>=*`7@au>ev#yb7rV#^p#%fL3vN=R3
S+%E;SE5g?)+tD&7j>u{aMu|;dhnXwK9Jl%7*{i%0K=jD3t}?8SHKZ~&HE}nSGP`$yPso9`^bRXoD9g*BBO+LvvN(<h5W~Hc40{4r|6M)h4do^)u_e>8;4Ax
n}isY=tdF`(&FD|@?df?_A~5u{9gH~JIa(X`GFOab3<yo-b3?*wqo_fv#ope-HgbEe(VY?I78$6xFU{L1O199KLeX63L6ZkaWetVSx!)AA6H0-p)pfon&8+1
0A@h)`+<KMGGv}<<qy6yN=NBA6$xQ$_$@Fl%FU%{n2vNNcy^To+WM*gQ&k7|RLsm9R)10ymE%x@d5M2Ei&@C8qh@>xvcT-<toa9UNTL?w&5I*|1ouJNEohj8
Bl^sl<~3<o@W@S=?UeuFy>I{Vdh~)F<{<U0;5EPL_2G(Ux)g|HPDJo@OIW_}&kJ$iZw5f746qAXS@5Pd3z|mCXXKKRJxZL$wj`hC<B>63n|vG{V6z7_i>sYp
p#@QF=zA_llIKzUUCn-H*323*iwT>w3rv9OGjF(7X~bp<1tRe*_i7-|bB=$Ytt9Sb9+j#0-559^MD>kJIJB6)7kOS`|B_{D!}rh^as$+6o7w?5FehuY^y{2P
DtuyybATvz=SHy^N2%KL$F*ZxTZDGsGKT#+f7md3@@EeS`UPhpI;;`$DT!f*gskT2wKrWUJI=!i%`r@pW!?<U)#Lj;D=!t-DV8(TAqH$%N~A;D*P-*tho-r0
!mnyOlq;^Hl-2+`a?tVYUc`O$l)Kl5;mQa>hl(E1v?U4I0zQNZ$_-G^)K-qh?nJ_2$TF1~_v5n(u|>{I9dmf}Tj<eKy7A>B3PvBA+ANZ5F_!zN{GxSO7m(?X
?h;@#UfPrCc>h*TI^lqlCDT<Rivl(Zf>)1#dh)t3J$;sg;sVO#XlWjAngze%RP6~E0$;)w@xW0DvQ8W36eJG`!#<4*=>l0A{9n*vR@tq+paHx>p#_&u!e+tH
G$6K}i)XQ~y(|+OrtPI}?scXdJnx4sl-I!IFsN$5?>d$Maw31OJJPQ`I$rMz&FZyWRMy7IwsJn><g~8G9_?iE|5;(stoy>ZXmGb>XDzkWyHGPnolzyyuYx!>
^C{D_fWwE$w=9dsTa&DZv@b1oPt&T_&Cg!-kPH<lWeiuG_=5dzZHpaS$%CqE>5uG`rvZQ-eJXXlt5Z_6jmlZOY&L-_BxtR^$LW*AD?ZuX9;%tUAv3z6%v0BT
yeo%?GFg;EIQr5J1c4v`2y<~1om@(<%t`}3P!@3dm+|MB4>gZxuN|;9*1jz6>ax&1!E^+#xHF|#9D2DSI-s5UbMAGJe9vJWxI<y#6>|j(?iCs!32TNBoHh;p
-Dg)ixwnAZ=(mEC$CU_{H3p-S_;14cw0Q%L8e}E7?E<VgiPLY<FN(TMYUmR$tN+)PNgB^*Z8h4P8!CLMSep(F>@5+oU6qqK<wCeSoDIWGT^T=Hv$$A>BXT2l
`Mw@6H%rxGL14xoX4}!@S=XkTFd!mU1`&p%X!eW&?y^05q^n|yH8X6qxGBFoSK%n?d2BmGx%bN)Dd+Umcm|zlt)gVFZ`-<e_tMMt`_rgotH8fxCNg9+COpiP
+VRX@5i9k~Vs32GJc4SyXX6(nvl>&wmob@)0>G-0`UrEhbsC$+e^J*Qa6xEk<G@}o^ZFggD9lGB*DqcM>q#3F>P+)=^JH-)8n=-S1e1njGPbzQ0m!h}Z!GrE
Cs@R9CeJ~u$zyD0Jy&|bVz{L~G;+w`Wir!GYKg?PyW9mkJ?Gh=#3PQx^-Be08h#!xRIvQgYvW4QJWTHEQbS4clv{--Z}6WtcZPZ2Dt;qKa;1A4Ey9?S9I|mR
nD*d&HhpCG`KD|wwU0qea3(oOxvToQFf$&g?!HiZw}t)E=OTbZ*$!-M<RI)~M}#{LNDgYeJUz`IwF#v!JhRDS%QVRQ%&QfQs?mn_*n%yw60Ik$lrTft2NtML
B%mBix;GOvUIiX~jPIqt8S_<}dnj8E`w^b|lZE_mR~z_g8bLz(%YTIRSC@~)Wq!Jjl-a*SSF$j_eUei*JO`{BQ%K|uV_jn$gXJ)6ngxVH+y-9VTHgK!JM<)?
EZnlF0dZBwN|jLWDtZ7?YgT2Btlyw}N+f;taEtcUZiHfplDqxlatAb2NIi0MVacy{c+3InQ>~L{r$8c0co0|_kgYui<0V8x<t3@l(#Ve#@M+DnujKelVS4l9
AnkQPk>-mpH&Y5LaBN1zpFT<7kUIo?Vg1+3yx^g?-eB^RY|@3My^TNu#I`+zNFE$pQ+V)xtu64`+i&5M#{qn6AS6ddwg$B@{!c&?4F!+e{@+!$T$R(h;0jj=
{=Zwv0ibN{*3EzmX%8DJgi$c=NNhCkccDF)y~d{dWq9;3B%z+N=}=C*&3*%#MWrn<p;p-#?COHHYLpX}ykiEMPh9DqjRUzF!XtC;^Qx`Qt2#Nl=8otyKu-_<
Ihs(I*v%K*6zA9(p~_BJEn;Q-r}F`I&)rett$edu*<uKk)OIoT2nW>EGn3OI9BuGR5sLjeohWu~peJ=%kqZxw#XRG)KF$D1%h-KF4$V@5=RWv{La9KG6~lGq
hC$XfLuU?OhwXoA`6TbqIbq9mcjHB<#(vF8=3f7*R9;Gg2M!c3*6OJxX*YMV7vZG`&V@`B+!6A^$rL{Xe#HOIB=*}uU~%j+!Ih}RIa{Kc!=UJ`Z}PEhhW{1^
w?zxK_pD%1#Bys`yP)m+%T!^_qakfr4!J?Mr}Qki24ARZpG&TXzHNd0DWzffS2xscZSRq*`1i00slL>+*L|hpd}Vj2nWOe6ohTJTzb-0xU<b=8SvaV$HS%|y
P_J5~&QA!zT6CKyxDWL+gk=qY@}53OMPcX=Lq=5W8TDpC&!r0$3}4AbIxARdf@AQJEIH5sSbx>wcyaB$`>{$3Qu|IiokEoSFh5t9xwIv3`K1wIcJ~~wzzsx@
iU4ipEO?67*r|z#LKH?idNo^RFwzYNYv`hBQzhGBcmlHMr~WW%8G^w0<qWu}ACxkD9D=W-4186Y#apB~m<I`a`LD#L98xx@*yB;kL(G8h&_Bn>Uu;#8PD4bQ
&1*n9yAqV?orM@dIV6B}`R8Hsg7;u0YQ?R3yM4}P;vo+PwAc|^914{Qr_>h_{0G6e2+*;*jeaozrXu?a!DlO%Ld5`KD?Q>cMT^YGe#^}Pk;3e~FwH6^z8WUC
bJ6?-{a<apPwZ<4kxS#sy_<QKa0#K1Rw}S4L=;&_Ay|Ug8k|s6ow<*q)%*dK`XBv>z8&q6D7`e#EA4mpreG{HXOz(DAY;>|x7wMa@yf$ZfLt>e3#@K6@4)LD
>%uU!4<tj<g3n8Iz->0*5(Eo!!XYx7wnOv?n^VU<B<7aG&9R5{<nau!`kqQp09TMeg0jA2!}U*4_zPqHhQAYFdfuY|L!yB`ewGgWkM(WIO%^x|3Q*(e?z3tj
;}aA<*q`e%aK0*)znzDWcnOhyFHg7GlE8SedT*~Pl!!^ut8S!8x^aKF^jxx?&me(@{hB1RhUCYmXAnIRiDFHI^5KxGj#xUgqMy^R4*-lLHA9;W&h6y8KR!xs
f(l0+xLu#C7G?_Z_Q>=}g-kqf3JhpDu7B_67>U})*SA!0HY&NR1~51?g8fY9QBlT=BbE*@_+S3cXANR5d?5X4*Zrr{R_kG7kVBR|@1RchP(WiOQ79eCkHn0c
9c>|KP^5r}>79L&c?=m~81KD1NZrnHiD2ND=rutz!u4Jy=C_)n`lV{Gse*s6Je=1X!v&|E+`CP<Wkx8Emal`o<IN*XN3V0$+Q@@{Cnz2G8`ETx(lugTbbP{{
Vf44@s3j~`SxM0-j1|n#$gvv)Z^hf;NVRSy&+P}PQXg|kC4DUvg=3%g6@vo|*EqYE;VC9lr{Y5rm5YjQ1!oC!y!w1oPw)SVK<P2zShEwDN%Y4IB%+YA)GhCA
JQPgHdS+}A<`13Ui+i?Fch8VKvw4T~9Ro35=6gcgh@NyPVsnOGB^NNOp0&IlFRuI0+~#-?XxNG^HWWd01i=uPuLH+UghroSz{2t^X>uTulDtw*h2}rGQt6SP
+Y;nXR@{cwo&_gtN=QRWS8*z#!a(tK-te*HbfSs2N?irf_ROjN{>WBNG)mQ!=#W!={Ajqo-yVV;da0=GGrV42DbEMrtF)dU4pj?I-sN`%AW#nw@f$RpbaiPG
_0%}hV=wpma(CvarDZ}YTC5cXw&^93<)WBpqORn0mEp9y!nJA_LnN?mp!&Qo{PO;Bxngjqg$D#*Y^#db?PKPD^t8+Lke;YqvBv=0M<$*5in;w2LB47M%piTA
3<ZxH-l2`}Hc3PaUgh(dYOL!C7k}u5fRmgOhV*nq%eN;dTje~$O**s8Fj#M6wn!4z3`VAZ)Nw0t`>BfyZfI1T96$j^Ix!Mk$q_)rv+jqI@~}a>@^EP7e!GIj
RySePe4$;XBJsEZ_bK#1K7aOyfcg~c@$1LEIG_I1iu!{i89OoXv_*9`iZ64B$>~y^Ph}|S=lu6ZFV^HjjMlLsxoKi#uIz`Ox3L7dnHF7(XY7HCSFYRy+Geb%
E^Fegb$<x_{PA2k3ce*K55n=dfukz6_`{s;%?LTHnuc-Xo5N=o?FB%HXtnkc<-_$4r7pjESv;^noM(dqADPIL^6_*c2tnS_1;{{Jfz2f82om>!oVXUgbmQ~(
MSLTT8|k)hL-Y8m8}OO}#v!eVFCT@Own!1XJi^;OL6juu+u`=q|L{cB)M^i>1aRzlrp_euksf(>oT)S=nF=-Z&Z+Rb^%<k-yO+cUH;fzt)yOlg-0#kiay0{?
raXzgBqWgi#M)6Q7#$N(0cZF@&(|lpU0+)yaCrP(DHn^V-G{nfk($hh1jo0R8z7N(^~I2GclHh9uPAE}oaXY)7OIQkI&Qz?rqW9N^}OI5jHZ@JK{9E-gdB2b
Vi>9MFV}GY<bZ>Ryixo*(#59kJX&kGmh<B#Z7^N=*mL_?XLZ>y(atcS=idzMa?XjeRPJ<%U-Kq8wVCUQ@H=pg?O!S4qyc!hRS;%H+f$(;V+Ba<9ro^_A04G8
-lC1aH@95}aLK~{@F_LMwM>a4XPe<wRvj%<FTd|i5Sgvnao<xhh0W>26x**}*?yHmnO-8Tn^R&0j!SI(@bxAKG63zsPL|f-w0c`uOq*dX7#uTiSMMQ$iAFxA
TAZ;uwTRy>^Z>dBGcW7NT|9SP7|RJLWD)on{t7($8Nd<QOOjB0sCmIAU2YYgIJ9=r2o`S5KP$3<NqyP=%jp5@fbNVFsDxU8hK|-p<r0}Q^3Ype7e_CrX%!&2
uCELZ3LG7}XYWcz>$$Q8WWPq00ra{F;bo4c%Vj{Nq*>$pPzcrqRhOwHY`--Fhze@V32(@lH%LaXmCj+#J4(gzsr7!!hb9}ml{Jhj(bp?c9oa?5v5MSO_Wdlc
_ZIyNNg?4f3b(Ko=5!wfC`Jt@q9Zh}zPv{i>n%66qLxuc9_?XmAADI^92k3E6>TQNL5=dBM0ljHSKB6r#@-0?#x6WL2qGkuL{~DwJ)325D&LdSxOHPhbxk|O
$9vE0B}HASsNV#z>Euw^=L|<2@nc~+BX@K$2M!!a2|^+V(;<5z;(vR0<@C;aS94X1xUFsO_N01tJTLy4uNNusp&#t#%nPY)<c_qynAb5#>6aid<KRKoA8AX4
7T-}ww=x6+lHVA19zv8M_3ikI$P2de+2jtcD60E$1KK`D=rOaBGv>aS&V>9`WEPCmsX9<l>`k4|#QjX)Z5o=8fXG#<qTT?bPwkE4K6?h0%80f%vlU3Z`c<s1
RxK#tE{8YY0jvD6wXQAareLLXIYlcsBr4f%MBf1Hh#`|1@>kHv^#T{Hhq23@S$X4h*EkAAm9P^-#oitE1~<Z_AMxh&R^*rVi)DPTSn4k_W#GElIhxL%_^7sP
VkL#EBF+7iA+bBSrJ}0gR!IFlaLMS$_q-3Q+v~ppU$*?ci9nG_evaSx2e|8+2&r+rgod4XL^;z57!7_K@MCrB3gAsWg7dY1P_4f;ko3)YUi5O7ATxSXXa+|o
Y(pJ08xC<s5-u(R^LkX^L9|xuL(S+l-6q&2lbF|wLle0?#C8Dphfvehk?+QfFM}|8!E{sr-L##%Zi>F{rh=5n^7n7t$i%pcxIwsloGRCg1^&15#D?k~JMSW}
s_!4#^x94Vl9YX`=JDn90{P>@39=b3W#Y};*o~7W3TLpqi;dmPlZ~4Vc3#d&eyf%uk*?im&F&}90-?D+zGO9xoCJ2hhO=|_&qS?Fpsv84p5Gh-&brEKTOP3_
m#p%iFT1^{VgZC}smpjwII%|E+d@7#L|Udayi3K50=BF9ww2tPV*7zL!eJ%Tmy2T2R@uophgI&gC4Nl3o=O*iBRL+PqGuZ@LL1sSWY!zZj@+4{`n(;mRo>pl
kNNY1)4xri36AFouzso}C+-)xA6RR4GRL=3_ld{(bgUX~$>$}LRn%P+VoqwWeC}6>@8Qk@LBxWzGQD}0f3M(FhRgfodwLs@H=*^J7NNnJ5QXcHvSYYHR({d0
Q=F_6`s|muo|!3wlE)gLGzV<0b*OI-km%vjC`WyS-l7BNmLj3lKNBlJhq);!Ed+adxeK8-Vzn$r+d&<6_)9+biX>dO)3er!CMN6*k~1h`)yv*l8mIIl>9=bm
oS}guX32$A6rI$SeaO?HpGpfU_>F;YneR{N6Vp1Uv4c33#0ecd*e?e>Hn)C5x!zNatlt(4@p^eu&@p4XYTE4=#nIbJtC2p-6Ly=ieb+WyH>7jlX}z#EAkm5U
pBafcn=Ch9t>+5OY=Lxk*KA)$mr*k9QAO63UK&6{(enV5@vk^eDx7{NSJx9H;=0u(9p=h=I&?WZ%@uTXYAsRK@g9pX&MIm|#$thUuObuLyNJE^y#1C36!}EM
RMX)@UVSt9M85S?Yt%JRW7zHywW9dRP4Y@(Hs)j3_ChgB8>P;gNs_s~KgwS+$-<JhWJb05Aq`L`#^OdO!4nHCup)(j1pqEhp(e$?MtkD}KWKU;O()S1lp0dA
L9_h@^o_OCe32mCZI)Mc&xzT4bl&jqe_9J!jG)V4X&NodWGJ{W0_{c#q|9I}&9pUsaXV`?{@Kvqx&}Z1{EIdncMC%|sRvz_n%n3s?fEminGzeu2X*$F|K^7-
o}4*;G_div(u?dya2en#NHAGa(~y(16IUQ|O8zOgyAht@jP5MeO@?wOZ2Q-s`Lc-hP<jM90b-uwXE4WwCUp_b0`R(o21hZ@{mYtdkD@xq9~D)9bDEl`{2X;V
aEGL(u9B3~U(H85qc@A;i@k|sLzI+dRH{Y4^NgAhG0}!d;eeMbZ}BKtfuR}qaWH44*Zxfa1p0SQsa`H+Ak`JdI%-uvsCNX!H!+J8bkc*3;j4ZrU<v)MVS9V%
M!`uzl;%P-crsC)1sh=|xc-_zM(pW7)mNDl5&)OLZ%$v{<-I$~*NFWUhgEFr#dQ@L)s~=Q#{<GNpTeJIkOg%f+a47B^ZDN!9d$6y5_W{lFT(~CN_2(NqWOF+
|Lw%3z8fFm8Q`<a4Hd-?(z@MFGfLL@P(9fZpr>Uzl>A!Nl;}CyV3kDgL<REMKnfO@#`*qjv{<agkxnE<GG2`=b=M4>vttxiz4Bt1+?DtkmiVEoQZAraKm273
I2g=9iw`yZmub<~atxZZ4L0DTvgGSXW&tPn$R=eSGcSvh@>MYLT^V_C{KjWA@gA(Hl$3*5uGZU#oCz{zpSGWcFLh?pT=E-dCJdUo@7kx`#r~m`QeK26pGvc_
n?i9%mD|n-`wJexAfPcp)_6r%+{O-`+$MP-*;Ys<S0RV(doGPn$%qN>TFP#Q>KhvqBjl)-9^;!_=vfgtP5SZC)z4$-Tw%-ZX7!L>z`A!&P^5p@T_}8_+K`xA
9#MUrDOGKewNjjAqz)a1dY@G&^^=H!FX+l5Eg?}LInvZWp6QS^>LQXw&kGmLK77tXP-zW*>*_oFjZvdC4IC6U-<ns&Y#fA^7Tc$T+JcpBWr(@&77_;#*fN-@
#mG!k+2JUSz^J`0xJ5xH5#5DV+=kE1X-&pH;F*Ij-bw+9OD}UJx4}@_>K6dC{XLU^00|~|+j2a13uzR@5L1^k`|*}z5Br7l-={c|QMzpj$aft=sCE`~0Xzy`
q$NgRHP*voHfiSQuLt+74{h9A#H#%Gvd!-WW=V~d=Z3Y(wJSxw><_kp6jncNRD7)hiw`9OC(#dI&3q7U|HPI`C>4=HH-VWG!xJ-nQy_My#A$sHxA+w}4-5|T
tT1?f4}M=*+FY4(qIKObOb2``4)|6LGwE^92&h+;5o|C_lS*lSf>pD;7(BFIDr<Ckt4LXEEhWJGL?&KY{p!5BwdrY&4<!Tp<p;NMPZY_R=L(5R_=e7OKBU_z
K=BWU;GlJRbp34_XV_?`uWvJ7*7H!|{~IMi4EQTzOPRZ+Xm^CJlJqi|UJ*Ny5s9!M25HB4233lL2~sK8Al#vq_6FHmvH->}YutHazpxOGmdVH;lbc)al>lPh
Z>VIUWMv~mngo|}Alj>oEO5HCrsrBI|MV4&E#ctMD!G&x)Xj?{@i^c^MI}w-*o>^r2Dcpy>1FBH?H|{F=7lV2H@t=L?_<mb0Vm9@>@8@cmI}|uJc(c$j~Jo=
U*ZIAx*!-HX{L(#nAkQOj-(+X1ae$X1m31Me&Zr}Np)3<hOB@p%RWu{`+C!adHSzKSSc^mIh+TzOz?Cc5Z$SW^;;rg%`hA(5Cz#$D>NqR4#UTY$ME>q#q{Uh
*w4BKiaM0Fd*A?wH8=i@F6j#wt^~Y+R&=V3(-;nLcO{P(q^L5m;NP362bQ7I>vZgKJ4EVGbCRlo&-~A|iSEDhmp4+r>q6;=1OHFSQ+u`T_fR|t(*Q0m&rkdQ
61}4@V>)ClSBR8RLyOJhFh{n8giA~AVCl;GZ^z}e4Yz)un`G27NMV)?W7l-`8Xh)?$LN>!1O3o~^y_90EBT<X*KRL+1hG57s9lhxm5H=RK!R11M~cW<>rVNg
u>cSD>FVhZbPBF|*$Gh2bA5x3GNld|xE7Eq9-@=M6^ak$4p#buE!OGW4Qr*xj%?PDiJpRAadUkB_Smw}p3lld^x<TCDGsTOW6L8!<2<Q%*xcgU>}940a}3dm
qp-Z$cXg5J?Y(++EIMJI_LR1>xAJ?MO=&b}!WgIztsDNNfmA1UZRc<}&+lj`<7tniWRs*A;W&AT`&coOE?Y(X@~q{B%@dF{*PB9U!~zwx<c>yIr*K&V@j}R5
dP$H9dPA$r(;R%H-x4{}`3v)jVImCo(9+O8-zyq|S@a#Pv6&rriYh)Ba>gEzF-rIkjZ?;OF~hh;f$&VcOhkP9=vX{h@R$*Yzh@rBe>9O<INvWQ_dQH*iR$-B
mR#Le7Qkn!G}Z7~<LR`WS?mGCj1_fU@T%~jcmo>XISf4w{)>|cLmpNTb#8`nup0&4Gz!B;3bLk9nmi(``#78X#fZ&S$^K}XwznK2$U`=nI@yi5>wS4Ng4n40
JY~9uJ?68(p~w|Krq<T2_XhxiJ2m+kEB^87e;17DOm^Q*woeh;9FQ>g&FK2mZ`J&T+ZYwTl@^-)vK6S<r@<uP=uPL{+Q!V83S@j<E4u@?(9~01FwYocwn0tL
7p6f(BqD{%(>A469>`Ni8(@4GklBH#GZPwqg=NRHsyol5+Bd(=)yh)fi}4FPpIMr|w0+g58*Yj;D{f>tdG0re%4#IUwea>+zFrE>m8DGv8rFe-P~583Qs{HP
I&suv5zkcpm)r&Qt?Ynp(jLO?e*nZ-u4#ctXDfoDzDfjgY>VlB=h|sESp7H-0NZ0698TSpD*H0LXqv;SuO`>F$Z39?4|=lY=rC_zwWOs4kx2-vYJDlPI1}eo
>X;?Bcv?8GssrlmN%ta?cun^E*Og%2kN{eOxy7H=!R5*Z?YC(wZ@~(=UvXBUA)>ga1#F^;xHS{BgHn>PAq_|;mzAtp_KC{>!qZZYnw|zfm6W>u5F$@7SG8Fm
>4yvWh?b|?J`$18DLpwKiZ$>DiJh><<*mM6Ev4!*v~LKV$m)&3i1VGl<Ppv^F^#Fp{1W7zahRs|FQp=K%8ge&pcz&h@zqP>njb)N05UUFo-xcsb$RHneX7m(
v&BHnqy;q_2&_-%-G0%B$4}sCn4!*qgQF>O6cOy?TOWR52$L<%5<JocDd1Opf(#G%T2MdekCcjVs^E}+BL_mWk!bmKoqYc^QQLGstReJ<ZtiUFc;Q9Jv5m6_
Au3vvylv@nfV8Q&T$Z_2>E@vgt5)=^gn|uNo_-YmNkdJYRY>%oFep&{l$%NT*(`}2o7tXA_!vqT?&ak`0KYH$^pm&0`%H+g*?x?AKx$g>sjhB%*g8si4`ra4
=+EGgrugRH3Q0_wApOvK6!i%w_1vZFc*`sr%b~+>&8u+j@7FR;;i@E2Q`nxWIgDU_ZyHbf^jL0V-B_1Z`{C#n)7oPmn7YD>%J@O6y8Qx%&d2;|CE`B(jDS)j
W=~V}$pK+EkX$BsEI`hXHF-X<^^c)Zt|_{ly#J_PHM8ntxHxsRu|eQz*W@7h3z|rW7Kz!52>J0Ws52py)KaKvXnN~ciu47udyC$LC8|i>KAM7wbH^;F`lSd&
08DH4+ls<kt_q?tu9WFjmmMKN(#5}uWkXDQTxvLUkcfdSc~`ji-oPIU$2`T%VwTmbRxOYhtgF^m`+FIKXVW8w@xomwFFgLB0w@A)4-dH3wSLZZJZ8ApqQ_dD
ISLN{QDETopr>0sv0jX^XAGm~0?QRw<)fWYi0iQJ8qM#4Z72Ygb>Qoruw23>LT*DjUlTV4+9A!Wx-PGbaQS-mN1K#wsrJJ<X^=109Tq0VMUXJAa5%G_!B|hs
LB@26qF3CP5!>@)KcSWHCYbK)lNv-=UD&B)#4zjK+qQTVR%%PB7J9F$KFLw=UiP*9Xt?NHnoG-qoRJYUCsHNUqt|P{yeYvMV3<h5boU(_&!xP*lr87n$=6+`
&I_ttD9Hy-hIaR!x0B(9p4E)~#jSG*__#68U%8>u=la6Or2O97(Pjl|(e^$)QNaRL=OjbY6DfAHRk{suc_-@EnLdh1uuzJhP^~!qMN0L9AmzPJY|`zSgyKMe
%AGw`KCFMnv!P4ETijX|6rt9klbT68%rh(yvWUTb+G!uGA}3o6R;`Tb|4MeKisI|ghriVV+}N^IzHy*u&yo6w)X*m4G5H*@+d9@$PQvJleC<&<C!E4aH$+m@
?8p64!tc9>8oShy$-E-Xyax6hv!~du<rck5v<|CNup=jWsFsidwVtB;QXLf+%VqQv`ZTg=h*I^&HVYn=q%N!0Q-c(My3GtV(QP>ge(0Z*VMRD~jy1ngPW+f!
nf}%?ZAL2*WwgDs)~i|i1q|$((QA5=7H@Ag{;3Drs#hs494o7VVH+D&zq48u*^B<}JQ0O+@H`hMJ5yFRLDV5b5ehbOjK@7YpYkew8A3EV{p8^sFY(bEkmxTM
8%Nx+{vxEpn_R3T*ny$}$vcElkzVRemO|BH>6YC=yu3sHqhSn9NY>ei%%o2=FHKI8eJAf)W?c$CSU*X}uU4nysD#D`MR=yJNN8)gD$c{!aMmRjVEi8dz_~>p
R%lXILgPS}=+!E^4&C`zhCiKGrr?y<N{D3;SWNK8ZHZM&da@9Fj#5GGE&d1_{JAz-blvYhtc5!H29)>1LW|!GEVc_f0N#LURwBuNiw|l)62+v5@`;iwY^+5B
`}d~(*c~*ltMiB<u)3b2baSy3l&=KV!1WTsI;I8lt5~0PqN;dMluwly7wEN89Izh(MACN#91vNVKSMU)_ukvzln#gsH{ZBB>Ts@p083lX$ih5sPQ;O;AmY3n
hPzL>Zs(-`gyblO69qT-+(0gi9gtEQ0l{r%##I3Z=;0S<-U+;kkvYSR^0~gjk~*o5(%DFI1v0^Qw<fpaK3G(hl_>yv4_@SR5q@1Q<SsQe9%u>;J??yDJDr=+
p113ZrN`ejlQii&+k<ZzGE?iTm<AIy&e5YMkEvQNxj1n6WOj1AKSaR%(RPbpD?Tzhs?WUUo|bx#Cjva_0E8GjAZ=Q_1+0hLgMZid7t~r0KE_osm~8VUg8_Mo
>aC()<U@b_f^>&Te<;eDHkJlER7p3G(MZ{wg!UmDTL8S{JOJ@+iq^P({x4avK%W03`Do!_VAN04^EH|n<@Je(9VGsE@Zp3b?YrEv3ey<Bm7#dY;S4TYgs>t>
EiP<z*}*VmbJI=_^-s|@T&KA13twv3Wmxvf-m7=2cK)&!^}3i?zU!T^r|x&76VzE-Z9Gb2`QN>B@rha2IS39B{CiLowdAV&ZES>;4DX;r(9mIVLemZ`{3U|3
((5_Db{xCl_^@74yyC1HKgJo3+p;!H9p_DtSUSN0#uTVQA<663SPMZZNkcVGce-HA@TcY_>qTV~#Dz-U(Vaovr`14MTjyRS%pW+UVKGfxbSADtcH%>@%b!ql
a(veau%|w85`W=>Z*~fkl<x9`amAATojBaD0pLTx>!I-w`9`7ZsB#$#qa8)Ul3ShE4epHDxQE20kKQj{G7+&WgPbGof(}!Yy=;;g6HbdTAilkZ2^!cycwFu&
t?NL$?c%ig?4N_1XOqe8#eH{@7e@kIDkHq=r*O2{Rt?1b3rIrVr#CAM#B6&|E!M8Fp(&S29I>TbQH#%sgYog9Q47nEiaT)3xBgf6iC_zDjwU@Wip5~_vv*RS
03WaP!MsKkLu_h_embEH-lx(oC<qfOizJ9wc@Do-ov*si*-)aF6U$DPNUvrQFjF^`OEqA$|G)hw@T1UbP3?@sJP8z+JIHK}!T%0Gr<p`lX8%zSZOUH0g{E7W
6%2jRW*p@*l2lOxcd=WIOrWGNLVTS}g`fkiFY9!rM**#t&9kl_a=AE1z2feGc^(%&SuJy7FJz`;AWBob5|V+NgpXFWqfFzlZP~q?-lywz3I7dfmBxhm?iGAC
uavyDkB9~&IlUF0`>TBD1I8Tsa!Q*t-Y^0DxS;<B3RYI9%W3fqDQBq;<u4aM54B4wRBJSSEp#w4mH^DL=c{!njSVpc4v%8UHF#HgO0SYo;i(*omda~m7(+u)
W)9iQ=57Rgyvph)*r8|QdgkmGt(92AjA+dbdrprd%?Xp*NSW`4-P`r&17GuiHlnb2{a_&SMMKX`uhe|`$P=aA&7Ino4rG%ldWJQk#I83V&)+6H5C?*wTYk3j
5ubV^1U|v=8N>Fkme|KHt@7u%-mX|qkrNE#BunI1T(tbpZE_=uu=zf$i5u>kG`wS<eUQpuU~mblV!5@*`s}%)6dSIclBbOa8tXz;e)&@UBS7lYt~7;mY?BG&
?u-~ADWB?mbCrr|(LpT*%Bf!Lq3Ho4PI$+;ZY4C3s2`r<_p~T&Dlt~#0EQ0Lj=cP^r;p|1Ne%k3#s_|a+7lCqzDbbO`UT>+sFRI8QkWoYx|{rnpJ-W{yL6Wv
)|!}ks@wcdIME(72wH8Cj=Y-=uX2C&`)JvcDDlId?HjA%vT()0DC=RPXY<mp0p4%O3Z_4L)w@3!_EbI>y==djdXk-6wG51&^VQOFEu!^6%ewdeOvS^Jh6z(M
pix3SGpuZ6vdx5>9Y0utv73sJEy|~l<}6K8kg57^Mexgcrq)@Nla$ohbn(R_SZZvMiUzX5X+WtgE@L=40^>YIzpVn%Jyg=u03$L{l;Q*mnbZRP%#R{GDgZF3
2XOz!g$Gzw<ADjQ#l}jm(Nrx>|7bzgm`-}Y`sQ@o5*Cqgu*1)0UPsorY%WjL9kx{{NV=lrBuN9B_CndX08usR{(LJ-2GXcR50ncms=acuhu!tRy$qFtX%-q=
zhY}ySHTGHo5KVcS7<-@S#2FqE;%>rr`U+CW?spm4Kh)Dz97RcI}L&=35qFQ`6@m(R|+)28`t&uNldSaDfzVOQF#F{zp_=&Vw~#z+GV|v!b?3=DD~d3jVXI@
NTJ~WFEq*zQ~rDgMpwQ-(2MuCqTf-{pta{HUIJTIG|cD*jn8EW0&3%jXeZrCebO^dy#-$gJM222rA1^;DMXWT=rO=D>|g4gr-clzHhu<xOlmqCl5ww*5_z_J
OUS`TG3On;A2Oz&DrJjq*(r$ZTWJ5?$90E9(X^rtSXwZFig_doJvK{dIns}wT_OX&uzXqyIZ(>FRES6v_=64;zk6v7I+u1G{3`*dRWqCnUUv`?7~dR))!>2{
A&##SrEqyh$TxzZu|v2(>}1|7rJIk_e1po1kawH@RU2>@F%F?g6_|6h_y8zsnuI-NALo@+Yfa<n*hfV4D91=kt(-TWv3u<mJtwfiMdUKZti4P=G(l6As~=~n
{+M;)NIusN_1r3<i~W_hDQEpC^U@^5H)OS-oS6rcRO=G8xd~zc&biVlwxAmeM}r9mh46q%w_#O(`46n!O!b4u=KDrBB_hE+08-xH`m4yR(RC!uSc+1k1g4-;
Z<Zgf@w!k4+O){J&uT*trvy{TA8&0vz!vk9Dibr74v51lAei@6%eB{9;-V_Gj8~b8KVUADM26T|bIePn)q=p{i1*XfS_%usrt!wyg<zPLrg;i<n~UjbhGyOQ
;+_@#E~RH-j5^PTj>FF_yK&cZ5CJwn*>+khx+R-Z=tzZUw={8xGG4q3sKye?-VIL19YFuBq~AW05qIwa`zD`>*PIM0_BuUs>>gsqi--Gwa2<jCwC3L(ogh8&
2dWOm4B;#WgbTmy(10JASlSO5@3YsWy305-U-6{M2JdGZ=|HI(4IhLI6ac)c+a=wQsVm)i{}aAqLjm2<6`t4-=L&W#%I~sNXmr*TVh6*8%7e4))YtMw{m=hK
P;OKn{m(p2DaKW0Jyc=d5#O>1v$U{KF#+Mbonz|B3vF!CmVJzytho0O&zo`n3?^Vfy^d;^85M=ross_69H(zfMFg{8>%PdB1srx*Z1oQZpj-$7Rg3yTg0r4h
Usdk90kHj(wn^g4Q~9PuY``?xONkD-U&??zuz5N4=%I{hRmttYm1EqV`gmXF`asW<?f7>O7}^FUS|)kb<?qh`U<DCb+hekJTKJhX?!=Y`xO)f8y2j}UgQqJ<
-CQOgy(tR(;&ydQJr3RR|87KuR}>2-D_<oS`uRS;fs=dd|6&~VpW_k6l28=0!sJ-7ZTEu^6JJJ+xO*S7cHySM-}NrMSGwSi-EujR#HKbr&6|#)%ouI2JqWvi
=Z2}QEpQLx`T;(#Yj)89=bnh#Z5PQAoa&83WL|UQ=_+#UPwU>A#Iy(tj0h|GT@yXC^&Rs9>Yse%36ltz|Mh52-Yr|VZ9i(Mpy+mtNq=G7yxr#bJ<VkkvJvDA
hO%GCLDgN%NwfFry6)%J)H}d?S2DJ*O+9K&C6+76u-+%?Ew}2}tDEelCxi4CtKPT<X6~|lX>K;x*rYx(GgQ9HGoe^d*dP{0z27xyPPmN(TDjv$W_O7`-N19Z
8SFqsoEvc%tzHm7(OIXyZ3&ZG(Yf2%SD2HhGP8zB4RU8Aga<v0KI}(*`q4-8@fAH_RiB7jj3|L_wK#1zkA0eRC^R%RgbE4%vK73>6wiBxjz}-uuHzxZgx2a6
G)Y>%qs-g)y21xlZ{r%q&ESw?vRLKI0;Weo0qQ+__x)J6gy+MM%+`IbsTS-o7j05E=k8WFpqvfqe>|R8(y2x+^V4ZXF(R)OD^<$m_jzoEw-8V**D7UR`a^9v
(d@uam5@)NtegE_2_$NNy%N=R_eso&!dknmLDrUQn_U<6Yqas5E<Mi@_@x}{mM69zXS0Zju$<a4%)j-WT6T?G(4iDxzTC~qguvBz@?&`k-mJyZmEO0LcX@((
-6;vasN+Qflm>(5z)+qmif@%w+Md$V>}cB8p|?`UBA94o#jxA=06UQH@Sz6F+L&jv?^Gp*Ncx@UHQ3iKhoS4Jl{aBYxaX{9{>Pr)8<*^0zX$-Im&es!G$CYy
cL01!dN<$(lLOXT!jtCU2AtWz@W_{?%PT95uFB?exe<{zvs5$S%gbna3|(mf^;Pq#c}&_rsTtQe$W$j-dSmjC^G@YY=-X*nsz3CYq9IvU;cXVbHyn0Gg3EaP
c_@qZ+6NF9e4#%a_c81?pykB&%Rf-(6Mg91j4j`4aI&ln)MjpFzy6QlwQu3|`lKN-R24^Husdjf+bgXQPNd^h#asCbk+kOZaHtgC20Rz%P6`emCh=!<MSIdK
N|tF4pvMpDM5c7MjLP8~z4IBs+0XW92>9|fl%css<X;T(joqZq#}J6ukW+?Nnl`##RL0>aR1_=LW@=m9`^VD1lllf>G1UTD`#=fGKhWR0a8X>U@>_G8Gy5>Z
`dL`&x;n+b>gTp%;nym2hEd;jVzcSonAtnuk(hoe<ad}V1m)Pt-kGDZj!DF?wLqa4sSN>p5a4?KPsmzMjc5rtYx~ue`DDTA)I1yMHW@2Yz!b&$yIBvtXdTE<
<1u@>hac^E<3GG<IcH<F;+D|D@=1AJf;Y6uPNF3?80d_SYWAdlo|Gf(%Gk*4a-x!=Lq_rgT<c<YGy+vOD++Lf)fPXy-+ARuQaZNQwUnYO2T{`mrP1zk1<$-p
_-U#-!U%a~pCqyV&0VJFKS^CFs~&c8o%OqzYBH0A4D9%iQG0kXh)_2{_fjJj{J3d$$U^iEIE(b|{lMhu+H&-RRiUr)q#M#H^bX2lm9w+p!dyS5k(dF|An3^(
y-0WdhiaoMw4RaqDR_QgSIUKtgU}@@8hDtfOfc-8WWXh-i7*vQkdcI5@vDGpuU#e^t|mTBy$aeEdCteV=A>~QD!7>>e=;oPwXx{#XQc*cZsaFbu2+i?c~Ak>
nfIvT{~#gCx|e9_@P`dy{G2JGYcqN+?BrXnt~XJzdxOB*DS5p?cR0;px~^~gy}N<x1nN3R6CWi%k(FBj6a20NxpFUx8*<75Nt<Iuc1*!WXDObW!th+^n@V7a
OldYOvPK=!>D5!$k+rxivb6I4UtSyWFCB1;YY6z|$|}|1Euz5yN%&M@891BH%FWN&b^pjR6wGvlM-q5n+hix9of1PNWL4Dq@vka)8ZU2Oiqm226y_SnYeG?!
wl*2IyLX@a<o^U?Q|#v)M0{!qq~hHMsJB;qbtM)Hz%gzmFg{d#4b}(9UZ;|=dOn#b5D#I_{tRk#&mN~4#Kd)AIIl#iSC*p_W8BurinRd2Zb36m7~P(<O|+I%
gePQtN31*R)5^8qTyL6?{1xuWZ9-k!-*!0;!yhzf6FI8{@k_nH`2Kx^i5R&`Ef1;n`S@NOo`}Z@yXyN8TVq03pdlqcxe%k4a~>Cs+)V+5P;+Nq;bO2jh-lP%
EeeMR9x`>N>nQMcC4DLT`Y|mt-mPY*LY3sehwQ>Nc-F=8<C>8_sRwc(KC1JZfijgOv3ums2Z9@*N*WCeQ8PDHs@#l5?SUfq6&~rZ@>)rgwGWGM!jyab|B6nk
weY|(c>9$zRI+*2pi966*S_{A^U{IqBA9hHpxL+hJXa|Z@v=96{w#vm$5cUpHZeBr#4M6|O8G)5%Kc(-h3cI>Q{U?_zUC^&<Uux4+u6%J<7rpBu^STuCEMN(
x@ozXwQ08r><)jE_JU(q13*aJEoV!8wurJkl6R+<!WD@-oV1`P{m-HUD!rspSbk`#C@E8b{G0wjfde;6D<wuvA({H}X9M!wV`{&<w<ely!w2WtjUIBA#*F@P
j_camh!CC{LX@}Kv)TpK68*)dmr#$#bG=Lc;@-jElVn+Wv6`Vqo|L^pL@LPXCriD&fN3#g!9-&PL62v{YyAE^w{PX*-|j6H;scf_ah%#Sv{P-FbNtHDIFIAl
*UtC<hWi)Tq&v9qRCmgxGFzYtIqfh)CzZJxsp^WQEjkM(-JSEKFLB&M1Fv|k)rjm#=#%bXltZa}1%uV$Q`mW|wi3_>Q(@2=1$bz>XgVgv`-TY>KlUu1tDtPF
NaAZr$*uIp=$?KqL&$iNGMD7Y(&4FAeGKGZh8+_N|E6|lL8nF(m)|cDD5wzoTbIxhvouNYMchcHTE%d3Y+qmAh2%k#b787E0Wv$R<TJM20xyeZ;LzT59k>Zd
hfA2<(ebDLm~SeCsMNiVRBBDEaYlAM*gK<SjT_k#O$8GPrV3TSqjp#XhxTU1vUv%3o3=-2``Y75{_S`vpCkoiZ9e51DM;=IL&G>18%BHN{Xkglw+YmI3P%-B
1r?_SfK?UWQ|N7-)Dk%6KOlSnC)=5a7rb14hsT0}sHJq^`L@u3`DyuCveC&&Ty_Bb<H#mfJ2CW4=#69x+1myzKiV`g_iXJo$3j94ni8a$<`XhgKQlX~rDp=o
WA<&rBP<q;ODSQ+JtVPIc5(N7xLbafk1}I4yEZSpJpPj8^G7`AeuY$TRt=CT4Pbb{|E0qu4f5kBcHZ{OpjixGQf^&{ZZ90}BvcMb{{rCA_dAdL0IW}PCEGS>
O}Fw|uZUPl4IWGQHdgVHtslQ#;+QmslIqk$5;t|NHaWh+=7U`M1nDgBrl}bX*vD{`3?TmUD}X-&Ip`*8xG^{V_xV~N0GtHyA+EEbZUnb9yHjT~D)|IXx#m$>
r35Tb7kE)ogvCg})_b-saXo8me!lj*&1{r^Wjc#z7d2|`w^03vy&WlLF{O!{(x|>lc+0_{#Tgwhv0u<wQ5QFy^KG;(P|<2R;2Q(@XzI;79rY>dFubblazOc}
6|Bs+PNG&PciPk*R~vU$#N3_H{i}T|)!yGW<CHr9ay|7@Vk~TLI7%BcWEtw?y?{r0Cq7i+X+Y}F7#6E*^GE@6`Lq!@fZa4v<GnaA_uhG}t)xOZ=L0XlTe)++
^Ir-)Jq9!)-WAP1>)j_p;L;Ks?@#Pa1eH0TL~(o_=Bz(&S!e2BWuE-XaY1gZjmGNJfwlDffDLQhi(4Y{oF!vvnYFN7EfykB>!T5gvl$<?Bbox2E}1&1n4N}N
{rT(J>^Q8r#ClZd{l<1GI~;3R0y;$rw@Rl(@OH14nRoR)rK>+CCTe!$T|%a~p+bcMt#D2!_xv&Bzb`vG(XCGojN6x<%WJf_RRo?!X^Z~MIwZ=rt&RGmL1n(!
Bj3SbPE=nhEdWU|IrSGdKwWR6`FOUc=tN#~g+qR1j<qLh8%lP6g@<^`s4hS$@i6phU4AWkV(Lp#th+tanVZHwm}7gP!kQeh$DU|TrpBB2>Uf)@Q}$(Ny1kqH
D0d1;gm5R>i?JSRptc!AX<!MwyVX{8i9@-|t5Yd<2B6<8UNHEaE^H9K1L;^ArBf#{5BuH$THiIQQXf(U!n|4-0<*pBAE@o!ki=_$Y<F5&GCxSk%<Qq$E3KFb
2e5a26U!!TNo)MT8%9w6Rqjxbs5{6x71c>fB1s5ch^w>1nrAVBj>aQ{;7wKJ?a25zaZGMSA5C+CI1MP^BjL{^@mIqqxx`H`)rrlWniF}#D4J45<FPy-u$c2{
h_;>xss*P(*YT(7&%V)i6U{EXIU-$o`i*xuCMU^Q@)rx=E9mkF!cF-xATYww-6&tkv7ww^xk1~LOt(rB8?YvK1RG0QdBcV2@u^)S<9gJ8D?%UDhktyx<aG8^
X_oz<W3W4ji?E#Ot0h6Zy8@g~={{4sGiNNzN-(pxJw^h^fyYMjKbaz$R&H5FSsac7xsBvGEnS=I8gCZS7lvS*u{Sl{MZX^r{C)Mwz4#zF94mcW!n;N}&A0Do
a1@n9m)%4w)Ya3MkWfMIBiPvKN&>keH7`bnRD)wbOD{dBo~{O8Ew%ECOY-NwStc<{%`wE<Bm+X2fEQ1m<T3d7!zLg+I59##$3fBv1KbqPMiPS<SVdU@PX9|D
ya*TFKkh9$HJ!QAOuKv^uz?wf9>EL!eLzICtpET3%T4^FNBYow00Eo)0nYmdZO)bivBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/ref_r6_prato.csv')

In [ ]:
# listas/treino_v2.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/treino_v2.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<twuA0bKwx8cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTl9(JVVGirv%q4sY9B44hig?`BE%r#o^Zq29?%ejt>DGIwE3W{!9Ii
^T8pa9s<-_nLk_t<B_+dejJAjh(+J+N8U?`f2xn<Go0Y8oPTpiTaK?%UG$Sjy(y^*#+tYyjUVk;J>~er0Lq#aW|rHOsotdtZQnaJZQK;e*L~nvL)e~YHZgq%
y|t{-ugR7Dcw@u4+RaA>@wR_)+(1(IvWUB%z5xJsyi@GBgq(^zep`iXdO5Te_WgU(g`i{+aPb>`3Kw-Pg`9|tJMZRWSk!W-!3HvP6$Ci@hZaj>UmZ{C1<}s<
)Ly&9Z13IhYP8cLI@P#*j_IgP${=Sh;pikjg<k4cBq6&X$GVF_^D|*MX<BDM)h=C=H9d!6;tm)sCXTboA$7A?zjnLNiJIEA3P7TGg|;U2bs0wGE+Uz;X9IH!
z#=ArZLUb*-j6G3B)i01JvZ4BkS;O|$sEE2SZah?`|w~RW$Z_}XumaOhKS~2h+4%)6CS1S?>%30l%EgXd)kR#fk@aDKlYnECfq>5pcqHj1xUcnweASV3pvbs
V_4;m!Z{ku(YLn3?*c!?MzW4^ms)>!>1Z)*1|a!?>ZYyi7|-e<5nj}PdxEP#Zk}QYrppJ=L>nijHKG{J0XMrqwF|kEAL-CbpFvkBmv!um%ihllIi6d>y~V<V
lcBE>!N=VU^b2cbk%pDGw1lBU?^(CZ(2o7QaD_VbF~`yoe8em}0kc8)@!GCcWxT+^x(TzSyg2W%ytCvV#}`Irb-G=hDQ<dGzf?LGr;!;iCQEMAX&bXvTh61$
X6nj?R&JLrzm<+e@;)a~-39}3-$K~i4<MhzJ>k@@JMg+GxHM;c9iN30DZ%XBFl({LhYno_)4I5=jz5Rz5X3hRwJ}azCik9{2hUNZkNws?t&}YscR*;$iwL~-
a>(d3AuR~T*Paq~M>fv~b^q8TuZ?M$?ncqB#{s{805GY5RG+G=tD|do66(z<e^F6s$yJEx!QaRzRny<;q_6af)m@!me_x=cK_75!%%l=3i;0$m_g^XX$XA2+
K!=A#1gL>$PKtCQ^>w~TR~o#M$szRRs|fk+$hDY`AswT4vYH;cKej<dW#=_Y<5#p0cxgN>tRC5}Qm*iHOcJv!E9_=|lfV6R!mk1+J{DLKc2uD9uE8GphIqf2
-~H2k^;}CaHT6%3#P3{5@|E}FK(M7ML+0#I&>OF3fzyWT)(-n7=rB>@Y<G54e}WIpV(Bu37Pw2#D&}Zt{vhS20IICED}Ljh083RWl&I$(^}Vh9_w4uV>yr6f
Dvr&*UT&(#;A~@Q?F5I`p!01E4~+%cLs0O{*cMDIy^_q-Ka3=3=&U}NuM6tO3}}+kaXy;Oio0M6=~4nJQi4%{Yo)DJ4xLjN%>~|thmqNY)8*@R_&UGhsI2aC
>g2)d(KxuFwI};W=n9ZO$`Y(+bCBCdu4!JYP`m}P_5~<QP)hA5govSa;^T(nQ@lK#yhB>CwiTiB^Gg=(YBQomCkmTFe|XkEqLc#y53K#u=LF~#7jhBx!@^^C
(1bWyi@gyVo+rXOpQX_-57*~(*q%7f91M~{-*g3_fq*I_njldYTe!t-kwrp<GDTVqjCw)yEtHhOD-0-$(Z^nwL#71~5%i<qzLc^I?KE$<y+3Jqe$iXE)qT?`
o#hLy<#!%$rv&i1E*Z^(`vdP_tWPO>sGp>lqQwreqOPdh_8`vz!R5L1FeqbRj|6uoR+4f#D&!*i!ZW`?fW`XyM=uSfTBdvBPL=7clnx)L))ekjY<$lC1=>g|
Ed`_*1~<W2u*+kh0c2>d4AjrKEb!V?>Xr5XQf(C{RJsyM<pL`5pzlhHi{~8WjFs2O)7b3stNagtbxvW38}XS+^q2U<SS6MTHWF3nGlW%>aW_Sde&q=C+z$;j
z3p+udJkf)`8?0;#k5UYn(z)4yU+LrRL3M^6CgFWizp}XabLn8Z@?Z0qBzaF`b#)fYt;g$qf-II<twwO^?r<{MbM66MKFdyOUmsXD-()WS8C=!)vr0zTilyN
m#ofrqoSo8G01Q`z}Q?{Y?pykTl$z$D3AvHVE2*7%DcwVgVrN+f6g<T4f~lW`l<H8Ema>$ezPa?{-)wJ>F13W@iU>ous&=lQc7U8djcC;r|V35u7$NB9+-@U
`eOllO(`8rs1Nv5d^vP4+!w`AF6S~_oTWlkW)_6*09a$0DG>YHPbL{W6m%s6!y`q6_(QfZq*V%R?N|t2{SrsV0KcVupym5?iIg9G-{lK)tR_lBg2|992cam>
1k*@+S|2iA<Vg!zIP5|&yG?7KVcT5NQE(&08)CY5<TT-tVbTyi5qXKQ&xx7DJ#3*N*z}&7OIs7so7gYY-8T9N)wB4QD<mQH#05=%=PI~Laz<jefi`nD3Vk|%
#M@9-ZY9SP#An4XqQVUo=!Ua@Ul56(%}AfISfQ7yEML}qEe4(Pfoj#|Nq(Hq$zliN%~Z}oFW9@?Q`rvvTFsUkTEj?R(kVFg2&&b5UX?dhzhrq~4q}KrNn}6m
Y6(f$Wa~rCmGX>5ys3}_{hm^kbj!$eh8g%(r3<easce7Io4AL7pX!_^uZ1%)>qSo4d7l^y*QfJ<e&H5P2<mpPm<3dPh?`+XuOHx)zBRP!em=g@)uqX_4Y%iV
s1yz+h-Y+&SsNF4#>?c9BcV(v>lLRdGv1_&7kc)^AW*%vP?ZBfb5sEp8cvmM4^)D>7;7{9Fgj~7fCu1Ve=1^2itK*(k<sj(!m!jY$m9W}c@OxV6`GVB&R0^0
gYuGDu@q*_l(nV#VIORsUzO|V{;5&1&y3I+g_H<H>Kb=<{g|*9g#K2%_S{3}|4VF6Ead^k0GkjPM&xADaw~x2LyHgE&pLzzOX=-jU!h2~o$08Co;cWQHS<k4
m?$&sysl%q`M`E%F8+-V!UHec|B5Czuj*C50MnX7gxs2;3-i`8+|Xi0{J7A3y<C|NPM!8b;-~fMOr#y(4}tEVfQbCKaiM(5G;}?NsNb`vRCz3Owsqk>#@0O6
#LKCgI5(JyODpXZDOj%k-x4hnM7ijF8VZkRX)FW~#SV(&*FZ|{Xyvzuly+mN>hBo|PM&)n?eiU&DHC*dPXT#<OI)(qR_rn=;@@z6EdAvzbce3!y6ba!LUqn{
qN?SOvUEB_P#IynlvZmVmJdG}XXXrR`V_L|H;qKT<d*%%&`vvnGc;GJ&#l<XC~D{Llj{>6Ya&<`oU!OQ1}s6vKVy7TF%C^P#*AAM8qg>_m&p71*f_OSgV~J%
jI|LL6tQX@jQ2b3(C;qwwb(!^13LLg=QS3*+34$+8Ke;t<(zt`)6ud29Gz%yOi#1f<$g9GBgWmg_ul8pnq{LXkM>;B_EMa!oG?5O6q86!)Q-wP@#CsE+o-II
=k4>((<}FO4tFY?z%|Be{yIpB%>!qM$YFei&$9-$DJ~|~WbbZiu(cJw4iZqxW0<1ksf`R&zgrdJ(okFlsOY+gAhRhk9k+~(Sb~~NyPBT$-MG$(fwz$8K5Vs_
*tVZSMqa+dcy>08XQ;VInZ!AbXlI#w<04e=Q$h|jc-FYFG#GOKt*%gLIziC@mXSprW1TbOM}!Fy#_)ziveKn3$xI~;yfp33d#E<VAO&i|kdh7W7(+!Axb_o`
-s&Sh8f4|w<*M6_&T#QhkJ*o&i@EIG7JZ&#EItN4)7X@x4B}N8c%OL49~XClys9ljPe!S3?xtuqU@<}2Dv@cPTDl1ivS2qbaZp7ZFT;u#78x`rIkCI!Hk7wO
;M3I{w}E%}knRHwEgAKQ1D<zid2NMZkgnVuFt+Sx?ZJry6L&vZ#W2yb=O;9LXp&G2W5WoKjGEJ!KF@gmp!>pMQ-o?!HF%Mu&&uPr^5vm?G;+QMh!rZZr(cP-
keG31)=*XX+oIKV-xa5(dxleKD$$o#ST}%^8cxAsPkAvFA_&jc(*gPB2!QTCzfWRdV7%<&ga*lgB}U$xqI=%QsS4T9!5&|lH}ni6E$*Q73@%v*zfdMtWg2jt
f%F27UdS|XBJDmcBXPrEd=?4km%B6^vC|sU%W<xAz1kdFT4Fb;B*!XDsrH(O4?lKR7ke9JOSZ%)V*;*g+bsbW;k`;?m+4Ht>>iuw`jtu`c(pUSLf>QNi6OJl
hOch_vfp`7^7MD?qvh$ED@`;}6zybmCoTt8*24$N*UYaVe1kR&==IPTYYsDdLsAl=_Il2qU9xsX#S(tHh~f(%Xy1153y~+o^O2qB_+n%8K_`?|R`YlMBwoPM
TGoI{)w9RHap?PQus9;Qp2}5$!hcWp&mPY&tR)?1zLN7K<!xSXEYa>1U8nBX#PrWmn(@vr{$1v~UhI!xyvK8rDpk7^VT<Fzwtg_HE`x-|hch;sRVnxAcrR5_
vJUdZi}a*#LDFd>-<us<k97HQtju2FU!Tm@!VlwkyJNgdt~p#KBgES$*(P>ok)R^*jqBDjg@LJUAK?#F&gOy!9Zik)8t$EwXI|Nxv4_81HzqBC;(+{L+|1cV
fn^CfWd>gB4bm`MY84Mh{NsFuyd+gyMg~IajJ$$HR?3ye5qs=I^x{4goPt@12zy;ydYs=cVq*Q<oY>@U)Zz`u`DrgHGo2}QN8F!_cq^|cGZA9JS`{jkcSd*9
V%Ta@UlvXv)U%f!yYo~<*1VH?=cj=^EzonbXo*y*C`N_7Uh_?)R|Jh(tN~1Y;)POWt%Y8G>YAL%@FmhHo8XIa-<P*8;6u~rLTVb0F`(zqn&AC1rv2j7;bxCJ
6wXd;*;48|de8PJbbah3uPd+r(pfJsG6?7d9q9Sq)|8w}@<i0ewkFB3lfdn@zg6dPvyKG2V`)0K42DlPHz`n_vozw&9=BFf^o~tP5%5&^ZWfxQlEX;SJ_-+o
(3KAC%pRz)CZHvu@)TQNm>M2C8|LM~P*%}#RO)OA4Q#LMLm+>JCNK>C#1t!IQ=1%X@Nw@CQpiX7@XHhke#=S+w&D|`7hJ%n(s`&$hj+u7nC5XV(Hq^Eowhrm
V1<sMz>)s6#QDXS7RAVm{5FlFoyx#E5OS#E7nzI=u=-`)a4$`MyiEC6d|$Q(NRn~Zn6T(GA<Dn-nxn^pHcc|=^?FSX!qJ)_8xO$KhJZt#BX_y&qE|IV+l55I
q4rEU&m&nWS(Mm~5P!nQL{4|pq!I|^^Z!X`Pt(<?gB<$@@`*1rg3va1Fa8xA@;@-Z+^WLqfpRc193N?K_py8$PI=A~FIpk6KYm>+(4gRF!r6B~uXvv0bdV)(
s_@thH()jZf#VuQ%SQ78D^NkARjT%79Z^h(7J?St(VG>Bv;sZYg^9-bRC@(+1>E8+#Pn7$W$q~0`t9qy)Ei^0tW+4<9QwEDamyrKQsiUO1YB#_S$cb0n{><Y
$)Z3rEvHS#eX9AJBv=C{HPcObBCL7$WP8SbkWG=v3h_Gl9sV45_Pxo>5OJ^jwxmRm&7i(E=RxcNr*=<4UP<K?1KrIB@7WR@i<~ofS~qp|PgBDzf;!&A5O(-F
`O2b|{2Ymy&YgAfy8ix<J8yTigg0mBAaAxu$)!6C>-)EZiMBGHjVT?@<svl}elgg3$%YV$&U?mI$t<!E(C#fE$qM@(3yv<^mE&`lrK@lS)&9jhjDIln@qi+e
U#w}?*BQNVo?VXkrRW|I4N2K#PR~85E)mXhqJ?a)(8)~7gGMy1imTT;8yn?VdeBvsc+xps1XT9_Dkc{|;^8yzQz)iz6yD_G?B<afX|{rZbB1&BvuQA??>8H-
4n<~Y<v!4xSU6a2QX}N*lC-@}m;dyCU1xh-VA12ZI$JLwQYUohA}@QkBP(?T<zT(}1<sFB0;eUvU3-v_#-e;%r;G3Xr6cBwy7EV=9sANSCgAUJzrCsT&w=Bj
(-kiSb%O1Z2^utzIhL6lS5O>!<NFIbNE$PpiTYllj=nT_2Run7b&gA7{_2MhP^SXhHx)G3>W=EJY*iAkYa=b_&q39o0;Qz!ht2-7-kRY%S;wZP$F1(i3#}+V
m8Ir0Ysy$p4W_P6DOETyxnl;to3>QkyIH!!F_Ge0h!oSs|EUBMOi1SS$TOnqK6}YD|0Ko7j)B=4O6|~rX}bxsn44go)M5`eX~^lL_|NP_0V7p)aPV`Ai8s;-
EnA~-1(!sfMf0K=8q<b+Z*kBX@V2e<ea7EgtS&bT1{z0{++G2Z!LAHB-QSAtc{#)iWEEqF^zw0#G0;5vmL#}^&$9|}<+I+13Wv5{YBEbmE7A#BS@<|@K?!sM
%Q!@y?TXXe)8q|sdz{V)dn|dX$Z;b;AHjE4N<CJ%GBMa7C!!LuHLgp>hVSt(hHLIR$#^+NB#W8`L#Xd5MzZjbie1@{JXb!NqdTE4dtdpFvyz>EOg`F$FHL*^
hMH%KAnx=WLol2ReSGLargPn$oD*};LtVD+d>~?v41B1T{_N4q+AAt>e*8hKeCb@hxXUygRA;?cHMr&GEa9fN&*`4idrs{IRkXx;9ELyBH&;BAj1j1|P5Y^<
co=KbBp|JmA97g!8uF;v59!DMSMPx>(DzPSo(_e+$L12#Yj!ct7gup2mcd5>*M-SvSBx!R40g%fC|~_3QiW0AvYX+o&->AJozbN7w8X<&l?CoM!IqK&rJ^-C
#63P>&9|D^!u$wyNoP2@Wb!W*9TVC>|HA#32TKE2wAZ^w=WE!JqZVoB-VxmEP_eqaI=^_wXxWyzrs?cI_PJ*62nAoRjS5o-mS4(e3X|tdyV*G}7?;2H{AWX)
tzk@cP0qCk4NgqSY0!MKcfXkvsn1xF)`EW8TeM9;mn=DY&!S>GUcYuXCXgz|+K^1#%xRFcta%EzE>EI@JyB(%D)OCvP8(zSB>GhPz?SR*Bl!TAlDUJuY=PYi
0K?Pirk5A}FVYFpp=z9|*c5Yo#V2a|nkmp>Z4DJ_U&i#jG=5VOJY%_0^SxJ&YR>LubksJ6s&2TPleyQ3(-hwp53!Ai`wNgAjVL#0LwwkNryiK<6c?jmG;T8G
5jJ8eraXwD^!hyiGY=<Z(88&{`@WXoSZMV>m9%@-mr7%t<H}Lk(;gC)<!oCm7P{WM)oQ%x13F^I1*klXB8n(CzVwLMKawihkg+FnCUN4VOBxI1@KIM^@$N`R
cEw@={fulQ7?-Q}zcwDI%3iYc#}>_yg3ix|TrP8oD1@d!Ow-iEdGaPX;VjQHXuh{79Ksb-mb)pyb&&k82I7B95*rxj3NGftdB0o2Df~DK!>D-}_8)#RLjp#C
WVTg@ug&3qOJQt?@YhtoV7&p@F`APZ6RCwwFxL_PGLaFxS@rkvG*nhwC~3k!AXYcjneb2)R<`O-t?NR&t-~&lck@czz&zEei$hn~>ZfXphm|&4L+!p+*6fBb
syK00T7Gx8<j&aOd%z8L0E_t}6)Nb=r;f2kd(P#qBRQru8zdTR@w#M7M>5B}HG-(rm&0sT%imrdZkGe%7u&c`9#8ir_hZl&^H7V<G^6qwM92;yZ(!8oZ6RPb
2edtA4?DH9T5(5dzCBpYG?9H%!!*sFrgR(I=&kZ*4QJWY^#lc<kthw7=im65eMH4Q!r}*A2U)3uIrBk*%P^FY|6w!17a%$TB*Pwny#hUISKHauWwp=icEl!f
xsVJFgS=j*g)!#!+>o7XJgBdozAjzEv}Oy-9rZh9_Qx+bw&eMa*S?MxMSl4FZ)UVp(Y%sOJOck+ErgHgMJ9#umQB@nkwsdhP9cDR0gflTHou2QL`9JiX#1+k
bqE5v2Z0i#Rq=!>*}MyI_PBEtb&HE1bJiQp&1_EC{{Zjad0ZTZRQN~b-cVm$l;B{J$TY<`v-095NE{KhGHsldr=#xd4A5)I0+0ClHhs2Ngax%|A=wo$_M|j|
Zo+0psP&mjXp$}b;Af&QXNMG1jr%<G5Q<qYfM^RNJQUr#X>YT4`_OHPh56RR7Ad4(G{TKJEH~1kV(9Ku2T0|=y-#+VR~lBEO@_(yPt$VkbSq($eyPP^^0wvG
Q#$!J1KKKYzm4ZcliV{+T9kLFDLk{A#|>Fk07k$>Lmd?Kj9R9d$m2_j_cJSd95Ss;#RZH~r8a$8YU<Zhq4sAZqu?;IE&?l-3$jPV@rTn;L~eshKWH-z-d7tg
ULVxYBg#6wjM_-D{9gAb$c*Y(LrJ@rTV5eiRG#44G0>oaejM6lKQOh4g?jMwSeiU__}GJKWwc!qLLJ{RU0-<R>-N2B6>4}PIoxUJRnpA~tBM<0!||lLaXvTv
oeBQolKfmEb3J5A_#&PLQ4hM?C(?aQ8mBmLs|rqflcGKcKDGH%L{~4W$9v6R3#K7mXAf)%>ipN6$MJV)FBE`9c10hoCg*G~zdjeud5Sjv&0OQguUim|T@R(~
<M~s)ZV|n`VYm`Ia9&gk2XPOMY(zQzE3ho3(P0SFG}$weCl7UrD%0C}@u8<;(CHpzXPYV}*UH(5KbvntR}!hpr;_l8QKW(u7T06}l%9S!OlRNb2#7s6h?I;p
KZ+Rxj#bP9mF$wI4z`U{8xDdhwtM&NMTX&j|CcSF;}H<K2%zv?B0(`&cHmN*vH3PvFNDbhXF!ph;S-=EfTFNDqPA0BQT(ly3mEklTs-q#xd(F~;pBMv1&s9J
31S9@x9474?^)kBNjmGD*1R=B3*#!!a%<`PdIHMbLg#hV21fuCd-F=)SB0(uSmS}C=Km2QVPDhJu+K>#sWH3tJ2wcHJ<QFI)A?r+s{=8(_GdT)so*NgGh>Vl
d?(ri8^)VAkN7@ope?X%!{0nU8&E=sB>YU(D;nC;N2pPs?kl(m(!d2(c_+T!mM<oL<_QQq7cQL&8Xp(G^~I!?P7-d)!Y2t|Cy{Ecj8Qj{YQHlt9i_nH3YqT$
@^$MWqOm_on~FLj6*mEs3-jw}v-H;X-HPTQE!FC;7o=i+p8Z_|)BUAk8nctt*hcTUQhXYECRC3rj5v4;D}nF-khCannOky@onT2A%pHKGT4%bC<Dfnrc|GcI
7J5i2<F!U$sFt_X?15vj<+}7h{M8;HndlIl7u(^xi;|2E8peF-E!^!2Pm!Cr9NZswztlfEe%*{B%Wl-{{EXfgZ)Y;XRt%x@;9?^FOvv(nY^0$il+&vPSOd8C
Zzir$hR!y*CM0Y<3K-Lx*jS*@J9XHtUeM-T+-_;f%WCB%Mi}YwHqBqeISgRKyJ5_z{@(XN5z`esCTP^#48op(X>eOn#|1FB?|hlU4sW1^>wHHRTq@O8vem<5
QGBcwTIi3>D%8hwt}a!z<JxYpdQC1J36Aq`>e@@3@&m&g4B}iw)}>XuqUlSrV9<~o!n&2qOx)&<!oF(TmezRwRFOVSj|7$}^`fbTrHNyanBau28@e%uM-{NO
4<QTJ57%XhsJbXkMSv<`X8ttPVbg=2M7bf^qiVc~QjIZ0+&wWayotma!p<9N!Gga9O-TZWWJ3`w*~)C^G}Lkt1-&;63?ZN9(ik7edK*wOvq*kHT*AZef~i};
A4Enp9pCqM%KL`(GpK^a0|3$Q<oqExy_vP4wTs=1u?L_)lAofTqWBvTJnp+rU!uwBPCww<mL?4iWmvf7?BoH1?$CdhS?u*4Ohk?rn#)x4jMwkeXm2)AjdIyE
x|AGtOXsF&ws_NmY_J=wM!8Ibn+3wQCGtbuOougwJhUrKgQNVQ4EZSD!8xeR%ujvx`@g*tOzbD|Up-KuYlm0_K(qO9)d?OeT<UzB27J@v-0--r1XW0Wp}!L=
Jn!xQ7Bv~jWXE!v3RZZ7nE!yLAD5v`0qu(8<jk}ufcScAqm;UP<?5s6^Jve3{~Oh_ipND9#WHA_f#?e5ANA51kFKI4#50q#5}x?2pKm;yDvA?GV}c}}u!Y`I
CiSDpWlYKT5OKa7gRQo8-fiUdr(VaNmc-|OBIp5-yDb{*vom=2(hxT#obqlD#znhjI~RUo7rlPF5lyFFQ}k~#U14Qn0tPSbN&z!2@#1zxRaJEj$P{^;Wyvbh
Bt|1r3)kn0<LCcNr@p}eCOcI616Rwqe!3lQgem!vQ^3X~=5o@MI%2V-8qmlCLH&>V%g--x-=A|_9RVtocTPe#3v}iI2M_444}TpM;<+c>))#i-GGkm7n~hU~
tR((QZb9A{rToEfd=xL|2wz5}Ev0f8!v%Dnu6{{Zz7{3>A$mYX(G{7j1_)&5Yo<B*sA{prCAYR*oRm%{moj)H_L>Z{|CY0H8h56eBfZ|jhT`~BVGgFNQhtfU
A>aGsS#m2C7Y1g!#V``&m-!B(M@LwPTJ|j!Gd8Z#Z7HI_DAmKRxoy*@@XmYB^U0|sqJJ7`ELK;~#4IY>Q7>j#0x-NLo48!cF0i(+$dT+ZbLH9><?T(E9%dQQ
Ajg5(z&i?yUPwjoC!DLO(|%XtYOb&g3=Fr80+91_VtTSThMZrcBRZt3GHFl>J4#rg739dSh5uAR$UuMk=A4@}@zmR6OlnohPO90~6cn6VwnS1#w5zypqbnp=
2?*6qBH~qCj#`IuM!}(Mj}>V?TK{)XpoXPl3-bgI7d=0QR}7|DeXGtf$+n|Vcv~f`6%=>ZhOZ*Rbu2hD#@RH_h-R?_5NjmWP-x^005E_J&^~g{^9!bsV*0sl
-)S)N1<0vPTvn639eU<b1pIdt&EX$^yp%|dme$`x{Rwj!<OOlLKJBTf1qx|Mtn;U$dL8F$Yg0cE#5YwePJ}W^n&Sb7@cRlxa(!MuOEh$I{Lzxz*MqAlUvAf_
2@jG01B7Sb56hd4;Dls$I3Se#e@Yey3UmW8SK;K?w^O=kL~@-^hvyutzi>CR5Kp)lM}UaD{4rA6?xt}a+1d4a{8`hdOa9d2JeZQil)S||{I;~JkQyiKO~(@i
K;-G*yLGcl*$zI@t}x#sG{)tE1-`Tv4g5{bT<qf*X`b_vGo4HWp1%GsOs8N7;k3{WWq+pQF6hxMO)}rMupA=q*RlO)55AnJX=5lRR(9BnWDEBB8<NkpY7>*M
Xh_=q+08$a^SB|jnrV?F_A~^OBP+)NmCmG?3!a@$+Bw&5b-e9jnyv?#n6(%kw_lxTzBAj$9`{i>18ZPVZtCL=*O)nKiG6K*p)cspkc-d{UR8M}$tTW86vf}D
LmqIo8yl1=t?)@bYM)<@4~h~@N#DdAHjoAq@M4-JL2&=lz0#rUGxBtniysYzyLpvYF-f*k)`A2~sJOf?tf4mFL}b1DtF2FG611y@MEqu(s#*ynZw4w{lP10I
2{`~<XjPjNhhS#XNEz2K#a+Ru$hc-t*z(=gV{N<4r3uAbLB6mgKbVr>CBv{{x*1mS0;*{c+x!3iq9{f{N8sS?2q{M<8ke;h|E?li5x(|>3gLds4)TQ?YGxb?
vsK;)8C0i8UV<DLvBGUW9SVK@eRdRn-NVbh{r?8Ec%-{MJ9Y2G;|-QA3gf^q6V$=4VfUwx4T|6B2NAaJ_Ju63j&id8{el~Lb;GTvEAP@CRc2`UkX$~QhG#}w
@t^_*NS&2X58)#4F=?laiZZ<%Wo?VEPZS07>B$dElY?P({WgTIoyeBRo4kQAvM|O@gCvj(h40u*VOHrmlViy{2?f6G3fudzk2I&VW`=V6%NSFw>ZxaT%3TPo
ARne^5qpVGKEk05F=-vpEDbXzaRmfm4h1$d!2xQy9Xz<+O-ij0!44#iebeWWQ|yd&b%9LyUNo=;-5l7abKZK23mc9sHme>!@M^i=TE+Ll_NLncsrOM+o*kpc
R3UTzsVArTujqKtvU@*dBV+A@Lr04vl%WM*=w0<Kfrx5|kaP2sADG~9ao_E7;7h>`=h?CvgqP;Q<5dZp?QcnIDhgW{Ga~dkSp)tewn1uC+K0&xS|uWHbq%lC
$F`BTcyPajxGgS!`X%i2h4(!v`4Ur#H8`z`TvU{dnheU`$2VRISJB^Jp&P4eqG{Cx?qT<^5z0E5-w23eng2?GHE3$cIMLEFb=9j70Js*2+Ec4A#J#2D@$8t{
rCS25&6N<A5WD^QDmm+hdKO5h#fvqF9_8pClB;<9%qzpiDXo{0sTO1k2HhfEYYcw#Fq@M6?S5N;^r(Z=)K#_9{9cu4-O#M45XZ@^i3(1TIV>=dilv~U-5SFc
&loLM#Ins}O;FPUGraR^RBh2Us68dx-~e%d5*J}5tm~8X5lVnjc8Pj^#GIIq@YY%dPbD3<0NvPKcTBRy2r)-A=LmP?6rs>_odf(50CxRkK*Lm2hO+|b(#jtf
Ho;(2O`LcUKlVM=0#L*561mPUT2HT|v;dIRCf?ypiY<DmE^pQ&kAzvbD+G^zV)O3z#i0nA(JYSCQ6vCNFKq=(pgv;^W}lGe_f4J|Vgp@9pRZ$kXNCG~nrMU4
ug;A5Q}Q}*yWdTZ%9?&jdmKw=Y1$JNaZ_gBkSq|;3Sj_gZy->|wkVdc>$|NlkirnmBO>FPQo?~cSv@F9Rid$mO<oPJ+P!C+H51^^QIi$Ifdn_FTpdtlQ?9u2
!RSJGn62WsfIp%Q0ZfVx^UgFVU}NCmWPdC^j~&`IR<XSaa6XkEUF5b6jG!t`!l$3u27>-t0n;R?QlDuG0A+s9X70C{6=K+Np-)$UH5<?{3TZ$x1^+@zPw##N
VH)luWsZj*JF~~*g|ZO76)CFNKFX^IYO*H3UQ7FHT9#0snjzuBP>a93<?e~TeztDn8Y+%z+HV|>gV|Ci`@tln5H6nsNQWD!FkL(D)r{|;$H%W9r4|{ho$$80
Rc{D2(w^vR`1<eGE0ioy)?uw5{**VbuBGh(Q8i>zTp!J}l03pbSZ8BNLQ=AzfB)k#LpD~*up)?>Bpz2-zU-^g-`aP^!RQ%Hb(FeB&)^_Y5#Xt*q1(%`CQSQQ
>T{93<CrVev(6d?Q#lW03{pOT46Bv1HDqniKmqLR`-rMuyqTIP-ww>_{VKLnZ$YKD)b%RGK-k`%*&D&7T7LeBB&{Hc_a(oe1zf2(Cvl|bzYOA1{)`O~HLpz7
UOc)A!&?PM`v2OBUs8f{6QoGnKdDi~bFCVVXVXub?JAcJytGOl3EN-qcu*YoZM@w~PLtLnq4_O}PMBDr5YmSMbHs9!&w^8eQ4Uhdlv$=bJ^VQKf*I<*7<^T1
xe_8UrJ-@sR57*?Y$+e<#><C6L*by_gH?W&UBu<ea2QwNN3ro;#5mh_q8GY<68+Mey5qw;0tm1Wcm2F0yY@RU>LLNDsYSi!p5bb78X_+`ouOPY+nDzl$??6Q
D?{W~XuWz5S}A`HcDTHgb7SzVN@@U&Iy@@hVc?P<v-WJemQIV{d8rA@Ud!w4D@_w?S}Z^{W-B433*!}sD0)&`xib0C8xzQ&I1qhUa01|2sRyN5IqgaW9Eq^%
%YB%0Ino&xR{P`*i!r%|ZFAgXmnpemW(7$|Gr-Kzlmv#B7p_*b!6e<>#Vx`@C=2m7--aevdfv^7tW8sgXDyDt5SJK-umJ{I38B4Wn;-kXH9+CN(q~IyY__C?
&e&G;Hy!1Kg~WqbarLG=!wZgI2COUo0;9zopzS-?@I+prDz|3Kt(+820EIC(gD_5Sdp8MltW3#i)RA>s&S63@l{@asOTNr~i?dtTVoOUi*!!+GUq-rLxL{{a
c09t#pRlCyNV}5f6WXSrqM&x)@aJfmI_-RXKZ-zc6}Q?X_2s;mW3f5PK-%@qtjE7gHd&||kd2%%YpJ{`i%J4aki{>!()SsIj9^^`$!<z%K&w_;E@%;hq+^XH
RVH<4D*lwDv<+I$z!j-&c^yDbdU?J7UlVvciA<|gJDY0c1CaijP+6F@Fr!DC0CrH($tLkOoJyKqXLCt>pGF;GvEQN{5rpMtdXQDHKiXEUaARZ!drQAMNhoOM
1jaMGP7u5c1ta<$mbP7CBl)k<LGUxm>R*xdy>d}q*Jof-8z-ZesWN0}`*vIhc3G45nxUjI_34ehW(+rcrPjOS!OH15u&JgF$}T`Z5t)ggJ8;S)9QANoF2jzU
H(vCJMNkPH01qPTGo)MrJf<-d*m%HNC1_pi;X6MzFM`;h=gkA*G5-t=jY4vszTB_{;%c+LcPs_-h}NHlC_FQ~qP_M*sLNN9mKVeM>6!nlf$Fv^>ywUE>B!rO
#b|pZ%0bHRqW;?-bjD*jvaf>gKQ2E|)ezcN;pGlmz*6!`@XNyTYcJ$=wW<jWln5&~LmW|&;&@5fyOT-nc5@Cgd36;Y>b$4&zv-%~*{R96F^Qufdg=x#<IHjI
xjoPGR2i}kaJ(to8%ywQ@6DIGbkE38{m4_z=!cP_7Spns!`9qIR5iJHxJ;Qy6P>*Of2fk@$my=ylYPI?GhsV6(iY1VnQMHY>2VE)=v!V4Uwwdc*5AT&x7?{{
dg8q2-AKmVvhh4vFssZ*b>EJYB=H34eT_w2TiXVum#MtFLS){*UQrVZqsz@rRzIND#Ia5m9%ftJUjHq55bY)Li!CWS&0i11fPoIQd{1Krm(H<#WVBVxIJti6
;BlBtDdb*@UD=uFaA5^2w1MAFEqW$vLhnB-N8GDONV0o>0<8vZ9!XIzE50=a8Ul|TQ*s+?j)bxdS}^(8@|#Q6jwb_;b5uhdOts9^;D@wJmza)zeAYmiQQdla
y|Q)M7VdZgV+r<dt7$3NeRH^~)kutZf-+kIR0RIbT_tt8+p;PiG7jR4ecuvec*3W%xNcMqfFWEh=DP~$sF#w%zXDfcQTIq*y6gj-+qqEb&Zup+a*U5rQ)vVo
*tLS+(!S3_jE7tB2=YhM-%nB71@#dTCtpx?e%Z{HyGs;)B;aHf*NjpUPs}4G3lv{H5`S@0Nm+qe<Yy8L%WHg)ohPY;PIrJA5!|+;8zn-#4=AV;m-*Zu5Dau_
ll;Z5a1C$KA2>y~d%WQ}&&g`1^=cf91I~ikjxMmnPvYdO?=^FN12KxcOLc?gZ*Hj8Bf)nHoM9?J6URfIzy_`NV=2jEc3lEw4Yjylm^K2aB3DgqVU?-%O#>qX
BwofhYCJ|O%>)#-(C+Hk0|WRWpz$lw;QUp3{#_&h-c?&kT@O;1T;+)bX_`V}$*jh5ekTEhxeGUGM!Yb;T;fB-<>B`6*G--lZ-$c~V|}1$dFrw}uOmo2#D~gV
lH0m%gizW_CStw>*#$1Hw}Xl)>h*;w)tkugcBtA_tx(rs7<WjCypGU74PY$UV>k66osjEB=a`=XLa_rX3(jt390*||d?Eh;zS07Y);Z0f#PhpvA}|L7jw>tz
RRRwKEqsISXd)F_3b~6KRtZ~hxam!X4}!9bqlQX8Z1-|49srk=1Ui*yZgDb5a*nZ1FWlHgDsIEdjfllaKhq(vy$mZNg#Db3I2HDqD7L_=qn?0`hVD&S9_X`l
)+B#3%xSEVhB&Qx`d4fyH&ru>*a=vgnR+?ImMS1>S3s|~-fH9^ttpJjxP@(Vs{eF&P(96e+v1T9Yk5SqM(<&xAhNrA1hG8mmtIs$3fJd@TvPiIQ1vLRLnj@A
ZwVe7$L3rB^`b8DrLPY;Vsrh5^iu?uado#PC-qvZUnP}aS7StK2UDd1a(d+K)&BgN*@0ZhBiCyFUqBt~0lj~)1enTAH!nP5S@zWtmx=*wgK+d!%2n&fIzQVZ
cHkn`%{u6jch97Fkd#tt6*!25IAT0S$$Wrm=NM1XnYWxeAnGUlrIU6U7onitcEN6`iJ{i~sva@c=&(5M;~00HN#-86D9JSoYHXNhq^lx#O_55f&R&8nIvHb{
qQ&>%C8L=3)Gh*0c#gJ|3FQSn;_lM?d1n?&e~>XYz3hM?jh(@iPhX*hhVWC0N(M;aVfSS<);)lM>$q(fEdKx4|Gt!EmC*%dR@w`P2ij%?zjmx_P+U6A*M<ge
qv|Uk^3`f4EN760k_a>A0@o0I<J(B1F+HXi5)HGu3d(Wd<z{zivXv8M?SjR`WBVUGx<*BbW0R@}NJs0?c06ItDD=S8ge)BQvNl9JKpP^d2@I;jCnGi@twndz
?ULGwS_pQs07t4R()p>8-*C?4o3X<g8IbnoY%Pv}*<8oKfa1oprbX@mp;%8vVl9qali+*ySGV`c)d#6?YQzoI*D&zbV|heB{%{%kpAw{Y6gFUwkSR`=-O%bA
bX#sdApzr6wCyrpv*LwP0Km`IOUDqv*HqNth^3n~j0W4Cc}vTR<;-zV%(*C~w-Ix_qO|#c9j_+PTmYBMzw5NUC2R4)sboX6RyeCY>_p)j8_J!Vnw)Z$<vTYz
aDs|h2)=(PKYn6db<s_56(7$<3xl9LRR>pm$kcy~Fj+XgS8nZz-mRvYSM})cbA@y^?9F$rk9{`C{4lE&JM}l}O1ryuzcJ~d<RH&kv0w~xag5o`IyFiakhF-2
pPy0gCV{-YK(oplMlcc4NRX%F*Lb!eF`&Nq;}CNwnu?lN`2*|d5U9TX^1IaTPh#-34O1=a8L3cc+Cy-ojC0y)vh}SeUEylwr0u*Q^Q{Dhpv*ijOX1*WrC-3|
2?lHA-Mk{!ej>GMvK|sgDq3lOqY})$^k|?n-{XmCw=zK8SyWlN(q}v)JQK`vK~ok-Vz|B$l@x?C!s0n>WJZtNxP<TExkTrHjB#XWqj%G6yBNr6!LGc-L6Uq2
;mDetbERVAb@*)mRe=D!Csj5HjSs{?B~!SepvSaxyt0BuP-aZeCWu#*hOuSQ;@;-~+2>LL`WELOL$eCDqM~zt26jd8j<jmwi*<S`pbeK}%i?NnZOV3;e{C?&
(Aqk7lwS^ERA@M6ElsBbqriOEg#y{LIMUGGBzAWRUAt-Z23jIJhqMCCTQtL-Ld4>q7U}97Q6l&ShWG`uN!W6n5yRkW!b1=N%y6Hfh>i60lsS1n^3UHMJ1|Lg
AG>5!ZJGWNIvZ8;`9dMxy3JNPPG`IMZvcH{M{M3Da7fpz-C2unZgpb|b;B$dN5?yJZTSyn7{%m60Zz|pIB~}<XJ52nX!&jh39CD>sw%D^>FwF>Q5<vWa3-;u
hb%R-!$kqqM9w0F5l`%myF&ioGF4((O40(K(DFBwj-D}9@Od}oP~RgsmN(L**;do*_4>-qEziG-_4Ol?2PYQTFHjNdHdbHjBNZlwL8Z)Y9LfVo*-1gzEJ_>|
qNYy@hx6QWUXE6II>&t8=>}y@{zZDNG)sqnWDM~cv4}+R6K$2n@=<P$d0{LCR};>GXsOEI(8+Hah_MuaUSGb&`1;XO+^L6!t`N6p1v!NVExWE_qP6m_kq>ls
zL;%w{zvZTDiS{q7}CyyS<I#Zz`vr=Z-H>PlS65W{!M-JMBfjd%9_<TKXi{<%FCL$U6Dq`$4tT8^G^RT!mlYU6~{-pf|_wdq`Dgf@+&fF=3)0-YsRTBD2bzZ
MLU}uoy4#=)u^`JTUJzC4dxxW{&JEKI$~U_Zk}5I7G=PlxMTZmSF5{}wm+DSRW<+kko&rD|5KtR_Oo}$%9Z@M%`fspX?MZo526E@Vk0Lq4~MpPQ4L1RKtCZw
w7}y}?t+5YsjCNb;&>fJer4GvZOo6&wUznmq&$K~Ngj5wLGU9(E;ZyQB#<LzFQeNC7(T0|1-BHH*>5hcOXII)l0QAMUaLusA|aX;sTo9Tn9Do!5TX_~6t9$m
iODC4*V5Yt`pkeXNF#GcoG@Ph)^Ds^p}b)bcv=r0NIHOTID|%<$I*pZsolFbHP5vZKL_ledp~vf$5g^|yh!1mV64IpUpV;i15y>f(A8y$q%!EOFr>K<siX3W
7I|0q7h{qIt8=rgNM;^aUH5u<$vZzjsqu$*8>4uXx>|s=_dm4F^4AqIQuUeN6P|?G_XH_<CksTgA6RoNq9$tnl31TGgza4{Fet-UOL58RdTA<3LRu|tD=NLg
r$LooF^%XTyD__<=mnxjXlvN6mWj2RFu%McXcOmmCbGrnoV2%zs5(?OPcH^%AJPFv%o`78!eA3_mx~nL<qr(HPBSf^{4gDeyx}OA@ZMndppK55a|)CG>O43m
yDulBLvPRlQ#y%=D0Esigy~BwL!6XhM~DF<wv{&QU+vr0J)eKNK{U1!Q~DVJ0Or02h_G{lh>JXG+UwScZh5e>;*{J*a1%!2;%PRlap01F@~r%rUSraS@H;}m
y7G#~IPBo~POIN@hVom$)YxKnMn&OsBP^v*3D?mQK>d3JdSQAU8y>KV!BjA$rg~sZr=I>g2x}FcEzJ`I`pc}Do+O>3E}2Q!G^T_fmMIMySb+ty>KwhM*!P<{
BTv#h36`FLXC*dZa!5Q>xS$`>*Pht@&4LmQqQnauabJ_pIxvs1!G!tMH3V_ocEDK4y>3T}g#Z5);-FWQ&{}h=w0uN^xqo&hf=u`+)ruadOMYXCy@11wYm?`S
)TmP_#NEr~jOlK_y=gg8siOPhD&=#z@L}P@mgBd2+`I0^Z1f*S{TX9o#l%~Jc6ar8p+sC^kCU>vbA1~++u{_vl(lHy4Or~3%MRHKQvUBMhszGDH1aElw~@2h
RbP3(YH$!a8rZinyzJoW;dD&$LeE|V&b8>gPf9^*0`cWW(IcSiH&1_wt1%yNBY6_Qakm(u7M<-PzKO>m=}wj%*HJWmTucKeL>7DPfL70%6LEqBvQ7KKX#3y{
g<RdzXe)_y5u0a3WW9g$oLUajmuu_?=_2|fKSjCoCK!t}70Vc29#NuGRFpE8`eY?|or6xxlBgg3dqG0y56@TAf_PgxEcShjoXP*F+Udp%T79+WtZsHAVZ3Ek
osC~z#o!GY0VWwu14lN2_&5gDRv4ziP{kzxWWyXw(2$1LG7wJ|z&IR+@4l}I(K<zw4-Bkhk6K5ealQU2J|YBKzI>W#)E84EXx@O$mp$J28kRsD{Cq|(0_)uL
LL@u>5v1gbUA3A9MT2$XxK<g9Ku;N;B1RUG-3jSD`yNrj*IthtwssmP>UU_5ytU4QgM_iBjL;wkZEIa27X`DRRwTvsZ9D)AU=Y7xG&{j6k;sOvGP`w9b=1eV
?mKQ17^tmZ72D&G4pzHA<p~CSe!5!z1W!LZqvbaJZx{IP`2b0tVdBQ)2C~|ySu5v4q0Lx)zq94TP`nDP&Z|bPX9<qQ)eSIXC1b9WpjkKd{S7$5=}($s+inm8
v6mVu9?)tl2|umJnud)Ia2W3_l3y;+9%3fvG5ulZ`BfXaf?c2Ocsxu>DN;|MFg6Ki(?hiip9IIhv49DOL!8*qDQZmiD+n^|Ko38x&N&aXb!V(ye?{6Y0zp<Q
!+&sbBqjVe#(?LMiD?T=wX_{yrB^7gJgYR7-o{6^SUw<Ea#9q<ntGc9f;n}kuooLA@v2f=gwKsw9LXd&=1N|ud9zXeR^NaSqOA1fF(7vKGP+q@2})uC`@tXt
?@4`lefmANO60Z*Ey$0wZSq0P;<rNfHb(12j9bqpH5uvC%Wj$WOE`bU4xbEjW1VYmoIGd%Zd#N(cR?D4g3F#_x2lgsKjEkv%ks#-Tow-7WPp52oo_BO>o495
H{eaq+aaxD2R^}FoNCa9^#)B1ZKHE1x~u|eQ48UkW_RT*LH|>{V!*O-6-v@_*=yF82*BDYRtgU3_2-S$@dr#|MxG1uclP-ym+1X7zLOe$S^LZD5C1Vu(@)8m
EG>0d;Gbi|O6FtZ?lLLiI$yCV;ZK7;p&aCVB3R;@*4FdBwv?V?DHtFblY?n=&iS|qchB*5&m=}dTmM1hvm~6Wv}&h}1KjSjr1&^F$2k^kY-pzXSRYn4oxfp7
1G;>xwFi%Eu(Hw5Cq6L|R(4`pSu}lyH_Af2*aZ{BHcP4K7#DL~IL)G1c|+U%`u;Ycv9FMWoZd1kND@MW5FATlLSlB3x^A6jk-Ik}h8OaET(pV{LaUfv#YzNo
z~2;@hf>+ZTH9Y72PpckA}EkS1IGZxCbla@a6&GSyC&_@jJMAonYcv&^+wTP8C=xd9gZP9)c#V3;S!y`KBRxAB8+e^0505!o$;2|i+u1Y?KRfWU7ey20-3a}
4_{pDUA-+M4>Xg1NXKaY6a@OGdli{SORXXb4Vr+xa7){0HZ>G1(D03eYJJx%FYQ=hw)eQTSTgr6#<t%%TQC*q#&+HwI{Dte=3W&hz)<R6=QTPen1y+kX$ggv
hmucfE~D-u3fG)v4MuH8^&1LG=&X$FzWPEG>4IqVpyRxpQx(AbcJN(<pGq!(vXC?XSr;{Il_{8Cs|uCs!bp-H)1#_YZg#V$W!o~?K(yb097u|{`_2v9TlWvR
eX*c0NPka&YxPh&sO0u<M@Mj0J|Y}CL!WDXJ(nTZ$Fg*A6y0wmbb<=sk}2Ur%1*urK*9Niadhf3A*iH^xcj~O1{m1boVS%i-Gzq0WELeaJmR9=fHebA7)#!v
Jm+UZC})qa5CHaz_W@Y9ccs?CA`pwEUfbI_bw}>wH5Tmk%L1q`y+w{jp>(VsKw2Y)`mSp_-jk7jb_b}nQ2+W;^tax!OZ(($+sD~LO+wl6UB21fA>~@H_zXg!
vtTOJEue>b`Hv24sjoqyav9r#JHqW;|DQh4{0gzwL{v5>#i~)bEWG)7clPn8Gbb0&c=1~%hMj;o^a;*XK*WWST2ym-SQx4r6V!9M#{`{$eM?rG8(r4>r=%Q8
W_k3LCW%k`a6>Bh|4g2q<mGwWA&p$u3E>~axLI5OU@b(BuhKI$C=0Q{^#Gda%2M3%jIv{bUXu{PAN4(a@s^|>SCAJvu|hh2TFXX1N-g54Qoau$*a)dcdSoS{
QMGFc|A4CKEfmU$w;B?LyC!w}KCd<rb^N;yo@wi_q_(k7$C``wwsdAFj%Ya>fgUSiY_GK9MG}K<R2vmI&W-T$ynF}>r9!_rS5&}ZQl$a6k^(syojD}}XhnJy
BP0<dULOe&2(a;2NrtQqs82B?gh6OnN)FU^daS+i$TZVpK{P5BJ;HrYM=5m}(mziqRgxk?fxV7NV&mSpGm*;Y=I69`sHbUFj#N}NM2pu=KX<@4RFP0VvA`ec
NWa>fp;(OjCQL99(HS@y(IxMCFZmpWiE)ZaA<3?MM|gC{2c#cF16|@)S65njdFu6)G?BnidrB6aC15=TXY80iw{!3vaE~S3oNWbMlfQerGzVS1<N5*Rd=$@!
pl~2moyE6n8u?dY>qM~Z$f2ThUd~(V*sR1IRFG=hiSY&3Um0iF(It2!6j<%<N?&F&Sd=ELSHS0;*o9N`I~k6}T|!RpVNVOVbA*SQMQw6ELj$pPzQR7S_Sonf
M{8DuFLgoYm;wBKGt*min7q*!ETO9(-gPKk`c11`D<-S5B0j;py6y#hfqszk9VzaFnC>cIE1K-DkGL(CsdjTRhgD$Y5dHaGt9z{cc3oEHaK`L`K_*KLbD`cv
0|S>Sq9O&NN0_5v=bL$Q^#@KFcPckJiN9GjBm6F7N<pp)|2Q+>@e3JiR{~JUZV?4&0Ts??OmxfTc7O`Pn{7l#=u(pLF6^YXFf$tn@M`bFcUJoNp%2~9E;6|7
q-x?lHo}O}2?MH@nNhKaPMz1#vzjZVKhydafq5uYxK9-xFo>2v_)jExNL<I+u2)nKIsg8>*$hIi2oE?=K7}RtBE;Z9KRRmVmudnN>V?Nv2zY-zbL1Ej_*5k?
lTI2(BxbH@#fkj0NYLc;>%*C<nILHv@Sfh8m7c6>oKT?$+^F786qi@(_ezMk=f<>&zzcScdJoWLJ`zM5KJcCw?v*h9-qT_2T<>TssF-2U6OX-LD8O)z%{*<P
bae_C5Mk0+PahQ3F>>i^jH7y91+=T=mJ>HxaBU8)P@>#0>^bmfyI*~ty>hL(x+V9M$mB>>D+Jvap3eYk4eX%c<r?t4!K8U%Hp?r%-1V=Yh|F*f!Gk&PLYnNZ
A8eph-RCht0_#)O!}znoG+-YJ+uJq!HR!hAf&)IUi${N@J8pz4QW^Oj?w<F&qI})z6<N7C`PEz<lug>oJfH7if&U6GwdE>mag*`egzoW6^KTE1T5!HZ%ShYD
)kQDO-X8&MEZE@QodY9!>g?d_>i&IET>6@fywKG8B!cV4qSja-QFA6RZ3P^R&c$3NB0aLCa4nQUouVK;Hf^?Pqt<(E`9u7G8FkVMNRIRCtLOG{4}8f;{APe*
4gyYHxUKO}M>(C+SVG7bZlN6k%VGmP=DHPeJh9^%*Ipl=?&GB3FtWj08?WnNEAl@~`F2~YmEJXuE#^Gk9A~MvaIHL=EYO|}H36X8g9{j#7b|FECy0}6UR)+8
9;8>94gG(9q?op?|8PME;6rT%zQD1z->oeIIrJlkexRxgUT|R-w)qz3gEuaF`7D3Hp4V)OptMcQE0GA&GyA%re$1fYs5muh0HwZXK7R2xP-_^y$O!iIG0et@
h>KUcXcf&yo||7SKTWbDHNbU=d1pSRTKYlH(Duxd<#1M4D)90T1sF8RoDwICIgtMnn<q(2nd0oTMXsn~yT}||3^0|-p5nOPKZNl8^xJ0mdZ`s2c=lJ)wHVe?
Nk4MQ(8tFMN?7f<eR>tyAK8R$;G=3`dKB$t%A%g|O)lNWZ>(z5N6?}OD+E5r0u>6>v^`L(9P8q0K9Hs{S<zH98M2lwa`Rwh@YusSP=P}+gyVyGT;FmTj4Wwx
q!J}ieVXm_^<wrblX?ok?LKvRJ1v>-;A&kVBBh>6q1d&c#?8Ac-+6(z+QW)Yh5aL4Jf8;dAT~?N(C4`pRrz`-@n1c#?di)BbGs!c!|^b}&JY-jfQ!yK_r3fE
-0~!;Wv3vWlRvtM`>s{p!-XeR&VG~~2ijhgLi8)1-f3x=kL9Q_W&c*Ords=w^NItqTJ{#DpjJ%8+}0IU>&H#YRNDy4qm52o)BP`WTwm&UO7>7$3vILt75PIy
TkQogPw#6=8NZvJY*oT0tYT)O=3tYC%fD5Z+5S#oC5Q4!F8Y;vh4Da%?Im|c)9KI~#%g(nOq6W>HvoiF{A;;+*X1f(_9^65rh%ti7b+^??Sum{DzScbXpu7l
2nT}G8*+q+*ka^5<%oYCUll&J2JfTOX$gQ!9eJ&X?Cha87f{LW{^je<$U-1g3LFtEcSopqkX1+2*+lb1(L%YMfxme5ze^;0$OQbm(^U}@?a(t`n}3~|CC${h
xhloX1#oxeVTej&sohP+rzUnYnOyY_C4G|pl_M^NjY3rMD+6Dq>Q)ZcrotKB3gI47X~E=H-ZlAnMwHC<I&w4$GYsO~<dT<cG6$d&b9DFGd(Lw}D1WK!KJGsK
_7$!+3wp|5BGHwt^pOOj6mq6CDOQ>K7gGmoJ!39C1I;phPutbmp&%vn48LgTK_@X?SR;02d)d=Tzu&)L;U4~iCUkg$bSDg3GUJm|*rW<#nggXSna^Gg|M#XA
U9<^ShNJKtYhyRA-pDEZ;ru}g%BI|g2q1Lizf1cp0hNH$3-H2-om1|}a6IrVtapv<55KXn2wmFV*mmW_XM~?j=@#12$x(bxExo%EKBXfHfs1i?kBSok<t8Ou
zbr}-@KIE(JOG%Dgn-DD>V(7J(m%&#0cE%@cU}Q&{yi-$WRwm31<Mo~sAJ;P(i}pg{oN_Qq5j5`0)@eQI#FzF_104<V3#`i?`Q$Nr+o=)-v96DdR7WFACE@R
g#=qvO;ZaIgW$$9E?VTJcrBNoK#O-<@lluev3C$&?|LU=cNB?uT3cV?fXtnDoQ$hFp<4=WWxhm?*rKBmzdO3~AV$^TKtf-t!`FP(&17h4t2@(xh)oJS0~)l^
N5~gzOW&yJUcK%O#rmh^wZIFzfa2Qgkb!KUcyEkeDLi82LmM8x==YVJ=c%w&_21l{ko*T80y<$&lL=Y0A}gzat+}9)aNw1^S&q8M<^5R_?Vrzfke*0l#a{@3
GwnO6ziyubaRM<oBWT7<Ik^`6^0<dV0i&6O)0m!@gIKB@TKV;Kl%(Lw(pv;0!R(gG%=klZ%BV{Ez-E^MhAPOqarrFn(3~7JJe@desV8{OKLJSekQv74OAZxw
bA7~9?}*#NmIDlCI6*F);E9*Uh|X6_5Q>SbywZDmV7>oTy{6-_7qG{kzDt)RaT?)LqM&ortpuV%C6UGoqs<=6lyZ}u)LHPx969FaL6Vh~%HBS(!zTyDFQJtM
59J=!1c=LA?3v(OKfIe$U+6nSuK(Hh>*us6GrN8<L<0F0M<86DOpL-m&$@5y8XnpUqXRP~y4hsrgeo)LIYVy&RZ$O01o9B62j{8EmF8gmxK<cnN~*i5Q&}m1
c>>AC$)_N&WP9}Y@QEL+BkKTsa5J*gYq!#U-6~t+$2>0E2r3c3Y^c>aKYlpWA+&yBSy8&WfDfZZ8%v>86iub9v;#x73LP<a$H7FyJxM+_WR*m;&F!_HJYjfc
I7>drg{EGfIrGZiV~3vA1vDASD^0CQu|3XR$O-|KZ^NTubZP#GDb%WiZ@D{R+$#Td81@R#+Rs&Z;@n3JO^jE5(4)yx@zZM%)m}$8H(Ns)jSDeLG`%DK;maa?
SBf(L%cF$ymO!&<9KyyKJufi$PTchftz($B;N6n}L)Mbw*(xICU%9GU|BXiheO|8XSYg4FU)7qH3QyFL3s6z$+qp&Wd;uh-icB&LU6ennB#qv9v{E%Nr8p3}
>i=044u$JXEm>@U2Dod@{Yp62X$b~|tXsgrOBMhI;Y&*+>CAnOhNslh=mcWCxIJFR?u;ZBZIxnAG#Nf{r~O<cC8d#?-TckVJa${bVsUie*=PM){zq(mcL-_9
1K>OSZByTC9rRp^D@~bIie`h#f_9f6vVkHlVcM_RyGYMX#0w<Up}eC^ye1kvED}{YhjcF$J=a9#u>4Bm@(cec&<L)6+a?=14#~TvQ{ZzZJ(8)G0acTMz>&JF
kH<!t`yna~wL6~l7;?9!J`CRYcsIcczHVXX=;Lp%GCvvs%X29-g0(%vy7dGv4f)=GxW^&oRX8vhzW(*|fP<X(Em&+^0FCBpQ9bcr+|LTt?`MYzU_gBe?vScg
u|mQQWq7tV8q5v4^Z2ZT<c*w)l;$X^5+MwEZYi0Qt(Zr76t0HvvI!^ZVHvT~lUm)gJpAwjO7c@^g*hJaSWlr|8H(`{E@^HESSRWfK$`dn*CS|7lmoqIja}pK
3b$5&QCwU7($2_sn9b{a`baqr<tlh8<;|<}Fp$bI6qu!|I_aoH$^}Fd%KaNI?ge=Xgk!5h8hSIY((bGK$MM5JYS!F%Khk9-bPMVkIS?Jhb@hZP9bvEROhlS5
u0&?)8BO~O`bVg;toRn32+ii(K8Va)8z}zN29&?V2ow*Uc46Uo8K<ggJ4=9@ZxAvPa|0t=e-&>36Vz!0N3XSC%7g7BE#1_P<kKjYfkbC}_(a`@7C}Iw$6)nQ
X7~go0pjv3b%<BQPG7}5bM0gk$B#W54o*BX*|U<H+J@)4enayzXIIjt(o<mBjFhQrWV}h5IWm7$Um1%XfWQY#B4?w>^aoK`{WV4K=zmpQ``|!k%68^dzLI2w
9Kdub4s{5o@eqCF8qTBd5YUWqOUjz6uh8%aB4$*7x?<J>mcj>^`(dQ+nP^+O(Q>^oy0P8XAHA`~dqdS{nG0*NKpZoIpf^ZsINVj9*DICBn1&y^nV;I7YCyyq
1x;QeBX}{19-(W4N76!yj&Doyw<%@3l5C&VE&LA$h5X#vtO`Z1TL>fgW!432ikft^x(hF5&M(whxp^Ib1OjYj@IzDpL9xzRssW`BW%qa<SYmS7b1NJ6F%l^r
GqqM@9r|a6*Iu<ea@XOvnbbH}+nGB1uPnXACj~Sx%7v)<M1sK^2G|5vuK0ay+ye-S2IEabn7di~?(ZWB-(m2)yS8%uflw_Ttier99DP*(5}d{au%%mVC%^1#
l@nBZN>;Zq_oxAQ>gbfr^9pjKy7U4(cW5<|wmJreYlgOK9<a>d;IWSASkG!j5@DgFq9V%n%G)C8GR15Y{L$}mj}VZFOHHuRsJdS{_nVzxI~IxM{9h9zE<eBE
#^=<Mo)9It?C{Y-B5=#r7a&$G^#k43m!6OPT`a*omA(4O?&qN!?|pG`+sO)mVd@)^%8LRUt)3H$4%5=TK!6KdG^Xrre2E}Q=Dw`6<NM_X{;jZo6&-{%I!XWW
^O}Wb?h)+%j7hz&bCse$mvAvW2KNqAFGs0C!s^YrW{#@lDa+!lQ##%;<3R7=)fk(<#!rjc{3Pzu6{$fbkRMjgp0Ch@0g-RBB`}%J=1SE(^tT8OL#C0wcbECh
CJ1Da=pE!|2noNjQd+H@18W2*fp}nX0EEp#Cins1Vz?=+WY>Z3(P&4vzJ@DMhqAnS=nSMnOy|Dq@d=<a)?%<6{mG+qGl$8xjAfr&FDjZyXyDddR&}9R9s&mR
sA#_*cWE=^jH-4MpuApR)M6jiG?Cj&NrSCBAm|joNZJ#|oZ(NXs(~+`-lHfWFC>2|-$g~n0_i>d`?`!N%S)#=^S{(|!#}_M&8-}FE7Q6{2fh%l*=;HijI_vc
J`2vd{dqYb68OA@Z?dPhJYO(FBs?&No66ZTjR(p&qj6v?t~|8#Ym3xs$2@J&GxUCrrJ@<(78zriqQY8Ay?U6!1Qr6Y{@vh>rn?eFq%NwVpMqu!^Q^JE0w$Tf
QkqCP{v`7$$4?lB&IOl{)Xs(_)K1kH)lN(U+FkHSu`<DS<!*>K)_mi$b#fn?%fL$<F*Qy8@hcIGl{(EvP;LjHMi}Ho#fNh{ThM*I+&*}s5$>*WefW8eypk2Q
Nx61#PC%d%EXsrhR-c!F2r*1j?w55IVX*V$#jymH@@M#qz!OCA<mjQm<Bjaat{*&$OFIyHhT?KbC%OT9X`_CHH^(RquP_ow%$qc%2lzJdoA1pa_^}>@waPZO
!oh74ibQmL_2Sg1Iudf*Ak6d<Mo`O)VM0IscB0>r8LVClZ_3z2XSlR9i6)_Om2G_y*}X1A_HM!P2t_c`Oa&?#Xi{#@+`;>65N1)xs4v#r<ym&+F$Ad2mTH{@
(zoV^L>a+JcAQEQ*eA~CRU|n~^YVPuFA2;0w0I1C|CCfA3pfB@Sg2)v9Z0(-ql6Atkj+d-H0+)aApTvG8b6_1MNLIxk8bGO0X@7f^<f|YL<%ZBCEc?JTzT0{
HxeL3KfnEa<|1&r(t8uY32wNG7l!7>)(W#OmI@>ugE}gHE#HPS0i|*_7~M*uzaO2DMJ5|X>_@bs%li0{e;6pBcT>A*(#MwPv+)wBe`mcx26UM=o~AQe_e|}<
vf%T0SAu_4<fS>^I*6Vt=eq*uu2T*eg1wPtp_a$DW|mX*ZjT25I|55eP|A=nEaFb$pUGuBban@y#C6Z1v)5Et?=|9X+G*v>4(D@4^KRCI#mQ3i2sZ?;$pIkH
8u6YwoaNkCLzc>0Iq$s*`xIxEB?|qH@Oe{;3YvXyQsMR&Xokq?%Ix9|0>PP+9LM-0n)>V%>6O2=)h=o^^fWjN1OSq^Q?qxrrm&m|nJ#$(QWK?ID=3>L?F0bP
7btpci4iNKbpQ~{Yv-3j2vtbjd>r3(KBJWvO}p^u+YZli@5dt#IX4sEcmo8AW1}T9n<61Ir@X#qyo!xHbSohm>OBtD?!SiApAY#XSz8VqARaC~{TTrlfJAVe
c&Bll26$ViImu~KGz8{`f1<IW1Ks2hxN4KAPLpVl;G{OJdkt_PP2;MLodeXP2B%_@T92$cR07gh-j*-~j;zD!9F<>w`M||n`>&kg0fi&f#(W^R?YgYqj>bc{
QlPW3k274qz)ZlpWf#?cz}CdFbj4`2K`I5cht=y7(!Lw(&6P+90St2jII=n04Em}un&doW>3WV}g`aLloPnXJu*a*O%P3K!gB9vr#Vuv=jBBa?PscDlRE1cQ
Wgqc5JP&kGQ2@`pr|z-lt<3~=fH-g?&E*x{vwa_7&XZaWo_4HZdz1!nimKUUs9&U#NcelNL>6!bq!{14xMV6RD7m5(gd=a|Tz(3&MttMffauxg6zIVizC+M8
pEJjltrTeH2a2b>mzPQ?3$p~=t%|e}#*&!ANh#Ws>ilEBW`%V`{=Z5!G9Ee~<Edl)>k9xNVU_EQ=p~D-hfTuh#vcU&4Pgs58(6u3gaNp40@{hLg0XM84|X-{
M|Lh@L2V<OTLM*=dOZpt@Jey0l<2#-PoW_)LK{cjHKC)tJI!8Mr>Twx#K7rKrl@Rox`(ONz39^lH`C*02&bR(pzQ`t{ii_-j#<sy&D#g%!G-V)-*iG&_k?0O
U~C$Xp?w3W*q~X=No9<+qAh%nK7D$EOxQBS@XV60BJUh~%C2}^WTYWLpB1*+v}GX3Uq(QAvW^*lbIMl`*_5G7+Q=rY)ziq!|GAgSH`Y3(PE<c1hWC4TuoLnQ
J6PDcR2clq@2j(z$m_Mvp68O7Gt4+i%elh-Pd{sc)U*Oz354Q%OAL$a0iZR$s@YmY)*5VZCBhw!BLoSj)DFflm9r8=eAw?Iz<b!dUt)9i+FM#t^OIr&Jp_7F
X4Pce7e{g@D)aY*82C1l!T-XvT;sIz{-Ed(nO$J4`~AVKN-$Jp{1t_4(m*DOw6j>6ThXKr0vp@8Wy5V|r=Ja~E;!05=1zrd7vWWu5&yyOOGJUXjRalm+|2rd
Q^Dvf^VlVk(DKt=ep?~*va`s8ce|>BKuGPuE8fEr*`mmnGVQZ<YGP0v7^3s>7pc9xG;)x<>Ae(hU7TfQZHi&e2~u9AQ}!Xd#T{ep*~yhBWHC!#u0XEKYyIRU
AAuYg<g)e<@;1~*kV(=B<wM!i<cy;N;Od)DZ`OxMj40qU@Fwq3s)ie|vv=4!)zbr4bKlW5le~D$s|_3PQI`3LjJqnI7j(Z9j<|9TlA2%<kX7e0#{Aft-R=u;
EZ7@l9L~HnQcUs*r3_jv1mkP<npH!D3DHjz(q%l5hk&dVRE98BS9{6F<%W8ZpgK^2Bz$K%(7Lm?GxL{5uveFjCE}U!AC|Iu@n-4*e#$i)qSmX_K=c?G_f@Ub
o0m-F;D(X;(D~o>$-*$O%Up~pvt0|HkF2$G;b8oh&0%Y|)J}YFJzBQw;(oBw*;?g=roT5&1LhIhwufvVj8azSwGfL^#?GNHy0`9L|913H+9#Dfn=y@eN4U-I
XyzNXUHIc1`fB@V9;m~(VHwI~syMC;!Df*({$k{Ve13xAA?GF(h&}Eec$<3N&{C{sJXCW!lq?_Gl6QMOX&J*Kqa<-4eY>N8r7zL6_JBoSlDQphTNQ<)&#~wQ
!S-8QU+qC;q^p-TVzgw!e<zni>S{!=iuuT41YG%e;WVd3nrVzwNAElz;P>A}$hLrqu=9>;YKX)B9F*B7mtFo4nVFodz8_$<!|N_RtQ32dF7*q$Z>!zK;F4~L
^icxf<8Dk@_Oe$X-2o-u_R#}zErF(_Xe`|;)kpy`q(3qYx+z&Ew|uxq9M!wN^IZNFm3j%qfl}xW<R90<40^$Se4FZ^0jbcB1#}&w+<?*bVt*NNr!SY5<#=pH
Y)4+4Zpv&XgUA^#Io-BOm!-Q1B#-E`O)VRo*zo~$i<|7xpICo;<K5<T0~Db$a|;5=9@cBz^z`hSMoi&)w$kgFpyMAMn-Z94XA}>}aHrFnc+tdH!m3}a;E_q`
C^Z_TbMn-ba5}s>Y|Ii6DXWet4IbaeP{&y)*7bEckkJOB?*_`k*q7YkJQtCL4n)Xo$q1j(PE6MdwlL^B8=1cRar9$}jwf(>D(;uVM_t}8$t0T?5uTPZ!D-6p
f20kRVj}I|22n|<uB;%BLp<>zwkry`^LC1WrFh2Hp=G`qMTh%x{cdkTQw}`V7U~_SBs>-CLR&hZ5GSjGi16mdKHUdxmFCR-;*xMix<D$2L`$z7GYU%59em)!
>lsy3M5HS>**QeChVV|SgyUyNiP|h|E4;BJ#jG85*pd^ObPQnT%WOb$u%u5H3Yun>;|s;Zc!z(NYxa4S)^_8Iym~b+-y(&`V*@wLIs(OidO3xs>OYovH)Z|N
)wf{Wg4dmrWs!DU7!t}ws=oZk=?|B_3cb$I=(s+C6Y_|>cnSWobI3$tpZ+r*3rorg(+SkfQVm{St1Bj)R-``Da%>f|d_2+Ldx;$IjI+@&i{0z4mZ=tZ%AhE&
8=U$!R4`xPDI;?)I^5@5e14TD9d#-r4Bh6zyUacU|B1gAy(27!Dg7@6%_Z%XQ)?KOX(O3sDLHwtpzYYAeeE6nSYs7**)mL9e2gSFxT`xwgl+jap7`&7NtiG1
^>2H(8U3I3s(&h+P-FY4F!f&(kp_1!`UquqBg=ZNPHoHt+oo|F3BLMVw(2)`wtqxddoJFx20dG7)9E;HkrCGu<7Et{uYcu1>K8*p;;OEZA-%2jYX+E2IlBaE
M3lHgaz$rnaOH@8M7^BEu%TO-L^(CXIkGaA+a8nq>{1Rzc$3+;paR$jUv<Y}@Is{LRNlRZ$l^)eq=BAf4A0@!2Fo`^0t(y!{+{F$r7)-<oXeq3y6<%|8%RX2
_<Kq&VukOf9Wjruy~cN1uwB+wrI!Mv(ERDNhRJ)qldScdb*;mUaN4Xrf%r1hk4<9BwXhikeeQN|%Qr?P#T#1A?)p_<JH$#Kwzx&G>v}o;f78ga6*!pQR4nPK
L8J)Hb;cJ<cHzFWGF+}ElJSP(PP}+JS>S0T`-V^xIaixg>(1jon^pW1Iy;ai;r&7aKQ2WhClO+MNJ#$s^a$U;AdB=()jz^_JdIdmkHz9*o5je#+csZ)T{G5`
q$BQfV6#xh;v=y3mGvXt`0jHR9xHQV`3asRLu~#j=@9bN`YLwJx%keMlQ-`*CIAQ#O_H1nPd6$aLVP|6!DlSX$ss(4y~1^<C`rpFOESxim2>07_Ctn?hiSsk
2SejaT6JBVAm7;J+|$4L$}oFdN_q$%|7h~^%x?tK!YX1BeX@69S3R`C^ou;mm$DY1E4AF6tLycML4G)&UIreSppmmaC(MpDdQq@zb*sC26Mli@Qx9eEJ#x6)
`!j4KfhlI7D|Yf&Oa_ws1vB}0eoQ&rt7eribFGP~y)xa;d+F7D>M7P=!yxZxafN*jOIP2y_;e^S(4+}wimQdi)g``l&&~8BiLVv4`-P2YY5F~Vfj}hTmXNJP
Q;k$jV!{z&7_<M5U)HL*n*TgcBh~V&#vgTx>T030n;7gc=m~nA|1FNcg64YCsu0B*srIEn@r^kb-PpWq_L4Sd#|s1Qk-{Gb7Uay<e=_TH4FY1wk+bj=e^6-}
yKY>`c`k~bzH+HIPV?3k(`ul?BWn>2<M{YZm*+tR?$C~GBRw$-y`FBYpbAOUGtl3BDoC+Vs6yV-4tefony+JcaBo-?AF{jmsmfKN72?URBAMur`}Z1y&SFqw
OxdjJxGL<p1o8NL2~o+9j*5X&bvV}Wj^}LijQDU}b#@n+I(3#a>Oh3SJ(ufCBebh|kIjLv%Pior@)fmrdWAd->O~r)c=-#@gQRF-X4I^hxALbl!B5ONJ+bEL
zG3Gx*!DQIJYMq1#cft&hSsU4NfBDbI2`duQ@?GQ<Diy|z-m+X<#}UjgOtJXX(aYT{VS+;z68S*C4xRHsB_K%@K+jZ1`U;%F0Kvs<>K;deyJImuw;=YwtU#3
8G#;WG6LxW2MrDFzd2U*(Vke!#av%_8}f^f=@ys{^XBt~YXg-}Bhcmg7qXiIgP)F2KZP5~L0UI6TorHpvbBMY5?K;v9Je$kJ3I`MD<wT)kE^7Qaw3&o6-%p*
qh8es<5Kje019|+1M54e0`vc##`U7!?purjI#bsec?5BOsa2nM7GZKH?Cr{_xZ>=!aL{ti^ABtLOD6g=*l*%h4k-{CUUrO*gzeXxWkOp`<-u5;F>SJi>-(o&
BG$wGa_p(lx=6*auaT|^@SRi0;9?DU?3zwwZn2bM7aLwtkmc9P2K{_0zNj*ch;3w7AhA<0iG1`T05zb-!vLc4j1jM_@(XiGgh4$v4ak9pw^4WEP^OL$P7D^#
T0CcKP9X@SaPg{vv$}m*o+X!zIuc+L8Ue&gylNk5%MO8kVt#!!OtJso!>+*L3%XKM4)UK}DSQpa<)AB@rF?W{$|&wxBpQelp2Ffq9%LT7Mu2c%0oa`9L-n0|
Bm+*23<h&Baf9s>|7x7F1Fr!h?zvv(1LDp~_%M^we@&B|Wl5_?%x)Pylin>@SLCs5g9kMZK*je(a#nEUO>o?Yr;TK(pDt<^xx2C_Ycp6aemKIQp{8#pY4)!(
HeYcN<~Gfob{YP3bwTTKqE}{`{*tAzuc!hjv7qp96@$X&n$y)j(Z?EkeB2QFn!6wN8qu54Yyd2hWtQe5f0DUwe_X@j2rbD`V~*pej!n*^x_FPOW5C@C%5aI^
yv+Ea_H2NHfmB$*K2;qLFv`n{4o+Bb?xvE@jJ)b9mi9P@7nVk1bNJgzi}5cec2KA_ZtC3$v|4*#*b_MkBYo*>(R#T{9aDdFA=Ku<6W?6M%5CiewrALLEhP4G
zghh%TK&E#A09suL@hv!(^wFJ;GgGXMl&ZS@<X${3*FWUTiU7%eipnuz*x6taCZC3#a|Q|AF3;3H<oICSwruZEFGt<7=2!fP<~;%<-R0XKk!<J*wu(ELd7~e
Sh9`0roL7K+$?dXgN3q9&AKQc136wJv##9ct6g)LL$i!^nWYi(Vt^`Kv1Nl^QlqJX!)(=sAf8p+JEQy$>uBD2$R)*@3N5(iM*ob9t`CHm@sBS&0`Com7+&U1
2CRKFh2T)h73sjiUnVS5nwdE4311_({|m3Vm1wo`U(26YNSUVn$v$dqL1N#@r#R{kZc3?0E@%c_VpOGGvbpN<{gBncHd#+j<*{$g*H}U%pW{Ra@&^|MdFUST
+Pp#Buw7Y<9IH0f`E!-?wkS$KsMe9~CJma&lPLl}1mx~qa%QLpO&JbK&<l?hWg>p)K$%zUo$F_&>o)vAtWp|i4bl@g4S#0#<r+(946<3EL6wakTc>0pfCSA$
){V$$NW|J*zC&}Wn8dl}#m`D>rWnu!3#C^{cFAA>vV$af6V|VW_;)<7V0?E!akIu$fW*m*y$EA$zq`GYPSj348S5_$Lh0~mMyl4I*lXaZV;9`=gHdQ>N48b4
kh9_;ARz_hhG9=dx*{AUt7>%$?{u>B#|Xw6w8G;XjIjT1|5pc4Y8e9j82``;L^Iy#s$s&X^ZRYV->GVeTHEMYAmn%TXK0LtUFPoMr5#0u?vkLmTF?-<SR{31
_-@e4|4EE7jD@hIN1sqCv<T3ASDGLm%4*%RApgPap+uPDimn`e9M98-QpE#25o1b^s(XQmby~w4se>6do5Wz3>d;<cr01068#sPMd&0xLpxyT5%Yx0PS4VXJ
TC?hgD86))4xj`rqRM+WZ2xP07qa%#lf~Ie!`kT`62}Nqpqi@BkJHth)?!seWDbU8%<nf)Ml~~>l+=67xk<l5vA^5I4H2Li?3*%c<Rw()`$2`-X4P}SY8ASD
xIl5b69#mDL30NzFh-{}n6PX3Ts411m=ZRfz$rJ!Ch`Uo)a-~VGZo!Z*_WAvOpxZW$zn;c3@SCHt*jywRdFLQ_RK1=sDg)yC`X>HYd>QhGSq2%tH6=e(z6)Y
mYpJ6-Bd0|yI6!g)j`5az+XvA34crd3c;-D7&yRbxf#j$KMS7y#K^t<WYMA==GGRm4}ZXUTq>!@7s2WempE@qJOx55LLp<&8TbHK=%H-995NWNR$&m=+d@0d
B?d9)B{Hj-FhCHP$QlO`#|z8df~rPnb#Fi5$-B>U1G7^M4e1ny|I~sKxySSo=$@6z=P<Shk5Hqn#A0Hu1G?}Wi5&LrWc$&A_VG{_-svYNG(W2X$J+zWRVHta
gTP)+66EH<xp&OLNe@X+6-9Lh({n?*i^T#~)O7RlYa^nF@AA}kDPRC|JpHZtsDUc%WIZk+j-FdTVv4&8yP3a=^QtetV?b$`HNoD3)ZyQy0x{I`5M=QA_=2B-
FhCkv(idEH@+5LW0i@X`LY{id5Ej-K;sOtTIR4RB87A#WBI*m-@4T54Y|Cs;<6jx4`~YWQ0I~-)NOU{{Nn2Q@`7AJlc~?ZwW5f_(2@c-6u*!6eGT;LnJGaSO
fc^DaI==Nrl4b0XGpYYl5b>ktzUWp-xr;5|bs}Evgt;a`cYeVr#70$RbO*Ua{OP@u$A%kNz1uDy^)#Z#PM-6|)dV*{p`<&CE&GU=6V*mw@wbbXXzgDSD9%QC
!NyijdDAIeX2N^)gS{G78aPOQlUSu#^jbyDyEfa=YBo7ei%YK6t^{N)>7qZLmz?{MhkC@$KF_7+tw>6S%)qzVu>aVf2Rma~+JUc_owz7M+^$7+os1}H62JY=
kcoZ$nD(>1j?wOtV?;+m$GKFZQHy(-Kdo*09{Nhg6(lF$Yc;=qpYpHZL2Os>idq<(8)LO9mL#TQJ))~TP_DvgtOj@5D@;=+nG@F>rVNHVyG`%m=)4TR>!<nf
M3e|70$GZA%;*qViwy@)>c_J@I>%bAV}7PSL;1iHXg(bSDx0wqyy5Xc)9f#jyO9%LklWqgN;TOZ5L16r0AFw!p6=t0oSfQ16%I4q^8~$E?BPl@>*P?4Niw+A
Tg1Z+WM=7;12}lBC%BSbwae&*6G?WgN@!GES}%2f7fOY7Q+1(c40%M;_O~lVR#6RV!~7eS$Q3=AqzSP}=Y--9)V83cSG}l8JG%=5?bs>&McD97uGq_$J5_z)
uVB3q0LtFU=w3^F1AurF*rz@0HH7o^hN)gU3Fq!!{?NDo1e^7StY3Yxb&6_K>)`~RmeSa3lV1WVhr;-|+~<NB`IIxVGjC-&xlDMyH6|n^p{@}smna73oBbT{
Em>-=AqKK}BAT6|%XTe3stmR|Qu%F9G=*YPIu3YTXK1?L7U?mlsFN4n4)S?o^)j)vZ2}JOd0p96n#s|NfDC-mZlE?sH|bFBkIx6HO;rGz1^EIaQ0K>$+d#5e
fh(_*+#r%4<3c`qu@o!nLFg1SajoysZ9@37)fls^WK@T*0&S;`D_s@6Ta24ZMm4~WKwj0Rs$8|9m=EQ~0ngwOCM#j)9ik}RP{k<KalhOOWCz@m7<-JrVC$^5
yh>cKGpvlRr!r!>c;~ve4X9semk@;tPOr|U+DXy9G7R-*oPdgprTg~nHSYuLCIr#Vyy{{moEm02a(Tos>?D1=J)Yu&opm0*GeoXC=9L;G29uZu`TF1zCHN|8
Ypj9$D=n<riJ>tW=YaW~>M9H=00JyAO*6|vX_mB{kFQ^?8<DzllDk>4ZBK$>Yej=X37h{zXNnrTm+%IPvgx*x`q}DXcNA28DvZ6K*2+DGUx`_-j(Qg4Eu3l!
;#^xJ=Gpqu6GrHl2(9u^T(*ovQ^D}4xUZh5EV*$^;EbeLlSylZHsK-s8?(E4XvmWaGHX#Tf~Utg8s7#_l<ikXnZ!!5t#>Ny`o?(D^L7=0ugGYokVC_FkU_pi
(jj5(i00|?RQ3efn}v_p<IfzCr)JSFs>zA-UcO_ESmg}pb=wthV>l+ZhFC1X-c>vGwbHT<|82-)EDekC!|NpftT$D+djnz!g$ay~UdVoS+4W3q+-$bKl)VX9
c`4y-h769Q`)l#ZVev}-&XJ8qH;8Y1LL^iponM^OK8t9?0gSc1B)~i2Ld-$lW?PJCX*polBGG*PeTn(WZE|8pc?13yl94a!e}{iw`+@G=pQ>+b1@hvoLbz)8
E2-ThfnYr;XQPM{6mD~H4r7`y5Tt~))9kenf_Iyli)>64{hr}c#Ypx)!aJikkfgDsQrQ7!Etm*IB)mt{LA(l7T@R}VlnHIRZqkuI2ua5J?JZ$V`LOagNdiK8
^KQ_fhCEKH!E$N_If^tDH{c~CUe;>LA*PUkgGhWL>fN}sk3FVk3DHGylxe<Gk!!<r+qm=B?)2CKEQ`(XA_MFx>t=dpHY^f>=Z4}GYJgsC;~id$FB7lD1k#Rm
9<s~5=-PEKeDoqRMfqjxO>aYMSp%Wk@%vuAXYFaMz-Q})NHem-x!CfEv)HF`=RK&a<VqZKM2saJT{LbHjws$;e%PKRPz0PPLw^oX*i{6gs<X7+tG!0iaumAz
evNQE<|r{aa(p+{7olaQ;YOwD3L=1AM*;>|77&&sHDvcI{6l)*ei7$#)q|yoBk69fYX=2y$<ff$F?L)qF<e@Y1dJ)5ZQAC5PLgogV6f~N%F_@|>hl-X$$0Lu
e&}RcIdU4Qq%u^&a&Vq?B+e92s9e`q)6DPR6!&ppc_o{gFLVzub5l5heYt}C0iGDWV?h?9d^wzUTgw|X1aN3vt}vrVilDdC3g?G`6A>%^QCw1x?2p_mV=pmC
M<=KQhn;D-e&t>?*_%Xfz8Ma#aG%0~;A&~;Ywn4uS-Kz+c=8{N^Z>xvWks=$C|0&#6Ox@j4H@hTRL(+#cn@h-`&qPf`9-z++bkzMFiZCdpbA{@*j?$<MGVHA
;8+FZFX01v+4CPpi83e2X^1U+veYuhQ#4xk9_ib6^84ThLWA6EMf^$4$k$XI52QAQw`D!mn57Yg)q(IJBzC|92{u;apX=mYfFaVpPN|aH9PudtlH&v{Dp-H2
lp@d05Z>-BCLgML%e?I)isSJ_!YuGMaD1}`Xr%!fI<CF&D!9r{aVrnG)?uIMSaPGXV#)(7AYb3JRJyJx{Ro_akiCDc)9~ZQ>6b*TJ|RZ2oj6$Sa}|nJc%*LE
-SQ*a)vIWi0Cdumr|0(RZ`?`E?|Q$lfeGU_&Q<x};_wMj@Eq2O`43<tP5&L!{9;mYC3fLv<T&}U|1(rbnuaein2Aw(8+pZe8jRzT_q@O`!`RG2UEhS&2}a1|
eg%kB0M}->y*e%6dISlC*LJU$hl6mhQY)gqPw2IIIhF^nS7VD4lV?o?CyP=OF8&O85sS5a;^9Ie2C`Xfpy|!HIh@}{N~+hTAq}KPC6XZ2EY46b@H`6LQIUA~
5YiW1zwYaSZO(gn2jNL@H9KI6RsCJ;vY{au^^-NHj8RlU5CLkelY6}E5K)NHbAaq%M#*9DS<z7qswC>b`N9Jga4L^&v7AjMt!Q!~Cqw_yXJx}DC%}$Whnj~-
{{w9N$4rkyyl{}lEghy3#?fu-i6mx&0Oqu~muk6cr`Gk*_9kOS5qW8w)qSJQZ$1;FWl`dIogvwXNv_Sp(~?2DY(U#UC6FW(@jf_q3FqP6V+Y~GAz3EI5Nsub
UmqCZ9&md!yxU+cN;~uckHN$%_YNEIDB(U|PyZf}vr_M|CF!u=W)+Bqw-g_F5nbw%UkCyRO5Y{zNFJDx1|qz0LoSdbtqftr<fYk4J)~>5iY8!7im`&W9-_UC
eN$p}*z|2aQ6usk9a-Q&5bC(32glXp_bVSF8K`>-;I`_E45OCgb=w=UA_>^(^$I4KL~gjhyIt<tW-RvCAJ}3V9CF;f7VY|HoWmyq>Vr<9I3)tYqfWzj5lMaA
W)YRKj&yydeL5~dKZW>pqNcZSE0$B7z?7CON0y}2ZbQn2O@Ui%|GeqvJEV*{a2kS~WEd84G?oIsbp2QMQ*=L=949e*^iWVme;+8x;q|;bd}ylkm<<khDLaAT
ZaX$$srHSz6EQ0!@3x2rJs&o&4l>;y1&rVpBJx-yRdzO5y#F|(&kyg@3F<;wvW&VcN|Z42fT+CO2pcDh>`{l*EzeVwSL01@xeQwMQv0>FCC1$f3qF>H5D`Kj
WD>;yU&u!|`GNqe&Fl)_BQ)&+bhXKxqB2e^1_9+Yyna-u0{0Pl<<ONv%}r}V1usA{KPe1QFra*DDu*niFZvtMaeQk;vful5zZR?o6cmIyTe|NSZWf?1NG@ZF
oG`R!Gm`q_HJ|k^|94YwI-<$JBjs{ANWQY|Ad*{9nV8mkUIwE*o2Xx`=V}h$UEJFq&spN}ei85I@sYCv0-~x5MiSJ$(?v^i<1hnUhjvG$Nql1#)n^FNhPEmS
<kF<yrw-Vy*w&6T&CRx7xbk#iSrmtUKDud@2JAx(7j|X~&(Sm19w(mH{daJNn!r^@m-931xv&m7X~tITgVdM9dsQB0JzW`#K;-QsjA-$xn=p(ecD(0H%*@3~
8N*r|T!0#0zC{3ASx{B?0%`3_J1(`T03>*IOyrZ?WJ8>SI?aIot^G{jFDHUCy|oGcUzN`5pl3Ke*AyOI8!}pQKHx%+i{<E*Eiy1y+va_8q%eMgi?~=2^N*F?
hDTXp8JDs@jbZ4(m{jVynLk&qm&J%`EZ9PS%EIW^y_|oENlxSY_ptC!6K~xeLPv@bsf;K!XzA9m$qa_JuK=?UHWiZ=5?108JAU!<E)aThk~4X!1?c$)N9D}t
b9FGU(mQx8b3#A01+`1%leU{Q7z#tEOea7?`H7XY&`7oh7_y9LV|}@nly08Q^aKN9mN^>qgK72$K0Xw&C)2J(8S3x*_<HYo&2mH44T0Kuy?#U!f0L>sE9S;9
{dxLfe))H^tLP{t89;^Z;HpAS_J{2L$J)<@*DLL;91-u_JWXZaKu!gY&&XVZXFR<&VanK9N~-*P`9Uzxkp$vpL8t0&3R54W;#Y><m30CxdE8GRo3SRdWvsaj
%!Ji7ot?`UefJ|sb@m$0fKsfez?ThNDnj7MpjtkWjA<sp{|cE5Pko?CTwby+1wK-eCw_-)^cvyMKHGH#UOdbjDDMAa6Ey5~{)GG&hxwMVoWNRHyFD1P!ef~)
I;w<3ZJ%OBgF=baGB>nr%T{ittM5O&r;_vkR|;q?H}bx*Adp01UV_nA67;_>oZ?enu7(O_fn(=CJwQeY4D~V7XI&{2Get$`(jfrRlH`o|46<2pyHo8ES3DEu
sqLR$?YnBGfOjK+ACRa<0Q#(y^L2jRpkohkI_Sd=vtSXnK@JE^dd<uf`p(OgYMXtogin~+s0f07u2XVYHP@iO_ok!EZ&Gnxn^x=9%fM1t3HRt3ywtTXV0p#s
YH#i?UfW7DG_wx%G)=j|Pc{hC&WBDQYuFU-v-=^@E)eT7LTOA*$UmqsNea+<Z2x4UqvYlWY|ERlsArTrq|nPc$?gMTvIJZ!*Crh}N6y0IyK)NNhh&Y=otR@W
lgCzHGsx^lP92Oo#_=XsI%<fq6^Qx5=k2bolTnjqg;LjAqD>x`I0X*0h-3o_lb^nmsABi7(E5Eu;*V^f)jkv@;UeiXu13Ok!qSq(1q)>Df-|BtY25nm0;@n%
QJ~x(Z`KJQ=1XP!*!xdzN{gFi+4_v7SO7Nb@+~@)YDU2lV(zx-SLKY};~zR|**)(b0JrcW+4o4hRB3-Sl_~Gw&)^s*RX3IgFkd!&7k&$m`Dg;Ph)5=pULBDe
x<$;plr9=F_-6q_46cZQ7-;V9ADKbd5ya#{rj22&*;~JGdC?(!dhI7vv28)S-hEK$<xVM~V1pU+%ETG2G=V!>-UwUm#iosM75hIknMgMtgma}i>!BfFnvI9t
#2P6ou(S^!|E$WRO~G4Vg22*<Tvzvkb(oN@_DKoAq+3fb?{?Zck)*;A$B5PjW(P(UXPwZBe(g5{z7nCGHcvL`Qg3bOWcOFzmLkZl)>bx;NvQ1YW*Ye$Yl#I!
QCNI$F;ON@J*Yjp(Oh#z$Nv~TapEnN8Wgr0;!sm4Bw}wja!T;FRM8BI0JFb8WL2><8H-uH_O9(EqAJyirym>fc*~-sLFrd8XWtyP+}ep5nD1_%mwM`hF*l#5
d5m0H_SZJ0re^b=Pa8ReRVoSy3MVshXaT^+w9F!z`{k==v-#N&$nRZY*)bbt-*{SrI@Y6VtJN0B%Llhp2I|Q%&|&RnUGUIu6*9+@@Cyb_1gD{aDYF?hyO4v=
#W#uJYe6$!<$RReXfZEz6aB!z)Fj{4mp&7*5azNNSwhQJ$c2bC+-U%tCpsmEaHfDHB!BjaVv$!YX0i`#|6<~M^Hk;QxGxF;-Xl?s|3I20!|}Tr2toyGmlr+I
7!dmR&;N~83<row`pmOHIDwH0C<0Xl=_^MnK=dE$h?i^T>Z&+1Zc1(}m14RRtkeGui)(1&wb?*Gjl#jQ7XCX3sFvtxF)>$bF_iEtT%BpRXc~<#rT@_^eL8vl
G9hS0s9P@rFFJv}4=1dOgDdj>=jj%94JDcpV$`#t?VYqsVMg?y%0XWTYeR|8#VqophU&BQU&<~UqUDgSd+7uM*5sLE+C80^+}Rq3<*%{hzD9c4bG<}3(FFWA
!ks@!bFXv_?RCh^V?~MJ>r+yBo+-aXYJ@(FB09RF*rrxO`iB*%E~LZdjJ5NI!Cc+J4cFDUKYKTM4iPy?nEZ$geF*$xP{5Ns@^6o98A_^h9`T$1+tLRYdkc`g
SwQdn2YC%*4$<08m6$nT;G6v$mA4_6;S}E7qu!&~jdk0}I|+JP5eo`$1kaZMUXcPO8m>TJ6HM**XInJCCNCn8BC4Eup#?4i765FB!CsAT=d)_Cqb0J(y$JvG
S^YH2erM%QZ2!tk!6=*`T?x1gPR`WDZMOEt&bC(bnvwQ!`gHFsDQei9;&Rz-Hn7pQ7DDzOnTw({ALv1Vf@AtI$sXHV48O$y;qkcUHnmVZzuW$98nZBOGEofR
K~DL@>fm1uL%2Lg_D^z#5Ei?#_frySj1qcq^hd<}zxH=5o-&t^DY<f|UB5%=_d<j+L%2U_jJ)Wb6GlqbS({D^hg#=?HXwrE@8&vbiA8=Ss4VLLcBR4zD#CUP
gw}XsMP?rX<lru~iON>x8pP5cJ+v;MYyHBimXWPVQ*{?acvm=%i7D#)uta=LJ<3X{Xv+P&G;b-0IvP9#3;l(r;+>1aLE?{0l&2P(@{rvG6O}8b%1K6WM4`n^
ZFDWq2ZhfT^Z?%|)<(&aK_v&_^X#nULpvKa6vxdRdXAC1R`sB(W+*^51)cq9mlP|#K7B}xJD0R^clJl{8$VkcQ)x)Q5*82qF{FDX;)i%rS&FXAQ0S*(y0Bqs
N^BMOv8V~YMQ~jwW|5kRg8P$(sfM7mOGd4+c|g=5$#ftsuZ#+&t<A3vh?tTr={AR%dtt6jz*8J+T2$=*g&b%M7+jihX=TAP2jr#V^XVi-VEW)n`c;skCS5%V
Oq_uy{UgG(3CK4opu&e-{^@ZMk{u6-?W{Ey=DdUW8#Hw`s_XmQ2L`u!L10*e6I(CXnWm2PzYhQ!Bts_L7&;lQn`JBEh)6w(Yh88776#~>^BS%frGfr*^f{Lp
IhCck&hieVRY$f14^yTp=M0$#C~-^I1@DYY`sxDQ8=X18sM|+fkDHXo(cxF(nlr&p6P2f<2xv{gw;P-@D8iHvj=l;4W=TSLpU;`nl6*<fATZ^Jr>o0!g%wZ@
Il9`GRmmL59Q!HhP17zupH&v76ZGC@5m%>Ng6si3w|q{@aVF#R+eB_Eh=f|W=HGA!VVZoBxc%$B*WJ^`GbKx$1~$0|SIgc)5Hu(K+SYahN*3_vX5A3*@Oa&v
5GFPr4g#YlFwQf_C{6nk!!3#K0H5++WNu)cIP;W=hJMA1-=ck4Nk|_e9&;I^WS$0}M)%NNdpoST)l)|zY>*gsN;8FFG1<9>f*ymEYij<w1F^^Q`J5k7f=??W
^?EsxJ9Djg&k%L1znn(g)N;bqV9!;Vgfu?*7K%hF7U*j@GaOK(*-GhaSjET0wcDWaYPvjEiE7d1Ej2#|t5+7rY*H{}^*{oK4BWlu<iIjb!S2bq%_HtRX%w3e
)OAEG6R;OGKQro^=5SlK$dA$Im|0uwnLoWws3zh&!;RQkIm}T^0?Pr{oQ`cX7VM>I|Eg?-BG<t2Tp}!u<-(HG_4+Rqt{ihBjlq9&{eZv_UChx3zk8+^VQ;@6
VPU7Jxd};J+M;wwpdTx_gW<<55Y_BbR}H_xiG|oEDJ>Ow7ecKF4UcHJil`%UW>Kbk8;tY-@M+xf(vXS*paC?=J0EW!s7BBKnZesGcH$}OtP4@Yea{&GW{_~B
$IZ$bSPdlRh}Q<Ej(iz!hHchJ+rg*EL2pKavO`>OD(^`D(At~$uwQgC@y>S>@tn9LX~U`Iv_Pwt+(qs6HJl}`IVuXl$jgfo=jr?&15>}<q~CbVHusa|LbiKF
`be(jTd`>NQw?AseL41>c9WDFIOF_GZETq6DLJb;f;ddn?gE}WV^ZtCXji}jRAzt_mt!A?_?@26FQ}W;{8ibVLYoIa+xN8zA4Ibce-+3otKV1&^*npivG=*>
cA}^nt|ilPxkdyLm`%#0om@nd*IET(RI*2OS<`jO1?RHDZr+6=D$~EL0LY5^_bm<9j@NP0Jc=h0AncI4MblkCP{AUnF1KqOaQXBG890A+7Wv0X*d|S3sGZkE
M2R1sF?RRNiujRFD<U1#Op~+}_qolaxS_U@hI$2P<pJ;9zC*s>T0u*o2nmS!7BGSSxvm$gn<Jci+H^}R{Z$`G*pws7?p`CCTd)>z@g_=QpBY{TDTlZF&h~4A
(K-ignU|+YU<@aL;|H8WhiRFBggoS4eldCs<<>+UgP>7SS3C@pA#t~j!f+EM|9KOAyO!69AduQQ&vAem;^X=B3)Un?<YT1(i8vvFt|V&6vm_(>HX{`7imRp4
`QB;J%!Q)daJ74py8yt$a{qm8@O?QCM&fH9@_nbwnoKqoIuQ88_8_;q+H>$^5q1qBiD0*a^iSO*S~!e$s|R9gAt8zwGz2e)MItLr+(lE;JTf0eIN(Z)00+$Q
O&v!D<#tb2E3eRf{|6V|u=T)=b>*@c^3sapPpzL;$K-yqKyl;ts{i1?ndDD1>X}SfE&&}d)Z@Yi4^?9b7}4qPzUvShxLM&EHD?miRRpDka#AU2`S||;B`zDW
;)+z_u1tCllZtfI$^G;(gP|Zlk;T(QsGf~a1~TiS(9~S6zsrI94-aF;&T@#tmO~q$>P22dHI!r#TL0zN*3d|1>K2z<h%|rU@uy2bl;jT)HN}Xen^0OC!Ibpt
lYoW<RfX5%Zm|k1U4T(Q2X_Jc9vV*uD=d@><BQu{Z*aCG<<v5IQFRPN@g<<S$Yf0KjFPaE`xr5`qEc9?37cLiocw~XI})9I1NkwnZQ=cP3s2*3cW<DqJX{$z
e>X%<d3xU|Q2;27nR|cWPf@+^kJd#!Uk7P%9u}>nrqz)k=m;h8YfbKhDwliakGj@3>|(P(L<4!pCi%zP?D;NKz!(}XP!n)$`=#+AJ$s%1ylX$^u9EfsT*xIm
kW5uFxU?9Hy8<p6N}^7wv6g8&NixMUHV_P?G~NQX<>I|qrTI%o%Xr`Q>~bSnr&4jXV|I4UV*o05qdpo2=!U4o?;%_@>*g!f!_AwuM*D9`yTU1kRifcGa+lPG
@*8qIdf@oluWrFXP;W1?A{;_@S!St{V)RDig?c=V$=@QgS#%a9ZD1Hxn!5cr`9m~380a}jl<m<kzGPb0#PYTSaeJ`WpB{*4x@OcUO4Z^He=o#gveujaZa?xw
7`gm)AwnFs$+!9<z2w?!I`2drnx?pm@$oB^sz1g*yi$OCamOoU^#HF^9IZob0+{p3*lz5aUlg#e^focE&Ty8oBEl2%^09*eogvKg`rqHDz&IO=%Hu5x1?$4n
W8zKb{eaRp0cL>F=WMR<iYa7q8ug<kgqt2Zdlr%|6Vmc$m+n<20uGbua|oe=B_RZK6vw58%BtvGcZ#9XubUQVg_N(yPRqs86X(O$c5t;3J(wh#&s2&_yw={=
Ra_PSWUoGtC)x&VYR;1}7l<Z(*BjVUNJ<JE%2X_r#pMsbg$4whFXCrF5n<BJVckf}9c*Y>7yFSp+eVSb$AD#I_Z=W0P|dxz=xCvLqh=NCt!#pWiH*~<27p~?
r>~ka*Il!aqksSHglDfxLtHPw6jp8w$d5(>p^{z1N<Cecv=V|9yYJ84_nw7PNub@(kWULoykU7=K9Pp3D^C7&_<D0x^TD5RcHaCggyQtm@#yXAdtRGEj!Xdy
4J@b0&idMyi#9FDIKn?bnRF3t^LXPtf}tAF-Oa4AL17A{eH?h1CstA``rt+I)2+IO*{eximi7(w&I!Eg4Zn-5c$FC5p)0p^3$D#*_!n%6YeEos#)_=;i?!5`
#jHXGyM|$%t^7^ekVK6&>~;p^PGG>D5WpN|2<Pt9O}rlrx?d{5Fx*UkA-+m^WUe)GLLl6X&i~3lFr;f^enlLgkP#FvceL8<rMY?$J4IHo8bcPPw&93F`5(XQ
JdgQva0Vw!rxABL!*|Wb@S*R7kyC(-A!9k-S8@YWY`yj<><>N}ax`?6=2u@xfeTLJH_Q<M48#db<ieE6X2eo3)|zqPV%#jYkWCoJ?H6Hn>!-Xz{zGn>zJdtp
;Z-9}E_VVwW^SEE0J3>`#R%!E;x2(y3t3%{(Nri0TK>)YR4sq%{&wz;mOk}cWXo-r*?C-m<f&%+d4S%%L3N6-whp6kr(I{yo9=JP)}q16q&td@1kkF#AW3oH
#z%Qn-_Mpc3vqz^5|WP8r9gxfrY&;Y8>ErB?%$93e%*b2S`t;A;YSJqaOPs>x2``TGmI~%Xa9rgupj=RJgXZ<=z&w!p43av@CdL@tMOgfMs?Gd6ma1VBWyJl
eJF#){P-B~)kr&N$P@NhleK^bq#SO_NN+c3`GmPwO{PkdnS>pt03K+9!M+=r;&&VF7D9|mRT@`qwyQq;l(}^L!;n@6!$2-9Xk{sZdE{v71!to?(r2*rC%x$F
S+h=fz{8d~sH@Bo#h9NLbJ+D#>(&?RQMjMk4}>K&4@6Ni-Vy}}WnS!*5H(r^;6l+ke(4i1McMlS0v(EI`zsxTlItRYt7b=x#n_!@^Wd6dLFy-#BcAyTXJ!3?
+CboA1V7O{&x)02J(@?FBGoeWv9en>P1`(8^96>gl?*<H7a5AI`E#T<#y1IQZ!_v0BeivQvXfQI^yu$7zruGXtPd;6*-SDZlW(u>D~K*ukl?m|=nFAEfF{=Z
OtHH>rJE>}@3wavFC*P~Jc4Tqb9nFSpNhMm!>Rlo_iEcUYzOUb-v;pRp^q9=n2*^)mz+AE7(r%3a>hDBM&$Sx7*&r2#4FiO9pou)B?|*-z*r7uZxb*`YI#!i
!U69d17rJJ$g#}0Y`<3RH+Vnm4~OEsWM`-Fb{g3eG=Ff8i5IpY=Fr9W^su?g6Uth}0KO9k5f;vxu@74pgXec|0n`skfkW*PaJiVJR=?y^Nr|^_(x++Owg>m9
I&_GEIm;hj5YzNxf0YpJ9+vXMZ!*2YI0U5Ss>xz%nK~*1=(-q(cisN5(NA=B*)`L9gptY39}ac7rOiGfU=lJ{i8JSZZx4SZi=8+5K0Hf%wYHcH408y<Tfr!`
5)zk=NjWMqcxobL&j1@MJG0*6<kaCJBeWfHBa-A%miF9;gxr;$b;nk!zhQ8jP7@mRsGf9mK|tLewi7^)n8$7GIz#}b)chJ%VH-T!n?eG;<MR70`2;O^v+Um{
Vc_|N8zzUvgGy2l#CY7rUOo4+y39Ce7x9jv-l7G3Q{u}9mu!b24y4x=v;jo97-JSwkgR&gI4AfdKU`PCBQ_Z=rOMe{(Uredq?&MH9PYb0yq&s`U7cEajPmmw
tjCu^vn_&-{PC=tTSa8mon=8zjRSnkJ3Rmo*Cbsz!5}=3zlR3NKQihfR%G+JA`QwuEoq!X17iHP7JGPE_U<d(-GO3^Y(oS2VDcpuXln;AyDQ;K3=|v8-&?wv
mU$4jdTh7MzIINb{)dS&rtnGJvxRbRklh08+Edq2i4{a@t#S=!0w5{ijgsaW4W_fKFg_d#>6>hMD*yb?xu3dqPIiRyEVmXx1VZmPSs@x72+E~YZgL_RtR0#3
nuZGCnbR^nvojzVaR%KR`2V^kxnq{^*l@$XiJ9t&-^E<A81EV|-I^rY62#rCwI%S)$=b}z;88B7AR>RMV8C%>hlU+++XBaiAeU~vdtIm9)0<fyKzs^&NzgG4
+>eoS2KF?+*L)7$k)dWmfL@^QI7c{U7ZxrFcaB2@aK7h0KJz*sZHPZWgtDq@$E==ToVW>*TcOYvTH_I_?k*?6ok=<6l!2|t_+Ytbdl^cS!fZpi#RDVcSZAlW
nAHdGGjHPPU+{$shyUdDu`BLaa4kV?xVr0}$oprpEBtf`FGxl$HxXWlo_KfXpjuE21?qTb{`RmzenZDG9smw)9pK`o_3VoMA~~E@8{HK=J11%>&ASEemZR!z
k)HGe^ubs_TG=Q@Qfm9p-NkTp7%)ZV#UiaG-5_d0@%ZA+nh5_4_Q~f%CKVo!BbLyBcdEr&qyTF~4djLg;l{+={vLMBQu^Vte$}3I`6BXDr|wPd-?w*4MOOPX
Yc+P^`npsd{5bTG)h!`*w2b!1SbOr4*p%skh~L$$oQ657frFXgWCFusoI^msr~K7qtl(1jZGX8_m0_~d>(NsXbG#RTY9%##ECG*55Hwi@o#imjO|k%#7@a*p
k;Vo@<E?1ep8%vh+g&5xDQx&vCu7THlO@2wl%|<?5myp2)U*g!oKd`jvJT{ed$qtFi+;U3Wn3RlY5S(-b^E&3sEP+V03h#3R=+Cq<A!UU!s*_-OKb2>Ivl4}
g+sS$;-aUwws5g{elyu`kl7G@V`Up2h9YA0=`}qq_6|v_#uKE&tzO14Ldn=$YblMSh+!o~GRMP#vVL-`!J{VVDz4pl%OcE;U<kd8k@IA_CQgIr>AXE`v8DC(
N0}1>4bWqA+m|6%OlVM{<22G<kg7EUc~XHsn?g@qImO6X#*?2Rp4G4QE`vDMy67`!nRkQ+^G^cvu#tvv;AK&v6DBDTKGRkw9ls<aG*zQ1m45SkYW!YL$fFy7
_6xDfR$RZPJ<a?wdJIHRu6jrnDI+6KmfMR8)<yU%d)W1rd<{JVgrT4BasvBlf?yj?n5R1YLqrq0pK4#lm?BV&)SxHx*YFDXQq4%8O;YcqiQJdKudsVcv<O!k
wfyYaQ}E*E(&O(&_yPAaEJ@lilAHM!x>kb_rIhQSOCFSpBP-02KgcqqHtiswRjb7t_ybC8HmSHE6_*rLVYacgG))lG&O^Lzr(ge7k`%9KObo%OZc{mfcogGt
9NrWYaPS??N_rnWVaWN)dBf<WT?%V01ZyeE%kqJ3B_)x{q^ol0xURbLIz8|QotjJ030ZFmvud<7_JQ~|H8RJk;j9^6LwKS$%h<N1fDJ&bITZzyhGBLRm*Zo$
XPy0c7-mV_-%o|ZE%Xj@oJ;_di^j3bFl<5yH%G7=1{H;p3&aPYehVb9NS!FU5VvXsb35_-vP#1a6)@ijJ+;R$bQk{#LGEh+*=v;Wt|i0*OVB`(XI3l*?!}*Z
hUcG<GrTY?>m-`VS7_G7_xQGAAwS^sNAoDnfjV`2$59g!zvs{E4VA`@Tt@zY0-S_}t}a1UrN-Rq6YQU;Ni6G{PY~rj9A;u6l~}XM7QIUAtg_qyT~uM8XforX
f!tQkLBs<N)I`x<RB!6jX33>qrXWnk;?mfLMS(JTB>_9OhQ;I<r@&2a+jVQHI6)$j(Igh8Iuh6RfWwg8LqnuVMXgjVNUZG^fFKAr(*33Z(6Q}&%7L1O-PcFq
9#Q_-2Q_~}Gd|7cQ^2+tKqhq~L<v&2Lc&xsEU~<u%4J2o314Tx1JPHh`q?H`S=Y}`LasHt(m^^O_3yTZxIS#6#jfq>Dg?Q%utIian|8-<u(te<iC>c~d`Icr
O4+@y<zi>w_r*_hh>mMfvg{}(^c@JK7gmCZbG>!}Wt-ix&@h~tnP*2gj4%@sFs&lJok~F@Ld;;L#<uSz2xvf)?mk5)b1l|ZbkY}qS}^>5zoUIbgmy>^3p4A)
fL3WRAihVZv*%;Xp%$D=@iL~Q^KC9=QnTQqYLR-4O{!BC3*V%{|LvjD1p!FfHEsEQQj8+$G9$~sgBx6l_D&Uflm<SPg+n)Q@5?AjWy3btm7kyZ@g+C9^V+uO
@j>2NRDcQTM#-q8pfrWgGo%e~U8x|wYwMNHgn~eCTnch$5a3-}-el<S@rpe($!N3Zc^?pRkEi%|^}sDnp1*B(o(v(j<`_?qd_21&^CJ0SkN>by`z|%qS_Y~-
snIQImBCn1IT}L}H}+H6`0%n5h*oS>elTm(nV>khfqoX1C84c)3y?Fv&jcCUiVCky6%$@03<K(rZd%4d)lukHb!F0aM)B;8(5gq92RUxEbUllX$tqls6TncE
(=w!+JOC<#-t>A9(TueQY1hREvAJgm{LSOa+m*S1mIxM#l`Le%(6MY}-Ohs{Y>mrU-B(j?{@cN*@E{kI4)1WZZ^rz^xT9>K^s%I&hKl0}zIF}#F;X5fZ<vZ@
0V7VgV97^#y`=nH#d+z)JX1(t`9X_}LgCsZrhV5(JDEy5%qXbF6p6NB%{cRKbTp$}lQ)>!i2{Di-HA~lCMI(;f#cVK>WMY<0A~3mk@>*>?<|b-a8g4InwSOH
QT>yPvaqQmO}Uad4zqVvyo*U$(xIy>8s?f1&{>nP5J*<@oeV*K*f{d@J)DW>NgSAAT@2YXAbuS1j>_wovzFqfZok$rqM*8WW<9b)cYbN*cvM3Q%Fd|xqFx11
#Buej@czh%PLlTaPsVJ&w~~{&f!m}IG!3?{=F08bkYTWcL6E#Y)&DwI+(XDTppR~fa~UAAcjnv<T?Cn}^U_PV5fiD$H&)-%Jo2ut)kATg-j@POZ4r}&vu`9c
1uF55f{K48GCjOl)xWQJKIib_CGe7S;&m>C5AUceG-_*Fc)5CEd|R1k?e-&m|8d`SQ*X~SnoZ3B9ngm~81rT^+_MntlP9$0xi8>qM;i&=+l5K=&Qvzf*zELC
ZWkvs2buqdECSR&0ZtLTR0tby;y&j+Mu2=n+Iint86QLCmGuLa=sf%IbYC*mDU+Yxb`=D^73EPt&}$a{k&~}qmd(qfAFUzMy*f`B$Dqm3u%9OT7AJw1EgtZx
#hA>;Pw}GQRg&_Ofy(hOj}iNy<3q=r2Pt9+$QC7+2MBe_o80TQcWY9fQ}Kq6(64t1qYK2VvF2clou%2-V6V6vc&k>m0Cb5~8|km@)?^Ica|01^k(WS}*u5%y
60A8%3I-B)BWEGX3sP%!C+H6Yue*~2U)!;2!KSb5=MaryM*B>Ut0v!)L-xO|_FNFA8!H^iJy3WEf4o`Ssh$-k=>3wue`2*|c^*hq`Q3uc;w1T&<F1RWBjHS_
up%XbZOz~ydl>VWp80dcG)5Q}^2bCmM^mY36Oi!;o$b_#(44h(F#Y-N2G5I=*(q3i0>i||72OeAydxVShZC)UnERj<KpWog1iDlVgd8~309!|4T#+>mH*e$;
;68~;w_%L|T}~K)s$kGLIC1g;4J%762=hX+>x@AA1*>>q=ds)O>|;k-8#6=!iMmJM92bxwDOSu9{{q+5pTY_shmDf~%o~MTYgNW!`khnTtgp(S9nqcyXxCXQ
`wwfE&)|LbO+O8-uPdfSsRYJjSbIq#_*Rq+pjFoo)oFtsFWt>{J>exhyr%7@P!$=CB45aF^Z6+4;WAZrbEE0wwVNdURzDeHQq*|$?mUoK-%6x}c_RafJ1z8O
|8aOMO);23%W%oCPtBNGH5k9K_M>U2zf2V635uKHnb;-FXmDvdeq;AhN=Clr8)#JM-n2y8w{_4eBTIiR8_SCqa!BiH*~(dim7LNVDZP7842yCuWK~`Dzecr<
A`9$r8beD9HcD@e$$LwmEe1M#-4j_6MW0T8kP<>RSZA+}nH#ZI^ukrAxiKIr`+ZYp3OlcAc~Ym?Fg;aw&du&lqWQJ(lrOMMdvyqbg$YoY5wFMx<bWY8BQ7>K
0zgySP${A{fTzoyVI?c+n`%uCgnGa<ovf!--Ds9~im?zFQ1J|{8UgpYzw9^kR)h24K~FhKjwl~mO<GBa9J#?q-PB7kwi#;+up5YzXJf@-=@{4!k5;O;69&PO
i;8uvEu4}lefP$bCb6XrU&9?V!DnChy1!yxjz6Vt()0spkoA(XVCFSsH`GNMi&daceqoH8WiaKA@i=ni=^J;$jCc^%)|GM6(g?4XMFjcJFXc8I;Pl1@<Q&!-
M}XdS=ZV|ZXzd5%u;>OZ{=m4|JBgUI%yW6AW|y_9llzV*sR6H@1TjlJ%0NZr_;W(U?tcmcr|H2lE9<`wTonBx-c^?2#@y&c#((dl$#KjbFK8We$Yp7vUcL3=
VeIqx;MHae1)6!U%J3Z~RflYy5Jj0Hs!!u)ky+V$Xtf=E-N&{Q{~4pS`n=_u(PQFMQjQm{&IX5*(36G%ZximG`h-waBF_O9B*+9GDj=`QDtBOO4-R|jU`#Wz
IP*!{o_^M@32>nb$Q7=kr_^x&>6<3_gynJqv`HzptTGf7y(rXY+D%jx1$HL12D@NdE=f+UBbY9#GtC~(Y=iVixC6WjRP{||sTchgtPF(uh?ah(kIZL7F`)cn
g(Ii=5|$noK<WdII!>PF*w29iC#W8CE>r;GUr_G0g|x<n)$S|Eu3gl9Pt{BoAO0aV55U$2QUwsxOmnVEW~?w)ZmrT)K@Q*n;`nX3A|z)4nSXH`di1*{Jwa58
Nwfg+HhAM=!6vtfeds5nl2`{6%W>yNP(0A1j6G2CyKs9%MEI=M%;2;rIsTR1zH5RRC=#JFja&q;Fq5|xYv2;H>h>jm*ud2r^C|VCysI2nU#X-<OY-!GD1(BG
9{{kM%zF>VZOwh91-lK5H5!YmuC87Z8_J{ZSpXTfvlaoU?>j3=G}D%t3I-*358&*ogzVE%<XQq4t1Hgg7c(r!b`g)|Gv*AF5p2ixM%+}trfgtBazqDGncvSV
%+8rwqlifm0v6guPt;kocWNjc7J%J++d^}LU>ew1-!u(hH^vtXiMG}9fX4EJK$FvbD5s_log&*skf$fNVx(C%AEmsVwL)-5J^KGQd3^JaPA1}FJcG;BD2^ZG
6KTzwXo~i+x5?mOf0GZ7v3cL`^&_n~4hUPVK=-5LFzrQ?+i+)=WqTo#-TklWZ>9ozJ}SB{DGN_pEVfGe*$VK?TJ53W!HPZp&66c#5qpZE15JV>;EIyH!GJlU
>_pI3-fgx-OP8g)xCmm}B{WNe114m_wrAsN6d3qXW}Ie|>-qGYat^tzZrs=e%TB1Jq4znvi~<b@>@q217_P~8e?A_*m3QHiYr8}z=&||IN0~;d;#Ouj)lq;w
bYCOI#kuGD0=fjsBzD(s6VQi>0fzS#>=~@t8d_;28FgL%rBFvI8~e|Uo3y8KxeRNaf*W>!4doj*bwcsdi}#YIMB!!Z(sK~$6Yb#z6SZ-It!e1BvY-_q4|W{7
>+^?JCr{ej#xu~Ftp6+!yA)EWo+yVCqG22Rk|X984uLzbZRlvtH?Bv)k(}A=rWamp7u=6B{>~cP?N=4|3#)mu7V?cg1k#3RWREw%9yJK>^Hnfy2f}3il_wGD
do6O`tY%??>jIZC98AUUVOd{xElq%}M)8Fq1F#mjQv#NgrysC;-NjY$Bu9~p?^jBIPn>uaJvqJFN_kE_u%Z4?^99a;gfy=PM_nF-oJxG;oY1{rSt=%R2}qg|
M1&y<zzVHENyc<X?8F1$icDCjy${|>-J>HWMNeY&JpoG+&U;coaXjr@t{|xds5>GGlBNhi04Px9=|1;TUr?o&Gpwo?fHqy?7kVM*KOBrUAu1t6U2-ZG0s?Se
$5Hr>Oa%2JnJqD3QK=!ZOd#>c$a0;C0X?>H9;ZV)en>>i;3odOZJSlK#6Px+qj<m$4;_J@C7sDKJ^dBRBci;A+ekHcu9)6o>Ie`HdH%k;mf$XNlVl4I?$Y#S
)OE?q>t)K=2w_M5QInGH#}<QSr~Tb!&@Ymg#P~~!@s)u-{KMun8=SL%^*?!mx3>#0VTKr(X$s3D(vzbd=gtfC!2NqLQ46xXOYj%jP7@4JOfEb5{k@}qA=h86
%wvac=<`JvGj+1gVe>fDz>f56sVoQPOa!qU%Sc}iV+)yyY66F%C|B|2aqVj-2~*Ui`~LYVl0e)~__IprkJJDUuN9?J$amEyoT`P!dO@_DPwm7iR^&PBJ2X7T
h-@1aJ={&{M{AWi^|CylB>m@z6qb2=^PSty)1ds~>K?pYg?-7_oIWnS7285lxG)uLg9XJQrjvL&7f0`AI3>=-ckQ({qCV8V7llW~X;KgUluc4P&|dv;`-^SW
Ab{y-xU`1$)zL(X`iOv8<Z%Otkep-gr?XVmK#>M;JD7eafpfC@AIjm0v3i)i)X*MDZ_2;1!mwC@uSe2S)QBYtF*QIL%f6H0g}^K=K<e#YL2H1aBu>mi5LZGd
;i;HHT=_SLZ-sK?wLnKM?WDmthHE>|@H6if?4YIK#=Qc>7MT%+5UX5cuuDGEUgmb}3+i6$skdiH36^xL&|Vfia_im>hC%iGi@^B#!IZOcep=NsK%In{S+ml^
et%wsV-YzBbmtN3wGsT<37MAIVc{={z5M_RxyFsyxRqk*R_JMpR6dU<Iwk#pZ#kC4n%(}y4{8n)`#Apk%I&JB(pBrpPX@1xqM`P!rw->80=Dydl$xRBOhy_Y
vAaiuy?q8*+8c`9!Cx8ED%x!0m2}z-M6TlEG`&J8GVUiOMBFn##us2ord$59r@X2xSH4?&+81D!?wJg+3a~IiGbS)O5M}*|v|>O!<;Jr0`2nHMsG)#lm`T_X
UpOWgcydxu*@D-4t9?@c9g?2u7Jo?MRP&=QRsI`%4g0)2kVCA2k$E6EmT;h_sPe4!;Vug?1Z1ids*P#(@$Q-Zt~qAtm1u10xw%WyIN{$*UY8jOE_65t#MDA~
CWCw-0}l>}0pA;e^F{=4u@w+4LfvhNNwwTZ%^o<o>QU!~yJI=qHdGAhV`CXp)Qn-h!2Ua7<9|2rJr`RcKC`1_?_KrZsmDRD+`C1Qxv!EvxKc4s2=Q!PWL!0B
Z9_yV+bMQd=yxl8D#DUx9IqzsnG+Ff3(n0-wO5X>11gTypfc|T$P5@Z|D`T0elL+e!~$V!%xcn!R3eb>JxTl7{K$yMLcv38A{wX8Isi?AR^<XAt-+jovIK<l
prcR;#`j~sQu`iERheeq{Jdv=_YEJNv&N3oVD8f_fG7*jWIwsvn;obqI+sVz-aOxPJ)ar3uRug8#orFzM-(SZ@LvuY3>&2G_xLXuHkYjDMMOKKB2)2&w;n=c
4vG5vduJEhUkL12A6A9NKl1V&hWW_JCwM7di=7;tSP;z%IE2;(UrF(ufOW%44ptX~#a%-g$YzU;Xz@ZQ-q^((dFc&d)k55IfpotS-E_J;aD)53;;!Ni{f8$`
*+WsBM2H6`xZNG(Ml17~n}tSR26Og)04_9n05KVg$~I(oAfW>rr^@C59RUQlnb4<UzVV*XpCJ5LcXB|n9F#v3A~qeaqYUmUya9VvIqAj|@LE3kL&jq4GUF<c
x<jVcRn4s4&G_WTQiKtK$lx-t(8TNai#_oQEK}N(pL|XcMEWsYZKw3LUsF+4;r7kckq1}VLw_-I;*;=^p~ik)UvHdWN+@C0xQ^)U;X^$-C2)@XJPQL&|71uY
ymZl8ckC+vK0FX+?x#NLq9rsIod=D2s*;sFxpmvO^dD_Y(K9(q@YH%gOg;srXY-V!t>V*`$Y=9?HJKyqxn0mvIPzCE_T@)eeSe)K063wkaMi2dDCG%pDaKXW
kSb<(<~Q~H$Ya243%zjw_qlJ*F)MzvNS>21!oC~4aM1DKMHhBP#fED32UN@o0VIf#{W+4$-6u%qHolK@k3aOStD(*`RTljh&Ydn%^XvV;5~R+P$i`O`@yd;1
d4beL-5HH|+(Z`^u#tI)*LG{}9}tRjr^8q}LDbDFypTYdJI4jHg97DKwgH|8H3y%L>*P=%A?BeQM7qqe`jASOv87(<opa0~8(6MCl4uBPQrSIPfKB~uk9Q%*
%YDJOPrjtCyPt4HSFm^TPu!dfhg$B&oM2{^O)f-nj6q~1)uAqGM*)!&PKAUftj#@q*CBk%ed;fg+@2ksC$;+qBg7`>7D06o@ROO@B~y8)Do(4dIH^Qae|t}|
h_})#<Xhb{)!XXOc*#(g8=ChrKD!BoaCG&RBuKBRNB<EEK)=(Eh*&*af_py1Mq{Q7Zm*TtwmQ(G#LpY>(0Fq6-hZQT-L2G1o_2fjrRIpB9u*vKsC1n%P0lR(
Sos@xpitgt?LP0R-kWQibOqX%BH{U~X7S9KJ22Dm2s<+Qb`EvkXc~7qFAL@kzU|${VnxOC1}pP%$KV`^R-4QxUTuhA!p+&;^ec>eu}?0`KHmT>mWiQ6ez<hS
v-ehRQWL3Vx}nxJqu>$*%~+iTu}|rBZ~;M-K2O~DoUfD5qIcHBmJ#M`%8WZ|-3i3?Q2^gX!u8H=%t{TDYVLKozo1fKs{~zRmCuK(HDUKnZMTCvl)75nlNIUg
*c7I729JT5gW&}1^TCYt=4OJy)p>al^t(hj_o5M9H;B23H&+mdU$r`&=cxt#cq#ivDW1F5`6af>E4T~~Vb9=rEXkjW?KB%=84sPTuwx&<W?f*Zo1<fp$cx|u
x__>vk|_#-;4{=}JRUM+aRe5aaeQE_seEDKJi^+7rbO{e1tBz@+|UAV#t&@IW0CF$4o(JqyaSw+My?FQ)A&ynz`%><*1zczNRL*&Cgn&_czpGU=zmdl2$W+`
aPt}I=J+4vW{!>BAIbc{9(XrF-j85k#dn)Jy1gEIDrP@dl@5xlRT_*}@(nn7G5*OIUP+u$L$p&e_MN8~aAOe2iBjk4d2<F+3adOygA<>k39G{)5Ub=@3O>jr
T@$=RmP8^71>CQMXN<5#!6u&$fT^lE{aEy60C$pOHnYfm^O>V{gM%7+8QAR9@hmZx7dr2CjxJpYVIFWwi7no;S$!u>i&DR1j<V&Vl8F4tR^xGPgnsou<0Vg*
sL|fovVUC&s`Bo;=d#T=Hv=0P1Jrr^1pA-4dt-xUuFw%0TL0klH8NXdikiKB?qgDH&c9!v195CQDLgTaGE;&INvDTp;g@q)p}iN%mL-CPt_Vwbifkved#&vC
-DAIklS;yHADsw~Pc2m**fZ<zzI+-5JxGXW6Atc0S~kQqp=Li8&&&w$Y%qC1SGSK`rY|l?gjJidwr8WJ3VhiP&TwM;!ty{M7{!Dk^MJ)reUnvVeQ>)WA#yNA
_Yug~r1<Djo)F2=c~kv+D@3WFss<UB3HqUhpOLY-1pwd2JsYixpS!6A`EAqDIK%5oVsn&;SYk)TY!HV~ej5|9!(tX<3xs>F9Aac~*{Va39E&zKWP3LJIj2O>
o$-qJEYa%JOMqylzPt5BE{a+wf(3ZA(&iZ7T@B1w&pqtI+18J5VGMUKMDtON6O9n48k6t$|Nk4!f$H%h*W{ZZe@4I)yE++7{2Tb5ffz)$ELtbjWb!2boba}H
lw}zsOz1wO+V4Lw;3MV+p&{xRr#!;EbSJ%P%zeB<y|~H@eyD#BoNi@^80KL-1kjOy*gymkE6-<(td92(L*e~jky?OAW3^G1dc(xJ2JMV+92>jw%*pb>4ascl
9NCsouJ_Wh#Fx-9YL`i;_I%nm0fJH&kyW4vfNA0(A~Qp~^`G_90p<5^3n&kI=HmI;mbtX*ltJ%)h2pUQGLnH23{sfRfVs&OpLT~frESCHRaP^t4BfT*JZxKZ
iiPtP3opFg*`inGCk089+;Q1-<#$ef{~xW*s*K~t;k=0!W1tjA+K+EEmUZAo({wd0)WCb~$r{<qwtr{UoN+=d+7fkyPq#T#q(@4E0_;nQU#CN_^fZjb071Ij
^pdf6n%jUx%x5<7ViI3guqX>B#L5y!(n<Q}6R!hq^WBj=$%>EqdWyF$&W=>?Xzt=VZNj%!Ma3({NnH{9YElp&A(PAsJTuReNEDfxybGMVHw{Im{+e76NioFA
4Z56~rbA&iq2W1%Nr?((<S-g<T*zdQsNZcdbz71jKQ@hBAuDq;=J?z5%Dy%;#Qz+Z#UWowLk_T(&GZi0u1f|gG)lS@lsIz{f#Z1)T4bBs20nLbiXli4{sT7c
j`}y_iU>uGF2iDr{IiS5)h=T*kf3YiO(q5AP@gc)k{<F>EG0(VehU$KK%_3Ap*^zVww53U3b8rytR+W7!CM-DcyT|(py3Q1MVY~4;Yfe5P@wsVYFAW|bea0`
rOGq>Gp(dt^dAK)-tR5jCfAUj=|()Ixe9zE<ImJaq*TpcpbYDkJT$tb66U7cg_9s`1WBlp9tTe?6h+pPvjXbcPGKbK_-GTSC<|kRD_94^$_$ubq7?{fio}rL
U;2BfOjmraofTMah*b#?WKTYdIy*vyss*sPc2_x42QCayUg5TKRAg6LJyClie{gEES7@~I$`EGVyA9jf)IC3q5iFfCQ94y@uwMWE($f`k+u_8eRnb(dj7jzc
@Cv=whbqY7pD(s;f?%@Z=DXg!Pr4-gh8<^fwzU{3@zkM{fpfbGkgMilVQ~i&U4g7cwFF#~s={;15}W+?_+J;Iwm>Iu{M~XIh62ccdxMS~4S%4N?;3Y_N+kxb
;4Z{H$p_|h#S5rO^6%W3LkBO)^%;K1nyA0)9D`j|D0|DKyQiuo-+6Ro+n<}z)E#yYq4%O;;dWXgsNq^6F9MUvUF$T`TEool#t=plIP$2cJ@ed)t-2|6ks^-n
dlu_>-=yzf1n-)KhI{PVq|r)};U`>FqPw(mZZTLU7F!4vm4;S-gjDA0J5>zOSp)f6ygsy(<07Q^3lD~a4m=}=C)0XPA@@p3ugl1mc-*&CB&5h(xm==Of}dyf
?o42+IIImAtHP}N;~jUJ?<JbIJMBkP?+h_yMtPhLwT!mY0Oa!sYpX+|vZEd%ek0u^!^s^|!xh4;++Jz>2HZR3)Z5CBWeo9GP-dOpl{9nRE?r(S=O)GV0_VtS
{ti0v@~C!a^|}IL+*oO3D%<5K4LBA=n1pKPOTA~l|A6wu0kIRudWU+X+KAOGP*44J*#gsr%#{mjR40nhcY5DgChes&QnAQIqwS|{)&38gLCC9n9!ej3!E`V$
P^E#4s6)@wRL7bX`vC1r7a`+2DiepEtny&t;}zomVQ8q2;aPCfAR=)j2xJKlWm7G0>+ULyL#7vXAUUt%a(7$^7TvU*6us?fS^V3-r5j@tLpUhkiel9`d&?C`
I>2*B>#-sP*Z*^9DdH7Ry(`jIWAr=g&ddRH7GbK`_am)bP*q_jVcXKehLN4P*33$VB;-R|h<s(PtiDz1q}JSMms%3Co5Ce$K&`>g9R?`(Lxr>b{+`BMFl0||
5w3CkS$#hS|LZ5FlhteHZ^h>dYvn`K7a~Le<I1powzOs7tE#;0vJoV8DYzm*b}pqJSO}!*j)l-jx4(|%Rq(<{gestPdd*?(2L!)v<-wYsm?~+FFyzm)(wn#_
B~0n@e-Eavz`~Ovi%REufN&jHZhtb&bdbz}Wi2>t)Mf{x6%y~fgX%XIC}+mO=|r@Z*G*GhQu7jt2~g?w;4W*%Z4)|BrBj*&RF&Lk@(!lwbg`Pfn_9{9rzp~;
fj>yjWBq1P1ZN~Pclx-c?YyQnUUnne_<I>jPTMyfS2J^%hw)#}6cA$mlM<&AxBg}&J4m(n>O)7itehq12FUIkG^c-@qJpTh=H(xBH+BxJD7+_5jR_|@q%7$d
_|y&}C5h3-1cdsTN>17j4kUiH3w+)0{iKSni$&&gi{Fl0_{^iM_-SFwk{2BHIA@BsKQ47~(sqifO!NEFF2H%JWux0o2+KNnBRuC~snk)~Tf`<8ote*`o*y#K
EH{UB`gbYYN+m?x5Dou76|9x9d;5EEM(PT5Dtt(W?2CAdZW?d5s-9X1g{ZdTgtA>(fp;Go%pD9at#29kkwX2n3x~%jq-=r%8T{If3MmKFu$Ck)W$W$N%}2X+
I&1DHAnCnzp9`ZaxqTkio#Q0F6RSW<F4>3mo1Z%(aDfxI#$*skL^#Mu4)jbVE+wgA;_OUmI+n$TQ)*m8soW{~*F`inT%$1ffF{s7)>(SvHE<!qaJ;0(=GZvM
5)&wiR$S-6)h_ombp}ht$s*RZ!X@6jrPDUEglxX0SE7)xfsCPJniv0M52gFgVFj4?5Ct;6H!q6fc8cbuqupl##kx-G6N?dFP|!4?!C%^nN<HOBZ$h6BS5ep^
IilkCjw4&7#i1VVhG3)Z-%7qv51*RD>`c|_4*Zsb4_y-k<t=LS|8O1GPt2d?hy}R2&^k?3uOcQA^qVGPmcDc}EpHe)_;RSNH0FUJw3KkQXIQ0$XZYlVK`r>X
tpsOUohjv9+jC9G2h0n4S=IU`8}x+rMI7$FU;YO39x>&<|JVnd00D4+f@D=y;zf1$f#%4#zBseSbY5H=qPeUSr(i@#cY~yPriZ|zFF(g0ZAS^6>JLe^%U~<+
bqH;yrVgAKOw!-KSp%@es_Na9<W0lRX-k;gqa8~{1BfnZCb~<y%ntCW&6fWtmTZudLkH;SYh98~pPM?Gra}6bE2Cl64m=yFbH;0JV*ho%6)SJ;YM($WH>&Dp
p>9QkG5zBGN7Qj7Sgro&yCc^EfBWF~*ES(@17d^jPEB)fyaHF~A{crZEXM_tawL8O0Z2-LI>Jp?EJRXU63i*yRSD{u^!Lf+7CGUGa!<7jtt}IPYmd=MhK>+0
I+?(VdVXoj_^Gb|4kekOIb5A~zlFWMLW3zeMwfQWR&OO`pl9mKm&}7^2V6n^{YHr*MY=w;IdIgZBXicZK7iVvV;yR;*vpGrN(bp64+K!sBOO@ebm~>tc1na%
#|JP0UM%bbGq_JC(%<+dX0TOQT1?^xVLlIES3zFVQy^fLG*lJ8-eusp`&Ej&bG9>pFBYRV`jy%+7b;S)@`Yz{Y?N~;s(LM_8=<}XTYAnkh0o6dZ>srM!vOm|
ucXeB!z7kOpf}v4{20*2TjH@^<e)!u8S}WFQpwTt1FO4)AL?W=D@8RWVQXl8M~YSNqctuzGQ82*41+}Sw>GwO4S39y{a~UK^}V^A{p5e-ks!Y-i;z*d#daaJ
2(L0_dnyrw$`1L-QBGS<q~DO-7bgl)`j}6@*MDiVV(^%OT+mPxHK}nh>=FFNXa42bCVF)(3u{zoa?H_nl<3&q|9&QvTWnxV35C)lvpqP+T)x<Fymi*bSE&hE
#fr4kGl)$I7XT|uN_*}$R-Wq{aEvm>a4;l}jlCZ*8Ai>V7J(YkneRIvE2blI;vEG<!seQfb)-6~Y-f3o5v#9hAi{8>)C;qC(tQe_{65ud{!Ypi6JO7Gt2s(!
@UR0Lp{S2sjD~c3xoIx;Q~$Dy^pIdnAfp2J?gwd9?SGY~WH%>t%bWRZaq@Zz_{e``YD*HtfEBObNr8niWeK|kV58$@C<MY)hwR1hH{%aEhz7CPa63oG9pS~E
VR5}Ya&_i?+F{57xwd0uq;xSYvz9mOf1SlE7N`j?Z~W%JA;E_ZS8ED>YeX}|MEShE4{`TiGR}o*2tWC7HE-E!WY$Th@m90`@SVSST)UqKw#Wu7>}kr_3y8A)
dyFGqpMWCg48~3TMkg^b&5e^V+^hM|^^LzHd~q<)8svmvk`b=F%Q|iy%A<lb4ne}P65Gvc-~fEM;`L-{`9P79P|Wq5jHw?30ZL0LZ%&Ds?2>u0D9@e=x^O&&
=fURf9};MaAQ2;=the4-*D64DM*?is!sxgZh+PX2zkpRGdEdI|efruPV5?Ffwp2{3a)_$$fTr9{2T^W+OK@O5cabF>I@AUI-O@*xKY5aiNMrQAksF!cmiNnB
Zr|i8t~jHH0N&=d3v||&j>oy(>BKUVkOl^ZmJ%GS#YFDU^CgL|1YyZ+_MG20E&1IuLaVvTS3=z_!y&~uJ*g6YM(<)0%GIN+gxyUFQLvMRW+(CcemanoAAbTj
oRs6_(tRr!A(ixdIqK-k4ma{WF*D~yUG~YPmD^O+;+o{9CeJ!^yQlB`m)5$owRy{R3vMU!X_U~lo-ak|g<T+r9yddHl|wBelF$oLS>A@+ip_&T_Aap=_8-VC
?R90@ClCASJyhD_H<TfwK5SB^k0C(rjD?{1jNjxgf+)K3RS8-ieZdZseVXW(if-I+hfy6uHD*?JKBdS4{g_Z0IsU;+PUBdrhF(x9ROM?%i|K-FVsPJy=jOZy
EP;IUGM86WFfj9ASchzzgbCmd$dS-9Qi$`5h@6n}4lLG4S}~MaFDvbkn_UI6Mqab?X7)-8SDPF-q^Nf$@+>R9?dT1I>F5(npZ+Trz)S;vg*GDGGIC2B-AEpQ
8C?1Tz4}j?#A&&v_XPO}bE$3Mk?3t$T8X@ju2iz;(HpvlW)PsX2beyd=<^Wy@nqK+apQ@8QqszLTh*;-+EJ!$6-wD?^&Svp#QP@01x0U(L5Io>rzo|l-3T%n
AML6^N=COV?>>}CLzlSdqel`*;#lxGsu=c)PoOCyWEr~Uw;UjWn~&Vv7Y4e_l;uB=niX00q+qebSpVpuq&BFY+jA9eDMDwRuUqri#mtdRkA=;jM*5L)A=_T5
EXR02cxoe2sQc-lY959Wn#I<%z5dYnSpPBAk7}YcRpIfoe}OY6K2#CN!gFUgU{n*qMQE3=?J#V(UJ)3TRWDI?GV7lPb;;hK3zr7;8sf=k3jCMiA(oW4;Hi<%
-)PXDry4joMtD4X5mi}ziAS$}L7<k$ND)IBkd2QuhN=dHHCeY$KKrboyGU_Bn<`LU@M_^cn}r^j!#E?zg*ecQ?+*d0HI~VEtIAvcG$7jN8%OT^G^<yw7@S-y
2{5~L;fvOp-Cth-EWXswho<td67YDnG{B7@o!&)=se^!!Rb2i+13A@$uWq;;2m9<FvOdI6K>-sVAvxn+UF*zhzG%9Lm$Ci}rQQG6ZXm@L42?{Fpr{S%3#bd$
MrHfF4|54KR5t>leS<hEb!KCNLiBEcor642Z1u&_G1G^Sx7d|(b)e&mZgV}Nc>6Na!O5Hekkg0lPfA%g1w@i3l1rt_D&vW;{8-!mzJjr2ms<g9|Dce6G6C_%
%wQa4BV?;@YrRq*s7&_XGKsC9g5c?&9z<IxltEY0wOuh?D)_hMAIS&BE#<z}G`3~b>9BcBh!=P&p!*AJ-}@an)zw3$H#mBdLGSYw<bQWTy3I=%fw#4SLc&7-
H+C|XAzNOA4%Yvwx1`Ok9iLxumNdA0!+QmOa)P;_*e`nWHM(UQpyx_Bg|23l0;gSo5Q6+l&?8FKuGgY#XJyOLy16xZ$D@64rcR;)l4Yy405MFXORboKNnAeS
0d4RtmDd!V8p<TDM-g&ta(;^RbQb7>QljS3yNW-KIJi^t*j`c~?_(n5!H$ekHUQ(Yb#T=o5Qfz6=(gxFRi{sbvj>D|am<a{Kd;z_X#aD~B;*CM8w97uvH{SP
_p*pS5B{RF@!e&AMvjdep9#y>Je!QtpsK$E&0Vu%%zjausTG4)&_&j4t<|@LEFzzcg#sM^o1R8Uu)Hknvf^PIR-Jn^PC=msb2R2zZF(5zlYVP}*hT%_n3Z6L
g<M5q>Hyj$XMM~S*+SPZ%ku;gUE!-|bQdr_v`E>|F?JuSuxo?H*)TZJ)+RDeQ(bNjH(yu;UlO3;jO3l995z(d*;)BDn;v0ID)v*3)LLY%?uMs%$sc1O;vI#U
OV<LIL$}ot{{d&#nUAK3Ujs3OE_qZg#*!eoMUu+zFWwM0`OBC^St=FAUTGqsLx05{9iZ#+`<c!g&KA3cx0iXDcMyP1rk`bXZg(_{+97{HeuRrqeGGpW9MVaM
)}vt}YF_}oZ(ubi?8iJFyf3BK>lSRnpygOvtD)@(p6!uO?1*cw6as##JHHC(Dngxz`>;djPzPQ}2>b(!pww)Ad*BUH%>`sz`IO+5zS)Y2+rzF=WxT(+!+JB&
KfOOvfBl&9%t?P+y{Eg3pqJ~{Jlg$b-OCD7z}9MSKYF`r*DkxiAfUt|qdT4Rq@sHjC^h`B$y2nN(Yf|qy_Pz|Ao-*7?^iracH^~$1G>nC?s@`iD75q=tzvgJ
ECrYU(8HDDt%QRsL@&8n!MS+ljezI$KgR5X(cHaz3#c?`@I2beUWa5sPX5YUUeQ9D@*e^=psaM|3Vh{8JRGGAw)#kcOj^$E;qv5r)F}`n?ZE`FW)GTkDE?`x
YTr%I0lp8Vsc(-@_m`B%rd5)K2^HzE#8aHx3)*U|-HT9K9k59;9y%2xgxBJFCO*I~;DZw_d=Y^E!5Ai)+36@EQtg}vyivnAm$!P!J&j#<x-U(*l9@cn6Fx<)
@}#iw1)1SGZyj>LZ#W3(fZpaP&+1I#o-S=YQ9ynW9G}Jm9?JE_94d-ixQl`nqQI0S>{ZMCoPxrnVZSbGe28mPixw(p?Pda+lL#P=PtL`pw-Ap*!)bAQ-EkHo
A?QfcYZvuDawy%i%z+!_0#8MXzowQSjlqT5laDi)#>Vg)lx-)toqTB4^)|U@6!!A_+v^OV+ko_%UBcKR2&5baQtcDg9pq0EB1r+k4our4`mE8HP;mS?*;=!(
_T4{s)KpqejjbjNN$@dx%-_EW+aAgYA?njP*$T;6gTIh{jBkkR&MwfDkaQ&)#!Q&k4FYKG{JYsM^{4fr{n-JLNuQ<0r;>1>)x+h9Ikb-rOoEY~PS4BCcyqA3
l<WOK)~cz&m)$$keG%Y<fGP!v!Vk(%i0`R~!}fDP=TwT9#!7;?Y|*;0p*ruzBG(PKOc|{k9DAH@p=*>Svmo^85K)te0_C$1^a?7>eIoXGXfGWIq#GD>3u^lf
H`uJ>wgx@%SW%;0NlfjF`Ls^PDsKe`t1L;mqtaRQ4ROvHuD74r(Y4)MF7D{n^AEO0PYF>GSR@q%y+iMCY@SJlnnp{-Gq1hL5<vNeLL6|zCBHL5PCaYaExch0
hi{-S<A|#`n98GrvOX+*qvV@R^8EClgD_$(&5NiQsLoB_k)buMMl+(gNV}Ro8zN8(RAbe*LHQw2@Rq!7oU_REGWEW4T8tOo+8ObWgrgtdSBw*pxG;sT-VjRP
!;z)@q9TcnD3DX<6EbL(6El%Agy2mtFry<4N+PPJ=VzPj#Tt@SSw#%Q(WV>y=6I@R24OvgQHnzj&&SA;G2cIvsYA;ake7cGhUC*2@h?x+tEgce{fzhfn37qE
v-IRs5-nH7!#=pY6<921441l50L$h9vy;>!wmSg}JUXMqsW3>|Exd%pF88t(ZWfgDB*Zgpz-^X21iAKU%iv5z?51^l9ZFqvJfk}c8t~RP<zr(4jZ^kdQryn(
NhRMn&nqK_AeCxJ_Yf{q4ZMvM;(DcL-Duqv&cq_Qj3k9Q$z<vs$#<02wWv&t)DY~E4F(%5K;d;*yS%bMT^sg%lcK$62M_FmrA^*NVd&k27tE7s{Cd0Gz@%6^
NA$+<h`9T&Ym2NTzdDUjHJqa<r18vXrbeLQ6(FBVd9YUN8~gZz^T7R9iM~QZuYJiCVw<d$z-}1xo?WPBfd2#0i>o?pEo5zz$4uz&BgL@+Dcx6WhY+{DDeFzq
8JH2)l|!PW>Jx#&=S3Uty(tp{ei?p|`7QgQe9s&RH-R8I8v2R9o2B;i&>5`OL-Vge+Txx#7obR4>tkh#nOIOlL%PFH?TvE=;26tXvnmQ;-->^B2vb-NY(A=o
KxKzYgi+ED?T;rcM7QzcH{9Jd1=a6!d068(%(sTEmf;w(Ss7FevmQGzGg;t~n_|G<T1eUZ;S^)zra+NdqhjQ{$(80aGDYOYy;Jx_&?3>y3*c_!mlyR0;)a*{
4X^J*0Sl{(_47$_*m+A{ji|fri<TMIR%I)*E3N0kRF%ag6Eek&H+6FXr@LQUtGFT$qrzq!P66J4ewN8SUXk1jC%?Ji9*3~*Bm(0b`X&gN$N-7G@dRGOUOaFf
l=a0;65F`{3IE@yy>{-&_gO>2(}sNjyT3JorrBFgS+X6nLFY*<w+xZ_H^TTRo9JbjdxM16j}q;M^|P>P+plS$n?FoQrh;>%@kA?jC+@zt+pj_S?zdfqz(;8a
7q2s=%<Bwq(lTnG(U)&CK=Dhj^+X3d38S_e!6sPP6_W+OmNRYhZboS)v0Biyvc^nk|8uB`8`g;8>0qANL;h}qv?W5`Ehv4`(-NCkZJcT(#xP%%UxG@dq?YZ>
U}48@bk)Mx{UR;nbH~bx8XcOmy&TiP39G+Y`-e<t!%%FtC!p%7sUST`67@zpfr_|DoW>d$f4KHk;&x1GM6woD>Akm8=2g+GHvh(oG05qn_Rg4UHywe$^3pY;
^EfKgJ1NNvVaKSWO7dnn%h!r)*K@<8HpRU9B|jn+T7E>LOsloaZqS@epKSkTDM+00HC<rMCP?o|;>)!v#QKD1Zn{yN*&F==4Sl!HVmEk}KZ33bgEv)Bes~-e
-0blhm3xJ^5cQ#=NDtyn>9ihFjpV7B{=f=fMil6L;sS&fJ22F$prt`xhchr?p{kYC`o{`Jmz(wm<GFeNd4g7w+a5OO(~sMQo;Hf2g1PKvDEyqCvXTE@Csulr
DZb-S*p&PxU{BrCN_aS_kfO>zvFNJkQTX%7O=}HTLWR!(Oa)#K@*slF(CBIDN;{vWg(Jv9?{`xrZN^%<gmmtnrBi-4sES#JK;0{@R{u+1ukvDVx1vR?gdS)V
N;wf*@1mV(e%!7XPkW9t-m(Ieg9pa+>Ot>;3!3LDncjSj{pNn1G-s8I_NgH4L%BDQR<OUnNGh~!o}0oz%9$y<J)=Yu&Qxi>NiYub=eTXsxP!m~&>2#e?Xng9
GA>k)6t`)`EdNAxa9;+)0J(Y0N6SmdS;cd<-g$+Ttc`w#BFAr`Bl0f(Yc-?M_<S9EvEGgpfV)e?X+U!eCbw?mBXbVc&NOom=bi8yqIrXK+=`R^Ao#5&{=4wi
N&ISh_*&@q=2{BE^qoaCv@>lgajQyu(_7M1Rdf1f(Bs)7Jk+VZYSkX`IhJKDZv0jwAzfSDzM~@vtWqJy{QWCN6m0e{uejJ-o+?VEw#|BZXD!&eBw%_|5X!Wn
9;!n2!3FLvTMMQhyP2XRazbyZIHsbn_ldTu(%fFq{!#YC?D$dHT^cUoIx<n3Zrm-3sDAUB7Q245UTF3o!E)WM__>~7!xz}K8G>bjV_p=dx2=PyYVF5}THq?z
6iX7Q<{Y=(Sv~6fPl3#8orH>~*r6y<M%RP=eG{S#{++O1YZ2~U*H|gl%Y!)?8n|<ze)EzS@D2DbZ8%s%egqnB*10C88eimyjDb%pgG*jox(n*;r9{&ns0Pn!
6O2z4naKFDHN6uJn`Feh;rkylqOV6ymLTH>f37j$Hoft+CJ2qZC{8dfQ~ROE4iqrVdd;fbq_T<&)RRFxVK~;*g5{R)m_qh9+AI6a!WI5Uesc;oyBA_15N4L^
;mf~*{CeI-ODm>%#A#s>zaa_y?d78Wtswx5362Aebep>qr3K|RG$cSv-;%%W9eg*L2n4}}bTk!o*scY56zjG+?k}DrpZr6_EH4N)H?pDw$Te0@Lz9H66xn21
gl4guZkVI2>Lc%BTMxke#5X=htNlALCI6I$iUU@w%5gn~X1<Z;j-F%?k|wE5=d9H2o$LS8dxvZiMmG&bpG8FNJ%#i|y+D2z$&x|@pISvNZf?~Pu^BLXqSF96
{#~1!NHXZVs1yt}BI1nuM^)lMoQEuyoRQ1$ND56@hWW`~UqwaRKHm$RvnAb__xONv%#SFgZl(GzTZh?uN|GLRQkmmCAgQyd-%uv_569li?qBf!s)3ly$TFtO
2Uq0xjmn5hmBQpDJKMA1UE}mt59^;UW5RT8ff>1_JRbp(74DE|aRGBDUinz&{OIkiAxPGUjYcq2kuj{1{ZE6DyrAf$l{WJIgM{3r3rAE)JJ`dzI{$DN@Twrs
J>bOQ(@n_C5-`)$8)eO#!QXt;QjEM7QFx-4r(S#v@qjT)pP1R5hb-+1Zd$E8)u(*zUCl(gB3UEIhi{ZYZbWgIc4Pdj@{n{}CU%p&^SDLxwbl(x#d=G!pdmS&
(2f;JvZMF$!zVj9lR19w0(Nb{2AAYbew@PSEn27W!wa8-T7knKp%Sux&Go6BP^b2X5EH9i!1<gcp@`ujV!S)a^H}qF1#ldC5fccPO7+rPhgtQq^m8o2(+CyN
<_PV2y@~#+u@)qsud!OjqL{ZBNf7D(lm(2pVUvpGfn^p^63u8;l{UoT7FZn)8dwpU)J;IC=&R5ABCvWFHd(kc0hdV$)O2`;TySbG=*1yjw&TMv_tfqJTbO)=
+-O{l78#SO+x0-bJQ7j?Hn!sAegAdb+SV!-*|hPXd2#8i{R+$XI60W=c7lcE^-7u!I9%EDDko3OP)pKi-q*)$toUl~VigUXK)17UJ5iNAsVn?Pz?BC9fAUi<
Z`>Xb9$cpWUADy^nT}VpztRQLp^DiYSTx-RAS=Gr?T0qDdxCVVO%Td4W(h0C2C3x{ywu9u<Sj3}PAc9K3s&#5xaP5eS9{4o&7eA)|FGL^Q@&ihK|n2OwlMpX
_i)@cEl@e~N)?HsU+{Hvsb^9>2|H6qa3EJL?m&61F~>8W`FBOPlwp05^y(<0-G<-@5ncXE9zaKo4TPs7G%O-${VXL?+y{OkK^0N>U{12ku9X@MoM=GQXGl0%
esb^ji-=Gf@2=^nBDFXdBu8TrfsX4BAV(*q|Hyuu`bmHn*qXY7DgYRu@3JN!5NIFnZuCX3N%1M2av+cz;29Ev*TVl1(5A@sziBwMI7Q8>gQMk0Edy4<q?x@!
rJ)%hZ|yW%P)G8A1@$#P*j4XGDidu)-g&ayHir)@n2Nj+aqfOI$I{8Ra8{K2eKjtuPVRRB>;uD5JB6=H2yph(jyVIG%d#htM1_JIU&^D@7x067Q8#|_QJb-p
Q2?($s(H=X+wz%Dz-L28*8vtY{nNGH3#6?Y0bWjeZgyijXGHA_0Qi<spi01myGFG1`si+P>;yQo2uwxZqj-jxORBsOBN{M|!oOC$XtCgMMwX+<DOAd{c&~MI
JcU3giyzd6@ae`&uxRETp)$c2&$isD+N9L1Xb_Cp)}7R%-h8*><`S_s_MKYv98Q=xx!&JyoTu*oQg&);;2QBXNt50E5+WBM@a`2+a*Zx1f0zTpM1{CN2_ww!
UBJ0*Pj$8wjgIgKwEKj$`CXZm<t;LjzWV@c1f3Ct9-5k0t|9_k&$30~zbm=Pttyoueezb6daOGjx#K$*gV|83P5BE`asC+v<?+Hc_6K@Uh`X--UT%j_Rv``=
uA++O-4B%OWDh%)DubXt$Uaa<5gUhELZf~&%wp)^4SP~SdN&Tf7L~gsRy^o5=y72*_pAyIyV7d^tEe7d7eEbpHK{Oti$)aQqMNGOg2F?w%;{3}_?2dgF%pUF
5gwSU1xSEhNVma}nj0`LxEshGMj{PIx2D3kP2YvO$O^SE#?{Yi0U15+*4(6;MkRef0I_>$!>SF<h!&jwIDY^~-bSaqr)tuF-&wv+|H$2!;uB+X9~!>cW5`a~
jB)Ka-;o-7P42pVD(JA#Vx%d46gNE1JEC9quNIbJ(~>RQJ&Xv-4EG42q#(hP<f6eN<}q~W3Cae&adP0+5++8Szt<d(rL09I$yd=$Mj5)&(6I=B`l%~XB`bCW
HfqV>eaKcJtdbE9xX@BTPGZfa1Ii=Gra4KVHavo81{=iIJj#6^+x#e1d7G!Vksj@XUtxw}#h0x=%zp=m!QAGeDSR<*YBK?$bTIXjc;pg>wROi$ZZlr+$dGGg
*$4UIDgcSINn<MB){iAB%6_5#1A&=cHJZtg(JwZ<8J#+jG`v%$h@%&*`o&}l@4rdi4nG&Hgj{7V2m6}(mVgA7Rx(bqM<*ZDbFMe?BN(JSO6CUXu)sUx%*-h-
{A76|pjCjhSrUZ8sfmty+=|7#K?w_7#Y)E->?(5+k!Qhp3>OIo$Zfn7@dAM-(x$L2a_d*P53C)0lsm%ZXj?pbQ+;C#h0HqUVU(Ai26$5^A~)NFzpfS95~PI5
mLCiMKovJ=1@Ou4On76yE;2Sq3dkD?xMqE1=CxCN<1#)fmhKHHL$GE@*&{}P*DSkdg-%Gi(h_46@}EKlF>sKqS-)q~9zUR;TZFPs1*e_kYb^+mRt6n=e*_BC
H*V2g?b4N@$sYf*oN}-;*~R>atRO+KF#Ke3o$1xA(qDof6{Quat?KKoJMBWf^BKJm2(n|E<oh8lt5wfH#|SVKEbEF<Vog*F((MP__10>-fRD$e?hR#HpC--}
W5}t}x2xbSRAr}u`U*^Dx=-I*z$|F{Hn`RZ--@<nw<?mllo|RTT}*yQ$dG*e@~Yv^N1Dj6SJE6f-mVwpcN}0jzR`V-_p88GtYJIOY#W|rtVQgIkI*KKlpY;Y
Zf<0eJy`VxZ+&}&;zN!Wy#|6zz4wWo)&vWQP9ho<`XW&An2V?B_1e-W@_W>gxL;z{7{V^%#Fh*M#(pM#lA0eiRgyIg4hgufd?MBNWjAeur*az9m{yempQmTO
NpPrj=LJRi5-B;E*EL&Mx^1x&jxF7A?`{7$inU61x{PU(E)%|%ArkzWf$${Eci;>Tfy@1;>Cl`0!QCzb0v<e@&0kqVMg{_~mmzje0=7%>b+W59Dh`c_!e`=7
_Q2fF*Bd^&9FNb<HHF#b>y!;rxRoy5u+5d^8}S4%N9^Z7EtPcIiLg;|7(!+)LWN*0&tru-u7q&uQh8=q{<obmd6@yfiGkd7^T&~#Cm*j;Yt?FaejTwoAlg+-
HqG*aRe)Is%1D)%B_;n~S8wZbJxy2X;=Z$->L_8~dwZOu8==D@Ia*shnHGgX3FEYLWG@YgoYYa49F1hxRbei!d$1fg_;>R(z&9Ee5Me7rwXIu>iNHQtk5_x-
e&V1*KaqXbKD1Oe2;al-rdf(f9XPQFWy(AFFC$RuuX9pB8sg}3&L<K6pS=l&b)}l?+x=f+mB|>F`mo2N$wjmCMsP>H7u^HA-j?<K$@J^N_vyLYGi)X`f&{);
P!4yB)X<(PlO*_Bt4V8JVu{nZ(@-0oh8^ZI3++ez6r?r#to$?tnQ*jtax;JAM>(mTDEoqPt0Hwni%l!RI!^?DF8H#CVgDi?YNC>DBs)S#>EEgM8c*2r#aZ=s
ZFIFiMVL2$P)9${`fwOHYsb5&%lg}v+!cqjZJs7>$~otWyyr{;`^j~b_JIwmk6yF$y06b=b;Jyts%s#m!a6>Dn{k=kfU0c@5n?6S@J<%$e_7_gs@rnYwbw}o
mh6#}p&Y>*QVghWKJFxnSd1hcnM8*BZLzMVs9M!A5$?M}MQQ{0i4KPs<M`7`nykLQii#vAI8iG_{EMlBa6L-|>s`SkS|ZwwpNHYcz#$vDs-0Asij>Ut8swwY
SC@(%hWN(p#Z@^cCdA;;bTABo(CDVK8FT`Cr0qI#e*rGU&Vb_d(-GUDEEf2%gIAl|L!es0hk`b3xV%|<=eI-uo3go_jZh{h-s|x>4BLg(&(<oRPbWog=yJR9
Ad1w^X&w*b3vN54ph4j4sL=Q!ELDygH}zo0prbr7ywV;=k^1KsGC;&zq9Pq;Zmhb`CKh*hx%OBvFYV#QOINp2EG-}VnId*E-g&HKt8KghqW4;9X`!36d~7Ww
k)V+bo<;TtWd#q;c`{e9K@mz(CJkT4l~ycYF%G;#h((RJhkRYq1w4#rC~|ZajYVWeI8IX`DPK`2uRjy9l%r1?RC`|Z5mEx_;-H+^lCFrepeClT3Z%aNV6HOP
j@`7QNe)}Be=o*&GxqM`Xinb%8PZve#lu^{oh;8~qPH3Ym+bH1LKRk!#r7WKsI?R<1qi+9_`^pQ>4)kdC-=4lz<{%jhGiE>u@F~Nj1#z}9Jx74&0xX2_5xJj
@g+2LVh;LQ<8jQ%&mGeMIkr0wz>ATE-YmjDf3LUIES=6laeAC21!(45b(4!&+K}|=zZeLE7(FJ~>`k&BlBjMKTvd^oI-g=sa{hzjl?11i^p;zSUXM1pKRcL7
^<o8>db5VrOn(3dO@^e1Qr$(8NWOz6qW(tNdJNqTK>vg9cU4BgNA#Kgl!jj1<kWE?)eq_TH@^n)ZU-gDZ48pIX}oKV{QN@jvn+ey&)&Rc3%d8>G}^Jcd0<k<
eM_PRsvVk#=^Su{Eu0wCOi3&)$AA;`Q1bgC)U7h;osBK57A`2V2LDYmfqm5`Y%dGO!;DsyNTO5odzmsfxSh6S=nqTLDH_MGl7*%I`Bjp?qhlL}g?kpXds9=H
WJG02gETiFJULjoE>*sp1bABzsR7zw>r*Uv76cf)!9k#hv|U56bjp6kwYMBzzjoBa(+c{a9fI5k0og23)f?02x9r4aSc~)Jm^!JvGRR?34Fn}atru&;0s{3@
x5yDzsHjB5X<5g2Pa)#7l_sO7MlB?_Nw1d|8~81Y7-U?=b*HjB_!V!t3hEGcm4^#<cUln4qx6kswms3GtT+=Y#<2U#F%_dYc{XvnMxV2A6lP`-by_<397&`y
Z$fW$2hz$3)G@Ms_9@fT{wA~}m36icUpzliSEIZqC1PTC0*S7uz+hYM3eu_PC-4dUOgw-_%T&jpw257GpJ+HhX5TRhGW0v*qkt98yFq~dP~ecwgpzP>K-0EM
T)&bNe-woB$}x*TY&(2}<!<z&s81xcA7W=mb%VOBKO`)?@mg`v6~9m9rO&%28d<?O2Htk!3{YrK!gqe6`t*u^;WfbH`^nA8z6rllqz}MJ*Em=>Ia^B;0i8=s
NhJRO(Jo-?B&5yL$%1A3{q_$a?-1!iKTT~LZ=Lx1rp!~vCkCjG0|Ef#NF+vq%%=oxL%@3_-s2yJKM#2p;*H1;S`qtB_uWRkJ43`_=n(3iOQ-<brsM)_;xoSt
H7PE~)EXUx7jIBKsILE=xiQ@Q6;wQxgz`Q${UkopA|V-A&UlwffKei5{21$b8t8?@RSAWpx$}zZVS7Qne$!~xy~g3Rq=K5%32P{LAilQ(I~g^3@Fbo|O(kEp
-lJ9IKZtPtC{y_jy^h@J=*~<IgjI_t+3Y;JuXE)@4YQ=Zo#?_5-mZ}5a<kr_s=$J)rLmGmgJxiFDkzK|emA45E@akBua)lE@FAp?{j{ezK&9Brf(fg9qK&{~
LHadfYsN}NhU4+pRVCo8n96_MOxdsA((gs&y7W@DU`%6bq!K|Gh&Z=}bhWu6Q32m|b0L$PApe3>xL1{k?|yb+#nYatkI6t-$oY5$7Wuq9K`oQ(0JcdO=Jc-;
w{cZ7_kredKBECMxakx-1Dqf|GAM6@N()lnNAj(pssFmFcaZ&&v=htD5VgxX4iOWaMZnKU5PQ!23Lzsv(Tj$Stlwl`5NQ3X5IGpfVFj}ECA`^9vF3F_pQSTn
V?AM(f&UURjZpDvpvUiUyCIejhv&sC)x-M&e_2%xm!OtoU4GwpJqBMfO-*}C_I>74rL^?iS!iee315|ExAc6W`40F#qspGo%-1XBph}pCxLQ$m9&P`?xCG7G
+0dfkUVs0HjwCFq0!6qKWdo<B99;kVh}ORO-r|~(0{X?oP-59g#15Z6rDKT_?)$;FL2n>{BI>X5#rYlxAy6vKGYm(gYp+TjahlmzLxz>IQ*Aax>rUR@HWJGH
yFe=tX5EghoSR9Op)1s|amSXtj%QxkG2O;G(mS9QA%1>y(?V^Ja2xz@`>ZP#meJA^oJMQRrWEswwvJZc-`XK3Oc%bwikq@D!drHf;|Z=>!sf6xU`d4?PZYIu
E|rvmTc5N{(f<}xAh0#L5T_e~ldc)PWMnQXKnxxX*&_P6p{=@5A{V202~RMh1-Y85fSqouk@pxd*>heHVf54Bp7_>qlsg?zzqhl6eV@iT`gFyMdp%tZsQv1o
e2vxwM?31I@!w5wj0LPpJshbH=-#Az&$&HlI@hT*GWK1LkA3ReZX;9TOJw0Sbd#b#XvOSmb5jONdN)%x440ypJV`3#E{^tVQahk?k_d&LQ3`Oi%l%_BCZwLt
FO|4{)S4m|z78G6{-UMYxka<6x(P^zYCrK%-#Yq65@qif(lSBM#lCqaPFP3>Zn1FZvVSN)MkM_&TBY@udc?qg$66GKQw16pb}w9Bm~1vVB%4ymkb57;ZsrA+
COD!l>5(i^{(c1hCO}O2yb3#nk7%kAzAFg|a>u?g8<AxYSZ}GO;KSgBjxqn<i8X?2frn$W<xrt_9Jn|wI)k@zuN0a(f)x^vf5&u|eB}xbf`|QNNBsqGC1A|V
Rd{YTNs}&@Y>JE?IN}r48`jS2fGs?!0hEaWEvsv36B`Yb!NAQjl;i4anPoq|xVaW9H2JB^O`8@{<ju0|_q{~Za#GG)wI4>e-kLTiSUvufh+tVA%Y}J<7pNB5
#$497LM+gsxXG4uh@6|(UGRNr%(q6-At|!Zdk*xWFZIKs+olJnN23R$?lYWNEP33PaH)>XqNtEqe@)@gM#v36fzwiH&1@WfvMc%X1P|~)<ABw@JRtx9tb9|W
1yKgwW7%Hg#UqK>NUbNGwv`gaXlofZv?MlZ)b_$#vs2p6jeY(Vi$~MAlZwmxSc99jnlG@GLKXF-qhLmV_V$oqimIl55tZ=cf2pQ7{cvp=?gV`!;Fb$(v{>BS
KigtWRbd!JTW=h!Z@JiUvU}A3t{G}>?d`Mps?kFXg=u8ME0Bw4JrT*dzbf1xH~z4fWJ90=k|x|_!3NM@)#37~wj!jeZj=ucJX?1~;}8s;SqX^=kfeWFQOZ^Y
y`ssMRm2jNja&UQjd+^pF%|;qJrOkbz`*!z-PPg7cc_-+(8<@IMZZ&TW;AL%&_o|oVwp?Vs|E62)PJniKnW))WiA?y17rRJz|;p}?b}$)_#QSAbX=h;PSA!*
zW{3;tK#y#0p<otV3p>t!fdf8Zi1ylU(Kw{+V005vIfl4<vfUUzb)KaH&~4y(VlHA@3WKY^p~G&ir8b_eS0fTWTxu3=A^{Qw803(Sube$^=sv3c=ed$&!tY|
10LQtI&q{oeepYPM<YD=m+cc_rEjY;2<Q-&`XJT;65YTkzN4>0_%DW6i^l6`EjZNd57tb|B-wREkH^zNZj!gaJU8jd#tK7!&6ssoJ3L7AO;rB0Ci?#Pj{lGi
V^7C#h#?wUS1wd%24oULjzN6Ku4x#Mv+d_N22%%yB*5d8IG;>II$DLf?G0u|Z~|%;hKCZ7XN9JC=N{WS8u2oZ@CXcvn(ra`sHuwdA&(ABX-u2VQ+WKLS?v2B
9ijlYd&C`5xp79w8^PAjKiNK$TI(~~2|<==BhXK0bCWw_B^hFSpUd@ET4)pu-%;DLwkjKp%uq#Jf=)b}W9eUMfn401Pe9Gla6qGosO;b+yr$bV6Rlj}{1Cf`
DTq$i0ukZ0i$K4!UO7A!{>xw%$pX3?ov=lf{Jg~Z04auR_(~-5c>TQ5GBgMM|Ngh3J^%)(K+zv9@4>P9Fd3R_QcrX&M%gS8iEEogTh*(>V$XtQl1$i$sAiho
MFkv#?&O`5I~K?S6d%&P4sl9~%_;3{o^Dg&y8$u6wFH?9UxMfRiK*ju{RHAAj7zM%+?jteHRu>H{OJZ|`Zrx80E@bMqIr--|H#+l+uZ$D<{MH^!7;w_%lmDI
Mysr9VMA<a-D2(>)sxEi;wip0!xyTA|FR91PXomgF?$o73{%-=1y<{V@H}ssKL5lb18iL}*II_%32nSB#JNz@72pzmgUCBU=HTi4ingS<I*>|if_leZm1niS
Vtutz=ko=rX>TPDnPe>Wy~_HoADs-{-15IvONvn34%gDeSZpoGJ2)biZ{6{KMpzxa#p--~CxGs=m~55E9d1SPkVr8VQ7l;}vBk0e;@NZzC|GzCmIikC;aLlI
!V{*cu{cTQX9p=>9}>5miveNUP0VM_NU*RHoj~U;Zk!B?GY}2Aot5T&h!e|DM!2QVsoLQ#W)HCY2<MwPRgULZp}9$UTBS;=pY>BDUP!T%=F5~`c%QFhEkA-2
Ra>^gqy*h%-1Rf~tn7NvD(A@GS_P{JseqQ;UAHpO{PtD0f&k7{`w+>Cvm$OwK1_#_lrDr=&2#&uF8rLALwC>&c%Y{jj}R@j!LlS3D(0%ln#!^3f#=V^3as-m
#iyvKMe{>7=fi#<`zo?eeo8_ci?#$BMsa^pt4eev>_99w?k8OIg69Li=uuOruzB2Pn`-T4V66+P;`nqBUjyij70Ef(VOggMdm|-tD7oEl7$erTOYnXrH79UG
a_%54g~%ndiO1g0?2@`RZ7dgwtypN3**hkDLNb(aqY^zWJ7wo<n=Tj1EbU?_22lFHDMi{&gr13iGQyt_lhP>gV?C_~X*k8G;gCR@fdUaIA;+ToB*Urue@!86
Spv8qwII90nMS+5tYE1o8-{&0)v?-N5h}-JAj$aH>ucd#t*W0amKW9VP~Fs|(6dYPksX<-z(=PMEk`f16Sfb`0szB>?DGwVcFX1Nz8(wYx(5YGM9lU9UY*b<
8?k+3=2<(NI%*qiS!h;odx>Xr5ma)HPx?ZXEmtGDMC*=xwY3E+48bwjL@JJqc{wqS<|fHA9c)H*veR^~3&_3Z1l-AfVqp|z@A?G(?rhwTGFSw8U>VH)rKeLr
MoWc5@YwE`$}~mCe<ZJEKfIS+MVukQq%T(B;ht@D!|6WX`SMI)pb2!-eDuS?e7f_IJ1i;pa9Tm^XdoGlcWAPIE1kTFH&U@A342lHFl*rN85{qVVNu2y55IOn
Ef!&iLkj{KW;8=4kD5eH;KJ&QGFD9-p6k8GQ__*DJvdaU(8n*V<^4s{u!(9Rh;|O!ofWL>7}e2K5XQFepZ>^3zXb?yG1tTUGN+CL4h3`>tFd2B&#c0GtdKV!
4g`2-K!0UFf2U1KyxR%Ituh{!kk<m``J(dT>%P>QpsoYeC7lu_n!c3G^eovkkzw1$zY#+>sMA|XTaAa&vz_bDD)}zq1s0lBWs<EbLZvensU)OJst`rE@{V+W
UOZEyQ|^+1n-8RoP0sq2Csr;W#;QMB=cZAjdufEmS^(shp%H0asJIo{+c!kQBG-_>yr2wQ$H%`s@6FuteEU0W8vxtygXuv<R%BwvA7gjhFFJy>_&9^!!V++M
3R$StCv;%KNv_mL%su|T>M?9c<NNDp|5@<eoP(oXvCvOA;pd_sCnIK=MO~JH7ihjj4FjU)#idrUZLrtdLPJGA#|}G+9zJVGTg0#i=)#S>6KQWG#!H~{@Un}2
dFfU&(li;o94)FC3ArVWB2zi7KAUgX>fc636&%9&ON#*^aa=4SRGI(0P<!G(chOO;oz@UkJE9S98=x=8$A*!`%9s-7El)uN$|X55WQPr2zLr``++4{qLBTFG
CY2y?KH;LmB2tu)eLg8u>MoYC+W>r8I+pK2;2*}4N2$oQ7Q8XDY@^}??ZO$qG1yJLmM(h&X_07VN@l>nx89A$v956ju})2b$c2K;Iv!+aI7nm5)LhD8K|v-Y
+iUNxV$geUP-1frDxH_8fhfNv0lp38Q=P?VG+$<lr_vEi5~Tn9*#D(_^f*lkqiTf5<#av^N;5$d^B;tH?<O*FZbx6ucpMX!!_FhmQrMiSHbLHI)kU{v*r?wA
cxvjwW>xqL75~l7tAT$0^VLfL_Q_PeCR9hbeHn#E^ZapjAP44~CD#TFtA2*l7q`r9)QPEvOLJany!4Rdn=BJ$8a->8HP()>t4VzidOJ=+yTm>HT_(LGi4iaT
5vP7;UW_D(pBjVJ>W{M~4wjxQotdr*-}0QSa+Lcqtr*|ra|}rVGJgCVEbc=l3OLQ5p_IeLOWxg-n)tZoc&iVC4t<G?uJ3)Ki5D_tx9p#91vIz8*56(1A=atb
LsWl{Sd@20aq1R~)lJL~4k=9~kK9DQqMsg5xf*e3xq!N7QG>-?Q&(FFbDf@{cBLS!t;^36n6CJH=lV$`jnCrV5-e{wU)_Dj`qDjpd)wGh(Genh0fm4+&}_lq
^vbO&Pkzt7p%v9GhgM6?sn+2;n!(r120fsjZ(ENL^Z82TsX%$51tVeB2z4>%aDdi~j1P>-s1Vz<!iueCN=bWA5ab3b3M$KqVF&c<<9OFQMC;LDF=D+%&EIPv
TX!*+uch5USxIqbDquUSU#|g-MgY-avaq?uwX2@$2`O9u1PFy46g84};Zw5-tOjAP6+Y7#;6~bh#hTfsx-2ydN8=zYqZ8}5+{61hiKmj6O+9!yY)An6W~8%V
D4Z5-ec+BW&YkHhSc4ZZlYr*9imLALol=tjAfLf_TxnC-O505x(NlLu&wYk|caXY<r_56uz8BVHe1hg9k<%Rg^h$4F*J11kL2)|v6Jj^x`Ow0UlchXDVDK<)
sJU>OHLM0$k%rRoEmR`iBhYQnx?N!!k`EvoZRHy-6op@Lfx)Z`ZN?7jHD5CD=AIWk>o=RofhG7ueycgd+U!5ZhsGI4f`^DH4-rVZP-Z{fmIzNIDUZT!EpGSB
X=ipZrNu88eI!RyYam4~v0zxo2&T?CLMBB&>Ln?gP#IkH4G-;^5KEtmOB39KlHFRb=BE%^^YXCeE?DkAw>vqWJU!@08MgG?LBnF+dOF)(*>-KQ)Rye9-P7--
6%R;n<jsyDM;9+=ux!UxL-RJo9W=5?aHn9MD+Pzaj1ByTt4v&@sS!)9&9iSGS<aE8J?VLE#yKoPz=d*F3&~K1-ojo|Mg<2UDnm7X37aJ2RI*e$B4yhP$)V&T
#jcOK>cC?z(B(?tJv0`^@{2N)3x~=`hb3rYpRUB5pi9WwXyILYblYlMTVjCKsE7QaHb}-{4GKh-|2ID|$T?$rfUz+jU7rmgf#6la61rjxMjfU7x`kyKD~4ve
{W`&TTg0<Z7_BZ3iLnjWh9{q$$J<z*`+`lPPP%!YzQZvvjiH+~{`^aAeZb?sS>T?liQa{Ah2fLpg6tSIZq*1RF=lvk=@rdE#&(zTKWSdaAt*XJRO~LjV=e8{
T_i5&g1v`dSu84?eJ}&G@*XQ8*w!uXOn`<SL$nxhYB$#hWOjZCl#W8}Rz-y{pIXI_=&5=-{#U1{r^`2eE{KJo=(H7)#`-Xqn5&9N+lXnBK5v8ejnQB?y?)zY
j|m8O>qjHOs{@)`5XX=y3r1UsiCeVM4Y^RpEOgIRp{wsWPeH!>Ir)x1tj*7Cq=CJ7+KYvzuCO9w;WM%+_1uTOwsyQ<vm;qA7Z$O54!vM(>AI9N__J$RS5bc>
^&nlW(`k;TEkW?YcUcW;5-EIIDyDEg3h|<Ol0jghh;Z^o7D2_A^M-5uFXXa&=-nhY+ye)`Lr7x*A3PdKh|>ZgykWotPq=j*)icFMSN}tt=EvQUw(C9Jxo~TI
6AKfSi}((jVHv*X;>?a`5aC%wVC~>LAtW*g85Zns#3W<Bl-PY0K_EQh{bjp~WR3|h&Oom+gPYS21AzOj!-A%1I)P;1{^VI)=6jfEw7BSBM*(CZC_!(|kkNcc
YA*~7`htXZW9xmg3!nOvywgO8RMG7OtdSh$uRF-#r>Gn)X1i^s`UzIB6OLuN;86dHC_nonXYVSx{55EE!-zL!Lm&Bc2m1W~lOf3&<GdNl16~*E@MkYyu)8&o
p&=bI<OzIW)|_%<)>iJ<$<CA*9<<wmZ*@b<?ydaYg83%VwyeOp(u?w<TA>WgH?e9gfYph40YcB_{-EI-K*5_hpr?2s%hKXM>ac<;IH`S(fLIO!OTta^#~B(4
vMxM}I$lz{QUpuTBP?NP5C~VlN!7@wF4QOmuM|lW-tx`t_cXPth|meFy^4FK?au<Uk<4)4Y;69eHc$#x&7^}UX3Xqy4}&Hdm+Feo=;UR~xj65Kk7Q#5;RfhY
ArU!x0FZ=wlz%jjt4*mF?kp&ttNG^v@4|n&`~)1^sbHWiQ&bd=8Y^z#?VN~I3u;KjmhQ-0{&a8LFs`JC<v(;vdJwIQlE9W!QXBSZ1HyvcqTucL@FY-bVFYB6
e0EZ(3tMvp%qis>iEKCraY3w_&Fky+!xo4RMc4OIoqJ2;*qC|u$?jkC!wl|QWPDQRE@Tk_a+MvfH3i5jwR4o^q+qVkcF%Fu$jtrc2ycR5Ag~Lb#|#%(tD{;w
Ee;dkEW7*1%ZKmW1gdLy7oj#{!QCp$)sbAOZ5ml5>oOgj9VyKJxZj0YvaYHMFh$|`$w(yB2fl0(qh?NE)B^JIO1js>g`N_H4r;tdaK)a1V|jx&4#Pk$b)YV#
p{B+eJDdv$)iPU>9pj=N#ZVK54&GPSniE|$00R_~?kS{96x0yqFWKr;{V9xZu|Ac7#HUmiKr3C6nc*5*N{DXfjpL+ui>MXH4ax?eU>K*?t-UUeZ^0aqBNfe-
YqQ*PJ}Z33KF~M``G_fVQy(K)YW?_#Cv|;i@3~+EticVWSjFdk_vQqLh7QikyF#7`HXV?i1K$`U?F&|BMiO9Bvj!8HGFWPifaw6W4SNdhWM+h+AJuVQ`XE6@
R#E>A{8BjBZ1mC1c&x;L0I9jDINeSYTF)e$?*Zi(L91$a@3}q2Z_iq=*s3BkZHPnP{LtRY_E=_t$XDD$kP8eq#>S;-_l>oX2TRk=3b9izw#A?3L|xVW3r-Q|
U&MTSy60Z^ne*>HSaeR;u96Js7$rpc2QAW6^9vOKUXs53;Kb}MlW5EgB=Er!*&3Ey1A=$sM9T*d_5#^Gux)a_&FmLC2tN^mk9T*{)!PmC9E4S`;&PkbxSID8
(5I0qWC?$Q`uTPdT4wvI3l82A9#X{W3g3r|fhxd4{h2q=bGf=QRIs2fyxh_R><lrOv8?c}Sek(3dnM$K|Gp^|MFgIbA$~qA%=z2GXhvnLF6jhAlXwsuFhrpm
b~ZsJGiQS`uzwcy>C{6gHnejd$W;`jVy$PZ0{bwU!SaoW^9g1=ly@`)a2mVi2ungbfh)BD`{UuvG87jzozhCLWS{gx%z;0lR^OCLM4`wP#I+(DlbYk<6lu->
#qIDEK+j8g7X%?vsN{9w8GGse)B;SM%LcB#83nHBGR+b&CjH5q)Jo{`&MSxKlKA!mR*<YcgqEDn&TyxyTP@7ImoS~y!F-oKYo5&yIf5hd-Z(wL{{}3TdN#-L
x}Ii8l<Z{1WwLHuy};a<eNlphYyRx@msOrrBDOsPo}3-~>+5X6oWwpHfAhLSRxiS|IbD}gAoI@Aklh(}ibzwHm7(EWbpY_h-kECVG}*-fZa9AoRww7=HUV}5
fZPPqxai&H;m1xBc!xp{eY}L}tj7zzLVI`Z>ATB|`Ev!As@c*smt%IDaia~fp<i@z=I-}VJt%2)xLOrZX)URz#SvrC%I0MSA>KA1Wi;5&0t8n8Nsfm0yOb#R
G0s9?>5AgU<{vrSzCPH_r1YKG<?twRFSR>8pZ%i6;t*z(`8KfQG9X#0+9Zpl_p-biePt{JAZQpj?oO6~c2<vx*wFQMom4pbSERWfw!UPrESmJ?;4sTY#-T>>
4i315LzBzmexhtYC^u9Y=f`;w2%IVFmOalV7Cs5Tff5`vIRL67t+RmU=E*ijDhMkBFG!R`%Q3Rkurm|?=3c{yDOt^BT^~1;z~2?MPjUZRqmsd+qj&2rz8>Db
<bdqg%ytpE$<mw`Klw7b!e~by3RRqhnvd3@%<wOB!(IgGhrppjU*w%^JkR&x{+@q1Hp7jrv-AyxuTPRft)d0ia%5&3%cmmWtp4RQ#t$YGOrs~qtU$V5Vjn(q
x!9|}Cx+Y5Ixi~>wXytLuS^H7my?ef-|z*tWAT8CA{}?i?y5QP&-{aCM(X1C{9hw@E}Nd!J2eHNW8V_eCgCKU>7PRt3Jz~tKpda*Ouh+spdCmOh>A`W<UGDf
`;|(^*DZA1DH3*}oCsfQ$2H#X%X|n*81ea!H@a5Kr;xL)$3#k2+j>q6Qeh`VQb0BM*-EFvLu$)9<FcP0_%)4$)pA{BM~8lG_9(Fad$=pKxuPT^LQCe6A&r%A
CAmS67WwpdJkbX)UrZZ&&NnRQ8{L&Rcu~&nehEW<c!5+F$~penJlkX*!~#UAnitZlq!CUi`lYg~<&>#FnRH;m_OHbmyM|9q`N7{B)~d8-K<`2S;?Vch$fUoH
q7IvK+GB$qZvl*=9j~NyO1t09A(lGUb#GzjkAgC4u;wY`7a}Z$8DX#*v>$s%>pPr2watrM7IXr@&@VViCjuPdG9nq0RQaj5C@kv}22wu#EZULr+B@f6?1ENt
6{ua<m8*iDqas@TIxt9y`y5m)U|?|^ntG<Z{+7ZKYL&+7BS1M_^=!+7lWn{DRZ`(DyeqGfI~V1J9D#Ql?iYwXEsdX_B*wX(YBWLLj{szW?{{Z)=T^7+UAndc
`h%y+Uem~&b<#!S_Th`Lf`M?#F`mRzUb53)=LRF_9O`bWZo6KI?-ZY^pan-qd(dw8x;P2Z!y)6~^=lCfAduQJ3hAP1m7QA#2>Squ!@*RB3-7&@wvCi$OJa%?
EwUD;Gg=_`UgU*}P7(++y-Vt)?*f{g`QxZMNJ?AmYR`3QC=L9j!Zw%(=V%e`(F4?(qJTwfc(5Yw*{8?I#?Q9^(r^O3DD140pCKE!A1=V;&22ntD9)I$5CxiJ
b0X$gLwfC(1O0j#C|nkNj9};5yBpOcdA@wj6^ejUm_3A^Pu5lq{L%~0aCudP<=FS@LAECe4(X}u+!ZO_;$aujz%{yX_A-l!SamkNLM90d_phkCClZS=)khnE
P;xLUByDQ{Ed>C^@0Qe7Pm0^T{YBuvEGEJ(0&`Dw=4iuzCek$uDNppQ94h(2fWq=zGiUFeSoC=M=>CwAhm9gmb%$LZOc;6XC5o)AiW{&>-TYD{dG062_2BS!
9RCD?8%MNrC&F^P+ZAEXF66t|d)o3{Dfd6MBmdt)r)}J`bUfT#Zx1-+EE|we6a*-Lcn&`5-!v1JikeHiP7T}#fULie*I{*)OuNBE$%~lqcnaz{u+ZN}Gd!Gc
5|m-7o4;`^xmVH@?{Nb7d{Hw2H~mqi@1nuiLO(9*pB8qZpbT1XCMIF6AfFQaxA_h3tBg&k`HOT{c8q0P`5rE}{$srlch;QPoHHv^k?n0N8@xQX5dI}{N-464
kJ}7;U`p*RnrCVulk~7m6X-)V+c-3sGoh+Ty~UC`uSgv$*#9Jk2`ldM!j>fiE?eWHdnCGgT<0s?qjWexfySPPSd$L#*|yquV&OxS`o-x{d)UR&Qb)J#bxjO{
<vw%AmaG92zQcXl3bCG`3H2*xg$f0lmf>*w0*(dU!k(&qpAm~JBS5Q~XB5;;61~!p?Za<2R;P#y6dEMtjF_Xwt|(-_BXE;_0(NhBKCSfMWi2ACvn0l!rB3j1
^veQpdK4va*z93bpV1O0Vv&oO@+WmU+<3PmPw=dBICK(g(S~TGWCN~Iq@B}Mw3D)e)hcIS^jq;$J`>rsoi1<$s=k3BXbC1>&}o;f_>l}d?{><W{qq@4b`c8x
Ki*)pH?&@w$GFX4HB74d?-XjPa9mzP8H865$&xeSt$$;}U;8Brk@8!K8lavg@`1V`!Fug-JbHDdxZ%U2$?mtA!hGK$7(n4X9Yoq{){+>mL;M=@7@JxszA{zN
QTP)#dl2XAAFpuFTL)MWYAU<kyxw63<Z^2Yo&c^c`nZh}PvRBT4Fh79gN~n@DNq4C=I9JHmFDixZdq-2F^-9oyidLU4~A3Yzj3*qEu4AS&*H8Kt6uKb!QOa}
OwJZ`YK06Vj2VuO{?v?Xh|vBE56lj4S(6@&i!|k*L)2t=ZVusU#Ax~=QKu#3UMZ(mAW3aA0pPlH7#~~}x##ox_344z5>X38&MIsTpSMnpAHe?)+ssZy@M?zw
uj@6fP>}XW9WZ^$#}#U={#&)4ixU$|6G$sf5<E{S_!`QOo8xD#qi_4Y6%`By=~{?uJH$|QoGG<uC6Stzyt|*ZDlhSTQ(Q`+d7)R5U=Tg!5V=cfXo)UK2q0)3
>KfF+ES0^9BWiQhwb>fKF;EQ&0JY|guYm#9ldN9d=!&3v9sXh{AkjNMu|G6tZ(_PK+@StAV;<L;!%&z#Z95#-yIVZ7n`@+0D?GFjOi?N69LEKA|H~J6^h=3X
sSjJ;6oVUMQG^0tT$c(*bf{GuiyTi~1qSxl>6vDVGH|W1?zaCG_zpTOef>6(y?~zjts462d{;!=8n~t|W(d@Ww;8}T1k&7NtP1E;(T&J{QDXBdhOb?)ISwUl
Hu7fXaKa|of=GEW*F$6%0(kkS%ybsf`%$IL)Li*_Ji^Uv`!_$TXExRUpeeKDQFm|6M*{3cLxz=MD4QplCUdROQ<^bYjml<wsYD~uw`2H;x2T#=3)!pg3nYaS
I*^6~?&x40BrXv8&~c7?4xJBIF<&@JDS6K!>m0O>Bss`1n{)C8?|SN`r1rFww!v4}83{0whDKyI_MV=@ZCj;+#qJ4NJt92{m6g4$HG4>)T<g-<Uc+-ahnkJG
D#gON?ulgm{Hh~I_k3hRVlke$P6riGB}wHUm?`33>@C`rOKgC1XO91WgjX{C+-YF5Cb1s7`4fy>5a3--;CV>V1gW3Cxqt#g`O$GwZTjM}Nx)0xc%8Rra&GY|
vf+cKZtg<tl95aAouax3v`xgd(vd%6DYf^HmVO8OP}(Q3TExigVg>>+X%0dU3CV&1lLL}kY3`-t;a?!rA;k9&0@3q~COwzSJbWT)jT@P2c{DF~Bu!tPV@@)p
5Lu|UdDWW>&#vK-ecL!fG{UhL2_FSub<?M<uV~8Hp!|4sam7}u0g56ZQ=l+TV2|i^BBLE+s}{1>6_K4Yh(<Etk*<5o`4SGGZ%QYuqqXIU$};F$GR1<>7Hnp4
YL&3oUh!w?hlyKmQi6Zb$<qL2999ey5?4?wJ%fRu|Djk-d$0Z8G}=kd$SjEiSMiOV3Q+qf^i^Ame3o{oVJgS5+G4`P0Vw1b6j<(4>v>7BN|V(oBXi<Wne6vb
VURAw1iUa*;gW|V95lFX8<LrmpDS<rTE+N!Le)^hG&6PY^}z(AJLpK5VSSrv$Oqkg{vo|FD>yAn_{TzL<uphWRn^yla1$>zipYq8zUoNv1lw!=mY2tsm)#BK
%VP~WfgcKC&=TvV&u+%a6P7{tJ8s4Q);sCua>K{c<>C_$4Nw1hq8waJEp5loj$4fY0mr7CF1tm}apLmr<WTy%*S#FZ!YH2iSRdnVk6PrRT$|_U<uT<K_EG)?
5DPj<T%EJhX*t*_aM>QPVgNA?_=Ur>5Bpm%gF>xiGo$W4GVkIPeB>tcRKV`Q^7$_o6@}?RDIfwJb}->z8%f4^XRUaDhu?Cz8@p8wxa=W_JI$0hHNBug_@`kg
;)uv1m0^5wtmqRHGH%AkmogRcsE&FjnzoQopc(S)#791}C>?Zi;Ym^-t=r!NesJTJBS3xY<=s0La;52$_6;#W(_#vvzPw79@sr-yWHu@jN<Ba#ig!UgyQm~1
BgM5b&?t*8f=~44i(}2)%DQi!jCsHIrV)rCY8Mz-5-VVhEa54^I(U#`?<OQV3r<!%&75%z7*_F{5+?kqdYy&Rs`7RU7Ryp*4#!>44n-*Cb3_XQ{XD1u62=!S
L@wAkHHW8jO(qshus^$1OQOoa&?N?~*P(p(^Ha|gs@nDq0#5pA*Na&aFUj6l+QNNkV(ri-f~_7?`%O_M>DT!WPFM`IA8y>DWLyOS*|}9t7TP7sr%&@zpPuLs
;NLt_#cVfUQ;IbAklEFLwCrF`+{5ltP?a7`;Tz$nM)!qAB>auC(#a<`1(fLRiotit$AiGqUUj553eGtVAQWn68W>+L!L_`mWc?YWuyC>nos?bQT8Br_dt-ww
5ehFcK6P8lg1l!vQ;fjhK}njU6)8U#>xe?&DOdu3Ju`}d96D{n4l}2%32n^%MEcGu4eb%<^D?#jfQ!ymB+8=t5X^KOZ?x<4XB++CBbg1|IYA%Hw<{<&a#VS#
=t#5LJ)*PZzn6FIC}esnk+Xn91<_x>KO-cH->GmNHLI46ZS_*<C7wMZmoHK@ZKvhFmd$kw1oa3jy9%=Vt|tdqLT#cqG&A(h4l_#spsEfZ6Obz}(oHxri!?i_
=e$d8m_23H;d4Fv<H#aR0M<UrgAKS*yHu&HAF52Tp}0uiTC~GIk(_x&?Wl4M-QxS;i#<!~oc`EIDv|8*9{9_{w6pqWxa*bJZA&c(J_^YN=t-B-!8vv8mJfz@
zk}7b^?rN)RSFjj52tU}^kwEfwUsHjQ#Hu&RV|*Jfl&FQI8#C(9W`aBLrtuo?!~-AQiscmMw7lz<}?$nad0G2L>Nc5a=UIr%34l5YXotBZ`56jpJBVu+%&gJ
y490^B&#v|!a)MRS>VNGe%|T#cSd1zsow?_3eySi6zE|--s%T;9s5zvV+Z#v;``x4-%Ft_Yo}yJJo&yaVXG0{;AekW=vN)*`h%!DhzroR<JUqdiLUTBLmEt|
og`f`z#Sb3>3NH!hkGQd5c%zd6mm9HJF!W5&$5SbAJ%{*6Ar&>1fx-OiQ>>h*_?C*pi+<2W!O@l4b$@Ca=zH?_Qll$uq^>rQ4^!e_1+pshi3>;goRWG&ES);
NwO%_b`8|9TTL@h-x7ab_T^OCTf0>(t~if$t$|p7s>ds7MfGWKB@W!@GAONbh>fn)mQyJ+&Kh%pTyb+uV{>9`f_;CniJ}-H1n5X@Cem@1n&zbml@@(OM&66q
eCV6l1PR&zs~NtX1DetoOGRB_%<k~}Zj9F$V#P}Dt2$0G<8*=JZL!k636@ckS@uR5Z4-EyrX>JOE&sg=vmI8;0Vd0e>Fte2C*h^d5i;FZ&Y#!yr_;1*=F&+^
XFUq6i5>xPF2W39cfB%>N_&xuE*>6Z&W22^mRZY!)Vrv8z5;X_v|I*nG+S8s1L=TC2sFi~@86j(!lnb;k}MDu&d%8eq!QTPc%3&vk!+<jp>oeJ6w~|gY7m+`
sakcv9^149Y712h;w{*kQbhKIqK_K2ovZCksl$f~aER)_95GPb2yk*_bEsG!7lhm)8_1c$1lSm;#S@5^4m@&Pva>S(Xw%z3oWw5Ji2k&0@G7Kc=0?Fb?J<-U
lmft5!6jA>n@cLlE<J)|`$Szdh>tqtxP_%G%JPv=<=Qc1=4^-WyV<F!OAvad!$xR+MDTY(?hVFI_T%3#LZLVHgV^C6^8*St+Vl?EC)J#$m+RmVX77A}QytqY
RQ~WTgxd{nny3}kM-~!OeQQnGP=?#SQ6~oLhj<fJYsUa$hawFixH4Nb5p&LU{5rGc+mck+&66kg(O8W2K|`bu!UIq%=Hbzjpo9vPt}>(N$<{0lZ)gX!DG=@@
F*sGG_lhPa_euKjtxf+c!qXsT-Zc$3%0sbnT61HF9|Q}=WnpEUeek(wIriI1O#D%sG()ucJ`Pm0mlfmlv31U&aR&(&m$+AAkc>0aBHWC;RU(M;Z)im8&MY$~
RqV0L@ezq|VOQnV(F!iR6mAb_ELCnE%23Je>In(e=ecQEv*$?)J)V6>p#*AsGL4NHPCR1~<tmIU(kSqYHA2jj#El0WYVov~bv=1e2O?o$5cLKbYUIF~!56}@
wc?;3Rgz%~W3iST#W~c_g7KccJ?mJKkKUg4lA}H--HSiv`l`v35^)l?KG@h}0J>FvGcSN7r7!`^>!9}iXAhx?W`xVUj_aZx>Hfi>74Q4+i007>A9S_3a+}Ky
vYiEAc=^ki@YFp!QKrxs0k)<nPjxM)59#_<E83@ZsY%C+cIsg^@}q*?x4g;7U-KKxIpM8ra0LqLKu_Aj6@Q<eN&8h$rRjC@r2@fkXl_o|bUE><kZ9i1!8W0h
pqeiF5KdXCA-wuKYO9-6H$**&kr@XSdi%Y`Bv+LmU|;f8erVVI1mRO(9dwiiM9DvcFnYaZCVjMeP28=q>WY86z1m3Sq<S5!Dt>8Ubso(oDoi6tLr9RSBr}0k
u@&*l@w?$APt(jv=lUNEwLLKH%qC}V^LH#TP40I_kYW0#sWb;{-h|AW%re|86Aq_D5<#!RSRMps4A=Joaj@ki*&ABX&fxFQ{aX=Fw3n&IE$nljcuTkZZK~44
2SBLmN+iO$Q7(5Lq{o~eS~M2F5d5wQIL<Z)J}Vq#i!eAFXSZ!U*TevvLinZd5@^&lO|4t5K2niMj8|mh=yLx_F!g}*YwykB=25*}@Ul>l5--E7QCW36#Topp
dlb_?AMIaIU#@=;l&-idy*qm3&-rdX=(X<7-Lwh&Bprd>jA1`eQANEQIA+-9q0<)xYBX>nf_5kmZdCbHhi>3pm!k|u6x_^^9H`6V^`q8_B@oqPG{v;%NbA*>
o}Nv={EV*?y5h~$!YV@lS<{%e>`zaO3(|T~idWDmp_y<S{QGWo=U`5z+A+wt<~ErRK%NWSORH$47XZXV0$IShtM`9>ovh{~Lk9cKpk>^5<l4onFse(;kce4U
(nYX_$xddWmLkX7lr*|eUl*s#?#HF28(fq{qFK%24o$*vPsOlm#NwIVSA9+;1@sqlxL*#{4jW6L&>Z=@R+US4wNb~Wu2l-0aCwbq9n-_iGhT>CD0mq!)O|r1
lA^J@A4I$`LW$o2oC)__Q<@ENFymxUkgyhG5?Ripo><mGJOgiuj4Ufvz&;Y`+%mTTpqr<TGT=0?-9d_fc9x2=rdu(g-}UD=Ac#@aDHX+0C`5hD{2zlc-ehID
dYt-bS-i9^CVUY~lkY3(_;(SN2b7u)*vY$MEslPQ^B%Wn(fBD4aoA^fF6Hi?KS!UW{_?>OFJ|P{R^yjTr{4tmv3APvylUh8o71$#eZ+hr%XB|M6V(Lz#%6t`
n5P~sA$Yfo0;Y)3!8A&tlW8SS?El(TvD(-}T@;Ie4|;aSOtI-6+^JG^(L@#~3*hltvTFh5c?*UAheFEe*pvMwqy;!VS?N~R865lIeIKl-OjPAvT~B-pcGn`0
B>VYNay^y7HV!pSFx=9;F9b~l&IOmhyxB`f=-H29&zpX@-xD;39QBK1nzgIYk(0^<Oob=!TpAJSeijYJ;+JD<b~gcF#{(z$GC;rh`MtYs({9>`{7dQG{cslu
gJIIu*SnjY-D71b&f$*h=n25hp%5<dLkBfy6(IF>B0$G#AT2MwIPKc6<7jTQ_hqsl;`cZShpw*$;F4R9b5pzxPOTg-q?Ubaj}+Ga)NU>7iT^MvwBy5ge@~+E
%9Tt4XQ8OTo<WtbUDCICdFa@;Z;v8YngV#9MD2Q12%WP;er48<3jZ@mptu7{qQPKMsk=baEjxkjKkNn$e7%W8?e7U0MzhbYDV5}GB#_6UV7yD$l8(WbZypib
X;7={47P9eBf|t%W`(OosSQzdz}*zrG`T5)JdMhR&YMi(haFj2AavDU9$e6^U&-^bU+O=0)>(OS@#4d~cpIQ=Ekh&Dan|-k;WVBbRIxP&|HFErTmn=2p!cQe
CoXSL!GKuetIfYSW$oj;=!RhFUPB>I%VH(=2Y_DdatQOgL^sQWd1*-Qt-(Y0E(^O1OxEc9PJdgjCp(@t9?e=qH=JoI@TZ&BI_wv+*!l9AyfPWrk@M>I{{_Bi
+70fLKnA=R!@{b0;U1B4Yf%>X4sPEc&t?)^-xPNXybc~cN*90Jt3ZJhr_DeZdZH(y(k_{p#UlItKMQA$rET~_#gC0B?54N`^rV^oXWrX#C8XyCYP|`kSk&wB
eWYc#rfgsg5Rjivc(m<N2AJ|EEMeAsAAA+7!Fpt=qS>!#TX1~QY+sN$7(NJMJB}6+qhYs_z4FbGn1AB+OZ#aMr8&g^MqZ?T_yjm}Vwpm07G`CzW*7lenF%o5
gCeHqTGg=QMDb<&cEeHBB>(jWj=EK{8?tUSCkGs<s{!%mr2I;d8NN~}hWU>W^mkEnxyjzT!;FH}TthP_2i+nqXj9waL;$gLU#?eN!I^4yI^d92X3K5{82vw9
xA-+y+neGnPHX_hXelHeIDnj}I{&fdX+s?(7}?VdlW#n{-j7SR*gaEed;mZcOM5Pbs5@E!1JWq<wn0AN)yvBNtF@k*m+K}ZpLVh0O#YcBh_o1|l68d`S7%uU
6wRSNWerIM{zLrp=QX9Vt<+(yOtW?Vh%%34S`K1S??iA7H3cl3uWr_rwj}h~LK?5bkG53*5rcqD{1}P!0y`QyJm!vU2a#cb)Ud~8snhm8f|T@z)?Z$`afs?p
Ze}|XzH;h`J_b@jv^%%~N6AQghu7^v05p6!?LCiv>J1}gt(f{>b&hA`9rE=Fk0Y6B6t9Bq?@}xuu2N)3UNqWC2nRYEoRP<`bEn<nGE8>mTdgqBi;fJiJqw5E
N8R}=X3XB9v_%KPl$5*}G1|@`uYKSNl$Z|!%st}GxKN{`+NYQ=NEyMR3K@S!;H7c3<T86BbtBOI?-7x`!WiF=%QCopN;`-xRuy-l#UiuS)-}Og_r7tAdpEc4
!>Wk(P=5Fez_5*TEaI0DyVk%qlUCNnFfIzeYUB_%4D@)C^Z6YIr_ryVbg3UUXIAOiV#(x*+k`VPO8p7sHpT|i>qM)t-rA0~0#6&nrdH*QID8btnLdO&N_I4+
Kjvpub@z9>g1Jm1?W_p&+gUv7M406)Z-rg1`>OW!Z-cmy5738yy6Sj;t^U;R+hDY+GbbX6zcnV@pmH^*1a=SCgL56TEPljd4?_fgBp}dTnXUqCH1Q3$SWByt
Dg&rU$@Q51iy=2-`XKc!ieF<_=+k_Mw((+vJV5;}6x#TdT=7xxuNGD84u#$OltA4pG^W=SblS8=&eIc(hM$EO(!)m}^PRl+cDHl@vgCNNHR&k;iP(}1eGB#5
s}=>0;fO~l>`<dRx!kDh)`ewHbMKIV${1*tu9pmYesWB9k0z~Rw)0>5MhlK4940E9m&k^3pRIa4q46O^{VY=w2^3RaWH2IhJl&Id+CiT7!du=LauXqkV)d$U
1=*Jv;Q{l6qU=)(tkAwexef?QzH+!I?jgm(=H3a3z$9OUxvc4^idonyGWdvRuXXXH!g>sj(EwXVNf2NY9y9-5P0?acgiu$KoiUr7T)+kKdNUwYTz(Z+)$cAw
0uR_>`#=m=@VJ%p%G2T7Sa^8SJQX1$)_gG$XXL3AkSyUx3Fpu$GsE0y>qN+DEKpndp?0X(+Ia|f?1+oS6Nj9sY!D6pi7gZ*G*p^U?MUI^XZr})xr;QGF$vi-
Vh%+M3ejbIEqYwMfG!~AxVBHO1GP*%HbrLE^IRceVD;XG`jc%=i2Q-1UDKYs7nMj;1-L{s2~?*-*M_akVQ)p})k)s4jzbs++yh84zQ)>XYVn!^pkKzT9WQjl
0*S_&fOOdBkhqMw`3`Z6X8km@6+9TNj6;l^@JQ165uEb0!USAg4LMA3nqis5qlSj!r9KXwoFME$T###-HJJw>VsS$D+VsRDfukvyWX9phUJ{Hn5I?zC)oDHg
h2oK#i-^2z8PI8gF>Ii80bQELKx!)VECFXIc=|s7HY&$0AZ+LW-?jzD?l?g#d)DMT(sBwn!-sy<PqQ|g-A1Q&8*|T5EAKoRD<lV*W|z-1n?T_&@NToDPJmB_
3{s-FOCZa7ha355J7db5VhGwL@BRjMrcR2z`X+BT0unVXTi2pN_7cGX)@!-=*kqcV|C4fhKNOtPL$G`uLI>nc`o9v?VEpo!$`&+mf$4(d(}Kg(>%C5|`$R*k
Xhwe~f$zccB5JXOyN3QofmHW$ThUq~c&kmdqF99zneXae5!X)d##D{FpwF>xcAy~*hU0*ydGnD`2db=FN$jPbNz{Jl6zf>e-<7SkTjZCLRCul%@)HH!X69`%
{6qy$J%yK21BEdV6uF9kxJb~C+w~OtK<c>pxdp}zz4CVz&DMY+LJotyxLF#c&dk7jXHMT@0M+0}YAt*h+Gf@Anx4fBQ8zgMk1OJuq740_dki;qa~KY4(zk6D
JZjplt>4B#R&4zueDLUT3tw>aOo(r8XkTb}zwLZJyZbY*`=p`~lr@jW%_yb|3*wgZGoh;!zaY%Sea0Z^q7Y<Uk=|d3_}~y<=hVPH<8}`SgW6CvAWU`stZMNv
xzh4y98EiKAYwu~9tkQ-R-%5{NgpD|&O2nC`61yY7mu5KE9jLZzNHA2Q5Q1x_3-T+&R;$xD9)hBv@-S0P_3Q@F)@N{I{6#^%jDu&GoGYrKYfj>@ywSdh%|qx
M`OXofmMJTv|v(do95)wVr)<_{>H^Jh^mN@AKqjY)%7_IwpSWOde$=I3^B*{ypJ;<L`C~z(}A(x6Fj0aVUUnCd}!rm`LDHx{TI_GRRpHOeJHOCUx<zRd-0yv
fAhrBL<q?i${UCYbFyTYW>kF$Ql_D^hGDnSG38jMY-wnFC!CU8GWyShKlIYm(a|*E5@%=`j^ZyeFpI->FY=$bn_G&`lj~Eu3|#q$2Zo2SxrZL1mzZli$(C8Z
0UWxDq}=fH=MDx?hD1uMN|L>;2CEDw79@|F&X`1zDK&(hjf?i{4*fuaE(#GK^-xSvADIK;-+AM?_REAVol?G_W}nF$qv=vsjgxvF$H^VX$~tKq!_0B{WOm0F
0wu;L95lxUnk};jG^uzNnJzBhAq^|&?t#nH*^Daw&(&23Q>gXofU^(fhdY8%rTy?25Ja1hUHZCMJ2yx}HmaKA9HcA>T^RrY7SR#rY`pNQo}k{F2SrylJ6=1C
m5=pNBMXC3HBuT?3DBrF?$<<cL`Mcn)zJX;VwP(Ko{mS5N*>~ni<qVK95%BHz~_3d=7fC(2(jd|H_5=O)nVoPZmgQX`PK^o=euc$-5L>6UM_A~33>Y{`-j9@
YJpA=GQ(txw08%l9jKmIjbcf^_l)_2%%s~>F81Js<whn>O9+X@Skg<<=#txfG8s~F!@PVYI<>~0&yb19ok;5ExWcAo1UKi=E5-~Ocq=m^ALTdaOK4)K`tn*j
!e}Q#M4;_e2eiicXXb*soq#FmiiZH&rNG$fja!5A*<>xcq+APy!qglcr4okE?06uBAXS9|Qwr8Wn^BXhSi76@)@Yu5Uj=q_jWZXwf$65*TC(secETwJ#dPIL
4RH`?o=ckKyvYjDMj*NlhfRL^K95N}3nyFtzDzrQyU7?vfSisFLzdV{6UHq)%D#>!{2M&^s5LYvqhFR*>-Vxra5qz+W^k4|2qdWmJr0_r{FDqnnFu_&EP6gg
&`xN>icPu8`>rLxU#QUTsh0Z$6yO!PNJSgTymvjy$<*1Y4z$1I-t|t*aB0dUZs63?L9!({1#O--jH&L9rC-!Ls5gor+skxfzh{pmrCSPm3XsrtWBO`M<XI7u
Rc$PF^~^|GPIC)bQht%mlytxvkWVm?Kna#|QLNF3+cs)YMXc+Iuf!<Wrtgsz*`$e2EW~Ey6j1zP)ix^+%q&zcSI4aN7}QI|jt)SO$!jlM8gW@y!tE@NHinZ&
IqpRDn=j#=ajxLEPAvwN20Y*BZxC2ziy(5TYY@i>A;f(FY>!+>F1+*MU}Y+dn*XTzi>g>#*!+)Gx0N;9GP;o-5?f?OMml3ia<$s}#B{6k1f_VK$->`XKAM5P
1Kg>YNPdr7Tk`dp0(b`pQGEcJ5~A~UbV`Qu-8MKWIUM1|bI9Fv0`)5K0{AQ@vziJMmdsOy07uE$r`sx`{4A)F*vFv*YGPgcR!F*xuG76b%+BryeFig025Co6
=1IR*b*+UK4-@k0iG(<uTF=iBpiU6|5<GfeEUESB6YHvdH&Ed;UbaYL3YR;5`3ibe)Oca|BRg-(_q-Nh{?}zf!!ZZyg4(2}^}4Lh#)}#*g;hx&&CE*(@MHoH
m!ex142nP9hYtaf<Sjo+Fy`e^-INo+!@EDx`ZVj^OmlZG|FoCVUuwLb^vTcE<YrqBz&*jjB;NLA<E~#(z7a>RS*cpypxL}sta>%Bntz@Pn{Q7L`2CI(d)efv
9HT^Uno$BRK?)x~=dML8fowL5*|x%3S;t(k3xsZdMt76&MKfXLwi4ekqat6P%;fU6Gz=7LtxE=**LXAg?4ITRid1H^NL9yE`~XrCc8Lp~)Nh*{ZU`9rlTd7I
kJc2VK19x<u?gABT=%k^A@d{7tyesTZ7#*)wW+IO__S6iR^)|>*DI9N-3MM%O8d}4R{!E@g$<6mkYVvKdwWqHakmA@9y!2$ln4rU5S5`foS}UAqutYghIV6<
gYjI79=RlPdD)82%lVkgjA9=>ACHhKtIuU2I%hki&Of)m3CftV$ETht*+xEIF$7Y$95NpeBm&5wgvz|mB;R@C&J1A((!3t5ZvkE?$5?p0p5J0cI`p4~Cz^8=
gucWk?a5f8#Sv+*+{wzhj~GZKCY51PW5G*_AcR=@+X;cWt{+Iee9u$fN(9X%nA7i_ff8%$`_fJ^tYzwjb&+V~6Pj+Fu>FtT2qJZ>ZF0_NnA4^Styy|6UiuV4
tk(i3Lp0!b`|4en9_|X6qc>i@*#$kb$5l|6F}GoNF}`F%2AP)48m~{>5B=av@~Zy~Edutc!h5(={{J^8H0P-`vD7S}75ucvZZZ4vx`%viZIR{<*?u?jlGVj%
+B=O-(B5~sk(Z6rRq9J_xk+Nc9szyAq}lIG$(6?!yK0Jop+uLq=Smi|J0bFamsaz4UNz=&(nb?ME^v!*YkO@5#GSq7v7e^;XSrO*b{k&YIVT<O{<PYV(=X-A
vh%V#hpluG5kr`4^ZmLj(7g-|_ufZ5ZYlLK{aSWV`m3#O+Ly@+D_Kra^U&z<OJ#ZeScMN<<+FN5XneYg00J}DWG-0CW>rZT(NIjqY+aDEa?qMcdf?x*D?3#Q
XJ49_<|gvXAxyDDDE6(3#Ud}l*GrI@FP6_<@F?&O*#tYJe_4BH3&Uel)N6*^px?t~IImy^2hO;sI5bPGUPB!uOM-!AX7U<wl*=6X-=f$gie$+}h`;lE2gtx%
N}Q2Z?{wvVb`4*86N<c`b}XN`$XdegpR~9*;x+(RrYY*$xfMdYU(q=ii@gqJ3m}m3tm9jQlr9E$`cUgh3KVYOjs&5T(;Vg=Y6=SW32h1`X5B+2n`m%O%c?v0
fQQaS76^Dm)A9oy3-K`1krRLsiOu-fv9N8c)w7Gi1=Skp2n+(a*>O0uIR=ey<3|_L;ty`sZI6Edj1;xM=Ua5KSxWI!kd(!+fPGa8@^E3pUu~(HYvZ?F4Xn_3
GMF&(%%wXJq<5WB<Av@QpmT$HT<>dEeIH?ku=~sEU<qB3kk$RPbSY9Z9HXaQM>)>ML|9<w>Fd-#rZ1by6}Q5j63tO%BinVGhhJTKUzJ*4N-NZ@rU-<t7rtBk
^wqmM2lK6=d=y$KaAD-(Eix7|xS+;GNT}X4fROD0=YFSfx)6ABZ!P_LuO@wMH^Y!Or(r44q^;*7M=!AU=*jk~w$u#&rZJI@qp6eIZC6^R<5~j7BfEPg`Q~Tz
QGsf@Ej<HkJX}<}kE+E#f*w~HcQKCgsvq$&qZ#Z^k8Bl`-z-ywjVX&{+ow>Jb`=j1l3UAyZH6(nc(Epu$vT}WWl|LR-`$}@Cq}fkM!A?SqBTG+zYF8O-5l?h
!3B$#D5XS&a(N;NF+5d%^|mfmTf30>y=+a#*0YGG%OXSVQhyGv55=*R?KO;A{DwZCI(LX7gy_|Bxp($<tx>T2&~*8sq1=anyjs&eFUH(6i8e=gxz1wC70hO+
knN!pqETlN@gVj0HZi=%gO)7>m@l0;GmZkUOwKFXy5jU%UN*&jml43yi{v5oN~1FgoX`My$J~+RJ$(P17;j9-F-+)oZ5}`EdMY?%*)8LHT`7Msi3NaV&jO>r
`ZKkX0~$Nye8-u<K{@oG3T@iH-Ypq08tLL?(yyt;*BV9S2=7o-Ar=KxCxe2ZIt#DE$;QaJPo19X!zm@{y|CLpd11vjIJS~h3FL)I$2ZVZgPJ(QFp9vTmfm0+
HwuJ2ckIT6#FKsz9wVrE;I>Dti02Q%yjMw9uCFjXxE=f!ZM%q?GjD=kK<FiU3%7cyyj~sm(2DLlckT0vr~}`ZIJ!AWvmjSgw$T#4JvHpzK41$`D=R>{JW%=h
{dh6`aw0M(?+XbCiSMnD{NHW0zd5NlO4CV`H;IBY`3%meaznV}tB$j!fvtJtEW)M75(h>cBUv4+^Cn~kGvkUzAB40rx&v9@qzB|{pKN-eS7M4q`xP|#=Di}c
20S@~%OJ`>v5eFSzSk9w{Z|Bl32YJ_BibIB7NSm-7AJ0Rdz~*3(~8`)<H!&<)F`+H^m3Mg)76$b3N-a{Q9-WJI5prV3IfJY<oJp~n4b0=-RnvyRXYnm_4Ll@
#ZXnjAgR4i7DHil<b_9RWX9L}Jj<YbSp2t8V>RU=ozcTA4b*-V!|!El8YyA&*DEjVXws`Tp*e69_H<H786ZmLK2CrsQJ#DGdy<q#cUuG}i=Vg<_~y#YRzt0j
&C(H3%#P9AXn!Gw)qUk!fOZ^pX~(TNGHWGanKs4-d#xtCLs+Zu`oC!wmDZI%+4&TEdgOcehcSju3&Z6Cb9aFlYOeB8-q0HtXXL8@OzG#m-jvV=;^GpuZX9>g
IBIB<Kx5Q?(gwJ>&-qiD%zmK6={qWS#UIkjHY{QIr3C1BBF#NtV~DkK9Ru2?mxZ*4CDG1es~la>P*fTrTl?Az2Dhgb5(ujR;L%7$Rn)I0T5?lTlUe_U;*S+h
eghppQ5%aO*h+l?)U*h}1i+N8W)wOD)Ca(z3-B_#koy(XDF42f`2#TliQ6)(vD|fNf!K!*7cIuBi&irkD~&Xal-wt?X7>+Ep_ZLQGUYWOOQ(drU(7(&7se;x
G29(QaPHT$uJ1cp7c<2Kd(|H9f@|jR<tS5^IXHy~CQ?HO<T&Zy8F-Szvh_*LOx+mqUHR-l;j%}eE#gK4?yo@gej);GpO(nXJt)vYZ6Yv_1J+Il-HjFyREr5l
r%q#AlC;eAKys06evY0;y*OoID=NmLK5?$1s~$nQc3hdQ`pVM56QTEo7KeA6A^!{FSrB@H5}NFE?!95F_6Wyty^~(AN7{j_vF9k!m}5K1Tjc?LP4NEKyYhS7
$Z0h$U<7igB{F)p>i=d5t4_DRejaySkGgE&4Vz{E(JQ7AoyQka0oI)VkI9EJn;$ceV8k3=v{px5^PRLf3c@T(&E%2}j!gxWA|RTHjg+w3EH%4tu4BbKIMjl}
l#G7AZ;;zJw}ipH8rJchi0K~9K^K7V-ml7m6h7_Vl!*xj-=mymM5X^{WBG0_fq^NR-IjG-_le+;r6fgaodhF+p!M4l<&ofpdn?Ll`nR#f&eBg+4H0X_Ch9g_
Pr&8#9klJt469;L;H&}x|G67uCPNiHWQsdOh3sVi>=)G99DE2VmpC8GUC^G3#&B|9Z~GUoRfC(`yES0Rufc=LyLQzIF+9l<0nj0vK;of)T@=O%f)rBsPA$Vk
D2v*$Awk-S(6?PoL{baf&9`78YIR27*77c>#evA1qiZVlA^<Lj)-_6ue}tR~m>SCyCZe=(_W999RAskd6%pSJti+L;u@+ErC*o+y6*tpq808e?N0~Pjs4iDR
59YQ%b$6#-X5cOK5u|h&JEfE%QZ`NKu3P_7?40)vcOuFKJ|mvucGhds<E)%&02xpYDRO*@lKr{kK1MPH$B<r<$7Ak#@2JlbJS3R(Ovr5v4RkYgFj3~tvd)Db
Nyh~n+cji_iWAEdW`Y98hMS=3pw=!c9mN_=0^&G3O{x}?KJ8j%{PJ4*XLHqfuNqTR*Y1nC$tHiMF7UqU>V(M=+^w#WrRI2h7MBLRR7$FO9EYKfF5?>>6fHE3
G;IatK}IV$4#0i%hj#Z-aiX=pkC}x%vgL|RLzK{SdtiU+L9a%{l>VoXSfGw=P<tCDXZ!i;oD=j>+#fGBpN`>6f}R-809TE*A32z$xW1kSo#hsI^#E$p&|0&V
<U$+(f=9O%I7PIVBX!1uAG=}KC(0gihpvEA?HTV2o8w=;{EpQT^3vQ$S$~^t3z6dM*v5IcY*tVcYQKvtMoJa>H(^)`N6@@iWO<fv^jP-d^wf_S16nxnvIS=X
`F2YEM#9<kQ0?2WTuWENq>A1>B0Z2Nr%Dm4LqsfkG~cahq!pZ5YFP8*_LE*g=2AfNrrgb3VcPmFJap`tqtbC>*TS~TB%0bKXtD+W)xg8x?7_<!lJaBTdnsAf
^A}c-Ocl;xZpLLPMH3txVbWr-_2!FI1+xErbJonbNNmsfQ*wh2e*XUNa0=OUdtc2n$X#oK6fpn?Z<4FmWba(>yxFiTux#Z?3VT_*i5^o8GR{lRbk*-n|Eyo_
_d%h>m7!z_LF*|k{UGoJE693WXRPLOaJ=@+R`<ae{aWCh4x7$^FTK8&3>r+U)qb53U7Gg5?I;m->8=hb>5}z$Da5g9{^F3mny6ah_FtGJ14L2<q|G5XEK@ZD
nWfcwtWt@~5QtSf%Wm26L4G1MIQq|MTB`t+keKkpSh{auskzHt2v+e}na|J^+5oQ*d^v(}QBOPb+H;Dr@onpN2l~ITxGi>or^BfZX*tHZ+z+q6G`75XHt6Ze
K^el|JtqYU^Z<1FfD?sT-HLa4bCtv7uQ?YFx^nwgM@{eU5SSqg`?`{<mecC|dLBK%VDKm*Gw*2J6Ys4TNoFpbN!dyq17u?)XbWLqLG7TfTRjnAnzBo3H@%&y
AZ0LrR_4t>QJ~dcMZC)^$LBY+b%Ai*{FHRss`y3Tn(UtXBqqzQyI4)49({k>l09$Qx)+8)jTLEe^>eL)(p3AnL5T26qEiy#Z+wdLvNX^YFU^1jyXlc}0hLrh
NSML~dOg?wfhb0sz5a*7T5M?7KCgVZgAWefud1&$KVFx3J&m3Yc%`cdV8BK^_Ql1$^>AI<%~N*G{=NVJg0cNJK{&TO00G>{1kn2!)TslRvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/treino_v2.csv')

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-u', '/tmp/nutrelle_codigo/treino_gpu.py', '--rodada', RODADA, '--saida', SAIDA, '--orcamento-h', str(ORCAMENTO_H)] + EXTRA
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in p.stdout:
    print(linha, end='')
print('código de saída:', p.wait())   # não levanta erro: o que ficou pronto continua na pasta de saída

In [ ]:
import json, os
m = os.path.join(SAIDA, 'manifesto.json')
if os.path.exists(m):
    m = json.load(open(m))
    for nome, c in m['configs'].items():
        print(f"{c['estado']:14s} {nome:46s} serviço: {c.get('servico', {}).get('estado')}")
    print('paridade (sétima execução: R4c e H0c-m refeitas × sexta):', m.get('paridade_d1'))
    print('recorte pelo prato:', {k: v for k, v in (m.get('prato') or {}).items() if k in ('regra', 'consultas', 'reprovados', 'n5k_ok')})
    print('confiança:', m.get('confianca'))
    print('bases externas:', {k: v.get('situacao') for k, v in (m.get('dominios') or {}).items()})
    print('tamanho da saída:', m.get('tamanho_saida'))
print(sorted(os.listdir(SAIDA)))